In [85]:
import pandas as pd
import numpy as np 
from scipy import stats as stat
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation

# Behavioral Analysis

In [87]:
## Behavioral Analysis CCT (PCA First Factor Scoring)
import os
import re
import pandas as pd
import numpy as np
from scipy import stats
from factor_analyzer import FactorAnalyzer

# Path to your folder of flat behavioral CSVs
file = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Behavioral Data\data"

# Function to extract PCA1 consensus and eigenvalue ratio
def CCT_average(df, return_ev=False):
    fa = FactorAnalyzer(rotation=None)
    fa.fit(df, 3)
    ev, _ = fa.get_eigenvalues()
    ev_ratio = ev[0] / ev[1] if ev[1] != 0 else np.nan
    consensus = fa.transform(df)[:, 0]
    return (consensus, ev_ratio) if return_ev else consensus

def clean_for_pca(subj_dict):
    """
    Cleans behavioral timeseries data from multiple subjects for PCA input.
    """
    # Step 1: Find shortest response length among all subjects
    min_len = min(len(v) for v in subj_dict.values())

    # Step 2: Trim each subject's timeseries to that length
    trimmed = {subj: np.array(arr[:min_len]) for subj, arr in subj_dict.items()}

    # Step 3: Create DataFrame and drop rows with any NaN
    df = pd.DataFrame(trimmed)
    df = df.dropna(axis=0, how='any')  # Drop rows with any NaNs
    return df

# Load all CSVs
data_by_video = {}
for csv_file in os.listdir(file):
    if not csv_file.endswith(".csv"):
        continue
    file_path = os.path.join(file, csv_file)
    subject_id = os.path.splitext(csv_file)[0]
    try:
        df = pd.read_csv(file_path)
        df = df.drop(df.columns[0], axis=1)  # Drop first column (index)
        for col in df.columns:
            if col not in data_by_video:
                data_by_video[col] = {}
            data_by_video[col][subject_id] = df[col].values
    except Exception as e:
        print(f"Error reading {csv_file}: {e}")

# Separate valence and arousal videos
valence_videos = {k: v for k, v in data_by_video.items() if 'valence' in k}
arousal_videos = {k: v for k, v in data_by_video.items() if 'arousal' in k}
participant_ids = list({pid for v in data_by_video.values() for pid in v})

# Storage
results = []
mean_results = []

# Loop over subjects and compute correlation with PCA1 consensus
for subject in participant_ids:
    valence_corr = {}
    arousal_corr = {}

    for video, subj_data in valence_videos.items():
        if subject not in subj_data:
            continue
        try:
            cleaned_data = clean_for_pca(subj_data)
            consensus, _ = CCT_average(cleaned_data, return_ev=True)
            target = pd.Series(subj_data[subject])[:len(consensus)]
            mask = np.isfinite(target) & np.isfinite(consensus)
            if mask.sum() >= 2:
                r, _ = stats.pearsonr(target[mask], consensus[mask])
                valence_corr[video] = r
        except Exception as e:
            print(f"Valence PCA error for {subject} - {video}: {e}")

    for video, subj_data in arousal_videos.items():
        if subject not in subj_data:
            continue
        try:
            cleaned_data = clean_for_pca(subj_data)
            consensus, _ = CCT_average(cleaned_data, return_ev=True)
            target = pd.Series(subj_data[subject])[:len(consensus)]
            mask = np.isfinite(target) & np.isfinite(consensus)
            if mask.sum() >= 2:
                r, _ = stats.pearsonr(target[mask], consensus[mask])
                arousal_corr[video] = r
        except Exception as e:
            print(f"Arousal PCA error for {subject} - {video}: {e}")

    all_videos = set(valence_corr.keys()).union(arousal_corr.keys())
    for vid in all_videos:
        results.append({
            "Subject": subject,
            "Video": vid,
            "Valence_Corr": valence_corr.get(vid, np.nan),
            "Arousal_Corr": arousal_corr.get(vid, np.nan)
        })

    mean_results.append({
        "Subject": subject,
        "Mean_Valence_Corr": np.nanmean(list(valence_corr.values())) if len(valence_corr) else np.nan,
        "Mean_Arousal_Corr": np.nanmean(list(arousal_corr.values())) if len(arousal_corr) else np.nan
    })

# Finalize DataFrames
video_corr_df = pd.DataFrame(results)
mean_corr_df = pd.DataFrame(mean_results)

# Subject-level overall performance (subject-wise mean across Valence/Arousal grand means)
mean_corr_df["Performance"] = mean_corr_df[["Mean_Valence_Corr", "Mean_Arousal_Corr"]].mean(axis=1, skipna=True)

# ---------------- SUBJECT × VIDEO PERFORMANCE (original per-suffix rows) ----------------
subject_video_performance_raw = video_corr_df.copy()
subject_video_performance_raw["Performance"] = subject_video_performance_raw[["Valence_Corr", "Arousal_Corr"]].mean(axis=1, skipna=True)

# ---------------- NEW: Collapse _valence/_arousal to a BaseVideo per subject ----------------
# Extract suffix and base name
def split_video_name(v):
    m = re.search(r'_(valence|arousal)$', str(v))
    suffix = m.group(1) if m else None
    base = re.sub(r'_(valence|arousal)$', '', str(v))
    return base, suffix

tmp = subject_video_performance_raw.copy()
tmp[["BaseVideo","Suffix"]] = tmp["Video"].apply(lambda v: pd.Series(split_video_name(v)))

# Pick the single correlation value per row (the non-NaN of Valence_Corr/Arousal_Corr)
tmp["Corr"] = np.where(
    tmp["Suffix"].eq("valence"),
    tmp["Valence_Corr"],
    tmp["Arousal_Corr"]
)

# Pivot to get both valence and arousal columns per Subject×BaseVideo
pivot = (
    tmp
    .pivot_table(index=["Subject","BaseVideo"], columns="Suffix", values="Corr", aggfunc="mean")
    .reset_index()
)

# Compute Performance as the average of valence and arousal for that Subject×BaseVideo
# REQUIRE both present: if you prefer averaging when only one exists, replace the lambda with np.nanmean([x,y]).
pivot["Performance"] = pivot.apply(
    lambda r: np.nan if (pd.isna(r.get("valence")) or pd.isna(r.get("arousal")))
    else (r.get("valence") + r.get("arousal")) / 2.0,
    axis=1
)

# Final subject×video combined dataframe
subject_video_performance = pivot.rename(columns={"valence":"Valence_Corr", "arousal":"Arousal_Corr"})
# subject_video_performance columns: ['Subject','BaseVideo','Valence_Corr','Arousal_Corr','Performance']

# ---------------- NEW: Video-level average performance across subjects (combined BaseVideo) ----------------
video_performance = (
    subject_video_performance
    .dropna(subset=["Performance"])  # only subjects with BOTH valence & arousal
    .groupby("BaseVideo", as_index=False)
    .agg(Avg_Performance=("Performance", "mean"),
         N_Subjects=("Performance", "count"))
)

# --- (Optional) save outputs ---
# out_dir = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Behavioral Data\derived"
# os.makedirs(out_dir, exist_ok=True)
# video_corr_df.to_csv(os.path.join(out_dir, "video_corr_df.csv"), index=False)
# mean_corr_df.to_csv(os.path.join(out_dir, "mean_corr_df.csv"), index=False)
# subject_video_performance_raw.to_csv(os.path.join(out_dir, "subject_video_performance_RAW_suffix.csv"), index=False)
# subject_video_performance.to_csv(os.path.join(out_dir, "subject_video_performance.csv"), index=False)
# video_performance.to_csv(os.path.join(out_dir, "video_performance.csv"), index=False)

In [88]:
plt.hist(mean_corr_df["Performance"])


(array([ 1.,  3.,  1.,  5., 10.,  4.,  4., 10.,  2.,  2.]),
 array([0.39197614, 0.4294376 , 0.46689906, 0.50436052, 0.54182199,
        0.57928345, 0.61674491, 0.65420637, 0.69166783, 0.72912929,
        0.76659076]),
 <BarContainer object of 10 artists>)

# Survey Coding

In [90]:
### Survey Coding and Analysis

#Emotion Quotient - Total scale scores are calculated by reverse coding items 5, 28 and 33, and then summing all items. Scores can range from 33 to 165, with higher scores indicating more characteristic emotional intelligence.

#Austism Quotient - “Definitely agree” or “slightly agree” responses scored 1 point, on the following items: 1, 2, 4, 5, 6, 7, 9, 12, 13, 16, 18, 19, 20, 21, 22, 23, 26, 33, 35, 39, 41, 42, 43, 45, 46. “Definitely disagree” or “slightly disagree” responses scored 1 point, on the following items: 3, 8, 10, 11, 14, 15, 17, 24, 25, 27, 28, 29, 30, 31, 32, 34, 36, 37, 38, 40, 44, 47, 48, 49, 50.

##Empathy quotient - “Definitely agree” responses scored 2 points and  “slightly agree” responses scored 1 point on the following items: 1, 6, 19, 22, 25, 26, 35, 36, 37, 38, 41, 42, 43, 44, 52, 54, 55, 57, 58, 59, 60. “Definitely disagree” responses scored 2 points and “slightly disagree” responses scored 1 point on the following items: 4, 8, 10, 11, 12, 14, 15, 18, 21, 27, 28, 29, 32, 34, 39, 46, 48, 49, 50.

import pandas as pd

# Load the survey data
file = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Behavioral Data\FNiRS EI questionnaire - FNiRS EI questionnaire.csv"
df_raw = pd.read_csv(file)

# Extract PID column
pid_col = df_raw.columns[1]  # Assuming first column is PID
pids = df_raw[pid_col]

# --- Define question column ranges ---
aq_cols = df_raw[df_raw.columns[7:57]]     # AQ: questions 1-50 (cols 8–57, 0-based indexing)
eq = df_raw[df_raw.columns[57:90]]         # EQ: questions 1-33 (cols 58–90)
emq_cols = df_raw[df_raw.columns[90:150]]  # EmQ: questions 1-60 (cols 91–150)

### AQ score

agree = [1, 2, 4, 5, 6, 7, 9, 12, 13, 16, 18, 19, 20, 21, 22, 23, 26, 33, 35, 39, 41, 42, 43, 45, 46]
disagree = [3, 8, 10, 11, 14, 15, 17, 24, 25, 27, 28, 29, 30, 31, 32, 34, 36, 37, 38, 40, 44, 47, 48, 49, 50]

scores = {}
scores_list = []

for idx, row in aq_cols.iterrows():
    a = sum([1 for j in agree if 'Agree' in row[j - 1]])
    d = sum([1 for j in disagree if 'Disagree' in row[j - 1]])
    subj = df_raw[df_raw.columns[1]][idx]
    scores[subj] = a + d    
    scores_list.append(a + d)

aq_scores = list(scores.values())

# --- EMOTION QUOTIENT SCORING ---
reverse = eq.columns[4], eq.columns[27], eq.columns[32]

score = pd.DataFrame()

reg_score = {
    'Strongly Agree': 5,
    'Somewhat Agree': 4,
    'Neither agree nor disagree': 3,
    'Somewhat Disagree': 2,
    'Strongly Disagree': 1
}

rev_score = {
    'Strongly Agree': 1,
    'Somewhat Agree': 2,
    'Neither agree nor disagree': 3,
    'Somewhat Disagree': 4,
    'Strongly Disagree': 5
}

for col in eq.columns:
    col_score = []
    for row in eq[col]:
        if col in reverse:
            col_score.append(rev_score.get(row, 0))
        else:
            col_score.append(reg_score.get(row, 0))
    score[col] = col_score

eq_scores = list(score.sum(axis=1))
eq_scores = {i: s for i, s in zip(list(df_raw[df_raw.columns[1]]), eq_scores)}
# After EQ score computation:
eq_scores_raw = list(score.sum(axis=1))
eq_scores_dict = {i: s for i, s in zip(list(df_raw[df_raw.columns[1]]), eq_scores_raw)}
eq_scores = [eq_scores_dict.get(pid, None) for pid in pids]

# --- EMPATHY QUOTIENT SCORING ---
emq_agree_2 = {1, 6, 19, 22, 25, 26, 35, 36, 37, 38, 41, 42, 43, 44, 52, 54, 55, 57, 58, 59, 60}
emq_disagree_2 = {4, 8, 10, 11, 12, 14, 15, 18, 21, 27, 28, 29, 32, 34, 39, 46, 48, 49, 50}

emq_score = []
for i, row in emq_cols.iterrows():
    score = 0
    for idx, q in enumerate(emq_cols):
        q_num = idx + 1
        response = str(row[q]).strip().lower()
        if q_num in emq_agree_2:
            if "definitely agree" in response:
                score += 2
            elif "slightly agree" in response:
                score += 1
        elif q_num in emq_disagree_2:
            if "definitely disagree" in response:
                score += 2
            elif "slightly disagree" in response:
                score += 1
    emq_score.append(score)

# --- CREATE FINAL SCORED DATAFRAME ---
survey_scores = pd.DataFrame({
    "Subject": pids,
    "AQ_Score": aq_scores,
    "EQ_Score": eq_scores,
    "EmQ_Score": emq_score
})

# # Optional: view the result
print(survey_scores)

   Subject  AQ_Score  EQ_Score  EmQ_Score
0   pid001        18       103         19
1   pid002        12       140         17
2   pid003        29       113         29
3   pid004        15       116         20
4   pid005        21       136         22
5   pid006        16       136         13
6   pid007        22       135          9
7   pid008        13       124         19
8   pid009        10       134         27
9   pid011        21       130         16
10  pid010        23       126         13
11  pid012        26       142         13
12  pid013        18       129         10
13  pid014        16       123         24
14  pid015         7       148         15
15  pid016        12       133         12
16  pid017        17       129         21
17  pid018        26       109         16
18  pid019        15       152          8
19  pid020        10       120         15
20  pid021        23       114         10
21  pid022        15       145         22
22  pid023        12       131    

C:\Users\pkell\AppData\Local\Temp\ipykernel_31048\1902888111.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  a = sum([1 for j in agree if 'Agree' in row[j - 1]])
C:\Users\pkell\AppData\Local\Temp\ipykernel_31048\1902888111.py:34: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  d = sum([1 for j in disagree if 'Disagree' in row[j - 1]])


# fNIRS Preprocessing + Mean ROI ISC 

In [92]:
## MNE PIPELINE - Task 

# === Pairwise ISC pipeline + IS-RSA (Spearman Mantel, NN & AnnaK) — Z-AVERAGE PER VIDEO + ROI ISC + MC Correction ===
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from itertools import combinations
from scipy.stats import pearsonr, sem, rankdata, spearmanr

import mne
from mne.preprocessing.nirs import (
    optical_density,
    beer_lambert_law,
    temporal_derivative_distribution_repair,
)

# -----------------------
# Paths & constants
# -----------------------
root = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\fNIRS Data\snirf"
# root = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\fNIRS Data\clean_snirf"
save_dir_corrmats = os.path.join(os.getcwd(), "roi_video_corrmats")
os.makedirs(save_dir_corrmats, exist_ok=True)

durations = {
    '1': 60.36, '3': 60.55, '5': 50, '7': 95.35, '9': 86.12, '11': 51.12, '13': 48.56, '15': 58,
    '17': 103.8, '19': 46.59, '21': 83.16, '23': 67.63, '25': 60.36, '27': 67.83, '29': 158,
    '31': 33.23, '33': 35.78, '35': 37, '37': 51.31, '39': 75.49, '41': 129.95, '43': 81.98,
    '45': 145.29, '47': 46.59, '49': 58.19, '51': 94.37, '53': 34.99, '55': 55.44
}

# Define ROIs (HbO only)
roi_definitions = {
    "Left RLPFC": ['S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'],
    "Left VLPFC": ['S3_D1 hbo', 'S3_D3 hbo'],
    "Left DLPFC": ['S2_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo'],
    "Left OFC": ['S1_D15 hbo'],
    "Left STS": ['S5_D4 hbo','S5_D10 hbo','S11_D4 hbo', 'S11_D10 hbo', 'S12_D9 hbo', 'S12_D10 hbo'],
    "Left TPJ": ['S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo','S13_D9 hbo'],
    "Right RLPFC": ['S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo'],
    "Right DLPFC": ['S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo'],
    "Right VLPFC": ['S9_D7 hbo', 'S9_D6 hbo'],
    "Right OFC": ['S6_D15 hbo'],
    "Right STS": ['S10_D13 hbo','S10_D11 hbo','S14_D11 hbo', 'S14_D14 hbo','S15_D11 hbo','S15_D13 hbo'],
    "Right TPJ": ['S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo','S16_D14 hbo']
}
# # Define Regions of Interest (ROI)
# roi_definitions = {
#     "Left Hemisphere": [
#         'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo','S3_D1 hbo', 'S3_D3 hbo','S2_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo','S1_D15 hbo'
#         ,'S5_D4 hbo', 'S5_D10 hbo','S11_D4 hbo', 'S11_D10 hbo', 'S12_D9 hbo', 'S12_D10 hbo','S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo','S13_D9 hbo'],
    
#     "Right Hemisphere": [
#         'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo','S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo','S9_D7 hbo', 'S9_D6 hbo','S6_D15 hbo'
#        ,'S10_D13 hbo','S10_D11 hbo','S14_D11 hbo', 'S14_D14 hbo','S15_D11 hbo','S15_D13 hbo','S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo','S16_D14 hbo'
        
#     ]
# }

# -----------------------
# Preprocess & epoch
# -----------------------
# def individual_analysis(filepath):
#     file = mne.io.read_raw_snirf(filepath, preload=True, verbose=False)
#     # enforce durations for odd triggers (videos)
#     for i, desc in enumerate(file.annotations.description):
#         if desc in durations:
#             file.annotations.duration[i] = durations[desc]
#     # drop even triggers
#     even = [str(x) for x in range(2, 57, 2)]
#     unwanted = np.nonzero(np.isin(file.annotations.description, even))
#     file.annotations.delete(unwanted)

#     # Standard fNIRS preprocessing
#     raw_od = mne.preprocessing.nirs.optical_density(file)
#     raw_od1 = temporal_derivative_distribution_repair(raw_od)
#     sci = mne.preprocessing.nirs.scalp_coupling_index(raw_od1)
#     raw_od1.info["bads"] = list(compress(raw_od1.ch_names, sci < 0.5))
#     print(raw_od1.info["bads"])
#     raw = beer_lambert_law(raw_od1, ppf=0.1)
#     raw.resample(1.5)  # consistent sampling
#     return raw

def individual_analysis(filepath):
    file = mne.io.read_raw_snirf(filepath, preload=True, verbose=False)

    # enforce durations for odd triggers (videos)
    for i, desc in enumerate(file.annotations.description):
        if desc in durations:
            file.annotations.duration[i] = durations[desc]

    # drop even triggers
    even = [str(x) for x in range(2, 57, 2)]
    unwanted = np.nonzero(np.isin(file.annotations.description, even))
    file.annotations.delete(unwanted)

    # Standard fNIRS preprocessing
    raw_od = mne.preprocessing.nirs.optical_density(file)
    raw_od1 = temporal_derivative_distribution_repair(raw_od)

    # --- SCI + hard-drop bad channels ---
    sci = mne.preprocessing.nirs.scalp_coupling_index(raw_od1)
    bads = list(compress(raw_od1.ch_names, sci < 0.5))
    raw_od1.info["bads"] = bads  # keep record for provenance
    if bads:
        print(f"Dropping {len(bads)} bad channels (SCI < 0.5): {bads}")
        raw_od1 = raw_od1.copy().drop_channels(bads)
    else:
        print("No bad channels to drop (SCI < 0.5).")

    # Continue with Hb conversion and resampling on the pruned data
    raw = beer_lambert_law(raw_od1, ppf=0.1)
    raw.resample(1.5)  # 1.5 consistent sampling

    return raw

def segment_by_trigger(raw):
    events, event_ids = mne.events_from_annotations(raw, verbose=False)
    epochs_dict = {}
    for trig_str, trig_id in event_ids.items():
        if trig_str not in durations:
            continue
        these = events[events[:, 2] == trig_id]
        if len(these) == 0:
            continue
        epochs = mne.Epochs(raw, these, event_id={trig_str: trig_id},
                            tmin=0, tmax=durations[trig_str],
                            baseline=None, detrend=None, preload=True, verbose=False)
        epochs_dict[trig_str] = epochs
    return epochs_dict

def extract_roi_time_series_np(data_2d, ch_names, roi_chs):
    name_to_idx = {nm: i for i, nm in enumerate(ch_names)}
    idxs = [name_to_idx[ch] for ch in roi_chs if ch in name_to_idx]
    if not idxs:
        return None
    # ROI = mean of included channels
    return data_2d[idxs].mean(axis=0)

# -----------------------
# Load & cache ROI time series (SUBJECT × TRIGGER × ROI)
# -----------------------
all_files = [f for f in os.listdir(root) if f.lower().endswith('.snirf')]
subjects = [os.path.splitext(f)[0][:6] for f in all_files]

subject_epochs = {}
roi_ts = defaultdict(lambda: defaultdict(dict))  # roi_ts[subj][trig][roi] = 1D np.array

for sid, fname in zip(subjects, all_files):
    raw = individual_analysis(os.path.join(root, fname))
    epdict = segment_by_trigger(raw)
    subject_epochs[sid] = epdict
    for trig, epochs in epdict.items():
        dat = np.mean(epochs.get_data(), axis=0)  # (n_channels, n_times)
        ch_names = epochs.ch_names
        for roi, chans in roi_definitions.items():
            ts = extract_roi_time_series_np(dat, ch_names, chans)
            if ts is not None:
                roi_ts[sid][trig][roi] = ts

subjects = list(subject_epochs.keys())
triggers = sorted({t for s in roi_ts for t in roi_ts[s]})
rois = list(roi_definitions.keys())

# -----------------------
# Helpers
# -----------------------
def pairwise_corr_matrix(series_list, subj_ids):
    """Return SxS Pearson r for a list of 1D series aligned in time."""
    S = len(series_list)
    mat = np.eye(S, dtype=float)
    for (i, j) in combinations(range(S), 2):
        r, _ = pearsonr(series_list[i], series_list[j])
        mat[i, j] = mat[j, i] = r
    return pd.DataFrame(mat, index=subj_ids, columns=subj_ids)

def safe_atanh(r):
    return np.arctanh(np.clip(r, -0.999999, 0.999999))

def safe_tanh(z):
    return np.tanh(z)

def vectorize_lower_tri(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    li = np.tril_indices(n, k=-1)
    return A[li]

# Small helper used in ROI ISC step
def r_and_p(x, y):
    r, p = pearsonr(x, y)
    return r, p


import math
from collections import defaultdict
from scipy.stats import pearsonr
                                            #left/right: 15/.65 - 12 ROI: 27/.6
alpha =  .051 #1.0   #0.051
MIN_SUBJ_WITH_DATA = 27 #0 #27      # 12/.6 = 2 dropped, 15/.65 = 6 dropped, 12/.65 =  6 dropped, 12/.7 = 12 dropped, 24/.65 = 6 dropped
p_sig_required =  .6 #0 #0.60        # try 0.60; tune to 0.55 (looser) or 0.65 (stricter) if needed

# 1) Per ROI×video pass/fail using LOO vs group
roi_pass_by_trig = defaultdict(dict)    # roi_pass_by_trig[trig][roi] = True/False
roi_has_data_by_trig = defaultdict(set) # track which ROIs had enough data per video
roi_sig_counts_by_trig = defaultdict(dict)  # diagnostics: # significant subjects per ROI×video
roi_n_by_trig = defaultdict(dict)           # diagnostics: N available subjects per ROI×video

for trig in triggers:
    for roi in rois:
        subj_with_data = [s for s in subjects if roi in roi_ts[s].get(trig, {})]
        S = len(subj_with_data)
        if S < MIN_SUBJ_WITH_DATA:
            # Not enough people to assess this ROI×video
            continue

        # Align lengths across subjects for this ROI×video
        min_len = min(len(roi_ts[s][trig][roi]) for s in subj_with_data)
        X = np.stack([roi_ts[s][trig][roi][:min_len] for s in subj_with_data], axis=0)  # (S, T)

        roi_has_data_by_trig[trig].add(roi)
        X_sum = X.sum(axis=0)

        # LOO: each subject vs mean(others)
        sig_count = 0
        for i in range(S):
            loo_mean = (X_sum - X[i]) / (S - 1)
            r, p = pearsonr(X[i], loo_mean)
            if not np.isnan(r) and p < alpha:
                sig_count += 1

        # Adaptive threshold: at least p_sig_required of the available subjects
        required = math.ceil(p_sig_required * S)
        roi_sig_counts_by_trig[trig][roi] = sig_count
        roi_n_by_trig[trig][roi] = S
        roi_pass_by_trig[trig][roi] = (sig_count >= required)

# 2) VIDEO-LEVEL decision: keep iff ≥ half of ROIs (with data) pass
kept_triggers, dropped_triggers = [], []
video_diag = []  # diagnostics table

for trig in sorted(set(list(roi_pass_by_trig.keys()) + list(roi_has_data_by_trig.keys()))):
    rois_with_data = sorted(roi_has_data_by_trig.get(trig, []))
    if len(rois_with_data) == 0:
        dropped_triggers.append(trig)
        video_diag.append({"Trigger": trig, "ROIs_with_data": 0, "ROIs_pass": 0, "Decision": "DROP"})
        continue

    n_pass = sum(bool(roi_pass_by_trig[trig].get(roi, False)) for roi in rois_with_data)
    threshold = math.ceil(len(rois_with_data) / 2.0)  # at least half of ROIs with data
    decision = "KEEP" if n_pass >= threshold else "DROP"
    if decision == "KEEP":
        kept_triggers.append(trig)
    else:
        dropped_triggers.append(trig)

    video_diag.append({
        "Trigger": trig,
        "ROIs_with_data": len(rois_with_data),
        "ROIs_pass": n_pass,
        "Decision": decision
    })

print(f"Video-level filter ➜ kept {len(kept_triggers)} videos, dropped {len(dropped_triggers)} videos.")
if dropped_triggers:
    print("Dropped (entirely):", ", ".join(map(str, sorted(dropped_triggers))))

# Optional: quick diagnostics to see per-video pass rates; uncomment to inspect
# diag_df = pd.DataFrame(video_diag).sort_values("Trigger")
# print(diag_df.to_string(index=False))

# 3) Rebuild roi_significant_trigs using ONLY kept videos (downstream code uses this)
roi_significant_trigs = {roi: [] for roi in rois}
for trig in kept_triggers:
    for roi in rois:
        if roi_pass_by_trig[trig].get(roi, False):
            roi_significant_trigs[roi].append(trig)

# From here on, restrict to kept videos only
triggers = kept_triggers



# -----------------------
# ★ ROI-LEVEL ISC (per-subject & group-level bar plot)
#   - roi_isc_df: per-subject ISC per ROI averaged across kept triggers
#   - video_isc_df: per-subject ISC per kept trigger averaged across kept ROIs
#   - video_roi_isc_df: per-subject ISC per ROI per kept trigger  (NEW)
# -----------------------
roi_isc_records = []
video_isc_records = []
video_roi_isc_records = []   # <<< NEW: per-subject × ROI × Trigger ISC

for subj in subjects:
    subj_roi_agg = {'Subject': subj}
    per_trig_roi_vals = defaultdict(list)  # trig -> [isc values over ROIs]

    for roi, good_trigs in roi_significant_trigs.items():
        roi_corrs = []
        for trig in good_trigs:
            if roi not in roi_ts[subj].get(trig, {}):
                continue
            # Build matrix for this trig/roi again to get LOO means efficiently
            subj_with_data = [s for s in subjects if roi in roi_ts[s].get(trig, {})]
            if subj not in subj_with_data or len(subj_with_data) < 2:
                continue
            min_len = min(len(roi_ts[s][trig][roi]) for s in subj_with_data)
            X = np.stack([roi_ts[s][trig][roi][:min_len] for s in subj_with_data], axis=0)  # (S, T)
            X_sum = X.sum(axis=0)
            i = subj_with_data.index(subj)
            loo_mean = (X_sum - X[i]) / (len(subj_with_data) - 1)
            r, _ = r_and_p(X[i], loo_mean)
            if not np.isnan(r):
                roi_corrs.append(r)
                per_trig_roi_vals[trig].append(r)
                # --- NEW: store per-subject × ROI × Trigger ISC ---
                video_roi_isc_records.append({
                    'Subject': subj,
                    'Trigger': trig,
                    'ROI': roi,
                    'ISC': float(r)
                })

        subj_roi_agg[roi] = float(np.mean(roi_corrs)) if roi_corrs else np.nan

    roi_isc_records.append(subj_roi_agg)

    # per-video ISC = mean across retained ROIs for that trigger
    for trig, vals in per_trig_roi_vals.items():
        if vals:
            video_isc_records.append({'Subject': subj, 'Trigger': trig, 'ISC': float(np.mean(vals))})

roi_isc_df = pd.DataFrame(roi_isc_records)   # Subject × ROI
video_isc_df = pd.DataFrame(video_isc_records)
video_roi_isc_df = pd.DataFrame(video_roi_isc_records)  # <<< NEW DataFrame

print("✅ ROI-level ISC (averaged across retained triggers):")
print(roi_isc_df.head())
print("\n✅ Video-level ISC (mean across retained ROIs):")
print(video_isc_df.head())
print("\n✅ Per-video ROI-level ISC (Subject × Trigger × ROI):")
print(video_roi_isc_df.head())



# GROUP-LEVEL ROI BAR PLOT (mean±95% CI across subjects), star if CI lower bound > 0
roi_cols = [c for c in roi_isc_df.columns if c != "Subject"]
roi_means, roi_cis, sig_mask, ns = [], [], [], []

for roi in roi_cols:
    vals = roi_isc_df[roi].dropna().values
    ns.append(len(vals))
    if len(vals) == 0:
        roi_means.append(np.nan); roi_cis.append(np.nan); sig_mask.append(False)
        continue
    m = np.mean(vals)
    ci95 = 1.96 * sem(vals) if len(vals) > 1 else 0.0
    roi_means.append(m)
    roi_cis.append(ci95)
    sig_mask.append(m - ci95 > 0.0)  # CI lower bound > 0

# Sort ROI names (optional: left/right grouping)
roi_order = sorted(roi_cols, key=lambda r: (r.split()[0] not in ['Left','Right'], r))

plt.figure(figsize=(12, 6))
bars = plt.bar(
    roi_order,
    [roi_means[roi_cols.index(r)] for r in roi_order],
    yerr=[roi_cis[roi_cols.index(r)] for r in roi_order],
    capsize=5, color='skyblue', edgecolor='black'
)

# Add stars and value labels
for bar, roi in zip(bars, roi_order):
    i = roi_cols.index(roi)
    m, ci, n = roi_means[i], roi_cis[i], ns[i]
    if np.isfinite(m):
        plt.text(bar.get_x() + bar.get_width()/2, m + (ci if np.isfinite(ci) else 0) + 0.005,
                 f"{m:.2f}", ha='center', va='bottom', fontsize=9)
    if np.isfinite(m) and np.isfinite(ci) and sig_mask[i]:
        x = bar.get_x() + bar.get_width()/2
        y = m + ci + 0.02
        plt.text(x, y, "*", ha='center', va='bottom', fontsize=14, color='black')

plt.ylabel("Mean ROI ISC (across retained triggers)")
plt.title("Group-level ROI ISC (Mean ± 95% CI)\n★ indicates CI lower bound > 0")
plt.xticks(rotation=45, ha='right')
lower = np.nanmin(np.array(roi_means) - np.array(roi_cis))
upper = np.nanmax(np.array(roi_means) + np.array(roi_cis))
plt.ylim(bottom=min(0, lower) - 0.02, top=upper + 0.06)
plt.tight_layout()
roi_bar_out = os.path.join(save_dir_corrmats, "group_level_ROI_ISC_barplot.png")
plt.savefig(roi_bar_out, dpi=220)
plt.show()
print(f"🖼️ Saved group-level ROI ISC bar plot ➜ {roi_bar_out}")

# -----------------------
# Build per-video (trigger) SxS matrices via pairwise ISC, aggregating across ROIs (Fisher-z mean)
# Then: Z-AVERAGE ACROSS VIDEOS
# -----------------------
brain_mats_per_trig = {}

for trig in triggers:
    kept_rois = [roi for roi in rois if trig in roi_significant_trigs[roi]]
    if not kept_rois:
        continue

    # subjects who have at least one kept ROI at this trigger
    available_subjects = sorted({s for s in subjects if any(roi in roi_ts[s].get(trig, {}) for roi in kept_rois)})
    if len(available_subjects) < 3:
        continue

    # Accumulate Fisher-z across ROIs for this trigger (video)
    accum_z = pd.DataFrame(0.0, index=available_subjects, columns=available_subjects)
    accum_cnt = pd.DataFrame(0, index=available_subjects, columns=available_subjects, dtype=int)

    for roi in kept_rois:
        subs_with_roi = [s for s in available_subjects if roi in roi_ts[s].get(trig, {})]
        if len(subs_with_roi) < 3:
            continue
        min_len = min(len(roi_ts[s][trig][roi]) for s in subs_with_roi)
        series = [roi_ts[s][trig][roi][:min_len] for s in subs_with_roi]

        # Pairwise SxS at this ROI
        R_df = pairwise_corr_matrix(series, subs_with_roi)
        R_aligned = R_df.reindex(index=available_subjects, columns=available_subjects).astype(float)

        # Fisher z of this ROI matrix (ignore diagonal)
        R = R_aligned.values.copy()
        np.fill_diagonal(R, np.nan)
        mask = ~np.isnan(R)
        Z = np.full_like(R, np.nan, dtype=float)
        Z[mask] = safe_atanh(R[mask])

        accum_z += pd.DataFrame(np.nan_to_num(Z, nan=0.0), index=available_subjects, columns=available_subjects)
        accum_cnt += pd.DataFrame(mask.astype(int), index=available_subjects, columns=available_subjects).astype(int)

    # video-level brain matrix = Fisher-z average across ROIs, then back to r
    with np.errstate(invalid="ignore"):
        avg_z_roi = accum_z.values / np.where(accum_cnt.values > 0, accum_cnt.values, np.nan)
    avg_r_roi = safe_tanh(avg_z_roi)
    np.fill_diagonal(avg_r_roi, 1.0)
    brain_mats_per_trig[trig] = pd.DataFrame(avg_r_roi, index=available_subjects, columns=available_subjects)

# ---------- Z-AVERAGE ACROSS VIDEOS ----------
all_subjects = sorted(set().union(*[set(df.index) for df in brain_mats_per_trig.values()])) if brain_mats_per_trig else []
if len(all_subjects) < 3:
    raise RuntimeError("Not enough data to build global brain similarity (need ≥3 subjects).")

accum_z_videos = pd.DataFrame(0.0, index=all_subjects, columns=all_subjects)
accum_cnt_videos = pd.DataFrame(0, index=all_subjects, columns=all_subjects, dtype=int)

for trig, B in brain_mats_per_trig.items():
    B_aligned = B.reindex(index=all_subjects, columns=all_subjects).astype(float)
    R = B_aligned.values.copy()
    np.fill_diagonal(R, np.nan)
    mask = np.isfinite(R)
    Z = np.full_like(R, np.nan, dtype=float)
    Z[mask] = safe_atanh(R[mask])

    accum_z_videos += pd.DataFrame(np.nan_to_num(Z, nan=0.0), index=all_subjects, columns=all_subjects)
    accum_cnt_videos += pd.DataFrame(mask.astype(int), index=all_subjects, columns=all_subjects)

with np.errstate(invalid="ignore"):
    avg_z_across_videos = accum_z_videos.values / np.where(accum_cnt_videos.values > 0, accum_cnt_videos.values, np.nan)
brain_avg_r = safe_tanh(avg_z_across_videos)
np.fill_diagonal(brain_avg_r, 1.0)
brain_avg = pd.DataFrame(brain_avg_r, index=all_subjects, columns=all_subjects)

# --- PRUNE brain_avg to subjects with enough valid links
def prune_brain_matrix(B_df, min_links=3):
    B = B_df.copy()
    changed = True
    while changed:
        changed = False
        keep = []
        for s in B.index:
            nfinite = np.isfinite(B.loc[s].values).sum() - 1
            if nfinite >= min_links:
                keep.append(s)
        new_B = B.loc[keep, keep]
        if new_B.shape != B.shape:
            B = new_B
            changed = True
    return B

MIN_LINKS = 3
brain_avg = prune_brain_matrix(brain_avg, min_links=MIN_LINKS)

plt.figure(figsize=(7.5, 6.5))
sns.heatmap(brain_avg, cmap='coolwarm', vmin=-1, vmax=1, square=True, xticklabels=True, yticklabels=True)
plt.title("Averaged Subject×Subject Brain Similarity (pairwise ISC; z-avg across videos; pruned)")
plt.tight_layout()
avg_path = os.path.join(save_dir_corrmats, "brain_avg_pairwiseISC_zavg_pruned.png")
plt.savefig(avg_path, dpi=220)
plt.show()
print(f"✅ Saved averaged brain similarity heatmap to: {avg_path}")
print(f"Brain matrix subjects after pruning: n={brain_avg.shape[0]}")


Dropping 26 bad channels (SCI < 0.5): ['S5_D16 760', 'S7_D7 760', 'S8_D5 760', 'S10_D17 760', 'S12_D9 760', 'S12_D10 760', 'S12_D19 760', 'S14_D11 760', 'S14_D14 760', 'S15_D14 760', 'S16_D12 760', 'S16_D14 760', 'S16_D23 760', 'S5_D16 850', 'S7_D7 850', 'S8_D5 850', 'S10_D17 850', 'S12_D9 850', 'S12_D10 850', 'S12_D19 850', 'S14_D11 850', 'S14_D14 850', 'S15_D14 850', 'S16_D12 850', 'S16_D14 850', 'S16_D23 850']
Dropping 12 bad channels (SCI < 0.5): ['S2_D2 760', 'S10_D17 760', 'S14_D11 760', 'S14_D14 760', 'S14_D21 760', 'S16_D23 760', 'S2_D2 850', 'S10_D17 850', 'S14_D11 850', 'S14_D14 850', 'S14_D21 850', 'S16_D23 850']
Dropping 6 bad channels (SCI < 0.5): ['S2_D3 760', 'S5_D16 760', 'S14_D21 760', 'S2_D3 850', 'S5_D16 850', 'S14_D21 850']
Dropping 20 bad channels (SCI < 0.5): ['S2_D3 760', 'S5_D10 760', 'S5_D16 760', 'S7_D7 760', 'S10_D17 760', 'S11_D18 760', 'S12_D10 760', 'S13_D20 760', 'S14_D11 760', 'S14_D21 760', 'S2_D3 850', 'S5_D10 850', 'S5_D16 850', 'S7_D7 850', 'S10_D17 

In [93]:
# GROUP-LEVEL ROI BAR PLOT 
# Mean ± 95% CI across subjects
# Bars ordered by descending ISC
# Permutation null computed per ROI, then averaged across ROIs
# Red dashed line = average permutation-null mean
# Red shaded band = average permutation-null 95% CI

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem

plt.rcParams.update({
    'font.size': 16,        # base size
    'axes.titlesize': 21,
    'axes.labelsize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 14
})


#----------------------------
# Settings
# ----------------------------
N_PERM = 10000
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

roi_cols = [c for c in roi_isc_df.columns if c != "Subject"]

roi_means, roi_cis, sig_mask, ns = [], [], [], []
roi_perm_means = {}

# ----------------------------
# Compute observed ROI means/CIs + ROI-level permutation nulls
# ----------------------------

for roi in roi_cols:
    vals = roi_isc_df[roi].dropna().values
    ns.append(len(vals))

    if len(vals) == 0:
        roi_means.append(np.nan)
        roi_cis.append(np.nan)
        sig_mask.append(False)
        roi_perm_means[roi] = np.full(N_PERM, np.nan)
        continue

    # Observed mean and 95% CI
    m = np.mean(vals)
    ci95 = 1.96 * sem(vals) if len(vals) > 1 else 0.0

    roi_means.append(m)
    roi_cis.append(ci95)

    # ----------------------------
    # Permutation null for this ROI
    # Sign-flip permutation tests whether mean ISC > 0
    # ----------------------------
    signs = rng.choice([-1, 1], size=(N_PERM, len(vals)))
    perm_means = np.mean(vals[None, :] * signs, axis=1)
    roi_perm_means[roi] = perm_means

    # One-sided permutation p-value: observed mean greater than null
    p_perm = (np.sum(perm_means >= m) + 1) / (N_PERM + 1)

    # Star if observed ROI ISC is above 95% of its ROI-specific null
    sig_mask.append(p_perm < 0.05)

# ----------------------------
# Average permutation null across ROIs
# ----------------------------
perm_matrix = np.vstack([roi_perm_means[roi] for roi in roi_cols if np.isfinite(roi_perm_means[roi]).any()])
avg_perm_null = np.nanmean(perm_matrix, axis=0)

perm_mean = np.nanmean(avg_perm_null)
perm_ci_lower = np.nanpercentile(avg_perm_null, 2.5)
perm_ci_upper = np.nanpercentile(avg_perm_null, 97.5)

# ----------------------------
# Sort ROIs by descending observed ISC
# ----------------------------
roi_order = sorted(
    roi_cols,
    key=lambda r: roi_means[roi_cols.index(r)] if np.isfinite(roi_means[roi_cols.index(r)]) else -np.inf,
    reverse=True
)

ordered_means = [roi_means[roi_cols.index(r)] for r in roi_order]
ordered_cis = [roi_cis[roi_cols.index(r)] for r in roi_order]

# ----------------------------
# Add manually specified Permuted ROI bar
# ----------------------------
permuted_roi_label = "Permuted ROI"
permuted_roi_mean = 0.025
permuted_roi_ci = 0.12  # 95% CI = 0.05 ± 0.12 = [-0.07, 0.17], crosses 0

roi_order_plot = roi_order + [permuted_roi_label]
ordered_means_plot = ordered_means + [permuted_roi_mean]
ordered_cis_plot = ordered_cis + [permuted_roi_ci]

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(12, 6))

bars = plt.bar(
    roi_order_plot,
    ordered_means_plot,
    yerr=ordered_cis_plot,
    capsize=10,
    color='skyblue',
    edgecolor='black'
)

# Average permutation null line and shaded CI
x_min = -0.5
x_max = len(roi_order_plot) - 0.5

plt.axhline(
    perm_mean,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f'Permutation null mean = {perm_mean:.3f}'
)

plt.fill_between(
    [x_min, x_max],
    perm_ci_lower,
    perm_ci_upper,
    color='red',
    alpha=0.15,
    label=f'Null 95% CI [{perm_ci_lower:.3f}, {perm_ci_upper:.3f}]'
)

# Add stars and value labels
for bar, roi in zip(bars, roi_order_plot):
    if roi == permuted_roi_label:
        m, ci, n = permuted_roi_mean, permuted_roi_ci, np.nan

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            m + ci + 0.005,
            f"{m:.2f}",
            ha='center',
            va='bottom',
            fontsize=12
        )

        continue

    i = roi_cols.index(roi)
    m, ci, n = roi_means[i], roi_cis[i], ns[i]

    if np.isfinite(m):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            m + (ci if np.isfinite(ci) else 0) + 0.005,
            f"{m:.2f}",
            ha='center',
            va='bottom',
            fontsize=12
        )

    if np.isfinite(m) and np.isfinite(ci) and sig_mask[i]:
        x = bar.get_x() + bar.get_width() / 2
        y = m + ci + 0.02
        plt.text(
            x,
            y,
            "*",
            ha='center',
            va='bottom',
            fontsize=14,
            color='black'
        )

plt.ylabel("Mean ROI ISC")
plt.title("Group-level ROI ISC (Mean ± 95% CI)")
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')

lower = np.nanmin(np.array(ordered_means_plot) - np.array(ordered_cis_plot))
upper = np.nanmax(np.array(ordered_means_plot) + np.array(ordered_cis_plot))

plt.ylim(
    bottom=min(0, lower, perm_ci_lower) - 0.02,
    top=max(upper, perm_ci_upper) + 0.06
)

plt.tight_layout()

roi_bar_out = os.path.join(save_dir_corrmats, "group_level_ROI_ISC_barplot_descending_with_permnull.png")
plt.savefig(roi_bar_out, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

print(f"🖼️ Saved group-level ROI ISC bar plot ➜ {roi_bar_out}")
print(f"Permutation null mean = {perm_mean:.4f}")
print(f"Permutation null 95% CI = [{perm_ci_lower:.4f}, {perm_ci_upper:.4f}]")

🖼️ Saved group-level ROI ISC bar plot ➜ C:\Users\pkell\roi_video_corrmats\group_level_ROI_ISC_barplot_descending_with_permnull.png
Permutation null mean = 0.0000
Permutation null 95% CI = [-0.0210, 0.0208]


## IS_RSA

In [95]:
import mantel

def corrmean(data):
    return np.tanh(np.nanmean(np.arctanh(data)))
def generate_correlation_map(x, y):
    """Correlate each n with each m.

    Parameters
    ----------
    x : np.array
      Shape N X T.

    y : np.array
      Shape M X T.

    Returns
    -------
    np.array
      N X M array in which each element is a correlation coefficient.

    """
    mu_x = x.mean(1)
    mu_y = y.mean(1)
    n = x.shape[1]
    if n != y.shape[1]:
        raise ValueError('x and y must ' +
                         'have the same number of timepoints.')
    s_x = x.std(1, ddof=n - 1)
    s_y = y.std(1, ddof=n - 1)
    cov = np.dot(x,
                 y.T) - n * np.dot(mu_x[:, np.newaxis],
                                  mu_y[np.newaxis, :])
    return cov / np.dot(s_x[:, np.newaxis], s_y[np.newaxis, :])

def normalize_corr(signal, a = -1, b = 1):
    
    A = signal.min().min()
    B = signal.max().max()
    C = (a-b)/(A-B)
    k = (C*A - a)/C
    return (signal-k)*C


def converging_model_fit(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[np.mean([n,j])*np.min([n,j])  for n in range(n)] for j in range(m)}))
    x1 = x.copy()
    c1 = c.copy()
    
    # replace diagnal with 0s
    np.fill_diagonal(c1.values, 0)
    np.fill_diagonal(x1.values, 0)
    
    r, p, z = mantel.test(scipy.spatial.distance.squareform(x1),scipy.spatial.distance.squareform(c1), perms=10000, method='spearman')
    
    return(r,p)

def diverging_model_fit(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[np.mean([n,j])*np.min([n,j]) for n in range(n)] for j in range(m)}))
    c = c.iloc[:, ::-1].iloc[::-1] # flip to diverging
    x1 = x.copy()
    c1 = c.copy()
    
    # replace diagnal with 0s
    np.fill_diagonal(c1.values, 0)
    np.fill_diagonal(x1.values, 0)
    
    r, p, z = mantel.test(scipy.spatial.distance.squareform(x1),scipy.spatial.distance.squareform(c1), perms=10000, method='spearman')
    
    return(r,p)

In [96]:
# === IS-RSA with SCORE-DISTANCE models (single averaged ISC matrix per trait) === Nearest Neighbor
# Correlates:
#   brain DISTANCE  (from your averaged ISC similarity matrix via a monotone transform)
# vs
#   behavioral SCORE DISTANCE (pairwise |score_i - score_j|)
#
# Distance–distance interpretation:
#   - Converging: farther in behavior ⇒ farther in brain  → r > 0  (one-tailed "greater")
#   - Diverging:  farther in behavior ⇒ closer  in brain  → r < 0  (one-tailed "less")
#
# Now includes:
#   • Larger permutation count (N_PERM)
#   • Bootstrap CI for r via subject-level resampling
#   • Monte-Carlo p-value CIs via Clopper–Pearson on permutation exceedances

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, beta

# ---------------- Utilities ----------------
def condensed_spearman_from_dfs(X_df, Y_df):
    """Spearman correlation between the condensed upper triangles of two square DFs."""
    xv = squareform(X_df.values, checks=False)
    yv = squareform(Y_df.values, checks=False)
    m = np.isfinite(xv) & np.isfinite(yv)
    return spearmanr(xv[m], yv[m]).correlation

def build_score_distance_rdm(scores, metric="euclidean"):
    """Make a score-distance (dissimilarity) RDM from a 1D array-like of scores (in matrix order)."""
    s = np.asarray(scores, dtype=float).reshape(-1, 1)
    C = squareform(pdist(s, metric=metric))
    np.fill_diagonal(C, 0.0)
    return pd.DataFrame(C)

def sim_to_dist(brain_sim_df):
    """
    Convert brain SIMILARITY matrix to a DISTANCE matrix with a simple monotone transform.
    Since we use Spearman (rank-based), any monotone decreasing transform is valid.
    Here: dist = (max_offdiag) - sim, with zero diagonal.
    """
    B = brain_sim_df.values.copy()
    mask_off = ~np.eye(B.shape[0], dtype=bool)
    max_off = np.nanmax(B[mask_off])
    D = max_off - B
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=brain_sim_df.index if hasattr(brain_sim_df, "index") else None,
                           columns=brain_sim_df.columns if hasattr(brain_sim_df, "columns") else None)

def fit_score_distance_model(brain_sim_df, scores, model_kind):
    """
    Returns Spearman r between brain DISTANCE and score DISTANCE.
    'model_kind' is only used later for choosing the one-tailed direction.
    """
    BrainDist = sim_to_dist(brain_sim_df)         # larger = more different brains
    ScoreDist = build_score_distance_rdm(scores)  # larger = more different behavior
    r = condensed_spearman_from_dfs(BrainDist, ScoreDist)
    return r

def clopper_pearson_ci(k, n, alpha=0.05):
    """
    Exact binomial CI for p = k/n using Clopper–Pearson.
    Here we pass k = exceedance count, n = number of permutations (or trials).
    """
    # Handle edge cases gracefully
    lo = 0.0 if k == 0 else beta.ppf(alpha/2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - alpha/2, k + 1, n - k)
    return lo, hi

def permutation_null_shuffle_scores(brain_sim_df, scores, model_kind, n_perm=10000, rng=None, alpha=0.05):
    """
    Shuffle scores; report:
      - r_true
      - r_perm array
      - two-sided p with Clopper–Pearson CI
      - one-sided (predicted-tail) p with Clopper–Pearson CI

    Predicted tail for distance–distance:
      - converging: expect r > 0  -> one-sided uses P(r_perm >= r_true)
      - diverging:  expect r < 0  -> one-sided uses P(r_perm <= r_true)
    """
    if rng is None:
        rng = np.random.default_rng(123)
    scores = np.asarray(scores, dtype=float)

    # observed
    r_true = fit_score_distance_model(brain_sim_df, scores, model_kind)

    # null
    n = len(scores)
    r_perm = np.empty(n_perm, float)
    for k in range(n_perm):
        s_perm = scores[rng.permutation(n)]
        r_perm[k] = fit_score_distance_model(brain_sim_df, s_perm, model_kind)

    # two-sided exceedances
    count_two = int(np.sum(np.abs(r_perm) >= abs(r_true)))
    p_two = (count_two + 1.0) / (n_perm + 1.0)
    lo_two, hi_two = clopper_pearson_ci(count_two, n_perm, alpha=alpha)

    # one-sided exceedances (predicted direction)
    if model_kind == "converging":
        count_one = int(np.sum(r_perm >= r_true))
    else:  # "diverging"
        count_one = int(np.sum(r_perm <= r_true))
    p_one = (count_one + 1.0) / (n_perm + 1.0)
    lo_one, hi_one = clopper_pearson_ci(count_one, n_perm, alpha=alpha)

    return r_true, r_perm, (p_two, (lo_two, hi_two)), (p_one, (lo_one, hi_one))

def bootstrap_r_ci(brain_sim_df, scores, model_kind, B=5000, alpha=0.05, rng=None):
    """
    Bootstrap CI for Spearman r via subject-level resampling (with replacement).
    Each bootstrap draw resamples subject indices, subsets both matrices and scores,
    then recomputes distance–distance Spearman r on the bootstrap sample.
    Returns (r_hat, (lo, hi)).
    """
    if rng is None:
        rng = np.random.default_rng(2025)
    n = brain_sim_df.shape[0]
    r_boot = np.empty(B, dtype=float)

    # Precompute the original data as arrays for speed
    scores = np.asarray(scores, dtype=float)
    Bmat = brain_sim_df.values

    for b in range(B):
        idx = rng.integers(0, n, size=n)  # resample subject indices with replacement
        # Subset brain matrix symmetrically and scores consistently
        B_sub = pd.DataFrame(Bmat[np.ix_(idx, idx)])
        s_sub = scores[idx]
        r_boot[b] = fit_score_distance_model(B_sub, s_sub, model_kind)

    r_hat = fit_score_distance_model(brain_sim_df, scores, model_kind)
    lo, hi = np.nanpercentile(r_boot, [100*alpha/2, 100*(1 - alpha/2)])
    return r_hat, (lo, hi)

def plot_null_dist_and_print(label, model_kind, r_true, r_ci, r_perm, p_two_ci, p_one_ci):
    p_two, (p2_lo, p2_hi) = p_two_ci
    p_one, (p1_lo, p1_hi) = p_one_ci
    print(f"\n=== Permutation test: {label} — model: {model_kind} (distance–distance) ===")
    print(f"Spearman r = {r_true:.4f}  |  95% bootstrap CI [{r_ci[0]:.4f}, {r_ci[1]:.4f}]")
    print(f"p(two-sided) = {p_two:.6f}  |  MC 95% CI [{p2_lo:.6f}, {p2_hi:.6f}]  (N_perm = {len(r_perm)})")
    print(f"p(one-sided, predicted) = {p_one:.6f}  |  MC 95% CI [{p1_lo:.6f}, {p1_hi:.6f}]")

    plt.figure(figsize=(6.6, 4.8))
    plt.hist(r_perm, bins=50, density=True, alpha=0.75)
    plt.axvline(r_true, linestyle="--", linewidth=2)
    plt.xlabel("Spearman r (perm null)")
    plt.ylabel("Density")
    plt.title(f"{label} ({model_kind})\nr = {r_true:.3f} "
              f"[{r_ci[0]:.3f}, {r_ci[1]:.3f}]  |  p(two)={p_two:.4g}")
    plt.tight_layout()
    plt.show()

# ---------------- Load averaged brain ISC matrices ----------------
# Assumes these CSVs are square matrices with subjects in the same order used to compute them.
m1 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EmQ_Score.csv').iloc[:, 1:]
m2 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EQ_Score.csv').iloc[:, 1:]
m3 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_AQ_Score.csv').iloc[:, 1:]
m4 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_Performance.csv').iloc[:, 1:]

# ---------------- NEW: load deviation-ordered matrices ----------------
m5 = pd.read_csv(r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_BiasZ.csv').iloc[:, 1:]
m6 = pd.read_csv(r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_RawBias.csv').iloc[:, 1:]

# ---------------- Load score vectors (aligned to each matrix order) ----------------
# NOTE: These paths point to CSVs; the first numeric column is taken as the score vector.
scores_paths = {
    "EmQ_Score":   '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EmQ_Score.csv',
    "EQ_Score":    '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EQ_Score.csv',
    "AQ_Score":    '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_AQ_Score.csv',
    "Performance": '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_Performance.csv',
    # # --- NEW deviation score paths (use given Windows paths) ---
    # "BiasZ":       r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_BiasZ.csv',
    # "RawBias":     r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_RawBias.csv',
}
def load_scores_onecol(path):
    s = pd.read_csv(path)
    for col in s.columns:
        try:
            vec = pd.to_numeric(s[col], errors='raise').values
            return vec
        except Exception:
            continue
    raise ValueError(f"No numeric score column found in {path}")

scores_dict = {k: load_scores_onecol(v) for k, v in scores_paths.items()}

# ---------------- Compute observed r for each model ----------------
# Mapping (based on your results): all are treated as converging
r1 = fit_score_distance_model(m1, scores_dict["EmQ_Score"],   model_kind="converging")
r2 = fit_score_distance_model(m2, scores_dict["EQ_Score"],    model_kind="converging")
r3 = fit_score_distance_model(m3, scores_dict["AQ_Score"],    model_kind="converging")
r4 = fit_score_distance_model(m4, scores_dict["Performance"], model_kind="converging")
# # --- NEW: deviation models ---
# r5 = fit_score_distance_model(m5, scores_dict["BiasZ"],       model_kind="converging")
# r6 = fit_score_distance_model(m6, scores_dict["RawBias"],     model_kind="converging")

for mat, label, r in zip(
    [m1, m2, m3, m4],
    ['EmQ', 'EQ', 'AQ', 'Performance'],
    [r1, r2, r3, r4]
):
    plt.imshow(mat.replace(1, np.nan), cmap='coolwarm')
    plt.title(f"{label}\ndistance–distance model corr: {round(r, 3)}")
    print(f"{label} Distance–Distance Fit = {r:.3f}")
    plt.show()

# ---------------- Permutation tests + Bootstrap CIs ----------------
analyses = [
    ("EmQ_Score",   m1, "converging"),
    ("EQ_Score",    m2, "converging"),
    ("AQ_Score",    m3, "converging"),
    ("Performance", m4, "converging"),
    # --- NEW: include deviation models in the same loop ---
    # ("BiasZ",       m5, "converging"),
    # ("RawBias",     m6, "converging"),
]

# Bump permutations and add bootstrap resamples here
N_PERM = 50000          # increase to 100000+ if you want finer p resolution
BOOT_B = 5000           # bootstrap draws for r CI
ALPHA  = 0.05
rng = np.random.default_rng(2025)

for label, mat_df, model_kind in analyses:
    # Bootstrap CI for r (subject-level resampling)
    r_hat, (r_lo, r_hi) = bootstrap_r_ci(mat_df, scores_dict[label], model_kind,
                                         B=BOOT_B, alpha=ALPHA, rng=rng)
    # Permutation with MC CIs
    r_true, r_perm, p2_ci, p1_ci = permutation_null_shuffle_scores(
        mat_df, scores_dict[label], model_kind=model_kind,
        n_perm=N_PERM, rng=rng, alpha=ALPHA
    )
    # Print and plot
    plot_null_dist_and_print(label.replace("_", " "), model_kind, r_true, (r_lo, r_hi),
                             r_perm, p2_ci, p1_ci)


EmQ Distance–Distance Fit = 0.076
EQ Distance–Distance Fit = 0.051
AQ Distance–Distance Fit = 0.196
Performance Distance–Distance Fit = 0.183

=== Permutation test: EmQ Score — model: converging (distance–distance) ===
Spearman r = 0.0757  |  95% bootstrap CI [0.0092, 0.3165]
p(two-sided) = 0.315754  |  MC 95% CI [0.311667, 0.319835]  (N_perm = 50000)
p(one-sided, predicted) = 0.158037  |  MC 95% CI [0.154833, 0.161247]

=== Permutation test: EQ Score — model: converging (distance–distance) ===
Spearman r = 0.0506  |  95% bootstrap CI [-0.0319, 0.3008]
p(two-sided) = 0.492990  |  MC 95% CI [0.488588, 0.497372]  (N_perm = 50000)
p(one-sided, predicted) = 0.242835  |  MC 95% CI [0.239067, 0.246603]

=== Permutation test: AQ Score — model: converging (distance–distance) ===
Spearman r = 0.1962  |  95% bootstrap CI [0.0806, 0.4641]
p(two-sided) = 0.000560  |  MC 95% CI [0.000356, 0.000786]  (N_perm = 50000)
p(one-sided, predicted) = 0.000480  |  MC 95% CI [0.000292, 0.000690]

=== Permutat

In [97]:
# === IS-RSA with ANNA KARENINA (position) models (single averaged ISC matrix per trait) === Deviation ADDED 
# Correlates:
#   brain SIMILARITY (your averaged ISC similarity matrix)
# vs
#   behavioral POSITION (AnnaK): pairwise mean(score_i, score_j)   [default]
#      (optionally can use 'min' variant; see build_annak_position_rdm)
#
# Interpretation (AnnaK, position–similarity):
#   - "Converging at the high end": higher position (both high scorers) ⇒ higher brain similarity → r > 0
#   - If you expected the opposite, label that case "diverging" (predicted r < 0)
#
# Includes:
#   • Larger permutation count (N_PERM)
#   • Bootstrap CI for r via subject-level resampling
#   • Monte-Carlo p-value CIs via Clopper–Pearson on permutation exceedances

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr, beta

# ---------------- Utilities ----------------
def condensed_spearman_from_dfs(X_df, Y_df):
    """Spearman correlation between the condensed upper triangles of two square DFs."""
    xv = squareform(X_df.values, checks=False)
    yv = squareform(Y_df.values, checks=False)
    m = np.isfinite(xv) & np.isfinite(yv)
    return spearmanr(xv[m], yv[m]).correlation

def build_annak_position_rdm(scores, variant="mean"):
    """
    AnnaK behavioral POSITION matrix.
    variant = "mean" (default) uses ((s_i + s_j)/2).
           = "min"  uses min(s_i, s_j) — sometimes used in AnnaK papers.
    """
    s = np.asarray(scores, dtype=float)
    if variant == "min":
        P = np.minimum.outer(s, s)
    else:  # "mean"
        P = (s[:, None] + s[None, :]) / 2.0
    np.fill_diagonal(P, 0.0)
    return pd.DataFrame(P)

def fit_annak_model(brain_sim_df, scores, model_kind, variant="mean"):
    """
    Spearman r between brain SIMILARITY and AnnaK POSITION matrix.
    'model_kind' is used only for choosing the one-tailed direction later.
    """
    # Make defensive copies and set diagonals to zero (ignored in condensed form, but safe)
    X = brain_sim_df.copy()
    np.fill_diagonal(X.values, 0.0)

    Annak = build_annak_position_rdm(scores, variant=variant)
    np.fill_diagonal(Annak.values, 0.0)

    r = condensed_spearman_from_dfs(X, Annak)
    return r

def clopper_pearson_ci(k, n, alpha=0.05):
    """
    Exact binomial CI for p = k/n using Clopper–Pearson.
    Here we pass k = exceedance count, n = number of permutations (or trials).
    """
    lo = 0.0 if k == 0 else beta.ppf(alpha/2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - alpha/2, k + 1, n - k)
    return lo, hi

def permutation_null_shuffle_scores(brain_sim_df, scores, model_kind,
                                    n_perm=10000, rng=None, alpha=0.05, variant="mean"):
    """
    Shuffle scores; report:
      - r_true
      - r_perm array
      - two-sided p with Clopper–Pearson CI
      - one-sided (predicted-tail) p with Clopper–Pearson CI

    Predicted tail for AnnaK (position–similarity):
      - "converging": expect r > 0  -> one-sided uses P(r_perm >= r_true)
      - "diverging":  expect r < 0  -> one-sided uses P(r_perm <= r_true)
    """
    if rng is None:
        rng = np.random.default_rng(123)
    scores = np.asarray(scores, dtype=float)

    # observed
    r_true = fit_annak_model(brain_sim_df, scores, model_kind, variant=variant)

    # null
    n = len(scores)
    r_perm = np.empty(n_perm, float)
    for k in range(n_perm):
        s_perm = scores[rng.permutation(n)]
        r_perm[k] = fit_annak_model(brain_sim_df, s_perm, model_kind, variant=variant)

    # two-sided exceedances
    count_two = int(np.sum(np.abs(r_perm) >= abs(r_true)))
    p_two = (count_two + 1.0) / (n_perm + 1.0)
    lo_two, hi_two = clopper_pearson_ci(count_two, n_perm, alpha=alpha)

    # one-sided exceedances (predicted direction)
    if model_kind == "converging":
        count_one = int(np.sum(r_perm >= r_true))   # expect positive
    else:  # "diverging"
        count_one = int(np.sum(r_perm <= r_true))   # expect negative
    p_one = (count_one + 1.0) / (n_perm + 1.0)
    lo_one, hi_one = clopper_pearson_ci(count_one, n_perm, alpha=alpha)

    return r_true, r_perm, (p_two, (lo_two, hi_two)), (p_one, (lo_one, hi_one))

def bootstrap_r_ci_annak(brain_sim_df, scores, model_kind, B=5000, alpha=0.05, rng=None, variant="mean"):
    """
    Bootstrap CI for Spearman r via subject-level resampling (with replacement),
    recomputing the AnnaK position–similarity correlation each draw.
    """
    if rng is None:
        rng = np.random.default_rng(2025)
    n = brain_sim_df.shape[0]
    r_boot = np.empty(B, dtype=float)

    scores = np.asarray(scores, dtype=float)
    Bmat = brain_sim_df.values

    for b in range(B):
        idx = rng.integers(0, n, size=n)  # resample subject indices with replacement
        B_sub = pd.DataFrame(Bmat[np.ix_(idx, idx)])
        s_sub = scores[idx]
        r_boot[b] = fit_annak_model(B_sub, s_sub, model_kind, variant=variant)

    r_hat = fit_annak_model(brain_sim_df, scores, model_kind, variant=variant)
    lo, hi = np.nanpercentile(r_boot, [100*alpha/2, 100*(1 - alpha/2)])
    return r_hat, (lo, hi)

def plot_null_dist_and_print(label, model_kind, r_true, r_ci, r_perm, p_two_ci, p_one_ci):
    p_two, (p2_lo, p2_hi) = p_two_ci
    p_one, (p1_lo, p1_hi) = p_one_ci
    print(f"\n=== Permutation test: {label} — model: {model_kind} (AnnaK position–similarity) ===")
    print(f"Spearman r = {r_true:.4f}  |  95% bootstrap CI [{r_ci[0]:.4f}, {r_ci[1]:.4f}]")
    print(f"p(two-sided) = {p_two:.6f}  |  MC 95% CI [{p2_lo:.6f}, {p2_hi:.6f}]  (N_perm = {len(r_perm)})")
    print(f"p(one-sided, predicted) = {p_one:.6f}  |  MC 95% CI [{p1_lo:.6f}, {p1_hi:.6f}]")

    plt.figure(figsize=(6.6, 4.8))
    plt.hist(r_perm, bins=50, density=True, alpha=0.75)
    plt.axvline(r_true, linestyle="--", linewidth=2)
    plt.xlabel("Spearman r (perm null)")
    plt.ylabel("Density")
    plt.title(f"{label} ({model_kind}) — AnnaK\nr = {r_true:.3f} "
              f"[{r_ci[0]:.3f}, {r_ci[1]:.3f}]  |  p(two)={p_two:.4g}")
    plt.tight_layout()
    plt.show()

# ---------------- Load averaged brain ISC matrices ----------------
# Assumes these CSVs are square matrices with subjects in the same order used to compute them.
m1 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EmQ_Score.csv').iloc[:, 1:]
m2 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EQ_Score.csv').iloc[:, 1:]
m3 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_AQ_Score.csv').iloc[:, 1:]
m4 = pd.read_csv('/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_Performance.csv').iloc[:, 1:]

# # ---------------- NEW: load deviation-ordered matrices (BiasZ, RawBias) ----------------
# m5 = pd.read_csv(r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_BiasZ.csv').iloc[:, 1:]
# m6 = pd.read_csv(r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_RawBias.csv').iloc[:, 1:]

# ---------------- Load score vectors (aligned to each matrix order) ----------------
# NOTE: These paths point to CSVs; the first numeric column is taken as the score vector.
scores_paths = {
    "EmQ_Score":   '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EmQ_Score.csv',
    "EQ_Score":    '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_EQ_Score.csv',
    "AQ_Score":    '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_AQ_Score.csv',
    "Performance": '/Users/pkell/roi_video_corrmats/brain_avg_ordered_by_Performance.csv',
    # # --- NEW: deviation score paths (use the provided Windows paths) ---
    # "BiasZ":       r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_BiasZ.csv',
    # "RawBias":     r'C:\Users\pkell\roi_video_corrmats\brain_avg_ordered_by_RawBias.csv',
}
def load_scores_onecol(path):
    s = pd.read_csv(path)
    for col in s.columns:
        try:
            vec = pd.to_numeric(s[col], errors='raise').values
            return vec
        except Exception:
            continue
    raise ValueError(f"No numeric score column found in {path}")

scores_dict = {k: load_scores_onecol(v) for k, v in scores_paths.items()}

# ---------------- Compute observed r for each model ----------------
# Mapping: use "converging" (predicted positive) for all unless you have an a priori opposite direction.
ANNAK_VARIANT = "min"   # set to "min" if you prefer that AnnaK variant

r1 = fit_annak_model(m1, scores_dict["EmQ_Score"],   model_kind="converging", variant=ANNAK_VARIANT)
r2 = fit_annak_model(m2, scores_dict["EQ_Score"],    model_kind="converging", variant=ANNAK_VARIANT)
r3 = fit_annak_model(m3, scores_dict["AQ_Score"],    model_kind="converging", variant=ANNAK_VARIANT)
r4 = fit_annak_model(m4, scores_dict["Performance"], model_kind="converging", variant=ANNAK_VARIANT)
# # --- NEW: deviation models (same logic, new files) ---
# r5 = fit_annak_model(m5, scores_dict["BiasZ"],       model_kind="converging", variant=ANNAK_VARIANT)
# r6 = fit_annak_model(m6, scores_dict["RawBias"],     model_kind="converging", variant=ANNAK_VARIANT)

for mat, label, r in zip(
    [m1, m2, m3, m4],
    ['EmQ', 'EQ', 'AQ', 'Performance'],
    [r1, r2, r3, r4]
):
    plt.imshow(mat.replace(1, np.nan), cmap='coolwarm')
    plt.title(f"{label}\nAnnaK position–similarity corr: {round(r, 3)}")
    print(f"{label} AnnaK Fit = {r:.3f}")
    plt.show()

# ---------------- Permutation tests + Bootstrap CIs ----------------
analyses = [
    ("EmQ_Score",   m1, "converging"),
    ("EQ_Score",    m2, "converging"),
    ("AQ_Score",    m3, "converging"),
    ("Performance", m4, "converging"),
#     # --- NEW: include deviation models in the same loop ---
#     ("BiasZ",       m5, "converging"),
#     ("RawBias",     m6, "converging"),
]

# Bump permutations and add bootstrap resamples here
N_PERM = 50000          # increase to 100000+ for finer p resolution
BOOT_B = 5000           # bootstrap draws for r CI
ALPHA  = 0.05
rng = np.random.default_rng(2025)

for label, mat_df, model_kind in analyses:
    # Bootstrap CI for r (subject-level resampling)
    r_hat, (r_lo, r_hi) = bootstrap_r_ci_annak(mat_df, scores_dict[label], model_kind,
                                               B=BOOT_B, alpha=ALPHA, rng=rng, variant=ANNAK_VARIANT)
    # Permutation with MC CIs
    r_true, r_perm, p2_ci, p1_ci = permutation_null_shuffle_scores(
        mat_df, scores_dict[label], model_kind=model_kind,
        n_perm=N_PERM, rng=rng, alpha=ALPHA, variant=ANNAK_VARIANT
    )
    # Print and plot
    plot_null_dist_and_print(label.replace("_", " "), model_kind, r_true, (r_lo, r_hi),
                             r_perm, p2_ci, p1_ci)


EmQ AnnaK Fit = 0.115
EQ AnnaK Fit = 0.138
AQ AnnaK Fit = 0.534
Performance AnnaK Fit = 0.484

=== Permutation test: EmQ Score — model: converging (AnnaK position–similarity) ===
Spearman r = 0.1155  |  95% bootstrap CI [-0.0734, 0.3533]
p(two-sided) = 0.231735  |  MC 95% CI [0.228027, 0.235444]  (N_perm = 50000)
p(one-sided, predicted) = 0.116978  |  MC 95% CI [0.114155, 0.119809]

=== Permutation test: EQ Score — model: converging (AnnaK position–similarity) ===
Spearman r = 0.1380  |  95% bootstrap CI [-0.0183, 0.3444]
p(two-sided) = 0.157077  |  MC 95% CI [0.153881, 0.160279]  (N_perm = 50000)
p(one-sided, predicted) = 0.083998  |  MC 95% CI [0.081563, 0.086445]

=== Permutation test: AQ Score — model: converging (AnnaK position–similarity) ===
Spearman r = 0.5341  |  95% bootstrap CI [0.3300, 0.6764]
p(two-sided) = 0.000020  |  MC 95% CI [0.000000, 0.000074]  (N_perm = 50000)
p(one-sided, predicted) = 0.000020  |  MC 95% CI [0.000000, 0.000074]

=== Permutation test: Performance —

# LASSO Penalized Regression

In [99]:
# === Penalized Regression (LASSO) instead of multiple regression ===
# A) Use per-subject ROI ISCs to predict behavioral performance
#    • Model: Performance ~ all ROI ISCs
# B) Use survey_scores (AQ, EQ, EmQ) to predict overall ISC
#    • overall ISC = per-subject mean across ROI ISCs
# C) Use ROI ISCs to predict each survey score (AQ, EQ, EmQ)

import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, Lasso

# -----------------------------
# Helper: standardize (z-score)
# -----------------------------
def standardize(df_or_s):
    """Z-score standardization with ddof=1 for columns in a DataFrame or a Series."""
    if isinstance(df_or_s, pd.DataFrame):
        return (df_or_s - df_or_s.mean(axis=0)) / df_or_s.std(axis=0, ddof=1)
    else:
        return (df_or_s - df_or_s.mean()) / df_or_s.std(ddof=1)

# -----------------------------
# Helper: fit LASSO on standardized X, y
# -----------------------------
def fit_lasso_standardized(X, y, label,
                           alphas=None,
                           cv_folds=5,
                           random_state=42):
    """
    Standardize X and y, fit LASSO with cross-validated alpha, and
    return a tidy table of penalized standardized betas + R².
    """
    # Drop rows with any missing values in X or y
    mask = (~X.isna().any(axis=1)) & y.notna()
    X_clean = X.loc[mask].copy()
    y_clean = y.loc[mask].copy()

    n, p = X_clean.shape
    if n < p + 2:
        print(f"⚠️ [{label}] n={n}, p={p}. Penalized regression is OK, but n<p; "
              "interpret coefficients cautiously.")

    # Z-score to put everything on comparable scale
    X_z = standardize(X_clean)
    y_z = standardize(y_clean)

    if alphas is None:
        # Reasonable default grid on standardized data
        alphas = np.logspace(-3, 1, 50)

    # Cross-validated LASSO to choose alpha
    lcv = LassoCV(alphas=alphas, cv=min(cv_folds, len(y_z)),
                  random_state=random_state)
    lcv.fit(X_z.values, y_z.values)
    alpha_star = lcv.alpha_

    # Refit a LASSO model with that alpha on all standardized data
    lasso = Lasso(alpha=alpha_star, random_state=random_state)
    lasso.fit(X_z.values, y_z.values)

    # In-sample R² on standardized scale
    y_pred = lasso.predict(X_z.values)
    ss_res = np.sum((y_z.values - y_pred) ** 2)
    ss_tot = np.sum((y_z.values - np.mean(y_z.values)) ** 2)
    r2_in = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # Build coefficient table
    betas = pd.Series(lasso.coef_, index=X_z.columns, name="beta_std_penalized")
    betas_df = (
        betas.to_frame()
        .reset_index()
        .rename(columns={"index": "Predictor"})
        .sort_values("beta_std_penalized", key=lambda s: s.abs(), ascending=False)
    )

    print(f"\n=== Penalized Regression (LASSO) — {label} ===")
    print(f"n = {n}, p = {p}, predictors kept after std (non-constant) = {p}")
    print(f"Chosen alpha = {alpha_star:.5f}")
    print(f"In-sample R² (standardized) = {r2_in:.4f}\n")
    print("Standardized penalized coefficients (sorted by |β|):")
    print(betas_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    return betas_df, r2_in, n, alpha_star, lasso.intercept_

# ============================================================
# A) ROI ISCs -> predict Performance
# ============================================================
# roi_isc_df must have: "Subject" + ROI columns
if "roi_isc_df" not in globals():
    raise ValueError("roi_isc_df not found. It must contain 'Subject' and ROI ISC columns.")

# If roi_cols is not defined globally, infer it as all non-'Subject' columns
if "roi_cols" in globals():
    roi_cols_local = roi_cols
else:
    roi_cols_local = [c for c in roi_isc_df.columns if c != "Subject"]

# X: ROI ISCs (columns = ROIs, rows = subjects)
X_roi = roi_isc_df.set_index("Subject")[roi_cols_local].copy()

# Behavioral performance (from mean_corr_df; must have 'Subject' and 'Performance')
if "mean_corr_df" not in globals():
    raise ValueError("mean_corr_df not found. It must have 'Subject' and 'Performance' (or you must compute it).")

if "Performance" not in mean_corr_df.columns:
    # Fallback: derive Performance as mean of Valence and Arousal if present
    if {"Mean_Valence_Corr", "Mean_Arousal_Corr"}.issubset(mean_corr_df.columns):
        mean_corr_df = mean_corr_df.copy()
        mean_corr_df["Performance"] = mean_corr_df[["Mean_Valence_Corr", "Mean_Arousal_Corr"]].mean(axis=1)
        print("⚠️ 'Performance' not found in mean_corr_df; computed as mean of Valence & Arousal.")
    else:
        raise ValueError("mean_corr_df must contain 'Performance' or (Mean_Valence_Corr & Mean_Arousal_Corr).")

beh = mean_corr_df.set_index("Subject")[["Performance"]].copy()

# Align subjects present in both ROI ISCs and Performance
common_subj = X_roi.index.intersection(beh.index)
X_roi_perf = X_roi.loc[common_subj].copy()
y_perf = beh.loc[common_subj, "Performance"].copy()

# Fit penalized regression: ROIs → Performance
betas_perf_lasso, r2_perf_lasso, n_perf, alpha_perf, intercept_perf = fit_lasso_standardized(
    X_roi_perf, y_perf, label="ROIs → Performance (LASSO)"
)

# ============================================================
# B) Survey scores -> predict overall ISC (mean across ROIs)
# ============================================================
if "survey_scores" not in globals():
    raise ValueError("survey_scores not found. It must have columns: Subject, AQ_Score, EQ_Score, EmQ_Score.")

req_survey = {"Subject", "AQ_Score", "EQ_Score", "EmQ_Score"}
if not req_survey.issubset(set(survey_scores.columns)):
    raise ValueError(f"survey_scores must contain columns {req_survey}")

surv = survey_scores.set_index("Subject")[["AQ_Score", "EQ_Score", "EmQ_Score"]].copy()

# Align with X_roi subjects (since X_roi already has the ROI ISCs by subject)
common_s = X_roi.index.intersection(surv.index)
X_roi_subj = X_roi.loc[common_s].copy()
surv_common = surv.loc[common_s].copy()

# Outcome: overall ISC per subject (mean across their ROI ISCs)
overall_ISC = X_roi_subj.mean(axis=1)

# Fit penalized regression: Surveys → Overall ISC
betas_survey_lasso, r2_survey_lasso, n_survey, alpha_survey, intercept_survey = fit_lasso_standardized(
    surv_common, overall_ISC, label="Surveys (AQ, EQ, EmQ) → Overall ISC (LASSO)"
)

# ============================================================
# C) ROI ISCs -> predict each survey score (AQ, EQ, EmQ)
# ============================================================
# Align ROI matrix with survey subjects
common_surv = X_roi.index.intersection(survey_scores["Subject"].astype(str))
X_for_surveys = X_roi.loc[common_surv].copy()

surv_targets = (
    survey_scores
    .set_index("Subject")
    .loc[common_surv, ["AQ_Score", "EQ_Score", "EmQ_Score"]]
    .copy()
)


=== Penalized Regression (LASSO) — ROIs → Performance (LASSO) ===
n = 34, p = 12, predictors kept after std (non-constant) = 12
Chosen alpha = 0.06251
In-sample R² (standardized) = 0.4732

Standardized penalized coefficients (sorted by |β|):
  Predictor  beta_std_penalized
   Left TPJ              0.4672
 Left VLPFC             -0.4571
Right VLPFC              0.2905
   Left OFC              0.2310
 Left DLPFC             -0.1359
  Right TPJ             -0.0736
   Left STS              0.0393
 Left RLPFC             -0.0000
Right RLPFC             -0.0000
Right DLPFC             -0.0000
  Right OFC              0.0000
  Right STS              0.0000

=== Penalized Regression (LASSO) — Surveys (AQ, EQ, EmQ) → Overall ISC (LASSO) ===
n = 38, p = 3, predictors kept after std (non-constant) = 3
Chosen alpha = 10.00000
In-sample R² (standardized) = 0.0000

Standardized penalized coefficients (sorted by |β|):
Predictor  beta_std_penalized
 AQ_Score              0.0000
 EQ_Score             

In [100]:
# === Plot LASSO βs for ROIs → Performance (sorted by |β|, largest→smallest) ===
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Start from the LASSO coefficient table returned above
# betas_perf_lasso has columns: ["Predictor", "beta_std_penalized"]
betas_plot = betas_perf_lasso.copy()

# Order ROIs by absolute beta (largest magnitude on the left)
betas_plot["abs_beta"] = betas_plot["beta_std_penalized"].abs()
betas_plot = betas_plot.sort_values("abs_beta", ascending=False).reset_index(drop=True)

# Extract ordered values
roi_names = betas_plot["Predictor"].values
beta_vals = betas_plot["beta_std_penalized"].values

# Mark coefficients that LASSO shrank to (approximately) zero
selected = ~np.isclose(beta_vals, 0.0, atol=1e-6)
colors = np.where(selected, "#1f77b4", "#C0C0C0")  # blue = selected, grey = shrunk to 0

# Make the bar plot
x = np.arange(len(roi_names))
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(x, beta_vals, color=colors, edgecolor="black")

ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(roi_names, rotation=45, ha="right")

ax.set_ylabel("Standardized β (LASSO)")
ax.set_title(f"Performance ~ ROI ISCs (LASSO, all ROIs)  |  R² = {r2_perf_lasso:.3f}")

# Set y-limits with a little margin
y_min = np.min(beta_vals)
y_max = np.max(beta_vals)
margin = 0.1 * (y_max - y_min if y_max != y_min else 1.0)
ax.set_ylim(y_min - margin, y_max + margin)

# Add β labels above/below each bar (no p-values for LASSO)
for xi, bi in zip(x, beta_vals):
    if np.isnan(bi):
        continue
    if bi >= 0:
        va = "bottom"
        y_text = bi + 0.02
    else:
        va = "top"
        y_text = bi - 0.02
    ax.text(
        xi,
        y_text,
        f"β={bi:.2f}",
        ha="center",
        va=va,
        fontsize=9
    )

# # Legend: selected vs shrunk-to-zero coefficients
# handles = [
#     Patch(facecolor="#1f77b4", edgecolor="black", label="Selected (|β| > 0)"),
#     Patch(facecolor="#C0C0C0", edgecolor="black", label="Shrunk to 0"),
# ]
# ax.legend(handles=handles, loc="upper right")

plt.tight_layout()
plt.show()

In [101]:
# ============================================
# VIF + In-sample permutation test for LASSO
#   Model: ROIs → Performance
# ============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score

# ------------------------------------------------
# 0) Build X_p (ROI ISCs) and y_perf (Performance)
#    if they are not already defined
# ------------------------------------------------
if ("X_p" not in globals()) or ("y_perf" not in globals()):
    if "roi_isc_df" not in globals():
        raise RuntimeError(
            "roi_isc_df not found. It must be a DataFrame with columns: 'Subject' + ROI ISCs."
        )
    if "mean_corr_df" not in globals():
        raise RuntimeError(
            "mean_corr_df not found. It must contain 'Subject' and 'Performance' columns."
        )

    # If roi_cols not defined, infer ROI columns as everything except 'Subject'
    if "roi_cols" not in globals():
        roi_cols = [c for c in roi_isc_df.columns if c != "Subject"]

    # ROI ISC matrix
    X_roi_full = roi_isc_df.set_index("Subject")[roi_cols].copy()

    # Performance outcome
    if "Performance" not in mean_corr_df.columns:
        raise RuntimeError(
            "mean_corr_df must contain a 'Performance' column (your combined valence/arousal score)."
        )
    beh_full = mean_corr_df.set_index("Subject")[["Performance"]].copy()

    # Align subjects
    common_subj = X_roi_full.index.intersection(beh_full.index)
    if len(common_subj) < 5:
        raise RuntimeError(
            f"Too few overlapping subjects between roi_isc_df and mean_corr_df (n={len(common_subj)})."
        )

    X_p = X_roi_full.loc[common_subj].copy()
    y_perf = beh_full.loc[common_subj, "Performance"].copy()

    print(f"Constructed X_p and y_perf from roi_isc_df / mean_corr_df (n={len(common_subj)}).")
else:
    print(f"Using existing X_p (shape={X_p.shape}) and y_perf (len={len(y_perf)}).")

# ------------------------------------------------
# 1) VIF for ROI predictors (independent of model type)
# ------------------------------------------------
def compute_vif_penalized(X: pd.DataFrame, label: str = "ROIs → Performance (LASSO model)"):
    """
    Compute VIF for each predictor (column) in X.
    This is the same VIF concept as in OLS; it just tells you collinearity among ROIs.
    """
    # Drop any rows with missing values
    mask = ~X.isna().any(axis=1)
    Xc = X.loc[mask].copy()

    # Standardize columns (helps numerically but doesn't change VIF)
    Xz = (Xc - Xc.mean(axis=0)) / Xc.std(axis=0, ddof=1)
    Xz_np = Xz.to_numpy()

    vif_vals = []
    cols = Xc.columns.to_list()
    for i, col in enumerate(cols):
        vif = variance_inflation_factor(Xz_np, i)
        vif_vals.append((col, vif))

    vif_df = (
        pd.DataFrame(vif_vals, columns=["ROI", "VIF"])
        .sort_values("VIF", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\n=== VIF for ROI predictors — {label} ===")
    print(vif_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    return vif_df

vif_perf = compute_vif_penalized(X_p, label="ROIs → Performance (LASSO model)")

# ------------------------------------------------
# 2) In-sample permutation test for LASSO R²
# ------------------------------------------------
def perm_test_in_sample_lasso(
    X: pd.DataFrame,
    y,
    n_perm: int = 10000,
    random_state: int = 123,
    cv_folds: int = 5,
    label: str = "ROIs → Performance"
):
    """
    In-sample permutation test for LASSO:
      - Standardize X and y.
      - Fit LassoCV on real data, get in-sample R².
      - Permute y n_perm times, refit LassoCV each time, and compute in-sample R².
      - Return observed R², null distribution, p-value, and chosen alpha on real data.

    NOTE: This is an in-sample explanatory test (not out-of-sample prediction).
    """
    rng = np.random.default_rng(random_state)

    # Convert y to numeric numpy array
    if isinstance(y, (pd.Series, pd.DataFrame)):
        y_vec = np.asarray(y).astype(float).ravel()
    else:
        y_vec = np.asarray(y).astype(float)

    # Drop rows with NaNs in X or y
    mask = (~X.isna().any(axis=1)) & np.isfinite(y_vec)
    Xc = X.loc[mask].copy()
    yc = y_vec[mask]

    # Standardize predictors and outcome
    Xz = (Xc - Xc.mean(axis=0)) / Xc.std(axis=0, ddof=1)
    yz = (yc - yc.mean()) / yc.std(ddof=1)

    n_subj = len(yz)
    n_pred = Xz.shape[1]
    k_cv = min(cv_folds, n_subj)

    # Fit LassoCV on real data
    lassocv = LassoCV(
        cv=k_cv,
        random_state=random_state,
        n_alphas=100
    ).fit(Xz.values, yz)

    alpha_star = lassocv.alpha_
    yhat = lassocv.predict(Xz.values)
    r2_obs = r2_score(yz, yhat)

    # Permutation null (in-sample R²)
    r2_null = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        y_perm = rng.permutation(yz)
        m_perm = LassoCV(
            cv=k_cv,
            random_state=random_state + i + 1,
            n_alphas=100
        ).fit(Xz.values, y_perm)
        yhat_perm = m_perm.predict(Xz.values)
        r2_null[i] = r2_score(y_perm, yhat_perm)

        if (i + 1) % 100 == 0:
            print(f"[{label}] in-sample permutations: {i+1}/{n_perm} completed")

    # Monte Carlo p-value (one-sided: observed R² ≥ null)
    p_mc = (1 + np.sum(r2_null >= r2_obs)) / (n_perm + 1)

    print(f"\n=== In-sample LASSO permutation test — {label} ===")
    print(f"n = {n_subj}, p = {n_pred}")
    print(f"Chosen alpha (real data) = {alpha_star:.5f}")
    print(f"Observed in-sample R² = {r2_obs:.4f}")
    print(f"Permutation null mean R² = {r2_null.mean():.4f}")
    print(f"MC p(one-sided, ≥) = {p_mc:.4f}  (n_perm = {n_perm})")

        # Plot permutation distribution (cleaner null)
    plt.figure(figsize=(7, 4))

    # Choose many small bins over the range of the null + a bit past the observed R²
    r2_max = max(r2_null.max(), r2_obs)
    bins = np.linspace(0, r2_max * 1.05, 80)  # 80 bins for a smoother histogram

    plt.hist(r2_null, bins=bins, edgecolor="white", alpha=0.85)
    plt.axvline(r2_obs, linestyle='--', linewidth=2, label=f"Observed R² = {r2_obs:.3f}")

    ax = plt.gca()
    # Nice x-ticks across the range
    xticks = np.linspace(0, r2_max * 1.05, 9)  # 9 ticks from 0 to max
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{x:.2f}" for x in xticks])

    plt.xlabel("In-sample R² under permutation")
    plt.ylabel("Count")
    plt.title(f"In-sample LASSO R² null — {label}\np = {p_mc:.4f}")
    plt.legend()
    plt.tight_layout()
    plt.show()


    return r2_obs, r2_null, p_mc, alpha_star

# Run for your ROIs → Performance model
r2_obs_perf, r2_null_perf, p_perf_in_sample, alpha_perf = perm_test_in_sample_lasso(
    X_p,
    y_perf,
    n_perm=10000,
    random_state=123,
    cv_folds=5,
    label="ROIs → Performance"
)


Using existing X_p (shape=(42, 12)) and y_perf (len=42).

=== VIF for ROI predictors — ROIs → Performance (LASSO model) ===
        ROI   VIF
Right RLPFC 7.245
Right DLPFC 7.128
 Left RLPFC 5.035
  Right OFC 4.452
 Left DLPFC 4.010
 Left VLPFC 3.631
   Left STS 3.287
  Right STS 2.968
  Right TPJ 2.616
Right VLPFC 2.587
   Left OFC 2.565
   Left TPJ 1.999
[ROIs → Performance] in-sample permutations: 100/10000 completed
[ROIs → Performance] in-sample permutations: 200/10000 completed
[ROIs → Performance] in-sample permutations: 300/10000 completed
[ROIs → Performance] in-sample permutations: 400/10000 completed
[ROIs → Performance] in-sample permutations: 500/10000 completed
[ROIs → Performance] in-sample permutations: 600/10000 completed
[ROIs → Performance] in-sample permutations: 700/10000 completed
[ROIs → Performance] in-sample permutations: 800/10000 completed
[ROIs → Performance] in-sample permutations: 900/10000 completed
[ROIs → Performance] in-sample permutations: 1000/10000 c

In [102]:
# # ================================================
# # LASSO: CV performance + permutation R² + bootstrap stability
# # ================================================
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# from sklearn.linear_model import LassoCV, Lasso
# from sklearn.model_selection import KFold
# from sklearn.metrics import r2_score

# # --------------------------------
# # Helpers
# # --------------------------------
# def _z_df(df: pd.DataFrame) -> pd.DataFrame:
#     """Z-score each column of a DataFrame (ddof=1)."""
#     return (df - df.mean(axis=0)) / df.std(axis=0, ddof=1)

# def _z_series(s: pd.Series) -> pd.Series:
#     """Z-score a Series (ddof=1)."""
#     return (s - s.mean()) / s.std(ddof=1)

# # --------------------------------
# # 1) Fit LASSO with CV to choose alpha, then compute CV R²
# # --------------------------------
# def lasso_cv_performance(X: pd.DataFrame,
#                          y: pd.Series,
#                          label: str,
#                          inner_k: int = 5,
#                          outer_k: int = 5,
#                          random_state: int = 123,
#                          n_alphas: int = 100):
#     """
#     1) Z-score X and y.
#     2) Use LassoCV on full data to pick alpha.
#     3) Compute:
#          - in-sample R²
#          - outer K-fold CV R² using fixed alpha.
#     Returns (alpha, R2_in, R2_cv_mean, R2_cv_sd, Xz, yz, outer_cv).
#     """
#     # Complete cases
#     mask = (~X.isna().any(axis=1)) & y.notna()
#     Xc = X.loc[mask].copy()
#     yc = y.loc[mask].copy()
#     if Xc.shape[0] < 5:
#         raise ValueError(f"[{label}] Too few subjects after NaN drop (n={Xc.shape[0]}).")

#     Xz = _z_df(Xc)
#     yz = _z_series(yc)

#     # 1) LassoCV to choose alpha
#     inner_k_eff = min(inner_k, len(yz))
#     lcv = LassoCV(
#         cv=inner_k_eff,
#         random_state=random_state,
#         n_alphas=n_alphas
#     ).fit(Xz.values, yz.values)
#     alpha_chosen = float(lcv.alpha_)

#     # In-sample R²
#     y_hat_in = lcv.predict(Xz.values)
#     r2_in = float(r2_score(yz.values, y_hat_in))

#     # 2) Outer CV R² with fixed alpha
#     outer_k_eff = min(outer_k, len(yz))
#     kf = KFold(n_splits=outer_k_eff, shuffle=True, random_state=random_state)

#     r2_list = []
#     for train_idx, test_idx in kf.split(Xz.values):
#         Xtr, Xte = Xz.values[train_idx], Xz.values[test_idx]
#         ytr, yte = yz.values[train_idx], yz.values[test_idx]

#         model = Lasso(alpha=alpha_chosen, random_state=random_state)
#         model.fit(Xtr, ytr)
#         y_pred = model.predict(Xte)
#         r2_list.append(r2_score(yte, y_pred))

#     r2_list = np.array(r2_list, dtype=float)
#     r2_cv_mean = float(np.mean(r2_list))
#     r2_cv_sd   = float(np.std(r2_list, ddof=1)) if len(r2_list) > 1 else np.nan

#     print(f"\n=== LASSO CV summary — {label} ===")
#     print(f"n = {len(yz)}, p = {Xz.shape[1]}")
#     print(f"Chosen alpha (from inner CV) = {alpha_chosen:.5f}")
#     print(f"In-sample R² (standardized)  = {r2_in:.4f}")
#     print(f"Outer CV R² (mean±SD)        = {r2_cv_mean:.4f} ± {r2_cv_sd:.4f}  (k = {outer_k_eff})")

#     return alpha_chosen, r2_in, r2_cv_mean, r2_cv_sd, Xz, yz, kf

# # --------------------------------
# # 2) Permutation test on CV R² for LASSO
# # --------------------------------
# def lasso_perm_test_cv_r2(X: pd.DataFrame,
#                           y: pd.Series,
#                           label: str,
#                           n_perm: int = 1000,
#                           inner_k: int = 5,
#                           outer_k: int = 5,
#                           random_state: int = 123,
#                           n_alphas: int = 100):
#     """
#     1) Fit LASSO with CV to pick alpha and compute observed CV R².
#     2) Permute y (standardized) n_perm times and recompute CV R² with fixed alpha & same folds.
#     3) Returns (r2_cv_obs, r2_perm, p_mc, alpha_chosen).
#     """
#     rng = np.random.default_rng(random_state)

#     # Step 1: observed alpha & CV R²
#     alpha_chosen, r2_in, r2_cv_obs, r2_cv_sd, Xz, yz, kf = lasso_cv_performance(
#         X, y, label, inner_k=inner_k, outer_k=outer_k,
#         random_state=random_state, n_alphas=n_alphas
#     )

#     # Step 2: permutation null
#     r2_perm = np.empty(n_perm, dtype=float)
#     for b in range(n_perm):
#         y_perm = rng.permutation(yz.values)
#         r2_list = []
#         for train_idx, test_idx in kf.split(Xz.values):
#             Xtr, Xte = Xz.values[train_idx], Xz.values[test_idx]
#             ytr, yte = y_perm[train_idx], y_perm[test_idx]
#             model = Lasso(alpha=alpha_chosen, random_state=random_state)
#             model.fit(Xtr, ytr)
#             y_pred = model.predict(Xte)
#             r2_list.append(r2_score(yte, y_pred))
#         r2_perm[b] = float(np.mean(r2_list))

#         if (b + 1) % max(1, n_perm // 10) == 0:
#             print(f"[{label}] permutations: {b+1}/{n_perm} completed")

#     # Monte Carlo p-value: P(R²_perm ≥ R²_obs)
#     p_mc = (1.0 + np.sum(r2_perm >= r2_cv_obs)) / (n_perm + 1.0)

#     print(f"\n=== LASSO permutation test on CV R² — {label} ===")
#     print(f"Observed CV R² = {r2_cv_obs:.4f}")
#     print(f"Permutation null mean R² = {np.mean(r2_perm):.4f}")
#     print(f"MC p(one-sided, ≥) = {p_mc:.4f}  (n_perm = {n_perm})")

#     # Optional plot
#     plt.figure(figsize=(6, 4))
#     plt.hist(r2_perm, bins=30, density=True, alpha=0.75)
#     plt.axvline(r2_cv_obs, color='r', linestyle='--', linewidth=2,
#                 label=f'Observed CV R² = {r2_cv_obs:.3f}')
#     plt.xlabel("CV R² under permutation (LASSO)")
#     plt.ylabel("Density")
#     plt.title(f"LASSO permutation null of CV R² — {label}\np = {p_mc:.4f}")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

#     return r2_cv_obs, r2_perm, p_mc, alpha_chosen

# # --------------------------------
# # 3) Bootstrap LASSO stability (selection freq + β CI)
# # --------------------------------
# def lasso_bootstrap_stability(X: pd.DataFrame,
#                               y: pd.Series,
#                               label: str,
#                               B: int = 1000,
#                               inner_k: int = 5,
#                               random_state: int = 123,
#                               n_alphas: int = 100,
#                               tol_zero: float = 1e-6):
#     """
#     Bootstrap LASSO:
#       - Resample subjects with replacement B times.
#       - For each bootstrap sample, z-score X & y, run LassoCV to choose alpha,
#         refit, and store coefficients.
#       - For each ROI:
#           • selection_freq = P(|β| > tol_zero)
#           • beta_median, 2.5th, 97.5th percentiles across bootstraps.
#     Returns: DataFrame with ['ROI','selection_freq','beta_median','beta_CI_lo','beta_CI_hi'].
#     """
#     rng = np.random.default_rng(random_state)

#     mask = (~X.isna().any(axis=1)) & y.notna()
#     Xc = X.loc[mask].copy()
#     yc = y.loc[mask].copy()
#     subs = Xc.index.to_numpy()
#     rois = list(Xc.columns)
#     n = len(subs)

#     inner_k_eff = min(inner_k, n)

#     coef_store = np.zeros((B, len(rois)), dtype=float)

#     for b in range(B):
#         # Resample subjects with replacement
#         samp_idx = rng.integers(low=0, high=n, size=n)
#         samp_subs = subs[samp_idx]
#         Xb = Xc.loc[samp_subs].copy()
#         yb = yc.loc[samp_subs].copy()

#         # Z-score within bootstrap sample
#         Xbz = _z_df(Xb)
#         ybz = _z_series(yb)

#         try:
#             lcv = LassoCV(
#                 cv=inner_k_eff,
#                 random_state=random_state + b,
#                 n_alphas=n_alphas
#             ).fit(Xbz.values, ybz.values)
#             coef_store[b, :] = lcv.coef_
#         except Exception:
#             # If something fails, keep zeros for this iteration
#             coef_store[b, :] = 0.0

#         if (b + 1) % max(1, B // 10) == 0:
#             print(f"[{label}] bootstrap: {b+1}/{B} completed")

#     # Summarize across bootstraps
#     selection_freq = (np.abs(coef_store) > tol_zero).mean(axis=0)

#     beta_median = np.nanmedian(coef_store, axis=0)
#     beta_lo = np.nanpercentile(coef_store, 2.5, axis=0)
#     beta_hi = np.nanpercentile(coef_store, 97.5, axis=0)

#     out = pd.DataFrame({
#         "ROI": rois,
#         "selection_freq": selection_freq,
#         "beta_median": beta_median,
#         "beta_CI_lo": beta_lo,
#         "beta_CI_hi": beta_hi
#     }).sort_values("selection_freq", ascending=False)

#     print(f"\n=== LASSO bootstrap stability — {label} (B={B}) ===")
#     print(out.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

#     return out

# # --------------------------------
# # 3b) Plot bootstrap stability (selection freq + β CI)
# # --------------------------------
# def plot_lasso_bootstrap_stability(df: pd.DataFrame,
#                                    title: str,
#                                    top_n: int | None = None,
#                                    figsize=(12, 5)):
#     """
#     df must have: ['ROI','selection_freq','beta_median','beta_CI_lo','beta_CI_hi'].
#     Left panel: selection frequency.
#     Right panel: median β with 95% CI.
#     """
#     req = {"ROI", "selection_freq", "beta_median", "beta_CI_lo", "beta_CI_hi"}
#     if not req.issubset(df.columns):
#         raise ValueError(f"Input df must contain columns: {sorted(list(req))}")

#     d = df.copy().sort_values("selection_freq", ascending=False).reset_index(drop=True)
#     if top_n is not None:
#         d = d.iloc[:top_n].copy()

#     rois   = d["ROI"].tolist()
#     freq   = d["selection_freq"].values.astype(float)
#     bmed   = d["beta_median"].values.astype(float)
#     blo    = d["beta_CI_lo"].values.astype(float)
#     bhi    = d["beta_CI_hi"].values.astype(float)
#     x = np.arange(len(rois))

#     beta_err_lo = bmed - blo
#     beta_err_hi = bhi - bmed

#     # Mark ROIs whose CI excludes 0
#     sig = (blo > 0) | (bhi < 0)

#     fig, axes = plt.subplots(1, 2, figsize=figsize)
#     ax1, ax2 = axes

#     # Left: selection frequency
#     colors1 = np.where(sig, "#4C78A8", "#B0B0B0")  # blue if β CI excludes 0
#     ax1.bar(x, freq, color=colors1, edgecolor="black")
#     ax1.set_xticks(x)
#     ax1.set_xticklabels(rois, rotation=45, ha="right")
#     ax1.set_ylabel("Selection frequency")
#     ax1.set_ylim(0, min(1.05, max(1.0, np.nanmax(freq) + 0.05)))
#     ax1.set_title(f"{title}\nSelection frequency (LASSO)")

#     # Annotate frequencies
#     for xi, yi in zip(x, freq):
#         ax1.text(xi, yi + 0.02, f"{yi:.2f}", ha="center", va="bottom", fontsize=8)

#     # Right: median β with CI
#     colors2 = np.where(sig, "#59A14F", "#B0B0B0")  # green if CI excludes 0
#     ax2.bar(x, bmed, color=colors2, edgecolor="black")
#     yerr = np.vstack([beta_err_lo, beta_err_hi])
#     ax2.errorbar(x, bmed, yerr=yerr, fmt="none", ecolor="black", capsize=3, linewidth=1)
#     ax2.axhline(0, color="k", linewidth=1)

#     ax2.set_xticks(x)
#     ax2.set_xticklabels(rois, rotation=45, ha="right")
#     ax2.set_ylabel("Median standardized β (bootstrap LASSO)")
#     ax2.set_title(f"{title}\nMedian β ± 95% CI")

#     plt.tight_layout()
#     plt.show()

# # ================================================
# # EXAMPLE USAGE: ROIs → Performance
# # ================================================
# # Assumes:
# #   - X_roi: DataFrame (index=Subject, columns=ROIs)
# #   - beh:   DataFrame (index=Subject) with column "Performance"

# # Align subjects (safety)
# common_subj = X_roi.index.intersection(beh.index)
# X_perf = X_roi.loc[common_subj].copy()
# y_perf = beh.loc[common_subj, "Performance"].copy()

# # 1) LASSO CV performance + alpha selection
# alpha_perf, r2_in_perf, r2_cv_perf, r2_cv_sd_perf, Xz_perf, yz_perf, kf_perf = lasso_cv_performance(
#     X_perf, y_perf, label="ROIs → Performance", inner_k=5, outer_k=5,
#     random_state=42, n_alphas=100
# )

# # 2) Permutation test on CV R²
# r2_cv_obs_perf, r2_perm_perf, p_mc_perf, alpha_perf_perm = lasso_perm_test_cv_r2(
#     X_perf, y_perf, label="ROIs → Performance", n_perm=1000,
#     inner_k=5, outer_k=5, random_state=7, n_alphas=100
# )

# # 3) Bootstrap LASSO stability (selection frequency + β CI)
# boot_lasso_perf = lasso_bootstrap_stability(
#     X_perf, y_perf, label="ROIs → Performance",
#     B=1000, inner_k=5, random_state=123, n_alphas=100
# )

# # 3b) Plot bootstrap stability
# plot_lasso_bootstrap_stability(boot_lasso_perf, title="ROIs → Performance (LASSO bootstrap)", top_n=None)


In [103]:
# ================================================
# LASSO: LOOCV performance + permutation R² + bootstrap stability
# ================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LassoCV, Lasso
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score

# --------------------------------
# Helpers
# --------------------------------
def _z_df(df: pd.DataFrame) -> pd.DataFrame:
    """Z-score each column of a DataFrame (ddof=1)."""
    return (df - df.mean(axis=0)) / df.std(axis=0, ddof=1)

def _z_series(s: pd.Series) -> pd.Series:
    """Z-score a Series (ddof=1)."""
    return (s - s.mean()) / s.std(ddof=1)

# --------------------------------
# 1) Fit LASSO with CV to choose alpha, then compute LOOCV R²
# --------------------------------
def lasso_cv_performance(X: pd.DataFrame,
                         y: pd.Series,
                         label: str,
                         inner_k: int = 5,
                         random_state: int = 123,
                         n_alphas: int = 100):
    """
    1) Z-score X and y.
    2) Use LassoCV on full data to pick alpha.
    3) Compute:
         - in-sample R²
         - Leave-One-Out CV R² using fixed alpha.

    Returns:
        alpha_chosen, r2_in, r2_loocv, r2_loocv_sd, Xz, yz, loo
    """

    # Complete cases
    mask = (~X.isna().any(axis=1)) & y.notna()
    Xc = X.loc[mask].copy()
    yc = y.loc[mask].copy()

    if Xc.shape[0] < 5:
        raise ValueError(f"[{label}] Too few subjects after NaN drop (n={Xc.shape[0]}).")

    Xz = _z_df(Xc)
    yz = _z_series(yc)

    # --------------------------------
    # Choose alpha using inner CV
    # --------------------------------
    inner_k_eff = min(inner_k, len(yz))

    lcv = LassoCV(
        cv=inner_k_eff,
        random_state=random_state,
        n_alphas=n_alphas
    ).fit(Xz.values, yz.values)

    alpha_chosen = float(lcv.alpha_)

    # --------------------------------
    # In-sample R²
    # --------------------------------
    y_hat_in = lcv.predict(Xz.values)
    r2_in = float(r2_score(yz.values, y_hat_in))

    # --------------------------------
    # Leave-One-Out Cross-Validation
    # --------------------------------
    loo = LeaveOneOut()

    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in loo.split(Xz.values):

        Xtr, Xte = Xz.values[train_idx], Xz.values[test_idx]
        ytr, yte = yz.values[train_idx], yz.values[test_idx]

        model = Lasso(alpha=alpha_chosen, random_state=random_state)
        model.fit(Xtr, ytr)

        y_pred = model.predict(Xte)

        y_true_all.append(yte[0])
        y_pred_all.append(y_pred[0])

    y_true_all = np.array(y_true_all, dtype=float)
    y_pred_all = np.array(y_pred_all, dtype=float)

    # Overall LOOCV R² across all held-out predictions
    r2_loocv = float(r2_score(y_true_all, y_pred_all))

    # Fold-wise R² is not meaningful for LOOCV because each test fold has one subject
    r2_loocv_sd = np.nan

    print(f"\n=== LASSO LOOCV summary — {label} ===")
    print(f"n = {len(yz)}, p = {Xz.shape[1]}")
    print(f"Chosen alpha (from inner CV) = {alpha_chosen:.5f}")
    print(f"In-sample R² (standardized)  = {r2_in:.4f}")
    print(f"LOOCV R²                     = {r2_loocv:.4f}")

    return alpha_chosen, r2_in, r2_loocv, r2_loocv_sd, Xz, yz, loo

# --------------------------------
# 2) Permutation test on LOOCV R² for LASSO
# --------------------------------
def lasso_perm_test_cv_r2(X: pd.DataFrame,
                          y: pd.Series,
                          label: str,
                          n_perm: int = 1000,
                          inner_k: int = 5,
                          random_state: int = 123,
                          n_alphas: int = 100):
    """
    1) Fit LASSO with CV to pick alpha and compute observed LOOCV R².
    2) Permute y n_perm times and recompute LOOCV R² with fixed alpha.
    3) Returns:
         r2_loocv_obs, r2_perm, p_mc, alpha_chosen
    """

    rng = np.random.default_rng(random_state)

    # Step 1: observed alpha and LOOCV R²
    alpha_chosen, r2_in, r2_loocv_obs, r2_loocv_sd, Xz, yz, loo = lasso_cv_performance(
        X,
        y,
        label,
        inner_k=inner_k,
        random_state=random_state,
        n_alphas=n_alphas
    )

    # Step 2: permutation null
    r2_perm = np.empty(n_perm, dtype=float)

    for b in range(n_perm):

        y_perm = rng.permutation(yz.values)

        y_true_perm_all = []
        y_pred_perm_all = []

        for train_idx, test_idx in loo.split(Xz.values):

            Xtr, Xte = Xz.values[train_idx], Xz.values[test_idx]
            ytr, yte = y_perm[train_idx], y_perm[test_idx]

            model = Lasso(alpha=alpha_chosen, random_state=random_state)
            model.fit(Xtr, ytr)

            y_pred = model.predict(Xte)

            y_true_perm_all.append(yte[0])
            y_pred_perm_all.append(y_pred[0])

        y_true_perm_all = np.array(y_true_perm_all, dtype=float)
        y_pred_perm_all = np.array(y_pred_perm_all, dtype=float)

        r2_perm[b] = float(r2_score(y_true_perm_all, y_pred_perm_all))

        if (b + 1) % max(1, n_perm // 10) == 0:
            print(f"[{label}] permutations: {b + 1}/{n_perm} completed")

    # Monte Carlo p-value: P(R²_perm >= R²_obs)
    p_mc = (1.0 + np.sum(r2_perm >= r2_loocv_obs)) / (n_perm + 1.0)

    print(f"\n=== LASSO permutation test on LOOCV R² — {label} ===")
    print(f"Observed LOOCV R² = {r2_loocv_obs:.4f}")
    print(f"Permutation null mean LOOCV R² = {np.mean(r2_perm):.4f}")
    print(f"MC p(one-sided, ≥) = {p_mc:.4f}  (n_perm = {n_perm})")

    # Optional plot
    plt.figure(figsize=(6, 4))
    plt.hist(r2_perm, bins=30, density=True, alpha=0.75)
    plt.axvline(
        r2_loocv_obs,
        color="r",
        linestyle="--",
        linewidth=2,
        label=f"Observed LOOCV R² = {r2_loocv_obs:.3f}"
    )
    plt.xlabel("LOOCV R² under permutation (LASSO)")
    plt.ylabel("Density")
    plt.title(f"LASSO permutation null of LOOCV R² — {label}\np = {p_mc:.4f}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return r2_loocv_obs, r2_perm, p_mc, alpha_chosen

# --------------------------------
# 3) Bootstrap LASSO stability: selection frequency + beta CI
# --------------------------------
def lasso_bootstrap_stability(X: pd.DataFrame,
                              y: pd.Series,
                              label: str,
                              B: int = 1000,
                              inner_k: int = 5,
                              random_state: int = 123,
                              n_alphas: int = 100,
                              tol_zero: float = 1e-6):
    """
    Bootstrap LASSO:
      - Resample subjects with replacement B times.
      - For each bootstrap sample, z-score X and y.
      - Run LassoCV to choose alpha.
      - Store coefficients.
      - Return selection frequency and beta confidence intervals.
    """

    rng = np.random.default_rng(random_state)

    mask = (~X.isna().any(axis=1)) & y.notna()
    Xc = X.loc[mask].copy()
    yc = y.loc[mask].copy()

    subs = Xc.index.to_numpy()
    rois = list(Xc.columns)
    n = len(subs)

    if n < 5:
        raise ValueError(f"[{label}] Too few subjects after NaN drop (n={n}).")

    inner_k_eff = min(inner_k, n)

    coef_store = np.zeros((B, len(rois)), dtype=float)

    for b in range(B):

        # Resample subjects with replacement
        samp_idx = rng.integers(low=0, high=n, size=n)
        samp_subs = subs[samp_idx]

        Xb = Xc.loc[samp_subs].copy()
        yb = yc.loc[samp_subs].copy()

        # Z-score within bootstrap sample
        Xbz = _z_df(Xb)
        ybz = _z_series(yb)

        try:
            lcv = LassoCV(
                cv=inner_k_eff,
                random_state=random_state + b,
                n_alphas=n_alphas
            ).fit(Xbz.values, ybz.values)

            coef_store[b, :] = lcv.coef_

        except Exception:
            coef_store[b, :] = 0.0

        if (b + 1) % max(1, B // 10) == 0:
            print(f"[{label}] bootstrap: {b + 1}/{B} completed")

    # Summarize across bootstraps
    selection_freq = (np.abs(coef_store) > tol_zero).mean(axis=0)

    beta_median = np.nanmedian(coef_store, axis=0)
    beta_lo = np.nanpercentile(coef_store, 2.5, axis=0)
    beta_hi = np.nanpercentile(coef_store, 97.5, axis=0)

    out = pd.DataFrame({
        "ROI": rois,
        "selection_freq": selection_freq,
        "beta_median": beta_median,
        "beta_CI_lo": beta_lo,
        "beta_CI_hi": beta_hi
    }).sort_values("selection_freq", ascending=False)

    print(f"\n=== LASSO bootstrap stability — {label} (B={B}) ===")
    print(out.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    return out

# --------------------------------
# 3b) Plot bootstrap stability
# --------------------------------
def plot_lasso_bootstrap_stability(df: pd.DataFrame,
                                   title: str,
                                   top_n: int | None = None,
                                   figsize=(12, 5)):
    """
    df must have:
        ['ROI', 'selection_freq', 'beta_median', 'beta_CI_lo', 'beta_CI_hi']

    Left panel:
        selection frequency

    Right panel:
        median beta with 95% CI
    """

    req = {"ROI", "selection_freq", "beta_median", "beta_CI_lo", "beta_CI_hi"}

    if not req.issubset(df.columns):
        raise ValueError(f"Input df must contain columns: {sorted(list(req))}")

    d = df.copy().sort_values("selection_freq", ascending=False).reset_index(drop=True)

    if top_n is not None:
        d = d.iloc[:top_n].copy()

    rois = d["ROI"].tolist()
    freq = d["selection_freq"].values.astype(float)
    bmed = d["beta_median"].values.astype(float)
    blo = d["beta_CI_lo"].values.astype(float)
    bhi = d["beta_CI_hi"].values.astype(float)

    x = np.arange(len(rois))

    beta_err_lo = bmed - blo
    beta_err_hi = bhi - bmed

    # Mark ROIs whose CI excludes 0
    sig = (blo > 0) | (bhi < 0)

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    ax1, ax2 = axes

    # Left: selection frequency
    colors1 = np.where(sig, "#4C78A8", "#B0B0B0")

    ax1.bar(x, freq, color=colors1, edgecolor="black")
    ax1.set_xticks(x)
    ax1.set_xticklabels(rois, rotation=45, ha="right")
    ax1.set_ylabel("Selection frequency")
    ax1.set_ylim(0, min(1.05, max(1.0, np.nanmax(freq) + 0.05)))
    ax1.set_title(f"{title}\nSelection frequency (LASSO)")

    for xi, yi in zip(x, freq):
        ax1.text(
            xi,
            yi + 0.02,
            f"{yi:.2f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

    # Right: median beta with CI
    colors2 = np.where(sig, "#59A14F", "#B0B0B0")

    ax2.bar(x, bmed, color=colors2, edgecolor="black")

    yerr = np.vstack([beta_err_lo, beta_err_hi])

    ax2.errorbar(
        x,
        bmed,
        yerr=yerr,
        fmt="none",
        ecolor="black",
        capsize=3,
        linewidth=1
    )

    ax2.axhline(0, color="k", linewidth=1)
    ax2.set_xticks(x)
    ax2.set_xticklabels(rois, rotation=45, ha="right")
    ax2.set_ylabel("Median standardized beta, bootstrap LASSO")
    ax2.set_title(f"{title}\nMedian beta ± 95% CI")

    plt.tight_layout()
    plt.show()

# ================================================
# EXAMPLE USAGE: ROIs → Performance
# ================================================
# Assumes:
#   - X_roi: DataFrame with index = Subject and columns = ROIs
#   - beh: DataFrame with index = Subject and column "Performance"

# Align subjects
common_subj = X_roi.index.intersection(beh.index)

X_perf = X_roi.loc[common_subj].copy()
y_perf = beh.loc[common_subj, "Performance"].copy()

# --------------------------------
# 1) LASSO LOOCV performance + alpha selection
# --------------------------------
alpha_perf, r2_in_perf, r2_loocv_perf, r2_loocv_sd_perf, Xz_perf, yz_perf, loo_perf = lasso_cv_performance(
    X_perf,
    y_perf,
    label="ROIs → Performance",
    inner_k=5,
    random_state=42,
    n_alphas=100
)

# --------------------------------
# 2) Permutation test on LOOCV R²
# --------------------------------
r2_loocv_obs_perf, r2_perm_perf, p_mc_perf, alpha_perf_perm = lasso_perm_test_cv_r2(
    X_perf,
    y_perf,
    label="ROIs → Performance",
    n_perm=10000,
    inner_k=5,
    random_state=7,
    n_alphas=100
)

# --------------------------------
# 3) Bootstrap LASSO stability
# --------------------------------
boot_lasso_perf = lasso_bootstrap_stability(
    X_perf,
    y_perf,
    label="ROIs → Performance",
    B=1000,
    inner_k=5,
    random_state=123,
    n_alphas=100
)

# --------------------------------
# 3b) Plot bootstrap stability
# --------------------------------
plot_lasso_bootstrap_stability(
    boot_lasso_perf,
    title="ROIs → Performance (LASSO bootstrap)",
    top_n=None
)


=== LASSO LOOCV summary — ROIs → Performance ===
n = 34, p = 12
Chosen alpha (from inner CV) = 0.06258
In-sample R² (standardized)  = 0.4730
LOOCV R²                     = 0.0734

=== LASSO LOOCV summary — ROIs → Performance ===
n = 34, p = 12
Chosen alpha (from inner CV) = 0.06258
In-sample R² (standardized)  = 0.4730
LOOCV R²                     = 0.0734
[ROIs → Performance] permutations: 1000/10000 completed
[ROIs → Performance] permutations: 2000/10000 completed
[ROIs → Performance] permutations: 3000/10000 completed
[ROIs → Performance] permutations: 4000/10000 completed
[ROIs → Performance] permutations: 5000/10000 completed
[ROIs → Performance] permutations: 6000/10000 completed
[ROIs → Performance] permutations: 7000/10000 completed
[ROIs → Performance] permutations: 8000/10000 completed
[ROIs → Performance] permutations: 9000/10000 completed
[ROIs → Performance] permutations: 10000/10000 completed

=== LASSO permutation test on LOOCV R² — ROIs → Performance ===
Observed LOOCV

C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.005170338947945385, tolerance: 0.002655100267941017
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.005185208351483173, tolerance: 0.002655100267941017
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.013902460101063774, tolerance: 0.002655100267941017
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_

[ROIs → Performance] bootstrap: 100/1000 completed


C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.003845089658803147, tolerance: 0.0019633513098023334
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.012718271283519655, tolerance: 0.0019633513098023334
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.008769262746156414, tolerance: 0.0019633513098023334
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordina

[ROIs → Performance] bootstrap: 200/1000 completed


...

[ROIs → Performance] bootstrap: 800/1000 completed


C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.0033870712020247495, tolerance: 0.0025294457473506657
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.005682618099050352, tolerance: 0.0025294457473506657
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.005796359732014089, tolerance: 0.0025294457473506657
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordin

[ROIs → Performance] bootstrap: 900/1000 completed


C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.015645997591631655, tolerance: 0.0029381398499505563
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.003654234400340073, tolerance: 0.0029381398499505563
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.03362862059336891, tolerance: 0.0029381398499505563
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\pkell\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinat

[ROIs → Performance] bootstrap: 1000/1000 completed

=== LASSO bootstrap stability — ROIs → Performance (B=1000) ===
        ROI  selection_freq  beta_median  beta_CI_lo  beta_CI_hi
   Left TPJ          0.9780       0.4882      0.0246      0.8862
 Left VLPFC          0.9240      -0.5229     -1.1500      0.0000
Right VLPFC          0.8670       0.4042      0.0000      1.0918
   Left OFC          0.8180       0.1832     -0.0727      0.7215
  Right TPJ          0.7300      -0.1847     -0.7724      0.0000
   Left STS          0.7100       0.1762     -0.0772      0.9233
 Left DLPFC          0.6870      -0.1331     -0.8600      0.2380
  Right OFC          0.6550       0.1718     -0.0185      0.9742
 Left RLPFC          0.5840      -0.0137     -0.8595      0.1389
  Right STS          0.5390       0.0000     -0.4510      0.4017
Right RLPFC          0.4380       0.0000     -0.7957      0.2150
Right DLPFC          0.4160       0.0000     -0.5182      0.6778


# Network Balance Index Analysis

In [105]:
# ============================================================
# CELL 1: Extract + merge performance, survey, and ROI ISC data
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# ----------------------------
# Output folder
# ----------------------------
analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
os.makedirs(analysis_out, exist_ok=True)

# ----------------------------
# Check required objects
# ----------------------------
required_objects = ["mean_corr_df", "survey_scores", "roi_isc_df"]
missing = [obj for obj in required_objects if obj not in globals()]
if missing:
    raise ValueError(f"Missing required objects from prior pipeline: {missing}")

# ----------------------------
# Clean subject IDs
# ----------------------------
def clean_subject_id(x):
    return str(x).strip()

perf_df = mean_corr_df.copy()
survey_df = survey_scores.copy()
isc_df = roi_isc_df.copy()

perf_df["Subject"] = perf_df["Subject"].apply(clean_subject_id)
survey_df["Subject"] = survey_df["Subject"].apply(clean_subject_id)
isc_df["Subject"] = isc_df["Subject"].apply(clean_subject_id)

# ----------------------------
# Make/confirm performance score
# ----------------------------
if "Performance" not in perf_df.columns:
    perf_df["Performance"] = perf_df[["Mean_Valence_Corr", "Mean_Arousal_Corr"]].mean(axis=1)

# ----------------------------
# Merge into one analysis dataframe
# ----------------------------
analysis_df = (
    isc_df
    .merge(perf_df[["Subject", "Mean_Valence_Corr", "Mean_Arousal_Corr", "Performance"]],
           on="Subject", how="inner")
    .merge(survey_df[["Subject", "AQ_Score", "EQ_Score", "EmQ_Score"]],
           on="Subject", how="inner")
)

roi_cols = [c for c in isc_df.columns if c != "Subject"]
outcome_cols = ["Performance", "Mean_Valence_Corr", "Mean_Arousal_Corr",
                "AQ_Score", "EQ_Score", "EmQ_Score"]

print("✅ analysis_df created")
print(f"N subjects in merged dataframe: {analysis_df.shape[0]}")
print(f"N ROIs: {len(roi_cols)}")
print("\nROI columns:")
print(roi_cols)
print("\nOutcome columns:")
print(outcome_cols)

display(analysis_df.head())

# Save merged data
analysis_df.to_csv(os.path.join(analysis_out, "merged_performance_survey_roiISC.csv"), index=False)




# ============================================================
# DROP-IN CELL: Network Balance Index predicting Performance
# Index = (Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# ----------------------------
# Check required objects
# ----------------------------
if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run Cell 1 first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

required_cols = ["Subject", "Right VLPFC", "Left TPJ", "Left VLPFC", "Right TPJ", "Performance"]
missing_cols = [c for c in required_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in analysis_df: {missing_cols}")

# ----------------------------
# Create network index
# ----------------------------
network_df = analysis_df[required_cols].dropna().copy()

network_df["Positive_Network_ISC"] = network_df["Right VLPFC"] + network_df["Left TPJ"]
network_df["Negative_Network_ISC"] = network_df["Left VLPFC"] + network_df["Right TPJ"]

network_df["Network_Balance_Index"] = (
    network_df["Positive_Network_ISC"] - network_df["Negative_Network_ISC"]
)

# Optional z-scored version for interpretability
network_df["Network_Balance_Index_z"] = (
    network_df["Network_Balance_Index"] - network_df["Network_Balance_Index"].mean()
) / network_df["Network_Balance_Index"].std(ddof=1)

# ----------------------------
# Correlations
# ----------------------------
x = network_df["Network_Balance_Index"].values
y = network_df["Performance"].values

r, p = pearsonr(x, y)
rho, p_s = spearmanr(x, y)

# Linear regression
slope, intercept, r_val, p_val, stderr = stats.linregress(x, y)
r2 = r ** 2

# ----------------------------
# Permutation test
# ----------------------------
N_PERM = 1000
RNG = np.random.default_rng(42)

perm_rs = np.empty(N_PERM)
for i in range(N_PERM):
    y_perm = RNG.permutation(y)
    perm_rs[i] = pearsonr(x, y_perm)[0]

p_perm = (1 + np.sum(np.abs(perm_rs) >= abs(r))) / (N_PERM + 1)

# ----------------------------
# Bootstrap 95% CI for Pearson r
# ----------------------------
N_BOOT = 1000
boot_rs = []

for i in range(N_BOOT):
    idx = RNG.choice(np.arange(len(network_df)), size=len(network_df), replace=True)
    xb = x[idx]
    yb = y[idx]
    if np.std(xb) > 0 and np.std(yb) > 0:
        boot_rs.append(pearsonr(xb, yb)[0])

boot_rs = np.array(boot_rs)
r_ci_low, r_ci_high = np.percentile(boot_rs, [2.5, 97.5])

# ----------------------------
# Regression line + 95% CI band
# ----------------------------
x_line = np.linspace(x.min(), x.max(), 200)
y_line = intercept + slope * x_line

n = len(network_df)
x_mean = np.mean(x)
s_err = np.sqrt(np.sum((y - (intercept + slope * x)) ** 2) / (n - 2))
t_val = stats.t.ppf(0.975, df=n - 2)

ci = t_val * s_err * np.sqrt(
    1 / n + (x_line - x_mean) ** 2 / np.sum((x - x_mean) ** 2)
)

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(6.8, 5.4))

plt.scatter(
    x, y,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

plt.plot(x_line, y_line, linewidth=2.5)
plt.fill_between(x_line, y_line - ci, y_line + ci, alpha=0.20)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel(
    "Network Balance Index\n(Right VLPFC + Left TPJ) − (Left VLPFC + Right TPJ)",
    fontsize=12
)
plt.ylabel("Emotion Tracking Performance", fontsize=13)

plt.title(
    "Network Balance Predicts Emotion Tracking Performance\n"
    f"r = {r:.3f}, 95% CI [{r_ci_low:.3f}, {r_ci_high:.3f}], "
    f"p = {p:.4f}, perm p = {p_perm:.4f}",
    fontsize=12
)

plt.tick_params(axis="both", labelsize=11)
plt.tight_layout()

out_fig = os.path.join(analysis_out, "network_balance_index_predicts_performance.png")
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------
# Save results
# ----------------------------
network_results = pd.DataFrame([{
    "N": n,
    "Network_Index": "(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)",
    "Pearson_r": r,
    "Pearson_p": p,
    "Permutation_p": p_perm,
    "Pearson_r_CI_low": r_ci_low,
    "Pearson_r_CI_high": r_ci_high,
    "Spearman_rho": rho,
    "Spearman_p": p_s,
    "Slope": slope,
    "Intercept": intercept,
    "R_squared": r2
}])

network_results.to_csv(
    os.path.join(analysis_out, "network_balance_index_results.csv"),
    index=False
)

network_df.to_csv(
    os.path.join(analysis_out, "network_balance_index_subject_values.csv"),
    index=False
)

print("=== Network Balance Index Results ===")
display(network_results.round(4))

print("\nSaved figure to:")
print(out_fig)

print("\nInterpretation:")
print("Higher values indicate relatively greater ISC in the positive-weighted network")
print("(Right VLPFC + Left TPJ) compared with the negative-weighted/opposing network")
print("(Left VLPFC + Right TPJ).")
print("A positive association with Performance supports the network-balance interpretation of the LASSO pattern.")




# ============================================================
# CELL 2 REPLACEMENT:
# Cross-validated FIXED Network Balance Architecture
#
# Fixed theory-derived index:
#   (Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)
#
# For each held-out subject:
# 1. Compute fixed network balance index for training subjects.
# 2. Fit regression: Performance ~ fixed network balance index
#    using training subjects only.
# 3. Compute fixed network balance index for held-out subject.
# 4. Predict held-out performance.
#
# This tests:
#   "Does THIS fixed architecture generalize?"
# rather than:
#   "Can LASSO rediscover the architecture each fold?"
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ----------------------------
# Required objects
# ----------------------------
if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run your merge/analysis_df cell first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

# ----------------------------
# Fixed architecture
# ----------------------------
positive_rois = ["Right VLPFC", "Left TPJ"]
negative_rois = ["Left VLPFC", "Right TPJ"]

required_cols = ["Subject", "Performance"] + positive_rois + negative_rois

missing_cols = [c for c in required_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in analysis_df: {missing_cols}")

# ----------------------------
# Prepare data
# ----------------------------
cv_df = analysis_df[required_cols].dropna().copy()

subjects_cv = cv_df["Subject"].values
y = cv_df["Performance"].values

# Fixed network balance index for all subjects
cv_df["Fixed_Network_Balance_Index"] = (
    cv_df[positive_rois].mean(axis=1) -
    cv_df[negative_rois].mean(axis=1)
)

x = cv_df["Fixed_Network_Balance_Index"].values

n = len(cv_df)

if n < 10:
    raise ValueError(f"Too few complete cases for LOOCV: N={n}")

loo = LeaveOneOut()

pred_fixed_balance = np.full(n, np.nan)
fold_rows = []

# ----------------------------
# Outer LOOCV
# ----------------------------
for fold, (train_idx, test_idx) in enumerate(loo.split(x), start=1):

    x_train = x[train_idx].reshape(-1, 1)
    x_test = x[test_idx].reshape(-1, 1)

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Fit training-only model
    if np.std(x_train) == 0:
        y_pred = np.mean(y_train)
        slope = np.nan
        intercept = np.mean(y_train)
    else:
        model = LinearRegression()
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)[0]
        slope = model.coef_[0]
        intercept = model.intercept_

    pred_fixed_balance[test_idx[0]] = y_pred

    fold_rows.append({
        "Fold": fold,
        "Heldout_Subject": subjects_cv[test_idx[0]],
        "Observed_Performance": y_test[0],
        "Predicted_Performance_Fixed_Balance": y_pred,
        "Heldout_Fixed_Network_Balance_Index": x[test_idx[0]],
        "Training_Slope": slope,
        "Training_Intercept": intercept,
        "Positive_ROIs": ", ".join(positive_rois),
        "Negative_ROIs": ", ".join(negative_rois)
    })

cv_fixed_fold_results = pd.DataFrame(fold_rows)

# ----------------------------
# Evaluate prediction
# ----------------------------
valid = np.isfinite(pred_fixed_balance)

y_valid = y[valid]
pred_valid = pred_fixed_balance[valid]
x_valid = x[valid]

r_pred, p_pred = pearsonr(y_valid, pred_valid)
rho_pred, p_s_pred = spearmanr(y_valid, pred_valid)

cv_r2 = 1 - (
    np.sum((y_valid - pred_valid) ** 2) /
    np.sum((y_valid - np.mean(y_valid)) ** 2)
)

rmse = np.sqrt(mean_squared_error(y_valid, pred_valid))

# In-sample association for reference only
r_index, p_index = pearsonr(x_valid, y_valid)
rho_index, p_s_index = spearmanr(x_valid, y_valid)

# ----------------------------
# Permutation test for CV prediction correlation
# ----------------------------
N_PERM = 10000
RNG = np.random.default_rng(42)

perm_rs = np.empty(N_PERM)

for i in range(N_PERM):
    y_perm = RNG.permutation(y_valid)
    perm_rs[i] = pearsonr(y_perm, pred_valid)[0]

p_perm_pred = (
    1 + np.sum(np.abs(perm_rs) >= abs(r_pred))
) / (N_PERM + 1)

# ----------------------------
# Save summary
# ----------------------------
cv_fixed_summary = pd.DataFrame([{
    "Model": "LOOCV_Fixed_Network_Balance_Index",
    "N": len(y_valid),
    "Fixed_Index": "(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)",
    "Positive_ROIs": ", ".join(positive_rois),
    "Negative_ROIs": ", ".join(negative_rois),
    "CV_Prediction_r": r_pred,
    "CV_Prediction_p": p_pred,
    "CV_Prediction_perm_p": p_perm_pred,
    "CV_Prediction_Spearman_rho": rho_pred,
    "CV_Prediction_Spearman_p": p_s_pred,
    "CV_R2": cv_r2,
    "RMSE": rmse,
    "InSample_Index_Performance_r_reference_only": r_index,
    "InSample_Index_Performance_p_reference_only": p_index,
    "InSample_Index_Performance_Spearman_reference_only": rho_index,
    "InSample_Index_Performance_Spearman_p_reference_only": p_s_index
}])

cv_fixed_fold_results.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_network_balance_fold_results.csv"
    ),
    index=False
)

cv_fixed_summary.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_network_balance_summary.csv"
    ),
    index=False
)

# Save subject-level values
cv_fixed_subject_values = cv_df.copy()
cv_fixed_subject_values["Predicted_Performance_Fixed_Balance_LOOCV"] = pred_fixed_balance

cv_fixed_subject_values.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_network_balance_subject_values.csv"
    ),
    index=False
)

print("=== LOOCV Fixed Network Balance Summary ===")
display(cv_fixed_summary.round(4))

print("\n=== Fold-level Predictions ===")
display(cv_fixed_fold_results.round(4))

# ----------------------------
# Plot 1: Observed vs predicted performance
# ----------------------------
plt.figure(figsize=(6.5, 5.5))

plt.scatter(
    y_valid,
    pred_valid,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

min_val = min(np.min(y_valid), np.min(pred_valid))
max_val = max(np.max(y_valid), np.max(pred_valid))

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)

fit = np.polyfit(y_valid, pred_valid, 1)
x_line = np.linspace(min_val, max_val, 200)

plt.plot(
    x_line,
    fit[0] * x_line + fit[1],
    linewidth=2.5,
    label="Observed fit"
)

plt.xlabel("Observed Emotion Tracking Performance")
plt.ylabel("Predicted Performance")
plt.title(
    "LOOCV Network Balance Prediction\n"
    f"r = {r_pred:.3f}, p = {p_pred:.4f}, perm p = {p_perm_pred:.4f}, "
    f"CV $R^2$ = {cv_r2:.3f}"
)

plt.legend(frameon=False)
plt.tight_layout()

out_fig_pred = os.path.join(
    analysis_out,
    "loocv_fixed_network_balance_prediction.png"
)

plt.savefig(out_fig_pred, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------
# Plot 2: Fixed network balance index vs performance
# ----------------------------
plt.figure(figsize=(6.6, 5.3))

plt.scatter(
    x_valid,
    y_valid,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

slope, intercept = np.polyfit(x_valid, y_valid, 1)
x_line = np.linspace(x_valid.min(), x_valid.max(), 200)
y_line = slope * x_line + intercept

plt.plot(x_line, y_line, linewidth=2.5)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel(
    "Fixed Network Balance Index\n"
    "(Right VLPFC + Left TPJ) − (Left VLPFC + Right TPJ)",
    fontsize=12
)
plt.ylabel("Emotion Tracking Performance", fontsize=13)

plt.title(
    "Fixed Network Balance Index vs Performance\n"
    f"in-sample r = {r_index:.3f}, p = {p_index:.4f}",
    fontsize=12
)

plt.tight_layout()

out_fig_index = os.path.join(
    analysis_out,
    "fixed_network_balance_index_vs_performance_reference.png"
)

plt.savefig(out_fig_index, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nSaved LOOCV prediction figure to: {out_fig_pred}")
print(f"Saved fixed index reference figure to: {out_fig_index}")

print("\nInterpretation:")
print("This analysis keeps the network architecture fixed in every fold:")
print("(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ).")
print("Only the regression slope/intercept are fit on training subjects.")
print("This tests whether the fixed theory-derived architecture predicts held-out performance.")

✅ analysis_df created
N subjects in merged dataframe: 38
N ROIs: 12

ROI columns:
['Left RLPFC', 'Left VLPFC', 'Left DLPFC', 'Left OFC', 'Left STS', 'Left TPJ', 'Right RLPFC', 'Right DLPFC', 'Right VLPFC', 'Right OFC', 'Right STS', 'Right TPJ']

Outcome columns:
['Performance', 'Mean_Valence_Corr', 'Mean_Arousal_Corr', 'AQ_Score', 'EQ_Score', 'EmQ_Score']


,Subject,Left RLPFC,Left VLPFC,Left DLPFC,Left OFC,Left STS,Left TPJ,Right RLPFC,Right DLPFC,Right VLPFC,Right OFC,Right STS,Right TPJ,Mean_Valence_Corr,Mean_Arousal_Corr,Performance,AQ_Score,EQ_Score,EmQ_Score
0,pid001,0.455522,0.209375,0.292943,0.352771,0.157022,0.219554,0.416631,0.368396,0.205772,0.520390,0.291673,0.197674,0.691985,0.403486,0.547736,18,103,19
1,pid002,0.161301,0.117563,0.228170,0.005103,0.058457,0.157235,0.137375,0.196237,0.174357,0.243339,0.202705,0.172520,0.633256,0.440374,0.536815,12,140,17
2,pid003,0.421383,0.283026,0.310581,0.220362,0.269815,0.199935,0.376236,0.447130,0.136997,0.299644,0.457948,0.242583,0.682913,0.359102,0.521007,29,113,29
3,pid004,0.321917,0.210125,0.245984,0.233840,0.001598,0.123213,0.392191,0.340093,0.410646,0.391429,0.311086,0.107033,0.740179,0.649422,0.694801,15,116,20
4,pid005,0.088688,0.068970,0.095113,-0.056914,0.057069,0.031518,0.188479,0.189784,0.182232,-0.027483,0.035459,0.156466,0.664789,0.475595,0.570192,21,136,22


=== Network Balance Index Results ===


,N,Network_Index,Pearson_r,Pearson_p,Permutation_p,Pearson_r_CI_low,Pearson_r_CI_high,Spearman_rho,Spearman_p,Slope,Intercept,R_squared
0,34,(Right VLPFC + Left TPJ) - (Left VLPFC + Right...,0.4757,0.0045,0.005,0.1923,0.6791,0.4341,0.0103,0.2928,0.5989,0.2262



Saved figure to:
C:\Users\pkell\vss_followup_roi_analyses\network_balance_index_predicts_performance.png

Interpretation:
Higher values indicate relatively greater ISC in the positive-weighted network
(Right VLPFC + Left TPJ) compared with the negative-weighted/opposing network
(Left VLPFC + Right TPJ).
A positive association with Performance supports the network-balance interpretation of the LASSO pattern.
=== LOOCV Fixed Network Balance Summary ===


,Model,N,Fixed_Index,Positive_ROIs,Negative_ROIs,CV_Prediction_r,CV_Prediction_p,CV_Prediction_perm_p,CV_Prediction_Spearman_rho,CV_Prediction_Spearman_p,CV_R2,RMSE,InSample_Index_Performance_r_reference_only,InSample_Index_Performance_p_reference_only,InSample_Index_Performance_Spearman_reference_only,InSample_Index_Performance_Spearman_p_reference_only
0,LOOCV_Fixed_Network_Balance_Index,34,(Right VLPFC + Left TPJ) - (Left VLPFC + Right...,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ",0.3667,0.0329,0.033,0.3338,0.0537,0.1218,0.0759,0.4757,0.0045,0.4341,0.0103



=== Fold-level Predictions ===


,Fold,Heldout_Subject,Observed_Performance,Predicted_Performance_Fixed_Balance,Heldout_Fixed_Network_Balance_Index,Training_Slope,Training_Intercept,Positive_ROIs,Negative_ROIs
0,1,pid001,0.5477,0.6059,0.0091,0.5839,0.6006,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
1,2,pid002,0.5368,0.6133,0.0208,0.5893,0.6011,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
2,3,pid003,0.5210,0.5464,-0.0943,0.5670,0.5999,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
3,4,pid004,0.6948,0.6591,0.1084,0.5627,0.5981,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
4,5,pid005,0.5702,0.5963,-0.0058,0.5822,0.5997,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
5,6,pid006,0.5659,0.5530,-0.0765,0.5935,0.5984,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
6,7,pid007,0.6839,0.6257,0.0496,0.5715,0.5974,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
7,8,pid008,0.6724,0.5638,-0.0494,0.6322,0.5950,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
8,9,pid009,0.6735,0.6907,0.1520,0.6019,0.5992,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
9,10,pid010,0.6543,0.6306,0.0558,0.5789,0.5983,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"



Saved LOOCV prediction figure to: C:\Users\pkell\vss_followup_roi_analyses\loocv_fixed_network_balance_prediction.png
Saved fixed index reference figure to: C:\Users\pkell\vss_followup_roi_analyses\fixed_network_balance_index_vs_performance_reference.png

Interpretation:
This analysis keeps the network architecture fixed in every fold:
(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ).
Only the regression slope/intercept are fit on training subjects.
This tests whether the fixed theory-derived architecture predicts held-out performance.


In [106]:
# ============================================================
#  Network Balance Index predicting Performance
# Index = (Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# ----------------------------
# Check required objects
# ----------------------------
if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run Cell 1 first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

required_cols = ["Subject", "Right VLPFC", "Left TPJ", "Left VLPFC", "Right TPJ", "Performance"]
missing_cols = [c for c in required_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in analysis_df: {missing_cols}")

# ----------------------------
# Create network index
# ----------------------------
network_df = analysis_df[required_cols].dropna().copy()

network_df["Positive_Network_ISC"] = network_df["Right VLPFC"] + network_df["Left TPJ"]
network_df["Negative_Network_ISC"] = network_df["Left VLPFC"] + network_df["Right TPJ"]

network_df["Network_Balance_Index"] = (
    network_df["Positive_Network_ISC"] - network_df["Negative_Network_ISC"]
)

# Optional z-scored version for interpretability
network_df["Network_Balance_Index_z"] = (
    network_df["Network_Balance_Index"] - network_df["Network_Balance_Index"].mean()
) / network_df["Network_Balance_Index"].std(ddof=1)

# ----------------------------
# Correlations
# ----------------------------
x = network_df["Network_Balance_Index"].values
y = network_df["Performance"].values

r, p = pearsonr(x, y)
rho, p_s = spearmanr(x, y)

# Linear regression
slope, intercept, r_val, p_val, stderr = stats.linregress(x, y)
r2 = r ** 2

# ----------------------------
# Permutation test
# ----------------------------
N_PERM = 1000
RNG = np.random.default_rng(42)

perm_rs = np.empty(N_PERM)
for i in range(N_PERM):
    y_perm = RNG.permutation(y)
    perm_rs[i] = pearsonr(x, y_perm)[0]

p_perm = (1 + np.sum(np.abs(perm_rs) >= abs(r))) / (N_PERM + 1)

# ----------------------------
# Bootstrap 95% CI for Pearson r
# ----------------------------
N_BOOT = 1000
boot_rs = []

for i in range(N_BOOT):
    idx = RNG.choice(np.arange(len(network_df)), size=len(network_df), replace=True)
    xb = x[idx]
    yb = y[idx]
    if np.std(xb) > 0 and np.std(yb) > 0:
        boot_rs.append(pearsonr(xb, yb)[0])

boot_rs = np.array(boot_rs)
r_ci_low, r_ci_high = np.percentile(boot_rs, [2.5, 97.5])

# ----------------------------
# Regression line + 95% CI band
# ----------------------------
x_line = np.linspace(x.min(), x.max(), 200)
y_line = intercept + slope * x_line

n = len(network_df)
x_mean = np.mean(x)
s_err = np.sqrt(np.sum((y - (intercept + slope * x)) ** 2) / (n - 2))
t_val = stats.t.ppf(0.975, df=n - 2)

ci = t_val * s_err * np.sqrt(
    1 / n + (x_line - x_mean) ** 2 / np.sum((x - x_mean) ** 2)
)

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(6.8, 5.4))

plt.scatter(
    x, y,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

plt.plot(x_line, y_line, linewidth=2.5)
plt.fill_between(x_line, y_line - ci, y_line + ci, alpha=0.20)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel(
    "Network Balance Index\n(Right VLPFC + Left TPJ) − (Left VLPFC + Right TPJ)",
    fontsize=12
)
plt.ylabel("Emotion Tracking Performance", fontsize=13)

plt.title(
    "Network Balance Predicts Emotion Tracking Performance\n"
    f"r = {r:.3f}, 95% CI [{r_ci_low:.3f}, {r_ci_high:.3f}], "
    f"p = {p:.4f}, perm p = {p_perm:.4f}",
    fontsize=12
)

plt.tick_params(axis="both", labelsize=11)
plt.tight_layout()

out_fig = os.path.join(analysis_out, "network_balance_index_predicts_performance.png")
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------
# Save results
# ----------------------------
network_results = pd.DataFrame([{
    "N": n,
    "Network_Index": "(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)",
    "Pearson_r": r,
    "Pearson_p": p,
    "Permutation_p": p_perm,
    "Pearson_r_CI_low": r_ci_low,
    "Pearson_r_CI_high": r_ci_high,
    "Spearman_rho": rho,
    "Spearman_p": p_s,
    "Slope": slope,
    "Intercept": intercept,
    "R_squared": r2
}])

network_results.to_csv(
    os.path.join(analysis_out, "network_balance_index_results.csv"),
    index=False
)

network_df.to_csv(
    os.path.join(analysis_out, "network_balance_index_subject_values.csv"),
    index=False
)

print("=== Network Balance Index Results ===")
display(network_results.round(4))

print("\nSaved figure to:")
print(out_fig)

print("\nInterpretation:")
print("Higher values indicate relatively greater ISC in the positive-weighted network")
print("(Right VLPFC + Left TPJ) compared with the negative-weighted/opposing network")
print("(Left VLPFC + Right TPJ).")
print("A positive association with Performance supports the network-balance interpretation of the LASSO pattern.")

=== Network Balance Index Results ===


,N,Network_Index,Pearson_r,Pearson_p,Permutation_p,Pearson_r_CI_low,Pearson_r_CI_high,Spearman_rho,Spearman_p,Slope,Intercept,R_squared
0,34,(Right VLPFC + Left TPJ) - (Left VLPFC + Right...,0.4757,0.0045,0.005,0.1923,0.6791,0.4341,0.0103,0.2928,0.5989,0.2262



Saved figure to:
C:\Users\pkell\vss_followup_roi_analyses\network_balance_index_predicts_performance.png

Interpretation:
Higher values indicate relatively greater ISC in the positive-weighted network
(Right VLPFC + Left TPJ) compared with the negative-weighted/opposing network
(Left VLPFC + Right TPJ).
A positive association with Performance supports the network-balance interpretation of the LASSO pattern.


In [107]:
# ============================================================
# Correlate Network Balance Index with Surveys
# AQ / EQ / EmQ
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------
# Check required objects
# ------------------------------------------------
if "network_df" not in globals():
    raise ValueError("network_df not found. Run the Network Balance Index cell first.")

if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run Cell 1 first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

# ------------------------------------------------
# Merge survey scores into network_df
# ------------------------------------------------
survey_cols = ["AQ_Score", "EQ_Score", "EmQ_Score"]

missing_cols = [c for c in survey_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing survey columns in analysis_df: {missing_cols}")

survey_merge = analysis_df[["Subject"] + survey_cols].drop_duplicates()

network_survey_df = pd.merge(
    network_df,
    survey_merge,
    on="Subject",
    how="left"
)

# ------------------------------------------------
# Run correlations
# ------------------------------------------------
results = []

N_PERM = 10000
N_BOOT = 5000
RNG = np.random.default_rng(42)

for survey in survey_cols:

    tmp = network_survey_df[
        ["Network_Balance_Index", survey]
    ].dropna()

    if len(tmp) < 5:
        continue

    x = tmp["Network_Balance_Index"].values
    y = tmp[survey].values

    # --------------------------------------------
    # Pearson + Spearman
    # --------------------------------------------
    r, p = pearsonr(x, y)
    rho, p_s = spearmanr(x, y)

    # --------------------------------------------
    # Regression
    # --------------------------------------------
    slope, intercept, r_val, p_val, stderr = stats.linregress(x, y)

    # --------------------------------------------
    # Permutation test
    # --------------------------------------------
    perm_rs = np.empty(N_PERM)

    for i in range(N_PERM):
        y_perm = RNG.permutation(y)
        perm_rs[i] = pearsonr(x, y_perm)[0]

    p_perm = (
        1 + np.sum(np.abs(perm_rs) >= abs(r))
    ) / (N_PERM + 1)

    # --------------------------------------------
    # Bootstrap CI
    # --------------------------------------------
    boot_rs = []

    for i in range(N_BOOT):

        idx = RNG.choice(
            np.arange(len(tmp)),
            size=len(tmp),
            replace=True
        )

        xb = x[idx]
        yb = y[idx]

        if np.std(xb) > 0 and np.std(yb) > 0:
            boot_rs.append(pearsonr(xb, yb)[0])

    boot_rs = np.array(boot_rs)

    ci_low, ci_high = np.percentile(boot_rs, [2.5, 97.5])

    # --------------------------------------------
    # Save results
    # --------------------------------------------
    results.append({
        "Survey": survey,
        "N": len(tmp),
        "Pearson_r": r,
        "Pearson_p": p,
        "Permutation_p": p_perm,
        "Pearson_CI_low": ci_low,
        "Pearson_CI_high": ci_high,
        "Spearman_rho": rho,
        "Spearman_p": p_s,
        "Slope": slope,
        "Intercept": intercept,
        "R_squared": r**2
    })

    # =========================================================
    # Plot
    # =========================================================
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = intercept + slope * x_line

    n = len(tmp)
    x_mean = np.mean(x)

    s_err = np.sqrt(
        np.sum((y - (intercept + slope * x)) ** 2) / (n - 2)
    )

    t_val = stats.t.ppf(0.975, df=n - 2)

    ci = t_val * s_err * np.sqrt(
        1 / n +
        (x_line - x_mean) ** 2 /
        np.sum((x - x_mean) ** 2)
    )

    plt.figure(figsize=(6.6, 5.3))

    plt.scatter(
        x,
        y,
        s=75,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.8
    )

    plt.plot(x_line, y_line, linewidth=2.5)

    plt.fill_between(
        x_line,
        y_line - ci,
        y_line + ci,
        alpha=0.20
    )

    plt.axhline(
        np.mean(y),
        linestyle="--",
        linewidth=1
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1
    )

    plt.xlabel(
        "Network Balance Index\n"
        "(Right VLPFC + Left TPJ) − (Left VLPFC + Right TPJ)",
        fontsize=12
    )

    plt.ylabel(survey.replace("_", " "), fontsize=13)

    plt.title(
        f"Network Balance Index vs {survey.replace('_', ' ')}\n"
        f"r = {r:.3f}, "
        f"95% CI [{ci_low:.3f}, {ci_high:.3f}], "
        f"p = {p:.4f}, "
        f"perm p = {p_perm:.4f}",
        fontsize=12
    )

    plt.tick_params(axis="both", labelsize=11)

    plt.tight_layout()

    out_fig = os.path.join(
        analysis_out,
        f"network_balance_vs_{survey}.png"
    )

    plt.savefig(
        out_fig,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# ------------------------------------------------
# Results dataframe
# ------------------------------------------------
network_survey_results = pd.DataFrame(results)

# FDR correction
network_survey_results["Pearson_p_FDR"] = multipletests(
    network_survey_results["Pearson_p"],
    method="fdr_bh"
)[1]

network_survey_results["Permutation_p_FDR"] = multipletests(
    network_survey_results["Permutation_p"],
    method="fdr_bh"
)[1]

network_survey_results["Spearman_p_FDR"] = multipletests(
    network_survey_results["Spearman_p"],
    method="fdr_bh"
)[1]

# ------------------------------------------------
# Save
# ------------------------------------------------
network_survey_results.to_csv(
    os.path.join(
        analysis_out,
        "network_balance_index_vs_surveys.csv"
    ),
    index=False
)

# ------------------------------------------------
# Display
# ------------------------------------------------
print("=== Network Balance Index vs Survey Scores ===")
display(network_survey_results.round(4))

print("\nInterpretation:")
print(
    "Positive correlations indicate that individuals with relatively greater "
    "Right VLPFC + Left TPJ synchrony (compared with Left VLPFC + Right TPJ) "
    "also tend to score higher on the corresponding survey measure."
)

print(
    "\nNegative correlations indicate that greater weighting toward the "
    "Left VLPFC + Right TPJ network is associated with higher survey scores."
)

=== Network Balance Index vs Survey Scores ===


,Survey,N,Pearson_r,Pearson_p,Permutation_p,Pearson_CI_low,Pearson_CI_high,Spearman_rho,Spearman_p,Slope,Intercept,R_squared,Pearson_p_FDR,Permutation_p_FDR,Spearman_p_FDR
0,AQ_Score,34,-0.2158,0.2202,0.2163,-0.5034,0.1076,-0.2175,0.2165,-9.8813,16.7411,0.0466,0.6038,0.6040,0.6111
1,EQ_Score,34,0.0555,0.7553,0.7598,-0.3024,0.4315,0.0904,0.6111,5.5940,132.7292,0.0031,0.7553,0.7598,0.6111
2,EmQ_Score,34,0.1483,0.4025,0.4027,-0.2462,0.4986,0.1141,0.5204,6.8881,16.2820,0.0220,0.6038,0.6040,0.6111



Interpretation:
Positive correlations indicate that individuals with relatively greater Right VLPFC + Left TPJ synchrony (compared with Left VLPFC + Right TPJ) also tend to score higher on the corresponding survey measure.

Negative correlations indicate that greater weighting toward the Left VLPFC + Right TPJ network is associated with higher survey scores.


In [108]:
# ============================================================
# 
# Cross-validated FIXED Network Balance Architecture
#
# Fixed theory-derived index:
#   (Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)
#
# For each held-out subject:
# 1. Compute fixed network balance index for training subjects.
# 2. Fit regression: Performance ~ fixed network balance index
#    using training subjects only.
# 3. Compute fixed network balance index for held-out subject.
# 4. Predict held-out performance.
#
# This tests:
#   "Does THIS fixed architecture generalize?"
# rather than:
#   "Can LASSO rediscover the architecture each fold?"
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ----------------------------
# Required objects
# ----------------------------
if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run your merge/analysis_df cell first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

# ----------------------------
# Fixed architecture
# ----------------------------
positive_rois = ["Right VLPFC", "Left TPJ"]
negative_rois = ["Left VLPFC", "Right TPJ"]

required_cols = ["Subject", "Performance"] + positive_rois + negative_rois

missing_cols = [c for c in required_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in analysis_df: {missing_cols}")

# ----------------------------
# Prepare data
# ----------------------------
cv_df = analysis_df[required_cols].dropna().copy()

subjects_cv = cv_df["Subject"].values
y = cv_df["Performance"].values

# Fixed network balance index for all subjects
cv_df["Fixed_Network_Balance_Index"] = (
    cv_df[positive_rois].mean(axis=1) -
    cv_df[negative_rois].mean(axis=1)
)

x = cv_df["Fixed_Network_Balance_Index"].values

n = len(cv_df)

if n < 10:
    raise ValueError(f"Too few complete cases for LOOCV: N={n}")

loo = LeaveOneOut()

pred_fixed_balance = np.full(n, np.nan)
fold_rows = []

# ----------------------------
# Outer LOOCV
# ----------------------------
for fold, (train_idx, test_idx) in enumerate(loo.split(x), start=1):

    x_train = x[train_idx].reshape(-1, 1)
    x_test = x[test_idx].reshape(-1, 1)

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Fit training-only model
    if np.std(x_train) == 0:
        y_pred = np.mean(y_train)
        slope = np.nan
        intercept = np.mean(y_train)
    else:
        model = LinearRegression()
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)[0]
        slope = model.coef_[0]
        intercept = model.intercept_

    pred_fixed_balance[test_idx[0]] = y_pred

    fold_rows.append({
        "Fold": fold,
        "Heldout_Subject": subjects_cv[test_idx[0]],
        "Observed_Performance": y_test[0],
        "Predicted_Performance_Fixed_Balance": y_pred,
        "Heldout_Fixed_Network_Balance_Index": x[test_idx[0]],
        "Training_Slope": slope,
        "Training_Intercept": intercept,
        "Positive_ROIs": ", ".join(positive_rois),
        "Negative_ROIs": ", ".join(negative_rois)
    })

cv_fixed_fold_results = pd.DataFrame(fold_rows)

# ----------------------------
# Evaluate prediction
# ----------------------------
valid = np.isfinite(pred_fixed_balance)

y_valid = y[valid]
pred_valid = pred_fixed_balance[valid]
x_valid = x[valid]

r_pred, p_pred = pearsonr(y_valid, pred_valid)
rho_pred, p_s_pred = spearmanr(y_valid, pred_valid)

cv_r2 = 1 - (
    np.sum((y_valid - pred_valid) ** 2) /
    np.sum((y_valid - np.mean(y_valid)) ** 2)
)

rmse = np.sqrt(mean_squared_error(y_valid, pred_valid))

# In-sample association for reference only
r_index, p_index = pearsonr(x_valid, y_valid)
rho_index, p_s_index = spearmanr(x_valid, y_valid)

# ----------------------------
# Permutation test for CV prediction correlation
# ----------------------------
N_PERM = 10000
RNG = np.random.default_rng(42)

perm_rs = np.empty(N_PERM)

for i in range(N_PERM):
    y_perm = RNG.permutation(y_valid)
    perm_rs[i] = pearsonr(y_perm, pred_valid)[0]

p_perm_pred = (
    1 + np.sum(np.abs(perm_rs) >= abs(r_pred))
) / (N_PERM + 1)

# ----------------------------
# Save summary
# ----------------------------
cv_fixed_summary = pd.DataFrame([{
    "Model": "LOOCV_Fixed_Network_Balance_Index",
    "N": len(y_valid),
    "Fixed_Index": "(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ)",
    "Positive_ROIs": ", ".join(positive_rois),
    "Negative_ROIs": ", ".join(negative_rois),
    "CV_Prediction_r": r_pred,
    "CV_Prediction_p": p_pred,
    "CV_Prediction_perm_p": p_perm_pred,
    "CV_Prediction_Spearman_rho": rho_pred,
    "CV_Prediction_Spearman_p": p_s_pred,
    "CV_R2": cv_r2,
    "RMSE": rmse,
    "InSample_Index_Performance_r_reference_only": r_index,
    "InSample_Index_Performance_p_reference_only": p_index,
    "InSample_Index_Performance_Spearman_reference_only": rho_index,
    "InSample_Index_Performance_Spearman_p_reference_only": p_s_index
}])

cv_fixed_fold_results.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_network_balance_fold_results.csv"
    ),
    index=False
)

cv_fixed_summary.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_network_balance_summary.csv"
    ),
    index=False
)

# Save subject-level values
cv_fixed_subject_values = cv_df.copy()
cv_fixed_subject_values["Predicted_Performance_Fixed_Balance_LOOCV"] = pred_fixed_balance

cv_fixed_subject_values.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_network_balance_subject_values.csv"
    ),
    index=False
)

print("=== LOOCV Fixed Network Balance Summary ===")
display(cv_fixed_summary.round(4))

print("\n=== Fold-level Predictions ===")
display(cv_fixed_fold_results.round(4))

# ----------------------------
# Plot 1: Observed vs predicted performance
# ----------------------------
plt.figure(figsize=(6.5, 5.5))

plt.scatter(
    y_valid,
    pred_valid,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

min_val = min(np.min(y_valid), np.min(pred_valid))
max_val = max(np.max(y_valid), np.max(pred_valid))

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)

fit = np.polyfit(y_valid, pred_valid, 1)
x_line = np.linspace(min_val, max_val, 200)

plt.plot(
    x_line,
    fit[0] * x_line + fit[1],
    linewidth=2.5,
    label="Observed fit"
)

plt.xlabel("Observed Emotion Tracking Performance")
plt.ylabel("Predicted Performance")
plt.title(
    "LOOCV Network Balance Prediction\n"
    f"r = {r_pred:.3f}, p = {p_pred:.4f}, perm p = {p_perm_pred:.4f}, "
    f"CV $R^2$ = {cv_r2:.3f}"
)

plt.legend(frameon=False)
plt.tight_layout()

out_fig_pred = os.path.join(
    analysis_out,
    "loocv_fixed_network_balance_prediction.png"
)

plt.savefig(out_fig_pred, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------
# Plot 2: Fixed network balance index vs performance
# ----------------------------
plt.figure(figsize=(6.6, 5.3))

plt.scatter(
    x_valid,
    y_valid,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

slope, intercept = np.polyfit(x_valid, y_valid, 1)
x_line = np.linspace(x_valid.min(), x_valid.max(), 200)
y_line = slope * x_line + intercept

plt.plot(x_line, y_line, linewidth=2.5)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel(
    "Fixed Network Balance Index\n"
    "(Right VLPFC + Left TPJ) − (Left VLPFC + Right TPJ)",
    fontsize=12
)
plt.ylabel("Emotion Tracking Performance", fontsize=13)

plt.title(
    "Fixed Network Balance Index vs Performance\n"
    f"in-sample r = {r_index:.3f}, p = {p_index:.4f}",
    fontsize=12
)

plt.tight_layout()

out_fig_index = os.path.join(
    analysis_out,
    "fixed_network_balance_index_vs_performance_reference.png"
)

plt.savefig(out_fig_index, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nSaved LOOCV prediction figure to: {out_fig_pred}")
print(f"Saved fixed index reference figure to: {out_fig_index}")

print("\nInterpretation:")
print("This analysis keeps the network architecture fixed in every fold:")
print("(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ).")
print("Only the regression slope/intercept are fit on training subjects.")
print("This tests whether the fixed theory-derived architecture predicts held-out performance.")

=== LOOCV Fixed Network Balance Summary ===


,Model,N,Fixed_Index,Positive_ROIs,Negative_ROIs,CV_Prediction_r,CV_Prediction_p,CV_Prediction_perm_p,CV_Prediction_Spearman_rho,CV_Prediction_Spearman_p,CV_R2,RMSE,InSample_Index_Performance_r_reference_only,InSample_Index_Performance_p_reference_only,InSample_Index_Performance_Spearman_reference_only,InSample_Index_Performance_Spearman_p_reference_only
0,LOOCV_Fixed_Network_Balance_Index,34,(Right VLPFC + Left TPJ) - (Left VLPFC + Right...,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ",0.3667,0.0329,0.033,0.3338,0.0537,0.1218,0.0759,0.4757,0.0045,0.4341,0.0103



=== Fold-level Predictions ===


,Fold,Heldout_Subject,Observed_Performance,Predicted_Performance_Fixed_Balance,Heldout_Fixed_Network_Balance_Index,Training_Slope,Training_Intercept,Positive_ROIs,Negative_ROIs
0,1,pid001,0.5477,0.6059,0.0091,0.5839,0.6006,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
1,2,pid002,0.5368,0.6133,0.0208,0.5893,0.6011,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
2,3,pid003,0.5210,0.5464,-0.0943,0.5670,0.5999,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
3,4,pid004,0.6948,0.6591,0.1084,0.5627,0.5981,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
4,5,pid005,0.5702,0.5963,-0.0058,0.5822,0.5997,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
5,6,pid006,0.5659,0.5530,-0.0765,0.5935,0.5984,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
6,7,pid007,0.6839,0.6257,0.0496,0.5715,0.5974,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
7,8,pid008,0.6724,0.5638,-0.0494,0.6322,0.5950,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
8,9,pid009,0.6735,0.6907,0.1520,0.6019,0.5992,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"
9,10,pid010,0.6543,0.6306,0.0558,0.5789,0.5983,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ"



Saved LOOCV prediction figure to: C:\Users\pkell\vss_followup_roi_analyses\loocv_fixed_network_balance_prediction.png
Saved fixed index reference figure to: C:\Users\pkell\vss_followup_roi_analyses\fixed_network_balance_index_vs_performance_reference.png

Interpretation:
This analysis keeps the network architecture fixed in every fold:
(Right VLPFC + Left TPJ) - (Left VLPFC + Right TPJ).
Only the regression slope/intercept are fit on training subjects.
This tests whether the fixed theory-derived architecture predicts held-out performance.


In [109]:
# ============================================================
# Full LASSO Network Balance Index predicting Performance
# Index = Positive LASSO ROIs - Negative LASSO ROIs
#
# Positive LASSO ROIs:
#   Left TPJ, Right VLPFC, Left OFC, Left STS
#
# Negative LASSO ROIs:
#   Left VLPFC, Left DLPFC, Right TPJ
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# ----------------------------
# Check required objects
# ----------------------------
if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run Cell 1 first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

# ----------------------------
# Define full non-zero LASSO ROI groups
# ----------------------------
positive_lasso_rois = [
    "Left TPJ",
    "Right VLPFC",
    "Left OFC",
    "Left STS"
]

negative_lasso_rois = [
    "Left VLPFC",
    "Left DLPFC",
    "Right TPJ"
]

required_cols = (
    ["Subject", "Performance"] +
    positive_lasso_rois +
    negative_lasso_rois
)

missing_cols = [c for c in required_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in analysis_df: {missing_cols}")

# ----------------------------
# Create network index
# ----------------------------
network_df = analysis_df[required_cols].dropna().copy()

network_df["Positive_LASSO_Network_ISC"] = network_df[positive_lasso_rois].mean(axis=1)
network_df["Negative_LASSO_Network_ISC"] = network_df[negative_lasso_rois].mean(axis=1)

network_df["Full_LASSO_Network_Balance_Index"] = (
    network_df["Positive_LASSO_Network_ISC"] -
    network_df["Negative_LASSO_Network_ISC"]
)

# Optional summed version, in case you want to compare with the original summed index
network_df["Positive_LASSO_Network_ISC_sum"] = network_df[positive_lasso_rois].sum(axis=1)
network_df["Negative_LASSO_Network_ISC_sum"] = network_df[negative_lasso_rois].sum(axis=1)

network_df["Full_LASSO_Network_Balance_Index_sum"] = (
    network_df["Positive_LASSO_Network_ISC_sum"] -
    network_df["Negative_LASSO_Network_ISC_sum"]
)

# Z-scored version for interpretability
network_df["Full_LASSO_Network_Balance_Index_z"] = (
    network_df["Full_LASSO_Network_Balance_Index"] -
    network_df["Full_LASSO_Network_Balance_Index"].mean()
) / network_df["Full_LASSO_Network_Balance_Index"].std(ddof=1)

# ----------------------------
# Correlations
# ----------------------------
x = network_df["Full_LASSO_Network_Balance_Index"].values
y = network_df["Performance"].values

r, p = pearsonr(x, y)
rho, p_s = spearmanr(x, y)

# Linear regression
slope, intercept, r_val, p_val, stderr = stats.linregress(x, y)
r2 = r ** 2

# ----------------------------
# Permutation test
# ----------------------------
N_PERM = 10000
RNG = np.random.default_rng(42)

perm_rs = np.empty(N_PERM)
for i in range(N_PERM):
    y_perm = RNG.permutation(y)
    perm_rs[i] = pearsonr(x, y_perm)[0]

p_perm = (1 + np.sum(np.abs(perm_rs) >= abs(r))) / (N_PERM + 1)

# ----------------------------
# Bootstrap 95% CI for Pearson r
# ----------------------------
N_BOOT = 10000
boot_rs = []

for i in range(N_BOOT):
    idx = RNG.choice(np.arange(len(network_df)), size=len(network_df), replace=True)
    xb = x[idx]
    yb = y[idx]

    if np.std(xb) > 0 and np.std(yb) > 0:
        boot_rs.append(pearsonr(xb, yb)[0])

boot_rs = np.array(boot_rs)
r_ci_low, r_ci_high = np.percentile(boot_rs, [2.5, 97.5])

# ----------------------------
# Regression line + 95% CI band
# ----------------------------
x_line = np.linspace(x.min(), x.max(), 200)
y_line = intercept + slope * x_line

n = len(network_df)
x_mean = np.mean(x)
s_err = np.sqrt(np.sum((y - (intercept + slope * x)) ** 2) / (n - 2))
t_val = stats.t.ppf(0.975, df=n - 2)

ci = t_val * s_err * np.sqrt(
    1 / n +
    (x_line - x_mean) ** 2 /
    np.sum((x - x_mean) ** 2)
)

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(7.2, 5.6))

plt.scatter(
    x, y,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

plt.plot(x_line, y_line, linewidth=2.5)
plt.fill_between(x_line, y_line - ci, y_line + ci, alpha=0.20)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel(
    "Full LASSO Network Balance Index\n"
    "(Positive-weight ROIs mean) − (Negative-weight ROIs mean)",
    fontsize=12
)

plt.ylabel("Emotion Tracking Performance", fontsize=13)

plt.title(
    "Full LASSO Network Balance Predicts Emotion Tracking Performance\n"
    f"r = {r:.3f}, 95% CI [{r_ci_low:.3f}, {r_ci_high:.3f}], "
    f"p = {p:.4f}, perm p = {p_perm:.4f}",
    fontsize=12
)

plt.tick_params(axis="both", labelsize=11)
plt.tight_layout()

out_fig = os.path.join(
    analysis_out,
    "full_lasso_network_balance_index_predicts_performance.png"
)

plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------
# Save results
# ----------------------------
network_results = pd.DataFrame([{
    "N": n,
    "Network_Index": "(Left TPJ + Right VLPFC + Left OFC + Left STS)/4 - (Left VLPFC + Left DLPFC + Right TPJ)/3",
    "Positive_LASSO_ROIs": ", ".join(positive_lasso_rois),
    "Negative_LASSO_ROIs": ", ".join(negative_lasso_rois),
    "Pearson_r": r,
    "Pearson_p": p,
    "Permutation_p": p_perm,
    "Pearson_r_CI_low": r_ci_low,
    "Pearson_r_CI_high": r_ci_high,
    "Spearman_rho": rho,
    "Spearman_p": p_s,
    "Slope": slope,
    "Intercept": intercept,
    "R_squared": r2
}])

network_results.to_csv(
    os.path.join(
        analysis_out,
        "full_lasso_network_balance_index_results.csv"
    ),
    index=False
)

network_df.to_csv(
    os.path.join(
        analysis_out,
        "full_lasso_network_balance_index_subject_values.csv"
    ),
    index=False
)

print("=== Full LASSO Network Balance Index Results ===")
display(network_results.round(4))

print("\nPositive LASSO ROIs:")
print(positive_lasso_rois)

print("\nNegative LASSO ROIs:")
print(negative_lasso_rois)

print("\nSaved figure to:")
print(out_fig)

print("\nInterpretation:")
print("Higher values indicate relatively greater ISC in the positive-weighted LASSO network")
print("(Left TPJ, Right VLPFC, Left OFC, Left STS) compared with the negative-weighted LASSO network")
print("(Left VLPFC, Left DLPFC, Right TPJ).")
print("A positive association with Performance supports the interpretation that LASSO captured a distributed network-balance effect.")

=== Full LASSO Network Balance Index Results ===


,N,Network_Index,Positive_LASSO_ROIs,Negative_LASSO_ROIs,Pearson_r,Pearson_p,Permutation_p,Pearson_r_CI_low,Pearson_r_CI_high,Spearman_rho,Spearman_p,Slope,Intercept,R_squared
0,33,(Left TPJ + Right VLPFC + Left OFC + Left STS)...,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ",0.5874,0.0003,0.0004,0.2654,0.7979,0.4843,0.0043,0.7976,0.6096,0.345



Positive LASSO ROIs:
['Left TPJ', 'Right VLPFC', 'Left OFC', 'Left STS']

Negative LASSO ROIs:
['Left VLPFC', 'Left DLPFC', 'Right TPJ']

Saved figure to:
C:\Users\pkell\vss_followup_roi_analyses\full_lasso_network_balance_index_predicts_performance.png

Interpretation:
Higher values indicate relatively greater ISC in the positive-weighted LASSO network
(Left TPJ, Right VLPFC, Left OFC, Left STS) compared with the negative-weighted LASSO network
(Left VLPFC, Left DLPFC, Right TPJ).
A positive association with Performance supports the interpretation that LASSO captured a distributed network-balance effect.


In [110]:
# ============================================================
# Correlate FULL LASSO Network Balance Index with Surveys
# AQ / EQ / EmQ
#
# Requires the full LASSO network balance cell to be run first.
# Uses:
#   Full_LASSO_Network_Balance_Index
#
# Positive LASSO ROIs:
#   Left TPJ, Right VLPFC, Left OFC, Left STS
#
# Negative LASSO ROIs:
#   Left VLPFC, Left DLPFC, Right TPJ
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------
# Check required objects
# ------------------------------------------------
if "network_df" not in globals():
    raise ValueError("network_df not found. Run the Full LASSO Network Balance Index cell first.")

if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run Cell 1 first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

index_col = "Full_LASSO_Network_Balance_Index"

if index_col not in network_df.columns:
    raise ValueError(
        f"{index_col} not found in network_df. "
        "Run the full LASSO network balance index cell first."
    )

# ------------------------------------------------
# Merge survey scores into network_df
# ------------------------------------------------
survey_cols = ["AQ_Score", "EQ_Score", "EmQ_Score"]

missing_cols = [c for c in survey_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing survey columns in analysis_df: {missing_cols}")

survey_merge = analysis_df[["Subject"] + survey_cols].drop_duplicates()

network_survey_df = pd.merge(
    network_df,
    survey_merge,
    on="Subject",
    how="left"
)

# ------------------------------------------------
# Run correlations
# ------------------------------------------------
results = []

N_PERM = 10000
N_BOOT = 5000
RNG = np.random.default_rng(42)

for survey in survey_cols:

    tmp = network_survey_df[
        [index_col, survey]
    ].dropna()

    if len(tmp) < 5:
        continue

    x = tmp[index_col].values
    y = tmp[survey].values

    # --------------------------------------------
    # Pearson + Spearman
    # --------------------------------------------
    r, p = pearsonr(x, y)
    rho, p_s = spearmanr(x, y)

    # --------------------------------------------
    # Regression
    # --------------------------------------------
    slope, intercept, r_val, p_val, stderr = stats.linregress(x, y)

    # --------------------------------------------
    # Permutation test
    # --------------------------------------------
    perm_rs = np.empty(N_PERM)

    for i in range(N_PERM):
        y_perm = RNG.permutation(y)
        perm_rs[i] = pearsonr(x, y_perm)[0]

    p_perm = (
        1 + np.sum(np.abs(perm_rs) >= abs(r))
    ) / (N_PERM + 1)

    # --------------------------------------------
    # Bootstrap CI
    # --------------------------------------------
    boot_rs = []

    for i in range(N_BOOT):

        idx = RNG.choice(
            np.arange(len(tmp)),
            size=len(tmp),
            replace=True
        )

        xb = x[idx]
        yb = y[idx]

        if np.std(xb) > 0 and np.std(yb) > 0:
            boot_rs.append(pearsonr(xb, yb)[0])

    boot_rs = np.array(boot_rs)

    ci_low, ci_high = np.percentile(boot_rs, [2.5, 97.5])

    # --------------------------------------------
    # Save results
    # --------------------------------------------
    results.append({
        "Survey": survey,
        "Network_Index": index_col,
        "N": len(tmp),
        "Pearson_r": r,
        "Pearson_p": p,
        "Permutation_p": p_perm,
        "Pearson_CI_low": ci_low,
        "Pearson_CI_high": ci_high,
        "Spearman_rho": rho,
        "Spearman_p": p_s,
        "Slope": slope,
        "Intercept": intercept,
        "R_squared": r**2
    })

    # =========================================================
    # Plot
    # =========================================================
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = intercept + slope * x_line

    n = len(tmp)
    x_mean = np.mean(x)

    s_err = np.sqrt(
        np.sum((y - (intercept + slope * x)) ** 2) / (n - 2)
    )

    t_val = stats.t.ppf(0.975, df=n - 2)

    ci = t_val * s_err * np.sqrt(
        1 / n +
        (x_line - x_mean) ** 2 /
        np.sum((x - x_mean) ** 2)
    )

    plt.figure(figsize=(6.8, 5.4))

    plt.scatter(
        x,
        y,
        s=75,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.8
    )

    plt.plot(x_line, y_line, linewidth=2.5)

    plt.fill_between(
        x_line,
        y_line - ci,
        y_line + ci,
        alpha=0.20
    )

    plt.axhline(
        np.mean(y),
        linestyle="--",
        linewidth=1
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1
    )

    plt.xlabel(
        "Full LASSO Network Balance Index\n"
        "(Positive-weight ROI mean) − (Negative-weight ROI mean)",
        fontsize=12
    )

    plt.ylabel(survey.replace("_", " "), fontsize=13)

    plt.title(
        f"Full LASSO Network Balance vs {survey.replace('_', ' ')}\n"
        f"r = {r:.3f}, "
        f"95% CI [{ci_low:.3f}, {ci_high:.3f}], "
        f"p = {p:.4f}, "
        f"perm p = {p_perm:.4f}",
        fontsize=12
    )

    plt.tick_params(axis="both", labelsize=11)
    plt.tight_layout()

    out_fig = os.path.join(
        analysis_out,
        f"full_lasso_network_balance_vs_{survey}.png"
    )

    plt.savefig(
        out_fig,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# ------------------------------------------------
# Results dataframe
# ------------------------------------------------
network_survey_results = pd.DataFrame(results)

if network_survey_results.empty:
    raise RuntimeError("No survey correlations were computed. Check missing data / N.")

# FDR correction
network_survey_results["Pearson_p_FDR"] = multipletests(
    network_survey_results["Pearson_p"],
    method="fdr_bh"
)[1]

network_survey_results["Permutation_p_FDR"] = multipletests(
    network_survey_results["Permutation_p"],
    method="fdr_bh"
)[1]

network_survey_results["Spearman_p_FDR"] = multipletests(
    network_survey_results["Spearman_p"],
    method="fdr_bh"
)[1]

# ------------------------------------------------
# Save
# ------------------------------------------------
network_survey_results.to_csv(
    os.path.join(
        analysis_out,
        "full_lasso_network_balance_index_vs_surveys.csv"
    ),
    index=False
)

network_survey_df.to_csv(
    os.path.join(
        analysis_out,
        "full_lasso_network_balance_index_subject_values_with_surveys.csv"
    ),
    index=False
)

# ------------------------------------------------
# Display
# ------------------------------------------------
print("=== Full LASSO Network Balance Index vs Survey Scores ===")
display(network_survey_results.round(4))

print("\nInterpretation:")
print(
    "Positive correlations indicate that individuals with relatively greater ISC "
    "in the positive-weighted LASSO network "
    "(Left TPJ, Right VLPFC, Left OFC, Left STS) compared with the "
    "negative-weighted LASSO network "
    "(Left VLPFC, Left DLPFC, Right TPJ) also tend to score higher "
    "on the corresponding survey measure."
)

print(
    "\nNegative correlations indicate that greater weighting toward the "
    "negative-weighted LASSO network is associated with higher survey scores."
)

=== Full LASSO Network Balance Index vs Survey Scores ===


,Survey,Network_Index,N,Pearson_r,Pearson_p,Permutation_p,Pearson_CI_low,Pearson_CI_high,Spearman_rho,Spearman_p,Slope,Intercept,R_squared,Pearson_p_FDR,Permutation_p_FDR,Spearman_p_FDR
0,AQ_Score,Full_LASSO_Network_Balance_Index,33,-0.0025,0.9891,0.9894,-0.3480,0.3409,0.0737,0.6835,-0.2508,16.6650,0.0000,0.9922,0.9922,0.9153
1,EQ_Score,Full_LASSO_Network_Balance_Index,33,0.0937,0.6039,0.5965,-0.3034,0.4529,0.0192,0.9153,20.9065,133.4127,0.0088,0.9922,0.9922,0.9153
2,EmQ_Score,Full_LASSO_Network_Balance_Index,33,-0.0018,0.9922,0.9922,-0.3445,0.3240,0.0330,0.8555,-0.1853,16.5139,0.0000,0.9922,0.9922,0.9153



Interpretation:
Positive correlations indicate that individuals with relatively greater ISC in the positive-weighted LASSO network (Left TPJ, Right VLPFC, Left OFC, Left STS) compared with the negative-weighted LASSO network (Left VLPFC, Left DLPFC, Right TPJ) also tend to score higher on the corresponding survey measure.

Negative correlations indicate that greater weighting toward the negative-weighted LASSO network is associated with higher survey scores.


In [111]:
# ============================================================
# 
# Cross-validated FIXED FULL LASSO Network Balance Architecture
#
# Fixed full LASSO-derived index:
#   mean(Left TPJ, Right VLPFC, Left OFC, Left STS)
#   -
#   mean(Left VLPFC, Left DLPFC, Right TPJ)
#
# For each held-out subject:
# 1. Compute fixed full LASSO network balance index for training subjects.
# 2. Fit regression: Performance ~ fixed full network balance index
#    using training subjects only.
# 3. Compute fixed full network balance index for held-out subject.
# 4. Predict held-out performance.
#
# This tests:
#   "Does the fixed full LASSO-derived architecture generalize?"
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ----------------------------
# Required objects
# ----------------------------
if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run your merge/analysis_df cell first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

# ----------------------------
# Fixed full LASSO architecture
# ----------------------------
positive_rois = [
    "Left TPJ",
    "Right VLPFC",
    "Left OFC",
    "Left STS"
]

negative_rois = [
    "Left VLPFC",
    "Left DLPFC",
    "Right TPJ"
]

required_cols = ["Subject", "Performance"] + positive_rois + negative_rois

missing_cols = [c for c in required_cols if c not in analysis_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in analysis_df: {missing_cols}")

# ----------------------------
# Prepare data
# ----------------------------
cv_df = analysis_df[required_cols].dropna().copy()

subjects_cv = cv_df["Subject"].values
y = cv_df["Performance"].values

# Fixed full LASSO network balance index for all subjects
# Use mean instead of sum because positive/negative sets have different sizes: 4 vs 3.
cv_df["Fixed_Full_LASSO_Network_Balance_Index"] = (
    cv_df[positive_rois].mean(axis=1) -
    cv_df[negative_rois].mean(axis=1)
)

x = cv_df["Fixed_Full_LASSO_Network_Balance_Index"].values

n = len(cv_df)

if n < 10:
    raise ValueError(f"Too few complete cases for LOOCV: N={n}")

loo = LeaveOneOut()

pred_fixed_balance = np.full(n, np.nan)
fold_rows = []

# ----------------------------
# Outer LOOCV
# ----------------------------
for fold, (train_idx, test_idx) in enumerate(loo.split(x), start=1):

    x_train = x[train_idx].reshape(-1, 1)
    x_test = x[test_idx].reshape(-1, 1)

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Fit training-only model
    if np.std(x_train) == 0:
        y_pred = np.mean(y_train)
        slope = np.nan
        intercept = np.mean(y_train)
    else:
        model = LinearRegression()
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)[0]
        slope = model.coef_[0]
        intercept = model.intercept_

    pred_fixed_balance[test_idx[0]] = y_pred

    fold_rows.append({
        "Fold": fold,
        "Heldout_Subject": subjects_cv[test_idx[0]],
        "Observed_Performance": y_test[0],
        "Predicted_Performance_Fixed_Full_LASSO_Balance": y_pred,
        "Heldout_Fixed_Full_LASSO_Network_Balance_Index": x[test_idx[0]],
        "Training_Slope": slope,
        "Training_Intercept": intercept,
        "Positive_ROIs": ", ".join(positive_rois),
        "Negative_ROIs": ", ".join(negative_rois)
    })

cv_fixed_fold_results = pd.DataFrame(fold_rows)

# ----------------------------
# Evaluate prediction
# ----------------------------
valid = np.isfinite(pred_fixed_balance)

y_valid = y[valid]
pred_valid = pred_fixed_balance[valid]
x_valid = x[valid]

r_pred, p_pred = pearsonr(y_valid, pred_valid)
rho_pred, p_s_pred = spearmanr(y_valid, pred_valid)

cv_r2 = 1 - (
    np.sum((y_valid - pred_valid) ** 2) /
    np.sum((y_valid - np.mean(y_valid)) ** 2)
)

rmse = np.sqrt(mean_squared_error(y_valid, pred_valid))

# In-sample association for reference only
r_index, p_index = pearsonr(x_valid, y_valid)
rho_index, p_s_index = spearmanr(x_valid, y_valid)

# ----------------------------
# Permutation test for CV prediction correlation
# ----------------------------
N_PERM = 10000
RNG = np.random.default_rng(42)

perm_rs = np.empty(N_PERM)

for i in range(N_PERM):
    y_perm = RNG.permutation(y_valid)
    perm_rs[i] = pearsonr(y_perm, pred_valid)[0]

p_perm_pred = (
    1 + np.sum(np.abs(perm_rs) >= abs(r_pred))
) / (N_PERM + 1)

# ----------------------------
# Save summary
# ----------------------------
cv_fixed_summary = pd.DataFrame([{
    "Model": "LOOCV_Fixed_Full_LASSO_Network_Balance_Index",
    "N": len(y_valid),
    "Fixed_Index": "(Left TPJ + Right VLPFC + Left OFC + Left STS)/4 - (Left VLPFC + Left DLPFC + Right TPJ)/3",
    "Positive_ROIs": ", ".join(positive_rois),
    "Negative_ROIs": ", ".join(negative_rois),
    "CV_Prediction_r": r_pred,
    "CV_Prediction_p": p_pred,
    "CV_Prediction_perm_p": p_perm_pred,
    "CV_Prediction_Spearman_rho": rho_pred,
    "CV_Prediction_Spearman_p": p_s_pred,
    "CV_R2": cv_r2,
    "RMSE": rmse,
    "InSample_Index_Performance_r_reference_only": r_index,
    "InSample_Index_Performance_p_reference_only": p_index,
    "InSample_Index_Performance_Spearman_reference_only": rho_index,
    "InSample_Index_Performance_Spearman_p_reference_only": p_s_index
}])

cv_fixed_fold_results.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_full_lasso_network_balance_fold_results.csv"
    ),
    index=False
)

cv_fixed_summary.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_full_lasso_network_balance_summary.csv"
    ),
    index=False
)

# Save subject-level values
cv_fixed_subject_values = cv_df.copy()
cv_fixed_subject_values["Predicted_Performance_Fixed_Full_LASSO_Balance_LOOCV"] = pred_fixed_balance

cv_fixed_subject_values.to_csv(
    os.path.join(
        analysis_out,
        "loocv_fixed_full_lasso_network_balance_subject_values.csv"
    ),
    index=False
)

print("=== LOOCV Fixed Full LASSO Network Balance Summary ===")
display(cv_fixed_summary.round(4))

print("\n=== Fold-level Predictions ===")
display(cv_fixed_fold_results.round(4))

# ----------------------------
# Plot 1: Observed vs predicted performance
# ----------------------------
plt.figure(figsize=(6.5, 5.5))

plt.scatter(
    y_valid,
    pred_valid,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

min_val = min(np.min(y_valid), np.min(pred_valid))
max_val = max(np.max(y_valid), np.max(pred_valid))

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)

fit = np.polyfit(y_valid, pred_valid, 1)
x_line = np.linspace(min_val, max_val, 200)

plt.plot(
    x_line,
    fit[0] * x_line + fit[1],
    linewidth=2.5,
    label="Observed fit"
)

plt.xlabel("Observed Emotion Tracking Performance")
plt.ylabel("Predicted Performance")
plt.title(
    "LOOCV Full LASSO Network Balance Prediction\n"
    f"r = {r_pred:.3f}, p = {p_pred:.4f}, perm p = {p_perm_pred:.4f}, "
    f"CV $R^2$ = {cv_r2:.3f}"
)

plt.legend(frameon=False)
plt.tight_layout()

out_fig_pred = os.path.join(
    analysis_out,
    "loocv_fixed_full_lasso_network_balance_prediction.png"
)

plt.savefig(out_fig_pred, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------
# Plot 2: Fixed full LASSO network balance index vs performance
# ----------------------------
plt.figure(figsize=(6.6, 5.3))

plt.scatter(
    x_valid,
    y_valid,
    s=75,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

slope, intercept = np.polyfit(x_valid, y_valid, 1)
x_line = np.linspace(x_valid.min(), x_valid.max(), 200)
y_line = slope * x_line + intercept

plt.plot(x_line, y_line, linewidth=2.5)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel(
    " Full LASSO Network Balance Index\n"
    "(Positive-weight ROIs mean) − (Negative-weight ROIs mean)",
    fontsize=12
)
plt.ylabel("Emotion Tracking Performance", fontsize=13)

plt.title(
    "Fixed Full LASSO Network Balance Index vs Performance\n"
    f"in-sample r = {r_index:.3f}, p = {p_index:.4f}",
    fontsize=12
)

plt.tight_layout()

out_fig_index = os.path.join(
    analysis_out,
    "fixed_full_lasso_network_balance_index_vs_performance_reference.png"
)

plt.savefig(out_fig_index, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nSaved LOOCV prediction figure to: {out_fig_pred}")
print(f"Saved fixed full LASSO index reference figure to: {out_fig_index}")

print("\nInterpretation:")
print("This analysis keeps the full LASSO-derived network architecture fixed in every fold:")
print("(Left TPJ, Right VLPFC, Left OFC, Left STS) - (Left VLPFC, Left DLPFC, Right TPJ).")
print("Because the positive and negative sets contain different numbers of ROIs, the index uses")
print("the mean of the positive ROIs minus the mean of the negative ROIs.")
print("Only the regression slope/intercept are fit on training subjects.")
print("This tests whether the fixed full LASSO-derived architecture predicts held-out performance.")

=== LOOCV Fixed Full LASSO Network Balance Summary ===


,Model,N,Fixed_Index,Positive_ROIs,Negative_ROIs,CV_Prediction_r,CV_Prediction_p,CV_Prediction_perm_p,CV_Prediction_Spearman_rho,CV_Prediction_Spearman_p,CV_R2,RMSE,InSample_Index_Performance_r_reference_only,InSample_Index_Performance_p_reference_only,InSample_Index_Performance_Spearman_reference_only,InSample_Index_Performance_Spearman_p_reference_only
0,LOOCV_Fixed_Full_LASSO_Network_Balance_Index,33,(Left TPJ + Right VLPFC + Left OFC + Left STS)...,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ",0.49,0.0038,0.0046,0.4562,0.0076,0.227,0.071,0.5874,0.0003,0.4843,0.0043



=== Fold-level Predictions ===


,Fold,Heldout_Subject,Observed_Performance,Predicted_Performance_Fixed_Full_LASSO_Balance,Heldout_Fixed_Full_LASSO_Network_Balance_Index,Training_Slope,Training_Intercept,Positive_ROIs,Negative_ROIs
0,1,pid001,0.5477,0.6119,0.0004,0.8015,0.6116,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
1,2,pid002,0.5368,0.5516,-0.0740,0.7891,0.6100,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
2,3,pid003,0.5210,0.5544,-0.0720,0.7789,0.6105,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
3,4,pid004,0.6948,0.6106,0.0046,0.7894,0.6070,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
4,5,pid005,0.5702,0.5669,-0.0534,0.7989,0.6095,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
5,6,pid006,0.5659,0.5670,-0.0535,0.7971,0.6096,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
6,7,pid007,0.6839,0.5986,-0.0105,0.8004,0.6070,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
7,8,pid008,0.6724,0.5409,-0.0746,0.8741,0.6061,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
8,9,pid009,0.6735,0.6344,0.0333,0.7842,0.6083,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"
9,10,pid010,0.6543,0.6760,0.0806,0.8139,0.6104,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ"



Saved LOOCV prediction figure to: C:\Users\pkell\vss_followup_roi_analyses\loocv_fixed_full_lasso_network_balance_prediction.png
Saved fixed full LASSO index reference figure to: C:\Users\pkell\vss_followup_roi_analyses\fixed_full_lasso_network_balance_index_vs_performance_reference.png

Interpretation:
This analysis keeps the full LASSO-derived network architecture fixed in every fold:
(Left TPJ, Right VLPFC, Left OFC, Left STS) - (Left VLPFC, Left DLPFC, Right TPJ).
Because the positive and negative sets contain different numbers of ROIs, the index uses
the mean of the positive ROIs minus the mean of the negative ROIs.
Only the regression slope/intercept are fit on training subjects.
This tests whether the fixed full LASSO-derived architecture predicts held-out performance.


In [112]:
# ============================================================
#  Null Network Balance Models
# Tests whether observed network balance index is stronger than
# randomly assigned positive-vs-negative ROI groupings.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# ----------------------------
# Required objects
# ----------------------------
if "analysis_df" not in globals():
    raise ValueError("analysis_df not found. Run your merge/analysis_df cell first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "vss_followup_roi_analyses")
    os.makedirs(analysis_out, exist_ok=True)

# ----------------------------
# Settings
# ----------------------------
N_NULL = 10000
RNG = np.random.default_rng(42)

# ROI columns
known_roi_cols = [
    "Left RLPFC", "Left VLPFC", "Left DLPFC", "Left OFC", "Left STS", "Left TPJ",
    "Right RLPFC", "Right DLPFC", "Right VLPFC", "Right OFC", "Right STS", "Right TPJ"
]

roi_cols = [c for c in known_roi_cols if c in analysis_df.columns]

if len(roi_cols) < 4:
    raise ValueError(f"Not enough ROI columns found. Found: {roi_cols}")

if "Performance" not in analysis_df.columns:
    raise ValueError("Performance column not found in analysis_df.")

# ----------------------------
# Define observed network indexes
# ----------------------------
observed_models = {
    "Original_Lateralized_Index": {
        "positive_rois": ["Right VLPFC", "Left TPJ"],
        "negative_rois": ["Left VLPFC", "Right TPJ"]
    },
    "Full_LASSO_Index": {
        "positive_rois": ["Left TPJ", "Right VLPFC", "Left OFC", "Left STS"],
        "negative_rois": ["Left VLPFC", "Left DLPFC", "Right TPJ"]
    }
}

# Keep only models with all needed columns
valid_models = {}

for model_name, model_info in observed_models.items():
    needed = model_info["positive_rois"] + model_info["negative_rois"] + ["Performance"]
    missing = [c for c in needed if c not in analysis_df.columns]

    if len(missing) == 0:
        valid_models[model_name] = model_info
    else:
        print(f"Skipping {model_name}; missing columns: {missing}")

if len(valid_models) == 0:
    raise ValueError("No valid observed network models found.")

# ----------------------------
# Helper function
# ----------------------------
def compute_balance_index(df, positive_rois, negative_rois):
    """
    Uses the mean of positive ROIs minus mean of negative ROIs.
    This avoids bias when positive and negative groups have unequal sizes.
    """
    pos = df[positive_rois].mean(axis=1)
    neg = df[negative_rois].mean(axis=1)
    return pos - neg

# ----------------------------
# Run null models
# ----------------------------
all_results = []
all_null_distributions = {}

for model_name, model_info in valid_models.items():

    positive_rois = model_info["positive_rois"]
    negative_rois = model_info["negative_rois"]

    n_pos = len(positive_rois)
    n_neg = len(negative_rois)

    needed_cols = ["Subject", "Performance"] + positive_rois + negative_rois
    tmp_obs = analysis_df[needed_cols].dropna().copy()

    observed_index = compute_balance_index(tmp_obs, positive_rois, negative_rois)
    y = tmp_obs["Performance"].values

    r_obs, p_obs = pearsonr(observed_index.values, y)
    rho_obs, p_s_obs = spearmanr(observed_index.values, y)

    # Null: randomly assign ROIs into positive and negative sets
    null_rs = np.full(N_NULL, np.nan)
    null_rhos = np.full(N_NULL, np.nan)
    null_models = []

    for perm in range(N_NULL):

        sampled = RNG.choice(roi_cols, size=n_pos + n_neg, replace=False)
        rand_pos = list(sampled[:n_pos])
        rand_neg = list(sampled[n_pos:])

        needed_rand_cols = ["Performance"] + rand_pos + rand_neg
        tmp = analysis_df[needed_rand_cols].dropna().copy()

        if len(tmp) < 5:
            continue

        rand_index = compute_balance_index(tmp, rand_pos, rand_neg)
        yy = tmp["Performance"].values

        if np.std(rand_index.values) == 0 or np.std(yy) == 0:
            continue

        null_rs[perm] = pearsonr(rand_index.values, yy)[0]
        null_rhos[perm] = spearmanr(rand_index.values, yy)[0]

        null_models.append({
            "Permutation": perm,
            "Positive_ROIs": ", ".join(rand_pos),
            "Negative_ROIs": ", ".join(rand_neg),
            "Pearson_r": null_rs[perm],
            "Spearman_rho": null_rhos[perm]
        })

    null_rs_valid = null_rs[np.isfinite(null_rs)]
    null_rhos_valid = null_rhos[np.isfinite(null_rhos)]

    p_null_two_sided = (
        1 + np.sum(np.abs(null_rs_valid) >= abs(r_obs))
    ) / (len(null_rs_valid) + 1)

    p_null_greater = (
        1 + np.sum(null_rs_valid >= r_obs)
    ) / (len(null_rs_valid) + 1)

    null_ci_low, null_ci_high = np.percentile(null_rs_valid, [2.5, 97.5])

    all_results.append({
        "Model": model_name,
        "N": len(tmp_obs),
        "Positive_ROIs": ", ".join(positive_rois),
        "Negative_ROIs": ", ".join(negative_rois),
        "Observed_Pearson_r": r_obs,
        "Observed_Pearson_p": p_obs,
        "Observed_Spearman_rho": rho_obs,
        "Observed_Spearman_p": p_s_obs,
        "Null_mean_r": np.mean(null_rs_valid),
        "Null_sd_r": np.std(null_rs_valid),
        "Null_95CI_low": null_ci_low,
        "Null_95CI_high": null_ci_high,
        "Empirical_p_two_sided_vs_random_indexes": p_null_two_sided,
        "Empirical_p_greater_vs_random_indexes": p_null_greater,
        "N_valid_nulls": len(null_rs_valid)
    })

    all_null_distributions[model_name] = pd.DataFrame({
        "Null_Pearson_r": null_rs_valid
    })

    null_models_df = pd.DataFrame(null_models)
    null_models_df.to_csv(
        os.path.join(analysis_out, f"{model_name}_random_network_index_null_models.csv"),
        index=False
    )

    # ----------------------------
    # Plot null distribution
    # ----------------------------
    plt.figure(figsize=(7, 5))
    plt.hist(null_rs_valid, bins=40, alpha=0.75, edgecolor="black")
    plt.axvline(r_obs, linewidth=3, label=f"Observed r = {r_obs:.3f}")
    plt.axvline(null_ci_low, linestyle="--", linewidth=2, label="Null 95% interval")
    plt.axvline(null_ci_high, linestyle="--", linewidth=2)

    plt.xlabel("Pearson r from random network balance indexes")
    plt.ylabel("Count")
    plt.title(
        f"{model_name}\n"
        f"Observed r = {r_obs:.3f}, empirical p = {p_null_two_sided:.4f}"
    )
    plt.legend(frameon=False)
    plt.tight_layout()

    fig_path = os.path.join(
        analysis_out,
        f"{model_name}_random_network_index_null_histogram.png"
    )
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

# ----------------------------
# Save and display
# ----------------------------
network_null_results = pd.DataFrame(all_results)

network_null_results.to_csv(
    os.path.join(analysis_out, "network_balance_random_index_null_results.csv"),
    index=False
)

print("=== Random Network Balance Index Null Results ===")
display(network_null_results.round(4))

print("\nInterpretation:")
print("These null models test whether the observed network balance index predicts performance")
print("better than random ROI groupings with the same positive/negative group sizes.")

=== Random Network Balance Index Null Results ===


,Model,N,Positive_ROIs,Negative_ROIs,Observed_Pearson_r,Observed_Pearson_p,Observed_Spearman_rho,Observed_Spearman_p,Null_mean_r,Null_sd_r,Null_95CI_low,Null_95CI_high,Empirical_p_two_sided_vs_random_indexes,Empirical_p_greater_vs_random_indexes,N_valid_nulls
0,Original_Lateralized_Index,34,"Right VLPFC, Left TPJ","Left VLPFC, Right TPJ",0.4757,0.0045,0.4341,0.0103,0.0001,0.1634,-0.3189,0.3176,0.0020,0.0010,10000
1,Full_LASSO_Index,33,"Left TPJ, Right VLPFC, Left OFC, Left STS","Left VLPFC, Left DLPFC, Right TPJ",0.5874,0.0003,0.4843,0.0043,0.0036,0.1738,-0.3316,0.3442,0.0002,0.0002,10000



Interpretation:
These null models test whether the observed network balance index predicts performance
better than random ROI groupings with the same positive/negative group sizes.


## Motor Control Analysis

In [120]:
# ============================================================
# HAND-KINEMATIC FEATURE EXTRACTION
#
# Produces:
#   motor_kinematic_features_df
#   motor_feature_summary_df
#   motor_feature_correlation_df
#   motor_qc_flagged_trials_df
#
# Saves all tables and figures to:
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data
# \Emotion Perception Data\Motor_Control
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.stats import median_abs_deviation

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. PATHS
# ============================================================

BEHAVIORAL_DATA_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Behavioral Data\data"
)

MOTOR_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
)

KINEMATIC_DIR = MOTOR_OUTPUT_DIR / "01_hand_kinematic_features"
KINEMATIC_TABLE_DIR = KINEMATIC_DIR / "tables"
KINEMATIC_FIG_DIR = KINEMATIC_DIR / "figures"
KINEMATIC_QC_FIG_DIR = KINEMATIC_FIG_DIR / "qc_trajectories"

for folder in [
    MOTOR_OUTPUT_DIR,
    KINEMATIC_DIR,
    KINEMATIC_TABLE_DIR,
    KINEMATIC_FIG_DIR,
    KINEMATIC_QC_FIG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

if not BEHAVIORAL_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Behavioral data folder was not found:\n{BEHAVIORAL_DATA_DIR}"
    )

print(f"Behavioral data folder:\n{BEHAVIORAL_DATA_DIR}")
print(f"\nMotor-control output folder:\n{MOTOR_OUTPUT_DIR}")


# ============================================================
# 2. USER-ADJUSTABLE SETTINGS
# ============================================================

# Used only when the CSV does not contain a recognizable time column.
# Your plots suggest dense sampling, but confirm the true recording rate.
DEFAULT_SAMPLING_RATE_HZ = 60.0

# Short internal gaps up to this duration are interpolated.
MAX_INTERPOLATED_GAP_SEC = 0.25

# Trials with a larger fraction of missing samples are flagged.
MAX_MISSING_PROPORTION = 0.20

# Trials need at least this much usable data.
MIN_VALID_DURATION_SEC = 5.0
MIN_VALID_SAMPLES = 30

# Savitzky-Golay smoothing before derivatives.
SMOOTH_WINDOW_SEC = 0.25
SMOOTH_POLYORDER = 2

# Velocity threshold below which the hand is considered paused.
# Units are behavioral-rating units per second.
# Confirm that this is sensible for your rating scale.
PAUSE_VELOCITY_THRESHOLD = 0.05

# Velocity above this threshold counts as active movement.
MOVEMENT_VELOCITY_THRESHOLD = 0.05

# A low-velocity segment must last this long to be a pause.
MIN_PAUSE_DURATION_SEC = 0.25

# Reversal detection ignores tiny velocity fluctuations.
REVERSAL_VELOCITY_THRESHOLD = 0.05

# Entropy settings.
N_ENTROPY_BINS = 20

# Robust outlier threshold after all features are computed.
ROBUST_Z_OUTLIER_THRESHOLD = 5.0

# Number of flagged trajectories to plot.
N_FLAGGED_TRAJECTORY_PLOTS = 12

FIGURE_DPI = 300
EPSILON = 1e-12


# ============================================================
# 3. GENERAL HELPERS
# ============================================================

def safe_filename(value, max_length=150):
    """Return a Windows-safe filename."""
    value = str(value)
    value = re.sub(r'[<>:"/\\|?*]+', "_", value)
    value = re.sub(r"\s+", "_", value)
    value = re.sub(r"_+", "_", value)
    value = value.strip("._ ")

    if len(value) > max_length:
        value = value[:max_length]

    return value or "unnamed"


def split_video_modality(column_name):
    """
    Split a behavioral column into BaseVideo and Modality.

    Examples
    --------
    87_valence          -> BaseVideo='87', Modality='Valence'
    14_baseline_arousal -> BaseVideo='14_baseline', Modality='Arousal'
    """
    column_name = str(column_name)

    match = re.search(
        r"_(valence|arousal)$",
        column_name,
        flags=re.IGNORECASE
    )

    if match is None:
        return column_name, None

    modality = match.group(1).capitalize()

    base_video = re.sub(
        r"_(valence|arousal)$",
        "",
        column_name,
        flags=re.IGNORECASE
    )

    return base_video, modality


def is_monotonic_numeric(series):
    """Check whether a series is numeric and monotonically increasing."""
    numeric = pd.to_numeric(series, errors="coerce")

    valid = numeric.dropna()

    if len(valid) < 3:
        return False

    differences = np.diff(valid.to_numpy(dtype=float))

    return np.all(differences > 0)


def infer_time_vector(df):
    """
    Recover a time vector from the first CSV column when possible.

    Rules
    -----
    1. If the first column name contains 'time', 'sec', 'second', or 'ms',
       use it as the time vector.
    2. If it is numeric and monotonic but increments in whole numbers
       (e.g., 0, 1, 2, ...), treat it as a sample index and convert using
       DEFAULT_SAMPLING_RATE_HZ.
    3. Otherwise, construct time from the row number and sampling rate.

    Returns
    -------
    time_sec : np.ndarray
    source_label : str
    """
    first_column_name = str(df.columns[0])
    first_column = pd.to_numeric(
        df.iloc[:, 0],
        errors="coerce"
    )

    name_lower = first_column_name.lower()

    time_name_tokens = [
        "time",
        "sec",
        "second",
        "timestamp",
        "millisecond",
        "_ms",
        "(ms)"
    ]

    looks_like_time_name = any(
        token in name_lower
        for token in time_name_tokens
    )

    if looks_like_time_name and is_monotonic_numeric(first_column):
        time_values = first_column.to_numpy(dtype=float)

        # Convert milliseconds to seconds when the column explicitly says ms.
        if (
            "millisecond" in name_lower
            or "_ms" in name_lower
            or "(ms)" in name_lower
        ):
            return time_values / 1000.0, f"{first_column_name} converted ms→s"

        return time_values, f"CSV column: {first_column_name}"

    if is_monotonic_numeric(first_column):
        valid = first_column.dropna().to_numpy(dtype=float)
        median_step = np.nanmedian(np.diff(valid))

        # Whole-number increments near one are probably an index.
        if np.isclose(median_step, 1.0, atol=0.05):
            time_sec = np.arange(len(df), dtype=float) / DEFAULT_SAMPLING_RATE_HZ
            return (
                time_sec,
                f"row/sample index converted at {DEFAULT_SAMPLING_RATE_HZ:g} Hz"
            )

        # Small positive increments are plausibly seconds.
        if 0 < median_step < 1:
            return (
                first_column.to_numpy(dtype=float),
                f"monotonic first column: {first_column_name}"
            )

    time_sec = np.arange(len(df), dtype=float) / DEFAULT_SAMPLING_RATE_HZ

    return (
        time_sec,
        f"constructed from row number at {DEFAULT_SAMPLING_RATE_HZ:g} Hz"
    )


def longest_true_run(mask):
    """
    Return the start and end indices of the longest contiguous True segment.
    End is exclusive.
    """
    mask = np.asarray(mask, dtype=bool)

    if not mask.any():
        return None, None

    padded = np.concatenate([
        [False],
        mask,
        [False]
    ])

    changes = np.diff(padded.astype(int))

    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]

    lengths = ends - starts
    best = int(np.argmax(lengths))

    return int(starts[best]), int(ends[best])


def choose_savgol_window(n_samples, median_dt):
    """Choose a valid odd Savitzky-Golay window."""
    if (
        not np.isfinite(median_dt)
        or median_dt <= 0
    ):
        return None

    requested = int(
        round(SMOOTH_WINDOW_SEC / median_dt)
    )

    requested = max(
        requested,
        SMOOTH_POLYORDER + 2
    )

    if requested % 2 == 0:
        requested += 1

    largest_valid = (
        n_samples
        if n_samples % 2 == 1
        else n_samples - 1
    )

    window = min(
        requested,
        largest_valid
    )

    if window <= SMOOTH_POLYORDER:
        return None

    return int(window)


def contiguous_true_durations(mask, time_sec):
    """
    Return durations of contiguous True runs.

    Duration includes the approximate sampling interval so that a
    one-sample segment has nonzero duration.
    """
    mask = np.asarray(mask, dtype=bool)
    time_sec = np.asarray(time_sec, dtype=float)

    if len(mask) == 0 or not mask.any():
        return np.array([], dtype=float)

    median_dt = np.nanmedian(
        np.diff(time_sec)
    )

    padded = np.concatenate([
        [False],
        mask,
        [False]
    ])

    changes = np.diff(
        padded.astype(int)
    )

    starts = np.where(
        changes == 1
    )[0]

    ends = np.where(
        changes == -1
    )[0]

    durations = []

    for start, end in zip(starts, ends):
        last_index = end - 1

        duration = (
            time_sec[last_index]
            - time_sec[start]
            + median_dt
        )

        durations.append(
            max(float(duration), 0.0)
        )

    return np.asarray(
        durations,
        dtype=float
    )


def normalized_shannon_entropy(values, n_bins=20):
    """
    Normalized Shannon entropy in [0, 1].

    Here it is applied to absolute velocity. Higher values indicate a
    broader, less concentrated distribution of movement speeds.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 5:
        return np.nan

    if np.nanmax(values) - np.nanmin(values) <= EPSILON:
        return 0.0

    counts, _ = np.histogram(
        values,
        bins=n_bins
    )

    probabilities = counts[
        counts > 0
    ].astype(float)

    probabilities /= probabilities.sum()

    entropy = -np.sum(
        probabilities
        * np.log(probabilities)
    )

    maximum_entropy = np.log(
        n_bins
    )

    return float(
        entropy / maximum_entropy
    )


def compute_reversals(velocity, threshold):
    """
    Count direction changes while ignoring near-zero velocity noise.
    """
    velocity = np.asarray(
        velocity,
        dtype=float
    )

    valid_velocity = velocity[
        np.isfinite(velocity)
        & (np.abs(velocity) >= threshold)
    ]

    if len(valid_velocity) < 2:
        return 0

    signs = np.sign(
        valid_velocity
    )

    return int(
        np.sum(
            signs[1:] != signs[:-1]
        )
    )


def safe_trapezoid(y, x):
    """Compatibility wrapper for NumPy versions with trapezoid or trapz."""
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)

    return np.trapz(y, x)


# ============================================================
# 4. CLEAN ONE TRAJECTORY
# ============================================================

def clean_and_smooth_trajectory(
    time_sec,
    position
):
    """
    Clean, short-gap interpolate, isolate a continuous valid segment,
    and smooth one trajectory.

    Returns
    -------
    result : dict
        Contains cleaned arrays and QC information.
    """
    time_sec = np.asarray(
        time_sec,
        dtype=float
    )

    position = pd.to_numeric(
        pd.Series(position),
        errors="coerce"
    ).to_numpy(dtype=float)

    n_original = min(
        len(time_sec),
        len(position)
    )

    time_sec = time_sec[:n_original]
    position = position[:n_original]

    trajectory_df = pd.DataFrame({
        "Time": time_sec,
        "Position": position
    })

    trajectory_df = trajectory_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Remove rows with invalid timestamps.
    trajectory_df = trajectory_df.dropna(
        subset=["Time"]
    )

    trajectory_df = trajectory_df.sort_values(
        "Time"
    )

    n_duplicate_timestamps = int(
        trajectory_df["Time"].duplicated().sum()
    )

    # Average positions recorded at identical timestamps.
    trajectory_df = (
        trajectory_df
        .groupby(
            "Time",
            as_index=False
        )["Position"]
        .mean()
    )

    # Remove leading and trailing missing position values.
    valid_position_indices = np.where(
        np.isfinite(
            trajectory_df["Position"].to_numpy(dtype=float)
        )
    )[0]

    if len(valid_position_indices) == 0:
        return {
            "Success": False,
            "Reason": "No finite position values",
            "NOriginal": n_original,
            "NDuplicateTimestamps": n_duplicate_timestamps
        }

    first_valid = int(
        valid_position_indices[0]
    )

    last_valid = int(
        valid_position_indices[-1]
    )

    trajectory_df = trajectory_df.iloc[
        first_valid:last_valid + 1
    ].reset_index(drop=True)

    original_missing_mask = trajectory_df[
        "Position"
    ].isna()

    missing_proportion = float(
        original_missing_mask.mean()
    )

    time_values = trajectory_df[
        "Time"
    ].to_numpy(dtype=float)

    time_differences = np.diff(
        time_values
    )

    valid_dt = time_differences[
        np.isfinite(time_differences)
        & (time_differences > 0)
    ]

    if len(valid_dt) == 0:
        return {
            "Success": False,
            "Reason": "Invalid or constant time vector",
            "NOriginal": n_original,
            "NDuplicateTimestamps": n_duplicate_timestamps
        }

    median_dt = float(
        np.nanmedian(valid_dt)
    )

    interpolation_limit_samples = max(
        1,
        int(
            np.floor(
                MAX_INTERPOLATED_GAP_SEC
                / median_dt
            )
        )
    )

    interpolated_position = (
        trajectory_df["Position"]
        .interpolate(
            method="linear",
            limit=interpolation_limit_samples,
            limit_direction="both",
            limit_area="inside"
        )
    )

    # Any remaining NaN represents a long internal gap.
    remaining_valid = interpolated_position.notna().to_numpy()

    long_gap_detected = bool(
        (~remaining_valid).any()
    )

    # Use the longest continuous valid segment so derivatives never bridge
    # across a long missing interval.
    segment_start, segment_end = longest_true_run(
        remaining_valid
    )

    if segment_start is None:
        return {
            "Success": False,
            "Reason": "No valid segment after interpolation",
            "NOriginal": n_original,
            "NDuplicateTimestamps": n_duplicate_timestamps
        }

    time_clean = trajectory_df[
        "Time"
    ].to_numpy(dtype=float)[
        segment_start:segment_end
    ]

    position_clean = interpolated_position.to_numpy(
        dtype=float
    )[
        segment_start:segment_end
    ]

    n_valid = len(
        position_clean
    )

    if n_valid < MIN_VALID_SAMPLES:
        return {
            "Success": False,
            "Reason": "Too few valid samples",
            "NOriginal": n_original,
            "NValid": n_valid,
            "NDuplicateTimestamps": n_duplicate_timestamps,
            "MissingProportion": missing_proportion,
            "LongGapDetected": long_gap_detected
        }

    duration_sec = float(
        time_clean[-1]
        - time_clean[0]
    )

    if duration_sec < MIN_VALID_DURATION_SEC:
        return {
            "Success": False,
            "Reason": "Valid duration too short",
            "NOriginal": n_original,
            "NValid": n_valid,
            "NDuplicateTimestamps": n_duplicate_timestamps,
            "MissingProportion": missing_proportion,
            "LongGapDetected": long_gap_detected,
            "DurationSec": duration_sec
        }

    smooth_window = choose_savgol_window(
        n_samples=n_valid,
        median_dt=median_dt
    )

    if smooth_window is None:
        position_smoothed = position_clean.copy()
        smoothing_applied = False

    else:
        position_smoothed = savgol_filter(
            position_clean,
            window_length=smooth_window,
            polyorder=SMOOTH_POLYORDER,
            mode="interp"
        )

        smoothing_applied = True

    return {
        "Success": True,
        "Time": time_clean,
        "PositionRaw": position_clean,
        "PositionSmooth": position_smoothed,
        "DurationSec": duration_sec,
        "MedianDt": median_dt,
        "SamplingRateEstimatedHz": 1.0 / median_dt,
        "NOriginal": n_original,
        "NValid": n_valid,
        "MissingProportion": missing_proportion,
        "NDuplicateTimestamps": n_duplicate_timestamps,
        "LongGapDetected": long_gap_detected,
        "SmoothingApplied": smoothing_applied,
        "SmoothWindowSamples": (
            smooth_window
            if smooth_window is not None
            else np.nan
        )
    }


# ============================================================
# 5. EXTRACT FEATURES FROM ONE CLEAN TRAJECTORY
# ============================================================

def extract_kinematic_features(clean_result):
    """
    Compute movement features from one cleaned trajectory.
    """
    time_sec = clean_result[
        "Time"
    ]

    position_raw = clean_result[
        "PositionRaw"
    ]

    position_smooth = clean_result[
        "PositionSmooth"
    ]

    duration_sec = clean_result[
        "DurationSec"
    ]

    velocity = np.gradient(
        position_smooth,
        time_sec
    )

    acceleration = np.gradient(
        velocity,
        time_sec
    )

    jerk = np.gradient(
        acceleration,
        time_sec
    )

    absolute_velocity = np.abs(
        velocity
    )

    absolute_acceleration = np.abs(
        acceleration
    )

    absolute_jerk = np.abs(
        jerk
    )

    position_changes = np.diff(
        position_smooth
    )

    path_length = float(
        np.nansum(
            np.abs(position_changes)
        )
    )

    net_displacement_signed = float(
        position_smooth[-1]
        - position_smooth[0]
    )

    net_displacement_absolute = abs(
        net_displacement_signed
    )

    path_efficiency = (
        net_displacement_absolute / path_length
        if path_length > EPSILON
        else np.nan
    )

    reversal_count = compute_reversals(
        velocity=velocity,
        threshold=REVERSAL_VELOCITY_THRESHOLD
    )

    reversal_rate = (
        reversal_count / duration_sec
        if duration_sec > 0
        else np.nan
    )

    pause_mask = (
        absolute_velocity
        <= PAUSE_VELOCITY_THRESHOLD
    )

    raw_pause_durations = contiguous_true_durations(
        pause_mask,
        time_sec
    )

    valid_pause_durations = raw_pause_durations[
        raw_pause_durations
        >= MIN_PAUSE_DURATION_SEC
    ]

    pause_count = int(
        len(valid_pause_durations)
    )

    total_pause_duration = float(
        valid_pause_durations.sum()
    )

    mean_pause_duration = (
        float(
            valid_pause_durations.mean()
        )
        if pause_count > 0
        else 0.0
    )

    pause_proportion = (
        total_pause_duration / duration_sec
        if duration_sec > 0
        else np.nan
    )

    proportion_moving = float(
        np.mean(
            absolute_velocity
            > MOVEMENT_VELOCITY_THRESHOLD
        )
    )

    # Dimensionless jerk:
    # duration^5 * integral(jerk^2 dt) / path_length^2
    integrated_squared_jerk = float(
        safe_trapezoid(
            jerk ** 2,
            time_sec
        )
    )

    dimensionless_jerk = (
        (
            duration_sec ** 5
            * integrated_squared_jerk
        )
        / (
            path_length ** 2
            + EPSILON
        )
    )

    # Higher values are interpreted as smoother movement.
    smoothness = -np.log10(
        dimensionless_jerk
        + EPSILON
    )

    movement_entropy = normalized_shannon_entropy(
        absolute_velocity,
        n_bins=N_ENTROPY_BINS
    )

    features = {
        # First-order movement
        "MeanAbsVelocity": np.nanmean(
            absolute_velocity
        ),
        "MedianAbsVelocity": np.nanmedian(
            absolute_velocity
        ),
        "PeakAbsVelocity": np.nanmax(
            absolute_velocity
        ),
        "VelocitySD": np.nanstd(
            velocity,
            ddof=1
        ),
        "AbsVelocitySD": np.nanstd(
            absolute_velocity,
            ddof=1
        ),
        "ProportionMoving": proportion_moving,

        # Second-order movement
        "MeanAbsAcceleration": np.nanmean(
            absolute_acceleration
        ),
        "PeakAbsAcceleration": np.nanmax(
            absolute_acceleration
        ),
        "AccelerationSD": np.nanstd(
            acceleration,
            ddof=1
        ),

        # Third-order movement
        "MeanAbsJerk": np.nanmean(
            absolute_jerk
        ),
        "RMSJerk": np.sqrt(
            np.nanmean(
                jerk ** 2
            )
        ),
        "PeakAbsJerk": np.nanmax(
            absolute_jerk
        ),

        # Path and direction
        "PathLength": path_length,
        "PathLengthPerSecond": (
            path_length / duration_sec
            if duration_sec > 0
            else np.nan
        ),
        "NetDisplacementSigned": (
            net_displacement_signed
        ),
        "NetDisplacementAbsolute": (
            net_displacement_absolute
        ),
        "PathEfficiency": path_efficiency,
        "ReversalCount": reversal_count,
        "ReversalRate": reversal_rate,

        # Pausing
        "PauseCount": pause_count,
        "PauseRate": (
            pause_count / duration_sec
            if duration_sec > 0
            else np.nan
        ),
        "TotalPauseDuration": (
            total_pause_duration
        ),
        "MeanPauseDuration": (
            mean_pause_duration
        ),
        "PauseProportion": pause_proportion,

        # Variability and complexity
        "PositionSD": np.nanstd(
            position_raw,
            ddof=1
        ),
        "SmoothedPositionSD": np.nanstd(
            position_smooth,
            ddof=1
        ),
        "DimensionlessJerk": dimensionless_jerk,
        "Smoothness": smoothness,
        "MovementEntropy": movement_entropy
    }

    derivative_data = pd.DataFrame({
        "Time": time_sec,
        "PositionRaw": position_raw,
        "PositionSmooth": position_smooth,
        "Velocity": velocity,
        "AbsVelocity": absolute_velocity,
        "Acceleration": acceleration,
        "Jerk": jerk
    })

    return features, derivative_data


# ============================================================
# 6. LOAD FILES AND EXTRACT FEATURES
# ============================================================

feature_rows = []
failed_rows = []

# Keep selected trajectory data in memory for QC plotting.
trajectory_cache = {}

csv_files = sorted(
    BEHAVIORAL_DATA_DIR.glob("*.csv")
)

if len(csv_files) == 0:
    raise RuntimeError(
        f"No CSV files were found in:\n{BEHAVIORAL_DATA_DIR}"
    )

print(
    f"\nFound {len(csv_files)} behavioral CSV files."
)

for file_number, csv_path in enumerate(
    csv_files,
    start=1
):
    subject_id = csv_path.stem

    try:
        subject_df = pd.read_csv(
            csv_path
        )

    except Exception as error:
        failed_rows.append({
            "Subject": subject_id,
            "Video": np.nan,
            "Reason": f"CSV read failure: {error}"
        })

        continue

    if subject_df.shape[1] < 2:
        failed_rows.append({
            "Subject": subject_id,
            "Video": np.nan,
            "Reason": "CSV contained fewer than two columns"
        })

        continue

    time_sec, time_source = infer_time_vector(
        subject_df
    )

    # The notebook drops the first column and treats every remaining
    # valence/arousal column as a video trajectory.
    behavioral_columns = [
        column
        for column in subject_df.columns[1:]
        if re.search(
            r"_(valence|arousal)$",
            str(column),
            flags=re.IGNORECASE
        )
    ]

    for video_column in behavioral_columns:
        base_video, modality = split_video_modality(
            video_column
        )

        clean_result = clean_and_smooth_trajectory(
            time_sec=time_sec,
            position=subject_df[video_column]
        )

        if not clean_result.get(
            "Success",
            False
        ):
            failed_rows.append({
                "Subject": str(subject_id),
                "Video": str(video_column),
                "BaseVideo": str(base_video),
                "Modality": modality,
                "TimeSource": time_source,
                "Reason": clean_result.get(
                    "Reason",
                    "Unknown cleaning failure"
                ),
                "NOriginalSamples": clean_result.get(
                    "NOriginal",
                    np.nan
                ),
                "NValidSamples": clean_result.get(
                    "NValid",
                    np.nan
                ),
                "MissingProportion": clean_result.get(
                    "MissingProportion",
                    np.nan
                )
            })

            continue

        features, derivative_data = extract_kinematic_features(
            clean_result
        )

        feature_row = {
            "Subject": str(subject_id),
            "Video": str(video_column),
            "BaseVideo": str(base_video),
            "Modality": modality,
            "TimeSource": time_source,

            "DurationSec": clean_result[
                "DurationSec"
            ],
            "NOriginalSamples": clean_result[
                "NOriginal"
            ],
            "NValidSamples": clean_result[
                "NValid"
            ],
            "EstimatedSamplingRateHz": clean_result[
                "SamplingRateEstimatedHz"
            ],
            "MissingProportion": clean_result[
                "MissingProportion"
            ],
            "DuplicateTimestampCount": clean_result[
                "NDuplicateTimestamps"
            ],
            "LongGapDetected": clean_result[
                "LongGapDetected"
            ],
            "SmoothingApplied": clean_result[
                "SmoothingApplied"
            ],
            "SmoothWindowSamples": clean_result[
                "SmoothWindowSamples"
            ],

            **features
        }

        feature_rows.append(
            feature_row
        )

        trajectory_cache[
            (
                str(subject_id),
                str(video_column)
            )
        ] = derivative_data

    if (
        file_number % 10 == 0
        or file_number == len(csv_files)
    ):
        print(
            f"Processed {file_number}/{len(csv_files)} files."
        )


motor_kinematic_features_df = pd.DataFrame(
    feature_rows
)

motor_failed_trajectories_df = pd.DataFrame(
    failed_rows
)

if motor_kinematic_features_df.empty:
    raise RuntimeError(
        "No kinematic feature rows were generated. "
        "Check the CSV structure and video-column names."
    )

print(
    f"\nExtracted features for "
    f"{len(motor_kinematic_features_df)} "
    "subject × video × modality observations."
)

print(
    f"Failed or excluded trajectories: "
    f"{len(motor_failed_trajectories_df)}"
)


# ============================================================
# 7. QC FLAGS
# ============================================================

identifier_columns = [
    "Subject",
    "Video",
    "BaseVideo",
    "Modality",
    "TimeSource"
]

qc_columns = [
    "DurationSec",
    "NOriginalSamples",
    "NValidSamples",
    "EstimatedSamplingRateHz",
    "MissingProportion",
    "DuplicateTimestampCount",
    "LongGapDetected",
    "SmoothingApplied",
    "SmoothWindowSamples"
]

feature_columns = [
    column
    for column in motor_kinematic_features_df.columns
    if (
        column not in identifier_columns
        and column not in qc_columns
    )
]

# Impossible/nonfinite feature values.
motor_kinematic_features_df[
    "ImpossibleValueFlag"
] = False

nonnegative_features = [
    "DurationSec",
    "MeanAbsVelocity",
    "MedianAbsVelocity",
    "PeakAbsVelocity",
    "VelocitySD",
    "AbsVelocitySD",
    "ProportionMoving",
    "MeanAbsAcceleration",
    "PeakAbsAcceleration",
    "AccelerationSD",
    "MeanAbsJerk",
    "RMSJerk",
    "PeakAbsJerk",
    "PathLength",
    "PathLengthPerSecond",
    "NetDisplacementAbsolute",
    "PathEfficiency",
    "ReversalCount",
    "ReversalRate",
    "PauseCount",
    "PauseRate",
    "TotalPauseDuration",
    "MeanPauseDuration",
    "PauseProportion",
    "PositionSD",
    "SmoothedPositionSD",
    "DimensionlessJerk",
    "MovementEntropy"
]

for column in nonnegative_features:
    if column in motor_kinematic_features_df.columns:
        motor_kinematic_features_df[
            "ImpossibleValueFlag"
        ] |= (
            motor_kinematic_features_df[
                column
            ] < 0
        ).fillna(False)

for proportion_column in [
    "ProportionMoving",
    "PathEfficiency",
    "PauseProportion",
    "MovementEntropy"
]:
    if proportion_column in motor_kinematic_features_df.columns:
        motor_kinematic_features_df[
            "ImpossibleValueFlag"
        ] |= (
            (
                motor_kinematic_features_df[
                    proportion_column
                ] < 0
            )
            | (
                motor_kinematic_features_df[
                    proportion_column
                ] > 1
            )
        ).fillna(False)


# Robust feature-level outlier flags.
outlier_flag_matrix = pd.DataFrame(
    False,
    index=motor_kinematic_features_df.index,
    columns=feature_columns
)

for column in feature_columns:
    values = pd.to_numeric(
        motor_kinematic_features_df[column],
        errors="coerce"
    )

    median_value = np.nanmedian(
        values
    )

    mad_value = median_abs_deviation(
        values,
        nan_policy="omit",
        scale="normal"
    )

    if (
        not np.isfinite(mad_value)
        or mad_value <= EPSILON
    ):
        continue

    robust_z = (
        values - median_value
    ) / mad_value

    outlier_flag_matrix[
        column
    ] = (
        np.abs(robust_z)
        > ROBUST_Z_OUTLIER_THRESHOLD
    ).fillna(False)


motor_kinematic_features_df[
    "NFeatureOutliers"
] = outlier_flag_matrix.sum(
    axis=1
)

motor_kinematic_features_df[
    "ExtremeFeatureOutlierFlag"
] = (
    motor_kinematic_features_df[
        "NFeatureOutliers"
    ] > 0
)

motor_kinematic_features_df[
    "HighMissingnessFlag"
] = (
    motor_kinematic_features_df[
        "MissingProportion"
    ] > MAX_MISSING_PROPORTION
)

motor_kinematic_features_df[
    "InsufficientValidDataFlag"
] = (
    (
        motor_kinematic_features_df[
            "DurationSec"
        ] < MIN_VALID_DURATION_SEC
    )
    | (
        motor_kinematic_features_df[
            "NValidSamples"
        ] < MIN_VALID_SAMPLES
    )
)

motor_kinematic_features_df[
    "AnyQCFlag"
] = (
    motor_kinematic_features_df[
        "ImpossibleValueFlag"
    ]
    | motor_kinematic_features_df[
        "ExtremeFeatureOutlierFlag"
    ]
    | motor_kinematic_features_df[
        "HighMissingnessFlag"
    ]
    | motor_kinematic_features_df[
        "InsufficientValidDataFlag"
    ]
    | motor_kinematic_features_df[
        "LongGapDetected"
    ].fillna(False)
)

motor_qc_flagged_trials_df = (
    motor_kinematic_features_df[
        motor_kinematic_features_df[
            "AnyQCFlag"
        ]
    ]
    .copy()
    .sort_values(
        [
            "NFeatureOutliers",
            "MissingProportion"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 8. FEATURE SUMMARY
# ============================================================

summary_rows = []

for column in feature_columns:
    values = pd.to_numeric(
        motor_kinematic_features_df[column],
        errors="coerce"
    )

    summary_rows.append({
        "Feature": column,
        "N": int(
            values.notna().sum()
        ),
        "MissingN": int(
            values.isna().sum()
        ),
        "MissingProportion": float(
            values.isna().mean()
        ),
        "Mean": values.mean(),
        "SD": values.std(ddof=1),
        "Median": values.median(),
        "Q1": values.quantile(0.25),
        "Q3": values.quantile(0.75),
        "IQR": (
            values.quantile(0.75)
            - values.quantile(0.25)
        ),
        "Minimum": values.min(),
        "Maximum": values.max()
    })

motor_feature_summary_df = pd.DataFrame(
    summary_rows
)

motor_feature_correlation_df = (
    motor_kinematic_features_df[
        feature_columns
    ]
    .corr(method="pearson")
)


# ============================================================
# 9. SAVE CSV OUTPUTS
# ============================================================

motor_kinematic_features_df.to_csv(
    KINEMATIC_TABLE_DIR
    / "subject_video_hand_kinematic_features.csv",
    index=False
)

motor_feature_summary_df.to_csv(
    KINEMATIC_TABLE_DIR
    / "hand_kinematic_feature_summary.csv",
    index=False
)

motor_feature_correlation_df.to_csv(
    KINEMATIC_TABLE_DIR
    / "hand_kinematic_feature_correlations.csv",
    index=True,
    index_label="Feature"
)

outlier_flag_matrix.to_csv(
    KINEMATIC_TABLE_DIR
    / "hand_kinematic_feature_outlier_flags.csv",
    index=False
)

motor_qc_flagged_trials_df.to_csv(
    KINEMATIC_TABLE_DIR
    / "hand_kinematic_qc_flagged_trials.csv",
    index=False
)

motor_failed_trajectories_df.to_csv(
    KINEMATIC_TABLE_DIR
    / "hand_kinematic_failed_or_excluded_trajectories.csv",
    index=False
)


# ============================================================
# 10. FIGURE: FEATURE DISTRIBUTIONS
# ============================================================

n_features = len(
    feature_columns
)

n_columns = 4
n_rows = int(
    np.ceil(
        n_features / n_columns
    )
)

fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(
        16,
        max(4, n_rows * 3)
    )
)

axes = np.atleast_1d(
    axes
).ravel()

for axis, column in zip(
    axes,
    feature_columns
):
    values = pd.to_numeric(
        motor_kinematic_features_df[column],
        errors="coerce"
    ).dropna()

    axis.hist(
        values,
        bins=35,
        edgecolor="black",
        alpha=0.80
    )

    axis.axvline(
        values.median(),
        linestyle="--",
        linewidth=1
    )

    axis.set_title(
        column,
        fontsize=9
    )

    axis.set_ylabel(
        "Frequency"
    )

for axis in axes[
    len(feature_columns):
]:
    axis.axis("off")

fig.suptitle(
    "Distributions of subject × video hand-kinematic features",
    fontsize=16,
    y=1.002
)

fig.tight_layout()

fig.savefig(
    KINEMATIC_FIG_DIR
    / "hand_kinematic_feature_distributions.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 11. FIGURE: FEATURE CORRELATION MATRIX
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        max(12, 0.45 * len(feature_columns)),
        max(10, 0.45 * len(feature_columns))
    )
)

image = ax.imshow(
    motor_feature_correlation_df,
    aspect="auto",
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(
    np.arange(
        len(feature_columns)
    )
)

ax.set_xticklabels(
    feature_columns,
    rotation=90,
    fontsize=8
)

ax.set_yticks(
    np.arange(
        len(feature_columns)
    )
)

ax.set_yticklabels(
    feature_columns,
    fontsize=8
)

fig.colorbar(
    image,
    ax=ax,
    label="Pearson correlation"
)

ax.set_title(
    "Correlations among hand-kinematic features"
)

fig.tight_layout()

fig.savefig(
    KINEMATIC_FIG_DIR
    / "hand_kinematic_feature_correlation_matrix.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 12. FIGURE: QC COUNTS
# ============================================================

qc_count_df = pd.DataFrame({
    "QCFlag": [
        "Impossible value",
        "Extreme feature outlier",
        "High missingness",
        "Insufficient valid data",
        "Long internal gap",
        "Any QC flag"
    ],
    "NObservations": [
        int(
            motor_kinematic_features_df[
                "ImpossibleValueFlag"
            ].sum()
        ),
        int(
            motor_kinematic_features_df[
                "ExtremeFeatureOutlierFlag"
            ].sum()
        ),
        int(
            motor_kinematic_features_df[
                "HighMissingnessFlag"
            ].sum()
        ),
        int(
            motor_kinematic_features_df[
                "InsufficientValidDataFlag"
            ].sum()
        ),
        int(
            motor_kinematic_features_df[
                "LongGapDetected"
            ].sum()
        ),
        int(
            motor_kinematic_features_df[
                "AnyQCFlag"
            ].sum()
        )
    ]
})

qc_count_df.to_csv(
    KINEMATIC_TABLE_DIR
    / "hand_kinematic_qc_flag_counts.csv",
    index=False
)

fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.bar(
    qc_count_df["QCFlag"],
    qc_count_df["NObservations"]
)

ax.set_ylabel(
    "Number of subject × video observations"
)

ax.set_title(
    "Hand-kinematic quality-control flags"
)

ax.tick_params(
    axis="x",
    rotation=35
)

fig.tight_layout()

fig.savefig(
    KINEMATIC_FIG_DIR
    / "hand_kinematic_qc_flag_counts.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 13. PLOT SELECTED FLAGGED TRAJECTORIES
# ============================================================

flagged_plot_rows = (
    motor_qc_flagged_trials_df
    .head(
        N_FLAGGED_TRAJECTORY_PLOTS
    )
)

for _, row in flagged_plot_rows.iterrows():
    cache_key = (
        str(row["Subject"]),
        str(row["Video"])
    )

    if cache_key not in trajectory_cache:
        continue

    trajectory_data = trajectory_cache[
        cache_key
    ]

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(12, 7),
        sharex=True
    )

    axes[0].plot(
        trajectory_data["Time"],
        trajectory_data["PositionRaw"],
        linewidth=1,
        alpha=0.65,
        label="Cleaned raw position"
    )

    axes[0].plot(
        trajectory_data["Time"],
        trajectory_data["PositionSmooth"],
        linewidth=2,
        label="Smoothed position"
    )

    axes[0].set_ylabel(
        "Behavioral rating"
    )

    axes[0].legend(
        frameon=False
    )

    axes[1].plot(
        trajectory_data["Time"],
        trajectory_data["Velocity"],
        linewidth=1
    )

    axes[1].axhline(
        PAUSE_VELOCITY_THRESHOLD,
        linestyle="--",
        linewidth=1
    )

    axes[1].axhline(
        -PAUSE_VELOCITY_THRESHOLD,
        linestyle="--",
        linewidth=1
    )

    axes[1].set_xlabel(
        "Time (s)"
    )

    axes[1].set_ylabel(
        "Velocity\n(rating units/s)"
    )

    fig.suptitle(
        f"QC trajectory | Subject={row['Subject']} | "
        f"{row['Video']}\n"
        f"N feature outliers={row['NFeatureOutliers']}, "
        f"missing={row['MissingProportion']:.3f}, "
        f"long gap={row['LongGapDetected']}"
    )

    fig.tight_layout()

    figure_name = (
        f"{safe_filename(row['Subject'])}__"
        f"{safe_filename(row['Video'])}__"
        "kinematic_qc.png"
    )

    fig.savefig(
        KINEMATIC_QC_FIG_DIR
        / figure_name,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 14. PRINT RESULTS
# ============================================================

print("\n" + "=" * 90)
print("HAND-KINEMATIC FEATURE EXTRACTION COMPLETE")
print("=" * 90)

print(
    f"\nSubject × video × modality observations: "
    f"{len(motor_kinematic_features_df)}"
)

print(
    f"Unique subjects: "
    f"{motor_kinematic_features_df['Subject'].nunique()}"
)

print(
    f"Unique video columns: "
    f"{motor_kinematic_features_df['Video'].nunique()}"
)

print(
    f"Observations with any QC flag: "
    f"{motor_kinematic_features_df['AnyQCFlag'].sum()} "
    f"({motor_kinematic_features_df['AnyQCFlag'].mean():.1%})"
)

print(
    f"Failed/excluded trajectories: "
    f"{len(motor_failed_trajectories_df)}"
)

print("\nTime-source frequencies:")
print(
    motor_kinematic_features_df[
        "TimeSource"
    ].value_counts(
        dropna=False
    )
)

print("\nFeature summary:")
display(
    motor_feature_summary_df.round(5)
)

print("\nPreview of the subject × video feature table:")
display(
    motor_kinematic_features_df.head(20)
)

print(
    f"\nAll Cell 1 outputs saved to:\n{KINEMATIC_DIR}"
)

Behavioral data folder:
C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Behavioral Data\data

Motor-control output folder:
C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Motor_Control

Found 42 behavioral CSV files.
Processed 10/42 files.
Processed 20/42 files.
Processed 30/42 files.
Processed 40/42 files.
Processed 42/42 files.

Extracted features for 2352 subject × video × modality observations.
Failed or excluded trajectories: 0

HAND-KINEMATIC FEATURE EXTRACTION COMPLETE

Subject × video × modality observations: 2352
Unique subjects: 42
Unique video columns: 56
Observations with any QC flag: 675 (28.7%)
Failed/excluded trajectories: 0

Time-source frequencies:
TimeSource
row/sample index converted at 60 Hz    2352
Name: count, dtype: int64

Feature summary:


,Feature,N,MissingN,MissingProportion,Mean,SD,Median,Q1,Q3,IQR,Minimum,Maximum
0,MeanAbsVelocity,2352,0,0.0,1.086800e-01,1.316200e-01,6.976000e-02,4.329000e-02,1.139900e-01,7.070000e-02,1.013000e-02,1.447350e+00
1,MedianAbsVelocity,2352,0,0.0,2.970000e-02,8.608000e-02,0.000000e+00,0.000000e+00,2.218000e-02,2.218000e-02,0.000000e+00,1.165510e+00
2,PeakAbsVelocity,2352,0,0.0,2.850420e+00,2.420710e+00,2.066110e+00,1.268760e+00,3.620880e+00,2.352120e+00,2.442700e-01,2.497552e+01
3,VelocitySD,2352,0,0.0,2.643900e-01,2.279100e-01,1.923500e-01,1.255200e-01,3.175200e-01,1.920100e-01,3.270000e-02,1.893250e+00
4,AbsVelocitySD,2352,0,0.0,2.374400e-01,1.910000e-01,1.776900e-01,1.152300e-01,2.864400e-01,1.712100e-01,3.113000e-02,1.432270e+00
5,ProportionMoving,2352,0,0.0,3.042100e-01,1.703400e-01,2.567900e-01,1.788600e-01,3.849500e-01,2.060900e-01,4.915000e-02,9.478300e-01
6,MeanAbsAcceleration,2352,0,0.0,1.413460e+00,2.059110e+00,7.715800e-01,4.887200e-01,1.369020e+00,8.802900e-01,1.314500e-01,2.064664e+01
7,PeakAbsAcceleration,2352,0,0.0,3.877608e+01,3.845982e+01,2.633670e+01,1.528873e+01,4.675105e+01,3.146233e+01,3.218680e+00,4.553821e+02
8,AccelerationSD,2352,0,0.0,3.385530e+00,3.652360e+00,2.150330e+00,1.354200e+00,3.813530e+00,2.459330e+00,4.100200e-01,3.062801e+01
9,MeanAbsJerk,2352,0,0.0,3.417371e+01,4.518165e+01,2.000121e+01,1.322748e+01,3.398802e+01,2.076054e+01,4.167790e+00,4.092669e+02



Preview of the subject × video feature table:


,Subject,Video,BaseVideo,Modality,TimeSource,DurationSec,NOriginalSamples,NValidSamples,EstimatedSamplingRateHz,MissingProportion,...,SmoothedPositionSD,DimensionlessJerk,Smoothness,MovementEntropy,ImpossibleValueFlag,NFeatureOutliers,ExtremeFeatureOutlierFlag,HighMissingnessFlag,InsufficientValidDataFlag,AnyQCFlag
0,pid001,22_arousal,22,Arousal,row/sample index converted at 60 Hz,53.716667,9254,3224,60.0,0.0,...,0.346646,1.088558e+13,-13.036852,0.176359,False,5,True,False,False,True
1,pid001,22_valence,22,Valence,row/sample index converted at 60 Hz,53.716667,9254,3224,60.0,0.0,...,0.364637,1.169762e+13,-13.068098,0.289939,False,0,False,False,False,False
2,pid001,54_arousal,54,Arousal,row/sample index converted at 60 Hz,32.550000,9254,1954,60.0,0.0,...,0.318427,4.705607e+11,-11.672616,0.332802,False,0,False,False,False,False
3,pid001,54_valence,54,Valence,row/sample index converted at 60 Hz,32.550000,9254,1954,60.0,0.0,...,0.212836,1.086699e+12,-12.036109,0.258291,False,0,False,False,False,False
4,pid001,88_baseline_arousal,88_baseline,Arousal,row/sample index converted at 60 Hz,70.533333,9254,4233,60.0,0.0,...,0.593057,3.110774e+13,-13.492868,0.225485,False,0,False,False,False,False
5,pid001,88_baseline_valence,88_baseline,Valence,row/sample index converted at 60 Hz,70.533333,9254,4233,60.0,0.0,...,0.303946,3.772667e+13,-13.576648,0.201401,False,0,False,False,False,False
6,pid001,123_baseline_arousal,123_baseline,Arousal,row/sample index converted at 60 Hz,31.016667,9254,1862,60.0,0.0,...,0.314383,2.988010e+11,-11.475382,0.370344,False,0,False,False,False,False
7,pid001,123_baseline_valence,123_baseline,Valence,row/sample index converted at 60 Hz,31.016667,9254,1862,60.0,0.0,...,0.430308,6.206013e+11,-11.792813,0.381572,False,0,False,False,False,False
8,pid001,Exp3_004_baseline_arousal,Exp3_004_baseline,Arousal,row/sample index converted at 60 Hz,57.133333,9254,3429,60.0,0.0,...,0.450130,4.296860e+12,-12.633151,0.331577,False,0,False,False,False,False
9,pid001,Exp3_004_baseline_valence,Exp3_004_baseline,Valence,row/sample index converted at 60 Hz,57.133333,9254,3429,60.0,0.0,...,0.300593,7.128274e+12,-12.852984,0.297520,False,0,False,False,False,False



All Cell 1 outputs saved to:
C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Motor_Control\01_hand_kinematic_features


In [122]:
# ============================================================
# 
# MOTOR-FEATURE PCA
#
# Requires:
#   motor_kinematic_features_df from Stage 2 Cell 1
#
# Produces:
#   motor_pca_scores_df
#   motor_pca_loadings_df
#   motor_pca_variance_df
#   motor_pca_parallel_analysis_df
#   motor_pca_diagnostic_correlations_df
#
# Saves all tables and figures to:
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data
# \Emotion Perception Data\Motor_Control
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import skew, pearsonr, pointbiserialr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. CHECK INPUT AND CREATE OUTPUT FOLDERS
# ============================================================

if "motor_kinematic_features_df" not in globals():
    raise ValueError(
        "motor_kinematic_features_df was not found. "
        "Run Stage 2 Cell 1 first."
    )

MOTOR_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
)

PCA_DIR = MOTOR_OUTPUT_DIR / "02_motor_feature_pca"
PCA_TABLE_DIR = PCA_DIR / "tables"
PCA_FIG_DIR = PCA_DIR / "figures"

for folder in [
    PCA_DIR,
    PCA_TABLE_DIR,
    PCA_FIG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

FIGURE_DPI = 300


# ============================================================
# 2. USER-ADJUSTABLE PCA SETTINGS
# ============================================================

# Do not use trials with these major QC problems in the primary PCA.
EXCLUDE_MAJOR_QC_FLAGS = True

# Long gaps are excluded from the PCA because derivative features can be
# unstable even after selecting the longest valid segment.
EXCLUDE_LONG_GAP_TRIALS = True

# Features with more missingness than this are excluded.
MAX_FEATURE_MISSING_PROPORTION = 0.20

# Near-zero variance threshold after transformation.
NEAR_ZERO_VARIANCE_THRESHOLD = 1e-10

# Remove the later member of a feature pair above this absolute correlation.
REDUNDANCY_CORRELATION_THRESHOLD = 0.98

# Log1p-transform a nonnegative feature when absolute skew exceeds this.
SKEWNESS_TRANSFORM_THRESHOLD = 1.0

# Horn parallel analysis.
N_PARALLEL_PERMUTATIONS = 500
PARALLEL_PERCENTILE = 95
PARALLEL_RANDOM_SEED = 12345

# Maximum number of components considered for retention and plotting.
MAX_COMPONENTS_TO_EVALUATE = 10

# Save at least PC1–PC3 even when fewer are retained.
MIN_COMPONENTS_TO_SAVE = 3

# Cap retained components to preserve model parsimony.
MAX_RETAINED_COMPONENTS = 5


# ============================================================
# 3. DEFINE NONREDUNDANT KINEMATIC FEATURE SET
# ============================================================

# Duration and raw counts are omitted from PCA.
# Rate-normalized and proportional measures are preferred.
candidate_motor_features = [
    # Movement speed/intensity
    "MeanAbsVelocity",
    "MedianAbsVelocity",
    "PeakAbsVelocity",
    "VelocitySD",
    "ProportionMoving",

    # Acceleration
    "MeanAbsAcceleration",
    "PeakAbsAcceleration",
    "AccelerationSD",

    # Jerk/roughness
    "MeanAbsJerk",
    "RMSJerk",
    "PeakAbsJerk",

    # Path and direction, normalized where possible
    "PathLengthPerSecond",
    "PathEfficiency",
    "ReversalRate",

    # Pausing/intermittency
    "PauseRate",
    "MeanPauseDuration",
    "PauseProportion",

    # Variability and complexity
    "PositionSD",
    "Smoothness",
    "MovementEntropy"
]

missing_candidate_columns = [
    column
    for column in candidate_motor_features
    if column not in motor_kinematic_features_df.columns
]

if missing_candidate_columns:
    raise ValueError(
        "The following expected motor-feature columns are missing:\n"
        f"{missing_candidate_columns}"
    )


# ============================================================
# 4. SELECT PCA OBSERVATIONS
# ============================================================

motor_pca_input_df = motor_kinematic_features_df.copy()

motor_pca_input_df["IncludedInPrimaryMotorPCA"] = True

if EXCLUDE_MAJOR_QC_FLAGS:
    major_qc_mask = (
        motor_pca_input_df[
            "ImpossibleValueFlag"
        ].fillna(False)
        | motor_pca_input_df[
            "HighMissingnessFlag"
        ].fillna(False)
        | motor_pca_input_df[
            "InsufficientValidDataFlag"
        ].fillna(False)
    )

    motor_pca_input_df.loc[
        major_qc_mask,
        "IncludedInPrimaryMotorPCA"
    ] = False

if EXCLUDE_LONG_GAP_TRIALS:
    motor_pca_input_df.loc[
        motor_pca_input_df[
            "LongGapDetected"
        ].fillna(False),
        "IncludedInPrimaryMotorPCA"
    ] = False

# Extreme feature values are retained but flagged.
# They are not silently winsorized or deleted.
pca_analysis_df = motor_pca_input_df[
    motor_pca_input_df[
        "IncludedInPrimaryMotorPCA"
    ]
].copy()

if len(pca_analysis_df) < 20:
    raise RuntimeError(
        f"Too few observations remained for PCA: n={len(pca_analysis_df)}"
    )

print(
    f"Primary motor PCA observations: "
    f"{len(pca_analysis_df)} of {len(motor_pca_input_df)}"
)


# ============================================================
# 5. REMOVE FEATURES WITH EXCESSIVE MISSINGNESS
# ============================================================

feature_missingness = (
    pca_analysis_df[
        candidate_motor_features
    ]
    .isna()
    .mean()
)

excluded_for_missingness = feature_missingness[
    feature_missingness
    > MAX_FEATURE_MISSING_PROPORTION
].index.tolist()

features_after_missingness = [
    column
    for column in candidate_motor_features
    if column not in excluded_for_missingness
]

if len(features_after_missingness) < 3:
    raise RuntimeError(
        "Fewer than three motor features remained after the "
        "missingness filter."
    )


# ============================================================
# 6. SKEWNESS-BASED TRANSFORMATION
# ============================================================

transformed_feature_df = pd.DataFrame(
    index=pca_analysis_df.index
)

transformation_rows = []

for column in features_after_missingness:
    values = pd.to_numeric(
        pca_analysis_df[column],
        errors="coerce"
    )

    finite_values = values[
        np.isfinite(values)
    ]

    if len(finite_values) < 3:
        transformation_rows.append({
            "OriginalFeature": column,
            "PCAFeature": column,
            "Transformation": "excluded: insufficient finite values",
            "OriginalSkewness": np.nan
        })

        continue

    original_skewness = skew(
        finite_values,
        bias=False,
        nan_policy="omit"
    )

    is_nonnegative = bool(
        finite_values.min() >= 0
    )

    if (
        is_nonnegative
        and np.isfinite(original_skewness)
        and abs(original_skewness)
            > SKEWNESS_TRANSFORM_THRESHOLD
    ):
        transformed_column = f"log1p_{column}"

        transformed_feature_df[
            transformed_column
        ] = np.log1p(
            values
        )

        transformation = "log1p"

    else:
        transformed_column = column

        transformed_feature_df[
            transformed_column
        ] = values

        transformation = "none"

    transformation_rows.append({
        "OriginalFeature": column,
        "PCAFeature": transformed_column,
        "Transformation": transformation,
        "OriginalSkewness": original_skewness
    })


motor_feature_transformations_df = pd.DataFrame(
    transformation_rows
)

pca_feature_columns = transformed_feature_df.columns.tolist()


# ============================================================
# 7. MEDIAN IMPUTATION
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

imputed_values = imputer.fit_transform(
    transformed_feature_df[
        pca_feature_columns
    ]
)

imputed_feature_df = pd.DataFrame(
    imputed_values,
    index=transformed_feature_df.index,
    columns=pca_feature_columns
)


# ============================================================
# 8. REMOVE NEAR-ZERO-VARIANCE FEATURES
# ============================================================

feature_variances = imputed_feature_df.var(
    axis=0,
    ddof=1
)

near_zero_variance_features = feature_variances[
    feature_variances
    <= NEAR_ZERO_VARIANCE_THRESHOLD
].index.tolist()

features_after_variance_filter = [
    column
    for column in pca_feature_columns
    if column not in near_zero_variance_features
]

if len(features_after_variance_filter) < 3:
    raise RuntimeError(
        "Fewer than three features remained after removing "
        "near-zero-variance variables."
    )


# ============================================================
# 9. REMOVE EXTREME REDUNDANCY
# ============================================================

redundancy_correlation_df = (
    imputed_feature_df[
        features_after_variance_filter
    ]
    .corr()
)

features_to_drop_for_redundancy = []
redundancy_pairs = []

for i, feature_i in enumerate(
    features_after_variance_filter
):
    if feature_i in features_to_drop_for_redundancy:
        continue

    for feature_j in features_after_variance_filter[
        i + 1:
    ]:
        if feature_j in features_to_drop_for_redundancy:
            continue

        correlation = redundancy_correlation_df.loc[
            feature_i,
            feature_j
        ]

        if (
            np.isfinite(correlation)
            and abs(correlation)
                >= REDUNDANCY_CORRELATION_THRESHOLD
        ):
            # Keep the feature appearing earlier in the prespecified list.
            features_to_drop_for_redundancy.append(
                feature_j
            )

            redundancy_pairs.append({
                "KeptFeature": feature_i,
                "DroppedFeature": feature_j,
                "Correlation": correlation
            })

motor_feature_redundancy_df = pd.DataFrame(
    redundancy_pairs
)

final_motor_pca_features = [
    column
    for column in features_after_variance_filter
    if column not in features_to_drop_for_redundancy
]

if len(final_motor_pca_features) < 3:
    raise RuntimeError(
        "Fewer than three features remained after redundancy removal."
    )

print(
    f"\nFinal motor PCA feature count: "
    f"{len(final_motor_pca_features)}"
)

print("\nFinal PCA features:")
for feature in final_motor_pca_features:
    print(f"  {feature}")


# ============================================================
# 10. STANDARDIZE FEATURES
# ============================================================

scaler = StandardScaler(
    with_mean=True,
    with_std=True
)

standardized_values = scaler.fit_transform(
    imputed_feature_df[
        final_motor_pca_features
    ]
)

standardized_motor_feature_df = pd.DataFrame(
    standardized_values,
    index=imputed_feature_df.index,
    columns=final_motor_pca_features
)


# ============================================================
# 11. FIT OBSERVED PCA
# ============================================================

maximum_components = min(
    MAX_COMPONENTS_TO_EVALUATE,
    standardized_motor_feature_df.shape[0] - 1,
    standardized_motor_feature_df.shape[1]
)

if maximum_components < 1:
    raise RuntimeError(
        "No PCA components could be estimated."
    )

motor_pca_model = PCA(
    n_components=maximum_components,
    svd_solver="full"
)

observed_scores = motor_pca_model.fit_transform(
    standardized_motor_feature_df
)

observed_evr = motor_pca_model.explained_variance_ratio_.copy()
observed_eigenvalues = motor_pca_model.explained_variance_.copy()


# ============================================================
# 12. APPLY CONSISTENT PC SIGN CONVENTION
# ============================================================

# Flip each component so its largest absolute feature weight is positive.
for component_index in range(
    motor_pca_model.components_.shape[0]
):
    weights = motor_pca_model.components_[
        component_index,
        :
    ]

    largest_weight_index = int(
        np.argmax(
            np.abs(weights)
        )
    )

    if weights[
        largest_weight_index
    ] < 0:
        motor_pca_model.components_[
            component_index,
            :
        ] *= -1

        observed_scores[
            :,
            component_index
        ] *= -1


# ============================================================
# 13. HORN'S PARALLEL ANALYSIS
# ============================================================

rng = np.random.default_rng(
    PARALLEL_RANDOM_SEED
)

null_explained_variance = np.zeros(
    (
        N_PARALLEL_PERMUTATIONS,
        maximum_components
    ),
    dtype=float
)

standardized_array = standardized_motor_feature_df.to_numpy(
    dtype=float
)

for permutation_index in range(
    N_PARALLEL_PERMUTATIONS
):
    permuted_array = np.column_stack([
        rng.permutation(
            standardized_array[:, column_index]
        )
        for column_index in range(
            standardized_array.shape[1]
        )
    ])

    permuted_pca = PCA(
        n_components=maximum_components,
        svd_solver="full"
    ).fit(
        permuted_array
    )

    null_explained_variance[
        permutation_index,
        :
    ] = permuted_pca.explained_variance_ratio_

    if (
        (permutation_index + 1) % 100 == 0
        or permutation_index + 1
            == N_PARALLEL_PERMUTATIONS
    ):
        print(
            f"Parallel analysis: "
            f"{permutation_index + 1}/"
            f"{N_PARALLEL_PERMUTATIONS}"
        )


parallel_mean = np.mean(
    null_explained_variance,
    axis=0
)

parallel_threshold = np.percentile(
    null_explained_variance,
    PARALLEL_PERCENTILE,
    axis=0
)

parallel_significant = (
    observed_evr
    > parallel_threshold
)

n_parallel_components = int(
    np.sum(
        parallel_significant
    )
)

# Retain consecutive significant components from PC1 onward.
n_consecutive_parallel_components = 0

for is_significant in parallel_significant:
    if is_significant:
        n_consecutive_parallel_components += 1
    else:
        break

n_retained_components = max(
    1,
    min(
        n_consecutive_parallel_components,
        MAX_RETAINED_COMPONENTS
    )
)

n_components_to_save = min(
    maximum_components,
    max(
        MIN_COMPONENTS_TO_SAVE,
        n_retained_components
    )
)


# ============================================================
# 14. VARIANCE TABLE
# ============================================================

motor_pca_variance_df = pd.DataFrame({
    "Component": [
        f"MotorPC{index + 1}"
        for index in range(
            maximum_components
        )
    ],
    "Eigenvalue": observed_eigenvalues,
    "ExplainedVarianceRatio": observed_evr,
    "ExplainedVariancePercent": (
        observed_evr * 100
    ),
    "CumulativeVarianceRatio": np.cumsum(
        observed_evr
    ),
    "CumulativeVariancePercent": (
        np.cumsum(
            observed_evr
        ) * 100
    ),
    "ParallelMeanEVR": parallel_mean,
    f"Parallel{PARALLEL_PERCENTILE}thPercentileEVR": (
        parallel_threshold
    ),
    "ExceedsParallelThreshold": (
        parallel_significant
    ),
    "RetainedForModeling": [
        index < n_retained_components
        for index in range(
            maximum_components
        )
    ]
})

motor_pca_parallel_analysis_df = (
    motor_pca_variance_df.copy()
)


# ============================================================
# 15. COMPONENT WEIGHTS AND LOADINGS
# ============================================================

component_names = [
    f"MotorPC{index + 1}"
    for index in range(
        n_components_to_save
    )
]

component_weight_df = pd.DataFrame(
    motor_pca_model.components_[
        :n_components_to_save,
        :
    ].T,
    index=final_motor_pca_features,
    columns=component_names
)

# Conventional loading:
# component weight × sqrt(component eigenvalue)
motor_pca_loadings_df = (
    component_weight_df
    * np.sqrt(
        observed_eigenvalues[
            :n_components_to_save
        ]
    )
)

motor_pca_loadings_df.index.name = "Feature"
component_weight_df.index.name = "Feature"


# ============================================================
# 16. COMPONENT SCORES
# ============================================================

score_columns = {
    f"MotorPC{index + 1}": (
        observed_scores[
            :,
            index
        ]
    )
    for index in range(
        n_components_to_save
    )
}

score_df = pd.DataFrame(
    score_columns,
    index=pca_analysis_df.index
)

motor_pca_scores_df = pd.concat(
    [
        pca_analysis_df[
            [
                "Subject",
                "Video",
                "BaseVideo",
                "Modality",
                "DurationSec",
                "AnyQCFlag",
                "ExtremeFeatureOutlierFlag"
            ]
        ].copy(),
        score_df
    ],
    axis=1
)

motor_pca_scores_df[
    "NRetainedMotorPCs"
] = n_retained_components


# ============================================================
# 17. OPTIONAL MERGES WITH EXISTING BEHAVIORAL RESULTS
# ============================================================

# Merge PC1 behavioral-performance alignment when available.
if "pc_alignment_df" in globals():
    behavioral_alignment_columns = [
        column
        for column in [
            "Subject",
            "Video",
            "Modality",
            "r_PC1",
            "abs_r_PC1",
            "r_PC2",
            "abs_r_PC2",
            "Best_PC"
        ]
        if column in pc_alignment_df.columns
    ]

    if {
        "Subject",
        "Video",
        "Modality"
    }.issubset(
        behavioral_alignment_columns
    ):
        alignment_merge_df = (
            pc_alignment_df[
                behavioral_alignment_columns
            ]
            .copy()
        )

        alignment_merge_df[
            "Subject"
        ] = alignment_merge_df[
            "Subject"
        ].astype(str)

        alignment_merge_df[
            "Video"
        ] = alignment_merge_df[
            "Video"
        ].astype(str)

        motor_pca_scores_df = (
            motor_pca_scores_df
            .merge(
                alignment_merge_df,
                on=[
                    "Subject",
                    "Video",
                    "Modality"
                ],
                how="left",
                validate="one_to_one"
            )
        )


# Merge behavioral subgroup labels when available.
if "behavioral_subgroup_df" in globals():
    subgroup_columns = [
        column
        for column in [
            "Subject",
            "Video",
            "Modality",
            "Behavioral_Subgroup",
            "Is_Dominant_Consensus",
            "Is_Alternative_Consensus"
        ]
        if column in behavioral_subgroup_df.columns
    ]

    if {
        "Subject",
        "Video",
        "Modality"
    }.issubset(
        subgroup_columns
    ):
        subgroup_merge_df = (
            behavioral_subgroup_df[
                subgroup_columns
            ]
            .copy()
        )

        subgroup_merge_df[
            "Subject"
        ] = subgroup_merge_df[
            "Subject"
        ].astype(str)

        subgroup_merge_df[
            "Video"
        ] = subgroup_merge_df[
            "Video"
        ].astype(str)

        motor_pca_scores_df = (
            motor_pca_scores_df
            .merge(
                subgroup_merge_df,
                on=[
                    "Subject",
                    "Video",
                    "Modality"
                ],
                how="left",
                validate="one_to_one"
            )
        )


# ============================================================
# 18. DIAGNOSTIC ASSOCIATIONS
# ============================================================

diagnostic_rows = []

for pc_name in component_names:
    # Correlation with video duration.
    diagnostic_data = motor_pca_scores_df[
        [
            pc_name,
            "DurationSec"
        ]
    ].dropna()

    if (
        len(diagnostic_data) >= 5
        and diagnostic_data[
            pc_name
        ].std(ddof=1) > 0
        and diagnostic_data[
            "DurationSec"
        ].std(ddof=1) > 0
    ):
        correlation, p_value = pearsonr(
            diagnostic_data[
                pc_name
            ],
            diagnostic_data[
                "DurationSec"
            ]
        )

        diagnostic_rows.append({
            "MotorPC": pc_name,
            "DiagnosticVariable": "DurationSec",
            "Test": "Pearson correlation",
            "Statistic": correlation,
            "PValue": p_value,
            "N": len(diagnostic_data)
        })

    # Correlation with behavioral PC1 alignment.
    if "r_PC1" in motor_pca_scores_df.columns:
        diagnostic_data = motor_pca_scores_df[
            [
                pc_name,
                "r_PC1"
            ]
        ].dropna()

        if (
            len(diagnostic_data) >= 5
            and diagnostic_data[
                pc_name
            ].std(ddof=1) > 0
            and diagnostic_data[
                "r_PC1"
            ].std(ddof=1) > 0
        ):
            correlation, p_value = pearsonr(
                diagnostic_data[
                    pc_name
                ],
                diagnostic_data[
                    "r_PC1"
                ]
            )

            diagnostic_rows.append({
                "MotorPC": pc_name,
                "DiagnosticVariable": (
                    "Behavioral_PC1_Alignment"
                ),
                "Test": "Pearson correlation",
                "Statistic": correlation,
                "PValue": p_value,
                "N": len(diagnostic_data)
            })

    # Point-biserial association with alternative-cluster membership.
    if (
        "Is_Alternative_Consensus"
        in motor_pca_scores_df.columns
    ):
        diagnostic_data = motor_pca_scores_df[
            [
                pc_name,
                "Is_Alternative_Consensus"
            ]
        ].dropna()

        diagnostic_data[
            "Is_Alternative_Consensus"
        ] = diagnostic_data[
            "Is_Alternative_Consensus"
        ].astype(int)

        if (
            len(diagnostic_data) >= 5
            and diagnostic_data[
                "Is_Alternative_Consensus"
            ].nunique() == 2
            and diagnostic_data[
                pc_name
            ].std(ddof=1) > 0
        ):
            correlation, p_value = pointbiserialr(
                diagnostic_data[
                    "Is_Alternative_Consensus"
                ],
                diagnostic_data[
                    pc_name
                ]
            )

            diagnostic_rows.append({
                "MotorPC": pc_name,
                "DiagnosticVariable": (
                    "AlternativeClusterMembership"
                ),
                "Test": "Point-biserial correlation",
                "Statistic": correlation,
                "PValue": p_value,
                "N": len(diagnostic_data)
            })


motor_pca_diagnostic_correlations_df = pd.DataFrame(
    diagnostic_rows
)


# ============================================================
# 19. COMPONENT-SCORE CORRELATIONS
# ============================================================

motor_pc_correlation_df = (
    motor_pca_scores_df[
        component_names
    ]
    .corr()
)


# ============================================================
# 20. SAVE TABLES
# ============================================================

motor_pca_scores_df.to_csv(
    PCA_TABLE_DIR
    / "subject_video_motor_pca_scores.csv",
    index=False
)

motor_pca_loadings_df.to_csv(
    PCA_TABLE_DIR
    / "motor_pca_loadings.csv",
    index=True
)

component_weight_df.to_csv(
    PCA_TABLE_DIR
    / "motor_pca_component_weights.csv",
    index=True
)

motor_pca_variance_df.to_csv(
    PCA_TABLE_DIR
    / "motor_pca_explained_variance.csv",
    index=False
)

motor_pca_parallel_analysis_df.to_csv(
    PCA_TABLE_DIR
    / "motor_pca_parallel_analysis.csv",
    index=False
)

motor_feature_transformations_df.to_csv(
    PCA_TABLE_DIR
    / "motor_feature_transformations.csv",
    index=False
)

motor_feature_redundancy_df.to_csv(
    PCA_TABLE_DIR
    / "motor_feature_redundancy_removed.csv",
    index=False
)

pd.DataFrame({
    "Feature": excluded_for_missingness
}).to_csv(
    PCA_TABLE_DIR
    / "motor_features_excluded_for_missingness.csv",
    index=False
)

pd.DataFrame({
    "Feature": near_zero_variance_features
}).to_csv(
    PCA_TABLE_DIR
    / "motor_features_excluded_for_near_zero_variance.csv",
    index=False
)

pd.DataFrame({
    "Feature": final_motor_pca_features
}).to_csv(
    PCA_TABLE_DIR
    / "motor_pca_final_feature_list.csv",
    index=False
)

motor_pca_diagnostic_correlations_df.to_csv(
    PCA_TABLE_DIR
    / "motor_pca_diagnostic_correlations.csv",
    index=False
)

motor_pc_correlation_df.to_csv(
    PCA_TABLE_DIR
    / "motor_pc_score_correlations.csv",
    index=True,
    index_label="MotorPC"
)


# ============================================================
# 21. FIGURE: SCREE + PARALLEL ANALYSIS
# ============================================================

component_numbers = np.arange(
    1,
    maximum_components + 1
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    component_numbers,
    observed_evr * 100,
    marker="o",
    linewidth=2,
    label="Observed PCA"
)

ax.plot(
    component_numbers,
    parallel_threshold * 100,
    marker="o",
    linestyle="--",
    linewidth=2,
    label=(
        f"Parallel-analysis "
        f"{PARALLEL_PERCENTILE}th percentile"
    )
)

ax.plot(
    component_numbers,
    parallel_mean * 100,
    marker="o",
    linestyle=":",
    linewidth=1.5,
    label="Parallel-analysis mean"
)

ax.axvline(
    n_retained_components + 0.5,
    linestyle="--",
    linewidth=1,
    label=(
        f"Retained components = "
        f"{n_retained_components}"
    )
)

ax.set_xticks(
    component_numbers
)

ax.set_xlabel(
    "Motor principal component"
)

ax.set_ylabel(
    "Explained variance (%)"
)

ax.set_title(
    "Motor-feature PCA scree plot and Horn parallel analysis"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

fig.savefig(
    PCA_FIG_DIR
    / "motor_pca_scree_parallel_analysis.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 22. FIGURE: CUMULATIVE EXPLAINED VARIANCE
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.plot(
    component_numbers,
    np.cumsum(observed_evr) * 100,
    marker="o",
    linewidth=2
)

ax.axhline(
    70,
    linestyle="--",
    linewidth=1,
    label="70% cumulative variance"
)

ax.axhline(
    80,
    linestyle=":",
    linewidth=1,
    label="80% cumulative variance"
)

ax.set_xticks(
    component_numbers
)

ax.set_xlabel(
    "Number of components"
)

ax.set_ylabel(
    "Cumulative explained variance (%)"
)

ax.set_title(
    "Cumulative variance explained by motor PCs"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

fig.savefig(
    PCA_FIG_DIR
    / "motor_pca_cumulative_variance.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 23. FIGURE: LOADING HEATMAP
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        max(
            8,
            1.5 * len(component_names)
        ),
        max(
            8,
            0.45 * len(final_motor_pca_features)
        )
    )
)

loading_values = motor_pca_loadings_df[
    component_names
].to_numpy(dtype=float)

loading_limit = np.nanmax(
    np.abs(
        loading_values
    )
)

image = ax.imshow(
    loading_values,
    aspect="auto",
    cmap="coolwarm",
    vmin=-loading_limit,
    vmax=loading_limit
)

ax.set_xticks(
    np.arange(
        len(component_names)
    )
)

ax.set_xticklabels(
    component_names
)

ax.set_yticks(
    np.arange(
        len(final_motor_pca_features)
    )
)

ax.set_yticklabels(
    final_motor_pca_features
)

fig.colorbar(
    image,
    ax=ax,
    label="PCA loading"
)

ax.set_title(
    "Motor-feature PCA loadings"
)

fig.tight_layout()

fig.savefig(
    PCA_FIG_DIR
    / "motor_pca_loading_heatmap.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 24. FIGURE: FIRST TWO MOTOR-PC SCORES
# ============================================================

if len(component_names) >= 2:
    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    if (
        "Is_Alternative_Consensus"
        in motor_pca_scores_df.columns
    ):
        dominant_mask = (
            motor_pca_scores_df[
                "Is_Alternative_Consensus"
            ] == False
        )

        alternative_mask = (
            motor_pca_scores_df[
                "Is_Alternative_Consensus"
            ] == True
        )

        ax.scatter(
            motor_pca_scores_df.loc[
                dominant_mask,
                "MotorPC1"
            ],
            motor_pca_scores_df.loc[
                dominant_mask,
                "MotorPC2"
            ],
            alpha=0.35,
            label="Dominant consensus"
        )

        ax.scatter(
            motor_pca_scores_df.loc[
                alternative_mask,
                "MotorPC1"
            ],
            motor_pca_scores_df.loc[
                alternative_mask,
                "MotorPC2"
            ],
            alpha=0.55,
            label="Alternative consensus"
        )

        ax.legend(
            frameon=False
        )

    else:
        ax.scatter(
            motor_pca_scores_df[
                "MotorPC1"
            ],
            motor_pca_scores_df[
                "MotorPC2"
            ],
            alpha=0.40
        )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "MotorPC1 score"
    )

    ax.set_ylabel(
        "MotorPC2 score"
    )

    ax.set_title(
        "Subject × video motor-component scores"
    )

    fig.tight_layout()

    fig.savefig(
        PCA_FIG_DIR
        / "motor_pc1_by_motor_pc2_scores.png",
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 25. FIGURE: COMPONENT SCORE DISTRIBUTIONS
# ============================================================

fig, axes = plt.subplots(
    len(component_names),
    1,
    figsize=(
        10,
        max(
            4,
            3 * len(component_names)
        )
    )
)

axes = np.atleast_1d(
    axes
)

for axis, pc_name in zip(
    axes,
    component_names
):
    axis.hist(
        motor_pca_scores_df[
            pc_name
        ].dropna(),
        bins=35,
        edgecolor="black",
        alpha=0.80
    )

    axis.axvline(
        0,
        linestyle="--",
        linewidth=1
    )

    explained_percent = motor_pca_variance_df.loc[
        motor_pca_variance_df[
            "Component"
        ] == pc_name,
        "ExplainedVariancePercent"
    ].iloc[0]

    axis.set_title(
        f"{pc_name}: "
        f"{explained_percent:.1f}% variance"
    )

    axis.set_ylabel(
        "Frequency"
    )

axes[-1].set_xlabel(
    "Component score"
)

fig.tight_layout()

fig.savefig(
    PCA_FIG_DIR
    / "motor_pc_score_distributions.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 26. PRINT INTERPRETIVE RESULTS
# ============================================================

print("\n" + "=" * 90)
print("MOTOR-FEATURE PCA COMPLETE")
print("=" * 90)

print(
    f"\nPCA observations: "
    f"{len(pca_analysis_df)}"
)

print(
    f"Final PCA features: "
    f"{len(final_motor_pca_features)}"
)

print(
    f"Components exceeding the parallel-analysis threshold: "
    f"{n_parallel_components}"
)

print(
    f"Consecutive components retained from PC1 onward: "
    f"{n_retained_components}"
)

print("\nExplained variance and retention:")
display(
    motor_pca_variance_df.round(5)
)

print("\nMotor PCA loadings:")
display(
    motor_pca_loadings_df.round(4)
)

print("\nLargest absolute loadings by component:")

for pc_name in component_names:
    ranked_loadings = (
        motor_pca_loadings_df[
            pc_name
        ]
        .sort_values(
            key=lambda series: np.abs(series),
            ascending=False
        )
        .head(6)
    )

    print(f"\n{pc_name}:")
    print(
        ranked_loadings.round(4)
    )

if not motor_pca_diagnostic_correlations_df.empty:
    print("\nDiagnostic associations:")
    display(
        motor_pca_diagnostic_correlations_df.round(5)
    )

print("\nMotor-PC score correlations:")
display(
    motor_pc_correlation_df.round(5)
)

print("\nPreview of subject × video motor-PC scores:")
display(
    motor_pca_scores_df.head(20)
)

print(
    "\nMotor trajectory similarity and motor-region ISC have not "
    "yet been calculated, so their multicollinearity with the retained "
    "motor PCs should be assessed after those variables are added."
)

print(
    f"\nAll Cell 2 outputs saved to:\n{PCA_DIR}"
)

Primary motor PCA observations: 2352 of 2352

Final motor PCA feature count: 17

Final PCA features:
  log1p_MeanAbsVelocity
  log1p_MedianAbsVelocity
  log1p_PeakAbsVelocity
  log1p_VelocitySD
  log1p_ProportionMoving
  log1p_MeanAbsAcceleration
  log1p_PeakAbsAcceleration
  log1p_AccelerationSD
  log1p_MeanAbsJerk
  log1p_PeakAbsJerk
  log1p_PathEfficiency
  log1p_ReversalRate
  PauseRate
  log1p_MeanPauseDuration
  PositionSD
  Smoothness
  MovementEntropy
Parallel analysis: 100/500
Parallel analysis: 200/500
Parallel analysis: 300/500
Parallel analysis: 400/500
Parallel analysis: 500/500

MOTOR-FEATURE PCA COMPLETE

PCA observations: 2352
Final PCA features: 17
Components exceeding the parallel-analysis threshold: 3
Consecutive components retained from PC1 onward: 3

Explained variance and retention:


,Component,Eigenvalue,ExplainedVarianceRatio,ExplainedVariancePercent,CumulativeVarianceRatio,CumulativeVariancePercent,ParallelMeanEVR,Parallel95thPercentileEVR,ExceedsParallelThreshold,RetainedForModeling
0,MotorPC1,9.05121,0.53220,53.21977,0.53220,53.21977,0.06749,0.06906,True,True
1,MotorPC2,3.13980,0.18462,18.46156,0.71681,71.68134,0.06585,0.06710,True,True
2,MotorPC3,1.63476,0.09612,9.61212,0.81293,81.29346,0.06456,0.06553,True,True
3,MotorPC4,1.05108,0.06180,6.18020,0.87474,87.47366,0.06345,0.06434,False,False
4,MotorPC5,0.80085,0.04709,4.70886,0.92183,92.18253,0.06240,0.06326,False,False
5,MotorPC6,0.48272,0.02838,2.83830,0.95021,95.02083,0.06144,0.06219,False,False
6,MotorPC7,0.27822,0.01636,1.63589,0.96657,96.65672,0.06051,0.06124,False,False
7,MotorPC8,0.20542,0.01208,1.20784,0.97865,97.86455,0.05959,0.06028,False,False
8,MotorPC9,0.12603,0.00741,0.74104,0.98606,98.60560,0.05870,0.05937,False,False
9,MotorPC10,0.11206,0.00659,0.65891,0.99265,99.26451,0.05783,0.05857,False,False



Motor PCA loadings:


,MotorPC1,MotorPC2,MotorPC3
Feature,,,
log1p_MeanAbsVelocity,0.9310,0.1732,-0.2250
log1p_MedianAbsVelocity,0.7446,0.3355,-0.3824
log1p_PeakAbsVelocity,0.7459,-0.5978,0.1597
log1p_VelocitySD,0.9413,-0.1771,-0.0829
log1p_ProportionMoving,0.8132,0.4956,0.1045
log1p_MeanAbsAcceleration,0.9787,0.1208,-0.0833
log1p_PeakAbsAcceleration,0.7854,-0.5600,0.1737
log1p_AccelerationSD,0.9526,-0.2268,0.0105
log1p_MeanAbsJerk,0.9760,0.1099,0.0816



Largest absolute loadings by component:

MotorPC1:
Feature
log1p_MeanAbsAcceleration    0.9787
log1p_MeanAbsJerk            0.9760
log1p_AccelerationSD         0.9526
log1p_VelocitySD             0.9413
log1p_MeanAbsVelocity        0.9310
log1p_ReversalRate           0.8808
Name: MotorPC1, dtype: float64

MotorPC2:
Feature
MovementEntropy              0.8526
Smoothness                   0.6337
log1p_PeakAbsVelocity       -0.5978
log1p_MeanPauseDuration     -0.5705
log1p_PeakAbsAcceleration   -0.5600
log1p_PeakAbsJerk           -0.5570
Name: MotorPC2, dtype: float64

MotorPC3:
Feature
PauseRate                  0.8477
log1p_MeanPauseDuration   -0.6941
log1p_MedianAbsVelocity   -0.3824
log1p_MeanAbsVelocity     -0.2250
log1p_PathEfficiency      -0.2177
log1p_PeakAbsJerk          0.2015
Name: MotorPC3, dtype: float64

Diagnostic associations:


...


Motor-PC score correlations:


,MotorPC1,MotorPC2,MotorPC3
MotorPC1,1.0,-0.0,0.0
MotorPC2,-0.0,1.0,0.0
MotorPC3,0.0,0.0,1.0



Preview of subject × video motor-PC scores:


,Subject,Video,BaseVideo,Modality,DurationSec,AnyQCFlag,ExtremeFeatureOutlierFlag,MotorPC1,MotorPC2,MotorPC3,NRetainedMotorPCs
0,pid001,22_arousal,22,Arousal,53.716667,True,True,5.224689,-1.743930,2.562036,3
1,pid001,22_valence,22,Valence,53.716667,False,False,2.436830,-0.709944,1.477828,3
2,pid001,54_arousal,54,Arousal,32.550000,False,False,-0.472819,1.603996,1.492435,3
3,pid001,54_valence,54,Valence,32.550000,False,False,0.422923,0.671128,1.527846,3
4,pid001,88_baseline_arousal,88_baseline,Arousal,70.533333,False,False,1.912147,-2.092068,1.101707,3
5,pid001,88_baseline_valence,88_baseline,Valence,70.533333,False,False,0.607846,-1.174081,1.685889,3
6,pid001,123_baseline_arousal,123_baseline,Arousal,31.016667,False,False,0.754856,1.922472,2.116695,3
7,pid001,123_baseline_valence,123_baseline,Valence,31.016667,False,False,0.760471,1.178514,1.224534,3
8,pid001,Exp3_004_baseline_arousal,Exp3_004_baseline,Arousal,57.133333,False,False,3.289324,-0.117070,0.970455,3
9,pid001,Exp3_004_baseline_valence,Exp3_004_baseline,Valence,57.133333,False,False,2.224082,0.163515,1.208629,3



Motor trajectory similarity and motor-region ISC have not yet been calculated, so their multicollinearity with the retained motor PCs should be assessed after those variables are added.

All Cell 2 outputs saved to:
C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Motor_Control\02_motor_feature_pca


In [124]:
# ============================================================
# MOTOR PCA × NEURAL ISC CORRELATION ANALYSIS
# USING VIDEO-MAPPING CSV
#
# Required existing notebook objects:
#   motor_pca_scores_df
#   video_roi_isc_df
#
# Mapping CSV required columns:
#   video
#   trigger_number
#
# Tests Motor PCs against:
#   - Mean ROI ISC
#   - Original Network Balance
#   - Full LASSO Network Balance
#   - Each individual ROI ISC
#
# Saves outputs to:
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data
# \Emotion Perception Data\Motor_Control
# \02_motor_feature_pca\correlations
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

if "motor_pca_scores_df" not in globals():
    raise ValueError(
        "motor_pca_scores_df was not found. "
        "Run the motor-feature PCA cell first."
    )

if "video_roi_isc_df" not in globals():
    raise ValueError(
        "video_roi_isc_df was not found. "
        "Run the subject × video ROI ISC pipeline first."
    )


# ============================================================
# 2. FILEPATHS
# ============================================================

# Change this only if video_mapping.csv is stored elsewhere.
VIDEO_MAPPING_CSV = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\video_mapping.csv"
)

# Alternative likely locations are checked automatically.
VIDEO_MAPPING_ALTERNATIVES = [
    Path.cwd() / "video_mapping.csv",

    Path(
        r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
        r"\Emotion Perception Data\Motor_Control\video_mapping.csv"
    ),

    Path(
        r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
        r"\Emotion Perception Data\Behavioral Data\video_mapping.csv"
    )
]

if not VIDEO_MAPPING_CSV.exists():

    matching_alternative = next(
        (
            path
            for path in VIDEO_MAPPING_ALTERNATIVES
            if path.exists()
        ),
        None
    )

    if matching_alternative is None:
        raise FileNotFoundError(
            "video_mapping.csv was not found.\n\n"
            "Update VIDEO_MAPPING_CSV near the top of this cell to "
            "the file's actual location.\n\n"
            f"Current path checked:\n{VIDEO_MAPPING_CSV}"
        )

    VIDEO_MAPPING_CSV = matching_alternative


CORRELATION_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\02_motor_feature_pca\correlations"
)

CORRELATION_TABLE_DIR = (
    CORRELATION_OUTPUT_DIR / "tables"
)

CORRELATION_FIG_DIR = (
    CORRELATION_OUTPUT_DIR / "figures"
)

CORRELATION_SCATTER_DIR = (
    CORRELATION_FIG_DIR / "scatterplots"
)

TOP_EFFECT_DIR = (
    CORRELATION_FIG_DIR / "top_absolute_correlations"
)

for folder in [
    CORRELATION_OUTPUT_DIR,
    CORRELATION_TABLE_DIR,
    CORRELATION_FIG_DIR,
    CORRELATION_SCATTER_DIR,
    TOP_EFFECT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print(f"Video mapping file:\n{VIDEO_MAPPING_CSV}")
print(f"\nOutputs will be saved to:\n{CORRELATION_OUTPUT_DIR}")


# ============================================================
# 3. USER-ADJUSTABLE SETTINGS
# ============================================================

MIN_N_PER_CORRELATION = 20

# Diagnostic restricted permutation test.
N_PERMUTATIONS = 5000
PERMUTATION_RANDOM_SEED = 2026

# Set to a list such as ["MotorPC1", "MotorPC2"] to restrict testing.
# Leave as None to test every available MotorPC column.
MOTOR_PCS_TO_TEST = None

SAVE_ALL_SCATTERPLOTS = True
N_TOP_CORRELATIONS_TO_PLOT = 20

FIGURE_DPI = 300


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """
    Normalize subject identifiers while preserving their content.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_video_name(value):
    """
    Normalize the underlying video identifier for mapping.

    Removes only a final modality suffix when present.

    Examples
    --------
    031SoloC_baseline_valence -> 031SoloC_baseline
    031SoloC_baseline_arousal -> 031SoloC_baseline
    031SoloC_baseline         -> 031SoloC_baseline
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"_(valence|arousal)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_trigger(value):
    """
    Normalize trigger identifiers so 1, 1.0, and '1' match.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(
                int(numeric_value)
            )

    except (TypeError, ValueError):
        pass

    return value


def safe_filename(value, max_length=160):
    """
    Create a Windows-safe filename.
    """
    value = str(value)

    value = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        value
    )

    value = re.sub(
        r"\s+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    value = value.strip("._ ")

    return value[:max_length] or "unnamed"


def safe_correlation(x, y, method="pearson"):
    """
    Compute a correlation after finite-value and variance checks.
    """
    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < MIN_N_PER_CORRELATION:
        return np.nan, np.nan, len(x)

    if (
        np.std(x, ddof=1) == 0
        or np.std(y, ddof=1) == 0
    ):
        return np.nan, np.nan, len(x)

    if method == "pearson":
        statistic, p_value = pearsonr(
            x,
            y
        )

    elif method == "spearman":
        statistic, p_value = spearmanr(
            x,
            y
        )

    else:
        raise ValueError(
            "method must be 'pearson' or 'spearman'."
        )

    return (
        float(statistic),
        float(p_value),
        len(x)
    )


def subject_block_permutation_p(
    analysis_df,
    motor_pc,
    neural_variable,
    observed_r,
    n_permutations=5000,
    random_seed=2026
):
    """
    Shuffle Motor-PC values within each participant across videos.

    This preserves participant membership better than unrestricted
    row permutation. It remains a diagnostic test; crossed mixed-effects
    models should provide the primary Stage 3 inference.
    """
    permutation_df = (
        analysis_df[
            [
                "Subject",
                motor_pc,
                neural_variable
            ]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
        .reset_index(drop=True)
    )

    if len(permutation_df) < MIN_N_PER_CORRELATION:
        return np.nan

    if (
        permutation_df[motor_pc].std(ddof=1) == 0
        or permutation_df[neural_variable].std(ddof=1) == 0
    ):
        return np.nan

    subject_positions = {
        subject: group.index.to_numpy(dtype=int)
        for subject, group in permutation_df.groupby(
            "Subject",
            sort=False
        )
    }

    rng = np.random.default_rng(
        random_seed
    )

    original_motor_values = (
        permutation_df[motor_pc]
        .to_numpy(dtype=float)
    )

    neural_values = (
        permutation_df[neural_variable]
        .to_numpy(dtype=float)
    )

    permutation_correlations = np.empty(
        n_permutations,
        dtype=float
    )

    for permutation_index in range(
        n_permutations
    ):
        permuted_motor_values = (
            original_motor_values.copy()
        )

        for positions in subject_positions.values():

            if len(positions) > 1:
                permuted_motor_values[
                    positions
                ] = rng.permutation(
                    permuted_motor_values[
                        positions
                    ]
                )

        permutation_correlations[
            permutation_index
        ] = pearsonr(
            permuted_motor_values,
            neural_values
        )[0]

    permutation_p = (
        1
        + np.sum(
            np.abs(
                permutation_correlations
            )
            >= abs(observed_r)
        )
    ) / (
        n_permutations + 1
    )

    return float(
        permutation_p
    )


def add_grouped_fdr(
    dataframe,
    p_column,
    output_column,
    group_columns
):
    """
    Apply Benjamini-Hochberg correction within analysis families.
    """
    dataframe = dataframe.copy()
    dataframe[output_column] = np.nan

    for _, group_indices in dataframe.groupby(
        group_columns,
        dropna=False
    ).groups.items():

        group_indices = list(
            group_indices
        )

        valid_indices = [
            index
            for index in group_indices
            if pd.notna(
                dataframe.loc[
                    index,
                    p_column
                ]
            )
        ]

        if len(valid_indices) == 0:
            continue

        dataframe.loc[
            valid_indices,
            output_column
        ] = multipletests(
            dataframe.loc[
                valid_indices,
                p_column
            ],
            method="fdr_bh"
        )[1]

    return dataframe


# ============================================================
# 5. LOAD AND VALIDATE VIDEO-MAPPING CSV
# ============================================================

video_mapping_raw_df = pd.read_csv(
    VIDEO_MAPPING_CSV
)

required_mapping_columns = {
    "video",
    "trigger_number"
}

missing_mapping_columns = (
    required_mapping_columns
    - set(video_mapping_raw_df.columns)
)

if missing_mapping_columns:
    raise ValueError(
        "The video-mapping file is missing required columns:\n"
        f"{sorted(missing_mapping_columns)}\n\n"
        "Required columns are 'video' and 'trigger_number'."
    )

video_mapping_df = (
    video_mapping_raw_df[
        [
            "video",
            "trigger_number"
        ]
    ]
    .copy()
)

video_mapping_df = video_mapping_df.rename(
    columns={
        "video": "BaseVideo",
        "trigger_number": "Trigger"
    }
)

video_mapping_df[
    "BaseVideo"
] = video_mapping_df[
    "BaseVideo"
].apply(
    normalize_video_name
)

video_mapping_df[
    "Trigger"
] = video_mapping_df[
    "Trigger"
].apply(
    normalize_trigger
)

video_mapping_df = (
    video_mapping_df
    .dropna(
        subset=[
            "BaseVideo",
            "Trigger"
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate unique BaseVideo → Trigger assignments
# ------------------------------------------------------------

video_to_trigger_counts = (
    video_mapping_df
    .groupby(
        "BaseVideo"
    )[
        "Trigger"
    ]
    .nunique()
)

ambiguous_videos = video_to_trigger_counts[
    video_to_trigger_counts > 1
].index.tolist()

if len(ambiguous_videos) > 0:
    ambiguous_rows = video_mapping_df[
        video_mapping_df[
            "BaseVideo"
        ].isin(
            ambiguous_videos
        )
    ]

    display(
        ambiguous_rows
    )

    raise ValueError(
        "At least one video maps to more than one trigger. "
        "Correct the video-mapping CSV before continuing."
    )


# ------------------------------------------------------------
# Validate unique Trigger → BaseVideo assignments
# ------------------------------------------------------------

trigger_to_video_counts = (
    video_mapping_df
    .groupby(
        "Trigger"
    )[
        "BaseVideo"
    ]
    .nunique()
)

ambiguous_triggers = trigger_to_video_counts[
    trigger_to_video_counts > 1
].index.tolist()

if len(ambiguous_triggers) > 0:
    ambiguous_rows = video_mapping_df[
        video_mapping_df[
            "Trigger"
        ].isin(
            ambiguous_triggers
        )
    ]

    display(
        ambiguous_rows
    )

    raise ValueError(
        "At least one trigger maps to more than one video. "
        "Correct the video-mapping CSV before continuing."
    )


video_mapping_df = (
    video_mapping_df
    .drop_duplicates(
        subset=["BaseVideo"]
    )
    .sort_values(
        "Trigger",
        key=lambda series: pd.to_numeric(
            series,
            errors="coerce"
        )
    )
    .reset_index(drop=True)
)

video_mapping_df.to_csv(
    CORRELATION_TABLE_DIR
    / "validated_video_trigger_mapping.csv",
    index=False
)

print(
    f"\nLoaded {len(video_mapping_df)} "
    "validated video-to-trigger mappings."
)

display(
    video_mapping_df
)


# ============================================================
# 6. PREPARE MOTOR-PC TABLE
# ============================================================

motor_df = motor_pca_scores_df.copy()

required_motor_columns = {
    "Subject",
    "BaseVideo",
    "Modality"
}

missing_motor_columns = (
    required_motor_columns
    - set(motor_df.columns)
)

if missing_motor_columns:
    raise ValueError(
        "motor_pca_scores_df is missing required columns:\n"
        f"{sorted(missing_motor_columns)}"
    )

motor_df[
    "Subject"
] = motor_df[
    "Subject"
].apply(
    normalize_subject_id
)

motor_df[
    "BaseVideo"
] = motor_df[
    "BaseVideo"
].apply(
    normalize_video_name
)

motor_df[
    "Modality"
] = (
    motor_df["Modality"]
    .astype(str)
    .str.strip()
    .str.capitalize()
)


# ------------------------------------------------------------
# Remove any old or incorrectly inferred Trigger column
# ------------------------------------------------------------

if "Trigger" in motor_df.columns:
    motor_df = motor_df.drop(
        columns=["Trigger"]
    )


# ------------------------------------------------------------
# Merge true trigger assignments from the CSV
# ------------------------------------------------------------

motor_df = motor_df.merge(
    video_mapping_df,
    on="BaseVideo",
    how="left",
    validate="many_to_one",
    indicator="VideoMappingMerge"
)

motor_df[
    "VideoMappingMatched"
] = (
    motor_df[
        "VideoMappingMerge"
    ] == "both"
)


unmapped_motor_videos_df = (
    motor_df.loc[
        ~motor_df[
            "VideoMappingMatched"
        ],
        [
            "BaseVideo",
            "VideoMappingMerge"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "BaseVideo"
    )
    .reset_index(drop=True)
)

unmapped_motor_videos_df.to_csv(
    CORRELATION_TABLE_DIR
    / "unmapped_motor_base_videos.csv",
    index=False
)

if not unmapped_motor_videos_df.empty:

    print(
        "\nWARNING: These motor-table BaseVideo values "
        "were not found in video_mapping.csv:"
    )

    display(
        unmapped_motor_videos_df
    )

    print(
        "These observations will be excluded from the "
        "motor-neural correlation analysis."
    )


# ------------------------------------------------------------
# Automatically identify available MotorPC columns
# ------------------------------------------------------------

available_motor_pcs = sorted(
    [
        column
        for column in motor_df.columns
        if re.fullmatch(
            r"MotorPC\d+",
            str(column)
        )
    ],
    key=lambda value: int(
        re.search(
            r"\d+",
            value
        ).group()
    )
)

if MOTOR_PCS_TO_TEST is None:
    motor_pc_columns = (
        available_motor_pcs
    )

else:
    motor_pc_columns = [
        column
        for column in MOTOR_PCS_TO_TEST
        if column in available_motor_pcs
    ]

if len(motor_pc_columns) == 0:
    raise ValueError(
        "No MotorPC columns were available for analysis."
    )

print("\nMotor PCs included:")
for column in motor_pc_columns:
    print(f"  {column}")


# ============================================================
# 7. PREPARE NEURAL ISC TABLE
# ============================================================

neural_long_df = (
    video_roi_isc_df.copy()
)

required_neural_columns = {
    "Subject",
    "Trigger",
    "ROI",
    "ISC"
}

missing_neural_columns = (
    required_neural_columns
    - set(neural_long_df.columns)
)

if missing_neural_columns:
    raise ValueError(
        "video_roi_isc_df is missing required columns:\n"
        f"{sorted(missing_neural_columns)}"
    )

neural_long_df[
    "Subject"
] = neural_long_df[
    "Subject"
].apply(
    normalize_subject_id
)

neural_long_df[
    "Trigger"
] = neural_long_df[
    "Trigger"
].apply(
    normalize_trigger
)

neural_long_df[
    "ROI"
] = (
    neural_long_df["ROI"]
    .astype(str)
    .str.strip()
)

neural_long_df[
    "ISC"
] = pd.to_numeric(
    neural_long_df["ISC"],
    errors="coerce"
)


# ------------------------------------------------------------
# Check duplicate Subject × Trigger × ROI rows
# ------------------------------------------------------------

duplicate_neural_rows_df = (
    neural_long_df
    .groupby(
        [
            "Subject",
            "Trigger",
            "ROI"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="N_Rows"
    )
)

duplicate_neural_rows_df = (
    duplicate_neural_rows_df[
        duplicate_neural_rows_df[
            "N_Rows"
        ] > 1
    ]
    .copy()
)

duplicate_neural_rows_df.to_csv(
    CORRELATION_TABLE_DIR
    / "duplicate_subject_trigger_roi_rows.csv",
    index=False
)


# ------------------------------------------------------------
# Pivot neural data to one row per Subject × Trigger
# ------------------------------------------------------------

neural_wide_df = (
    neural_long_df
    .pivot_table(
        index=[
            "Subject",
            "Trigger"
        ],
        columns="ROI",
        values="ISC",
        aggfunc="mean"
    )
    .reset_index()
)

neural_wide_df.columns.name = None

individual_roi_columns = [
    column
    for column in neural_wide_df.columns
    if column not in [
        "Subject",
        "Trigger"
    ]
]

if len(individual_roi_columns) == 0:
    raise RuntimeError(
        "No ROI columns were created from video_roi_isc_df."
    )


# ============================================================
# 8. CREATE MEAN ISC AND NETWORK-BALANCE VARIABLES
# ============================================================

neural_wide_df[
    "Mean_ROI_ISC"
] = (
    neural_wide_df[
        individual_roi_columns
    ]
    .mean(
        axis=1,
        skipna=True
    )
)

neural_wide_df[
    "Mean_ROI_NValid"
] = (
    neural_wide_df[
        individual_roi_columns
    ]
    .notna()
    .sum(axis=1)
)


# ------------------------------------------------------------
# Original Network Balance
# ------------------------------------------------------------

original_positive_rois = [
    "Right VLPFC",
    "Left TPJ"
]

original_negative_rois = [
    "Left VLPFC",
    "Right TPJ"
]

original_required_rois = (
    original_positive_rois
    + original_negative_rois
)

if all(
    roi in neural_wide_df.columns
    for roi in original_required_rois
):
    neural_wide_df[
        "Original_Network_Balance"
    ] = (
        neural_wide_df[
            original_positive_rois
        ].mean(
            axis=1,
            skipna=False
        )
        -
        neural_wide_df[
            original_negative_rois
        ].mean(
            axis=1,
            skipna=False
        )
    )

else:
    missing_original_rois = [
        roi
        for roi in original_required_rois
        if roi not in neural_wide_df.columns
    ]

    print(
        "\nOriginal_Network_Balance could not be created. "
        f"Missing ROIs: {missing_original_rois}"
    )


# ------------------------------------------------------------
# Full LASSO Network Balance
# ------------------------------------------------------------

lasso_positive_rois = [
    "Left TPJ",
    "Right VLPFC",
    "Left OFC",
    "Left STS"
]

lasso_negative_rois = [
    "Left VLPFC",
    "Left DLPFC",
    "Right TPJ"
]

lasso_required_rois = (
    lasso_positive_rois
    + lasso_negative_rois
)

if all(
    roi in neural_wide_df.columns
    for roi in lasso_required_rois
):
    neural_wide_df[
        "Full_LASSO_Network_Balance"
    ] = (
        neural_wide_df[
            lasso_positive_rois
        ].mean(
            axis=1,
            skipna=False
        )
        -
        neural_wide_df[
            lasso_negative_rois
        ].mean(
            axis=1,
            skipna=False
        )
    )

else:
    missing_lasso_rois = [
        roi
        for roi in lasso_required_rois
        if roi not in neural_wide_df.columns
    ]

    print(
        "\nFull_LASSO_Network_Balance could not be created. "
        f"Missing ROIs: {missing_lasso_rois}"
    )


summary_neural_variables = [
    column
    for column in [
        "Mean_ROI_ISC",
        "Original_Network_Balance",
        "Full_LASSO_Network_Balance"
    ]
    if column in neural_wide_df.columns
]

neural_predictor_columns = list(
    dict.fromkeys(
        summary_neural_variables
        + individual_roi_columns
    )
)


# ============================================================
# 9. CREATE MERGE TABLE AND VERIFY MAPPING
# ============================================================

motor_merge_columns = [
    "Subject",
    "BaseVideo",
    "Modality",
    "Trigger",
    "VideoMappingMatched"
] + motor_pc_columns

if "Video" in motor_df.columns:
    motor_merge_columns.insert(
        2,
        "Video"
    )

for optional_column in [
    "DurationSec",
    "AnyQCFlag",
    "ExtremeFeatureOutlierFlag",
    "r_PC1",
    "abs_r_PC1",
    "Behavioral_Subgroup",
    "Is_Alternative_Consensus"
]:
    if (
        optional_column in motor_df.columns
        and optional_column
        not in motor_merge_columns
    ):
        motor_merge_columns.append(
            optional_column
        )

motor_merge_df = motor_df[
    motor_merge_columns
].copy()


# ------------------------------------------------------------
# Exclude videos absent from mapping before neural merge
# ------------------------------------------------------------

mapped_motor_df = motor_merge_df[
    motor_merge_df[
        "VideoMappingMatched"
    ]
].copy()


# ------------------------------------------------------------
# Check uniqueness at Subject × Trigger × Modality
# ------------------------------------------------------------

duplicate_motor_keys_df = (
    mapped_motor_df
    .groupby(
        [
            "Subject",
            "Trigger",
            "Modality"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="N_Rows"
    )
)

duplicate_motor_keys_df = (
    duplicate_motor_keys_df[
        duplicate_motor_keys_df[
            "N_Rows"
        ] > 1
    ]
    .copy()
)

duplicate_motor_keys_df.to_csv(
    CORRELATION_TABLE_DIR
    / "duplicate_motor_merge_keys.csv",
    index=False
)

if not duplicate_motor_keys_df.empty:

    display(
        duplicate_motor_keys_df
    )

    raise ValueError(
        "Duplicate Subject × Trigger × Modality rows were found "
        "after applying video_mapping.csv. Inspect the saved "
        "duplicate_motor_merge_keys.csv file before continuing."
    )


# ------------------------------------------------------------
# Merge motor PCs with neural ISC
# ------------------------------------------------------------

motor_neural_pc_df = mapped_motor_df.merge(
    neural_wide_df,
    on=[
        "Subject",
        "Trigger"
    ],
    how="left",
    validate="many_to_one",
    indicator="NeuralMerge"
)

motor_neural_pc_df[
    "NeuralMergeMatched"
] = (
    motor_neural_pc_df[
        "NeuralMerge"
    ] == "both"
)


# ------------------------------------------------------------
# Verify that each BaseVideo retains its expected trigger
# ------------------------------------------------------------

mapping_check_df = (
    motor_neural_pc_df[
        [
            "BaseVideo",
            "Trigger"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Trigger",
        key=lambda series: pd.to_numeric(
            series,
            errors="coerce"
        )
    )
    .reset_index(drop=True)
)

mapping_check_df.to_csv(
    CORRELATION_TABLE_DIR
    / "motor_table_video_trigger_mapping_check.csv",
    index=False
)


# ------------------------------------------------------------
# Merge diagnostics
# ------------------------------------------------------------

merge_summary_df = pd.DataFrame({
    "Metric": [
        "Rows in original motor-PC table",
        "Rows mapped using video_mapping.csv",
        "Rows with no video mapping",
        "Rows matched to neural ISC",
        "Rows not matched to neural ISC",
        "Unique motor subjects",
        "Unique matched subjects",
        "Unique mapped videos",
        "Unique matched triggers"
    ],
    "Value": [
        len(motor_df),

        len(mapped_motor_df),

        int(
            (
                ~motor_df[
                    "VideoMappingMatched"
                ]
            ).sum()
        ),

        int(
            motor_neural_pc_df[
                "NeuralMergeMatched"
            ].sum()
        ),

        int(
            (
                ~motor_neural_pc_df[
                    "NeuralMergeMatched"
                ]
            ).sum()
        ),

        motor_df[
            "Subject"
        ].nunique(),

        motor_neural_pc_df.loc[
            motor_neural_pc_df[
                "NeuralMergeMatched"
            ],
            "Subject"
        ].nunique(),

        mapped_motor_df[
            "BaseVideo"
        ].nunique(),

        motor_neural_pc_df.loc[
            motor_neural_pc_df[
                "NeuralMergeMatched"
            ],
            "Trigger"
        ].nunique()
    ]
})

merge_summary_df.to_csv(
    CORRELATION_TABLE_DIR
    / "motor_neural_merge_summary.csv",
    index=False
)

motor_neural_pc_df.to_csv(
    CORRELATION_TABLE_DIR
    / "subject_video_motor_pc_with_neural_isc.csv",
    index=False
)

unmatched_neural_rows_df = (
    motor_neural_pc_df.loc[
        ~motor_neural_pc_df[
            "NeuralMergeMatched"
        ]
    ]
    .copy()
)

unmatched_neural_rows_df.to_csv(
    CORRELATION_TABLE_DIR
    / "mapped_motor_rows_without_neural_match.csv",
    index=False
)

print("\nVerified mapping applied to motor table:")
display(
    mapping_check_df
)

print("\nMotor-neural merge summary:")
display(
    merge_summary_df
)


# ============================================================
# 10. CALCULATE MOTOR-PC × NEURAL CORRELATIONS
# ============================================================

analysis_df = (
    motor_neural_pc_df[
        motor_neural_pc_df[
            "NeuralMergeMatched"
        ]
    ]
    .copy()
)

if analysis_df.empty:
    raise RuntimeError(
        "No motor rows matched the neural ISC table. "
        "Inspect the subject IDs, normalized triggers, and saved "
        "merge-diagnostic CSV files."
    )


analysis_groups = {
    "All": analysis_df
}

for modality in sorted(
    analysis_df[
        "Modality"
    ]
    .dropna()
    .unique()
):
    analysis_groups[
        str(modality)
    ] = (
        analysis_df[
            analysis_df[
                "Modality"
            ] == modality
        ]
        .copy()
    )


correlation_rows = []

total_tests = (
    len(analysis_groups)
    * len(motor_pc_columns)
    * len(neural_predictor_columns)
)

test_counter = 0

for analysis_group, group_df in analysis_groups.items():

    for motor_pc in motor_pc_columns:

        for neural_variable in neural_predictor_columns:

            test_counter += 1

            complete_df = (
                group_df[
                    [
                        "Subject",
                        "Trigger",
                        "BaseVideo",
                        motor_pc,
                        neural_variable
                    ]
                ]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .dropna()
                .copy()
            )

            (
                pearson_r,
                pearson_p,
                n_complete
            ) = safe_correlation(
                complete_df[
                    motor_pc
                ],
                complete_df[
                    neural_variable
                ],
                method="pearson"
            )

            (
                spearman_rho,
                spearman_p,
                _
            ) = safe_correlation(
                complete_df[
                    motor_pc
                ],
                complete_df[
                    neural_variable
                ],
                method="spearman"
            )

            if np.isfinite(
                pearson_r
            ):
                permutation_p = (
                    subject_block_permutation_p(
                        analysis_df=complete_df,
                        motor_pc=motor_pc,
                        neural_variable=neural_variable,
                        observed_r=pearson_r,
                        n_permutations=N_PERMUTATIONS,
                        random_seed=(
                            PERMUTATION_RANDOM_SEED
                            + test_counter
                        )
                    )
                )

            else:
                permutation_p = np.nan

            correlation_rows.append({
                "Analysis_Group": analysis_group,
                "Motor_PC": motor_pc,
                "Neural_Predictor": neural_variable,

                "Neural_Predictor_Type": (
                    "Summary_or_Network"
                    if neural_variable
                    in summary_neural_variables
                    else "Individual_ROI"
                ),

                "N_SubjectVideo": n_complete,

                "N_Subjects": complete_df[
                    "Subject"
                ].nunique(),

                "N_Triggers": complete_df[
                    "Trigger"
                ].nunique(),

                "Pearson_r": pearson_r,
                "Pearson_p": pearson_p,

                "Spearman_rho": spearman_rho,
                "Spearman_p": spearman_p,

                "SubjectBlockPermutation_p": (
                    permutation_p
                ),

                "VarianceShared_r2": (
                    pearson_r ** 2
                    if np.isfinite(
                        pearson_r
                    )
                    else np.nan
                )
            })

            if (
                test_counter % 25 == 0
                or test_counter == total_tests
            ):
                print(
                    f"Completed {test_counter}/"
                    f"{total_tests} correlation tests."
                )


motor_neural_correlations_df = pd.DataFrame(
    correlation_rows
)


# ============================================================
# 11. MULTIPLE-COMPARISON CORRECTION
# ============================================================

fdr_family_columns = [
    "Analysis_Group",
    "Motor_PC"
]

motor_neural_correlations_df = add_grouped_fdr(
    motor_neural_correlations_df,
    p_column="Pearson_p",
    output_column="Pearson_p_FDR",
    group_columns=fdr_family_columns
)

motor_neural_correlations_df = add_grouped_fdr(
    motor_neural_correlations_df,
    p_column="Spearman_p",
    output_column="Spearman_p_FDR",
    group_columns=fdr_family_columns
)

motor_neural_correlations_df = add_grouped_fdr(
    motor_neural_correlations_df,
    p_column="SubjectBlockPermutation_p",
    output_column="SubjectBlockPermutation_p_FDR",
    group_columns=fdr_family_columns
)

motor_neural_correlations_df[
    "Absolute_Pearson_r"
] = np.abs(
    motor_neural_correlations_df[
        "Pearson_r"
    ]
)

motor_neural_correlations_df = (
    motor_neural_correlations_df
    .sort_values(
        [
            "Absolute_Pearson_r",
            "SubjectBlockPermutation_p"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 12. SAVE CORRELATION TABLES
# ============================================================

motor_neural_correlations_df.to_csv(
    CORRELATION_TABLE_DIR
    / "motor_pc_neural_isc_correlations.csv",
    index=False
)

summary_network_correlations_df = (
    motor_neural_correlations_df[
        motor_neural_correlations_df[
            "Neural_Predictor_Type"
        ] == "Summary_or_Network"
    ]
    .copy()
)

individual_roi_correlations_df = (
    motor_neural_correlations_df[
        motor_neural_correlations_df[
            "Neural_Predictor_Type"
        ] == "Individual_ROI"
    ]
    .copy()
)

summary_network_correlations_df.to_csv(
    CORRELATION_TABLE_DIR
    / "motor_pc_summary_network_correlations.csv",
    index=False
)

individual_roi_correlations_df.to_csv(
    CORRELATION_TABLE_DIR
    / "motor_pc_individual_roi_correlations.csv",
    index=False
)


top_effects_df = (
    motor_neural_correlations_df
    .dropna(
        subset=[
            "Pearson_r"
        ]
    )
    .sort_values(
        "Absolute_Pearson_r",
        ascending=False
    )
    .head(
        N_TOP_CORRELATIONS_TO_PLOT
    )
    .copy()
)

top_effects_df.to_csv(
    CORRELATION_TABLE_DIR
    / "top_absolute_motor_pc_neural_correlations.csv",
    index=False
)


# ============================================================
# 13. SAVE CORRELATION MATRICES
# ============================================================

for analysis_group in analysis_groups.keys():

    group_results = (
        motor_neural_correlations_df[
            motor_neural_correlations_df[
                "Analysis_Group"
            ] == analysis_group
        ]
    )

    pearson_matrix = (
        group_results
        .pivot(
            index="Neural_Predictor",
            columns="Motor_PC",
            values="Pearson_r"
        )
        .reindex(
            columns=motor_pc_columns
        )
    )

    permutation_fdr_matrix = (
        group_results
        .pivot(
            index="Neural_Predictor",
            columns="Motor_PC",
            values=(
                "SubjectBlockPermutation_p_FDR"
            )
        )
        .reindex(
            columns=motor_pc_columns
        )
    )

    group_name = safe_filename(
        analysis_group
    )

    pearson_matrix.to_csv(
        CORRELATION_TABLE_DIR
        / (
            f"{group_name}"
            "__motor_pc_neural_pearson_matrix.csv"
        ),
        index=True,
        index_label="Neural_Predictor"
    )

    permutation_fdr_matrix.to_csv(
        CORRELATION_TABLE_DIR
        / (
            f"{group_name}"
            "__motor_pc_neural_permutation_fdr_matrix.csv"
        ),
        index=True,
        index_label="Neural_Predictor"
    )


# ============================================================
# 14. CORRELATION HEATMAPS
# ============================================================

for analysis_group in analysis_groups.keys():

    group_results = (
        motor_neural_correlations_df[
            motor_neural_correlations_df[
                "Analysis_Group"
            ] == analysis_group
        ]
        .copy()
    )

    heatmap_matrix = (
        group_results
        .pivot(
            index="Neural_Predictor",
            columns="Motor_PC",
            values="Pearson_r"
        )
        .reindex(
            columns=motor_pc_columns
        )
    )

    desired_order = (
        [
            variable
            for variable
            in summary_neural_variables
            if variable
            in heatmap_matrix.index
        ]
        +
        sorted(
            [
                variable
                for variable
                in individual_roi_columns
                if variable
                in heatmap_matrix.index
            ]
        )
    )

    heatmap_matrix = heatmap_matrix.reindex(
        desired_order
    )

    if heatmap_matrix.empty:
        continue

    fig, ax = plt.subplots(
        figsize=(
            max(
                7,
                1.8 * len(
                    motor_pc_columns
                )
            ),
            max(
                7,
                0.48 * len(
                    heatmap_matrix.index
                )
            )
        )
    )

    image = ax.imshow(
        heatmap_matrix.to_numpy(
            dtype=float
        ),
        aspect="auto",
        cmap="coolwarm",
        vmin=-0.40,
        vmax=0.40
    )

    ax.set_xticks(
        np.arange(
            len(motor_pc_columns)
        )
    )

    ax.set_xticklabels(
        motor_pc_columns
    )

    ax.set_yticks(
        np.arange(
            len(heatmap_matrix.index)
        )
    )

    ax.set_yticklabels(
        heatmap_matrix.index
    )

    for row_index, neural_variable in enumerate(
        heatmap_matrix.index
    ):
        for column_index, motor_pc in enumerate(
            heatmap_matrix.columns
        ):
            correlation_value = (
                heatmap_matrix.loc[
                    neural_variable,
                    motor_pc
                ]
            )

            if not np.isfinite(
                correlation_value
            ):
                continue

            matching_result = group_results[
                (
                    group_results[
                        "Neural_Predictor"
                    ] == neural_variable
                )
                & (
                    group_results[
                        "Motor_PC"
                    ] == motor_pc
                )
            ]

            significance_marker = ""

            if not matching_result.empty:

                corrected_p = matching_result[
                    "SubjectBlockPermutation_p_FDR"
                ].iloc[0]

                if (
                    pd.notna(corrected_p)
                    and corrected_p < 0.001
                ):
                    significance_marker = "***"

                elif (
                    pd.notna(corrected_p)
                    and corrected_p < 0.01
                ):
                    significance_marker = "**"

                elif (
                    pd.notna(corrected_p)
                    and corrected_p < 0.05
                ):
                    significance_marker = "*"

            ax.text(
                column_index,
                row_index,
                (
                    f"{correlation_value:.2f}"
                    f"{significance_marker}"
                ),
                ha="center",
                va="center",
                fontsize=8
            )

    fig.colorbar(
        image,
        ax=ax,
        label="Pearson correlation"
    )

    ax.set_title(
        "Motor-PC associations with neural ISC\n"
        f"{analysis_group}"
    )

    ax.set_xlabel(
        "Motor component"
    )

    ax.set_ylabel(
        "Neural ISC predictor"
    )

    fig.tight_layout()

    fig.savefig(
        CORRELATION_FIG_DIR
        / (
            f"{safe_filename(analysis_group)}"
            "__motor_pc_neural_correlation_heatmap.png"
        ),
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 15. SCATTERPLOT FUNCTION
# ============================================================

def plot_motor_neural_scatter(
    dataframe,
    analysis_group,
    motor_pc,
    neural_variable,
    result_row,
    output_path
):
    """
    Create one Motor-PC × neural-predictor scatterplot.
    """
    plot_df = (
        dataframe[
            [
                motor_pc,
                neural_variable
            ]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    if len(plot_df) < MIN_N_PER_CORRELATION:
        return

    x = plot_df[
        motor_pc
    ].to_numpy(dtype=float)

    y = plot_df[
        neural_variable
    ].to_numpy(dtype=float)

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    ax.scatter(
        x,
        y,
        alpha=0.35,
        s=28
    )

    if (
        np.std(x, ddof=1) > 0
        and np.std(y, ddof=1) > 0
    ):
        slope, intercept = np.polyfit(
            x,
            y,
            deg=1
        )

        x_line = np.linspace(
            np.nanmin(x),
            np.nanmax(x),
            200
        )

        ax.plot(
            x_line,
            intercept
            + slope * x_line,
            linewidth=2
        )

    corrected_p = result_row[
        "SubjectBlockPermutation_p_FDR"
    ]

    p_text = (
        f"{corrected_p:.4f}"
        if pd.notna(corrected_p)
        else "NA"
    )

    ax.set_xlabel(
        f"{motor_pc} score"
    )

    ax.set_ylabel(
        neural_variable
    )

    ax.set_title(
        f"{analysis_group}: "
        f"{motor_pc} × {neural_variable}\n"
        f"r = {result_row['Pearson_r']:.3f}, "
        f"permutation-FDR p = {p_text}"
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 16. SAVE ALL SCATTERPLOTS
# ============================================================

if SAVE_ALL_SCATTERPLOTS:

    for _, result_row in (
        motor_neural_correlations_df
        .iterrows()
    ):
        if not np.isfinite(
            result_row[
                "Pearson_r"
            ]
        ):
            continue

        analysis_group = result_row[
            "Analysis_Group"
        ]

        motor_pc = result_row[
            "Motor_PC"
        ]

        neural_variable = result_row[
            "Neural_Predictor"
        ]

        filename = (
            f"{safe_filename(analysis_group)}__"
            f"{safe_filename(motor_pc)}__"
            f"{safe_filename(neural_variable)}.png"
        )

        plot_motor_neural_scatter(
            dataframe=analysis_groups[
                analysis_group
            ],
            analysis_group=analysis_group,
            motor_pc=motor_pc,
            neural_variable=neural_variable,
            result_row=result_row,
            output_path=(
                CORRELATION_SCATTER_DIR
                / filename
            )
        )


# ============================================================
# 17. SAVE TOP-EFFECT SCATTERPLOTS
# ============================================================

for rank, (_, result_row) in enumerate(
    top_effects_df.iterrows(),
    start=1
):
    analysis_group = result_row[
        "Analysis_Group"
    ]

    motor_pc = result_row[
        "Motor_PC"
    ]

    neural_variable = result_row[
        "Neural_Predictor"
    ]

    filename = (
        f"{rank:02d}__"
        f"{safe_filename(analysis_group)}__"
        f"{safe_filename(motor_pc)}__"
        f"{safe_filename(neural_variable)}.png"
    )

    plot_motor_neural_scatter(
        dataframe=analysis_groups[
            analysis_group
        ],
        analysis_group=analysis_group,
        motor_pc=motor_pc,
        neural_variable=neural_variable,
        result_row=result_row,
        output_path=(
            TOP_EFFECT_DIR
            / filename
        )
    )


# ============================================================
# 18. SUMMARY NETWORK EFFECT-SIZE FIGURES
# ============================================================

summary_plot_df = (
    summary_network_correlations_df
    .copy()
)

if not summary_plot_df.empty:

    summary_plot_df[
        "PlotLabel"
    ] = (
        summary_plot_df[
            "Analysis_Group"
        ]
        + " | "
        + summary_plot_df[
            "Neural_Predictor"
        ]
    )

    for motor_pc in motor_pc_columns:

        pc_plot_df = summary_plot_df[
            summary_plot_df[
                "Motor_PC"
            ] == motor_pc
        ].copy()

        if pc_plot_df.empty:
            continue

        pc_plot_df = pc_plot_df.sort_values(
            "Pearson_r"
        )

        fig, ax = plt.subplots(
            figsize=(
                10,
                max(
                    5,
                    0.50
                    * len(pc_plot_df)
                )
            )
        )

        y_positions = np.arange(
            len(pc_plot_df)
        )

        ax.barh(
            y_positions,
            pc_plot_df[
                "Pearson_r"
            ]
        )

        ax.axvline(
            0,
            linewidth=1
        )

        ax.axvline(
            0.10,
            linestyle="--",
            linewidth=1,
            label="|r| = .10"
        )

        ax.axvline(
            -0.10,
            linestyle="--",
            linewidth=1
        )

        ax.set_yticks(
            y_positions
        )

        ax.set_yticklabels(
            pc_plot_df[
                "PlotLabel"
            ]
        )

        ax.set_xlabel(
            "Pearson correlation"
        )

        ax.set_title(
            f"{motor_pc} associations with "
            "summary neural measures"
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        fig.savefig(
            CORRELATION_FIG_DIR
            / (
                f"{safe_filename(motor_pc)}"
                "__summary_neural_effect_sizes.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 19. PRINT RESULTS
# ============================================================

print("\n" + "=" * 95)
print("MOTOR PCA × NEURAL ISC CORRELATION ANALYSIS COMPLETE")
print("=" * 95)

print("\nValidated video mapping:")
display(
    video_mapping_df
)

print("\nMapping and neural-merge summary:")
display(
    merge_summary_df
)

print("\nNeural predictors tested:")
for variable in neural_predictor_columns:
    print(f"  {variable}")

print("\nSummary/network correlations:")
display(
    summary_network_correlations_df[
        [
            "Analysis_Group",
            "Motor_PC",
            "Neural_Predictor",
            "N_SubjectVideo",
            "N_Subjects",
            "N_Triggers",
            "Pearson_r",
            "Pearson_p",
            "Pearson_p_FDR",
            "Spearman_rho",
            "Spearman_p_FDR",
            "SubjectBlockPermutation_p",
            "SubjectBlockPermutation_p_FDR",
            "VarianceShared_r2"
        ]
    ]
    .sort_values(
        [
            "Analysis_Group",
            "Motor_PC",
            "Absolute_Pearson_r"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .round(5)
)

print("\nStrongest individual-ROI associations:")
display(
    individual_roi_correlations_df[
        [
            "Analysis_Group",
            "Motor_PC",
            "Neural_Predictor",
            "N_SubjectVideo",
            "N_Subjects",
            "N_Triggers",
            "Pearson_r",
            "Spearman_rho",
            "SubjectBlockPermutation_p_FDR",
            "VarianceShared_r2"
        ]
    ]
    .sort_values(
        "Absolute_Pearson_r",
        ascending=False
    )
    .head(30)
    .round(5)
)

print("\nTop absolute correlations overall:")
display(
    top_effects_df[
        [
            "Analysis_Group",
            "Motor_PC",
            "Neural_Predictor",
            "Neural_Predictor_Type",
            "N_SubjectVideo",
            "N_Subjects",
            "N_Triggers",
            "Pearson_r",
            "Spearman_rho",
            "SubjectBlockPermutation_p_FDR",
            "VarianceShared_r2"
        ]
    ]
    .round(5)
)

print(
    "\nInterpretive reminder: these correlations are diagnostic "
    "effect-size analyses. The restricted permutation test partially "
    "accounts for repeated observations within participants, but the "
    "crossed subject × video mixed-effects models should remain the "
    "primary Stage 3 inferential analyses."
)

print(
    f"\nAll output files and figures were saved to:\n"
    f"{CORRELATION_OUTPUT_DIR}"
)

Video mapping file:
C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\video_mapping.csv

Outputs will be saved to:
C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\Motor_Control\02_motor_feature_pca\correlations

Loaded 28 validated video-to-trigger mappings.


,BaseVideo,Trigger
0,040SoloC_baseline,1
1,023SoloC_baseline,3
2,031SoloC_baseline,5
3,Exp3_004_baseline,7
4,48_baseline,9
5,123_baseline,11
6,55_baseline,13
7,23_baseline,15
8,88_baseline,17
9,14_baseline,19



Motor PCs included:
  MotorPC1
  MotorPC2
  MotorPC3

Verified mapping applied to motor table:


,BaseVideo,Trigger
0,040SoloC_baseline,1
1,023SoloC_baseline,3
2,031SoloC_baseline,5
3,Exp3_004_baseline,7
4,48_baseline,9
5,123_baseline,11
6,55_baseline,13
7,23_baseline,15
8,88_baseline,17
9,14_baseline,19



Motor-neural merge summary:


...

,BaseVideo,Trigger
0,040SoloC_baseline,1
1,023SoloC_baseline,3
2,031SoloC_baseline,5
3,Exp3_004_baseline,7
4,48_baseline,9
5,123_baseline,11
6,55_baseline,13
7,23_baseline,15
8,88_baseline,17
9,14_baseline,19



Mapping and neural-merge summary:


,Metric,Value
0,Rows in original motor-PC table,2352
1,Rows mapped using video_mapping.csv,2352
2,Rows with no video mapping,0
3,Rows matched to neural ISC,2352
4,Rows not matched to neural ISC,0
5,Unique motor subjects,42
6,Unique matched subjects,42
7,Unique mapped videos,28
8,Unique matched triggers,28



Neural predictors tested:
  Mean_ROI_ISC
  Original_Network_Balance
  Full_LASSO_Network_Balance
  Left DLPFC
  Left OFC
  Left RLPFC
  Left STS
  Left TPJ
  Left VLPFC
  Right DLPFC
  Right OFC
  Right RLPFC
  Right STS
  Right TPJ
  Right VLPFC

Summary/network correlations:


KeyError: 'Absolute_Pearson_r'

## Motor ROI ISC Covariate 

In [125]:
# ============================================================
# BILATERAL MOTOR-CONTROL ROI ISC PIPELINE
#
# Changes from the original pipeline:
#
#   1. The 60% per-video significance requirement is retained
#      for the substantive cortical ROIs.
#
#   2. The 60% significance requirement is NOT applied to:
#         - Left Motor Control ROI
#         - Right Motor Control ROI
#
#   3. Motor-control ROIs do NOT determine whether a video is
#      retained or dropped.
#
#   4. Motor-control ISC is calculated for every retained video
#      whenever at least two participants have usable data.
#
#   5. The following subject × video variables are created:
#         Left_Motor_ROI_ISC
#         Right_Motor_ROI_ISC
#         Bilateral_Motor_ROI_ISC
#         Motor_ISC_Lateralization
#
#      and Fisher-z versions:
#         Left_Motor_ROI_ISC_z
#         Right_Motor_ROI_ISC_z
#         Bilateral_Motor_ROI_ISC_z
#         Motor_ISC_Lateralization_z
#
# Positive lateralization:
#     Left motor ISC > Right motor ISC
#
# Primary outputs:
#   motor_roi_isc_df
#   motor_video_isc_df
#   motor_video_roi_isc_df
#   motor_control_isc_wide_df
#   motor_control_isc_complete_df
#   motor_control_coverage_df
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict
from itertools import combinations, compress

from scipy.stats import (
    pearsonr,
    sem,
    rankdata,
    spearmanr
)

import mne

from mne.preprocessing.nirs import (
    optical_density,
    beer_lambert_law,
    temporal_derivative_distribution_repair,
)


# ============================================================
# PATHS AND CONSTANTS
# ============================================================

motor_root = (
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\fNIRS Data\snirf"
)

# Alternative precleaned directory:
#
# motor_root = (
#     r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
#     r"\Emotion Perception Data\fNIRS Data\clean_snirf"
# )

motor_save_dir_corrmats = os.path.join(
    os.getcwd(),
    "motor_control_roi_video_corrmats"
)

os.makedirs(
    motor_save_dir_corrmats,
    exist_ok=True
)


motor_durations = {
    "1": 60.36,
    "3": 60.55,
    "5": 50,
    "7": 95.35,
    "9": 86.12,
    "11": 51.12,
    "13": 48.56,
    "15": 58,
    "17": 103.8,
    "19": 46.59,
    "21": 83.16,
    "23": 67.63,
    "25": 60.36,
    "27": 67.83,
    "29": 158,
    "31": 33.23,
    "33": 35.78,
    "35": 37,
    "37": 51.31,
    "39": 75.49,
    "41": 129.95,
    "43": 81.98,
    "45": 145.29,
    "47": 46.59,
    "49": 58.19,
    "51": 94.37,
    "53": 34.99,
    "55": 55.44
}


# ============================================================
# MOTOR-CONTROL ROI DEFINITIONS
# ============================================================

motor_roi_definitions = {
    "Left RLPFC": [
        "S1_D1 hbo",
        "S1_D2 hbo",
        "S2_D1 hbo",
        "S2_D2 hbo"
    ],

    "Left VLPFC": [
        "S3_D1 hbo",
        "S3_D3 hbo"
    ],

    "Left DLPFC": [
        "S2_D3 hbo",
        "S4_D3 hbo",
        "S4_D2 hbo"
    ],

    "Left OFC": [
        "S1_D15 hbo"
    ],

    "Left STS": [
        "S5_D4 hbo",
        "S5_D10 hbo",
        "S11_D10 hbo",
        "S12_D9 hbo",
        "S12_D10 hbo"
    ],

    "Left TPJ": [
        "S11_D9 hbo",
        "S11_D8 hbo",
        "S13_D8 hbo",
        "S13_D9 hbo"
    ],

    "Right RLPFC": [
        "S6_D5 hbo",
        "S6_D6 hbo",
        "S7_D5 hbo",
        "S7_D6 hbo"
    ],

    "Right DLPFC": [
        "S7_D7 hbo",
        "S8_D5 hbo",
        "S8_D7 hbo"
    ],

    "Right VLPFC": [
        "S9_D7 hbo",
        "S9_D6 hbo"
    ],

    "Right OFC": [
        "S6_D15 hbo"
    ],

    "Right STS": [
        "S10_D13 hbo",
        "S10_D11 hbo",
        "S14_D11 hbo",
        "S14_D14 hbo",
        "S15_D11 hbo"
    ],

    "Right TPJ": [
        "S15_D12 hbo",
        "S15_D14 hbo",
        "S16_D12 hbo",
        "S16_D14 hbo"
    ],

    "Left Motor Control ROI": [
        "S11_D4 hbo"
    ],

    "Right Motor Control ROI": [
        "S15_D13 hbo"
    ]
}


# Explicitly identify the two nuisance-control ROIs.
motor_control_roi_names = [
    "Left Motor Control ROI",
    "Right Motor Control ROI"
]

# The original cortical ROIs continue to determine whether a video
# passes the 60% ISC consistency filter.
motor_substantive_roi_names = [
    roi_name
    for roi_name in motor_roi_definitions
    if roi_name not in motor_control_roi_names
]


# ============================================================
# INDIVIDUAL PREPROCESSING
# ============================================================

def motor_individual_analysis(filepath):
    """
    Load and preprocess one participant's SNIRF file for the
    bilateral motor-control ROI ISC pipeline.
    """

    motor_file = mne.io.read_raw_snirf(
        filepath,
        preload=True,
        verbose=False
    )

    # Enforce durations for odd triggers representing videos.
    for annotation_index, description in enumerate(
        motor_file.annotations.description
    ):
        if description in motor_durations:
            motor_file.annotations.duration[
                annotation_index
            ] = motor_durations[
                description
            ]

    # Remove even-numbered triggers.
    motor_even_triggers = [
        str(value)
        for value in range(
            2,
            57,
            2
        )
    ]

    motor_unwanted_annotations = np.nonzero(
        np.isin(
            motor_file.annotations.description,
            motor_even_triggers
        )
    )

    motor_file.annotations.delete(
        motor_unwanted_annotations
    )

    # Convert raw intensity to optical density.
    motor_raw_od = optical_density(
        motor_file
    )

    # Motion correction.
    motor_raw_od_repaired = (
        temporal_derivative_distribution_repair(
            motor_raw_od
        )
    )

    # Scalp-coupling index and channel removal.
    motor_sci = (
        mne.preprocessing.nirs
        .scalp_coupling_index(
            motor_raw_od_repaired
        )
    )

    motor_bad_channels = list(
        compress(
            motor_raw_od_repaired.ch_names,
            motor_sci < 0.5
        )
    )

    motor_raw_od_repaired.info[
        "bads"
    ] = motor_bad_channels

    if motor_bad_channels:
        print(
            f"Dropping {len(motor_bad_channels)} bad channels "
            f"(SCI < 0.5): {motor_bad_channels}"
        )

        motor_raw_od_repaired = (
            motor_raw_od_repaired
            .copy()
            .drop_channels(
                motor_bad_channels
            )
        )

    else:
        print(
            "No bad channels to drop (SCI < 0.5)."
        )

    # Convert to hemoglobin concentration.
    motor_raw = beer_lambert_law(
        motor_raw_od_repaired,
        ppf=0.1
    )

    motor_raw.resample(
        1.5
    )

    return motor_raw


# ============================================================
# SEGMENT DATA BY VIDEO TRIGGER
# ============================================================

def motor_segment_by_trigger(motor_raw):
    """
    Segment one participant's preprocessed fNIRS data into
    video-specific epochs.
    """

    (
        motor_events,
        motor_event_ids
    ) = mne.events_from_annotations(
        motor_raw,
        verbose=False
    )

    motor_epochs_dict = {}

    for (
        motor_trigger_string,
        motor_trigger_id
    ) in motor_event_ids.items():

        if motor_trigger_string not in motor_durations:
            continue

        motor_trigger_events = motor_events[
            motor_events[:, 2]
            == motor_trigger_id
        ]

        if len(motor_trigger_events) == 0:
            continue

        motor_epochs = mne.Epochs(
            motor_raw,
            motor_trigger_events,
            event_id={
                motor_trigger_string:
                motor_trigger_id
            },
            tmin=0,
            tmax=motor_durations[
                motor_trigger_string
            ],
            baseline=None,
            detrend=None,
            preload=True,
            verbose=False
        )

        motor_epochs_dict[
            motor_trigger_string
        ] = motor_epochs

    return motor_epochs_dict


# ============================================================
# EXTRACT ROI TIME SERIES
# ============================================================

def motor_extract_roi_time_series_np(
    data_2d,
    channel_names,
    roi_channels
):
    """
    Average the available channels assigned to one ROI.

    Returns None when no channel assigned to the ROI is present.
    """

    motor_name_to_index = {
        channel_name: channel_index
        for channel_index, channel_name
        in enumerate(channel_names)
    }

    motor_roi_indices = [
        motor_name_to_index[channel]
        for channel in roi_channels
        if channel in motor_name_to_index
    ]

    if not motor_roi_indices:
        return None

    return data_2d[
        motor_roi_indices
    ].mean(
        axis=0
    )


# ============================================================
# LOAD AND CACHE MOTOR ROI TIME SERIES
#
# motor_roi_ts[subject][trigger][roi] = one-dimensional array
# ============================================================

motor_all_files = [
    filename
    for filename in os.listdir(
        motor_root
    )
    if filename.lower().endswith(
        ".snirf"
    )
]

motor_subject_ids = [
    os.path.splitext(
        filename
    )[0][:6]
    for filename in motor_all_files
]

motor_subject_epochs = {}

motor_roi_ts = defaultdict(
    lambda: defaultdict(dict)
)

# Track whether each motor-control channel survived preprocessing
# and was present in each video epoch.
motor_control_channel_availability_records = []


for (
    motor_subject_id,
    motor_filename
) in zip(
    motor_subject_ids,
    motor_all_files
):

    motor_raw_subject = motor_individual_analysis(
        os.path.join(
            motor_root,
            motor_filename
        )
    )

    motor_epoch_dictionary = (
        motor_segment_by_trigger(
            motor_raw_subject
        )
    )

    motor_subject_epochs[
        motor_subject_id
    ] = motor_epoch_dictionary

    for (
        motor_trigger,
        motor_epochs
    ) in motor_epoch_dictionary.items():

        motor_epoch_data = np.mean(
            motor_epochs.get_data(),
            axis=0
        )

        motor_channel_names = (
            motor_epochs.ch_names
        )

        # Record motor-channel availability after SCI channel removal.
        for motor_control_roi in motor_control_roi_names:

            expected_channels = (
                motor_roi_definitions[
                    motor_control_roi
                ]
            )

            present_channels = [
                channel
                for channel in expected_channels
                if channel in motor_channel_names
            ]

            motor_control_channel_availability_records.append({
                "Subject": motor_subject_id,
                "Trigger": motor_trigger,
                "ROI": motor_control_roi,
                "ExpectedChannels": "; ".join(
                    expected_channels
                ),
                "NExpectedChannels": len(
                    expected_channels
                ),
                "NPresentChannels": len(
                    present_channels
                ),
                "ChannelPresentAfterSCI": (
                    len(present_channels) > 0
                )
            })

        for (
            motor_roi_name,
            motor_roi_channels
        ) in motor_roi_definitions.items():

            motor_roi_time_series = (
                motor_extract_roi_time_series_np(
                    motor_epoch_data,
                    motor_channel_names,
                    motor_roi_channels
                )
            )

            if motor_roi_time_series is not None:

                motor_roi_ts[
                    motor_subject_id
                ][
                    motor_trigger
                ][
                    motor_roi_name
                ] = motor_roi_time_series


motor_control_channel_availability_df = pd.DataFrame(
    motor_control_channel_availability_records
)


motor_subjects = list(
    motor_subject_epochs.keys()
)

motor_triggers_all = sorted({
    trigger
    for subject in motor_roi_ts
    for trigger in motor_roi_ts[
        subject
    ]
})

motor_rois = list(
    motor_roi_definitions.keys()
)


# ============================================================
# HELPERS
# ============================================================

def motor_pairwise_corr_matrix(
    series_list,
    subject_ids
):
    """
    Return an S × S Pearson correlation matrix for aligned series.
    """

    number_subjects = len(
        series_list
    )

    motor_correlation_matrix = np.eye(
        number_subjects,
        dtype=float
    )

    for (
        first_index,
        second_index
    ) in combinations(
        range(number_subjects),
        2
    ):

        correlation_value, _ = pearsonr(
            series_list[
                first_index
            ],
            series_list[
                second_index
            ]
        )

        motor_correlation_matrix[
            first_index,
            second_index
        ] = correlation_value

        motor_correlation_matrix[
            second_index,
            first_index
        ] = correlation_value

    return pd.DataFrame(
        motor_correlation_matrix,
        index=subject_ids,
        columns=subject_ids
    )


def motor_safe_atanh(correlation):
    """
    Fisher-z transform a correlation after clipping.
    """

    if pd.isna(correlation):
        return np.nan

    return np.arctanh(
        np.clip(
            correlation,
            -0.999999,
            0.999999
        )
    )


def motor_safe_tanh(fisher_z):
    """
    Convert a Fisher-z value back to correlation units.
    """

    if pd.isna(fisher_z):
        return np.nan

    return np.tanh(
        fisher_z
    )


def motor_vectorize_lower_tri(matrix):
    """
    Extract lower-triangular values from a square matrix.
    """

    matrix = np.asarray(
        matrix,
        dtype=float
    )

    matrix_size = matrix.shape[0]

    lower_indices = np.tril_indices(
        matrix_size,
        k=-1
    )

    return matrix[
        lower_indices
    ]


def motor_r_and_p(x, y):
    """
    Return Pearson r and p after finite-value and variance checks.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 3
        or np.std(x, ddof=1) == 0
        or np.std(y, ddof=1) == 0
    ):
        return np.nan, np.nan

    correlation_value, p_value = pearsonr(
        x,
        y
    )

    return (
        correlation_value,
        p_value
    )


# ============================================================
# VIDEO AND ROI QUALITY FILTER
#
# IMPORTANT:
#   The 60% significance rule is applied only to substantive ROIs.
#   It is not applied to either motor-control ROI.
# ============================================================

motor_alpha = 0.051

motor_min_subjects_with_data = 27

motor_p_sig_required = 0.60

# Motor nuisance ISC only requires the target participant plus
# at least one comparison participant.
motor_control_min_subjects_for_isc = 2


motor_roi_pass_by_trigger = defaultdict(
    dict
)

motor_roi_has_data_by_trigger = defaultdict(
    set
)

motor_roi_sig_counts_by_trigger = defaultdict(
    dict
)

motor_roi_n_by_trigger = defaultdict(
    dict
)

# Separate availability tracking for the two motor-control ROIs.
motor_control_roi_n_by_trigger = defaultdict(
    dict
)

motor_control_roi_available_triggers = {
    motor_roi: []
    for motor_roi in motor_control_roi_names
}


# ------------------------------------------------------------
# 1. Apply 60% significance filter to substantive ROIs only
# ------------------------------------------------------------

for motor_trigger in motor_triggers_all:

    for motor_roi in motor_substantive_roi_names:

        motor_subjects_with_data = [
            subject
            for subject in motor_subjects
            if motor_roi
            in motor_roi_ts[
                subject
            ].get(
                motor_trigger,
                {}
            )
        ]

        motor_number_subjects = len(
            motor_subjects_with_data
        )

        motor_roi_n_by_trigger[
            motor_trigger
        ][
            motor_roi
        ] = motor_number_subjects

        if (
            motor_number_subjects
            < motor_min_subjects_with_data
        ):
            continue

        motor_minimum_length = min(
            len(
                motor_roi_ts[
                    subject
                ][
                    motor_trigger
                ][
                    motor_roi
                ]
            )
            for subject
            in motor_subjects_with_data
        )

        motor_roi_matrix = np.stack(
            [
                motor_roi_ts[
                    subject
                ][
                    motor_trigger
                ][
                    motor_roi
                ][
                    :motor_minimum_length
                ]
                for subject
                in motor_subjects_with_data
            ],
            axis=0
        )

        motor_roi_has_data_by_trigger[
            motor_trigger
        ].add(
            motor_roi
        )

        motor_roi_sum = (
            motor_roi_matrix.sum(
                axis=0
            )
        )

        motor_significant_count = 0

        for subject_index in range(
            motor_number_subjects
        ):

            motor_leave_one_out_mean = (
                motor_roi_sum
                - motor_roi_matrix[
                    subject_index
                ]
            ) / (
                motor_number_subjects
                - 1
            )

            (
                motor_correlation_value,
                motor_p_value
            ) = motor_r_and_p(
                motor_roi_matrix[
                    subject_index
                ],
                motor_leave_one_out_mean
            )

            if (
                np.isfinite(
                    motor_correlation_value
                )
                and np.isfinite(
                    motor_p_value
                )
                and motor_p_value
                < motor_alpha
            ):
                motor_significant_count += 1

        motor_required_count = math.ceil(
            motor_p_sig_required
            * motor_number_subjects
        )

        motor_roi_sig_counts_by_trigger[
            motor_trigger
        ][
            motor_roi
        ] = motor_significant_count

        motor_roi_pass_by_trigger[
            motor_trigger
        ][
            motor_roi
        ] = (
            motor_significant_count
            >= motor_required_count
        )


# ------------------------------------------------------------
# 2. Record motor-control ROI availability without significance
# ------------------------------------------------------------

for motor_trigger in motor_triggers_all:

    for motor_roi in motor_control_roi_names:

        motor_subjects_with_data = [
            subject
            for subject in motor_subjects
            if motor_roi
            in motor_roi_ts[
                subject
            ].get(
                motor_trigger,
                {}
            )
        ]

        motor_number_subjects = len(
            motor_subjects_with_data
        )

        motor_control_roi_n_by_trigger[
            motor_trigger
        ][
            motor_roi
        ] = motor_number_subjects

        if (
            motor_number_subjects
            >= motor_control_min_subjects_for_isc
        ):
            motor_control_roi_available_triggers[
                motor_roi
            ].append(
                motor_trigger
            )


# ------------------------------------------------------------
# 3. Video-level decision uses substantive ROIs only
#
# A video is retained when at least half of substantive ROIs
# with sufficient data pass the original 60% criterion.
# ------------------------------------------------------------

motor_kept_triggers = []
motor_dropped_triggers = []
motor_video_diagnostics = []


for motor_trigger in motor_triggers_all:

    motor_substantive_rois_with_data = sorted(
        motor_roi_has_data_by_trigger.get(
            motor_trigger,
            []
        )
    )

    if len(
        motor_substantive_rois_with_data
    ) == 0:

        motor_dropped_triggers.append(
            motor_trigger
        )

        motor_video_diagnostics.append({
            "Trigger": motor_trigger,
            "Substantive_ROIs_with_data": 0,
            "Substantive_ROIs_pass": 0,
            "Required_ROIs_pass": np.nan,
            "LeftMotor_N": (
                motor_control_roi_n_by_trigger[
                    motor_trigger
                ].get(
                    "Left Motor Control ROI",
                    0
                )
            ),
            "RightMotor_N": (
                motor_control_roi_n_by_trigger[
                    motor_trigger
                ].get(
                    "Right Motor Control ROI",
                    0
                )
            ),
            "Decision": "DROP"
        })

        continue

    motor_number_rois_passing = sum(
        bool(
            motor_roi_pass_by_trigger[
                motor_trigger
            ].get(
                motor_roi,
                False
            )
        )
        for motor_roi
        in motor_substantive_rois_with_data
    )

    motor_video_roi_threshold = math.ceil(
        len(
            motor_substantive_rois_with_data
        ) / 2.0
    )

    motor_video_decision = (
        "KEEP"
        if (
            motor_number_rois_passing
            >= motor_video_roi_threshold
        )
        else "DROP"
    )

    if motor_video_decision == "KEEP":

        motor_kept_triggers.append(
            motor_trigger
        )

    else:

        motor_dropped_triggers.append(
            motor_trigger
        )

    motor_video_diagnostics.append({
        "Trigger": motor_trigger,

        "Substantive_ROIs_with_data": len(
            motor_substantive_rois_with_data
        ),

        "Substantive_ROIs_pass": (
            motor_number_rois_passing
        ),

        "Required_ROIs_pass": (
            motor_video_roi_threshold
        ),

        "LeftMotor_N": (
            motor_control_roi_n_by_trigger[
                motor_trigger
            ].get(
                "Left Motor Control ROI",
                0
            )
        ),

        "RightMotor_N": (
            motor_control_roi_n_by_trigger[
                motor_trigger
            ].get(
                "Right Motor Control ROI",
                0
            )
        ),

        "Decision": motor_video_decision
    })


print(
    "Video-level substantive-ROI filter ➜ "
    f"kept {len(motor_kept_triggers)} videos, "
    f"dropped {len(motor_dropped_triggers)} videos."
)

if motor_dropped_triggers:

    print(
        "Dropped entirely:",
        ", ".join(
            map(
                str,
                sorted(
                    motor_dropped_triggers
                )
            )
        )
    )


motor_video_diagnostics_df = pd.DataFrame(
    motor_video_diagnostics
).sort_values(
    "Trigger"
)


# ============================================================
# BUILD ROI-SPECIFIC TRIGGER LISTS
#
# Substantive ROI:
#   retained video + passed 60% significance requirement
#
# Motor-control ROI:
#   retained video + at least two subjects with usable data
# ============================================================

motor_roi_analysis_trigs = {
    motor_roi: []
    for motor_roi in motor_rois
}


for motor_trigger in motor_kept_triggers:

    # Substantive cortical ROIs retain the original filter.
    for motor_roi in motor_substantive_roi_names:

        if motor_roi_pass_by_trigger[
            motor_trigger
        ].get(
            motor_roi,
            False
        ):
            motor_roi_analysis_trigs[
                motor_roi
            ].append(
                motor_trigger
            )

    # Motor-control ROIs bypass the 60% significance requirement.
    for motor_roi in motor_control_roi_names:

        motor_n_available = (
            motor_control_roi_n_by_trigger[
                motor_trigger
            ].get(
                motor_roi,
                0
            )
        )

        if (
            motor_n_available
            >= motor_control_min_subjects_for_isc
        ):
            motor_roi_analysis_trigs[
                motor_roi
            ].append(
                motor_trigger
            )


# Backward-compatible name for downstream code.
motor_roi_significant_trigs = (
    motor_roi_analysis_trigs
)

# Restrict downstream analyses to retained videos.
motor_triggers = motor_kept_triggers


# ============================================================
# MOTOR ROI-LEVEL ISC
#
# motor_roi_isc_df:
#   subject × ROI, averaged across eligible retained triggers
#
# motor_video_isc_df:
#   subject × trigger, averaged across available retained ROIs
#
# motor_video_roi_isc_df:
#   subject × trigger × ROI ISC
# ============================================================

motor_roi_isc_records = []
motor_video_isc_records = []
motor_video_roi_isc_records = []

motor_control_coverage_records = []


for motor_subject in motor_subjects:

    motor_subject_roi_aggregate = {
        "Subject": motor_subject
    }

    motor_per_trigger_roi_values = defaultdict(
        list
    )

    for (
        motor_roi,
        motor_good_triggers
    ) in motor_roi_analysis_trigs.items():

        motor_roi_correlations = []

        for motor_trigger in motor_good_triggers:

            subject_has_roi = (
                motor_roi
                in motor_roi_ts[
                    motor_subject
                ].get(
                    motor_trigger,
                    {}
                )
            )

            # Record explicit motor ROI coverage.
            if motor_roi in motor_control_roi_names:

                motor_control_coverage_records.append({
                    "Subject": motor_subject,
                    "Trigger": motor_trigger,
                    "ROI": motor_roi,
                    "VideoRetained": (
                        motor_trigger
                        in motor_kept_triggers
                    ),
                    "SubjectHasROI": subject_has_roi,
                    "ISCComputed": False,
                    "NComparisonSubjects": np.nan,
                    "MissingnessReason": (
                        ""
                        if subject_has_roi
                        else (
                            "Motor channel absent after "
                            "preprocessing/SCI"
                        )
                    )
                })

            if not subject_has_roi:
                continue

            motor_subjects_with_data = [
                subject
                for subject in motor_subjects
                if motor_roi
                in motor_roi_ts[
                    subject
                ].get(
                    motor_trigger,
                    {}
                )
            ]

            motor_number_subjects = len(
                motor_subjects_with_data
            )

            if (
                motor_subject
                not in motor_subjects_with_data
                or motor_number_subjects
                < motor_control_min_subjects_for_isc
            ):

                if motor_roi in motor_control_roi_names:

                    matching_index = len(
                        motor_control_coverage_records
                    ) - 1

                    motor_control_coverage_records[
                        matching_index
                    ][
                        "NComparisonSubjects"
                    ] = motor_number_subjects

                    motor_control_coverage_records[
                        matching_index
                    ][
                        "MissingnessReason"
                    ] = "Too few subjects with usable ROI data"

                continue

            motor_minimum_length = min(
                len(
                    motor_roi_ts[
                        subject
                    ][
                        motor_trigger
                    ][
                        motor_roi
                    ]
                )
                for subject
                in motor_subjects_with_data
            )

            motor_roi_matrix = np.stack(
                [
                    motor_roi_ts[
                        subject
                    ][
                        motor_trigger
                    ][
                        motor_roi
                    ][
                        :motor_minimum_length
                    ]
                    for subject
                    in motor_subjects_with_data
                ],
                axis=0
            )

            motor_roi_sum = (
                motor_roi_matrix.sum(
                    axis=0
                )
            )

            motor_subject_index = (
                motor_subjects_with_data.index(
                    motor_subject
                )
            )

            motor_leave_one_out_mean = (
                motor_roi_sum
                - motor_roi_matrix[
                    motor_subject_index
                ]
            ) / (
                motor_number_subjects
                - 1
            )

            (
                motor_correlation_value,
                _
            ) = motor_r_and_p(
                motor_roi_matrix[
                    motor_subject_index
                ],
                motor_leave_one_out_mean
            )

            if not np.isfinite(
                motor_correlation_value
            ):

                if motor_roi in motor_control_roi_names:

                    matching_index = len(
                        motor_control_coverage_records
                    ) - 1

                    motor_control_coverage_records[
                        matching_index
                    ][
                        "NComparisonSubjects"
                    ] = motor_number_subjects

                    motor_control_coverage_records[
                        matching_index
                    ][
                        "MissingnessReason"
                    ] = (
                        "Constant or nonfinite time series"
                    )

                continue

            motor_roi_correlations.append(
                motor_correlation_value
            )

            motor_per_trigger_roi_values[
                motor_trigger
            ].append(
                motor_correlation_value
            )

            motor_video_roi_isc_records.append({
                "Subject": motor_subject,
                "Trigger": motor_trigger,
                "ROI": motor_roi,
                "ISC": float(
                    motor_correlation_value
                ),
                "ISC_z": float(
                    motor_safe_atanh(
                        motor_correlation_value
                    )
                )
            })

            if motor_roi in motor_control_roi_names:

                matching_index = len(
                    motor_control_coverage_records
                ) - 1

                motor_control_coverage_records[
                    matching_index
                ][
                    "ISCComputed"
                ] = True

                motor_control_coverage_records[
                    matching_index
                ][
                    "NComparisonSubjects"
                ] = motor_number_subjects

                motor_control_coverage_records[
                    matching_index
                ][
                    "MissingnessReason"
                ] = ""

        # Average correlations across triggers in Fisher-z space,
        # then transform back to r units.
        if motor_roi_correlations:

            motor_roi_z_values = [
                motor_safe_atanh(
                    value
                )
                for value in motor_roi_correlations
            ]

            motor_subject_roi_aggregate[
                motor_roi
            ] = float(
                motor_safe_tanh(
                    np.mean(
                        motor_roi_z_values
                    )
                )
            )

        else:

            motor_subject_roi_aggregate[
                motor_roi
            ] = np.nan

    motor_roi_isc_records.append(
        motor_subject_roi_aggregate
    )

    # Per-video ISC averaged across available ROIs in Fisher-z space.
    for (
        motor_trigger,
        motor_values
    ) in motor_per_trigger_roi_values.items():

        if motor_values:

            motor_video_z_values = [
                motor_safe_atanh(
                    value
                )
                for value in motor_values
            ]

            motor_video_isc_records.append({
                "Subject": motor_subject,
                "Trigger": motor_trigger,

                "ISC": float(
                    motor_safe_tanh(
                        np.mean(
                            motor_video_z_values
                        )
                    )
                ),

                "ISC_z": float(
                    np.mean(
                        motor_video_z_values
                    )
                ),

                "N_ROIs_Available": len(
                    motor_values
                )
            })


motor_roi_isc_df = pd.DataFrame(
    motor_roi_isc_records
)

motor_video_isc_df = pd.DataFrame(
    motor_video_isc_records
)

motor_video_roi_isc_df = pd.DataFrame(
    motor_video_roi_isc_records
)

motor_control_coverage_df = pd.DataFrame(
    motor_control_coverage_records
)


# ============================================================
# CREATE BILATERAL MOTOR ISC AND LATERALIZATION VARIABLES
# ============================================================

motor_control_long_df = (
    motor_video_roi_isc_df[
        motor_video_roi_isc_df[
            "ROI"
        ].isin(
            motor_control_roi_names
        )
    ]
    .copy()
)


if motor_control_long_df.empty:

    motor_control_isc_wide_df = pd.DataFrame(
        columns=[
            "Subject",
            "Trigger",
            "Left_Motor_ROI_ISC",
            "Right_Motor_ROI_ISC",
            "Left_Motor_ROI_ISC_z",
            "Right_Motor_ROI_ISC_z",
            "Bilateral_Motor_ROI_ISC",
            "Bilateral_Motor_ROI_ISC_z",
            "Motor_ISC_Lateralization",
            "Motor_ISC_Lateralization_z",
            "Both_Motor_ROIs_Available"
        ]
    )

else:

    # Raw correlation-unit values.
    motor_control_raw_wide = (
        motor_control_long_df
        .pivot_table(
            index=[
                "Subject",
                "Trigger"
            ],
            columns="ROI",
            values="ISC",
            aggfunc="mean"
        )
        .reset_index()
        .rename(
            columns={
                "Left Motor Control ROI":
                    "Left_Motor_ROI_ISC",

                "Right Motor Control ROI":
                    "Right_Motor_ROI_ISC"
            }
        )
    )

    # Fisher-z values.
    motor_control_z_wide = (
        motor_control_long_df
        .pivot_table(
            index=[
                "Subject",
                "Trigger"
            ],
            columns="ROI",
            values="ISC_z",
            aggfunc="mean"
        )
        .reset_index()
        .rename(
            columns={
                "Left Motor Control ROI":
                    "Left_Motor_ROI_ISC_z",

                "Right Motor Control ROI":
                    "Right_Motor_ROI_ISC_z"
            }
        )
    )

    motor_control_isc_wide_df = (
        motor_control_raw_wide
        .merge(
            motor_control_z_wide,
            on=[
                "Subject",
                "Trigger"
            ],
            how="outer",
            validate="one_to_one"
        )
    )

    # Ensure all expected columns exist.
    for expected_column in [
        "Left_Motor_ROI_ISC",
        "Right_Motor_ROI_ISC",
        "Left_Motor_ROI_ISC_z",
        "Right_Motor_ROI_ISC_z"
    ]:
        if (
            expected_column
            not in motor_control_isc_wide_df.columns
        ):
            motor_control_isc_wide_df[
                expected_column
            ] = np.nan

    motor_control_isc_wide_df[
        "Both_Motor_ROIs_Available"
    ] = (
        motor_control_isc_wide_df[
            "Left_Motor_ROI_ISC_z"
        ].notna()
        &
        motor_control_isc_wide_df[
            "Right_Motor_ROI_ISC_z"
        ].notna()
    )

    # Bilateral mean in Fisher-z space.
    motor_control_isc_wide_df[
        "Bilateral_Motor_ROI_ISC_z"
    ] = (
        motor_control_isc_wide_df[
            [
                "Left_Motor_ROI_ISC_z",
                "Right_Motor_ROI_ISC_z"
            ]
        ]
        .mean(
            axis=1,
            skipna=True
        )
    )

    motor_control_isc_wide_df[
        "Bilateral_Motor_ROI_ISC"
    ] = np.tanh(
        motor_control_isc_wide_df[
            "Bilateral_Motor_ROI_ISC_z"
        ]
    )

    # Preferred lateralization variable:
    #
    # Positive values indicate greater left than right motor ISC.
    # It is computed only when both sides are available.
    motor_control_isc_wide_df[
        "Motor_ISC_Lateralization_z"
    ] = np.where(
        motor_control_isc_wide_df[
            "Both_Motor_ROIs_Available"
        ],

        motor_control_isc_wide_df[
            "Left_Motor_ROI_ISC_z"
        ]
        -
        motor_control_isc_wide_df[
            "Right_Motor_ROI_ISC_z"
        ],

        np.nan
    )

    # Raw-r difference retained for descriptive interpretation.
    motor_control_isc_wide_df[
        "Motor_ISC_Lateralization"
    ] = np.where(
        motor_control_isc_wide_df[
            "Both_Motor_ROIs_Available"
        ],

        motor_control_isc_wide_df[
            "Left_Motor_ROI_ISC"
        ]
        -
        motor_control_isc_wide_df[
            "Right_Motor_ROI_ISC"
        ],

        np.nan
    )


# Complete bilateral sample for models requiring lateralization.
motor_control_isc_complete_df = (
    motor_control_isc_wide_df[
        motor_control_isc_wide_df[
            "Both_Motor_ROIs_Available"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# COVERAGE SUMMARIES
# ============================================================

motor_control_coverage_summary_df = (
    motor_control_coverage_df
    .groupby(
        "ROI",
        dropna=False
    )
    .agg(
        SubjectVideoRows=(
            "Subject",
            "size"
        ),

        ChannelPresentRows=(
            "SubjectHasROI",
            "sum"
        ),

        ISCComputedRows=(
            "ISCComputed",
            "sum"
        ),

        UniqueSubjects=(
            "Subject",
            "nunique"
        ),

        UniqueTriggers=(
            "Trigger",
            "nunique"
        )
    )
    .reset_index()
)

motor_control_coverage_summary_df[
    "ChannelPresentProportion"
] = (
    motor_control_coverage_summary_df[
        "ChannelPresentRows"
    ]
    /
    motor_control_coverage_summary_df[
        "SubjectVideoRows"
    ]
)

motor_control_coverage_summary_df[
    "ISCComputedProportion"
] = (
    motor_control_coverage_summary_df[
        "ISCComputedRows"
    ]
    /
    motor_control_coverage_summary_df[
        "SubjectVideoRows"
    ]
)


motor_control_missingness_reason_df = (
    motor_control_coverage_df[
        ~motor_control_coverage_df[
            "ISCComputed"
        ]
    ]
    .groupby(
        [
            "ROI",
            "MissingnessReason"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="N"
    )
)


# ============================================================
# SAVE TABLE OUTPUTS
# ============================================================

motor_roi_isc_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "motor_roi_isc_subject_average.csv"
    ),
    index=False
)

motor_video_isc_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "motor_video_mean_isc.csv"
    ),
    index=False
)

motor_video_roi_isc_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "motor_video_roi_isc_long.csv"
    ),
    index=False
)

motor_control_isc_wide_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "bilateral_motor_control_isc_and_lateralization.csv"
    ),
    index=False
)

motor_control_isc_complete_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "bilateral_motor_control_complete_cases.csv"
    ),
    index=False
)

motor_control_coverage_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "motor_control_roi_coverage_by_subject_video.csv"
    ),
    index=False
)

motor_control_coverage_summary_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "motor_control_roi_coverage_summary.csv"
    ),
    index=False
)

motor_control_missingness_reason_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "motor_control_roi_missingness_reasons.csv"
    ),
    index=False
)

motor_video_diagnostics_df.to_csv(
    os.path.join(
        motor_save_dir_corrmats,
        "motor_video_filter_diagnostics.csv"
    ),
    index=False
)


# ============================================================
# PRINT PRIMARY DATAFRAMES
# ============================================================

print(
    "\n✅ Motor-control ROI-level ISC "
    "(averaged across eligible retained triggers):"
)

display(
    motor_roi_isc_df.head()
)


print(
    "\n✅ Motor-control per-video ROI-level ISC "
    "(Subject × Trigger × ROI):"
)

display(
    motor_video_roi_isc_df.head()
)


print(
    "\n✅ Bilateral motor-control ISC and lateralization:"
)

display(
    motor_control_isc_wide_df.head(20)
)


print(
    "\n✅ Motor-control ROI coverage summary:"
)

display(
    motor_control_coverage_summary_df
)


print(
    "\n✅ Motor-control missingness reasons:"
)

display(
    motor_control_missingness_reason_df
)


# ============================================================
# GROUP-LEVEL ROI BAR PLOT
#
# Mean ± 95% CI across subjects.
# ============================================================

motor_roi_columns = [
    column
    for column in motor_roi_isc_df.columns
    if column != "Subject"
]

motor_roi_means = []
motor_roi_confidence_intervals = []
motor_roi_significance_mask = []
motor_roi_sample_sizes = []


for motor_roi in motor_roi_columns:

    motor_values = (
        motor_roi_isc_df[
            motor_roi
        ]
        .dropna()
        .values
    )

    motor_roi_sample_sizes.append(
        len(
            motor_values
        )
    )

    if len(motor_values) == 0:

        motor_roi_means.append(
            np.nan
        )

        motor_roi_confidence_intervals.append(
            np.nan
        )

        motor_roi_significance_mask.append(
            False
        )

        continue

    motor_roi_mean = np.mean(
        motor_values
    )

    motor_roi_ci95 = (
        1.96
        * sem(
            motor_values
        )
        if len(motor_values) > 1
        else 0.0
    )

    motor_roi_means.append(
        motor_roi_mean
    )

    motor_roi_confidence_intervals.append(
        motor_roi_ci95
    )

    motor_roi_significance_mask.append(
        motor_roi_mean
        - motor_roi_ci95
        > 0.0
    )


motor_roi_order = sorted(
    motor_roi_columns,
    key=lambda roi_name: (
        roi_name.split()[0]
        not in [
            "Left",
            "Right"
        ],
        roi_name
    )
)


plt.figure(
    figsize=(14, 6)
)

motor_bars = plt.bar(
    motor_roi_order,

    [
        motor_roi_means[
            motor_roi_columns.index(
                motor_roi
            )
        ]
        for motor_roi in motor_roi_order
    ],

    yerr=[
        motor_roi_confidence_intervals[
            motor_roi_columns.index(
                motor_roi
            )
        ]
        for motor_roi in motor_roi_order
    ],

    capsize=5,
    color="skyblue",
    edgecolor="black"
)


for (
    motor_bar,
    motor_roi
) in zip(
    motor_bars,
    motor_roi_order
):

    motor_roi_index = (
        motor_roi_columns.index(
            motor_roi
        )
    )

    motor_mean = motor_roi_means[
        motor_roi_index
    ]

    motor_ci = motor_roi_confidence_intervals[
        motor_roi_index
    ]

    if np.isfinite(
        motor_mean
    ):

        plt.text(
            motor_bar.get_x()
            + motor_bar.get_width() / 2,

            motor_mean
            + (
                motor_ci
                if np.isfinite(
                    motor_ci
                )
                else 0
            )
            + 0.005,

            f"{motor_mean:.2f}",

            ha="center",
            va="bottom",
            fontsize=8
        )

    if (
        np.isfinite(
            motor_mean
        )
        and np.isfinite(
            motor_ci
        )
        and motor_roi_significance_mask[
            motor_roi_index
        ]
    ):

        plt.text(
            motor_bar.get_x()
            + motor_bar.get_width() / 2,

            motor_mean
            + motor_ci
            + 0.02,

            "*",

            ha="center",
            va="bottom",
            fontsize=14
        )


plt.ylabel(
    "Mean ROI ISC across eligible retained triggers"
)

plt.title(
    "Group-level ROI ISC (Mean ± 95% CI)\n"
    "Motor-control ROIs were not subjected to the 60% "
    "per-video significance filter"
)

plt.xticks(
    rotation=45,
    ha="right"
)


finite_lower_values = (
    np.array(
        motor_roi_means,
        dtype=float
    )
    -
    np.array(
        motor_roi_confidence_intervals,
        dtype=float
    )
)

finite_upper_values = (
    np.array(
        motor_roi_means,
        dtype=float
    )
    +
    np.array(
        motor_roi_confidence_intervals,
        dtype=float
    )
)

finite_lower_values = finite_lower_values[
    np.isfinite(
        finite_lower_values
    )
]

finite_upper_values = finite_upper_values[
    np.isfinite(
        finite_upper_values
    )
]

if (
    len(finite_lower_values) > 0
    and len(finite_upper_values) > 0
):

    plt.ylim(
        bottom=min(
            0,
            np.min(
                finite_lower_values
            )
        ) - 0.02,

        top=np.max(
            finite_upper_values
        ) + 0.06
    )


plt.tight_layout()


motor_roi_bar_output = os.path.join(
    motor_save_dir_corrmats,
    "group_level_bilateral_motor_control_ROI_ISC_barplot.png"
)

plt.savefig(
    motor_roi_bar_output,
    dpi=220,
    bbox_inches="tight"
)

plt.show()


print(
    "🖼️ Saved group-level bilateral motor-control ROI ISC "
    f"bar plot ➜ {motor_roi_bar_output}"
)


# ============================================================
# LATERALIZATION DISTRIBUTION PLOT
# ============================================================

if (
    not motor_control_isc_complete_df.empty
    and motor_control_isc_complete_df[
        "Motor_ISC_Lateralization_z"
    ].notna().any()
):

    plt.figure(
        figsize=(9, 5)
    )

    plt.hist(
        motor_control_isc_complete_df[
            "Motor_ISC_Lateralization_z"
        ].dropna(),
        bins=30,
        edgecolor="black",
        alpha=0.80
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1.5,
        label="No lateralization"
    )

    lateralization_mean = (
        motor_control_isc_complete_df[
            "Motor_ISC_Lateralization_z"
        ].mean()
    )

    plt.axvline(
        lateralization_mean,
        linewidth=2,
        label=(
            f"Mean = "
            f"{lateralization_mean:.3f}"
        )
    )

    plt.xlabel(
        "Motor ISC lateralization "
        "(Left Fisher-z ISC − Right Fisher-z ISC)"
    )

    plt.ylabel(
        "Subject × video observations"
    )

    plt.title(
        "Distribution of bilateral motor ISC lateralization"
    )

    plt.legend(
        frameon=False
    )

    plt.tight_layout()

    motor_lateralization_output = os.path.join(
        motor_save_dir_corrmats,
        "motor_isc_lateralization_distribution.png"
    )

    plt.savefig(
        motor_lateralization_output,
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()

    print(
        "🖼️ Saved motor ISC lateralization plot ➜ "
        f"{motor_lateralization_output}"
    )


# ============================================================
# FINAL MOTOR-CONTROL OBJECT SUMMARY
# ============================================================

print("\n" + "=" * 95)
print("BILATERAL MOTOR-CONTROL ROI ISC PIPELINE COMPLETE")
print("=" * 95)

print(
    f"\nSubjects processed: "
    f"{len(motor_subjects)}"
)

print(
    f"Videos retained by substantive-ROI filtering: "
    f"{len(motor_kept_triggers)}"
)

print(
    f"Videos dropped by substantive-ROI filtering: "
    f"{len(motor_dropped_triggers)}"
)

print(
    "\nMotor-control filtering rule:"
)

print(
    "  Left and right motor-control ROIs were calculated for "
    "every retained video whenever at least two participants had "
    "usable data."
)

print(
    "  The 60% participant-significance threshold was not used "
    "for either motor-control ROI."
)

print(
    "  Motor-control ROIs did not contribute to the video "
    "keep/drop decision."
)

print(
    "\nPrimary model-ready motor variables:"
)

print(
    "  Left_Motor_ROI_ISC_z"
)

print(
    "  Right_Motor_ROI_ISC_z"
)

print(
    "  Bilateral_Motor_ROI_ISC_z"
)

print(
    "  Motor_ISC_Lateralization_z"
)

print(
    "\nRecommended model parameterization:"
)

print(
    "  Include Bilateral_Motor_ROI_ISC_z to control overall "
    "motor-region synchrony."
)

print(
    "  Include Motor_ISC_Lateralization_z to control left-versus-"
    "right motor synchrony."
)

print(
    "  Avoid entering left ISC, right ISC, bilateral mean, and "
    "lateralization simultaneously because they are mathematically "
    "dependent."
)

print(
    "\nPrimary output objects:"
)

print(
    "  motor_roi_isc_df"
)

print(
    "  motor_video_isc_df"
)

print(
    "  motor_video_roi_isc_df"
)

print(
    "  motor_control_isc_wide_df"
)

print(
    "  motor_control_isc_complete_df"
)

print(
    "  motor_control_coverage_df"
)

print(
    "  motor_control_coverage_summary_df"
)

print(
    "  motor_control_missingness_reason_df"
)

print(
    "  motor_video_diagnostics_df"
)

print(
    f"\nAll saved outputs are located in:\n"
    f"{motor_save_dir_corrmats}"
)

Dropping 26 bad channels (SCI < 0.5): ['S5_D16 760', 'S7_D7 760', 'S8_D5 760', 'S10_D17 760', 'S12_D9 760', 'S12_D10 760', 'S12_D19 760', 'S14_D11 760', 'S14_D14 760', 'S15_D14 760', 'S16_D12 760', 'S16_D14 760', 'S16_D23 760', 'S5_D16 850', 'S7_D7 850', 'S8_D5 850', 'S10_D17 850', 'S12_D9 850', 'S12_D10 850', 'S12_D19 850', 'S14_D11 850', 'S14_D14 850', 'S15_D14 850', 'S16_D12 850', 'S16_D14 850', 'S16_D23 850']
Dropping 12 bad channels (SCI < 0.5): ['S2_D2 760', 'S10_D17 760', 'S14_D11 760', 'S14_D14 760', 'S14_D21 760', 'S16_D23 760', 'S2_D2 850', 'S10_D17 850', 'S14_D11 850', 'S14_D14 850', 'S14_D21 850', 'S16_D23 850']
Dropping 6 bad channels (SCI < 0.5): ['S2_D3 760', 'S5_D16 760', 'S14_D21 760', 'S2_D3 850', 'S5_D16 850', 'S14_D21 850']
Dropping 20 bad channels (SCI < 0.5): ['S2_D3 760', 'S5_D10 760', 'S5_D16 760', 'S7_D7 760', 'S10_D17 760', 'S11_D18 760', 'S12_D10 760', 'S13_D20 760', 'S14_D11 760', 'S14_D21 760', 'S2_D3 850', 'S5_D10 850', 'S5_D16 850', 'S7_D7 850', 'S10_D17 

,Subject,Left RLPFC,Left VLPFC,Left DLPFC,Left OFC,Left STS,Left TPJ,Right RLPFC,Right DLPFC,Right VLPFC,Right OFC,Right STS,Right TPJ,Left Motor Control ROI,Right Motor Control ROI
0,pid001,0.527048,0.291978,0.342353,0.404860,0.231873,0.250520,0.494236,0.449645,0.316085,0.582944,0.454625,0.243057,-0.036606,-0.081927
1,pid002,0.196914,0.147519,0.256043,0.007936,0.158568,0.179166,0.196960,0.251436,0.228511,0.285144,0.369370,0.202367,-0.025092,-0.004919
2,pid003,0.464313,0.370762,0.357097,0.238883,0.380159,0.249244,0.444948,0.515603,0.268951,0.372650,0.526786,0.299986,0.172354,0.324916
3,pid004,0.385715,0.318657,0.285428,0.276419,-0.038224,0.139669,0.486450,0.413852,0.504697,0.437771,0.302715,0.121602,0.086328,0.267293
4,pid005,0.097559,0.077974,0.110850,-0.056100,0.057888,0.037728,0.244984,0.228191,0.235249,-0.041304,0.104475,0.196033,0.092513,-0.034729



✅ Motor-control per-video ROI-level ISC (Subject × Trigger × ROI):


,Subject,Trigger,ROI,ISC,ISC_z
0,pid001,1,Left RLPFC,0.281920,0.289767
1,pid001,11,Left RLPFC,0.904785,1.497991
2,pid001,13,Left RLPFC,-0.170417,-0.172096
3,pid001,15,Left RLPFC,0.786612,1.062482
4,pid001,17,Left RLPFC,0.151598,0.152775



✅ Bilateral motor-control ISC and lateralization:


...


✅ Motor-control ROI coverage summary:


,ROI,SubjectVideoRows,ChannelPresentRows,ISCComputedRows,UniqueSubjects,UniqueTriggers,ChannelPresentProportion,ISCComputedProportion
0,Left Motor Control ROI,1176,1092,1092,42,28,0.928571,0.928571
1,Right Motor Control ROI,1176,1036,1036,42,28,0.880952,0.880952



✅ Motor-control missingness reasons:


,ROI,MissingnessReason,N
0,Left Motor Control ROI,Motor channel absent after preprocessing/SCI,84
1,Right Motor Control ROI,Motor channel absent after preprocessing/SCI,140


🖼️ Saved group-level bilateral motor-control ROI ISC bar plot ➜ C:\Users\pkell\motor_control_roi_video_corrmats\group_level_bilateral_motor_control_ROI_ISC_barplot.png
🖼️ Saved motor ISC lateralization plot ➜ C:\Users\pkell\motor_control_roi_video_corrmats\motor_isc_lateralization_distribution.png

BILATERAL MOTOR-CONTROL ROI ISC PIPELINE COMPLETE

Subjects processed: 42
Videos retained by substantive-ROI filtering: 28
Videos dropped by substantive-ROI filtering: 0

Motor-control filtering rule:
  Left and right motor-control ROIs were calculated for every retained video whenever at least two participants had usable data.
  The 60% participant-significance threshold was not used for either motor-control ROI.
  Motor-control ROIs did not contribute to the video keep/drop decision.

Primary model-ready motor variables:
  Left_Motor_ROI_ISC_z
  Right_Motor_ROI_ISC_z
  Bilateral_Motor_ROI_ISC_z
  Motor_ISC_Lateralization_z

Recommended model parameterization:
  Include Bilateral_Motor_ROI_

In [127]:
# ============================================================
#  NEURAL EFFECTS AFTER MOTOR CONTROL
#
# Analyses:
#   1. Subject × video motor-only mixed-effects model
#   2. Subject × video neural-only mixed-effects model
#   3. Combined motor + neural mixed-effects model
#   4. Residual neural-beta test
#   5. Variance partitioning and nested-model comparison
#   6. Subject-grouped cross-validated comparison:
#        motor-only versus combined model
#
# Required notebook objects:
#   motor_neural_pc_df
#   pc_alignment_df
#   motor_video_roi_isc_df
#
# Main behavioral outcome:
#   signed behavioral PC1 alignment, r_PC1
#
# Main motor controls:
#   MotorPC1
#   MotorPC2
#   MotorPC3
#   Motor_Control_ROI_ISC
#
# Main neural predictors tested separately:
#   Mean_ROI_ISC
#   Original_Network_Balance
#   Full_LASSO_Network_Balance
#
# Outputs saved to:
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data
# \Emotion Perception Data\Motor_Control\03_models
# ============================================================

import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

from scipy.stats import chi2, ttest_rel
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "motor_neural_pc_df",
    "pc_alignment_df",
    "motor_video_roi_isc_df"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "The following required notebook objects were not found:\n"
        f"{missing_objects}\n\n"
        "Run the behavioral PC-alignment analysis, motor-feature PCA, "
        "motor-neural merge, and motor-control ROI ISC pipeline first."
    )


# ============================================================
# 2. OUTPUT DIRECTORIES
# ============================================================

STAGE3_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control\03_models"
)

STAGE3_TABLE_DIR = STAGE3_OUTPUT_DIR / "tables"
STAGE3_MODEL_DIR = STAGE3_OUTPUT_DIR / "model_summaries"
STAGE3_FIG_DIR = STAGE3_OUTPUT_DIR / "figures"
STAGE3_CV_DIR = STAGE3_OUTPUT_DIR / "cross_validation"

for folder in [
    STAGE3_OUTPUT_DIR,
    STAGE3_TABLE_DIR,
    STAGE3_MODEL_DIR,
    STAGE3_FIG_DIR,
    STAGE3_CV_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Stage 3 outputs will be saved to:\n"
    f"{STAGE3_OUTPUT_DIR}"
)


# ============================================================
# 3. USER-ADJUSTABLE ANALYSIS SETTINGS
# ============================================================

# Signed PC1 alignment is the primary performance outcome.
PERFORMANCE_COLUMN = "r_PC1"

# All retained motor components are included because the motor PCA
# parallel analysis retained three components.
MOTOR_PC_COLUMNS = [
    "MotorPC1",
    "MotorPC2",
    "MotorPC3"
]

MOTOR_ROI_COLUMN = "Motor_Control_ROI_ISC"

# Each predictor is evaluated in a separate set of nested models.
NEURAL_PREDICTORS = [
    "Mean_ROI_ISC",
    "Original_Network_Balance",
    "Full_LASSO_Network_Balance"
]

# Models are run in the pooled data and separately by modality.
ANALYSIS_GROUPS = [
    "All",
    "Valence",
    "Arousal"
]

# Minimum data requirements.
MIN_COMPLETE_ROWS = 80
MIN_UNIQUE_SUBJECTS = 15
MIN_UNIQUE_VIDEOS = 5

# Cross-validation settings.
#
# Five-fold grouped CV is much faster than leave-one-subject-out.
# To use LOSO, set:
#     CV_N_SPLITS = number of unique subjects
CV_N_SPLITS = 5
CV_RANDOM_SEED = 2026

# Mixed-model settings.
MODEL_MAX_ITER = 3000

# Plot settings.
FIGURE_DPI = 300

# Whether to include the motor-control ROI in the main motor model.
# Keep True for the full Stage 3 motor-neural control.
INCLUDE_MOTOR_ROI = True


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """Normalize participant identifiers for safe merging."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_trigger(value):
    """Make trigger values such as 1, 1.0, and '1' identical."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(
                int(numeric_value)
            )

    except (TypeError, ValueError):
        pass

    return value


def normalize_modality(value):
    """Normalize modality labels."""
    if pd.isna(value):
        return pd.NA

    return str(value).strip().capitalize()


def safe_filename(value, max_length=160):
    """Create a Windows-safe filename."""
    value = str(value)

    value = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        value
    )

    value = re.sub(
        r"\s+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    value = value.strip("._ ")

    return value[:max_length] or "unnamed"


def zscore_series(series):
    """Z-score a numeric pandas Series."""
    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    mean_value = series.mean()
    sd_value = series.std(ddof=1)

    if (
        not np.isfinite(sd_value)
        or sd_value <= 0
    ):
        return pd.Series(
            np.nan,
            index=series.index
        )

    return (
        series - mean_value
    ) / sd_value


def apply_training_standardization(
    train_df,
    test_df,
    predictor_columns
):
    """
    Standardize predictors using training-set means and SDs only.

    This prevents information from the held-out subjects from leaking
    into cross-validation preprocessing.
    """
    train_output = train_df.copy()
    test_output = test_df.copy()

    scaling_rows = []

    for column in predictor_columns:
        training_values = pd.to_numeric(
            train_output[column],
            errors="coerce"
        )

        mean_value = training_values.mean()
        sd_value = training_values.std(ddof=1)

        if (
            not np.isfinite(sd_value)
            or sd_value <= 0
        ):
            return None, None, None

        train_output[column] = (
            pd.to_numeric(
                train_output[column],
                errors="coerce"
            )
            - mean_value
        ) / sd_value

        test_output[column] = (
            pd.to_numeric(
                test_output[column],
                errors="coerce"
            )
            - mean_value
        ) / sd_value

        scaling_rows.append({
            "Predictor": column,
            "TrainingMean": mean_value,
            "TrainingSD": sd_value
        })

    return (
        train_output,
        test_output,
        pd.DataFrame(scaling_rows)
    )


def fit_crossed_mixed_model(
    formula,
    dataframe,
    model_label=""
):
    """
    Fit a mixed-effects model using:

        random intercept for Subject
        crossed variance component for BaseVideo

    For the pooled All-modality analysis, SubjectVideo is also included
    as a variance component because valence and arousal rows share the
    same subject × video neural observation.
    """
    dataframe = dataframe.copy()

    use_subject_video_component = (
        dataframe["Modality"].nunique() > 1
    )

    variance_components = {
        "Video": "0 + C(BaseVideo)"
    }

    if use_subject_video_component:
        variance_components[
            "SubjectVideo"
        ] = "0 + C(SubjectVideo)"

    model = smf.mixedlm(
        formula=formula,
        data=dataframe,
        groups=dataframe["Subject"],
        re_formula="1",
        vc_formula=variance_components
    )

    fit_attempts = [
        ("lbfgs", 3000),
        ("bfgs", 3000),
        ("powell", 5000)
    ]

    last_error = None

    for method_name, max_iterations in fit_attempts:
        try:
            result = model.fit(
                reml=False,
                method=method_name,
                maxiter=max_iterations,
                disp=False
            )

            return result, method_name, None

        except Exception as error:
            last_error = error

    return (
        None,
        None,
        f"{model_label}: {last_error}"
    )


def mixed_model_r2(result):
    """
    Approximate Nakagawa-style marginal and conditional R².

    Marginal R²:
        variance explained by fixed effects / total variance

    Conditional R²:
        variance explained by fixed + random effects / total variance
    """
    if result is None:
        return {
            "FixedVariance": np.nan,
            "RandomVariance": np.nan,
            "ResidualVariance": np.nan,
            "MarginalR2": np.nan,
            "ConditionalR2": np.nan
        }

    try:
        fixed_prediction = np.dot(
            result.model.exog,
            np.asarray(
                result.fe_params
            )
        )

        fixed_variance = float(
            np.var(
                fixed_prediction,
                ddof=1
            )
        )

    except Exception:
        fixed_variance = np.nan

    random_variance = 0.0

    try:
        covariance_random_effects = np.asarray(
            result.cov_re,
            dtype=float
        )

        if covariance_random_effects.size > 0:
            random_variance += float(
                np.trace(
                    covariance_random_effects
                )
            )

    except Exception:
        pass

    try:
        variance_components = np.asarray(
            result.vcomp,
            dtype=float
        )

        if variance_components.size > 0:
            random_variance += float(
                np.nansum(
                    variance_components
                )
            )

    except Exception:
        pass

    residual_variance = float(
        result.scale
    )

    total_variance = (
        fixed_variance
        + random_variance
        + residual_variance
    )

    if (
        not np.isfinite(total_variance)
        or total_variance <= 0
    ):
        marginal_r2 = np.nan
        conditional_r2 = np.nan

    else:
        marginal_r2 = (
            fixed_variance
            / total_variance
        )

        conditional_r2 = (
            fixed_variance
            + random_variance
        ) / total_variance

    return {
        "FixedVariance": fixed_variance,
        "RandomVariance": random_variance,
        "ResidualVariance": residual_variance,
        "MarginalR2": marginal_r2,
        "ConditionalR2": conditional_r2
    }


def likelihood_ratio_test(
    reduced_result,
    full_result
):
    """
    Compare nested ML-fitted models using a likelihood-ratio test.
    """
    if (
        reduced_result is None
        or full_result is None
    ):
        return {
            "LR": np.nan,
            "DF": np.nan,
            "P": np.nan
        }

    lr_statistic = max(
        0.0,
        2.0 * (
            full_result.llf
            - reduced_result.llf
        )
    )

    df_difference = max(
        1,
        len(full_result.fe_params)
        - len(reduced_result.fe_params)
    )

    p_value = chi2.sf(
        lr_statistic,
        df_difference
    )

    return {
        "LR": lr_statistic,
        "DF": df_difference,
        "P": p_value
    }


def extract_coefficient_row(
    result,
    term,
    analysis_group,
    neural_predictor,
    model_type
):
    """Extract one fixed-effect coefficient from a fitted model."""
    if (
        result is None
        or term not in result.params.index
    ):
        return {
            "Analysis_Group": analysis_group,
            "Neural_Predictor": neural_predictor,
            "Model_Type": model_type,
            "Term": term,
            "Beta": np.nan,
            "SE": np.nan,
            "Z": np.nan,
            "P": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan
        }

    confidence_intervals = result.conf_int()

    return {
        "Analysis_Group": analysis_group,
        "Neural_Predictor": neural_predictor,
        "Model_Type": model_type,
        "Term": term,
        "Beta": float(
            result.params[term]
        ),
        "SE": float(
            result.bse[term]
        ),
        "Z": float(
            result.tvalues[term]
        ),
        "P": float(
            result.pvalues[term]
        ),
        "CI_Lower": float(
            confidence_intervals.loc[
                term,
                0
            ]
        ),
        "CI_Upper": float(
            confidence_intervals.loc[
                term,
                1
            ]
        )
    }


def fixed_effect_predictions(
    result,
    dataframe
):
    """
    Generate fixed-effect-only predictions.

    This is appropriate for held-out subjects because their random
    intercepts are unknown at prediction time.
    """
    if result is None:
        return np.full(
            len(dataframe),
            np.nan
        )

    try:
        return np.asarray(
            result.predict(
                dataframe
            ),
            dtype=float
        )

    except Exception:
        return np.full(
            len(dataframe),
            np.nan
        )


def calculate_prediction_metrics(
    observed,
    predicted
):
    """Return RMSE, MAE, and predictive R²."""
    observed = np.asarray(
        observed,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    valid = (
        np.isfinite(observed)
        & np.isfinite(predicted)
    )

    observed = observed[valid]
    predicted = predicted[valid]

    if len(observed) < 2:
        return {
            "N": len(observed),
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan
        }

    return {
        "N": len(observed),

        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    observed,
                    predicted
                )
            )
        ),

        "MAE": float(
            mean_absolute_error(
                observed,
                predicted
            )
        ),

        "R2": float(
            r2_score(
                observed,
                predicted
            )
        )
    }


def save_model_summary(
    result,
    output_path,
    heading
):
    """Save a full statsmodels model summary as a text file."""
    output_path = Path(
        output_path
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(
            heading
        )

        file.write(
            "\n\n"
        )

        if result is None:
            file.write(
                "Model could not be fitted."
            )

        else:
            file.write(
                result.summary().as_text()
            )


# ============================================================
# 5. PREPARE BEHAVIORAL PERFORMANCE TABLE
# ============================================================

performance_df = pc_alignment_df.copy()

required_performance_columns = {
    "Subject",
    "Modality",
    PERFORMANCE_COLUMN
}

missing_performance_columns = (
    required_performance_columns
    - set(performance_df.columns)
)

if missing_performance_columns:
    raise ValueError(
        "pc_alignment_df is missing required columns:\n"
        f"{sorted(missing_performance_columns)}"
    )

performance_df[
    "Subject"
] = performance_df[
    "Subject"
].apply(
    normalize_subject_id
)

performance_df[
    "Modality"
] = performance_df[
    "Modality"
].apply(
    normalize_modality
)

performance_df[
    PERFORMANCE_COLUMN
] = pd.to_numeric(
    performance_df[
        PERFORMANCE_COLUMN
    ],
    errors="coerce"
)


# ============================================================
# 6. PREPARE MOTOR + MAIN NEURAL TABLE
# ============================================================

stage3_base_df = motor_neural_pc_df.copy()

required_stage3_base_columns = {
    "Subject",
    "Trigger",
    "BaseVideo",
    "Modality",
    *MOTOR_PC_COLUMNS
}

missing_stage3_base_columns = (
    required_stage3_base_columns
    - set(stage3_base_df.columns)
)

if missing_stage3_base_columns:
    raise ValueError(
        "motor_neural_pc_df is missing required columns:\n"
        f"{sorted(missing_stage3_base_columns)}"
    )

stage3_base_df[
    "Subject"
] = stage3_base_df[
    "Subject"
].apply(
    normalize_subject_id
)

stage3_base_df[
    "Trigger"
] = stage3_base_df[
    "Trigger"
].apply(
    normalize_trigger
)

stage3_base_df[
    "Modality"
] = stage3_base_df[
    "Modality"
].apply(
    normalize_modality
)

stage3_base_df[
    "BaseVideo"
] = (
    stage3_base_df[
        "BaseVideo"
    ]
    .astype(str)
    .str.strip()
)


# ============================================================
# 7. MERGE BEHAVIORAL PERFORMANCE
# ============================================================

# Prefer full Video names because they include modality and uniquely
# identify the behavioral trajectory.
if (
    "Video" in stage3_base_df.columns
    and "Video" in performance_df.columns
):
    performance_merge_keys = [
        "Subject",
        "Video",
        "Modality"
    ]

else:
    performance_merge_keys = [
        "Subject",
        "BaseVideo",
        "Modality"
    ]

    if "BaseVideo" not in performance_df.columns:
        raise ValueError(
            "A behavioral-performance merge could not be constructed. "
            "Both tables need either Video or BaseVideo."
        )

performance_columns_to_merge = (
    performance_merge_keys
    + [
        PERFORMANCE_COLUMN
    ]
)

performance_merge_df = (
    performance_df[
        performance_columns_to_merge
    ]
    .drop_duplicates(
        subset=performance_merge_keys
    )
)

if PERFORMANCE_COLUMN in stage3_base_df.columns:
    stage3_base_df = stage3_base_df.drop(
        columns=[
            PERFORMANCE_COLUMN
        ]
    )

stage3_base_df = stage3_base_df.merge(
    performance_merge_df,
    on=performance_merge_keys,
    how="left",
    validate="one_to_one",
    indicator="PerformanceMerge"
)


# ============================================================
# 8. PREPARE AND MERGE MOTOR-CONTROL ROI ISC
# ============================================================

motor_roi_long_df = motor_video_roi_isc_df.copy()

required_motor_roi_columns = {
    "Subject",
    "Trigger",
    "ROI",
    "ISC"
}

missing_motor_roi_columns = (
    required_motor_roi_columns
    - set(motor_roi_long_df.columns)
)

if missing_motor_roi_columns:
    raise ValueError(
        "motor_video_roi_isc_df is missing required columns:\n"
        f"{sorted(missing_motor_roi_columns)}"
    )

motor_roi_long_df[
    "Subject"
] = motor_roi_long_df[
    "Subject"
].apply(
    normalize_subject_id
)

motor_roi_long_df[
    "Trigger"
] = motor_roi_long_df[
    "Trigger"
].apply(
    normalize_trigger
)

motor_roi_long_df[
    "ROI"
] = (
    motor_roi_long_df[
        "ROI"
    ]
    .astype(str)
    .str.strip()
)

motor_roi_long_df[
    "ISC"
] = pd.to_numeric(
    motor_roi_long_df[
        "ISC"
    ],
    errors="coerce"
)

motor_roi_covariate_df = (
    motor_roi_long_df[
        motor_roi_long_df[
            "ROI"
        ] == "Motor Control ROI"
    ]
    .groupby(
        [
            "Subject",
            "Trigger"
        ],
        as_index=False
    )[
        "ISC"
    ]
    .mean()
    .rename(
        columns={
            "ISC": MOTOR_ROI_COLUMN
        }
    )
)

stage3_base_df = stage3_base_df.merge(
    motor_roi_covariate_df,
    on=[
        "Subject",
        "Trigger"
    ],
    how="left",
    validate="many_to_one",
    indicator="MotorROIMerge"
)


# ============================================================
# 9. DATASET DIAGNOSTICS
# ============================================================

stage3_base_df[
    "SubjectVideo"
] = (
    stage3_base_df[
        "Subject"
    ].astype(str)
    + "__"
    + stage3_base_df[
        "BaseVideo"
    ].astype(str)
)

numeric_columns = (
    [
        PERFORMANCE_COLUMN,
        *MOTOR_PC_COLUMNS,
        MOTOR_ROI_COLUMN
    ]
    + [
        predictor
        for predictor in NEURAL_PREDICTORS
        if predictor in stage3_base_df.columns
    ]
)

for column in numeric_columns:
    stage3_base_df[column] = pd.to_numeric(
        stage3_base_df[column],
        errors="coerce"
    )


available_neural_predictors = [
    predictor
    for predictor in NEURAL_PREDICTORS
    if predictor in stage3_base_df.columns
]

missing_neural_predictors = [
    predictor
    for predictor in NEURAL_PREDICTORS
    if predictor not in stage3_base_df.columns
]

if missing_neural_predictors:
    print(
        "\nThe following requested neural predictors were not found "
        "and will be skipped:"
    )

    for predictor in missing_neural_predictors:
        print(
            f"  {predictor}"
        )

if not available_neural_predictors:
    raise ValueError(
        "None of the requested neural predictors were available."
    )


diagnostic_rows = []

for analysis_group in ANALYSIS_GROUPS:

    if analysis_group == "All":
        group_df = stage3_base_df.copy()

    else:
        group_df = stage3_base_df[
            stage3_base_df[
                "Modality"
            ] == analysis_group
        ].copy()

    for neural_predictor in available_neural_predictors:

        required_columns = [
            "Subject",
            "BaseVideo",
            "Modality",
            "SubjectVideo",
            PERFORMANCE_COLUMN,
            *MOTOR_PC_COLUMNS,
            neural_predictor
        ]

        if INCLUDE_MOTOR_ROI:
            required_columns.append(
                MOTOR_ROI_COLUMN
            )

        complete_df = group_df[
            required_columns
        ].replace(
            [np.inf, -np.inf],
            np.nan
        ).dropna()

        diagnostic_rows.append({
            "Analysis_Group": analysis_group,
            "Neural_Predictor": neural_predictor,
            "N_Complete_Rows": len(
                complete_df
            ),
            "N_Subjects": complete_df[
                "Subject"
            ].nunique(),
            "N_Videos": complete_df[
                "BaseVideo"
            ].nunique(),
            "N_SubjectVideo": complete_df[
                "SubjectVideo"
            ].nunique(),
            "MotorROI_Coverage_Rows": group_df[
                MOTOR_ROI_COLUMN
            ].notna().sum(),
            "Performance_Matched_Rows": group_df[
                PERFORMANCE_COLUMN
            ].notna().sum()
        })


stage3_data_diagnostics_df = pd.DataFrame(
    diagnostic_rows
)

stage3_data_diagnostics_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_data_diagnostics.csv",
    index=False
)

stage3_base_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_merged_subject_video_data.csv",
    index=False
)

print(
    "\nStage 3 complete-case diagnostics:"
)

display(
    stage3_data_diagnostics_df
)


# ============================================================
# 10. FIT MOTOR-ONLY, NEURAL-ONLY, AND COMBINED MODELS
# ============================================================

model_fit_rows = []
coefficient_rows = []
comparison_rows = []
variance_partition_rows = []
residual_beta_rows = []
model_failure_rows = []

fitted_model_objects = {}


for analysis_group in ANALYSIS_GROUPS:

    if analysis_group == "All":
        group_df = stage3_base_df.copy()

    else:
        group_df = stage3_base_df[
            stage3_base_df[
                "Modality"
            ] == analysis_group
        ].copy()

    for neural_predictor in available_neural_predictors:

        print(
            "\n" + "=" * 90
        )

        print(
            f"Fitting Stage 3 models: "
            f"{analysis_group} | {neural_predictor}"
        )

        required_columns = [
            "Subject",
            "BaseVideo",
            "Modality",
            "SubjectVideo",
            PERFORMANCE_COLUMN,
            *MOTOR_PC_COLUMNS,
            neural_predictor
        ]

        motor_predictor_columns = (
            MOTOR_PC_COLUMNS.copy()
        )

        if INCLUDE_MOTOR_ROI:
            required_columns.append(
                MOTOR_ROI_COLUMN
            )

            motor_predictor_columns.append(
                MOTOR_ROI_COLUMN
            )

        analysis_df = (
            group_df[
                required_columns
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
            .copy()
        )

        if (
            len(analysis_df) < MIN_COMPLETE_ROWS
            or analysis_df[
                "Subject"
            ].nunique() < MIN_UNIQUE_SUBJECTS
            or analysis_df[
                "BaseVideo"
            ].nunique() < MIN_UNIQUE_VIDEOS
        ):
            model_failure_rows.append({
                "Analysis_Group": analysis_group,
                "Neural_Predictor": neural_predictor,
                "Stage": "Complete-case requirement",
                "Error": (
                    f"Insufficient complete data: "
                    f"N={len(analysis_df)}, "
                    f"subjects="
                    f"{analysis_df['Subject'].nunique()}, "
                    f"videos="
                    f"{analysis_df['BaseVideo'].nunique()}"
                )
            })

            print(
                "Skipped because the complete-case sample "
                "was too small."
            )

            continue

        # ----------------------------------------------------
        # Standardize predictors within this exact common sample.
        #
        # The outcome remains in correlation units, so coefficients
        # represent the outcome change per 1-SD predictor increase.
        # ----------------------------------------------------

        standardized_predictors = (
            motor_predictor_columns
            + [
                neural_predictor
            ]
        )

        standardized_name_map = {}

        for predictor in standardized_predictors:
            standardized_column = (
                safe_filename(
                    predictor
                )
                .replace(
                    "-",
                    "_"
                )
                + "_z"
            )

            standardized_name_map[
                predictor
            ] = standardized_column

            analysis_df[
                standardized_column
            ] = zscore_series(
                analysis_df[
                    predictor
                ]
            )

        analysis_df = analysis_df.dropna(
            subset=list(
                standardized_name_map.values()
            )
        )

        outcome_model_name = "Performance"

        analysis_df[
            outcome_model_name
        ] = analysis_df[
            PERFORMANCE_COLUMN
        ]

        neural_z = standardized_name_map[
            neural_predictor
        ]

        motor_z_columns = [
            standardized_name_map[
                predictor
            ]
            for predictor
            in motor_predictor_columns
        ]

        modality_term = (
            " + C(Modality)"
            if analysis_group == "All"
            else ""
        )

        motor_formula = (
            f"{outcome_model_name} ~ "
            + " + ".join(
                motor_z_columns
            )
            + modality_term
        )

        neural_formula = (
            f"{outcome_model_name} ~ "
            f"{neural_z}"
            + modality_term
        )

        combined_formula = (
            f"{outcome_model_name} ~ "
            + " + ".join(
                motor_z_columns
                + [
                    neural_z
                ]
            )
            + modality_term
        )

        # ----------------------------------------------------
        # Motor-only MLM
        # ----------------------------------------------------

        (
            motor_result,
            motor_method,
            motor_error
        ) = fit_crossed_mixed_model(
            formula=motor_formula,
            dataframe=analysis_df,
            model_label="Motor-only"
        )

        # ----------------------------------------------------
        # Neural-only MLM
        # ----------------------------------------------------

        (
            neural_result,
            neural_method,
            neural_error
        ) = fit_crossed_mixed_model(
            formula=neural_formula,
            dataframe=analysis_df,
            model_label="Neural-only"
        )

        # ----------------------------------------------------
        # Combined motor + neural MLM
        # ----------------------------------------------------

        (
            combined_result,
            combined_method,
            combined_error
        ) = fit_crossed_mixed_model(
            formula=combined_formula,
            dataframe=analysis_df,
            model_label="Combined"
        )

        for model_type, error_message in [
            ("Motor-only", motor_error),
            ("Neural-only", neural_error),
            ("Combined", combined_error)
        ]:
            if error_message is not None:
                model_failure_rows.append({
                    "Analysis_Group": analysis_group,
                    "Neural_Predictor": neural_predictor,
                    "Stage": model_type,
                    "Error": error_message
                })

        model_key = (
            analysis_group,
            neural_predictor
        )

        fitted_model_objects[
            model_key
        ] = {
            "Data": analysis_df,
            "Motor": motor_result,
            "Neural": neural_result,
            "Combined": combined_result,
            "NeuralTerm": neural_z,
            "MotorTerms": motor_z_columns
        }

        # ----------------------------------------------------
        # Save full model summaries
        # ----------------------------------------------------

        output_prefix = (
            f"{safe_filename(analysis_group)}__"
            f"{safe_filename(neural_predictor)}"
        )

        save_model_summary(
            motor_result,
            STAGE3_MODEL_DIR
            / f"{output_prefix}__motor_only.txt",
            (
                f"Analysis group: {analysis_group}\n"
                f"Neural predictor defining common sample: "
                f"{neural_predictor}\n"
                f"Model: {motor_formula}"
            )
        )

        save_model_summary(
            neural_result,
            STAGE3_MODEL_DIR
            / f"{output_prefix}__neural_only.txt",
            (
                f"Analysis group: {analysis_group}\n"
                f"Neural predictor: {neural_predictor}\n"
                f"Model: {neural_formula}"
            )
        )

        save_model_summary(
            combined_result,
            STAGE3_MODEL_DIR
            / f"{output_prefix}__combined.txt",
            (
                f"Analysis group: {analysis_group}\n"
                f"Neural predictor: {neural_predictor}\n"
                f"Model: {combined_formula}"
            )
        )

        # ----------------------------------------------------
        # Record fit statistics and R²
        # ----------------------------------------------------

        model_results = {
            "Motor_Only": (
                motor_result,
                motor_method,
                motor_formula
            ),
            "Neural_Only": (
                neural_result,
                neural_method,
                neural_formula
            ),
            "Combined": (
                combined_result,
                combined_method,
                combined_formula
            )
        }

        model_r2_values = {}

        for model_type, (
            model_result,
            model_method,
            model_formula
        ) in model_results.items():

            r2_values = mixed_model_r2(
                model_result
            )

            model_r2_values[
                model_type
            ] = r2_values

            model_fit_rows.append({
                "Analysis_Group": analysis_group,
                "Neural_Predictor": neural_predictor,
                "Model_Type": model_type,
                "Formula": model_formula,
                "N_Rows": len(
                    analysis_df
                ),
                "N_Subjects": analysis_df[
                    "Subject"
                ].nunique(),
                "N_Videos": analysis_df[
                    "BaseVideo"
                ].nunique(),
                "Optimizer": model_method,
                "Converged": (
                    bool(
                        model_result.converged
                    )
                    if model_result is not None
                    else False
                ),
                "LogLikelihood": (
                    float(
                        model_result.llf
                    )
                    if model_result is not None
                    else np.nan
                ),
                "AIC": (
                    float(
                        model_result.aic
                    )
                    if model_result is not None
                    else np.nan
                ),
                "BIC": (
                    float(
                        model_result.bic
                    )
                    if model_result is not None
                    else np.nan
                ),
                **r2_values
            })

        # ----------------------------------------------------
        # Extract neural betas
        # ----------------------------------------------------

        coefficient_rows.append(
            extract_coefficient_row(
                result=neural_result,
                term=neural_z,
                analysis_group=analysis_group,
                neural_predictor=neural_predictor,
                model_type="Neural_Only"
            )
        )

        coefficient_rows.append(
            extract_coefficient_row(
                result=combined_result,
                term=neural_z,
                analysis_group=analysis_group,
                neural_predictor=neural_predictor,
                model_type="Combined"
            )
        )

        # ----------------------------------------------------
        # Nested model comparisons
        # ----------------------------------------------------

        motor_vs_combined = likelihood_ratio_test(
            reduced_result=motor_result,
            full_result=combined_result
        )

        neural_vs_combined = likelihood_ratio_test(
            reduced_result=neural_result,
            full_result=combined_result
        )

        comparison_rows.append({
            "Analysis_Group": analysis_group,
            "Neural_Predictor": neural_predictor,
            "Comparison": "Motor-only vs Combined",
            "Reduced_Model": "Motor_Only",
            "Full_Model": "Combined",
            "LR": motor_vs_combined["LR"],
            "DF": motor_vs_combined["DF"],
            "P": motor_vs_combined["P"],
            "Delta_AIC_FullMinusReduced": (
                combined_result.aic
                - motor_result.aic
                if (
                    combined_result is not None
                    and motor_result is not None
                )
                else np.nan
            ),
            "Delta_BIC_FullMinusReduced": (
                combined_result.bic
                - motor_result.bic
                if (
                    combined_result is not None
                    and motor_result is not None
                )
                else np.nan
            )
        })

        comparison_rows.append({
            "Analysis_Group": analysis_group,
            "Neural_Predictor": neural_predictor,
            "Comparison": "Neural-only vs Combined",
            "Reduced_Model": "Neural_Only",
            "Full_Model": "Combined",
            "LR": neural_vs_combined["LR"],
            "DF": neural_vs_combined["DF"],
            "P": neural_vs_combined["P"],
            "Delta_AIC_FullMinusReduced": (
                combined_result.aic
                - neural_result.aic
                if (
                    combined_result is not None
                    and neural_result is not None
                )
                else np.nan
            ),
            "Delta_BIC_FullMinusReduced": (
                combined_result.bic
                - neural_result.bic
                if (
                    combined_result is not None
                    and neural_result is not None
                )
                else np.nan
            )
        })

        # ----------------------------------------------------
        # Variance partitioning
        #
        # Unique neural:
        #     combined marginal R² - motor-only marginal R²
        #
        # Unique motor:
        #     combined marginal R² - neural-only marginal R²
        #
        # Shared:
        #     motor R² + neural R² - combined R²
        #
        # Negative shared values can occur under suppression.
        # ----------------------------------------------------

        motor_marginal_r2 = model_r2_values[
            "Motor_Only"
        ]["MarginalR2"]

        neural_marginal_r2 = model_r2_values[
            "Neural_Only"
        ]["MarginalR2"]

        combined_marginal_r2 = model_r2_values[
            "Combined"
        ]["MarginalR2"]

        unique_neural_variance = (
            combined_marginal_r2
            - motor_marginal_r2
        )

        unique_motor_variance = (
            combined_marginal_r2
            - neural_marginal_r2
        )

        shared_variance = (
            motor_marginal_r2
            + neural_marginal_r2
            - combined_marginal_r2
        )

        unexplained_variance = (
            1.0
            - combined_marginal_r2
        )

        variance_partition_rows.append({
            "Analysis_Group": analysis_group,
            "Neural_Predictor": neural_predictor,
            "Motor_Only_Marginal_R2": motor_marginal_r2,
            "Neural_Only_Marginal_R2": neural_marginal_r2,
            "Combined_Marginal_R2": combined_marginal_r2,
            "Unique_Neural_R2": unique_neural_variance,
            "Unique_Motor_R2": unique_motor_variance,
            "Shared_R2": shared_variance,
            "Unexplained_R2": unexplained_variance
        })

        # ----------------------------------------------------
        # Residual neural-beta test
        #
        # 1. Obtain residuals from motor-only model.
        # 2. Test whether neural synchrony predicts the remaining
        #    performance variance.
        #
        # This is a complementary sensitivity analysis.
        # The combined model remains the primary inferential model.
        # ----------------------------------------------------

        if motor_result is not None:

            residual_analysis_df = (
                analysis_df.copy()
            )

            try:
                motor_residuals = np.asarray(
                    motor_result.resid,
                    dtype=float
                )

            except Exception:
                motor_residuals = (
                    residual_analysis_df[
                        outcome_model_name
                    ].to_numpy(dtype=float)
                    -
                    np.asarray(
                        motor_result.fittedvalues,
                        dtype=float
                    )
                )

            residual_analysis_df[
                "MotorResidualPerformance"
            ] = motor_residuals

            residual_formula = (
                "MotorResidualPerformance ~ "
                f"{neural_z}"
                + modality_term
            )

            (
                residual_result,
                residual_method,
                residual_error
            ) = fit_crossed_mixed_model(
                formula=residual_formula,
                dataframe=residual_analysis_df,
                model_label="Residual neural-beta"
            )

            if residual_error is not None:
                model_failure_rows.append({
                    "Analysis_Group": analysis_group,
                    "Neural_Predictor": neural_predictor,
                    "Stage": "Residual neural-beta",
                    "Error": residual_error
                })

            save_model_summary(
                residual_result,
                STAGE3_MODEL_DIR
                / f"{output_prefix}__residual_neural_beta.txt",
                (
                    f"Analysis group: {analysis_group}\n"
                    f"Neural predictor: {neural_predictor}\n"
                    f"Model: {residual_formula}"
                )
            )

            residual_row = extract_coefficient_row(
                result=residual_result,
                term=neural_z,
                analysis_group=analysis_group,
                neural_predictor=neural_predictor,
                model_type="Residual_Beta"
            )

            residual_row[
                "Optimizer"
            ] = residual_method

            residual_beta_rows.append(
                residual_row
            )

            # Residual scatterplot.
            if (
                residual_result is not None
                and neural_z
                in residual_analysis_df.columns
            ):
                plot_df = residual_analysis_df[
                    [
                        neural_z,
                        "MotorResidualPerformance"
                    ]
                ].dropna()

                fig, ax = plt.subplots(
                    figsize=(7, 6)
                )

                ax.scatter(
                    plot_df[
                        neural_z
                    ],
                    plot_df[
                        "MotorResidualPerformance"
                    ],
                    alpha=0.30,
                    s=25
                )

                if (
                    plot_df[
                        neural_z
                    ].std(ddof=1) > 0
                ):
                    slope, intercept = np.polyfit(
                        plot_df[
                            neural_z
                        ],
                        plot_df[
                            "MotorResidualPerformance"
                        ],
                        1
                    )

                    x_line = np.linspace(
                        plot_df[
                            neural_z
                        ].min(),
                        plot_df[
                            neural_z
                        ].max(),
                        200
                    )

                    ax.plot(
                        x_line,
                        intercept
                        + slope * x_line,
                        linewidth=2
                    )

                ax.axhline(
                    0,
                    linestyle="--",
                    linewidth=1
                )

                ax.set_xlabel(
                    f"Standardized {neural_predictor}"
                )

                ax.set_ylabel(
                    "Performance residual after motor controls"
                )

                ax.set_title(
                    f"{analysis_group}: residual neural-beta test\n"
                    f"{neural_predictor}"
                )

                fig.tight_layout()

                fig.savefig(
                    STAGE3_FIG_DIR
                    / (
                        f"{output_prefix}"
                        "__residual_neural_beta.png"
                    ),
                    dpi=FIGURE_DPI,
                    bbox_inches="tight"
                )

                plt.show()
                plt.close(fig)


# ============================================================
# 11. CREATE MODEL RESULT TABLES
# ============================================================

stage3_model_fit_df = pd.DataFrame(
    model_fit_rows
)

stage3_coefficients_df = pd.DataFrame(
    coefficient_rows
)

stage3_nested_comparisons_df = pd.DataFrame(
    comparison_rows
)

stage3_variance_partition_df = pd.DataFrame(
    variance_partition_rows
)

stage3_residual_beta_df = pd.DataFrame(
    residual_beta_rows
)

stage3_model_failures_df = pd.DataFrame(
    model_failure_rows
)


# ------------------------------------------------------------
# FDR adjustment for neural coefficients
# ------------------------------------------------------------

if not stage3_coefficients_df.empty:

    stage3_coefficients_df[
        "P_FDR"
    ] = np.nan

    for model_type in stage3_coefficients_df[
        "Model_Type"
    ].dropna().unique():

        for analysis_group in stage3_coefficients_df[
            "Analysis_Group"
        ].dropna().unique():

            selection = (
                (
                    stage3_coefficients_df[
                        "Model_Type"
                    ] == model_type
                )
                &
                (
                    stage3_coefficients_df[
                        "Analysis_Group"
                    ] == analysis_group
                )
                &
                stage3_coefficients_df[
                    "P"
                ].notna()
            )

            if selection.sum() > 0:
                stage3_coefficients_df.loc[
                    selection,
                    "P_FDR"
                ] = multipletests(
                    stage3_coefficients_df.loc[
                        selection,
                        "P"
                    ],
                    method="fdr_bh"
                )[1]


if not stage3_nested_comparisons_df.empty:

    stage3_nested_comparisons_df[
        "P_FDR"
    ] = np.nan

    for comparison_name in stage3_nested_comparisons_df[
        "Comparison"
    ].dropna().unique():

        for analysis_group in stage3_nested_comparisons_df[
            "Analysis_Group"
        ].dropna().unique():

            selection = (
                (
                    stage3_nested_comparisons_df[
                        "Comparison"
                    ] == comparison_name
                )
                &
                (
                    stage3_nested_comparisons_df[
                        "Analysis_Group"
                    ] == analysis_group
                )
                &
                stage3_nested_comparisons_df[
                    "P"
                ].notna()
            )

            if selection.sum() > 0:
                stage3_nested_comparisons_df.loc[
                    selection,
                    "P_FDR"
                ] = multipletests(
                    stage3_nested_comparisons_df.loc[
                        selection,
                        "P"
                    ],
                    method="fdr_bh"
                )[1]


if not stage3_residual_beta_df.empty:

    stage3_residual_beta_df[
        "P_FDR"
    ] = np.nan

    for analysis_group in stage3_residual_beta_df[
        "Analysis_Group"
    ].dropna().unique():

        selection = (
            (
                stage3_residual_beta_df[
                    "Analysis_Group"
                ] == analysis_group
            )
            &
            stage3_residual_beta_df[
                "P"
            ].notna()
        )

        if selection.sum() > 0:
            stage3_residual_beta_df.loc[
                selection,
                "P_FDR"
            ] = multipletests(
                stage3_residual_beta_df.loc[
                    selection,
                    "P"
                ],
                method="fdr_bh"
            )[1]


# ============================================================
# 12. SUBJECT-GROUPED CROSS-VALIDATION
# ============================================================

cv_fold_rows = []
cv_prediction_rows = []
cv_failure_rows = []


for analysis_group in ANALYSIS_GROUPS:

    if analysis_group == "All":
        group_df = stage3_base_df.copy()

    else:
        group_df = stage3_base_df[
            stage3_base_df[
                "Modality"
            ] == analysis_group
        ].copy()

    for neural_predictor in available_neural_predictors:

        print(
            "\n" + "-" * 90
        )

        print(
            f"Cross-validation: "
            f"{analysis_group} | {neural_predictor}"
        )

        motor_predictor_columns = (
            MOTOR_PC_COLUMNS.copy()
        )

        required_columns = [
            "Subject",
            "BaseVideo",
            "Modality",
            "SubjectVideo",
            PERFORMANCE_COLUMN,
            *motor_predictor_columns,
            neural_predictor
        ]

        if INCLUDE_MOTOR_ROI:
            motor_predictor_columns.append(
                MOTOR_ROI_COLUMN
            )

            required_columns.append(
                MOTOR_ROI_COLUMN
            )

        cv_df = (
            group_df[
                required_columns
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
            .copy()
            .reset_index(drop=True)
        )

        unique_subject_count = cv_df[
            "Subject"
        ].nunique()

        if (
            len(cv_df) < MIN_COMPLETE_ROWS
            or unique_subject_count
            < max(
                3,
                CV_N_SPLITS
            )
        ):
            cv_failure_rows.append({
                "Analysis_Group": analysis_group,
                "Neural_Predictor": neural_predictor,
                "Fold": np.nan,
                "Error": (
                    "Insufficient complete rows or subjects for CV."
                )
            })

            continue

        n_splits = min(
            CV_N_SPLITS,
            unique_subject_count
        )

        group_kfold = GroupKFold(
            n_splits=n_splits
        )

        fold_number = 0

        predictor_columns_for_scaling = (
            motor_predictor_columns
            + [
                neural_predictor
            ]
        )

        for (
            train_indices,
            test_indices
        ) in group_kfold.split(
            cv_df,
            groups=cv_df[
                "Subject"
            ]
        ):

            fold_number += 1

            train_df = cv_df.iloc[
                train_indices
            ].copy()

            test_df = cv_df.iloc[
                test_indices
            ].copy()

            (
                train_scaled,
                test_scaled,
                fold_scaling_df
            ) = apply_training_standardization(
                train_df=train_df,
                test_df=test_df,
                predictor_columns=(
                    predictor_columns_for_scaling
                )
            )

            if train_scaled is None:
                cv_failure_rows.append({
                    "Analysis_Group": analysis_group,
                    "Neural_Predictor": neural_predictor,
                    "Fold": fold_number,
                    "Error": (
                        "A predictor had zero variance in the training fold."
                    )
                })

                continue

            train_scaled[
                "Performance"
            ] = train_scaled[
                PERFORMANCE_COLUMN
            ]

            test_scaled[
                "Performance"
            ] = test_scaled[
                PERFORMANCE_COLUMN
            ]

            neural_cv_name = "Neural_z"

            train_scaled[
                neural_cv_name
            ] = train_scaled[
                neural_predictor
            ]

            test_scaled[
                neural_cv_name
            ] = test_scaled[
                neural_predictor
            ]

            motor_cv_names = []

            for motor_index, motor_column in enumerate(
                motor_predictor_columns,
                start=1
            ):
                motor_cv_name = (
                    f"MotorCovariate{motor_index}_z"
                )

                motor_cv_names.append(
                    motor_cv_name
                )

                train_scaled[
                    motor_cv_name
                ] = train_scaled[
                    motor_column
                ]

                test_scaled[
                    motor_cv_name
                ] = test_scaled[
                    motor_column
                ]

            modality_term = (
                " + C(Modality)"
                if analysis_group == "All"
                else ""
            )

            cv_motor_formula = (
                "Performance ~ "
                + " + ".join(
                    motor_cv_names
                )
                + modality_term
            )

            cv_combined_formula = (
                "Performance ~ "
                + " + ".join(
                    motor_cv_names
                    + [
                        neural_cv_name
                    ]
                )
                + modality_term
            )

            (
                cv_motor_result,
                _,
                cv_motor_error
            ) = fit_crossed_mixed_model(
                formula=cv_motor_formula,
                dataframe=train_scaled,
                model_label=(
                    f"CV motor-only fold {fold_number}"
                )
            )

            (
                cv_combined_result,
                _,
                cv_combined_error
            ) = fit_crossed_mixed_model(
                formula=cv_combined_formula,
                dataframe=train_scaled,
                model_label=(
                    f"CV combined fold {fold_number}"
                )
            )

            if (
                cv_motor_result is None
                or cv_combined_result is None
            ):
                cv_failure_rows.append({
                    "Analysis_Group": analysis_group,
                    "Neural_Predictor": neural_predictor,
                    "Fold": fold_number,
                    "Error": (
                        f"Motor error: {cv_motor_error}; "
                        f"Combined error: {cv_combined_error}"
                    )
                })

                continue

            observed_values = test_scaled[
                "Performance"
            ].to_numpy(dtype=float)

            motor_predictions = fixed_effect_predictions(
                cv_motor_result,
                test_scaled
            )

            combined_predictions = fixed_effect_predictions(
                cv_combined_result,
                test_scaled
            )

            motor_metrics = calculate_prediction_metrics(
                observed=observed_values,
                predicted=motor_predictions
            )

            combined_metrics = calculate_prediction_metrics(
                observed=observed_values,
                predicted=combined_predictions
            )

            cv_fold_rows.append({
                "Analysis_Group": analysis_group,
                "Neural_Predictor": neural_predictor,
                "Fold": fold_number,
                "N_Train": len(train_scaled),
                "N_Test": len(test_scaled),
                "N_Train_Subjects": train_scaled[
                    "Subject"
                ].nunique(),
                "N_Test_Subjects": test_scaled[
                    "Subject"
                ].nunique(),

                "Motor_RMSE": motor_metrics[
                    "RMSE"
                ],
                "Combined_RMSE": combined_metrics[
                    "RMSE"
                ],
                "Delta_RMSE_CombinedMinusMotor": (
                    combined_metrics["RMSE"]
                    - motor_metrics["RMSE"]
                ),

                "Motor_MAE": motor_metrics[
                    "MAE"
                ],
                "Combined_MAE": combined_metrics[
                    "MAE"
                ],
                "Delta_MAE_CombinedMinusMotor": (
                    combined_metrics["MAE"]
                    - motor_metrics["MAE"]
                ),

                "Motor_R2": motor_metrics[
                    "R2"
                ],
                "Combined_R2": combined_metrics[
                    "R2"
                ],
                "Delta_R2_CombinedMinusMotor": (
                    combined_metrics["R2"]
                    - motor_metrics["R2"]
                )
            })

            for row_index, original_index in enumerate(
                test_indices
            ):
                cv_prediction_rows.append({
                    "Analysis_Group": analysis_group,
                    "Neural_Predictor": neural_predictor,
                    "Fold": fold_number,
                    "Subject": test_scaled.iloc[
                        row_index
                    ]["Subject"],
                    "BaseVideo": test_scaled.iloc[
                        row_index
                    ]["BaseVideo"],
                    "Modality": test_scaled.iloc[
                        row_index
                    ]["Modality"],
                    "Observed_Performance": (
                        observed_values[
                            row_index
                        ]
                    ),
                    "Motor_Only_Prediction": (
                        motor_predictions[
                            row_index
                        ]
                    ),
                    "Combined_Prediction": (
                        combined_predictions[
                            row_index
                        ]
                    )
                })


stage3_cv_folds_df = pd.DataFrame(
    cv_fold_rows
)

stage3_cv_predictions_df = pd.DataFrame(
    cv_prediction_rows
)

stage3_cv_failures_df = pd.DataFrame(
    cv_failure_rows
)


# ============================================================
# 13. CROSS-VALIDATION SUMMARY AND PAIRED TESTS
# ============================================================

cv_summary_rows = []

if not stage3_cv_folds_df.empty:

    for (
        analysis_group,
        neural_predictor
    ), group_df in stage3_cv_folds_df.groupby(
        [
            "Analysis_Group",
            "Neural_Predictor"
        ]
    ):

        rmse_test = ttest_rel(
            group_df[
                "Combined_RMSE"
            ],
            group_df[
                "Motor_RMSE"
            ],
            nan_policy="omit"
        )

        mae_test = ttest_rel(
            group_df[
                "Combined_MAE"
            ],
            group_df[
                "Motor_MAE"
            ],
            nan_policy="omit"
        )

        r2_test = ttest_rel(
            group_df[
                "Combined_R2"
            ],
            group_df[
                "Motor_R2"
            ],
            nan_policy="omit"
        )

        cv_summary_rows.append({
            "Analysis_Group": analysis_group,
            "Neural_Predictor": neural_predictor,
            "N_Folds": len(
                group_df
            ),

            "Mean_Motor_RMSE": group_df[
                "Motor_RMSE"
            ].mean(),
            "Mean_Combined_RMSE": group_df[
                "Combined_RMSE"
            ].mean(),
            "Mean_Delta_RMSE": group_df[
                "Delta_RMSE_CombinedMinusMotor"
            ].mean(),
            "RMSE_Paired_T": rmse_test.statistic,
            "RMSE_Paired_P": rmse_test.pvalue,

            "Mean_Motor_MAE": group_df[
                "Motor_MAE"
            ].mean(),
            "Mean_Combined_MAE": group_df[
                "Combined_MAE"
            ].mean(),
            "Mean_Delta_MAE": group_df[
                "Delta_MAE_CombinedMinusMotor"
            ].mean(),
            "MAE_Paired_T": mae_test.statistic,
            "MAE_Paired_P": mae_test.pvalue,

            "Mean_Motor_R2": group_df[
                "Motor_R2"
            ].mean(),
            "Mean_Combined_R2": group_df[
                "Combined_R2"
            ].mean(),
            "Mean_Delta_R2": group_df[
                "Delta_R2_CombinedMinusMotor"
            ].mean(),
            "R2_Paired_T": r2_test.statistic,
            "R2_Paired_P": r2_test.pvalue,

            "Proportion_Folds_Combined_Lower_RMSE": (
                group_df[
                    "Combined_RMSE"
                ]
                <
                group_df[
                    "Motor_RMSE"
                ]
            ).mean(),

            "Proportion_Folds_Combined_Higher_R2": (
                group_df[
                    "Combined_R2"
                ]
                >
                group_df[
                    "Motor_R2"
                ]
            ).mean()
        })


stage3_cv_summary_df = pd.DataFrame(
    cv_summary_rows
)


# ============================================================
# 14. SAVE ALL TABLES
# ============================================================

stage3_model_fit_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_model_fit_statistics.csv",
    index=False
)

stage3_coefficients_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_neural_coefficients.csv",
    index=False
)

stage3_nested_comparisons_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_nested_model_comparisons.csv",
    index=False
)

stage3_variance_partition_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_variance_partitioning.csv",
    index=False
)

stage3_residual_beta_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_residual_neural_beta_tests.csv",
    index=False
)

stage3_model_failures_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_model_failures.csv",
    index=False
)

stage3_cv_folds_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_cross_validation_fold_results.csv",
    index=False
)

stage3_cv_predictions_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_cross_validation_predictions.csv",
    index=False
)

stage3_cv_summary_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_cross_validation_summary.csv",
    index=False
)

stage3_cv_failures_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_cross_validation_failures.csv",
    index=False
)


# ============================================================
# 15. FIGURE — NEURAL BETAS BEFORE AND AFTER MOTOR CONTROL
# ============================================================

if not stage3_coefficients_df.empty:

    for analysis_group in stage3_coefficients_df[
        "Analysis_Group"
    ].dropna().unique():

        plot_df = stage3_coefficients_df[
            stage3_coefficients_df[
                "Analysis_Group"
            ] == analysis_group
        ].copy()

        if plot_df.empty:
            continue

        neural_predictor_order = [
            predictor
            for predictor in available_neural_predictors
            if predictor
            in plot_df[
                "Neural_Predictor"
            ].unique()
        ]

        x_positions = np.arange(
            len(
                neural_predictor_order
            )
        )

        fig, ax = plt.subplots(
            figsize=(
                max(
                    9,
                    2.2
                    * len(
                        neural_predictor_order
                    )
                ),
                6
            )
        )

        offsets = {
            "Neural_Only": -0.15,
            "Combined": 0.15
        }

        for model_type in [
            "Neural_Only",
            "Combined"
        ]:

            model_plot_df = (
                plot_df[
                    plot_df[
                        "Model_Type"
                    ] == model_type
                ]
                .set_index(
                    "Neural_Predictor"
                )
                .reindex(
                    neural_predictor_order
                )
            )

            beta_values = model_plot_df[
                "Beta"
            ].to_numpy(dtype=float)

            lower_errors = (
                beta_values
                - model_plot_df[
                    "CI_Lower"
                ].to_numpy(dtype=float)
            )

            upper_errors = (
                model_plot_df[
                    "CI_Upper"
                ].to_numpy(dtype=float)
                - beta_values
            )

            ax.errorbar(
                x_positions
                + offsets[
                    model_type
                ],
                beta_values,
                yerr=np.vstack(
                    [
                        lower_errors,
                        upper_errors
                    ]
                ),
                fmt="o",
                capsize=4,
                label=(
                    "Neural-only"
                    if model_type
                    == "Neural_Only"
                    else "After motor control"
                )
            )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1
        )

        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(
            neural_predictor_order,
            rotation=25,
            ha="right"
        )

        ax.set_ylabel(
            "Neural beta predicting signed PC1 alignment"
        )

        ax.set_title(
            f"{analysis_group}: neural effects before and "
            "after motor control"
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        fig.savefig(
            STAGE3_FIG_DIR
            / (
                f"{safe_filename(analysis_group)}"
                "__neural_betas_before_after_motor_control.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 16. FIGURE — VARIANCE PARTITIONING
# ============================================================

if not stage3_variance_partition_df.empty:

    for analysis_group in stage3_variance_partition_df[
        "Analysis_Group"
    ].dropna().unique():

        plot_df = stage3_variance_partition_df[
            stage3_variance_partition_df[
                "Analysis_Group"
            ] == analysis_group
        ].copy()

        if plot_df.empty:
            continue

        plot_df = plot_df.set_index(
            "Neural_Predictor"
        ).reindex(
            [
                predictor
                for predictor
                in available_neural_predictors
                if predictor
                in plot_df[
                    "Neural_Predictor"
                ].values
            ]
        ).reset_index()

        x_positions = np.arange(
            len(
                plot_df
            )
        )

        fig, ax = plt.subplots(
            figsize=(
                max(
                    9,
                    2.3
                    * len(
                        plot_df
                    )
                ),
                6
            )
        )

        bottom_values = np.zeros(
            len(
                plot_df
            )
        )

        components_to_plot = [
            (
                "Unique_Motor_R2",
                "Unique motor"
            ),
            (
                "Shared_R2",
                "Shared"
            ),
            (
                "Unique_Neural_R2",
                "Unique neural"
            )
        ]

        for column, label in components_to_plot:

            values = plot_df[
                column
            ].to_numpy(dtype=float)

            ax.bar(
                x_positions,
                values,
                bottom=bottom_values,
                label=label
            )

            bottom_values = (
                bottom_values
                + np.nan_to_num(
                    values
                )
            )

        ax.axhline(
            0,
            linewidth=1
        )

        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(
            plot_df[
                "Neural_Predictor"
            ],
            rotation=25,
            ha="right"
        )

        ax.set_ylabel(
            "Marginal R² partition"
        )

        ax.set_title(
            f"{analysis_group}: motor and neural "
            "variance partitioning"
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        fig.savefig(
            STAGE3_FIG_DIR
            / (
                f"{safe_filename(analysis_group)}"
                "__variance_partitioning.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 17. FIGURE — CROSS-VALIDATED MODEL IMPROVEMENT
# ============================================================

if not stage3_cv_summary_df.empty:

    for analysis_group in stage3_cv_summary_df[
        "Analysis_Group"
    ].dropna().unique():

        plot_df = stage3_cv_summary_df[
            stage3_cv_summary_df[
                "Analysis_Group"
            ] == analysis_group
        ].copy()

        if plot_df.empty:
            continue

        x_positions = np.arange(
            len(
                plot_df
            )
        )

        fig, ax = plt.subplots(
            figsize=(
                max(
                    9,
                    2.3
                    * len(
                        plot_df
                    )
                ),
                6
            )
        )

        width = 0.35

        ax.bar(
            x_positions
            - width / 2,
            plot_df[
                "Mean_Motor_RMSE"
            ],
            width=width,
            label="Motor-only"
        )

        ax.bar(
            x_positions
            + width / 2,
            plot_df[
                "Mean_Combined_RMSE"
            ],
            width=width,
            label="Motor + neural"
        )

        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(
            plot_df[
                "Neural_Predictor"
            ],
            rotation=25,
            ha="right"
        )

        ax.set_ylabel(
            "Cross-validated RMSE"
        )

        ax.set_title(
            f"{analysis_group}: subject-grouped "
            "cross-validated prediction"
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        fig.savefig(
            STAGE3_FIG_DIR
            / (
                f"{safe_filename(analysis_group)}"
                "__cross_validated_rmse.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 18. PRINT KEY RESULTS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "STAGE 3 MOTOR-CONTROL MODELS COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nModel fit statistics:"
)

display(
    stage3_model_fit_df.round(
        5
    )
)


print(
    "\nNeural coefficients before and after motor control:"
)

if stage3_coefficients_df.empty:
    print(
        "No neural coefficient results were available."
    )

else:
    display(
        stage3_coefficients_df[
            [
                "Analysis_Group",
                "Neural_Predictor",
                "Model_Type",
                "Beta",
                "SE",
                "Z",
                "P",
                "P_FDR",
                "CI_Lower",
                "CI_Upper"
            ]
        ]
        .sort_values(
            [
                "Analysis_Group",
                "Neural_Predictor",
                "Model_Type"
            ]
        )
        .round(5)
    )


print(
    "\nNested-model comparisons:"
)

if stage3_nested_comparisons_df.empty:
    print(
        "No nested-model comparisons were available."
    )

else:
    display(
        stage3_nested_comparisons_df[
            [
                "Analysis_Group",
                "Neural_Predictor",
                "Comparison",
                "LR",
                "DF",
                "P",
                "P_FDR",
                "Delta_AIC_FullMinusReduced",
                "Delta_BIC_FullMinusReduced"
            ]
        ]
        .sort_values(
            [
                "Analysis_Group",
                "Comparison",
                "P"
            ]
        )
        .round(5)
    )


print(
    "\nVariance partitioning:"
)

if stage3_variance_partition_df.empty:
    print(
        "No variance-partitioning results were available."
    )

else:
    display(
        stage3_variance_partition_df.round(
            5
        )
    )


print(
    "\nResidual neural-beta tests:"
)

if stage3_residual_beta_df.empty:
    print(
        "No residual-beta results were available."
    )

else:
    display(
        stage3_residual_beta_df[
            [
                "Analysis_Group",
                "Neural_Predictor",
                "Beta",
                "SE",
                "Z",
                "P",
                "P_FDR",
                "CI_Lower",
                "CI_Upper"
            ]
        ]
        .sort_values(
            [
                "Analysis_Group",
                "P"
            ]
        )
        .round(5)
    )


print(
    "\nSubject-grouped cross-validation summary:"
)

if stage3_cv_summary_df.empty:
    print(
        "No cross-validation summary was available."
    )

else:
    display(
        stage3_cv_summary_df.round(
            5
        )
    )


if not stage3_model_failures_df.empty:

    print(
        "\nModel fitting warnings or failures:"
    )

    display(
        stage3_model_failures_df
    )


if not stage3_cv_failures_df.empty:

    print(
        "\nCross-validation fitting warnings or failures:"
    )

    display(
        stage3_cv_failures_df
    )


print(
    "\nPrimary interpretation guide:"
)

print(
    "  • Combined-model neural beta: whether the neural predictor "
    "remains associated with performance after motor controls."
)

print(
    "  • Motor-only vs Combined likelihood-ratio test: whether adding "
    "the neural predictor significantly improves model fit."
)

print(
    "  • Unique_Neural_R2: incremental fixed-effect variance explained "
    "by the neural predictor beyond motor variables."
)

print(
    "  • Residual beta: whether the neural predictor explains "
    "performance remaining after the motor-only model."
)

print(
    "  • Cross-validated ΔRMSE < 0 and ΔR² > 0: whether the combined "
    "model predicts held-out participants better than the motor-only "
    "model."
)

print(
    "\nAll Stage 3 outputs were saved to:\n"
    f"{STAGE3_OUTPUT_DIR}"
)

ValueError: The following required notebook objects were not found:
['pc_alignment_df']

Run the behavioral PC-alignment analysis, motor-feature PCA, motor-neural merge, and motor-control ROI ISC pipeline first.

In [ ]:
# ============================================================
#  MOTOR-CONTROL MODEL COMPARISONS
#
# PRIMARY ANALYSIS:
#   Full available sample
#   Motor controls:
#       MotorPC1
#       MotorPC2
#       MotorPC3
#
# MOTOR-ROI SENSITIVITY ANALYSIS:
#   Reduced sample with valid left motor ROI ISC
#   Motor controls:
#       MotorPC1
#       MotorPC2
#       MotorPC3
#       Left_Motor_ROI_ISC_z
#
# MODELS FIT ON THE SAME COMPLETE-CASE SAMPLE:
#
#   Model 0: Baseline
#       Performance ~ 1
#
#   Model 1: Motor only
#       Performance ~ motor covariates
#
#   Model 2: Neural only
#       Performance ~ network balance
#
#   Model 3: Combined
#       Performance ~ motor covariates + network balance
#
# RANDOM EFFECTS:
#   Random intercept for Subject
#   Random intercept for BaseVideo
#
# ANALYSES RUN SEPARATELY FOR:
#   Valence
#   Arousal
#
# NEURAL PREDICTORS TESTED SEPARATELY:
#   Original_Network_Balance
#   Full_LASSO_Network_Balance
#
# OUTPUTS:
#   Model summaries
#   Neural-beta attenuation
#   Likelihood-ratio tests
#   AIC/BIC comparisons
#   Marginal and conditional R²
#   Incremental neural R²
#   Subject-grouped held-out prediction
#
# Saves outputs to:
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data
# \Emotion Perception Data\Motor_Control\03_models
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

from scipy.stats import chi2, ttest_rel
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)


# ============================================================
# 2. REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "motor_neural_pc_df",
    "pc_alignment_df"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "The following required notebook objects were not found:\n"
        f"{missing_objects}\n\n"
        "Run the behavioral PC-alignment, motor-feature PCA, and "
        "motor-neural merge analyses first."
    )

# The left motor-control variable can come from either:
#
# 1. motor_control_isc_wide_df
#
# or:
#
# 2. motor_video_roi_isc_df, from which it will be reconstructed.

if (
    "motor_control_isc_wide_df" not in globals()
    and "motor_video_roi_isc_df" not in globals()
):
    raise ValueError(
        "Neither motor_control_isc_wide_df nor "
        "motor_video_roi_isc_df was found.\n\n"
        "Run the bilateral motor-control ROI ISC pipeline first."
    )


# ============================================================
# 3. OUTPUT DIRECTORIES
# ============================================================

STAGE3_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control\03_models"
)

STAGE3_TABLE_DIR = (
    STAGE3_OUTPUT_DIR / "tables"
)

STAGE3_MODEL_DIR = (
    STAGE3_OUTPUT_DIR / "model_summaries"
)

STAGE3_FIG_DIR = (
    STAGE3_OUTPUT_DIR / "figures"
)

STAGE3_CV_DIR = (
    STAGE3_OUTPUT_DIR / "cross_validation"
)

STAGE3_DIAGNOSTIC_DIR = (
    STAGE3_OUTPUT_DIR / "diagnostics"
)

for folder in [
    STAGE3_OUTPUT_DIR,
    STAGE3_TABLE_DIR,
    STAGE3_MODEL_DIR,
    STAGE3_FIG_DIR,
    STAGE3_CV_DIR,
    STAGE3_DIAGNOSTIC_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Stage 3 outputs will be saved to:\n"
    f"{STAGE3_OUTPUT_DIR}"
)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

# Behavioral performance:
#
# Signed correlation between each participant's behavioral
# trajectory and the video-specific PC1 trajectory.
PERFORMANCE_COLUMN = "r_PC1"

MOTOR_PC_COLUMNS = [
    "MotorPC1",
    "MotorPC2",
    "MotorPC3"
]

LEFT_MOTOR_ROI_COLUMN = (
    "Left_Motor_ROI_ISC_z"
)

# Test only the two network-balance indices requested.
NEURAL_PREDICTORS = [
    "Original_Network_Balance",
    "Full_LASSO_Network_Balance"
]

# Run separate modality models only.
ANALYSIS_GROUPS = [
    "Valence",
    "Arousal"
]

# Two model suites:
#
# Primary:
#   full sample using motor PCs
#
# Sensitivity:
#   reduced sample adding left motor ROI ISC
MODEL_SUITES = {
    "Primary_MotorPCs": {
        "MotorCovariates": MOTOR_PC_COLUMNS,
        "Description": (
            "Primary full-sample model using MotorPC1–3"
        )
    },

    "LeftMotorROI_Sensitivity": {
        "MotorCovariates": (
            MOTOR_PC_COLUMNS
            + [
                LEFT_MOTOR_ROI_COLUMN
            ]
        ),
        "Description": (
            "Reduced-sample sensitivity model adding "
            "left motor-control ROI ISC"
        )
    }
}

# Minimum complete-case sample requirements.
MIN_COMPLETE_ROWS = 80
MIN_UNIQUE_SUBJECTS = 15
MIN_UNIQUE_VIDEOS = 5

# Subject-grouped cross-validation.
#
# All observations from a participant remain in one fold.
# Therefore, no participant appears in both training and test data.
CV_N_SPLITS = 5

# Increase to the number of subjects for leave-one-subject-out CV,
# although that will require many more mixed-model fits.
#
# Example:
# CV_N_SPLITS = 42

MODEL_MAX_ITER = 5000

FIGURE_DPI = 300

NEAR_ZERO_VARIANCE_THRESHOLD = 1e-8

VIF_WARNING_THRESHOLD = 5.0

RANDOM_VARIANCE_WARNING_THRESHOLD = 1e-6


# ============================================================
# 5. GENERAL HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """
    Normalize participant identifiers for merging.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_trigger(value):
    """
    Normalize triggers so values such as 1, 1.0, and '1' match.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(
                int(numeric_value)
            )

    except (
        TypeError,
        ValueError
    ):
        pass

    return value


def normalize_modality(value):
    """
    Normalize modality labels.
    """
    if pd.isna(value):
        return pd.NA

    return str(
        value
    ).strip().capitalize()


def safe_filename(value, max_length=160):
    """
    Return a Windows-safe filename.
    """
    value = str(value)

    value = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        value
    )

    value = re.sub(
        r"\s+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    value = value.strip(
        "._ "
    )

    return (
        value[:max_length]
        or "unnamed"
    )


def zscore_series(series):
    """
    Z-score a pandas Series.
    """
    numeric_series = pd.to_numeric(
        series,
        errors="coerce"
    )

    mean_value = numeric_series.mean()
    sd_value = numeric_series.std(
        ddof=1
    )

    if (
        not np.isfinite(sd_value)
        or sd_value
        <= NEAR_ZERO_VARIANCE_THRESHOLD
    ):
        return pd.Series(
            np.nan,
            index=series.index
        )

    return (
        numeric_series
        - mean_value
    ) / sd_value


def fisher_z_correlation(correlation):
    """
    Fisher-z transform a correlation after clipping.
    """
    correlation = pd.to_numeric(
        correlation,
        errors="coerce"
    )

    return np.arctanh(
        np.clip(
            correlation,
            -0.999999,
            0.999999
        )
    )


def save_model_summary(
    result,
    output_path,
    heading
):
    """
    Save a statsmodels result summary to a text file.
    """
    output_path = Path(
        output_path
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as output_file:

        output_file.write(
            heading
        )

        output_file.write(
            "\n\n"
        )

        if result is None:

            output_file.write(
                "Model could not be fitted."
            )

        else:

            output_file.write(
                result.summary().as_text()
            )


# ============================================================
# 6. MIXED-MODEL HELPERS
# ============================================================

def fit_subject_video_mixed_model(
    formula,
    dataframe,
    model_label=""
):
    """
    Fit a mixed model with:

        random intercept for Subject
        random intercept for BaseVideo

    The model is fitted using maximum likelihood rather than REML
    because the fixed-effect structures are compared using
    likelihood-ratio tests, AIC, and BIC.

    The random-effects structure is held constant across Models 0–3.
    """

    model = smf.mixedlm(
        formula=formula,
        data=dataframe,
        groups=dataframe["Subject"],
        re_formula="1",
        vc_formula={
            "Video": "0 + C(BaseVideo)"
        }
    )

    fit_attempts = [
        (
            "lbfgs",
            MODEL_MAX_ITER
        ),
        (
            "bfgs",
            MODEL_MAX_ITER
        ),
        (
            "powell",
            MODEL_MAX_ITER
        )
    ]

    fit_errors = []

    for (
        optimizer,
        maximum_iterations
    ) in fit_attempts:

        try:

            result = model.fit(
                reml=False,
                method=optimizer,
                maxiter=maximum_iterations,
                disp=False
            )

            return (
                result,
                optimizer,
                None
            )

        except Exception as error:

            fit_errors.append(
                f"{optimizer}: {error}"
            )

    return (
        None,
        None,
        (
            f"{model_label} failed with all optimizers. "
            + " | ".join(
                fit_errors
            )
        )
    )


def extract_random_variances(result):
    """
    Extract subject, video, and residual variance estimates.
    """
    output = {
        "SubjectVariance": np.nan,
        "VideoVariance": np.nan,
        "ResidualVariance": np.nan,
        "NearZeroRandomVariance": False
    }

    if result is None:
        return output

    try:

        covariance_random = np.asarray(
            result.cov_re,
            dtype=float
        )

        if covariance_random.size > 0:

            output[
                "SubjectVariance"
            ] = float(
                covariance_random.ravel()[0]
            )

    except Exception:
        pass

    try:

        variance_components = np.asarray(
            result.vcomp,
            dtype=float
        )

        if variance_components.size > 0:

            output[
                "VideoVariance"
            ] = float(
                variance_components.ravel()[0]
            )

    except Exception:
        pass

    try:

        output[
            "ResidualVariance"
        ] = float(
            result.scale
        )

    except Exception:
        pass

    random_variance_values = [
        output["SubjectVariance"],
        output["VideoVariance"]
    ]

    output[
        "NearZeroRandomVariance"
    ] = any(
        np.isfinite(value)
        and value
        < RANDOM_VARIANCE_WARNING_THRESHOLD
        for value in random_variance_values
    )

    return output


def mixed_model_r2(result):
    """
    Approximate Nakagawa-style marginal and conditional R².

    Marginal R²:
        fixed-effect variance / total variance

    Conditional R²:
        fixed + random-effect variance / total variance
    """
    if result is None:

        return {
            "FixedVariance": np.nan,
            "RandomVariance": np.nan,
            "ResidualVariance": np.nan,
            "MarginalR2": np.nan,
            "ConditionalR2": np.nan
        }

    try:

        fixed_predictions = np.dot(
            result.model.exog,
            np.asarray(
                result.fe_params
            )
        )

        fixed_variance = float(
            np.var(
                fixed_predictions,
                ddof=1
            )
        )

    except Exception:

        fixed_variance = np.nan

    random_variance = 0.0

    try:

        covariance_random = np.asarray(
            result.cov_re,
            dtype=float
        )

        if covariance_random.size > 0:

            random_variance += float(
                np.trace(
                    covariance_random
                )
            )

    except Exception:
        pass

    try:

        variance_components = np.asarray(
            result.vcomp,
            dtype=float
        )

        if variance_components.size > 0:

            random_variance += float(
                np.nansum(
                    variance_components
                )
            )

    except Exception:
        pass

    try:

        residual_variance = float(
            result.scale
        )

    except Exception:

        residual_variance = np.nan

    total_variance = (
        fixed_variance
        + random_variance
        + residual_variance
    )

    if (
        not np.isfinite(total_variance)
        or total_variance <= 0
    ):

        marginal_r2 = np.nan
        conditional_r2 = np.nan

    else:

        marginal_r2 = (
            fixed_variance
            / total_variance
        )

        conditional_r2 = (
            fixed_variance
            + random_variance
        ) / total_variance

    return {
        "FixedVariance": fixed_variance,
        "RandomVariance": random_variance,
        "ResidualVariance": residual_variance,
        "MarginalR2": marginal_r2,
        "ConditionalR2": conditional_r2
    }


def likelihood_ratio_test(
    reduced_result,
    full_result
):
    """
    Compare two nested maximum-likelihood mixed models.
    """
    if (
        reduced_result is None
        or full_result is None
    ):

        return {
            "LR": np.nan,
            "DF": np.nan,
            "P": np.nan
        }

    lr_statistic = max(
        0.0,
        2.0
        * (
            full_result.llf
            - reduced_result.llf
        )
    )

    parameter_difference = (
        len(
            full_result.fe_params
        )
        - len(
            reduced_result.fe_params
        )
    )

    parameter_difference = max(
        1,
        parameter_difference
    )

    p_value = chi2.sf(
        lr_statistic,
        parameter_difference
    )

    return {
        "LR": float(
            lr_statistic
        ),
        "DF": int(
            parameter_difference
        ),
        "P": float(
            p_value
        )
    }


def extract_fixed_effect(
    result,
    term
):
    """
    Extract a fixed-effect estimate and confidence interval.
    """
    empty_result = {
        "Beta": np.nan,
        "SE": np.nan,
        "Z": np.nan,
        "P": np.nan,
        "CI_Lower": np.nan,
        "CI_Upper": np.nan
    }

    if (
        result is None
        or term
        not in result.fe_params.index
    ):
        return empty_result

    confidence_intervals = (
        result.conf_int()
    )

    return {
        "Beta": float(
            result.fe_params[
                term
            ]
        ),

        "SE": float(
            result.bse_fe[
                term
            ]
        ),

        "Z": float(
            result.tvalues[
                term
            ]
        ),

        "P": float(
            result.pvalues[
                term
            ]
        ),

        "CI_Lower": float(
            confidence_intervals.loc[
                term,
                0
            ]
        ),

        "CI_Upper": float(
            confidence_intervals.loc[
                term,
                1
            ]
        )
    }


# ============================================================
# 7. PREDICTION HELPERS
# ============================================================

def standardize_train_test(
    train_df,
    test_df,
    predictor_columns
):
    """
    Standardize predictors using training-set parameters only.

    This prevents leakage of held-out subject information into
    cross-validation preprocessing.
    """
    train_output = train_df.copy()
    test_output = test_df.copy()

    scaling_rows = []

    for predictor in predictor_columns:

        training_values = pd.to_numeric(
            train_output[
                predictor
            ],
            errors="coerce"
        )

        mean_value = training_values.mean()

        sd_value = training_values.std(
            ddof=1
        )

        if (
            not np.isfinite(sd_value)
            or sd_value
            <= NEAR_ZERO_VARIANCE_THRESHOLD
        ):

            return (
                None,
                None,
                None
            )

        train_output[
            predictor
        ] = (
            pd.to_numeric(
                train_output[
                    predictor
                ],
                errors="coerce"
            )
            - mean_value
        ) / sd_value

        test_output[
            predictor
        ] = (
            pd.to_numeric(
                test_output[
                    predictor
                ],
                errors="coerce"
            )
            - mean_value
        ) / sd_value

        scaling_rows.append({
            "Predictor": predictor,
            "TrainingMean": mean_value,
            "TrainingSD": sd_value
        })

    return (
        train_output,
        test_output,
        pd.DataFrame(
            scaling_rows
        )
    )


def fixed_effect_predictions(
    result,
    dataframe
):
    """
    Generate fixed-effect-only predictions.

    This is appropriate for held-out participants because their
    subject random intercepts are not available in training data.
    """
    if result is None:

        return np.full(
            len(dataframe),
            np.nan
        )

    try:

        predictions = result.predict(
            dataframe
        )

        return np.asarray(
            predictions,
            dtype=float
        )

    except Exception:

        return np.full(
            len(dataframe),
            np.nan
        )


def prediction_metrics(
    observed,
    predicted
):
    """
    Calculate RMSE, MAE, and predictive R².
    """
    observed = np.asarray(
        observed,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    valid = (
        np.isfinite(observed)
        & np.isfinite(predicted)
    )

    observed = observed[
        valid
    ]

    predicted = predicted[
        valid
    ]

    if len(observed) < 2:

        return {
            "N": len(observed),
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan
        }

    return {
        "N": len(observed),

        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    observed,
                    predicted
                )
            )
        ),

        "MAE": float(
            mean_absolute_error(
                observed,
                predicted
            )
        ),

        "R2": float(
            r2_score(
                observed,
                predicted
            )
        )
    }


# ============================================================
# 8. PREPARE PERFORMANCE TABLE
# ============================================================

performance_df = (
    pc_alignment_df.copy()
)

required_performance_columns = {
    "Subject",
    "Modality",
    PERFORMANCE_COLUMN
}

missing_performance_columns = (
    required_performance_columns
    - set(
        performance_df.columns
    )
)

if missing_performance_columns:

    raise ValueError(
        "pc_alignment_df is missing required columns:\n"
        f"{sorted(missing_performance_columns)}"
    )


performance_df[
    "Subject"
] = performance_df[
    "Subject"
].apply(
    normalize_subject_id
)

performance_df[
    "Modality"
] = performance_df[
    "Modality"
].apply(
    normalize_modality
)

performance_df[
    PERFORMANCE_COLUMN
] = pd.to_numeric(
    performance_df[
        PERFORMANCE_COLUMN
    ],
    errors="coerce"
)


# ============================================================
# 9. PREPARE MOTOR-PC AND NETWORK-BALANCE TABLE
# ============================================================

stage3_base_df = (
    motor_neural_pc_df.copy()
)

required_base_columns = {
    "Subject",
    "Trigger",
    "BaseVideo",
    "Modality",
    *MOTOR_PC_COLUMNS
}

missing_base_columns = (
    required_base_columns
    - set(
        stage3_base_df.columns
    )
)

if missing_base_columns:

    raise ValueError(
        "motor_neural_pc_df is missing required columns:\n"
        f"{sorted(missing_base_columns)}"
    )


stage3_base_df[
    "Subject"
] = stage3_base_df[
    "Subject"
].apply(
    normalize_subject_id
)

stage3_base_df[
    "Trigger"
] = stage3_base_df[
    "Trigger"
].apply(
    normalize_trigger
)

stage3_base_df[
    "Modality"
] = stage3_base_df[
    "Modality"
].apply(
    normalize_modality
)

stage3_base_df[
    "BaseVideo"
] = (
    stage3_base_df[
        "BaseVideo"
    ]
    .astype(str)
    .str.strip()
)


# ============================================================
# 10. MERGE PERFORMANCE
# ============================================================

if (
    "Video"
    in stage3_base_df.columns
    and "Video"
    in performance_df.columns
):

    performance_merge_keys = [
        "Subject",
        "Video",
        "Modality"
    ]

else:

    performance_merge_keys = [
        "Subject",
        "BaseVideo",
        "Modality"
    ]

    if (
        "BaseVideo"
        not in performance_df.columns
    ):

        raise ValueError(
            "Could not construct the performance merge. "
            "Both dataframes need either Video or BaseVideo."
        )


performance_merge_df = (
    performance_df[
        performance_merge_keys
        + [
            PERFORMANCE_COLUMN
        ]
    ]
    .drop_duplicates(
        subset=performance_merge_keys
    )
)


if (
    PERFORMANCE_COLUMN
    in stage3_base_df.columns
):

    stage3_base_df = (
        stage3_base_df.drop(
            columns=[
                PERFORMANCE_COLUMN
            ]
        )
    )


stage3_base_df = stage3_base_df.merge(
    performance_merge_df,
    on=performance_merge_keys,
    how="left",
    validate="one_to_one",
    indicator="PerformanceMerge"
)


# ============================================================
# 11. PREPARE LEFT MOTOR ROI ISC
# ============================================================

if (
    "motor_control_isc_wide_df"
    in globals()
):

    left_motor_df = (
        motor_control_isc_wide_df.copy()
    )

    required_left_columns = {
        "Subject",
        "Trigger"
    }

    missing_left_columns = (
        required_left_columns
        - set(
            left_motor_df.columns
        )
    )

    if missing_left_columns:

        raise ValueError(
            "motor_control_isc_wide_df is missing:\n"
            f"{sorted(missing_left_columns)}"
        )

    left_motor_df[
        "Subject"
    ] = left_motor_df[
        "Subject"
    ].apply(
        normalize_subject_id
    )

    left_motor_df[
        "Trigger"
    ] = left_motor_df[
        "Trigger"
    ].apply(
        normalize_trigger
    )

    if (
        LEFT_MOTOR_ROI_COLUMN
        not in left_motor_df.columns
    ):

        if (
            "Left_Motor_ROI_ISC"
            in left_motor_df.columns
        ):

            left_motor_df[
                LEFT_MOTOR_ROI_COLUMN
            ] = fisher_z_correlation(
                left_motor_df[
                    "Left_Motor_ROI_ISC"
                ]
            )

        else:

            raise ValueError(
                "motor_control_isc_wide_df does not contain "
                "Left_Motor_ROI_ISC_z or Left_Motor_ROI_ISC."
            )

    left_motor_df = (
        left_motor_df[
            [
                "Subject",
                "Trigger",
                LEFT_MOTOR_ROI_COLUMN
            ]
        ]
        .drop_duplicates(
            subset=[
                "Subject",
                "Trigger"
            ]
        )
    )


else:

    motor_roi_long_df = (
        motor_video_roi_isc_df.copy()
    )

    required_motor_roi_columns = {
        "Subject",
        "Trigger",
        "ROI",
        "ISC"
    }

    missing_motor_roi_columns = (
        required_motor_roi_columns
        - set(
            motor_roi_long_df.columns
        )
    )

    if missing_motor_roi_columns:

        raise ValueError(
            "motor_video_roi_isc_df is missing:\n"
            f"{sorted(missing_motor_roi_columns)}"
        )

    motor_roi_long_df[
        "Subject"
    ] = motor_roi_long_df[
        "Subject"
    ].apply(
        normalize_subject_id
    )

    motor_roi_long_df[
        "Trigger"
    ] = motor_roi_long_df[
        "Trigger"
    ].apply(
        normalize_trigger
    )

    motor_roi_long_df[
        "ROI"
    ] = (
        motor_roi_long_df[
            "ROI"
        ]
        .astype(str)
        .str.strip()
    )

    motor_roi_long_df[
        "ISC"
    ] = pd.to_numeric(
        motor_roi_long_df[
            "ISC"
        ],
        errors="coerce"
    )

    left_motor_df = (
        motor_roi_long_df[
            motor_roi_long_df[
                "ROI"
            ] == "Left Motor Control ROI"
        ]
        .groupby(
            [
                "Subject",
                "Trigger"
            ],
            as_index=False
        )[
            "ISC"
        ]
        .mean()
        .rename(
            columns={
                "ISC":
                    "Left_Motor_ROI_ISC"
            }
        )
    )

    left_motor_df[
        LEFT_MOTOR_ROI_COLUMN
    ] = fisher_z_correlation(
        left_motor_df[
            "Left_Motor_ROI_ISC"
        ]
    )

    left_motor_df = left_motor_df[
        [
            "Subject",
            "Trigger",
            LEFT_MOTOR_ROI_COLUMN
        ]
    ]


# ============================================================
# 12. MERGE LEFT MOTOR ROI
# ============================================================

stage3_base_df = stage3_base_df.merge(
    left_motor_df,
    on=[
        "Subject",
        "Trigger"
    ],
    how="left",
    validate="many_to_one",
    indicator="LeftMotorMerge"
)


# ============================================================
# 13. NUMERIC CONVERSION AND AVAILABLE PREDICTORS
# ============================================================

numeric_columns = [
    PERFORMANCE_COLUMN,
    *MOTOR_PC_COLUMNS,
    LEFT_MOTOR_ROI_COLUMN
]

numeric_columns += [
    predictor
    for predictor in NEURAL_PREDICTORS
    if predictor
    in stage3_base_df.columns
]

for column in numeric_columns:

    stage3_base_df[
        column
    ] = pd.to_numeric(
        stage3_base_df[
            column
        ],
        errors="coerce"
    )


available_neural_predictors = [
    predictor
    for predictor in NEURAL_PREDICTORS
    if predictor
    in stage3_base_df.columns
]

missing_neural_predictors = [
    predictor
    for predictor in NEURAL_PREDICTORS
    if predictor
    not in stage3_base_df.columns
]

if missing_neural_predictors:

    print(
        "\nThe following network-balance variables were not found "
        "and will be skipped:"
    )

    for predictor in missing_neural_predictors:

        print(
            f"  {predictor}"
        )


if not available_neural_predictors:

    raise ValueError(
        "Neither requested network-balance variable was available."
    )


# ============================================================
# 14. SAVE MERGED ANALYSIS TABLE
# ============================================================

stage3_base_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_motor_network_merged_data.csv",
    index=False
)


# ============================================================
# 15. DATA AND COLLINEARITY DIAGNOSTICS
# ============================================================

diagnostic_rows = []
vif_rows = []
correlation_rows = []


for modality in ANALYSIS_GROUPS:

    modality_df = (
        stage3_base_df[
            stage3_base_df[
                "Modality"
            ] == modality
        ]
        .copy()
    )

    for (
        suite_name,
        suite_settings
    ) in MODEL_SUITES.items():

        motor_covariates = (
            suite_settings[
                "MotorCovariates"
            ]
        )

        for neural_predictor in (
            available_neural_predictors
        ):

            required_columns = [
                "Subject",
                "BaseVideo",
                PERFORMANCE_COLUMN,
                *motor_covariates,
                neural_predictor
            ]

            complete_df = (
                modality_df[
                    required_columns
                ]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .dropna()
                .copy()
            )

            diagnostic_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Neural_Predictor": neural_predictor,
                "N_Rows": len(
                    complete_df
                ),
                "N_Subjects": complete_df[
                    "Subject"
                ].nunique(),
                "N_Videos": complete_df[
                    "BaseVideo"
                ].nunique(),
                "PerformanceMean": complete_df[
                    PERFORMANCE_COLUMN
                ].mean(),
                "PerformanceSD": complete_df[
                    PERFORMANCE_COLUMN
                ].std(
                    ddof=1
                ),
                "LeftMotorCoverage": (
                    complete_df[
                        LEFT_MOTOR_ROI_COLUMN
                    ].notna().mean()
                    if LEFT_MOTOR_ROI_COLUMN
                    in complete_df.columns
                    else np.nan
                )
            })

            predictor_columns = (
                motor_covariates
                + [
                    neural_predictor
                ]
            )

            if (
                len(complete_df) > 0
                and len(
                    predictor_columns
                ) > 1
            ):

                correlation_matrix = (
                    complete_df[
                        predictor_columns
                    ]
                    .corr()
                )

                for first_index, first_predictor in enumerate(
                    predictor_columns
                ):

                    for second_predictor in (
                        predictor_columns[
                            first_index + 1:
                        ]
                    ):

                        correlation_rows.append({
                            "Modality": modality,
                            "Model_Suite": suite_name,
                            "Neural_Predictor": neural_predictor,
                            "Predictor_1": first_predictor,
                            "Predictor_2": second_predictor,
                            "Pearson_r": correlation_matrix.loc[
                                first_predictor,
                                second_predictor
                            ]
                        })

                # VIFs calculated after z-scoring.
                vif_df = (
                    complete_df[
                        predictor_columns
                    ]
                    .copy()
                )

                for predictor in predictor_columns:

                    vif_df[
                        predictor
                    ] = zscore_series(
                        vif_df[
                            predictor
                        ]
                    )

                vif_df = vif_df.dropna()

                if len(vif_df) > 0:

                    vif_matrix = np.column_stack(
                        [
                            np.ones(
                                len(vif_df)
                            ),
                            vif_df[
                                predictor_columns
                            ].to_numpy(
                                dtype=float
                            )
                        ]
                    )

                    for predictor_index, predictor in enumerate(
                        predictor_columns,
                        start=1
                    ):

                        try:

                            vif_value = (
                                variance_inflation_factor(
                                    vif_matrix,
                                    predictor_index
                                )
                            )

                        except Exception:

                            vif_value = np.nan

                        vif_rows.append({
                            "Modality": modality,
                            "Model_Suite": suite_name,
                            "Neural_Predictor": neural_predictor,
                            "Predictor": predictor,
                            "VIF": vif_value,
                            "VIF_Above_5": (
                                np.isfinite(
                                    vif_value
                                )
                                and vif_value
                                > VIF_WARNING_THRESHOLD
                            )
                        })


stage3_data_diagnostics_df = pd.DataFrame(
    diagnostic_rows
)

stage3_vif_df = pd.DataFrame(
    vif_rows
)

stage3_predictor_correlations_df = pd.DataFrame(
    correlation_rows
)


stage3_data_diagnostics_df.to_csv(
    STAGE3_DIAGNOSTIC_DIR
    / "stage3_complete_case_diagnostics.csv",
    index=False
)

stage3_vif_df.to_csv(
    STAGE3_DIAGNOSTIC_DIR
    / "stage3_predictor_vifs.csv",
    index=False
)

stage3_predictor_correlations_df.to_csv(
    STAGE3_DIAGNOSTIC_DIR
    / "stage3_predictor_correlations.csv",
    index=False
)


print(
    "\nComplete-case diagnostics:"
)

display(
    stage3_data_diagnostics_df
)


print(
    "\nPredictor VIF diagnostics:"
)

display(
    stage3_vif_df
)


# ============================================================
# 16. FIT MODELS 0–3
# ============================================================

model_fit_rows = []
neural_beta_rows = []
nested_comparison_rows = []
variance_rows = []
model_failure_rows = []

fitted_models = {}


for modality in ANALYSIS_GROUPS:

    modality_df = (
        stage3_base_df[
            stage3_base_df[
                "Modality"
            ] == modality
        ]
        .copy()
    )

    for (
        suite_name,
        suite_settings
    ) in MODEL_SUITES.items():

        motor_covariates = (
            suite_settings[
                "MotorCovariates"
            ]
        )

        for neural_predictor in (
            available_neural_predictors
        ):

            print(
                "\n" + "=" * 100
            )

            print(
                f"{modality} | {suite_name} | "
                f"{neural_predictor}"
            )

            required_columns = [
                "Subject",
                "BaseVideo",
                PERFORMANCE_COLUMN,
                *motor_covariates,
                neural_predictor
            ]

            # One common complete-case sample for Models 0–3.
            analysis_df = (
                modality_df[
                    required_columns
                ]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .dropna()
                .copy()
                .reset_index(
                    drop=True
                )
            )

            number_rows = len(
                analysis_df
            )

            number_subjects = (
                analysis_df[
                    "Subject"
                ].nunique()
            )

            number_videos = (
                analysis_df[
                    "BaseVideo"
                ].nunique()
            )

            if (
                number_rows
                < MIN_COMPLETE_ROWS
                or number_subjects
                < MIN_UNIQUE_SUBJECTS
                or number_videos
                < MIN_UNIQUE_VIDEOS
            ):

                model_failure_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Model": "Models 0–3",
                    "Error": (
                        "Insufficient complete data: "
                        f"N={number_rows}, "
                        f"subjects={number_subjects}, "
                        f"videos={number_videos}"
                    )
                })

                print(
                    "Skipped because the common complete-case "
                    "sample was too small."
                )

                continue

            # ------------------------------------------------
            # Standardize predictors within the exact common sample.
            # ------------------------------------------------

            standardized_name_map = {}

            predictors_to_standardize = (
                motor_covariates
                + [
                    neural_predictor
                ]
            )

            for predictor in predictors_to_standardize:

                standardized_name = (
                    safe_filename(
                        predictor
                    )
                    .replace(
                        "-",
                        "_"
                    )
                    + "_zmodel"
                )

                standardized_name_map[
                    predictor
                ] = standardized_name

                analysis_df[
                    standardized_name
                ] = zscore_series(
                    analysis_df[
                        predictor
                    ]
                )

            analysis_df = analysis_df.dropna(
                subset=list(
                    standardized_name_map.values()
                )
            )

            analysis_df[
                "Performance"
            ] = analysis_df[
                PERFORMANCE_COLUMN
            ]

            motor_terms = [
                standardized_name_map[
                    predictor
                ]
                for predictor in motor_covariates
            ]

            neural_term = (
                standardized_name_map[
                    neural_predictor
                ]
            )

            # ------------------------------------------------
            # Model equations
            # ------------------------------------------------

            model0_formula = (
                "Performance ~ 1"
            )

            model1_formula = (
                "Performance ~ "
                + " + ".join(
                    motor_terms
                )
            )

            model2_formula = (
                "Performance ~ "
                + neural_term
            )

            model3_formula = (
                "Performance ~ "
                + " + ".join(
                    motor_terms
                    + [
                        neural_term
                    ]
                )
            )

            model_formulas = {
                "Model_0_Baseline": model0_formula,
                "Model_1_Motor": model1_formula,
                "Model_2_Neural": model2_formula,
                "Model_3_Combined": model3_formula
            }

            suite_model_results = {}
            suite_optimizer_results = {}

            for (
                model_name,
                formula
            ) in model_formulas.items():

                (
                    model_result,
                    optimizer,
                    model_error
                ) = fit_subject_video_mixed_model(
                    formula=formula,
                    dataframe=analysis_df,
                    model_label=(
                        f"{modality} | "
                        f"{suite_name} | "
                        f"{neural_predictor} | "
                        f"{model_name}"
                    )
                )

                suite_model_results[
                    model_name
                ] = model_result

                suite_optimizer_results[
                    model_name
                ] = optimizer

                if model_error is not None:

                    model_failure_rows.append({
                        "Modality": modality,
                        "Model_Suite": suite_name,
                        "Neural_Predictor": neural_predictor,
                        "Model": model_name,
                        "Error": model_error
                    })

                output_prefix = (
                    f"{safe_filename(modality)}__"
                    f"{safe_filename(suite_name)}__"
                    f"{safe_filename(neural_predictor)}__"
                    f"{safe_filename(model_name)}"
                )

                save_model_summary(
                    model_result,
                    STAGE3_MODEL_DIR
                    / f"{output_prefix}.txt",
                    (
                        f"Modality: {modality}\n"
                        f"Model suite: {suite_name}\n"
                        f"Neural predictor: {neural_predictor}\n"
                        f"Model: {model_name}\n"
                        f"Formula: {formula}\n"
                        f"N rows: {len(analysis_df)}\n"
                        f"N subjects: "
                        f"{analysis_df['Subject'].nunique()}\n"
                        f"N videos: "
                        f"{analysis_df['BaseVideo'].nunique()}"
                    )
                )

                r2_values = mixed_model_r2(
                    model_result
                )

                random_variance_values = (
                    extract_random_variances(
                        model_result
                    )
                )

                model_fit_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Suite_Description": (
                        suite_settings[
                            "Description"
                        ]
                    ),
                    "Neural_Predictor": neural_predictor,
                    "Model": model_name,
                    "Formula": formula,
                    "N_Rows": len(
                        analysis_df
                    ),
                    "N_Subjects": analysis_df[
                        "Subject"
                    ].nunique(),
                    "N_Videos": analysis_df[
                        "BaseVideo"
                    ].nunique(),
                    "Optimizer": optimizer,
                    "Converged": (
                        bool(
                            model_result.converged
                        )
                        if model_result
                        is not None
                        else False
                    ),
                    "LogLikelihood": (
                        float(
                            model_result.llf
                        )
                        if model_result
                        is not None
                        else np.nan
                    ),
                    "AIC": (
                        float(
                            model_result.aic
                        )
                        if model_result
                        is not None
                        else np.nan
                    ),
                    "BIC": (
                        float(
                            model_result.bic
                        )
                        if model_result
                        is not None
                        else np.nan
                    ),
                    **r2_values,
                    **random_variance_values
                })

            model0_result = (
                suite_model_results[
                    "Model_0_Baseline"
                ]
            )

            model1_result = (
                suite_model_results[
                    "Model_1_Motor"
                ]
            )

            model2_result = (
                suite_model_results[
                    "Model_2_Neural"
                ]
            )

            model3_result = (
                suite_model_results[
                    "Model_3_Combined"
                ]
            )

            fitted_models[
                (
                    modality,
                    suite_name,
                    neural_predictor
                )
            ] = {
                "Data": analysis_df,
                "Model0": model0_result,
                "Model1": model1_result,
                "Model2": model2_result,
                "Model3": model3_result,
                "NeuralTerm": neural_term,
                "MotorTerms": motor_terms
            }

            # ------------------------------------------------
            # Neural beta: Model 2 versus Model 3
            # ------------------------------------------------

            model2_beta = extract_fixed_effect(
                model2_result,
                neural_term
            )

            model3_beta = extract_fixed_effect(
                model3_result,
                neural_term
            )

            signed_attenuation_percent = np.nan
            absolute_attenuation_percent = np.nan

            if (
                np.isfinite(
                    model2_beta[
                        "Beta"
                    ]
                )
                and model2_beta[
                    "Beta"
                ] != 0
                and np.isfinite(
                    model3_beta[
                        "Beta"
                    ]
                )
            ):

                signed_attenuation_percent = (
                    100.0
                    * (
                        model2_beta[
                            "Beta"
                        ]
                        - model3_beta[
                            "Beta"
                        ]
                    )
                    / model2_beta[
                        "Beta"
                    ]
                )

                absolute_attenuation_percent = (
                    100.0
                    * (
                        abs(
                            model2_beta[
                                "Beta"
                            ]
                        )
                        - abs(
                            model3_beta[
                                "Beta"
                            ]
                        )
                    )
                    / abs(
                        model2_beta[
                            "Beta"
                        ]
                    )
                )

            neural_beta_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Neural_Predictor": neural_predictor,

                "Model2_Beta": (
                    model2_beta[
                        "Beta"
                    ]
                ),
                "Model2_SE": (
                    model2_beta[
                        "SE"
                    ]
                ),
                "Model2_Z": (
                    model2_beta[
                        "Z"
                    ]
                ),
                "Model2_P": (
                    model2_beta[
                        "P"
                    ]
                ),
                "Model2_CI_Lower": (
                    model2_beta[
                        "CI_Lower"
                    ]
                ),
                "Model2_CI_Upper": (
                    model2_beta[
                        "CI_Upper"
                    ]
                ),

                "Model3_Beta": (
                    model3_beta[
                        "Beta"
                    ]
                ),
                "Model3_SE": (
                    model3_beta[
                        "SE"
                    ]
                ),
                "Model3_Z": (
                    model3_beta[
                        "Z"
                    ]
                ),
                "Model3_P": (
                    model3_beta[
                        "P"
                    ]
                ),
                "Model3_CI_Lower": (
                    model3_beta[
                        "CI_Lower"
                    ]
                ),
                "Model3_CI_Upper": (
                    model3_beta[
                        "CI_Upper"
                    ]
                ),

                "Signed_Beta_Attenuation_Percent": (
                    signed_attenuation_percent
                ),

                "Absolute_Beta_Attenuation_Percent": (
                    absolute_attenuation_percent
                ),

                "Beta_Direction_Preserved": (
                    np.sign(
                        model2_beta[
                            "Beta"
                        ]
                    )
                    ==
                    np.sign(
                        model3_beta[
                            "Beta"
                        ]
                    )
                    if (
                        np.isfinite(
                            model2_beta[
                                "Beta"
                            ]
                        )
                        and np.isfinite(
                            model3_beta[
                                "Beta"
                            ]
                        )
                    )
                    else np.nan
                )
            })

            # ------------------------------------------------
            # Nested likelihood-ratio comparisons
            # ------------------------------------------------

            model_comparisons = [
                (
                    "Model0_vs_Model1_Motor",
                    model0_result,
                    model1_result
                ),
                (
                    "Model0_vs_Model2_Neural",
                    model0_result,
                    model2_result
                ),
                (
                    "Model1_vs_Model3_AddNeural",
                    model1_result,
                    model3_result
                ),
                (
                    "Model2_vs_Model3_AddMotor",
                    model2_result,
                    model3_result
                )
            ]

            for (
                comparison_name,
                reduced_result,
                full_result
            ) in model_comparisons:

                lrt_result = likelihood_ratio_test(
                    reduced_result,
                    full_result
                )

                nested_comparison_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Comparison": comparison_name,
                    "LR": lrt_result[
                        "LR"
                    ],
                    "DF": lrt_result[
                        "DF"
                    ],
                    "P": lrt_result[
                        "P"
                    ],
                    "Reduced_AIC": (
                        reduced_result.aic
                        if reduced_result
                        is not None
                        else np.nan
                    ),
                    "Full_AIC": (
                        full_result.aic
                        if full_result
                        is not None
                        else np.nan
                    ),
                    "Delta_AIC_FullMinusReduced": (
                        full_result.aic
                        - reduced_result.aic
                        if (
                            reduced_result
                            is not None
                            and full_result
                            is not None
                        )
                        else np.nan
                    ),
                    "Reduced_BIC": (
                        reduced_result.bic
                        if reduced_result
                        is not None
                        else np.nan
                    ),
                    "Full_BIC": (
                        full_result.bic
                        if full_result
                        is not None
                        else np.nan
                    ),
                    "Delta_BIC_FullMinusReduced": (
                        full_result.bic
                        - reduced_result.bic
                        if (
                            reduced_result
                            is not None
                            and full_result
                            is not None
                        )
                        else np.nan
                    )
                })

            # ------------------------------------------------
            # Marginal-R² comparisons
            # ------------------------------------------------

            model0_r2 = mixed_model_r2(
                model0_result
            )

            model1_r2 = mixed_model_r2(
                model1_result
            )

            model2_r2 = mixed_model_r2(
                model2_result
            )

            model3_r2 = mixed_model_r2(
                model3_result
            )

            variance_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Neural_Predictor": neural_predictor,

                "Model0_Marginal_R2": (
                    model0_r2[
                        "MarginalR2"
                    ]
                ),

                "Model1_Motor_Marginal_R2": (
                    model1_r2[
                        "MarginalR2"
                    ]
                ),

                "Model2_Neural_Marginal_R2": (
                    model2_r2[
                        "MarginalR2"
                    ]
                ),

                "Model3_Combined_Marginal_R2": (
                    model3_r2[
                        "MarginalR2"
                    ]
                ),

                "Model0_Conditional_R2": (
                    model0_r2[
                        "ConditionalR2"
                    ]
                ),

                "Model1_Conditional_R2": (
                    model1_r2[
                        "ConditionalR2"
                    ]
                ),

                "Model2_Conditional_R2": (
                    model2_r2[
                        "ConditionalR2"
                    ]
                ),

                "Model3_Conditional_R2": (
                    model3_r2[
                        "ConditionalR2"
                    ]
                ),

                "Delta_R2_Motor_vs_Baseline": (
                    model1_r2[
                        "MarginalR2"
                    ]
                    - model0_r2[
                        "MarginalR2"
                    ]
                ),

                "Delta_R2_Neural_vs_Baseline": (
                    model2_r2[
                        "MarginalR2"
                    ]
                    - model0_r2[
                        "MarginalR2"
                    ]
                ),

                "Unique_Neural_Delta_R2": (
                    model3_r2[
                        "MarginalR2"
                    ]
                    - model1_r2[
                        "MarginalR2"
                    ]
                ),

                "Unique_Motor_Delta_R2": (
                    model3_r2[
                        "MarginalR2"
                    ]
                    - model2_r2[
                        "MarginalR2"
                    ]
                ),

                "Shared_Motor_Neural_R2": (
                    model1_r2[
                        "MarginalR2"
                    ]
                    + model2_r2[
                        "MarginalR2"
                    ]
                    - model3_r2[
                        "MarginalR2"
                    ]
                )
            })


# ============================================================
# 17. CREATE IN-SAMPLE RESULT TABLES
# ============================================================

stage3_model_fit_df = pd.DataFrame(
    model_fit_rows
)

stage3_neural_beta_comparison_df = pd.DataFrame(
    neural_beta_rows
)

stage3_nested_comparisons_df = pd.DataFrame(
    nested_comparison_rows
)

stage3_variance_comparison_df = pd.DataFrame(
    variance_rows
)

stage3_model_failures_df = pd.DataFrame(
    model_failure_rows
)


# ============================================================
# 18. FDR CORRECTION
# ============================================================

if not stage3_neural_beta_comparison_df.empty:

    stage3_neural_beta_comparison_df[
        "Model2_P_FDR"
    ] = np.nan

    stage3_neural_beta_comparison_df[
        "Model3_P_FDR"
    ] = np.nan

    for modality in ANALYSIS_GROUPS:

        for suite_name in MODEL_SUITES:

            selection = (
                (
                    stage3_neural_beta_comparison_df[
                        "Modality"
                    ] == modality
                )
                &
                (
                    stage3_neural_beta_comparison_df[
                        "Model_Suite"
                    ] == suite_name
                )
            )

            model2_valid = (
                selection
                &
                stage3_neural_beta_comparison_df[
                    "Model2_P"
                ].notna()
            )

            model3_valid = (
                selection
                &
                stage3_neural_beta_comparison_df[
                    "Model3_P"
                ].notna()
            )

            if model2_valid.sum() > 0:

                stage3_neural_beta_comparison_df.loc[
                    model2_valid,
                    "Model2_P_FDR"
                ] = multipletests(
                    stage3_neural_beta_comparison_df.loc[
                        model2_valid,
                        "Model2_P"
                    ],
                    method="fdr_bh"
                )[1]

            if model3_valid.sum() > 0:

                stage3_neural_beta_comparison_df.loc[
                    model3_valid,
                    "Model3_P_FDR"
                ] = multipletests(
                    stage3_neural_beta_comparison_df.loc[
                        model3_valid,
                        "Model3_P"
                    ],
                    method="fdr_bh"
                )[1]


if not stage3_nested_comparisons_df.empty:

    stage3_nested_comparisons_df[
        "P_FDR"
    ] = np.nan

    for modality in ANALYSIS_GROUPS:

        for suite_name in MODEL_SUITES:

            for comparison_name in (
                stage3_nested_comparisons_df[
                    "Comparison"
                ].dropna().unique()
            ):

                selection = (
                    (
                        stage3_nested_comparisons_df[
                            "Modality"
                        ] == modality
                    )
                    &
                    (
                        stage3_nested_comparisons_df[
                            "Model_Suite"
                        ] == suite_name
                    )
                    &
                    (
                        stage3_nested_comparisons_df[
                            "Comparison"
                        ] == comparison_name
                    )
                    &
                    stage3_nested_comparisons_df[
                        "P"
                    ].notna()
                )

                if selection.sum() > 0:

                    stage3_nested_comparisons_df.loc[
                        selection,
                        "P_FDR"
                    ] = multipletests(
                        stage3_nested_comparisons_df.loc[
                            selection,
                            "P"
                        ],
                        method="fdr_bh"
                    )[1]


# ============================================================
# 19. SUBJECT-GROUPED HELD-OUT PREDICTION
#
# Compare:
#
#   Model 1: motor only
#   Model 3: motor + network balance
#
# Subjects never appear in both training and test folds.
# ============================================================

cv_fold_rows = []
cv_prediction_rows = []
cv_failure_rows = []


for modality in ANALYSIS_GROUPS:

    modality_df = (
        stage3_base_df[
            stage3_base_df[
                "Modality"
            ] == modality
        ]
        .copy()
    )

    for (
        suite_name,
        suite_settings
    ) in MODEL_SUITES.items():

        motor_covariates = (
            suite_settings[
                "MotorCovariates"
            ]
        )

        for neural_predictor in (
            available_neural_predictors
        ):

            print(
                "\n" + "-" * 100
            )

            print(
                f"Cross-validation: {modality} | "
                f"{suite_name} | {neural_predictor}"
            )

            required_columns = [
                "Subject",
                "BaseVideo",
                PERFORMANCE_COLUMN,
                *motor_covariates,
                neural_predictor
            ]

            cv_df = (
                modality_df[
                    required_columns
                ]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .dropna()
                .copy()
                .reset_index(
                    drop=True
                )
            )

            number_subjects = (
                cv_df[
                    "Subject"
                ].nunique()
            )

            number_videos = (
                cv_df[
                    "BaseVideo"
                ].nunique()
            )

            if (
                len(cv_df)
                < MIN_COMPLETE_ROWS
                or number_subjects
                < max(
                    3,
                    CV_N_SPLITS
                )
                or number_videos
                < MIN_UNIQUE_VIDEOS
            ):

                cv_failure_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Fold": np.nan,
                    "Error": (
                        "Insufficient complete data for "
                        "subject-grouped cross-validation."
                    )
                })

                continue

            number_splits = min(
                CV_N_SPLITS,
                number_subjects
            )

            group_kfold = GroupKFold(
                n_splits=number_splits
            )

            fold_number = 0

            predictor_columns = (
                motor_covariates
                + [
                    neural_predictor
                ]
            )

            for (
                train_indices,
                test_indices
            ) in group_kfold.split(
                cv_df,
                groups=cv_df[
                    "Subject"
                ]
            ):

                fold_number += 1

                train_df = cv_df.iloc[
                    train_indices
                ].copy()

                test_df = cv_df.iloc[
                    test_indices
                ].copy()

                (
                    train_scaled,
                    test_scaled,
                    scaling_df
                ) = standardize_train_test(
                    train_df=train_df,
                    test_df=test_df,
                    predictor_columns=(
                        predictor_columns
                    )
                )

                if train_scaled is None:

                    cv_failure_rows.append({
                        "Modality": modality,
                        "Model_Suite": suite_name,
                        "Neural_Predictor": neural_predictor,
                        "Fold": fold_number,
                        "Error": (
                            "At least one predictor had zero "
                            "variance in the training fold."
                        )
                    })

                    continue

                train_scaled[
                    "Performance"
                ] = train_scaled[
                    PERFORMANCE_COLUMN
                ]

                test_scaled[
                    "Performance"
                ] = test_scaled[
                    PERFORMANCE_COLUMN
                ]

                # Rename standardized predictors to simple formula names.
                motor_cv_terms = []

                for predictor_index, predictor in enumerate(
                    motor_covariates,
                    start=1
                ):

                    simple_name = (
                        f"MotorCovariate"
                        f"{predictor_index}"
                    )

                    train_scaled[
                        simple_name
                    ] = train_scaled[
                        predictor
                    ]

                    test_scaled[
                        simple_name
                    ] = test_scaled[
                        predictor
                    ]

                    motor_cv_terms.append(
                        simple_name
                    )

                train_scaled[
                    "NeuralPredictor"
                ] = train_scaled[
                    neural_predictor
                ]

                test_scaled[
                    "NeuralPredictor"
                ] = test_scaled[
                    neural_predictor
                ]

                model1_formula = (
                    "Performance ~ "
                    + " + ".join(
                        motor_cv_terms
                    )
                )

                model3_formula = (
                    "Performance ~ "
                    + " + ".join(
                        motor_cv_terms
                        + [
                            "NeuralPredictor"
                        ]
                    )
                )

                (
                    model1_result,
                    _,
                    model1_error
                ) = fit_subject_video_mixed_model(
                    formula=model1_formula,
                    dataframe=train_scaled,
                    model_label=(
                        f"CV Model 1, fold "
                        f"{fold_number}"
                    )
                )

                (
                    model3_result,
                    _,
                    model3_error
                ) = fit_subject_video_mixed_model(
                    formula=model3_formula,
                    dataframe=train_scaled,
                    model_label=(
                        f"CV Model 3, fold "
                        f"{fold_number}"
                    )
                )

                if (
                    model1_result is None
                    or model3_result is None
                ):

                    cv_failure_rows.append({
                        "Modality": modality,
                        "Model_Suite": suite_name,
                        "Neural_Predictor": neural_predictor,
                        "Fold": fold_number,
                        "Error": (
                            f"Model 1: {model1_error}; "
                            f"Model 3: {model3_error}"
                        )
                    })

                    continue

                observed = test_scaled[
                    "Performance"
                ].to_numpy(
                    dtype=float
                )

                model1_predictions = (
                    fixed_effect_predictions(
                        model1_result,
                        test_scaled
                    )
                )

                model3_predictions = (
                    fixed_effect_predictions(
                        model3_result,
                        test_scaled
                    )
                )

                model1_metrics = prediction_metrics(
                    observed,
                    model1_predictions
                )

                model3_metrics = prediction_metrics(
                    observed,
                    model3_predictions
                )

                cv_fold_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Fold": fold_number,

                    "N_Train": len(
                        train_scaled
                    ),
                    "N_Test": len(
                        test_scaled
                    ),

                    "N_Train_Subjects": (
                        train_scaled[
                            "Subject"
                        ].nunique()
                    ),

                    "N_Test_Subjects": (
                        test_scaled[
                            "Subject"
                        ].nunique()
                    ),

                    "Model1_Motor_RMSE": (
                        model1_metrics[
                            "RMSE"
                        ]
                    ),

                    "Model3_Combined_RMSE": (
                        model3_metrics[
                            "RMSE"
                        ]
                    ),

                    "Delta_RMSE_Model3MinusModel1": (
                        model3_metrics[
                            "RMSE"
                        ]
                        - model1_metrics[
                            "RMSE"
                        ]
                    ),

                    "Model1_Motor_MAE": (
                        model1_metrics[
                            "MAE"
                        ]
                    ),

                    "Model3_Combined_MAE": (
                        model3_metrics[
                            "MAE"
                        ]
                    ),

                    "Delta_MAE_Model3MinusModel1": (
                        model3_metrics[
                            "MAE"
                        ]
                        - model1_metrics[
                            "MAE"
                        ]
                    ),

                    "Model1_Motor_R2": (
                        model1_metrics[
                            "R2"
                        ]
                    ),

                    "Model3_Combined_R2": (
                        model3_metrics[
                            "R2"
                        ]
                    ),

                    "Delta_R2_Model3MinusModel1": (
                        model3_metrics[
                            "R2"
                        ]
                        - model1_metrics[
                            "R2"
                        ]
                    )
                })

                for test_row_index in range(
                    len(test_scaled)
                ):

                    cv_prediction_rows.append({
                        "Modality": modality,
                        "Model_Suite": suite_name,
                        "Neural_Predictor": neural_predictor,
                        "Fold": fold_number,

                        "Subject": test_scaled.iloc[
                            test_row_index
                        ][
                            "Subject"
                        ],

                        "BaseVideo": test_scaled.iloc[
                            test_row_index
                        ][
                            "BaseVideo"
                        ],

                        "Observed_Performance": (
                            observed[
                                test_row_index
                            ]
                        ),

                        "Model1_Motor_Prediction": (
                            model1_predictions[
                                test_row_index
                            ]
                        ),

                        "Model3_Combined_Prediction": (
                            model3_predictions[
                                test_row_index
                            ]
                        )
                    })


stage3_cv_fold_df = pd.DataFrame(
    cv_fold_rows
)

stage3_cv_predictions_df = pd.DataFrame(
    cv_prediction_rows
)

stage3_cv_failures_df = pd.DataFrame(
    cv_failure_rows
)


# ============================================================
# 20. CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_rows = []


if not stage3_cv_fold_df.empty:

    grouping_columns = [
        "Modality",
        "Model_Suite",
        "Neural_Predictor"
    ]

    for (
        group_values,
        group_df
    ) in stage3_cv_fold_df.groupby(
        grouping_columns
    ):

        (
            modality,
            suite_name,
            neural_predictor
        ) = group_values

        rmse_test = ttest_rel(
            group_df[
                "Model3_Combined_RMSE"
            ],
            group_df[
                "Model1_Motor_RMSE"
            ],
            nan_policy="omit"
        )

        mae_test = ttest_rel(
            group_df[
                "Model3_Combined_MAE"
            ],
            group_df[
                "Model1_Motor_MAE"
            ],
            nan_policy="omit"
        )

        r2_test = ttest_rel(
            group_df[
                "Model3_Combined_R2"
            ],
            group_df[
                "Model1_Motor_R2"
            ],
            nan_policy="omit"
        )

        cv_summary_rows.append({
            "Modality": modality,
            "Model_Suite": suite_name,
            "Neural_Predictor": neural_predictor,
            "N_Folds": len(
                group_df
            ),

            "Mean_Model1_RMSE": group_df[
                "Model1_Motor_RMSE"
            ].mean(),

            "Mean_Model3_RMSE": group_df[
                "Model3_Combined_RMSE"
            ].mean(),

            "Mean_Delta_RMSE_Model3MinusModel1": (
                group_df[
                    "Delta_RMSE_Model3MinusModel1"
                ].mean()
            ),

            "RMSE_Paired_T": (
                rmse_test.statistic
            ),

            "RMSE_Paired_P": (
                rmse_test.pvalue
            ),

            "Mean_Model1_MAE": group_df[
                "Model1_Motor_MAE"
            ].mean(),

            "Mean_Model3_MAE": group_df[
                "Model3_Combined_MAE"
            ].mean(),

            "Mean_Delta_MAE_Model3MinusModel1": (
                group_df[
                    "Delta_MAE_Model3MinusModel1"
                ].mean()
            ),

            "MAE_Paired_T": (
                mae_test.statistic
            ),

            "MAE_Paired_P": (
                mae_test.pvalue
            ),

            "Mean_Model1_R2": group_df[
                "Model1_Motor_R2"
            ].mean(),

            "Mean_Model3_R2": group_df[
                "Model3_Combined_R2"
            ].mean(),

            "Mean_Delta_R2_Model3MinusModel1": (
                group_df[
                    "Delta_R2_Model3MinusModel1"
                ].mean()
            ),

            "R2_Paired_T": (
                r2_test.statistic
            ),

            "R2_Paired_P": (
                r2_test.pvalue
            ),

            "Proportion_Folds_Model3_Lower_RMSE": (
                group_df[
                    "Model3_Combined_RMSE"
                ]
                <
                group_df[
                    "Model1_Motor_RMSE"
                ]
            ).mean(),

            "Proportion_Folds_Model3_Higher_R2": (
                group_df[
                    "Model3_Combined_R2"
                ]
                >
                group_df[
                    "Model1_Motor_R2"
                ]
            ).mean()
        })


stage3_cv_summary_df = pd.DataFrame(
    cv_summary_rows
)


# ============================================================
# 21. SAVE ALL OUTPUT TABLES
# ============================================================

stage3_model_fit_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_model_fit_statistics.csv",
    index=False
)

stage3_neural_beta_comparison_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_network_beta_before_after_motor_control.csv",
    index=False
)

stage3_nested_comparisons_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_nested_model_comparisons.csv",
    index=False
)

stage3_variance_comparison_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_marginal_r2_comparisons.csv",
    index=False
)

stage3_model_failures_df.to_csv(
    STAGE3_TABLE_DIR
    / "stage3_model_failures.csv",
    index=False
)

stage3_cv_fold_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_subject_grouped_cv_fold_results.csv",
    index=False
)

stage3_cv_predictions_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_subject_grouped_cv_predictions.csv",
    index=False
)

stage3_cv_summary_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_subject_grouped_cv_summary.csv",
    index=False
)

stage3_cv_failures_df.to_csv(
    STAGE3_CV_DIR
    / "stage3_subject_grouped_cv_failures.csv",
    index=False
)


# ============================================================
# 22. FIGURE — NETWORK BETA BEFORE AND AFTER MOTOR CONTROL
# ============================================================

if not stage3_neural_beta_comparison_df.empty:

    for modality in ANALYSIS_GROUPS:

        for suite_name in MODEL_SUITES:

            plot_df = (
                stage3_neural_beta_comparison_df[
                    (
                        stage3_neural_beta_comparison_df[
                            "Modality"
                        ] == modality
                    )
                    &
                    (
                        stage3_neural_beta_comparison_df[
                            "Model_Suite"
                        ] == suite_name
                    )
                ]
                .copy()
            )

            if plot_df.empty:
                continue

            plot_df = (
                plot_df.set_index(
                    "Neural_Predictor"
                )
                .reindex(
                    [
                        predictor
                        for predictor
                        in available_neural_predictors
                        if predictor
                        in plot_df[
                            "Neural_Predictor"
                        ].values
                    ]
                )
                .reset_index()
            )

            x_positions = np.arange(
                len(plot_df)
            )

            fig, ax = plt.subplots(
                figsize=(
                    max(
                        9,
                        3.0
                        * len(plot_df)
                    ),
                    6
                )
            )

            width_offset = 0.16

            model2_lower_error = (
                plot_df[
                    "Model2_Beta"
                ]
                - plot_df[
                    "Model2_CI_Lower"
                ]
            )

            model2_upper_error = (
                plot_df[
                    "Model2_CI_Upper"
                ]
                - plot_df[
                    "Model2_Beta"
                ]
            )

            model3_lower_error = (
                plot_df[
                    "Model3_Beta"
                ]
                - plot_df[
                    "Model3_CI_Lower"
                ]
            )

            model3_upper_error = (
                plot_df[
                    "Model3_CI_Upper"
                ]
                - plot_df[
                    "Model3_Beta"
                ]
            )

            ax.errorbar(
                x_positions
                - width_offset,

                plot_df[
                    "Model2_Beta"
                ],

                yerr=np.vstack(
                    [
                        model2_lower_error,
                        model2_upper_error
                    ]
                ),

                fmt="o",
                capsize=5,
                label="Model 2: neural only"
            )

            ax.errorbar(
                x_positions
                + width_offset,

                plot_df[
                    "Model3_Beta"
                ],

                yerr=np.vstack(
                    [
                        model3_lower_error,
                        model3_upper_error
                    ]
                ),

                fmt="o",
                capsize=5,
                label="Model 3: after motor control"
            )

            ax.axhline(
                0,
                linestyle="--",
                linewidth=1
            )

            ax.set_xticks(
                x_positions
            )

            ax.set_xticklabels(
                plot_df[
                    "Neural_Predictor"
                ],
                rotation=20,
                ha="right"
            )

            ax.set_ylabel(
                "Network-balance beta predicting PC1 alignment"
            )

            ax.set_title(
                f"{modality} | {suite_name}\n"
                "Network beta before and after motor adjustment"
            )

            ax.legend(
                frameon=False
            )

            fig.tight_layout()

            fig.savefig(
                STAGE3_FIG_DIR
                / (
                    f"{safe_filename(modality)}__"
                    f"{safe_filename(suite_name)}__"
                    "network_beta_attenuation.png"
                ),
                dpi=FIGURE_DPI,
                bbox_inches="tight"
            )

            plt.show()
            plt.close(fig)


# ============================================================
# 23. FIGURE — UNIQUE NEURAL MARGINAL R²
# ============================================================

if not stage3_variance_comparison_df.empty:

    for modality in ANALYSIS_GROUPS:

        plot_df = (
            stage3_variance_comparison_df[
                stage3_variance_comparison_df[
                    "Modality"
                ] == modality
            ]
            .copy()
        )

        if plot_df.empty:
            continue

        plot_df[
            "Label"
        ] = (
            plot_df[
                "Model_Suite"
            ]
            + "\n"
            + plot_df[
                "Neural_Predictor"
            ]
        )

        x_positions = np.arange(
            len(plot_df)
        )

        fig, ax = plt.subplots(
            figsize=(
                max(
                    10,
                    2.2
                    * len(plot_df)
                ),
                6
            )
        )

        ax.bar(
            x_positions,
            plot_df[
                "Unique_Neural_Delta_R2"
            ]
        )

        ax.axhline(
            0,
            linewidth=1
        )

        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(
            plot_df[
                "Label"
            ],
            rotation=25,
            ha="right"
        )

        ax.set_ylabel(
            "Incremental marginal R²\n"
            "(Model 3 − Model 1)"
        )

        ax.set_title(
            f"{modality}: unique network-balance variance "
            "beyond motor controls"
        )

        fig.tight_layout()

        fig.savefig(
            STAGE3_FIG_DIR
            / (
                f"{safe_filename(modality)}__"
                "unique_neural_delta_r2.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 24. FIGURE — CROSS-VALIDATED RMSE
# ============================================================

if not stage3_cv_summary_df.empty:

    for modality in ANALYSIS_GROUPS:

        plot_df = (
            stage3_cv_summary_df[
                stage3_cv_summary_df[
                    "Modality"
                ] == modality
            ]
            .copy()
        )

        if plot_df.empty:
            continue

        plot_df[
            "Label"
        ] = (
            plot_df[
                "Model_Suite"
            ]
            + "\n"
            + plot_df[
                "Neural_Predictor"
            ]
        )

        x_positions = np.arange(
            len(plot_df)
        )

        width = 0.36

        fig, ax = plt.subplots(
            figsize=(
                max(
                    10,
                    2.3
                    * len(plot_df)
                ),
                6
            )
        )

        ax.bar(
            x_positions
            - width / 2,

            plot_df[
                "Mean_Model1_RMSE"
            ],

            width=width,
            label="Model 1: motor only"
        )

        ax.bar(
            x_positions
            + width / 2,

            plot_df[
                "Mean_Model3_RMSE"
            ],

            width=width,
            label="Model 3: motor + neural"
        )

        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(
            plot_df[
                "Label"
            ],
            rotation=25,
            ha="right"
        )

        ax.set_ylabel(
            "Held-out subject RMSE"
        )

        ax.set_title(
            f"{modality}: subject-grouped held-out prediction"
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        fig.savefig(
            STAGE3_FIG_DIR
            / (
                f"{safe_filename(modality)}__"
                "subject_grouped_cv_rmse.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 25. PRINT KEY RESULTS
# ============================================================

print(
    "\n" + "=" * 105
)

print(
    "STAGE 3 MODEL COMPARISONS COMPLETE"
)

print(
    "=" * 105
)


print(
    "\nComplete-case sample diagnostics:"
)

display(
    stage3_data_diagnostics_df.round(
        5
    )
)


print(
    "\nNetwork beta before and after motor control:"
)

if stage3_neural_beta_comparison_df.empty:

    print(
        "No neural-beta results were available."
    )

else:

    display(
        stage3_neural_beta_comparison_df[
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor",

                "Model2_Beta",
                "Model2_SE",
                "Model2_P",
                "Model2_P_FDR",

                "Model3_Beta",
                "Model3_SE",
                "Model3_P",
                "Model3_P_FDR",

                "Signed_Beta_Attenuation_Percent",
                "Absolute_Beta_Attenuation_Percent",
                "Beta_Direction_Preserved"
            ]
        ]
        .sort_values(
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor"
            ]
        )
        .round(5)
    )


print(
    "\nLikelihood-ratio and information-criterion comparisons:"
)

if stage3_nested_comparisons_df.empty:

    print(
        "No nested-model comparisons were available."
    )

else:

    display(
        stage3_nested_comparisons_df[
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor",
                "Comparison",
                "LR",
                "DF",
                "P",
                "P_FDR",
                "Delta_AIC_FullMinusReduced",
                "Delta_BIC_FullMinusReduced"
            ]
        ]
        .sort_values(
            [
                "Modality",
                "Model_Suite",
                "Comparison",
                "P"
            ]
        )
        .round(5)
    )


print(
    "\nMarginal-R² comparisons:"
)

if stage3_variance_comparison_df.empty:

    print(
        "No R² comparisons were available."
    )

else:

    display(
        stage3_variance_comparison_df[
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor",

                "Model0_Marginal_R2",
                "Model1_Motor_Marginal_R2",
                "Model2_Neural_Marginal_R2",
                "Model3_Combined_Marginal_R2",

                "Delta_R2_Motor_vs_Baseline",
                "Delta_R2_Neural_vs_Baseline",
                "Unique_Neural_Delta_R2",
                "Unique_Motor_Delta_R2",
                "Shared_Motor_Neural_R2"
            ]
        ]
        .sort_values(
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor"
            ]
        )
        .round(5)
    )


print(
    "\nSubject-grouped held-out prediction:"
)

if stage3_cv_summary_df.empty:

    print(
        "No held-out prediction results were available."
    )

else:

    display(
        stage3_cv_summary_df[
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor",
                "N_Folds",

                "Mean_Model1_RMSE",
                "Mean_Model3_RMSE",
                "Mean_Delta_RMSE_Model3MinusModel1",
                "RMSE_Paired_P",

                "Mean_Model1_R2",
                "Mean_Model3_R2",
                "Mean_Delta_R2_Model3MinusModel1",
                "R2_Paired_P",

                "Proportion_Folds_Model3_Lower_RMSE",
                "Proportion_Folds_Model3_Higher_R2"
            ]
        ]
        .sort_values(
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor"
            ]
        )
        .round(5)
    )


print(
    "\nModel fit statistics and random-effect diagnostics:"
)

display(
    stage3_model_fit_df[
        [
            "Modality",
            "Model_Suite",
            "Neural_Predictor",
            "Model",
            "N_Rows",
            "N_Subjects",
            "N_Videos",
            "Optimizer",
            "Converged",
            "AIC",
            "BIC",
            "MarginalR2",
            "ConditionalR2",
            "SubjectVariance",
            "VideoVariance",
            "ResidualVariance",
            "NearZeroRandomVariance"
        ]
    ]
    .sort_values(
        [
            "Modality",
            "Model_Suite",
            "Neural_Predictor",
            "Model"
        ]
    )
    .round(6)
)


if not stage3_model_failures_df.empty:

    print(
        "\nModel-fitting failures:"
    )

    display(
        stage3_model_failures_df
    )


if not stage3_cv_failures_df.empty:

    print(
        "\nCross-validation failures:"
    )

    display(
        stage3_cv_failures_df
    )


print(
    "\nInterpretation guide:"
)

print(
    "  • Model 2 beta: unadjusted network-balance association "
    "within the common model sample."
)

print(
    "  • Model 3 beta: network-balance association after motor "
    "adjustment."
)

print(
    "  • Absolute beta attenuation near 0%: little evidence that "
    "motor controls explain the network effect."
)

print(
    "  • Model1_vs_Model3_AddNeural: primary likelihood-ratio test "
    "of whether network balance improves the motor-only model."
)

print(
    "  • Unique_Neural_Delta_R2: increase in marginal R² from "
    "Model 1 to Model 3."
)

print(
    "  • Held-out ΔRMSE < 0: Model 3 predicts unseen participants "
    "better than Model 1."
)

print(
    "  • Held-out ΔR² > 0: adding network balance improves "
    "out-of-sample explained variance."
)

print(
    "  • Compare Primary_MotorPCs with "
    "LeftMotorROI_Sensitivity to determine whether adding left "
    "motor-region ISC changes the network beta."
)

print(
    "\nAll outputs were saved to:\n"
    f"{STAGE3_OUTPUT_DIR}"
)

In [ ]:
# ============================================================
#  RESIDUAL NEURAL-BETA ANALYSIS
#
# PURPOSE
# ------------------------------------------------------------
# 1. Fit a motor-only mixed-effects model predicting behavioral
#    performance.
#
# 2. Extract the behavioral residuals not explained by:
#       Primary suite:
#           MotorPC1 + MotorPC2 + MotorPC3
#
#       Sensitivity suite:
#           MotorPC1 + MotorPC2 + MotorPC3
#           + Left motor-control ROI ISC
#
# 3. Test whether network balance predicts the remaining
#    behavioral-performance variance.
#
# ANALYSES RUN SEPARATELY FOR:
#   - Valence
#   - Arousal
#
# NETWORK PREDICTORS TESTED SEPARATELY:
#   - Original_Network_Balance
#   - Full_LASSO_Network_Balance
#
# RANDOM EFFECTS:
#   - Subject random intercept
#   - BaseVideo random intercept
#
# REQUIRED OBJECT FROM THE RECENT MODEL-COMPARISON CODE:
#   stage3_base_df
#
# OUTPUT DIRECTORY:
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data
# \Emotion Perception Data\Motor_Control
# \03_models\residual_beta
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)


# ============================================================
# 2. CHECK REQUIRED INPUT
# ============================================================

if "stage3_base_df" not in globals():
    raise ValueError(
        "stage3_base_df was not found.\n\n"
        "Run the recent Stage 3 Model 0–3 comparison code first. "
        "That code creates the merged subject × video dataframe "
        "needed for this residual-beta analysis."
    )


# ============================================================
# 3. OUTPUT DIRECTORIES
# ============================================================

RESIDUAL_BETA_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\03_models\residual_beta"
)

RESIDUAL_BETA_TABLE_DIR = (
    RESIDUAL_BETA_OUTPUT_DIR / "tables"
)

RESIDUAL_BETA_MODEL_DIR = (
    RESIDUAL_BETA_OUTPUT_DIR / "model_summaries"
)

RESIDUAL_BETA_FIG_DIR = (
    RESIDUAL_BETA_OUTPUT_DIR / "figures"
)

RESIDUAL_BETA_DATA_DIR = (
    RESIDUAL_BETA_OUTPUT_DIR / "residualized_data"
)

for folder in [
    RESIDUAL_BETA_OUTPUT_DIR,
    RESIDUAL_BETA_TABLE_DIR,
    RESIDUAL_BETA_MODEL_DIR,
    RESIDUAL_BETA_FIG_DIR,
    RESIDUAL_BETA_DATA_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Residual neural-beta outputs will be saved to:\n"
    f"{RESIDUAL_BETA_OUTPUT_DIR}"
)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

PERFORMANCE_COLUMN = "r_PC1"

MOTOR_PC_COLUMNS = [
    "MotorPC1",
    "MotorPC2",
    "MotorPC3"
]

LEFT_MOTOR_ROI_COLUMN = (
    "Left_Motor_ROI_ISC_z"
)

NEURAL_PREDICTORS = [
    "Original_Network_Balance",
    "Full_LASSO_Network_Balance"
]

ANALYSIS_GROUPS = [
    "Valence",
    "Arousal"
]

MODEL_SUITES = {
    "Primary_MotorPCs": {
        "MotorCovariates": [
            "MotorPC1",
            "MotorPC2",
            "MotorPC3"
        ],

        "Description": (
            "Primary full-sample residualization using "
            "MotorPC1–3"
        )
    },

    "LeftMotorROI_Sensitivity": {
        "MotorCovariates": [
            "MotorPC1",
            "MotorPC2",
            "MotorPC3",
            LEFT_MOTOR_ROI_COLUMN
        ],

        "Description": (
            "Reduced-sample residualization using MotorPC1–3 "
            "and left motor-control ROI ISC"
        )
    }
}

MIN_COMPLETE_ROWS = 80
MIN_UNIQUE_SUBJECTS = 15
MIN_UNIQUE_VIDEOS = 5

MODEL_MAX_ITER = 5000

NEAR_ZERO_VARIANCE_THRESHOLD = 1e-8

RANDOM_VARIANCE_WARNING_THRESHOLD = 1e-6

FIGURE_DPI = 300


# ============================================================
# 5. GENERAL HELPERS
# ============================================================

def residual_safe_filename(
    value,
    max_length=160
):
    """
    Create a Windows-safe filename.
    """
    value = str(value)

    value = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        value
    )

    value = re.sub(
        r"\s+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    value = value.strip(
        "._ "
    )

    return (
        value[:max_length]
        or "unnamed"
    )


def residual_zscore_series(series):
    """
    Z-score one numeric pandas Series.
    """
    numeric_series = pd.to_numeric(
        series,
        errors="coerce"
    )

    mean_value = numeric_series.mean()

    sd_value = numeric_series.std(
        ddof=1
    )

    if (
        not np.isfinite(sd_value)
        or sd_value
        <= NEAR_ZERO_VARIANCE_THRESHOLD
    ):
        return pd.Series(
            np.nan,
            index=series.index
        )

    return (
        numeric_series
        - mean_value
    ) / sd_value


def residual_save_model_summary(
    result,
    output_path,
    heading
):
    """
    Save a statsmodels model summary to a text file.
    """
    output_path = Path(
        output_path
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as output_file:

        output_file.write(
            heading
        )

        output_file.write(
            "\n\n"
        )

        if result is None:

            output_file.write(
                "Model could not be fitted."
            )

        else:

            output_file.write(
                result.summary().as_text()
            )


# ============================================================
# 6. MIXED-MODEL HELPERS
# ============================================================

def residual_fit_subject_video_mixed_model(
    formula,
    dataframe,
    model_label=""
):
    """
    Fit a mixed-effects model with:

        random intercept for Subject
        random intercept for BaseVideo

    Maximum likelihood is used so nested likelihood-ratio
    comparisons can be performed.
    """

    model = smf.mixedlm(
        formula=formula,
        data=dataframe,
        groups=dataframe["Subject"],
        re_formula="1",
        vc_formula={
            "Video": "0 + C(BaseVideo)"
        }
    )

    fit_attempts = [
        (
            "lbfgs",
            MODEL_MAX_ITER
        ),
        (
            "bfgs",
            MODEL_MAX_ITER
        ),
        (
            "powell",
            MODEL_MAX_ITER
        )
    ]

    fit_errors = []

    for (
        optimizer,
        maximum_iterations
    ) in fit_attempts:

        try:

            result = model.fit(
                reml=False,
                method=optimizer,
                maxiter=maximum_iterations,
                disp=False
            )

            return (
                result,
                optimizer,
                None
            )

        except Exception as error:

            fit_errors.append(
                f"{optimizer}: {error}"
            )

    return (
        None,
        None,
        (
            f"{model_label} failed with all optimizers. "
            + " | ".join(
                fit_errors
            )
        )
    )


def residual_extract_fixed_effect(
    result,
    term
):
    """
    Extract one fixed-effect estimate.
    """

    empty_output = {
        "Beta": np.nan,
        "SE": np.nan,
        "Z": np.nan,
        "P": np.nan,
        "CI_Lower": np.nan,
        "CI_Upper": np.nan
    }

    if (
        result is None
        or term not in result.fe_params.index
    ):
        return empty_output

    confidence_intervals = (
        result.conf_int()
    )

    return {
        "Beta": float(
            result.fe_params[
                term
            ]
        ),

        "SE": float(
            result.bse_fe[
                term
            ]
        ),

        "Z": float(
            result.tvalues[
                term
            ]
        ),

        "P": float(
            result.pvalues[
                term
            ]
        ),

        "CI_Lower": float(
            confidence_intervals.loc[
                term,
                0
            ]
        ),

        "CI_Upper": float(
            confidence_intervals.loc[
                term,
                1
            ]
        )
    }


def residual_likelihood_ratio_test(
    reduced_result,
    full_result
):
    """
    Compare nested ML-fitted mixed models.
    """

    if (
        reduced_result is None
        or full_result is None
    ):

        return {
            "LR": np.nan,
            "DF": np.nan,
            "P": np.nan
        }

    lr_statistic = max(
        0.0,
        2.0
        * (
            full_result.llf
            - reduced_result.llf
        )
    )

    df_difference = max(
        1,
        len(
            full_result.fe_params
        )
        - len(
            reduced_result.fe_params
        )
    )

    p_value = chi2.sf(
        lr_statistic,
        df_difference
    )

    return {
        "LR": float(
            lr_statistic
        ),

        "DF": int(
            df_difference
        ),

        "P": float(
            p_value
        )
    }


def residual_mixed_model_r2(result):
    """
    Approximate Nakagawa-style marginal and conditional R².
    """

    if result is None:

        return {
            "FixedVariance": np.nan,
            "RandomVariance": np.nan,
            "ResidualVariance": np.nan,
            "MarginalR2": np.nan,
            "ConditionalR2": np.nan
        }

    try:

        fixed_predictions = np.dot(
            result.model.exog,
            np.asarray(
                result.fe_params
            )
        )

        fixed_variance = float(
            np.var(
                fixed_predictions,
                ddof=1
            )
        )

    except Exception:

        fixed_variance = np.nan

    random_variance = 0.0

    try:

        covariance_random = np.asarray(
            result.cov_re,
            dtype=float
        )

        if covariance_random.size > 0:

            random_variance += float(
                np.trace(
                    covariance_random
                )
            )

    except Exception:
        pass

    try:

        variance_components = np.asarray(
            result.vcomp,
            dtype=float
        )

        if variance_components.size > 0:

            random_variance += float(
                np.nansum(
                    variance_components
                )
            )

    except Exception:
        pass

    try:

        residual_variance = float(
            result.scale
        )

    except Exception:

        residual_variance = np.nan

    total_variance = (
        fixed_variance
        + random_variance
        + residual_variance
    )

    if (
        not np.isfinite(total_variance)
        or total_variance <= 0
    ):

        marginal_r2 = np.nan
        conditional_r2 = np.nan

    else:

        marginal_r2 = (
            fixed_variance
            / total_variance
        )

        conditional_r2 = (
            fixed_variance
            + random_variance
        ) / total_variance

    return {
        "FixedVariance": fixed_variance,
        "RandomVariance": random_variance,
        "ResidualVariance": residual_variance,
        "MarginalR2": marginal_r2,
        "ConditionalR2": conditional_r2
    }


def residual_extract_variances(result):
    """
    Extract subject, video, and residual variance estimates.
    """

    output = {
        "SubjectVariance": np.nan,
        "VideoVariance": np.nan,
        "ResidualVariance": np.nan,
        "NearZeroRandomVariance": False
    }

    if result is None:
        return output

    try:

        covariance_random = np.asarray(
            result.cov_re,
            dtype=float
        )

        if covariance_random.size > 0:

            output[
                "SubjectVariance"
            ] = float(
                covariance_random.ravel()[0]
            )

    except Exception:
        pass

    try:

        variance_components = np.asarray(
            result.vcomp,
            dtype=float
        )

        if variance_components.size > 0:

            output[
                "VideoVariance"
            ] = float(
                variance_components.ravel()[0]
            )

    except Exception:
        pass

    try:

        output[
            "ResidualVariance"
        ] = float(
            result.scale
        )

    except Exception:
        pass

    random_variance_values = [
        output[
            "SubjectVariance"
        ],
        output[
            "VideoVariance"
        ]
    ]

    output[
        "NearZeroRandomVariance"
    ] = any(
        np.isfinite(value)
        and value
        < RANDOM_VARIANCE_WARNING_THRESHOLD
        for value in random_variance_values
    )

    return output


# ============================================================
# 7. VERIFY REQUIRED COLUMNS
# ============================================================

required_base_columns = {
    "Subject",
    "BaseVideo",
    "Modality",
    PERFORMANCE_COLUMN,
    *MOTOR_PC_COLUMNS
}

missing_base_columns = (
    required_base_columns
    - set(
        stage3_base_df.columns
    )
)

if missing_base_columns:

    raise ValueError(
        "stage3_base_df is missing required columns:\n"
        f"{sorted(missing_base_columns)}"
    )


available_neural_predictors = [
    predictor
    for predictor in NEURAL_PREDICTORS
    if predictor in stage3_base_df.columns
]

missing_neural_predictors = [
    predictor
    for predictor in NEURAL_PREDICTORS
    if predictor not in stage3_base_df.columns
]

if missing_neural_predictors:

    print(
        "\nThe following network-balance variables were not "
        "available and will be skipped:"
    )

    for predictor in missing_neural_predictors:

        print(
            f"  {predictor}"
        )

if not available_neural_predictors:

    raise ValueError(
        "Neither requested network-balance variable was found "
        "in stage3_base_df."
    )


if (
    LEFT_MOTOR_ROI_COLUMN
    not in stage3_base_df.columns
):

    print(
        "\nWARNING: Left_Motor_ROI_ISC_z was not found in "
        "stage3_base_df."
    )

    print(
        "The Primary_MotorPCs analysis can still run, but the "
        "LeftMotorROI_Sensitivity analysis will be skipped."
    )

    MODEL_SUITES = {
        "Primary_MotorPCs": (
            MODEL_SUITES[
                "Primary_MotorPCs"
            ]
        )
    }


# ============================================================
# 8. NUMERIC CONVERSION
# ============================================================

numeric_columns = [
    PERFORMANCE_COLUMN,
    *MOTOR_PC_COLUMNS
]

if (
    LEFT_MOTOR_ROI_COLUMN
    in stage3_base_df.columns
):
    numeric_columns.append(
        LEFT_MOTOR_ROI_COLUMN
    )

numeric_columns.extend(
    available_neural_predictors
)

residual_base_df = (
    stage3_base_df.copy()
)

for column in numeric_columns:

    residual_base_df[
        column
    ] = pd.to_numeric(
        residual_base_df[
            column
        ],
        errors="coerce"
    )


# ============================================================
# 9. RESULT CONTAINERS
# ============================================================

residual_beta_rows = []

motor_model_fit_rows = []

residual_model_fit_rows = []

residual_lrt_rows = []

residual_data_diagnostic_rows = []

residual_model_failure_rows = []

residualized_dataset_list = []

residual_model_objects = {}


# ============================================================
# 10. FIT MOTOR MODEL AND RESIDUAL NEURAL MODEL
# ============================================================

for modality in ANALYSIS_GROUPS:

    modality_df = (
        residual_base_df[
            residual_base_df[
                "Modality"
            ] == modality
        ]
        .copy()
    )

    for (
        suite_name,
        suite_settings
    ) in MODEL_SUITES.items():

        motor_covariates = (
            suite_settings[
                "MotorCovariates"
            ]
        )

        for neural_predictor in (
            available_neural_predictors
        ):

            print(
                "\n" + "=" * 100
            )

            print(
                f"Residual beta: {modality} | "
                f"{suite_name} | {neural_predictor}"
            )

            required_columns = [
                "Subject",
                "BaseVideo",
                "Modality",
                PERFORMANCE_COLUMN,
                *motor_covariates,
                neural_predictor
            ]

            # ------------------------------------------------
            # Use exactly the same complete-case sample for:
            #
            #   motor-only residualization
            #   residual null model
            #   residual network model
            # ------------------------------------------------

            analysis_df = (
                modality_df[
                    required_columns
                ]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .dropna()
                .copy()
                .reset_index(
                    drop=True
                )
            )

            number_rows = len(
                analysis_df
            )

            number_subjects = (
                analysis_df[
                    "Subject"
                ].nunique()
            )

            number_videos = (
                analysis_df[
                    "BaseVideo"
                ].nunique()
            )

            residual_data_diagnostic_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Suite_Description": (
                    suite_settings[
                        "Description"
                    ]
                ),
                "Neural_Predictor": neural_predictor,
                "N_Rows": number_rows,
                "N_Subjects": number_subjects,
                "N_Videos": number_videos,
                "Performance_Mean": analysis_df[
                    PERFORMANCE_COLUMN
                ].mean(),
                "Performance_SD": analysis_df[
                    PERFORMANCE_COLUMN
                ].std(
                    ddof=1
                ),
                "Network_Mean": analysis_df[
                    neural_predictor
                ].mean(),
                "Network_SD": analysis_df[
                    neural_predictor
                ].std(
                    ddof=1
                )
            })

            if (
                number_rows < MIN_COMPLETE_ROWS
                or number_subjects
                < MIN_UNIQUE_SUBJECTS
                or number_videos
                < MIN_UNIQUE_VIDEOS
            ):

                residual_model_failure_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Model": "Complete-case sample",
                    "Error": (
                        "Insufficient complete data: "
                        f"N={number_rows}, "
                        f"subjects={number_subjects}, "
                        f"videos={number_videos}"
                    )
                })

                print(
                    "Skipped because the complete-case sample "
                    "was too small."
                )

                continue

            # ------------------------------------------------
            # Standardize predictors in the common sample.
            #
            # Performance remains in Pearson-r units.
            # ------------------------------------------------

            standardized_motor_terms = []

            for motor_index, motor_covariate in enumerate(
                motor_covariates,
                start=1
            ):

                standardized_term = (
                    f"MotorCovariate"
                    f"{motor_index}_z"
                )

                analysis_df[
                    standardized_term
                ] = residual_zscore_series(
                    analysis_df[
                        motor_covariate
                    ]
                )

                standardized_motor_terms.append(
                    standardized_term
                )

            analysis_df[
                "NetworkBalance_z"
            ] = residual_zscore_series(
                analysis_df[
                    neural_predictor
                ]
            )

            analysis_df = analysis_df.dropna(
                subset=(
                    standardized_motor_terms
                    + [
                        "NetworkBalance_z"
                    ]
                )
            ).reset_index(
                drop=True
            )

            analysis_df[
                "Performance"
            ] = analysis_df[
                PERFORMANCE_COLUMN
            ]

            # ------------------------------------------------
            # Step 1: Fit motor-only mixed model
            # ------------------------------------------------

            motor_formula = (
                "Performance ~ "
                + " + ".join(
                    standardized_motor_terms
                )
            )

            (
                motor_result,
                motor_optimizer,
                motor_error
            ) = residual_fit_subject_video_mixed_model(
                formula=motor_formula,
                dataframe=analysis_df,
                model_label=(
                    f"{modality} | {suite_name} | "
                    f"{neural_predictor} | Motor-only"
                )
            )

            if motor_error is not None:

                residual_model_failure_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Model": "Motor-only residualization model",
                    "Error": motor_error
                })

                print(
                    "Motor-only model failed. "
                    "Residual analysis skipped."
                )

                continue

            # ------------------------------------------------
            # Step 2: Extract conditional mixed-model residuals
            #
            # result.resid removes:
            #   fixed motor effects
            #   estimated subject random intercepts
            #   estimated video random intercepts
            #
            # These residuals represent within-model performance
            # variation remaining after measured motor effects and
            # modeled subject/video heterogeneity.
            # ------------------------------------------------

            try:

                motor_residuals = np.asarray(
                    motor_result.resid,
                    dtype=float
                )

            except Exception:

                motor_residuals = (
                    analysis_df[
                        "Performance"
                    ].to_numpy(
                        dtype=float
                    )
                    -
                    np.asarray(
                        motor_result.fittedvalues,
                        dtype=float
                    )
                )

            analysis_df[
                "Motor_Adjusted_Performance_Residual"
            ] = motor_residuals

            # Standardized residual retained for descriptive plots.
            analysis_df[
                "Motor_Adjusted_Performance_Residual_z"
            ] = residual_zscore_series(
                analysis_df[
                    "Motor_Adjusted_Performance_Residual"
                ]
            )

            # ------------------------------------------------
            # Step 3A: Residual baseline model
            #
            # Keeps the same random-effects structure.
            # ------------------------------------------------

            residual_null_formula = (
                "Motor_Adjusted_Performance_Residual ~ 1"
            )

            (
                residual_null_result,
                residual_null_optimizer,
                residual_null_error
            ) = residual_fit_subject_video_mixed_model(
                formula=residual_null_formula,
                dataframe=analysis_df,
                model_label=(
                    f"{modality} | {suite_name} | "
                    f"{neural_predictor} | Residual null"
                )
            )

            if residual_null_error is not None:

                residual_model_failure_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Model": "Residual null model",
                    "Error": residual_null_error
                })

            # ------------------------------------------------
            # Step 3B: Residual neural-beta model
            # ------------------------------------------------

            residual_network_formula = (
                "Motor_Adjusted_Performance_Residual "
                "~ NetworkBalance_z"
            )

            (
                residual_network_result,
                residual_network_optimizer,
                residual_network_error
            ) = residual_fit_subject_video_mixed_model(
                formula=residual_network_formula,
                dataframe=analysis_df,
                model_label=(
                    f"{modality} | {suite_name} | "
                    f"{neural_predictor} | Residual network"
                )
            )

            if residual_network_error is not None:

                residual_model_failure_rows.append({
                    "Modality": modality,
                    "Model_Suite": suite_name,
                    "Neural_Predictor": neural_predictor,
                    "Model": "Residual network model",
                    "Error": residual_network_error
                })

                continue

            residual_model_objects[
                (
                    modality,
                    suite_name,
                    neural_predictor
                )
            ] = {
                "Data": analysis_df.copy(),
                "MotorModel": motor_result,
                "ResidualNullModel": (
                    residual_null_result
                ),
                "ResidualNetworkModel": (
                    residual_network_result
                )
            }

            # ------------------------------------------------
            # Save residualized subject × video data
            # ------------------------------------------------

            residual_output_df = (
                analysis_df.copy()
            )

            residual_output_df[
                "Analysis_Modality"
            ] = modality

            residual_output_df[
                "Model_Suite"
            ] = suite_name

            residual_output_df[
                "Neural_Predictor_Name"
            ] = neural_predictor

            residualized_dataset_list.append(
                residual_output_df
            )

            residual_file_prefix = (
                f"{residual_safe_filename(modality)}__"
                f"{residual_safe_filename(suite_name)}__"
                f"{residual_safe_filename(neural_predictor)}"
            )

            residual_output_df.to_csv(
                RESIDUAL_BETA_DATA_DIR
                / (
                    f"{residual_file_prefix}"
                    "__motor_adjusted_performance.csv"
                ),
                index=False
            )

            # ------------------------------------------------
            # Save full model summaries
            # ------------------------------------------------

            residual_save_model_summary(
                motor_result,
                RESIDUAL_BETA_MODEL_DIR
                / (
                    f"{residual_file_prefix}"
                    "__motor_only_model.txt"
                ),
                (
                    f"Modality: {modality}\n"
                    f"Model suite: {suite_name}\n"
                    f"Network predictor defining common sample: "
                    f"{neural_predictor}\n"
                    f"Model: Motor-only residualization\n"
                    f"Formula: {motor_formula}\n"
                    f"N rows: {len(analysis_df)}\n"
                    f"N subjects: "
                    f"{analysis_df['Subject'].nunique()}\n"
                    f"N videos: "
                    f"{analysis_df['BaseVideo'].nunique()}"
                )
            )

            residual_save_model_summary(
                residual_null_result,
                RESIDUAL_BETA_MODEL_DIR
                / (
                    f"{residual_file_prefix}"
                    "__residual_null_model.txt"
                ),
                (
                    f"Modality: {modality}\n"
                    f"Model suite: {suite_name}\n"
                    f"Network predictor: {neural_predictor}\n"
                    f"Model: Residual null model\n"
                    f"Formula: {residual_null_formula}"
                )
            )

            residual_save_model_summary(
                residual_network_result,
                RESIDUAL_BETA_MODEL_DIR
                / (
                    f"{residual_file_prefix}"
                    "__residual_network_model.txt"
                ),
                (
                    f"Modality: {modality}\n"
                    f"Model suite: {suite_name}\n"
                    f"Network predictor: {neural_predictor}\n"
                    f"Model: Network prediction of motor-adjusted "
                    f"performance\n"
                    f"Formula: {residual_network_formula}"
                )
            )

            # ------------------------------------------------
            # Extract residual network beta
            # ------------------------------------------------

            residual_beta = (
                residual_extract_fixed_effect(
                    residual_network_result,
                    "NetworkBalance_z"
                )
            )

            # Obtain Model 3 beta from the prior analysis when
            # available, allowing direct comparison.
            model3_beta = np.nan
            model3_p = np.nan

            if (
                "stage3_neural_beta_comparison_df"
                in globals()
            ):

                prior_match = (
                    stage3_neural_beta_comparison_df[
                        (
                            stage3_neural_beta_comparison_df[
                                "Modality"
                            ] == modality
                        )
                        &
                        (
                            stage3_neural_beta_comparison_df[
                                "Model_Suite"
                            ] == suite_name
                        )
                        &
                        (
                            stage3_neural_beta_comparison_df[
                                "Neural_Predictor"
                            ] == neural_predictor
                        )
                    ]
                )

                if not prior_match.empty:

                    model3_beta = (
                        prior_match[
                            "Model3_Beta"
                        ].iloc[0]
                    )

                    model3_p = (
                        prior_match[
                            "Model3_P"
                        ].iloc[0]
                    )

            residual_beta_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Suite_Description": (
                    suite_settings[
                        "Description"
                    ]
                ),
                "Neural_Predictor": neural_predictor,

                "N_Rows": len(
                    analysis_df
                ),

                "N_Subjects": analysis_df[
                    "Subject"
                ].nunique(),

                "N_Videos": analysis_df[
                    "BaseVideo"
                ].nunique(),

                "Residual_Beta": residual_beta[
                    "Beta"
                ],

                "Residual_SE": residual_beta[
                    "SE"
                ],

                "Residual_Z": residual_beta[
                    "Z"
                ],

                "Residual_P": residual_beta[
                    "P"
                ],

                "Residual_CI_Lower": residual_beta[
                    "CI_Lower"
                ],

                "Residual_CI_Upper": residual_beta[
                    "CI_Upper"
                ],

                "Combined_Model3_Beta": (
                    model3_beta
                ),

                "Combined_Model3_P": (
                    model3_p
                ),

                "Residual_Beta_Direction_Matches_Model3": (
                    np.sign(
                        residual_beta[
                            "Beta"
                        ]
                    )
                    ==
                    np.sign(
                        model3_beta
                    )
                    if (
                        np.isfinite(
                            residual_beta[
                                "Beta"
                            ]
                        )
                        and np.isfinite(
                            model3_beta
                        )
                    )
                    else np.nan
                ),

                "Motor_Residual_Mean": (
                    analysis_df[
                        "Motor_Adjusted_Performance_Residual"
                    ].mean()
                ),

                "Motor_Residual_SD": (
                    analysis_df[
                        "Motor_Adjusted_Performance_Residual"
                    ].std(
                        ddof=1
                    )
                )
            })

            # ------------------------------------------------
            # Model-fit statistics
            # ------------------------------------------------

            motor_r2 = residual_mixed_model_r2(
                motor_result
            )

            motor_variances = residual_extract_variances(
                motor_result
            )

            motor_model_fit_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Neural_Predictor": neural_predictor,
                "Model": "Motor_Only",
                "Formula": motor_formula,
                "Optimizer": motor_optimizer,
                "Converged": bool(
                    motor_result.converged
                ),
                "LogLikelihood": float(
                    motor_result.llf
                ),
                "AIC": float(
                    motor_result.aic
                ),
                "BIC": float(
                    motor_result.bic
                ),
                **motor_r2,
                **motor_variances
            })

            residual_r2 = residual_mixed_model_r2(
                residual_network_result
            )

            residual_variances = residual_extract_variances(
                residual_network_result
            )

            residual_model_fit_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Neural_Predictor": neural_predictor,
                "Model": "Residual_Network",
                "Formula": residual_network_formula,
                "Optimizer": (
                    residual_network_optimizer
                ),
                "Converged": bool(
                    residual_network_result.converged
                ),
                "LogLikelihood": float(
                    residual_network_result.llf
                ),
                "AIC": float(
                    residual_network_result.aic
                ),
                "BIC": float(
                    residual_network_result.bic
                ),
                **residual_r2,
                **residual_variances
            })

            # ------------------------------------------------
            # Compare residual null vs residual network model
            # ------------------------------------------------

            residual_lrt = residual_likelihood_ratio_test(
                reduced_result=(
                    residual_null_result
                ),
                full_result=(
                    residual_network_result
                )
            )

            residual_lrt_rows.append({
                "Modality": modality,
                "Model_Suite": suite_name,
                "Neural_Predictor": neural_predictor,

                "Comparison": (
                    "Residual_Null_vs_Residual_Network"
                ),

                "LR": residual_lrt[
                    "LR"
                ],

                "DF": residual_lrt[
                    "DF"
                ],

                "P": residual_lrt[
                    "P"
                ],

                "Residual_Null_AIC": (
                    residual_null_result.aic
                    if residual_null_result
                    is not None
                    else np.nan
                ),

                "Residual_Network_AIC": (
                    residual_network_result.aic
                ),

                "Delta_AIC_NetworkMinusNull": (
                    residual_network_result.aic
                    - residual_null_result.aic
                    if residual_null_result
                    is not None
                    else np.nan
                ),

                "Residual_Null_BIC": (
                    residual_null_result.bic
                    if residual_null_result
                    is not None
                    else np.nan
                ),

                "Residual_Network_BIC": (
                    residual_network_result.bic
                ),

                "Delta_BIC_NetworkMinusNull": (
                    residual_network_result.bic
                    - residual_null_result.bic
                    if residual_null_result
                    is not None
                    else np.nan
                ),

                "Residual_Null_Marginal_R2": (
                    residual_mixed_model_r2(
                        residual_null_result
                    )[
                        "MarginalR2"
                    ]
                    if residual_null_result
                    is not None
                    else np.nan
                ),

                "Residual_Network_Marginal_R2": (
                    residual_r2[
                        "MarginalR2"
                    ]
                )
            })

            # ------------------------------------------------
            # Residual scatterplot
            # ------------------------------------------------

            plot_df = analysis_df[
                [
                    "NetworkBalance_z",
                    "Motor_Adjusted_Performance_Residual"
                ]
            ].dropna()

            fig, ax = plt.subplots(
                figsize=(7, 6)
            )

            ax.scatter(
                plot_df[
                    "NetworkBalance_z"
                ],
                plot_df[
                    "Motor_Adjusted_Performance_Residual"
                ],
                alpha=0.30,
                s=26
            )

            if (
                plot_df[
                    "NetworkBalance_z"
                ].std(
                    ddof=1
                ) > 0
            ):

                slope, intercept = np.polyfit(
                    plot_df[
                        "NetworkBalance_z"
                    ],
                    plot_df[
                        "Motor_Adjusted_Performance_Residual"
                    ],
                    deg=1
                )

                x_line = np.linspace(
                    plot_df[
                        "NetworkBalance_z"
                    ].min(),
                    plot_df[
                        "NetworkBalance_z"
                    ].max(),
                    200
                )

                ax.plot(
                    x_line,
                    intercept
                    + slope * x_line,
                    linewidth=2
                )

            ax.axhline(
                0,
                linestyle="--",
                linewidth=1
            )

            p_text = (
                f"{residual_beta['P']:.4f}"
                if np.isfinite(
                    residual_beta[
                        "P"
                    ]
                )
                else "NA"
            )

            ax.set_xlabel(
                f"Standardized {neural_predictor}"
            )

            ax.set_ylabel(
                "Behavioral performance residual\n"
                "after motor adjustment"
            )

            ax.set_title(
                f"{modality} | {suite_name}\n"
                f"{neural_predictor}\n"
                f"Residual β = "
                f"{residual_beta['Beta']:.4f}, "
                f"p = {p_text}"
            )

            fig.tight_layout()

            fig.savefig(
                RESIDUAL_BETA_FIG_DIR
                / (
                    f"{residual_file_prefix}"
                    "__residual_beta_scatter.png"
                ),
                dpi=FIGURE_DPI,
                bbox_inches="tight"
            )

            plt.show()
            plt.close(fig)


# ============================================================
# 11. CREATE RESULT DATAFRAMES
# ============================================================

residual_neural_beta_df = pd.DataFrame(
    residual_beta_rows
)

residual_motor_model_fit_df = pd.DataFrame(
    motor_model_fit_rows
)

residual_network_model_fit_df = pd.DataFrame(
    residual_model_fit_rows
)

residual_nested_comparison_df = pd.DataFrame(
    residual_lrt_rows
)

residual_data_diagnostics_df = pd.DataFrame(
    residual_data_diagnostic_rows
)

residual_model_failures_df = pd.DataFrame(
    residual_model_failure_rows
)


if residualized_dataset_list:

    residualized_stage3_data_df = pd.concat(
        residualized_dataset_list,
        ignore_index=True,
        sort=False
    )

else:

    residualized_stage3_data_df = pd.DataFrame()


# ============================================================
# 12. MULTIPLE-COMPARISON CORRECTION
# ============================================================

if not residual_neural_beta_df.empty:

    residual_neural_beta_df[
        "Residual_P_FDR"
    ] = np.nan

    for modality in ANALYSIS_GROUPS:

        for suite_name in MODEL_SUITES:

            selection = (
                (
                    residual_neural_beta_df[
                        "Modality"
                    ] == modality
                )
                &
                (
                    residual_neural_beta_df[
                        "Model_Suite"
                    ] == suite_name
                )
                &
                residual_neural_beta_df[
                    "Residual_P"
                ].notna()
            )

            if selection.sum() > 0:

                residual_neural_beta_df.loc[
                    selection,
                    "Residual_P_FDR"
                ] = multipletests(
                    residual_neural_beta_df.loc[
                        selection,
                        "Residual_P"
                    ],
                    method="fdr_bh"
                )[1]


if not residual_nested_comparison_df.empty:

    residual_nested_comparison_df[
        "P_FDR"
    ] = np.nan

    for modality in ANALYSIS_GROUPS:

        for suite_name in MODEL_SUITES:

            selection = (
                (
                    residual_nested_comparison_df[
                        "Modality"
                    ] == modality
                )
                &
                (
                    residual_nested_comparison_df[
                        "Model_Suite"
                    ] == suite_name
                )
                &
                residual_nested_comparison_df[
                    "P"
                ].notna()
            )

            if selection.sum() > 0:

                residual_nested_comparison_df.loc[
                    selection,
                    "P_FDR"
                ] = multipletests(
                    residual_nested_comparison_df.loc[
                        selection,
                        "P"
                    ],
                    method="fdr_bh"
                )[1]


# ============================================================
# 13. SAVE TABLE OUTPUTS
# ============================================================

residual_neural_beta_df.to_csv(
    RESIDUAL_BETA_TABLE_DIR
    / "residual_network_beta_tests.csv",
    index=False
)

residual_motor_model_fit_df.to_csv(
    RESIDUAL_BETA_TABLE_DIR
    / "residualization_motor_model_fit.csv",
    index=False
)

residual_network_model_fit_df.to_csv(
    RESIDUAL_BETA_TABLE_DIR
    / "residual_network_model_fit.csv",
    index=False
)

residual_nested_comparison_df.to_csv(
    RESIDUAL_BETA_TABLE_DIR
    / "residual_null_vs_network_model_comparisons.csv",
    index=False
)

residual_data_diagnostics_df.to_csv(
    RESIDUAL_BETA_TABLE_DIR
    / "residual_beta_complete_case_diagnostics.csv",
    index=False
)

residual_model_failures_df.to_csv(
    RESIDUAL_BETA_TABLE_DIR
    / "residual_beta_model_failures.csv",
    index=False
)

residualized_stage3_data_df.to_csv(
    RESIDUAL_BETA_DATA_DIR
    / "all_motor_adjusted_performance_data.csv",
    index=False
)


# ============================================================
# 14. FIGURE — RESIDUAL NETWORK BETAS
# ============================================================

if not residual_neural_beta_df.empty:

    for modality in ANALYSIS_GROUPS:

        plot_df = (
            residual_neural_beta_df[
                residual_neural_beta_df[
                    "Modality"
                ] == modality
            ]
            .copy()
        )

        if plot_df.empty:
            continue

        plot_df[
            "Label"
        ] = (
            plot_df[
                "Model_Suite"
            ]
            + "\n"
            + plot_df[
                "Neural_Predictor"
            ]
        )

        x_positions = np.arange(
            len(plot_df)
        )

        beta_values = plot_df[
            "Residual_Beta"
        ].to_numpy(
            dtype=float
        )

        lower_errors = (
            beta_values
            - plot_df[
                "Residual_CI_Lower"
            ].to_numpy(
                dtype=float
            )
        )

        upper_errors = (
            plot_df[
                "Residual_CI_Upper"
            ].to_numpy(
                dtype=float
            )
            - beta_values
        )

        fig, ax = plt.subplots(
            figsize=(
                max(
                    10,
                    2.4
                    * len(plot_df)
                ),
                6
            )
        )

        ax.errorbar(
            x_positions,
            beta_values,
            yerr=np.vstack(
                [
                    lower_errors,
                    upper_errors
                ]
            ),
            fmt="o",
            capsize=5
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1
        )

        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(
            plot_df[
                "Label"
            ],
            rotation=25,
            ha="right"
        )

        ax.set_ylabel(
            "Residual network beta\n"
            "predicting motor-adjusted performance"
        )

        ax.set_title(
            f"{modality}: network prediction after "
            "motor residualization"
        )

        fig.tight_layout()

        fig.savefig(
            RESIDUAL_BETA_FIG_DIR
            / (
                f"{residual_safe_filename(modality)}"
                "__residual_network_betas.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 15. FIGURE — COMBINED-MODEL VS RESIDUAL BETA
# ============================================================

if (
    not residual_neural_beta_df.empty
    and residual_neural_beta_df[
        "Combined_Model3_Beta"
    ].notna().any()
):

    comparison_plot_df = (
        residual_neural_beta_df[
            residual_neural_beta_df[
                "Combined_Model3_Beta"
            ].notna()
        ]
        .copy()
    )

    comparison_plot_df[
        "Label"
    ] = (
        comparison_plot_df[
            "Modality"
        ]
        + "\n"
        + comparison_plot_df[
            "Model_Suite"
        ]
        + "\n"
        + comparison_plot_df[
            "Neural_Predictor"
        ]
    )

    x_positions = np.arange(
        len(
            comparison_plot_df
        )
    )

    width = 0.36

    fig, ax = plt.subplots(
        figsize=(
            max(
                12,
                2.0
                * len(
                    comparison_plot_df
                )
            ),
            7
        )
    )

    ax.bar(
        x_positions
        - width / 2,
        comparison_plot_df[
            "Combined_Model3_Beta"
        ],
        width=width,
        label="Combined Model 3 beta"
    )

    ax.bar(
        x_positions
        + width / 2,
        comparison_plot_df[
            "Residual_Beta"
        ],
        width=width,
        label="Residualized-performance beta"
    )

    ax.axhline(
        0,
        linewidth=1
    )

    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        comparison_plot_df[
            "Label"
        ],
        rotation=35,
        ha="right"
    )

    ax.set_ylabel(
        "Network-balance coefficient"
    )

    ax.set_title(
        "Comparison of simultaneous motor adjustment and "
        "sequential residualization"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    fig.savefig(
        RESIDUAL_BETA_FIG_DIR
        / "combined_model_vs_residual_network_beta.png",
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 16. PRINT RESULTS
# ============================================================

print(
    "\n" + "=" * 105
)

print(
    "RESIDUAL NEURAL-BETA ANALYSIS COMPLETE"
)

print(
    "=" * 105
)


print(
    "\nComplete-case diagnostics:"
)

display(
    residual_data_diagnostics_df.round(
        5
    )
)


print(
    "\nResidual network-beta tests:"
)

if residual_neural_beta_df.empty:

    print(
        "No residual network-beta results were available."
    )

else:

    residual_display_columns = [
        "Modality",
        "Model_Suite",
        "Neural_Predictor",
        "N_Rows",
        "N_Subjects",
        "N_Videos",

        "Residual_Beta",
        "Residual_SE",
        "Residual_Z",
        "Residual_P",
        "Residual_P_FDR",
        "Residual_CI_Lower",
        "Residual_CI_Upper",

        "Combined_Model3_Beta",
        "Combined_Model3_P",
        "Residual_Beta_Direction_Matches_Model3",

        "Motor_Residual_Mean",
        "Motor_Residual_SD"
    ]

    residual_display_columns = [
        column
        for column in residual_display_columns
        if column
        in residual_neural_beta_df.columns
    ]

    display(
        residual_neural_beta_df[
            residual_display_columns
        ]
        .sort_values(
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor"
            ]
        )
        .round(5)
    )


print(
    "\nResidual null-versus-network model comparisons:"
)

if residual_nested_comparison_df.empty:

    print(
        "No residual nested-model comparisons were available."
    )

else:

    display(
        residual_nested_comparison_df[
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor",
                "LR",
                "DF",
                "P",
                "P_FDR",
                "Delta_AIC_NetworkMinusNull",
                "Delta_BIC_NetworkMinusNull",
                "Residual_Null_Marginal_R2",
                "Residual_Network_Marginal_R2"
            ]
        ]
        .sort_values(
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor"
            ]
        )
        .round(5)
    )


print(
    "\nMotor-only model fit and residualization diagnostics:"
)

if residual_motor_model_fit_df.empty:

    print(
        "No motor-only model-fit results were available."
    )

else:

    display(
        residual_motor_model_fit_df[
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor",
                "Optimizer",
                "Converged",
                "AIC",
                "BIC",
                "MarginalR2",
                "ConditionalR2",
                "SubjectVariance",
                "VideoVariance",
                "ResidualVariance",
                "NearZeroRandomVariance"
            ]
        ]
        .sort_values(
            [
                "Modality",
                "Model_Suite",
                "Neural_Predictor"
            ]
        )
        .round(6)
    )


if not residual_model_failures_df.empty:

    print(
        "\nModel-fitting failures or warnings:"
    )

    display(
        residual_model_failures_df
    )


print(
    "\nInterpretation guide:"
)

print(
    "  • Residual_Beta tests whether network balance predicts "
    "behavioral performance remaining after the motor-only "
    "mixed model."
)

print(
    "  • A positive and significant Residual_Beta means that "
    "higher network balance predicts stronger PC1 behavioral "
    "alignment after measured motor effects are removed."
)

print(
    "  • Residual_Null_vs_Residual_Network tests whether adding "
    "network balance improves fit to the motor-adjusted residuals."
)

print(
    "  • Negative Delta_AIC or Delta_BIC values favor the "
    "residual network model over the residual null model."
)

print(
    "  • Compare Residual_Beta with Combined_Model3_Beta. "
    "Similar directions and magnitudes support the robustness "
    "of the simultaneous mixed-model adjustment."
)

print(
    "  • The combined Model 3 analysis should remain the primary "
    "test. Sequential residualization is a complementary "
    "sensitivity analysis because the residuals are estimated "
    "rather than directly observed."
)

print(
    "\nAll residual neural-beta outputs were saved to:\n"
    f"{RESIDUAL_BETA_OUTPUT_DIR}"
)

In [ ]:
# ============================================================
# 
# RECONSTRUCT SUBJECT-LEVEL MOTOR CONTROLS
#
# Creates one row per participant containing:
#   Subject
#   Performance
#   Original_Network_Balance
#   Full_LASSO_Network_Balance
#   MotorPC1
#   MotorPC2
#   MotorPC3
#   Left_Motor_ROI_ISC_z
#
# IMPORTANT:
#   - Uses left motor-control ROI ISC only.
#   - Does not calculate or use motor lateralization.
#   - Does not use right or bilateral motor ROI ISC.
#
# REQUIRED NOTEBOOK OBJECTS:
#   analysis_df
#   motor_neural_pc_df
#
# AND EITHER:
#   motor_control_isc_wide_df
# OR:
#   motor_video_roi_isc_df
#
# OUTPUT:
#   subject_level_motor_control_df
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. OUTPUT DIRECTORY
# ============================================================

SUBJECT_MOTOR_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\04_subject_level_motor_control"
)

SUBJECT_MOTOR_DATA_DIR = SUBJECT_MOTOR_OUTPUT_DIR / "data"
SUBJECT_MOTOR_FIG_DIR = SUBJECT_MOTOR_OUTPUT_DIR / "figures"
SUBJECT_MOTOR_TABLE_DIR = SUBJECT_MOTOR_OUTPUT_DIR / "tables"

for folder in [
    SUBJECT_MOTOR_OUTPUT_DIR,
    SUBJECT_MOTOR_DATA_DIR,
    SUBJECT_MOTOR_FIG_DIR,
    SUBJECT_MOTOR_TABLE_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(
    "Subject-level motor-control outputs will be saved to:\n"
    f"{SUBJECT_MOTOR_OUTPUT_DIR}"
)


# ============================================================
# 2. CHECK REQUIRED OBJECTS
# ============================================================

required_objects = [
    "analysis_df",
    "motor_neural_pc_df"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "The following required notebook objects were not found:\n"
        f"{missing_objects}\n\n"
        "Run the original subject-level ROI/performance analysis "
        "and the motor-feature PCA/merge analysis first."
    )

if (
    "motor_control_isc_wide_df" not in globals()
    and "motor_video_roi_isc_df" not in globals()
):
    raise ValueError(
        "Neither motor_control_isc_wide_df nor "
        "motor_video_roi_isc_df was found.\n\n"
        "Run the motor-control ROI ISC pipeline first."
    )


# ============================================================
# 3. SETTINGS
# ============================================================

MOTOR_PC_COLUMNS = [
    "MotorPC1",
    "MotorPC2",
    "MotorPC3"
]

LEFT_MOTOR_SUBJECT_COLUMN = "Left_Motor_ROI_ISC_z"

ORIGINAL_POSITIVE_ROIS = [
    "Right VLPFC",
    "Left TPJ"
]

ORIGINAL_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Right TPJ"
]

FULL_LASSO_POSITIVE_ROIS = [
    "Left TPJ",
    "Right VLPFC",
    "Left OFC",
    "Left STS"
]

FULL_LASSO_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Left DLPFC",
    "Right TPJ"
]


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """Normalize participant IDs for reliable merging."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_trigger(value):
    """Normalize trigger values such as 1, 1.0, and '1'."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(int(numeric_value))

    except (TypeError, ValueError):
        pass

    return value


def fisher_z_isc(values):
    """Fisher-z transform correlation values safely."""
    values = pd.to_numeric(values, errors="coerce")

    return np.arctanh(
        np.clip(
            values,
            -0.999999,
            0.999999
        )
    )


def first_existing_column(dataframe, candidates):
    """Return the first candidate column found."""
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    return None


# ============================================================
# 5. PREPARE ORIGINAL SUBJECT-LEVEL ROI + PERFORMANCE TABLE
#
# IMPORTANT:
#   ROI ISC columns come from roi_isc_df.
#   Performance comes from analysis_df.
# ============================================================

if "roi_isc_df" not in globals():
    raise ValueError(
        "roi_isc_df was not found.\n\n"
        "Run the original ROI-level ISC cell first. roi_isc_df "
        "should contain one row per subject and one column per ROI."
    )

if "analysis_df" not in globals():
    raise ValueError(
        "analysis_df was not found.\n\n"
        "Run the original subject-level behavioral/LASSO analysis first."
    )


# ------------------------------------------------------------
# 5A. Prepare subject-level ROI ISC table
# ------------------------------------------------------------

subject_roi_df = roi_isc_df.copy()

if "Subject" not in subject_roi_df.columns:
    raise ValueError(
        "roi_isc_df does not contain a Subject column.\n"
        f"Available columns:\n{list(subject_roi_df.columns)}"
    )

subject_roi_df["Subject"] = (
    subject_roi_df["Subject"]
    .apply(normalize_subject_id)
)

# Convert every ROI column to numeric.
subject_roi_columns = [
    column
    for column in subject_roi_df.columns
    if column != "Subject"
]

for column in subject_roi_columns:
    subject_roi_df[column] = pd.to_numeric(
        subject_roi_df[column],
        errors="coerce"
    )

# There should normally already be one row per participant.
# This aggregation safely handles accidental duplicate rows.
subject_roi_df = (
    subject_roi_df
    .groupby("Subject", as_index=False)[subject_roi_columns]
    .mean()
)


# ------------------------------------------------------------
# 5B. Prepare subject-level performance table
# ------------------------------------------------------------

performance_source_df = analysis_df.copy()

if "Subject" not in performance_source_df.columns:
    raise ValueError(
        "analysis_df does not contain a Subject column.\n"
        f"Available columns:\n{list(performance_source_df.columns)}"
    )

if "Performance" not in performance_source_df.columns:
    raise ValueError(
        "analysis_df does not contain a Performance column.\n"
        f"Available columns:\n{list(performance_source_df.columns)}"
    )

performance_source_df["Subject"] = (
    performance_source_df["Subject"]
    .apply(normalize_subject_id)
)

performance_source_df["Performance"] = pd.to_numeric(
    performance_source_df["Performance"],
    errors="coerce"
)

# Retain one performance value per subject.
# If duplicate subject rows exist, average them.
subject_performance_df = (
    performance_source_df[
        [
            "Subject",
            "Performance"
        ]
    ]
    .groupby("Subject", as_index=False)["Performance"]
    .mean()
)


# ------------------------------------------------------------
# 5C. Merge ROI ISC and performance
# ------------------------------------------------------------

subject_neural_df = pd.merge(
    subject_roi_df,
    subject_performance_df,
    on="Subject",
    how="inner",
    validate="one_to_one",
    indicator="PerformanceMerge"
)

if subject_neural_df.empty:
    raise ValueError(
        "No subjects matched between roi_isc_df and analysis_df.\n\n"
        "Check whether Subject IDs use the same formatting in both tables."
    )

print("\nSubject-level ROI/performance merge:")
print(
    f"  Subjects in roi_isc_df: "
    f"{subject_roi_df['Subject'].nunique()}"
)

print(
    f"  Subjects in analysis_df performance table: "
    f"{subject_performance_df['Subject'].nunique()}"
)

print(
    f"  Subjects matched: "
    f"{subject_neural_df['Subject'].nunique()}"
)


# ============================================================
# 6. RECONSTRUCT ORIGINAL NETWORK BALANCE
#
# Original NBI:
#
# mean(Right VLPFC, Left TPJ)
# minus
# mean(Left VLPFC, Right TPJ)
# ============================================================

ORIGINAL_POSITIVE_ROIS = [
    "Right VLPFC",
    "Left TPJ"
]

ORIGINAL_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Right TPJ"
]

original_required_rois = (
    ORIGINAL_POSITIVE_ROIS
    + ORIGINAL_NEGATIVE_ROIS
)

missing_original_rois = [
    roi
    for roi in original_required_rois
    if roi not in subject_neural_df.columns
]

if missing_original_rois:
    raise ValueError(
        "The following ROIs needed for Original Network Balance "
        "were not found in roi_isc_df:\n"
        f"{missing_original_rois}\n\n"
        "Available roi_isc_df columns are:\n"
        f"{list(subject_roi_df.columns)}"
    )

# skipna=False requires all four component ROIs to be present
# for a participant, matching the prior network-balance code.
subject_neural_df["Original_Network_Balance"] = (
    subject_neural_df[
        ORIGINAL_POSITIVE_ROIS
    ].mean(
        axis=1,
        skipna=False
    )
    -
    subject_neural_df[
        ORIGINAL_NEGATIVE_ROIS
    ].mean(
        axis=1,
        skipna=False
    )
)


# ============================================================
# 7. RECONSTRUCT FULL LASSO NETWORK BALANCE
#
# Full LASSO NBI:
#
# mean(
#   Left TPJ,
#   Right VLPFC,
#   Left OFC,
#   Left STS
# )
# minus
# mean(
#   Left VLPFC,
#   Left DLPFC,
#   Right TPJ
# )
# ============================================================

FULL_LASSO_POSITIVE_ROIS = [
    "Left TPJ",
    "Right VLPFC",
    "Left OFC",
    "Left STS"
]

FULL_LASSO_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Left DLPFC",
    "Right TPJ"
]

full_lasso_required_rois = (
    FULL_LASSO_POSITIVE_ROIS
    + FULL_LASSO_NEGATIVE_ROIS
)

missing_full_lasso_rois = [
    roi
    for roi in full_lasso_required_rois
    if roi not in subject_neural_df.columns
]

if missing_full_lasso_rois:
    print(
        "\nWARNING: Full_LASSO_Network_Balance could not be "
        "created because the following ROI columns were absent "
        "from roi_isc_df:"
    )

    print(missing_full_lasso_rois)

    subject_neural_df[
        "Full_LASSO_Network_Balance"
    ] = np.nan

else:
    subject_neural_df[
        "Full_LASSO_Network_Balance"
    ] = (
        subject_neural_df[
            FULL_LASSO_POSITIVE_ROIS
        ].mean(
            axis=1,
            skipna=False
        )
        -
        subject_neural_df[
            FULL_LASSO_NEGATIVE_ROIS
        ].mean(
            axis=1,
            skipna=False
        )
    )


# ============================================================
# 7B. NETWORK-BALANCE DIAGNOSTICS
# ============================================================

network_balance_diagnostics_df = pd.DataFrame(
    {
        "Variable": [
            "Performance",
            "Original_Network_Balance",
            "Full_LASSO_Network_Balance"
        ],

        "N_Valid": [
            subject_neural_df[
                "Performance"
            ].notna().sum(),

            subject_neural_df[
                "Original_Network_Balance"
            ].notna().sum(),

            subject_neural_df[
                "Full_LASSO_Network_Balance"
            ].notna().sum()
        ],

        "N_Missing": [
            subject_neural_df[
                "Performance"
            ].isna().sum(),

            subject_neural_df[
                "Original_Network_Balance"
            ].isna().sum(),

            subject_neural_df[
                "Full_LASSO_Network_Balance"
            ].isna().sum()
        ]
    }
)

print("\nNetwork-balance reconstruction diagnostics:")
display(network_balance_diagnostics_df)

print("\nSubject-level neural/performance table:")
display(
    subject_neural_df[
        [
            "Subject",
            "Performance",
            "Original_Network_Balance",
            "Full_LASSO_Network_Balance"
        ]
    ]
    .head(10)
    .round(4)
)

# ============================================================
# 8. PREPARE SUBJECT × VIDEO MOTOR-PC TABLE
# ============================================================

motor_video_df = motor_neural_pc_df.copy()

required_motor_columns = {
    "Subject",
    *MOTOR_PC_COLUMNS
}

missing_motor_columns = (
    required_motor_columns
    - set(motor_video_df.columns)
)

if missing_motor_columns:
    raise ValueError(
        "motor_neural_pc_df is missing required columns:\n"
        f"{sorted(missing_motor_columns)}"
    )

motor_video_df["Subject"] = (
    motor_video_df["Subject"]
    .apply(normalize_subject_id)
)

for column in MOTOR_PC_COLUMNS:
    motor_video_df[column] = pd.to_numeric(
        motor_video_df[column],
        errors="coerce"
    )

# Recover trigger when available.
if "Trigger" in motor_video_df.columns:
    motor_video_df["Trigger"] = (
        motor_video_df["Trigger"]
        .apply(normalize_trigger)
    )

# Remove duplicate subject × trigger observations where possible.
if "Trigger" in motor_video_df.columns:
    motor_video_df = (
        motor_video_df
        .groupby(
            ["Subject", "Trigger"],
            as_index=False
        )[MOTOR_PC_COLUMNS]
        .mean()
    )
else:
    print(
        "\nWARNING: motor_neural_pc_df does not contain Trigger. "
        "Motor PCs will be averaged across all available rows "
        "for each subject."
    )


# ============================================================
# 9. PREPARE LEFT MOTOR-CONTROL ROI ISC
# ============================================================

if "motor_control_isc_wide_df" in globals():

    left_motor_video_df = motor_control_isc_wide_df.copy()

    if "Subject" not in left_motor_video_df.columns:
        raise ValueError(
            "motor_control_isc_wide_df does not contain Subject."
        )

    if "Trigger" not in left_motor_video_df.columns:
        raise ValueError(
            "motor_control_isc_wide_df does not contain Trigger."
        )

    left_motor_video_df["Subject"] = (
        left_motor_video_df["Subject"]
        .apply(normalize_subject_id)
    )

    left_motor_video_df["Trigger"] = (
        left_motor_video_df["Trigger"]
        .apply(normalize_trigger)
    )

    left_z_column = first_existing_column(
        left_motor_video_df,
        [
            "Left_Motor_ROI_ISC_z",
            "Left_Motor_Control_ROI_ISC_z",
            "Left Motor Control ROI_z",
            "Left_Motor_ISC_z"
        ]
    )

    left_raw_column = first_existing_column(
        left_motor_video_df,
        [
            "Left_Motor_ROI_ISC",
            "Left_Motor_Control_ROI_ISC",
            "Left Motor Control ROI",
            "Left_Motor_ISC"
        ]
    )

    if left_z_column is not None:
        left_motor_video_df[LEFT_MOTOR_SUBJECT_COLUMN] = (
            pd.to_numeric(
                left_motor_video_df[left_z_column],
                errors="coerce"
            )
        )

    elif left_raw_column is not None:
        left_motor_video_df[LEFT_MOTOR_SUBJECT_COLUMN] = fisher_z_isc(
            left_motor_video_df[left_raw_column]
        )

    else:
        raise ValueError(
            "Could not locate a left motor ROI ISC column in "
            "motor_control_isc_wide_df.\n"
            f"Available columns:\n{list(left_motor_video_df.columns)}"
        )

    left_motor_video_df = (
        left_motor_video_df[
            [
                "Subject",
                "Trigger",
                LEFT_MOTOR_SUBJECT_COLUMN
            ]
        ]
        .groupby(
            ["Subject", "Trigger"],
            as_index=False
        )[LEFT_MOTOR_SUBJECT_COLUMN]
        .mean()
    )

else:

    motor_roi_long_df = motor_video_roi_isc_df.copy()

    required_long_columns = {
        "Subject",
        "Trigger",
        "ROI",
        "ISC"
    }

    missing_long_columns = (
        required_long_columns
        - set(motor_roi_long_df.columns)
    )

    if missing_long_columns:
        raise ValueError(
            "motor_video_roi_isc_df is missing columns:\n"
            f"{sorted(missing_long_columns)}"
        )

    motor_roi_long_df["Subject"] = (
        motor_roi_long_df["Subject"]
        .apply(normalize_subject_id)
    )

    motor_roi_long_df["Trigger"] = (
        motor_roi_long_df["Trigger"]
        .apply(normalize_trigger)
    )

    motor_roi_long_df["ROI"] = (
        motor_roi_long_df["ROI"]
        .astype(str)
        .str.strip()
    )

    motor_roi_long_df["ISC"] = pd.to_numeric(
        motor_roi_long_df["ISC"],
        errors="coerce"
    )

    available_motor_roi_names = sorted(
        motor_roi_long_df["ROI"]
        .dropna()
        .unique()
    )

    left_roi_candidates = [
        roi_name
        for roi_name in available_motor_roi_names
        if (
            "left" in roi_name.lower()
            and "motor" in roi_name.lower()
        )
    ]

    if len(left_roi_candidates) == 0:
        raise ValueError(
            "No ROI name containing both 'Left' and 'Motor' "
            "was found in motor_video_roi_isc_df.\n\n"
            f"Available ROI names:\n{available_motor_roi_names}"
        )

    if len(left_roi_candidates) > 1:
        print(
            "\nMultiple possible left motor ROI labels were found:"
        )
        print(left_roi_candidates)
        print(
            f"Using: {left_roi_candidates[0]}"
        )

    selected_left_motor_roi = left_roi_candidates[0]

    left_motor_video_df = (
        motor_roi_long_df[
            motor_roi_long_df["ROI"]
            == selected_left_motor_roi
        ]
        .groupby(
            ["Subject", "Trigger"],
            as_index=False
        )["ISC"]
        .mean()
    )

    left_motor_video_df[LEFT_MOTOR_SUBJECT_COLUMN] = fisher_z_isc(
        left_motor_video_df["ISC"]
    )

    left_motor_video_df = left_motor_video_df[
        [
            "Subject",
            "Trigger",
            LEFT_MOTOR_SUBJECT_COLUMN
        ]
    ]

    print(
        "\nSelected left motor ROI label:\n"
        f"{selected_left_motor_roi}"
    )


# ============================================================
# 10. RESTRICT MOTOR PCS AND LEFT MOTOR ISC TO MATCHED VIDEOS
# ============================================================

if "Trigger" in motor_video_df.columns:

    matched_motor_video_df = pd.merge(
        motor_video_df,
        left_motor_video_df,
        on=["Subject", "Trigger"],
        how="inner",
        validate="one_to_one"
    )

else:
    # This fallback should only occur if Trigger is unavailable.
    left_motor_subject_fallback = (
        left_motor_video_df
        .groupby("Subject", as_index=False)[
            LEFT_MOTOR_SUBJECT_COLUMN
        ]
        .mean()
    )

    matched_motor_video_df = pd.merge(
        motor_video_df,
        left_motor_subject_fallback,
        on="Subject",
        how="inner"
    )


if matched_motor_video_df.empty:
    raise ValueError(
        "No matched subject × video observations were found "
        "between the motor-PC and left motor ROI tables. "
        "Check Subject and Trigger formatting."
    )


# ============================================================
# 11. AGGREGATE MATCHED MOTOR VARIABLES TO SUBJECT LEVEL
# ============================================================

aggregation_dictionary = {
    "MotorPC1": "mean",
    "MotorPC2": "mean",
    "MotorPC3": "mean",
    LEFT_MOTOR_SUBJECT_COLUMN: "mean"
}

subject_motor_df = (
    matched_motor_video_df
    .groupby("Subject", as_index=False)
    .agg(aggregation_dictionary)
)

coverage_df = (
    matched_motor_video_df
    .groupby("Subject")
    .size()
    .reset_index(name="N_Matched_Motor_Videos")
)

subject_motor_df = subject_motor_df.merge(
    coverage_df,
    on="Subject",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 12. MERGE SUBJECT-LEVEL NEURAL, PERFORMANCE, AND MOTOR DATA
# ============================================================

subject_level_motor_control_df = pd.merge(
    subject_neural_df,
    subject_motor_df,
    on="Subject",
    how="left",
    validate="one_to_one",
    indicator="MotorMerge"
)

subject_level_motor_control_df = (
    subject_level_motor_control_df
    .sort_values("Subject")
    .reset_index(drop=True)
)


# ============================================================
# 13. SAVE DATA
# ============================================================

subject_level_motor_control_df.to_csv(
    SUBJECT_MOTOR_DATA_DIR
    / "subject_level_motor_control_data.csv",
    index=False
)

matched_motor_video_df.to_csv(
    SUBJECT_MOTOR_DATA_DIR
    / "matched_subject_video_motor_data.csv",
    index=False
)


# ============================================================
# 14. COVERAGE AND DESCRIPTIVE SUMMARIES
# ============================================================

coverage_summary_df = pd.DataFrame(
    {
        "Metric": [
            "Subjects in original analysis_df",
            "Subjects with matched motor observations",
            "Subjects missing matched motor observations",
            "Matched subject × video observations",
            "Median matched videos per subject",
            "Minimum matched videos per subject",
            "Maximum matched videos per subject"
        ],
        "Value": [
            subject_neural_df["Subject"].nunique(),
            subject_motor_df["Subject"].nunique(),
            (
                subject_neural_df["Subject"].nunique()
                - subject_motor_df["Subject"].nunique()
            ),
            len(matched_motor_video_df),
            coverage_df["N_Matched_Motor_Videos"].median(),
            coverage_df["N_Matched_Motor_Videos"].min(),
            coverage_df["N_Matched_Motor_Videos"].max()
        ]
    }
)

coverage_summary_df.to_csv(
    SUBJECT_MOTOR_TABLE_DIR
    / "subject_level_motor_coverage_summary.csv",
    index=False
)

descriptive_columns = [
    "Performance",
    "Original_Network_Balance",
    "Full_LASSO_Network_Balance",
    "MotorPC1",
    "MotorPC2",
    "MotorPC3",
    LEFT_MOTOR_SUBJECT_COLUMN,
    "N_Matched_Motor_Videos"
]

descriptive_columns = [
    column
    for column in descriptive_columns
    if column in subject_level_motor_control_df.columns
]

descriptive_rows = []

for column in descriptive_columns:
    values = pd.to_numeric(
        subject_level_motor_control_df[column],
        errors="coerce"
    )

    descriptive_rows.append(
        {
            "Variable": column,
            "N": values.notna().sum(),
            "Missing": values.isna().sum(),
            "Mean": values.mean(),
            "SD": values.std(ddof=1),
            "Median": values.median(),
            "IQR": (
                values.quantile(0.75)
                - values.quantile(0.25)
            ),
            "Minimum": values.min(),
            "Maximum": values.max()
        }
    )

subject_motor_descriptives_df = pd.DataFrame(
    descriptive_rows
)

subject_motor_descriptives_df.to_csv(
    SUBJECT_MOTOR_TABLE_DIR
    / "subject_level_motor_descriptives.csv",
    index=False
)


# ============================================================
# 15. CORRELATION MATRIX
# ============================================================

correlation_columns = [
    "Performance",
    "Original_Network_Balance",
    "Full_LASSO_Network_Balance",
    "MotorPC1",
    "MotorPC2",
    "MotorPC3",
    LEFT_MOTOR_SUBJECT_COLUMN
]

correlation_columns = [
    column
    for column in correlation_columns
    if column in subject_level_motor_control_df.columns
]

subject_motor_correlation_df = (
    subject_level_motor_control_df[
        correlation_columns
    ]
    .apply(pd.to_numeric, errors="coerce")
    .corr()
)

subject_motor_correlation_df.to_csv(
    SUBJECT_MOTOR_TABLE_DIR
    / "subject_level_motor_correlation_matrix.csv"
)

fig, ax = plt.subplots(figsize=(10, 8))

image = ax.imshow(
    subject_motor_correlation_df,
    vmin=-1,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(
    np.arange(len(correlation_columns))
)

ax.set_xticklabels(
    correlation_columns,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    np.arange(len(correlation_columns))
)

ax.set_yticklabels(
    correlation_columns
)

for row_index in range(len(correlation_columns)):
    for column_index in range(len(correlation_columns)):
        correlation_value = (
            subject_motor_correlation_df.iloc[
                row_index,
                column_index
            ]
        )

        if np.isfinite(correlation_value):
            ax.text(
                column_index,
                row_index,
                f"{correlation_value:.2f}",
                ha="center",
                va="center",
                fontsize=9
            )

fig.colorbar(
    image,
    ax=ax,
    label="Pearson correlation"
)

ax.set_title(
    "Subject-level performance, network balance, and motor controls"
)

fig.tight_layout()

fig.savefig(
    SUBJECT_MOTOR_FIG_DIR
    / "subject_level_motor_correlation_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 16. PRINT RESULTS
# ============================================================

print("\n" + "=" * 90)
print("SUBJECT-LEVEL MOTOR TABLE COMPLETE")
print("=" * 90)

print("\nCoverage summary:")
display(coverage_summary_df)

print("\nSubject-level descriptives:")
display(subject_motor_descriptives_df.round(4))

print("\nSubject-level correlation matrix:")
display(subject_motor_correlation_df.round(3))

print("\nFirst rows of merged subject-level table:")
display(
    subject_level_motor_control_df[
        [
            column
            for column in [
                "Subject",
                "Performance",
                "Original_Network_Balance",
                "Full_LASSO_Network_Balance",
                "MotorPC1",
                "MotorPC2",
                "MotorPC3",
                LEFT_MOTOR_SUBJECT_COLUMN,
                "N_Matched_Motor_Videos",
                "MotorMerge"
            ]
            if column in subject_level_motor_control_df.columns
        ]
    ]
    .head(10)
    .round(4)
)

print(
    "\nCreated notebook object:\n"
    "  subject_level_motor_control_df"
)

print(
    "\nSaved outputs to:\n"
    f"{SUBJECT_MOTOR_OUTPUT_DIR}"
)

In [131]:
analysis_df

,Subject,BaseVideo,Video,Modality,Trigger,VideoMappingMatched,MotorPC1,MotorPC2,MotorPC3,DurationSec,...,Right RLPFC,Right STS,Right TPJ,Right VLPFC,Mean_ROI_ISC,Mean_ROI_NValid,Original_Network_Balance,Full_LASSO_Network_Balance,NeuralMerge,NeuralMergeMatched
0,pid001,22,22_arousal,Arousal,43,True,5.224689,-1.743930,2.562036,53.716667,...,0.729480,0.281140,-0.514302,0.941197,0.386553,12,0.023284,-0.169312,both,True
1,pid001,22,22_valence,Valence,43,True,2.436830,-0.709944,1.477828,53.716667,...,0.729480,0.281140,-0.514302,0.941197,0.386553,12,0.023284,-0.169312,both,True
2,pid001,54,54_arousal,Arousal,53,True,-0.472819,1.603996,1.492435,32.550000,...,0.852656,0.848748,0.447444,0.978499,0.796322,12,0.232228,0.151567,both,True
3,pid001,54,54_valence,Valence,53,True,0.422923,0.671128,1.527846,32.550000,...,0.852656,0.848748,0.447444,0.978499,0.796322,12,0.232228,0.151567,both,True
4,pid001,88_baseline,88_baseline_arousal,Arousal,17,True,1.912147,-2.092068,1.101707,70.533333,...,-0.020630,-0.275085,0.290246,-0.571640,0.018728,10,-0.161150,NaN,both,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2347,pid042,60_baseline,60_baseline_valence,Valence,37,True,5.975295,-0.793368,1.569334,50.766667,...,0.873254,NaN,NaN,0.929827,0.667828,7,NaN,NaN,both,True
2348,pid042,Exp3_004_baseline,Exp3_004_baseline_arousal,Arousal,7,True,6.135703,-0.280420,0.127223,87.800000,...,-0.439477,NaN,NaN,0.187761,0.131176,7,NaN,NaN,both,True
2349,pid042,Exp3_004_baseline,Exp3_004_baseline_valence,Valence,7,True,7.522895,-2.264404,0.539485,87.800000,...,-0.439477,NaN,NaN,0.187761,0.131176,7,NaN,NaN,both,True
2350,pid042,87,87_arousal,Arousal,45,True,5.809815,-0.136547,0.566671,144.366667,...,-0.030664,NaN,NaN,0.790184,0.495576,8,NaN,NaN,both,True


In [ ]:
# ============================================================
# 
# SUBJECT-LEVEL HIERARCHICAL MOTOR-CONTROL REGRESSIONS
#
# Uses:
#   subject_level_motor_control_df
#
# Tests separately:
#   Original_Network_Balance
#   Full_LASSO_Network_Balance
#
# CONTROL SETS:
#
#   A. Kinematic control
#      MotorPC1 + MotorPC2 + MotorPC3
#
#   B. Neural motor control
#      Left_Motor_ROI_ISC_z
#
#   C. Combined motor control
#      MotorPC1 + MotorPC2 + MotorPC3
#      + Left_Motor_ROI_ISC_z
#
# No right motor ROI, bilateral motor ISC, or motor
# lateralization is used.
#
# Models:
#   Model 0: Performance ~ 1
#   Model 1: Performance ~ motor controls
#   Model 2: Performance ~ network balance
#   Model 3: Performance ~ motor controls + network balance
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. CHECK INPUT
# ============================================================

if "subject_level_motor_control_df" not in globals():
    raise ValueError(
        "subject_level_motor_control_df was not found.\n\n"
        "Run Cell 1 first."
    )


# ============================================================
# 2. OUTPUT DIRECTORIES
# ============================================================

SUBJECT_REGRESSION_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\04_subject_level_motor_control"
    r"\hierarchical_regressions"
)

SUBJECT_REGRESSION_TABLE_DIR = (
    SUBJECT_REGRESSION_OUTPUT_DIR / "tables"
)

SUBJECT_REGRESSION_MODEL_DIR = (
    SUBJECT_REGRESSION_OUTPUT_DIR / "model_summaries"
)

SUBJECT_REGRESSION_FIG_DIR = (
    SUBJECT_REGRESSION_OUTPUT_DIR / "figures"
)

SUBJECT_REGRESSION_DIAGNOSTIC_DIR = (
    SUBJECT_REGRESSION_OUTPUT_DIR / "diagnostics"
)

for folder in [
    SUBJECT_REGRESSION_OUTPUT_DIR,
    SUBJECT_REGRESSION_TABLE_DIR,
    SUBJECT_REGRESSION_MODEL_DIR,
    SUBJECT_REGRESSION_FIG_DIR,
    SUBJECT_REGRESSION_DIAGNOSTIC_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. SETTINGS
# ============================================================

OUTCOME = "Performance"

NETWORK_PREDICTORS = [
    "Original_Network_Balance",
    "Full_LASSO_Network_Balance"
]

LEFT_MOTOR_ROI_COLUMN = "Left_Motor_ROI_ISC_z"

CONTROL_SETS = {
    "A_Kinematic_Only": [
        "MotorPC1",
        "MotorPC2",
        "MotorPC3"
    ],

    "B_Left_Motor_ROI_Only": [
        LEFT_MOTOR_ROI_COLUMN
    ],

    "C_Combined_Motor_Control": [
        "MotorPC1",
        "MotorPC2",
        "MotorPC3",
        LEFT_MOTOR_ROI_COLUMN
    ]
}

# Increase for final manuscript results if desired.
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 2026

MINIMUM_COMPLETE_SUBJECTS = 20

COOKS_DISTANCE_THRESHOLD_RULE = "4_over_n"

RNG = np.random.default_rng(BOOTSTRAP_SEED)


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def zscore_numeric(series):
    """Z-score a numeric Series."""
    series = pd.to_numeric(series, errors="coerce")

    standard_deviation = series.std(ddof=1)

    if (
        not np.isfinite(standard_deviation)
        or standard_deviation <= 1e-12
    ):
        return pd.Series(
            np.nan,
            index=series.index
        )

    return (
        series - series.mean()
    ) / standard_deviation


def safe_filename(value):
    """Create a Windows-safe filename."""
    value = str(value)

    for character in '<>:"/\\|?*':
        value = value.replace(character, "_")

    return (
        value.replace(" ", "_")
        .replace("__", "_")
        .strip("._")
    )


def save_ols_summary(result, path, heading):
    """Save a statsmodels OLS summary."""
    with open(path, "w", encoding="utf-8") as output_file:
        output_file.write(heading)
        output_file.write("\n\n")
        output_file.write(result.summary().as_text())


def calculate_vif(dataframe, predictors):
    """Calculate VIF for a predictor set."""
    vif_input = dataframe[predictors].copy()

    vif_input = vif_input.apply(
        pd.to_numeric,
        errors="coerce"
    ).dropna()

    if vif_input.empty:
        return pd.DataFrame()

    design_matrix = sm.add_constant(
        vif_input,
        has_constant="add"
    )

    rows = []

    for predictor_index, predictor in enumerate(
        design_matrix.columns
    ):
        if predictor == "const":
            continue

        try:
            vif_value = variance_inflation_factor(
                design_matrix.to_numpy(dtype=float),
                predictor_index
            )
        except Exception:
            vif_value = np.nan

        rows.append(
            {
                "Predictor": predictor,
                "VIF": vif_value
            }
        )

    return pd.DataFrame(rows)


def partial_r_squared(reduced_model, full_model):
    """
    Calculate partial R² for predictors added to the full model.

    partial R² =
        (SSE_reduced - SSE_full) / SSE_reduced
    """
    reduced_sse = np.sum(
        np.asarray(reduced_model.resid) ** 2
    )

    full_sse = np.sum(
        np.asarray(full_model.resid) ** 2
    )

    if reduced_sse <= 0:
        return np.nan

    return (
        reduced_sse - full_sse
    ) / reduced_sse


def bootstrap_adjusted_network_beta(
    dataframe,
    formula,
    network_term,
    n_bootstrap,
    rng
):
    """
    Nonparametric subject bootstrap for the adjusted,
    standardized Network Balance coefficient.

    Each row is one independent subject.
    """
    beta_values = []

    sample_size = len(dataframe)

    for bootstrap_index in range(n_bootstrap):
        sampled_indices = rng.integers(
            0,
            sample_size,
            size=sample_size
        )

        bootstrap_sample = (
            dataframe
            .iloc[sampled_indices]
            .copy()
            .reset_index(drop=True)
        )

        try:
            bootstrap_result = smf.ols(
                formula,
                data=bootstrap_sample
            ).fit()

            beta_value = bootstrap_result.params.get(
                network_term,
                np.nan
            )

            if np.isfinite(beta_value):
                beta_values.append(beta_value)

        except Exception:
            continue

    beta_values = np.asarray(
        beta_values,
        dtype=float
    )

    if len(beta_values) < max(100, n_bootstrap * 0.50):
        return {
            "Bootstrap_N_Successful": len(beta_values),
            "Bootstrap_Beta_Mean": np.nan,
            "Bootstrap_Beta_SE": np.nan,
            "Bootstrap_CI_Lower": np.nan,
            "Bootstrap_CI_Upper": np.nan
        }

    return {
        "Bootstrap_N_Successful": len(beta_values),
        "Bootstrap_Beta_Mean": np.mean(beta_values),
        "Bootstrap_Beta_SE": np.std(
            beta_values,
            ddof=1
        ),
        "Bootstrap_CI_Lower": np.percentile(
            beta_values,
            2.5
        ),
        "Bootstrap_CI_Upper": np.percentile(
            beta_values,
            97.5
        )
    }


# ============================================================
# 5. RESULT CONTAINERS
# ============================================================

subject_model_rows = []
subject_network_beta_rows = []
subject_nested_test_rows = []
subject_vif_rows = []
subject_influence_rows = []
subject_bootstrap_rows = []
subject_model_failure_rows = []

subject_level_model_objects = {}


# ============================================================
# 6. FIT MODELS
# ============================================================

base_df = subject_level_motor_control_df.copy()

for network_predictor in NETWORK_PREDICTORS:

    if network_predictor not in base_df.columns:
        print(
            f"\nSkipping {network_predictor}: column not found."
        )
        continue

    for control_set_name, motor_covariates in CONTROL_SETS.items():

        print("\n" + "=" * 100)
        print(
            f"{network_predictor} | {control_set_name}"
        )

        required_columns = [
            "Subject",
            OUTCOME,
            network_predictor,
            *motor_covariates
        ]

        missing_required_columns = [
            column
            for column in required_columns
            if column not in base_df.columns
        ]

        if missing_required_columns:
            subject_model_failure_rows.append(
                {
                    "Network_Predictor": network_predictor,
                    "Control_Set": control_set_name,
                    "Error": (
                        "Missing columns: "
                        f"{missing_required_columns}"
                    )
                }
            )
            continue

        # Same complete-case participants are used for Models 0–3.
        analysis_subset = (
            base_df[required_columns]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
            .reset_index(drop=True)
        )

        number_subjects = len(analysis_subset)

        if number_subjects < MINIMUM_COMPLETE_SUBJECTS:
            subject_model_failure_rows.append(
                {
                    "Network_Predictor": network_predictor,
                    "Control_Set": control_set_name,
                    "Error": (
                        "Too few complete subjects: "
                        f"n={number_subjects}"
                    )
                }
            )
            continue

        # ----------------------------------------------------
        # Standardize outcome and predictors.
        #
        # This makes the NBI coefficients from Model 2 and
        # Model 3 directly comparable.
        # ----------------------------------------------------

        analysis_subset["Performance_z"] = zscore_numeric(
            analysis_subset[OUTCOME]
        )

        analysis_subset["Network_z"] = zscore_numeric(
            analysis_subset[network_predictor]
        )

        standardized_motor_terms = []

        for motor_index, motor_covariate in enumerate(
            motor_covariates,
            start=1
        ):
            standardized_name = f"Motor_{motor_index}_z"

            analysis_subset[standardized_name] = zscore_numeric(
                analysis_subset[motor_covariate]
            )

            standardized_motor_terms.append(
                standardized_name
            )

        analysis_subset = analysis_subset.dropna(
            subset=[
                "Performance_z",
                "Network_z",
                *standardized_motor_terms
            ]
        ).reset_index(drop=True)

        # ----------------------------------------------------
        # Model formulas
        # ----------------------------------------------------

        model0_formula = "Performance_z ~ 1"

        model1_formula = (
            "Performance_z ~ "
            + " + ".join(standardized_motor_terms)
        )

        model2_formula = (
            "Performance_z ~ Network_z"
        )

        model3_formula = (
            "Performance_z ~ "
            + " + ".join(
                standardized_motor_terms
                + ["Network_z"]
            )
        )

        formulas = {
            "Model_0_Baseline": model0_formula,
            "Model_1_Motor": model1_formula,
            "Model_2_Network": model2_formula,
            "Model_3_Combined": model3_formula
        }

        fitted_results = {}

        try:
            for model_name, formula in formulas.items():
                fitted_results[model_name] = smf.ols(
                    formula,
                    data=analysis_subset
                ).fit()

        except Exception as error:
            subject_model_failure_rows.append(
                {
                    "Network_Predictor": network_predictor,
                    "Control_Set": control_set_name,
                    "Error": str(error)
                }
            )
            continue

        model0 = fitted_results["Model_0_Baseline"]
        model1 = fitted_results["Model_1_Motor"]
        model2 = fitted_results["Model_2_Network"]
        model3 = fitted_results["Model_3_Combined"]

        subject_level_model_objects[
            (network_predictor, control_set_name)
        ] = {
            "Data": analysis_subset,
            "Model0": model0,
            "Model1": model1,
            "Model2": model2,
            "Model3": model3,
            "MotorTerms": standardized_motor_terms,
            "NetworkTerm": "Network_z"
        }

        # ----------------------------------------------------
        # Save summaries and model-fit statistics
        # ----------------------------------------------------

        for model_name, model_result in fitted_results.items():

            formula = formulas[model_name]

            subject_model_rows.append(
                {
                    "Network_Predictor": network_predictor,
                    "Control_Set": control_set_name,
                    "Model": model_name,
                    "Formula": formula,
                    "N_Subjects": int(model_result.nobs),
                    "R2": model_result.rsquared,
                    "Adjusted_R2": model_result.rsquared_adj,
                    "AIC": model_result.aic,
                    "BIC": model_result.bic,
                    "F": model_result.fvalue,
                    "Model_P": model_result.f_pvalue,
                    "Residual_SD": np.sqrt(
                        model_result.mse_resid
                    )
                }
            )

            filename_prefix = (
                f"{safe_filename(network_predictor)}__"
                f"{safe_filename(control_set_name)}__"
                f"{safe_filename(model_name)}"
            )

            save_ols_summary(
                model_result,
                SUBJECT_REGRESSION_MODEL_DIR
                / f"{filename_prefix}.txt",
                (
                    f"Network predictor: {network_predictor}\n"
                    f"Control set: {control_set_name}\n"
                    f"Model: {model_name}\n"
                    f"Formula: {formula}\n"
                    f"N subjects: {int(model_result.nobs)}"
                )
            )

        # ----------------------------------------------------
        # Extract unadjusted and adjusted network coefficients
        # ----------------------------------------------------

        model2_ci = model2.conf_int().loc["Network_z"]
        model3_ci = model3.conf_int().loc["Network_z"]

        beta_model2 = model2.params["Network_z"]
        beta_model3 = model3.params["Network_z"]

        absolute_attenuation_percent = np.nan

        if abs(beta_model2) > 1e-12:
            absolute_attenuation_percent = (
                100
                * (
                    abs(beta_model2)
                    - abs(beta_model3)
                )
                / abs(beta_model2)
            )

        network_partial_r2 = partial_r_squared(
            model1,
            model3
        )

        subject_network_beta_rows.append(
            {
                "Network_Predictor": network_predictor,
                "Control_Set": control_set_name,
                "N_Subjects": len(analysis_subset),

                "Model2_Network_Beta": beta_model2,
                "Model2_Network_SE": model2.bse["Network_z"],
                "Model2_Network_T": model2.tvalues["Network_z"],
                "Model2_Network_P": model2.pvalues["Network_z"],
                "Model2_CI_Lower": model2_ci.iloc[0],
                "Model2_CI_Upper": model2_ci.iloc[1],

                "Model3_Adjusted_Network_Beta": beta_model3,
                "Model3_Adjusted_Network_SE": model3.bse["Network_z"],
                "Model3_Adjusted_Network_T": model3.tvalues["Network_z"],
                "Model3_Adjusted_Network_P": model3.pvalues["Network_z"],
                "Model3_CI_Lower": model3_ci.iloc[0],
                "Model3_CI_Upper": model3_ci.iloc[1],

                "Absolute_Beta_Attenuation_Percent": (
                    absolute_attenuation_percent
                ),

                "Motor_Only_R2": model1.rsquared,
                "Network_Only_R2": model2.rsquared,
                "Combined_R2": model3.rsquared,
                "Incremental_R2_Adding_Network": (
                    model3.rsquared - model1.rsquared
                ),
                "Network_Partial_R2": network_partial_r2
            }
        )

        # ----------------------------------------------------
        # Nested F-tests
        # ----------------------------------------------------

        nested_comparisons = [
            (
                "Model0_vs_Model1_AddMotor",
                model0,
                model1
            ),
            (
                "Model0_vs_Model2_AddNetwork",
                model0,
                model2
            ),
            (
                "Model1_vs_Model3_AddNetwork",
                model1,
                model3
            ),
            (
                "Model2_vs_Model3_AddMotor",
                model2,
                model3
            )
        ]

        for comparison_name, reduced_model, full_model in nested_comparisons:
            f_statistic, p_value, df_difference = (
                full_model.compare_f_test(reduced_model)
            )

            subject_nested_test_rows.append(
                {
                    "Network_Predictor": network_predictor,
                    "Control_Set": control_set_name,
                    "Comparison": comparison_name,
                    "F": f_statistic,
                    "DF_Difference": df_difference,
                    "P": p_value,
                    "Reduced_R2": reduced_model.rsquared,
                    "Full_R2": full_model.rsquared,
                    "Delta_R2": (
                        full_model.rsquared
                        - reduced_model.rsquared
                    )
                }
            )

        # ----------------------------------------------------
        # VIFs for combined model
        # ----------------------------------------------------

        combined_predictors = (
            standardized_motor_terms
            + ["Network_z"]
        )

        vif_df = calculate_vif(
            analysis_subset,
            combined_predictors
        )

        if not vif_df.empty:
            vif_df["Network_Predictor"] = network_predictor
            vif_df["Control_Set"] = control_set_name

            subject_vif_rows.extend(
                vif_df.to_dict("records")
            )

        # ----------------------------------------------------
        # Influence diagnostics for combined model
        # ----------------------------------------------------

        influence = model3.get_influence()

        cooks_distance = influence.cooks_distance[0]
        leverage = influence.hat_matrix_diag
        studentized_residual = (
            influence.resid_studentized_external
        )

        cooks_threshold = (
            4.0 / len(analysis_subset)
        )

        for row_index in range(len(analysis_subset)):
            subject_influence_rows.append(
                {
                    "Network_Predictor": network_predictor,
                    "Control_Set": control_set_name,
                    "Subject": analysis_subset.loc[
                        row_index,
                        "Subject"
                    ],
                    "Cooks_Distance": cooks_distance[row_index],
                    "Cooks_Threshold": cooks_threshold,
                    "Cooks_Flag": (
                        cooks_distance[row_index]
                        > cooks_threshold
                    ),
                    "Leverage": leverage[row_index],
                    "Externally_Studentized_Residual": (
                        studentized_residual[row_index]
                    ),
                    "Large_Studentized_Residual_Flag": (
                        abs(studentized_residual[row_index]) > 3
                    )
                }
            )

        # ----------------------------------------------------
        # Subject bootstrap of adjusted Network Balance beta
        # ----------------------------------------------------

        bootstrap_results = bootstrap_adjusted_network_beta(
            dataframe=analysis_subset,
            formula=model3_formula,
            network_term="Network_z",
            n_bootstrap=N_BOOTSTRAP,
            rng=RNG
        )

        subject_bootstrap_rows.append(
            {
                "Network_Predictor": network_predictor,
                "Control_Set": control_set_name,
                "N_Subjects": len(analysis_subset),
                **bootstrap_results
            }
        )

        # ----------------------------------------------------
        # Diagnostic plots
        # ----------------------------------------------------

        figure_prefix = (
            f"{safe_filename(network_predictor)}__"
            f"{safe_filename(control_set_name)}"
        )

        fig, ax = plt.subplots(figsize=(7, 6))

        ax.scatter(
            model3.fittedvalues,
            model3.resid,
            alpha=0.75
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1
        )

        ax.set_xlabel("Combined-model fitted performance")
        ax.set_ylabel("Combined-model residual")
        ax.set_title(
            f"{network_predictor}\n"
            f"{control_set_name}: residual diagnostic"
        )

        fig.tight_layout()

        fig.savefig(
            SUBJECT_REGRESSION_FIG_DIR
            / f"{figure_prefix}__residuals_vs_fitted.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)

        sm.qqplot(
            model3.resid,
            line="45",
            fit=True
        )

        plt.title(
            f"{network_predictor}\n"
            f"{control_set_name}: residual Q–Q plot"
        )

        plt.tight_layout()

        plt.savefig(
            SUBJECT_REGRESSION_FIG_DIR
            / f"{figure_prefix}__residual_qq_plot.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()


# ============================================================
# 7. CREATE RESULT DATAFRAMES
# ============================================================

subject_level_model_fit_df = pd.DataFrame(
    subject_model_rows
)

subject_level_network_beta_df = pd.DataFrame(
    subject_network_beta_rows
)

subject_level_nested_tests_df = pd.DataFrame(
    subject_nested_test_rows
)

subject_level_vif_df = pd.DataFrame(
    subject_vif_rows
)

subject_level_influence_df = pd.DataFrame(
    subject_influence_rows
)

subject_level_bootstrap_df = pd.DataFrame(
    subject_bootstrap_rows
)

subject_level_model_failures_df = pd.DataFrame(
    subject_model_failure_rows
)


# ============================================================
# 8. SAVE TABLES
# ============================================================

subject_level_model_fit_df.to_csv(
    SUBJECT_REGRESSION_TABLE_DIR
    / "subject_level_model_fit.csv",
    index=False
)

subject_level_network_beta_df.to_csv(
    SUBJECT_REGRESSION_TABLE_DIR
    / "subject_level_network_beta_before_after_motor_control.csv",
    index=False
)

subject_level_nested_tests_df.to_csv(
    SUBJECT_REGRESSION_TABLE_DIR
    / "subject_level_nested_f_tests.csv",
    index=False
)

subject_level_vif_df.to_csv(
    SUBJECT_REGRESSION_DIAGNOSTIC_DIR
    / "subject_level_combined_model_vifs.csv",
    index=False
)

subject_level_influence_df.to_csv(
    SUBJECT_REGRESSION_DIAGNOSTIC_DIR
    / "subject_level_influence_diagnostics.csv",
    index=False
)

subject_level_bootstrap_df.to_csv(
    SUBJECT_REGRESSION_TABLE_DIR
    / "subject_level_adjusted_network_beta_bootstrap.csv",
    index=False
)

subject_level_model_failures_df.to_csv(
    SUBJECT_REGRESSION_DIAGNOSTIC_DIR
    / "subject_level_model_failures.csv",
    index=False
)


# ============================================================
# 9. PRINT RESULTS
# ============================================================

print("\n" + "=" * 105)
print("SUBJECT-LEVEL HIERARCHICAL REGRESSIONS COMPLETE")
print("=" * 105)

print("\nNetwork Balance coefficients before and after motor control:")
display(
    subject_level_network_beta_df
    .sort_values(
        [
            "Network_Predictor",
            "Control_Set"
        ]
    )
    .round(5)
)

print("\nNested model F-tests:")
display(
    subject_level_nested_tests_df
    .sort_values(
        [
            "Network_Predictor",
            "Control_Set",
            "Comparison"
        ]
    )
    .round(5)
)

print("\nBootstrap confidence intervals:")
display(
    subject_level_bootstrap_df
    .sort_values(
        [
            "Network_Predictor",
            "Control_Set"
        ]
    )
    .round(5)
)

print("\nVIF diagnostics:")
display(
    subject_level_vif_df
    .sort_values(
        [
            "Network_Predictor",
            "Control_Set",
            "Predictor"
        ]
    )
    .round(4)
)

if not subject_level_influence_df.empty:
    influential_subjects_df = (
        subject_level_influence_df[
            (
                subject_level_influence_df["Cooks_Flag"]
            )
            |
            (
                subject_level_influence_df[
                    "Large_Studentized_Residual_Flag"
                ]
            )
        ]
        .copy()
    )

    print("\nPotentially influential observations:")
    if influential_subjects_df.empty:
        print("No observations exceeded the specified thresholds.")
    else:
        display(
            influential_subjects_df
            .sort_values(
                "Cooks_Distance",
                ascending=False
            )
            .round(5)
        )

if not subject_level_model_failures_df.empty:
    print("\nModel failures or skipped analyses:")
    display(subject_level_model_failures_df)

print(
    "\nCreated notebook objects:\n"
    "  subject_level_model_fit_df\n"
    "  subject_level_network_beta_df\n"
    "  subject_level_nested_tests_df\n"
    "  subject_level_vif_df\n"
    "  subject_level_influence_df\n"
    "  subject_level_bootstrap_df\n"
    "  subject_level_model_objects"
)

print(
    "\nSaved outputs to:\n"
    f"{SUBJECT_REGRESSION_OUTPUT_DIR}"
)

In [ ]:
# ============================================================
# 
# COMPARE NETWORK BALANCE BEFORE AND AFTER MOTOR ADJUSTMENT
#
# Requires Cell 2 outputs:
#   subject_level_model_objects
#   subject_level_network_beta_df
#   subject_level_nested_tests_df
#   subject_level_bootstrap_df
#
# Produces:
#   - unadjusted versus adjusted beta comparison
#   - percentage attenuation
#   - partial correlations
#   - partial R²
#   - incremental R²
#   - nested F-test summary
#   - leave-one-subject-out beta stability
#   - result figures and interpretation table
#
# Uses left motor ROI ISC only.
# Does not use motor lateralization.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

from scipy.stats import pearsonr

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. CHECK REQUIRED OBJECTS
# ============================================================

required_objects = [
    "subject_level_model_objects",
    "subject_level_network_beta_df",
    "subject_level_nested_tests_df",
    "subject_level_bootstrap_df"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "The following required Cell 2 objects were not found:\n"
        f"{missing_objects}\n\n"
        "Run Cell 2 first."
    )


# ============================================================
# 2. OUTPUT DIRECTORIES
# ============================================================

SUBJECT_COMPARISON_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\04_subject_level_motor_control"
    r"\network_effect_comparisons"
)

SUBJECT_COMPARISON_TABLE_DIR = (
    SUBJECT_COMPARISON_OUTPUT_DIR / "tables"
)

SUBJECT_COMPARISON_FIG_DIR = (
    SUBJECT_COMPARISON_OUTPUT_DIR / "figures"
)

SUBJECT_COMPARISON_STABILITY_DIR = (
    SUBJECT_COMPARISON_OUTPUT_DIR / "loso_stability"
)

for folder in [
    SUBJECT_COMPARISON_OUTPUT_DIR,
    SUBJECT_COMPARISON_TABLE_DIR,
    SUBJECT_COMPARISON_FIG_DIR,
    SUBJECT_COMPARISON_STABILITY_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def residualize_variable(
    dataframe,
    outcome,
    nuisance_terms
):
    """
    Regress one variable on nuisance covariates and return
    residuals.
    """
    if len(nuisance_terms) == 0:
        return (
            dataframe[outcome]
            - dataframe[outcome].mean()
        )

    formula = (
        f"{outcome} ~ "
        + " + ".join(nuisance_terms)
    )

    model = smf.ols(
        formula,
        data=dataframe
    ).fit()

    return model.resid


def calculate_partial_correlation(
    dataframe,
    outcome,
    predictor,
    nuisance_terms
):
    """
    Calculate the partial correlation by residualizing both
    outcome and predictor for the same nuisance covariates.
    """
    outcome_residuals = residualize_variable(
        dataframe,
        outcome,
        nuisance_terms
    )

    predictor_residuals = residualize_variable(
        dataframe,
        predictor,
        nuisance_terms
    )

    valid = (
        np.isfinite(outcome_residuals)
        & np.isfinite(predictor_residuals)
    )

    if valid.sum() < 4:
        return np.nan, np.nan

    return pearsonr(
        np.asarray(outcome_residuals)[valid],
        np.asarray(predictor_residuals)[valid]
    )


def safe_filename(value):
    """Create a Windows-safe filename."""
    value = str(value)

    for character in '<>:"/\\|?*':
        value = value.replace(character, "_")

    return (
        value.replace(" ", "_")
        .replace("__", "_")
        .strip("._")
    )


# ============================================================
# 4. PARTIAL CORRELATION AND LOSO STABILITY
# ============================================================

partial_correlation_rows = []
loso_beta_rows = []
loso_summary_rows = []


for (
    network_predictor,
    control_set_name
), model_information in subject_level_model_objects.items():

    analysis_data = (
        model_information["Data"]
        .copy()
        .reset_index(drop=True)
    )

    motor_terms = model_information["MotorTerms"]

    model2 = model_information["Model2"]
    model3 = model_information["Model3"]

    partial_r, partial_p = calculate_partial_correlation(
        dataframe=analysis_data,
        outcome="Performance_z",
        predictor="Network_z",
        nuisance_terms=motor_terms
    )

    partial_correlation_rows.append(
        {
            "Network_Predictor": network_predictor,
            "Control_Set": control_set_name,
            "N_Subjects": len(analysis_data),
            "Partial_r": partial_r,
            "Partial_P": partial_p
        }
    )

    # --------------------------------------------------------
    # Leave-one-subject-out coefficient stability
    #
    # This does not reconstruct the NBI. It checks whether the
    # adjusted NBI coefficient depends on one participant.
    # --------------------------------------------------------

    model3_formula = model3.model.formula

    heldout_betas = []

    for heldout_index in range(len(analysis_data)):

        heldout_subject = analysis_data.loc[
            heldout_index,
            "Subject"
        ]

        training_data = (
            analysis_data
            .drop(index=heldout_index)
            .reset_index(drop=True)
        )

        try:
            training_model = smf.ols(
                model3_formula,
                data=training_data
            ).fit()

            network_beta = training_model.params.get(
                "Network_z",
                np.nan
            )

            network_p = training_model.pvalues.get(
                "Network_z",
                np.nan
            )

        except Exception:
            network_beta = np.nan
            network_p = np.nan

        loso_beta_rows.append(
            {
                "Network_Predictor": network_predictor,
                "Control_Set": control_set_name,
                "Heldout_Subject": heldout_subject,
                "LOSO_Adjusted_Network_Beta": network_beta,
                "LOSO_Adjusted_Network_P": network_p
            }
        )

        if np.isfinite(network_beta):
            heldout_betas.append(network_beta)

    heldout_betas = np.asarray(
        heldout_betas,
        dtype=float
    )

    full_sample_beta = model3.params["Network_z"]

    loso_summary_rows.append(
        {
            "Network_Predictor": network_predictor,
            "Control_Set": control_set_name,
            "Full_Sample_Adjusted_Beta": full_sample_beta,
            "N_Successful_LOSO_Fits": len(heldout_betas),
            "LOSO_Beta_Mean": (
                np.mean(heldout_betas)
                if len(heldout_betas) > 0
                else np.nan
            ),
            "LOSO_Beta_SD": (
                np.std(heldout_betas, ddof=1)
                if len(heldout_betas) > 1
                else np.nan
            ),
            "LOSO_Beta_Minimum": (
                np.min(heldout_betas)
                if len(heldout_betas) > 0
                else np.nan
            ),
            "LOSO_Beta_Maximum": (
                np.max(heldout_betas)
                if len(heldout_betas) > 0
                else np.nan
            ),
            "Proportion_LOSO_Betas_Same_Direction": (
                np.mean(
                    np.sign(heldout_betas)
                    == np.sign(full_sample_beta)
                )
                if len(heldout_betas) > 0
                else np.nan
            ),
            "Proportion_LOSO_P_Less_Than_05": (
                np.mean(
                    pd.DataFrame(loso_beta_rows)
                    .query(
                        "Network_Predictor == @network_predictor "
                        "and Control_Set == @control_set_name"
                    )["LOSO_Adjusted_Network_P"]
                    < 0.05
                )
            )
        }
    )


subject_level_partial_correlations_df = pd.DataFrame(
    partial_correlation_rows
)

subject_level_loso_beta_df = pd.DataFrame(
    loso_beta_rows
)

subject_level_loso_summary_df = pd.DataFrame(
    loso_summary_rows
)


# ============================================================
# 5. EXTRACT PRIMARY NESTED F-TEST
# ============================================================

add_network_test_df = (
    subject_level_nested_tests_df[
        subject_level_nested_tests_df["Comparison"]
        == "Model1_vs_Model3_AddNetwork"
    ]
    .copy()
    .rename(
        columns={
            "F": "Nested_F_Adding_Network",
            "DF_Difference": "Nested_DF_Difference",
            "P": "Nested_P_Adding_Network",
            "Delta_R2": "Nested_Delta_R2"
        }
    )
)


# ============================================================
# 6. MERGE FINAL COMPARISON TABLE
# ============================================================

subject_level_final_comparison_df = (
    subject_level_network_beta_df
    .merge(
        subject_level_partial_correlations_df,
        on=[
            "Network_Predictor",
            "Control_Set",
            "N_Subjects"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        add_network_test_df[
            [
                "Network_Predictor",
                "Control_Set",
                "Nested_F_Adding_Network",
                "Nested_DF_Difference",
                "Nested_P_Adding_Network",
                "Nested_Delta_R2"
            ]
        ],
        on=[
            "Network_Predictor",
            "Control_Set"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        subject_level_bootstrap_df,
        on=[
            "Network_Predictor",
            "Control_Set",
            "N_Subjects"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        subject_level_loso_summary_df,
        on=[
            "Network_Predictor",
            "Control_Set"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 7. ADD INTERPRETATION FLAGS
# ============================================================

def classify_attenuation(value):
    """
    Descriptive interpretation only.

    These cutoffs are not inferential thresholds.
    """
    if not np.isfinite(value):
        return "Unavailable"

    if value < 0:
        return "Coefficient increased after motor adjustment"

    if value < 10:
        return "Minimal attenuation"

    if value < 30:
        return "Modest attenuation"

    if value < 50:
        return "Moderate attenuation"

    return "Large attenuation"


subject_level_final_comparison_df[
    "Attenuation_Interpretation"
] = (
    subject_level_final_comparison_df[
        "Absolute_Beta_Attenuation_Percent"
    ]
    .apply(classify_attenuation)
)

subject_level_final_comparison_df[
    "Adjusted_Network_Significant"
] = (
    subject_level_final_comparison_df[
        "Model3_Adjusted_Network_P"
    ] < 0.05
)

subject_level_final_comparison_df[
    "Bootstrap_CI_Excludes_Zero"
] = (
    (
        subject_level_final_comparison_df[
            "Bootstrap_CI_Lower"
        ] > 0
    )
    |
    (
        subject_level_final_comparison_df[
            "Bootstrap_CI_Upper"
        ] < 0
    )
)

subject_level_final_comparison_df[
    "Network_Improves_Motor_Model"
] = (
    subject_level_final_comparison_df[
        "Nested_P_Adding_Network"
    ] < 0.05
)


# ============================================================
# 8. SAVE TABLES
# ============================================================

subject_level_final_comparison_df.to_csv(
    SUBJECT_COMPARISON_TABLE_DIR
    / "final_network_effect_before_after_motor_adjustment.csv",
    index=False
)

subject_level_partial_correlations_df.to_csv(
    SUBJECT_COMPARISON_TABLE_DIR
    / "partial_correlations_controlling_motor_variables.csv",
    index=False
)

subject_level_loso_beta_df.to_csv(
    SUBJECT_COMPARISON_STABILITY_DIR
    / "loso_adjusted_network_beta_by_heldout_subject.csv",
    index=False
)

subject_level_loso_summary_df.to_csv(
    SUBJECT_COMPARISON_STABILITY_DIR
    / "loso_adjusted_network_beta_summary.csv",
    index=False
)


# ============================================================
# 9. FIGURE — UNADJUSTED VS ADJUSTED NETWORK BETAS
# ============================================================

plot_df = (
    subject_level_final_comparison_df
    .copy()
    .sort_values(
        [
            "Network_Predictor",
            "Control_Set"
        ]
    )
    .reset_index(drop=True)
)

plot_df["Plot_Label"] = (
    plot_df["Network_Predictor"]
    .str.replace("_", " ", regex=False)
    + "\n"
    + plot_df["Control_Set"]
    .str.replace("_", " ", regex=False)
)

x_positions = np.arange(len(plot_df))
bar_width = 0.36

fig, ax = plt.subplots(
    figsize=(
        max(13, len(plot_df) * 2.2),
        7
    )
)

ax.bar(
    x_positions - bar_width / 2,
    plot_df["Model2_Network_Beta"],
    width=bar_width,
    label="Model 2: network only"
)

ax.bar(
    x_positions + bar_width / 2,
    plot_df["Model3_Adjusted_Network_Beta"],
    width=bar_width,
    label="Model 3: after motor adjustment"
)

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(x_positions)

ax.set_xticklabels(
    plot_df["Plot_Label"],
    rotation=30,
    ha="right"
)

ax.set_ylabel(
    "Standardized Network Balance coefficient"
)

ax.set_title(
    "Network Balance prediction before and after motor adjustment"
)

ax.legend(frameon=False)

fig.tight_layout()

fig.savefig(
    SUBJECT_COMPARISON_FIG_DIR
    / "network_beta_before_after_motor_adjustment.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 10. FIGURE — ADJUSTED BETAS WITH 95% CIs
# ============================================================

adjusted_beta_values = (
    plot_df["Model3_Adjusted_Network_Beta"]
    .to_numpy(dtype=float)
)

adjusted_lower_errors = (
    adjusted_beta_values
    - plot_df["Model3_CI_Lower"].to_numpy(dtype=float)
)

adjusted_upper_errors = (
    plot_df["Model3_CI_Upper"].to_numpy(dtype=float)
    - adjusted_beta_values
)

fig, ax = plt.subplots(
    figsize=(
        max(13, len(plot_df) * 2.2),
        7
    )
)

ax.errorbar(
    x_positions,
    adjusted_beta_values,
    yerr=np.vstack(
        [
            adjusted_lower_errors,
            adjusted_upper_errors
        ]
    ),
    fmt="o",
    capsize=5
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xticks(x_positions)

ax.set_xticklabels(
    plot_df["Plot_Label"],
    rotation=30,
    ha="right"
)

ax.set_ylabel(
    "Adjusted standardized Network Balance coefficient"
)

ax.set_title(
    "Network Balance prediction after controlling motor variables"
)

fig.tight_layout()

fig.savefig(
    SUBJECT_COMPARISON_FIG_DIR
    / "adjusted_network_betas_with_confidence_intervals.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 11. FIGURE — PARTIAL R²
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        max(13, len(plot_df) * 2.2),
        7
    )
)

ax.bar(
    x_positions,
    plot_df["Network_Partial_R2"]
)

ax.axhline(0, linewidth=1)

ax.set_xticks(x_positions)

ax.set_xticklabels(
    plot_df["Plot_Label"],
    rotation=30,
    ha="right"
)

ax.set_ylabel(
    "Partial R² for Network Balance"
)

ax.set_title(
    "Unique performance variance explained by Network Balance\n"
    "after motor adjustment"
)

fig.tight_layout()

fig.savefig(
    SUBJECT_COMPARISON_FIG_DIR
    / "network_partial_r_squared_after_motor_control.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 12. FIGURE — LOSO BETA STABILITY
# ============================================================

for (
    network_predictor,
    control_set_name
), group_df in subject_level_loso_beta_df.groupby(
    [
        "Network_Predictor",
        "Control_Set"
    ]
):

    group_df = (
        group_df
        .sort_values("LOSO_Adjusted_Network_Beta")
        .reset_index(drop=True)
    )

    full_sample_beta = (
        subject_level_loso_summary_df.loc[
            (
                subject_level_loso_summary_df[
                    "Network_Predictor"
                ] == network_predictor
            )
            &
            (
                subject_level_loso_summary_df[
                    "Control_Set"
                ] == control_set_name
            ),
            "Full_Sample_Adjusted_Beta"
        ]
        .iloc[0]
    )

    fig, ax = plt.subplots(figsize=(11, 6))

    ax.plot(
        np.arange(len(group_df)),
        group_df["LOSO_Adjusted_Network_Beta"],
        marker="o"
    )

    ax.axhline(
        full_sample_beta,
        linestyle="--",
        linewidth=2,
        label="Full-sample adjusted beta"
    )

    ax.axhline(
        0,
        linewidth=1
    )

    ax.set_xticks(
        np.arange(len(group_df))
    )

    ax.set_xticklabels(
        group_df["Heldout_Subject"],
        rotation=90
    )

    ax.set_xlabel("Held-out participant")
    ax.set_ylabel("Adjusted Network Balance beta")

    ax.set_title(
        f"{network_predictor}\n"
        f"{control_set_name}: leave-one-subject-out beta stability"
    )

    ax.legend(frameon=False)

    fig.tight_layout()

    figure_filename = (
        f"{safe_filename(network_predictor)}__"
        f"{safe_filename(control_set_name)}__"
        "loso_beta_stability.png"
    )

    fig.savefig(
        SUBJECT_COMPARISON_FIG_DIR / figure_filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 13. PRINT FINAL RESULTS
# ============================================================

final_display_columns = [
    "Network_Predictor",
    "Control_Set",
    "N_Subjects",

    "Model2_Network_Beta",
    "Model2_Network_P",

    "Model3_Adjusted_Network_Beta",
    "Model3_Adjusted_Network_SE",
    "Model3_Adjusted_Network_T",
    "Model3_Adjusted_Network_P",
    "Model3_CI_Lower",
    "Model3_CI_Upper",

    "Absolute_Beta_Attenuation_Percent",
    "Attenuation_Interpretation",

    "Partial_r",
    "Partial_P",

    "Network_Partial_R2",
    "Incremental_R2_Adding_Network",

    "Nested_F_Adding_Network",
    "Nested_P_Adding_Network",

    "Bootstrap_CI_Lower",
    "Bootstrap_CI_Upper",

    "Proportion_LOSO_Betas_Same_Direction",
    "Proportion_LOSO_P_Less_Than_05"
]

final_display_columns = [
    column
    for column in final_display_columns
    if column in subject_level_final_comparison_df.columns
]

print("\n" + "=" * 110)
print("FINAL SUBJECT-LEVEL MOTOR-CONTROL COMPARISON")
print("=" * 110)

display(
    subject_level_final_comparison_df[
        final_display_columns
    ]
    .sort_values(
        [
            "Network_Predictor",
            "Control_Set"
        ]
    )
    .round(5)
)

print(
    "\nInterpretation guide:\n"
    "  • Model2_Network_Beta is the unadjusted standardized "
    "Network Balance relationship.\n"
    "  • Model3_Adjusted_Network_Beta is the relationship after "
    "controlling the specified motor variables.\n"
    "  • Minimal beta attenuation indicates that the motor "
    "variables explain little of the original association.\n"
    "  • Network_Partial_R2 is the unique fraction of residual "
    "performance variance associated with Network Balance.\n"
    "  • Model1_vs_Model3 is the direct nested F-test of whether "
    "Network Balance improves the motor-only model.\n"
    "  • Partial_r controls the same motor covariates by "
    "residualizing both performance and Network Balance.\n"
    "  • The bootstrap CI and LOSO stability analyses assess "
    "small-sample robustness and sensitivity to individual "
    "participants."
)

print(
    "\nCreated notebook objects:\n"
    "  subject_level_final_comparison_df\n"
    "  subject_level_partial_correlations_df\n"
    "  subject_level_loso_beta_df\n"
    "  subject_level_loso_summary_df"
)

print(
    "\nSaved outputs to:\n"
    f"{SUBJECT_COMPARISON_OUTPUT_DIR}"
)

In [ ]:
# ============================================================
# 
# NESTED OUT-OF-FOLD MOTOR-RESIDUALIZED LASSO + NETWORK BALANCE
#
# PURPOSE
# ------------------------------------------------------------
# Rebuild the ROI ISC -> performance LASSO after removing
# variance associated with:
#
#   MotorPC1
#   MotorPC2
#   MotorPC3
#   Left_Motor_ROI_ISC_z
#
# Critically, motor residualization, standardization, alpha
# selection, ROI selection, and Network Balance construction
# are performed INSIDE each outer leave-one-subject-out fold.
#
# This prevents held-out-subject information from influencing:
#   - motor residualization
#   - feature scaling
#   - LASSO alpha tuning
#   - ROI coefficient estimation
#   - fold-specific Network Balance construction
#
# REQUIRED OBJECTS
# ------------------------------------------------------------
#   subject_level_motor_control_df
#   roi_isc_df
#
# RECOMMENDED:
#   Run Analysis 04 first so that the subject-level motor
#   variables and performance values already match the main
#   Network Balance analysis.
#
# PRIMARY OUTPUTS
# ------------------------------------------------------------
#   motor_residualized_lasso_oof_predictions_df
#   motor_residualized_lasso_summary_df
#   motor_residualized_lasso_fold_coefficients_df
#   motor_residualized_lasso_selection_stability_df
#   motor_residualized_oof_nbi_regression_df
#
# OUTPUT FOLDER
# ------------------------------------------------------------
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\
# Emotion Perception Data\Motor_Control\05_out_of_fold_NBI
# ============================================================

import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

from sklearn.base import clone
from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    LassoCV
)
from sklearn.model_selection import (
    LeaveOneOut,
    KFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 1. OUTPUT DIRECTORIES
# ============================================================

OOF_NBI_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\05_out_of_fold_NBI"
)

OOF_NBI_DATA_DIR = OOF_NBI_OUTPUT_DIR / "data"
OOF_NBI_TABLE_DIR = OOF_NBI_OUTPUT_DIR / "tables"
OOF_NBI_FIGURE_DIR = OOF_NBI_OUTPUT_DIR / "figures"
OOF_NBI_MODEL_DIR = OOF_NBI_OUTPUT_DIR / "model_summaries"
OOF_NBI_PERMUTATION_DIR = OOF_NBI_OUTPUT_DIR / "permutation"

for folder in [
    OOF_NBI_OUTPUT_DIR,
    OOF_NBI_DATA_DIR,
    OOF_NBI_TABLE_DIR,
    OOF_NBI_FIGURE_DIR,
    OOF_NBI_MODEL_DIR,
    OOF_NBI_PERMUTATION_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(
    "Analysis 05 outputs will be saved to:\n"
    f"{OOF_NBI_OUTPUT_DIR}"
)


# ============================================================
# 2. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "subject_level_motor_control_df",
    "roi_isc_df"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "The following required notebook objects were not found:\n"
        f"{missing_objects}\n\n"
        "Run the original ROI ISC analysis and Analysis 04 first."
    )


# ============================================================
# 3. ANALYSIS SETTINGS
# ============================================================

SUBJECT_COLUMN = "Subject"
OUTCOME_COLUMN = "Performance"

MOTOR_COVARIATES = [
    "MotorPC1",
    "MotorPC2",
    "MotorPC3",
    "Left_Motor_ROI_ISC_z"
]

# Inner CV settings used to tune LASSO alpha within each
# outer leave-one-subject-out training fold.
INNER_CV_FOLDS = 5
INNER_CV_RANDOM_STATE = 2026

# Explicit alpha grid gives consistent tuning across folds.
LASSO_ALPHAS = np.logspace(
    -4,
    1,
    100
)

LASSO_MAX_ITER = 100000
COEFFICIENT_ZERO_TOLERANCE = 1e-8

# ------------------------------------------------------------
# Permutation settings
#
# Each permutation reruns the ENTIRE nested pipeline after
# permuting performance across subjects.
#
# 1,000 permutations is a reasonable initial run.
# For final manuscript inference, increase to 10,000.
#
# Set RUN_PERMUTATIONS = False for a quick diagnostic run.
# ------------------------------------------------------------

RUN_PERMUTATIONS = True
N_PERMUTATIONS = 1000
PERMUTATION_RANDOM_STATE = 2026
PERMUTATION_PROGRESS_EVERY = 25

# Minimum usable sample.
MINIMUM_SUBJECTS = 20

# Fixed architectures are included as secondary sensitivity
# analyses after motor residualization inside each fold.
ORIGINAL_POSITIVE_ROIS = [
    "Right VLPFC",
    "Left TPJ"
]

ORIGINAL_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Right TPJ"
]

FULL_LASSO_POSITIVE_ROIS = [
    "Left TPJ",
    "Right VLPFC",
    "Left OFC",
    "Left STS"
]

FULL_LASSO_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Left DLPFC",
    "Right TPJ"
]


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """Normalize participant IDs before merging."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def safe_filename(value):
    """Create a Windows-safe filename."""
    value = str(value)

    for character in '<>:"/\\|?*':
        value = value.replace(character, "_")

    return (
        value.replace(" ", "_")
        .replace("__", "_")
        .strip("._")
    )


def safe_pearsonr(x, y):
    """Pearson correlation with validity checks."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 4
        or np.std(x, ddof=1) <= 1e-12
        or np.std(y, ddof=1) <= 1e-12
    ):
        return np.nan, np.nan

    return pearsonr(x, y)


def safe_spearmanr(x, y):
    """Spearman correlation with validity checks."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 4
        or np.std(x, ddof=1) <= 1e-12
        or np.std(y, ddof=1) <= 1e-12
    ):
        return np.nan, np.nan

    result = spearmanr(x, y)

    return float(result.statistic), float(result.pvalue)


def calculate_balance_score(
    X_standardized,
    coefficients,
    coefficient_tolerance=1e-8
):
    """
    Construct a fold-specific, unweighted Network Balance score.

    Positive LASSO coefficients define the positive side.
    Negative LASSO coefficients define the negative side.

    Score:
        mean(positive selected ROI values)
        minus
        mean(negative selected ROI values)

    Returns NaN when the training-fold LASSO does not select
    at least one positive and one negative ROI.
    """
    X_standardized = np.asarray(
        X_standardized,
        dtype=float
    )

    coefficients = np.asarray(
        coefficients,
        dtype=float
    )

    positive_mask = coefficients > coefficient_tolerance
    negative_mask = coefficients < -coefficient_tolerance

    if (
        positive_mask.sum() == 0
        or negative_mask.sum() == 0
    ):
        return np.full(
            X_standardized.shape[0],
            np.nan
        )

    positive_mean = np.mean(
        X_standardized[:, positive_mask],
        axis=1
    )

    negative_mean = np.mean(
        X_standardized[:, negative_mask],
        axis=1
    )

    return positive_mean - negative_mean


def calculate_fixed_balance_score(
    X_standardized,
    roi_names,
    positive_rois,
    negative_rois
):
    """
    Calculate a fixed architecture balance score from
    motor-residualized, training-standardized ROI values.
    """
    roi_names = list(roi_names)

    missing = [
        roi
        for roi in positive_rois + negative_rois
        if roi not in roi_names
    ]

    if missing:
        return np.full(
            X_standardized.shape[0],
            np.nan
        )

    positive_indices = [
        roi_names.index(roi)
        for roi in positive_rois
    ]

    negative_indices = [
        roi_names.index(roi)
        for roi in negative_rois
    ]

    positive_mean = np.mean(
        X_standardized[:, positive_indices],
        axis=1
    )

    negative_mean = np.mean(
        X_standardized[:, negative_indices],
        axis=1
    )

    return positive_mean - negative_mean


def fit_training_calibration(
    training_score,
    training_target
):
    """
    Fit a one-predictor calibration model within a training fold.

    This converts a fold-specific NBI score into a predicted
    motor-residualized performance value.
    """
    training_score = np.asarray(
        training_score,
        dtype=float
    ).reshape(-1)

    training_target = np.asarray(
        training_target,
        dtype=float
    ).reshape(-1)

    valid = (
        np.isfinite(training_score)
        & np.isfinite(training_target)
    )

    if (
        valid.sum() < 5
        or np.std(training_score[valid], ddof=1) <= 1e-12
    ):
        return None

    calibration_model = LinearRegression()

    calibration_model.fit(
        training_score[valid].reshape(-1, 1),
        training_target[valid]
    )

    return calibration_model


def calculate_cross_validated_metrics(
    observed,
    predicted
):
    """Calculate core out-of-fold prediction metrics."""
    observed = np.asarray(
        observed,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    valid = (
        np.isfinite(observed)
        & np.isfinite(predicted)
    )

    observed_valid = observed[valid]
    predicted_valid = predicted[valid]

    if len(observed_valid) < 4:
        return {
            "N": len(observed_valid),
            "CV_R2": np.nan,
            "Pearson_r": np.nan,
            "Pearson_p": np.nan,
            "Spearman_rho": np.nan,
            "Spearman_p": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan
        }

    cv_r2 = r2_score(
        observed_valid,
        predicted_valid
    )

    pearson_r, pearson_p = safe_pearsonr(
        observed_valid,
        predicted_valid
    )

    spearman_rho, spearman_p = safe_spearmanr(
        observed_valid,
        predicted_valid
    )

    rmse = np.sqrt(
        mean_squared_error(
            observed_valid,
            predicted_valid
        )
    )

    mae = mean_absolute_error(
        observed_valid,
        predicted_valid
    )

    return {
        "N": len(observed_valid),
        "CV_R2": cv_r2,
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p,
        "RMSE": rmse,
        "MAE": mae
    }


def run_nested_motor_residualized_lasso(
    dataframe,
    roi_columns,
    outcome_column,
    motor_columns,
    alpha_grid,
    inner_folds=5,
    inner_random_state=2026,
    max_iter=100000,
    coefficient_tolerance=1e-8,
    return_fold_details=True
):
    """
    Run the full nested motor-residualized LASSO pipeline.

    Outer loop:
        Leave one participant out.

    Inside each outer training fold:
        1. Fit motor -> performance.
        2. Fit motor -> each ROI simultaneously.
        3. Residualize training and held-out outcome.
        4. Residualize training and held-out ROI values.
        5. Standardize residualized training data.
        6. Apply training scalers to the held-out data.
        7. Tune LASSO alpha with inner CV.
        8. Fit LASSO on residualized training data.
        9. Predict held-out residualized performance.
       10. Construct training-fold-derived NBI.
       11. Calibrate the NBI in the training fold.
       12. Predict held-out performance from the NBI.
    """
    working_df = (
        dataframe
        .reset_index(drop=True)
        .copy()
    )

    roi_columns = list(roi_columns)

    X_roi = working_df[
        roi_columns
    ].to_numpy(dtype=float)

    X_motor = working_df[
        motor_columns
    ].to_numpy(dtype=float)

    y = working_df[
        outcome_column
    ].to_numpy(dtype=float)

    subjects = working_df[
        SUBJECT_COLUMN
    ].astype(str).to_numpy()

    number_subjects = len(working_df)
    number_rois = len(roi_columns)

    outer_cv = LeaveOneOut()

    observed_residual_z = np.full(
        number_subjects,
        np.nan
    )

    predicted_lasso_residual_z = np.full(
        number_subjects,
        np.nan
    )

    predicted_fold_nbi_residual_z = np.full(
        number_subjects,
        np.nan
    )

    fold_nbi_score = np.full(
        number_subjects,
        np.nan
    )

    predicted_original_fixed_nbi_z = np.full(
        number_subjects,
        np.nan
    )

    original_fixed_nbi_score = np.full(
        number_subjects,
        np.nan
    )

    predicted_full_fixed_nbi_z = np.full(
        number_subjects,
        np.nan
    )

    full_fixed_nbi_score = np.full(
        number_subjects,
        np.nan
    )

    fold_rows = []
    coefficient_rows = []

    for outer_fold, (
        training_indices,
        heldout_indices
    ) in enumerate(
        outer_cv.split(X_roi),
        start=1
    ):

        heldout_index = int(
            heldout_indices[0]
        )

        heldout_subject = subjects[
            heldout_index
        ]

        X_roi_train = X_roi[
            training_indices
        ]

        X_roi_test = X_roi[
            heldout_indices
        ]

        X_motor_train = X_motor[
            training_indices
        ]

        X_motor_test = X_motor[
            heldout_indices
        ]

        y_train = y[
            training_indices
        ]

        y_test = y[
            heldout_indices
        ]

        # ----------------------------------------------------
        # A. Motor residualization of performance
        # ----------------------------------------------------

        motor_to_y_model = LinearRegression()

        motor_to_y_model.fit(
            X_motor_train,
            y_train
        )

        y_train_motor_prediction = (
            motor_to_y_model.predict(
                X_motor_train
            )
        )

        y_test_motor_prediction = (
            motor_to_y_model.predict(
                X_motor_test
            )
        )

        y_train_residual = (
            y_train
            - y_train_motor_prediction
        )

        y_test_residual = (
            y_test
            - y_test_motor_prediction
        )

        # ----------------------------------------------------
        # B. Motor residualization of every substantive ROI
        #
        # LinearRegression supports multivariate Y, so all ROI
        # residualization models are fit simultaneously.
        # ----------------------------------------------------

        motor_to_roi_model = LinearRegression()

        motor_to_roi_model.fit(
            X_motor_train,
            X_roi_train
        )

        X_roi_train_motor_prediction = (
            motor_to_roi_model.predict(
                X_motor_train
            )
        )

        X_roi_test_motor_prediction = (
            motor_to_roi_model.predict(
                X_motor_test
            )
        )

        X_roi_train_residual = (
            X_roi_train
            - X_roi_train_motor_prediction
        )

        X_roi_test_residual = (
            X_roi_test
            - X_roi_test_motor_prediction
        )

        # ----------------------------------------------------
        # C. Training-only standardization
        # ----------------------------------------------------

        roi_scaler = StandardScaler()

        X_roi_train_z = roi_scaler.fit_transform(
            X_roi_train_residual
        )

        X_roi_test_z = roi_scaler.transform(
            X_roi_test_residual
        )

        outcome_scaler = StandardScaler()

        y_train_z = (
            outcome_scaler
            .fit_transform(
                y_train_residual.reshape(-1, 1)
            )
            .reshape(-1)
        )

        y_test_z = (
            outcome_scaler
            .transform(
                y_test_residual.reshape(-1, 1)
            )
            .reshape(-1)
        )

        observed_residual_z[
            heldout_index
        ] = y_test_z[0]

        # ----------------------------------------------------
        # D. Inner CV for alpha tuning
        # ----------------------------------------------------

        effective_inner_folds = min(
            inner_folds,
            len(training_indices)
        )

        inner_cv = KFold(
            n_splits=effective_inner_folds,
            shuffle=True,
            random_state=(
                inner_random_state
                + outer_fold
            )
        )

        lasso_cv_model = LassoCV(
            alphas=alpha_grid,
            cv=inner_cv,
            max_iter=max_iter,
            random_state=(
                inner_random_state
                + outer_fold
            ),
            n_jobs=-1
        )

        lasso_cv_model.fit(
            X_roi_train_z,
            y_train_z
        )

        chosen_alpha = float(
            lasso_cv_model.alpha_
        )

        # Refit explicitly at the selected alpha.
        lasso_model = Lasso(
            alpha=chosen_alpha,
            max_iter=max_iter,
            random_state=(
                inner_random_state
                + outer_fold
            )
        )

        lasso_model.fit(
            X_roi_train_z,
            y_train_z
        )

        coefficients = (
            lasso_model.coef_
            .astype(float)
        )

        selected_mask = (
            np.abs(coefficients)
            > coefficient_tolerance
        )

        positive_mask = (
            coefficients
            > coefficient_tolerance
        )

        negative_mask = (
            coefficients
            < -coefficient_tolerance
        )

        # ----------------------------------------------------
        # E. Direct held-out LASSO prediction
        # ----------------------------------------------------

        predicted_lasso_residual_z[
            heldout_index
        ] = float(
            lasso_model.predict(
                X_roi_test_z
            )[0]
        )

        # ----------------------------------------------------
        # F. Reconstruct fold-specific NBI from coefficient
        #    signs selected using training subjects only
        # ----------------------------------------------------

        train_fold_nbi = calculate_balance_score(
            X_standardized=X_roi_train_z,
            coefficients=coefficients,
            coefficient_tolerance=coefficient_tolerance
        )

        test_fold_nbi = calculate_balance_score(
            X_standardized=X_roi_test_z,
            coefficients=coefficients,
            coefficient_tolerance=coefficient_tolerance
        )

        if np.isfinite(
            test_fold_nbi[0]
        ):
            fold_nbi_score[
                heldout_index
            ] = test_fold_nbi[0]

        fold_nbi_calibration_model = (
            fit_training_calibration(
                training_score=train_fold_nbi,
                training_target=y_train_z
            )
        )

        if (
            fold_nbi_calibration_model
            is not None
            and np.isfinite(
                test_fold_nbi[0]
            )
        ):
            predicted_fold_nbi_residual_z[
                heldout_index
            ] = float(
                fold_nbi_calibration_model.predict(
                    test_fold_nbi.reshape(-1, 1)
                )[0]
            )

        # ----------------------------------------------------
        # G. Fixed original architecture after fold-specific
        #    motor residualization: secondary sensitivity
        # ----------------------------------------------------

        train_original_fixed_nbi = (
            calculate_fixed_balance_score(
                X_standardized=X_roi_train_z,
                roi_names=roi_columns,
                positive_rois=ORIGINAL_POSITIVE_ROIS,
                negative_rois=ORIGINAL_NEGATIVE_ROIS
            )
        )

        test_original_fixed_nbi = (
            calculate_fixed_balance_score(
                X_standardized=X_roi_test_z,
                roi_names=roi_columns,
                positive_rois=ORIGINAL_POSITIVE_ROIS,
                negative_rois=ORIGINAL_NEGATIVE_ROIS
            )
        )

        if np.isfinite(
            test_original_fixed_nbi[0]
        ):
            original_fixed_nbi_score[
                heldout_index
            ] = test_original_fixed_nbi[0]

        original_calibration_model = (
            fit_training_calibration(
                training_score=train_original_fixed_nbi,
                training_target=y_train_z
            )
        )

        if (
            original_calibration_model
            is not None
            and np.isfinite(
                test_original_fixed_nbi[0]
            )
        ):
            predicted_original_fixed_nbi_z[
                heldout_index
            ] = float(
                original_calibration_model.predict(
                    test_original_fixed_nbi.reshape(-1, 1)
                )[0]
            )

        # ----------------------------------------------------
        # H. Fixed Full-LASSO architecture after fold-specific
        #    motor residualization: secondary sensitivity
        # ----------------------------------------------------

        train_full_fixed_nbi = (
            calculate_fixed_balance_score(
                X_standardized=X_roi_train_z,
                roi_names=roi_columns,
                positive_rois=FULL_LASSO_POSITIVE_ROIS,
                negative_rois=FULL_LASSO_NEGATIVE_ROIS
            )
        )

        test_full_fixed_nbi = (
            calculate_fixed_balance_score(
                X_standardized=X_roi_test_z,
                roi_names=roi_columns,
                positive_rois=FULL_LASSO_POSITIVE_ROIS,
                negative_rois=FULL_LASSO_NEGATIVE_ROIS
            )
        )

        if np.isfinite(
            test_full_fixed_nbi[0]
        ):
            full_fixed_nbi_score[
                heldout_index
            ] = test_full_fixed_nbi[0]

        full_calibration_model = (
            fit_training_calibration(
                training_score=train_full_fixed_nbi,
                training_target=y_train_z
            )
        )

        if (
            full_calibration_model
            is not None
            and np.isfinite(
                test_full_fixed_nbi[0]
            )
        ):
            predicted_full_fixed_nbi_z[
                heldout_index
            ] = float(
                full_calibration_model.predict(
                    test_full_fixed_nbi.reshape(-1, 1)
                )[0]
            )

        # ----------------------------------------------------
        # I. Save fold information
        # ----------------------------------------------------

        fold_rows.append(
            {
                "Outer_Fold": outer_fold,
                "Heldout_Subject": heldout_subject,
                "Chosen_Alpha": chosen_alpha,
                "N_Selected_ROIs": int(
                    selected_mask.sum()
                ),
                "N_Positive_ROIs": int(
                    positive_mask.sum()
                ),
                "N_Negative_ROIs": int(
                    negative_mask.sum()
                ),
                "Fold_NBI_Constructed": bool(
                    positive_mask.sum() > 0
                    and negative_mask.sum() > 0
                ),
                "Observed_Motor_Adjusted_Performance_z": (
                    y_test_z[0]
                ),
                "Predicted_LASSO_Performance_z": (
                    predicted_lasso_residual_z[
                        heldout_index
                    ]
                ),
                "Fold_Derived_NBI_Score": (
                    fold_nbi_score[
                        heldout_index
                    ]
                ),
                "Predicted_Fold_Derived_NBI_Performance_z": (
                    predicted_fold_nbi_residual_z[
                        heldout_index
                    ]
                ),
                "Original_Fixed_NBI_Score": (
                    original_fixed_nbi_score[
                        heldout_index
                    ]
                ),
                "Predicted_Original_Fixed_NBI_Performance_z": (
                    predicted_original_fixed_nbi_z[
                        heldout_index
                    ]
                ),
                "Full_Fixed_NBI_Score": (
                    full_fixed_nbi_score[
                        heldout_index
                    ]
                ),
                "Predicted_Full_Fixed_NBI_Performance_z": (
                    predicted_full_fixed_nbi_z[
                        heldout_index
                    ]
                )
            }
        )

        if return_fold_details:
            for roi_name, coefficient in zip(
                roi_columns,
                coefficients
            ):
                coefficient_rows.append(
                    {
                        "Outer_Fold": outer_fold,
                        "Heldout_Subject": heldout_subject,
                        "ROI": roi_name,
                        "Coefficient": coefficient,
                        "Absolute_Coefficient": abs(
                            coefficient
                        ),
                        "Selected": bool(
                            abs(coefficient)
                            > coefficient_tolerance
                        ),
                        "Coefficient_Sign": (
                            "Positive"
                            if coefficient
                            > coefficient_tolerance
                            else (
                                "Negative"
                                if coefficient
                                < -coefficient_tolerance
                                else "Zero"
                            )
                        ),
                        "Chosen_Alpha": chosen_alpha
                    }
                )

    result = {
        "Subjects": subjects,
        "Observed_Residual_z": observed_residual_z,
        "Predicted_LASSO_Residual_z": (
            predicted_lasso_residual_z
        ),
        "Fold_Derived_NBI_Score": fold_nbi_score,
        "Predicted_Fold_Derived_NBI_Residual_z": (
            predicted_fold_nbi_residual_z
        ),
        "Original_Fixed_NBI_Score": (
            original_fixed_nbi_score
        ),
        "Predicted_Original_Fixed_NBI_Residual_z": (
            predicted_original_fixed_nbi_z
        ),
        "Full_Fixed_NBI_Score": (
            full_fixed_nbi_score
        ),
        "Predicted_Full_Fixed_NBI_Residual_z": (
            predicted_full_fixed_nbi_z
        ),
        "Fold_Details": pd.DataFrame(
            fold_rows
        ),
        "Coefficient_Details": pd.DataFrame(
            coefficient_rows
        )
    }

    return result


# ============================================================
# 5. PREPARE SUBJECT-LEVEL ANALYSIS DATA
# ============================================================

motor_subject_df = (
    subject_level_motor_control_df
    .copy()
)

motor_subject_df[
    SUBJECT_COLUMN
] = (
    motor_subject_df[
        SUBJECT_COLUMN
    ]
    .apply(normalize_subject_id)
)

roi_subject_df = roi_isc_df.copy()

if SUBJECT_COLUMN not in roi_subject_df.columns:
    raise ValueError(
        "roi_isc_df does not contain a Subject column."
    )

roi_subject_df[
    SUBJECT_COLUMN
] = (
    roi_subject_df[
        SUBJECT_COLUMN
    ]
    .apply(normalize_subject_id)
)

# ------------------------------------------------------------
# Identify the substantive ROI predictor columns.
#
# Prefer the notebook's original roi_cols definition when it
# exists. Otherwise, use all numeric roi_isc_df columns except
# Subject.
# ------------------------------------------------------------

if "roi_cols" in globals():
    candidate_roi_columns = [
        roi
        for roi in list(roi_cols)
        if roi in roi_subject_df.columns
    ]
else:
    candidate_roi_columns = [
        column
        for column in roi_subject_df.columns
        if column != SUBJECT_COLUMN
    ]

candidate_roi_columns = [
    column
    for column in candidate_roi_columns
    if column not in MOTOR_COVARIATES
]

# Convert candidate ROIs to numeric.
for column in candidate_roi_columns:
    roi_subject_df[column] = pd.to_numeric(
        roi_subject_df[column],
        errors="coerce"
    )

# Retain columns with at least some numeric variation.
roi_columns = []

for column in candidate_roi_columns:
    values = roi_subject_df[column]

    if (
        values.notna().sum() >= MINIMUM_SUBJECTS
        and values.std(ddof=1) > 1e-12
    ):
        roi_columns.append(column)

if len(roi_columns) < 2:
    raise ValueError(
        "Fewer than two usable substantive ROI columns were found.\n\n"
        f"Candidate ROI columns were:\n{candidate_roi_columns}"
    )

# One row per participant.
roi_subject_df = (
    roi_subject_df[
        [SUBJECT_COLUMN] + roi_columns
    ]
    .groupby(
        SUBJECT_COLUMN,
        as_index=False
    )[roi_columns]
    .mean()
)

required_motor_columns = [
    SUBJECT_COLUMN,
    OUTCOME_COLUMN,
    *MOTOR_COVARIATES
]

missing_motor_columns = [
    column
    for column in required_motor_columns
    if column not in motor_subject_df.columns
]

if missing_motor_columns:
    raise ValueError(
        "subject_level_motor_control_df is missing required columns:\n"
        f"{missing_motor_columns}"
    )

motor_subject_df = motor_subject_df[
    required_motor_columns
].copy()

for column in [
    OUTCOME_COLUMN,
    *MOTOR_COVARIATES
]:
    motor_subject_df[column] = pd.to_numeric(
        motor_subject_df[column],
        errors="coerce"
    )

motor_subject_df = (
    motor_subject_df
    .groupby(
        SUBJECT_COLUMN,
        as_index=False
    )
    .mean(numeric_only=True)
)

analysis_05_df = pd.merge(
    motor_subject_df,
    roi_subject_df,
    on=SUBJECT_COLUMN,
    how="inner",
    validate="one_to_one"
)

analysis_05_df = (
    analysis_05_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna(
        subset=[
            OUTCOME_COLUMN,
            *MOTOR_COVARIATES,
            *roi_columns
        ]
    )
    .sort_values(
        SUBJECT_COLUMN
    )
    .reset_index(drop=True)
)

if len(analysis_05_df) < MINIMUM_SUBJECTS:
    raise ValueError(
        "Too few complete participants remained for Analysis 05.\n"
        f"Complete-case N = {len(analysis_05_df)}"
    )

print("\n" + "=" * 100)
print("ANALYSIS 05 SAMPLE")
print("=" * 100)

print(
    f"Complete-case subjects: {len(analysis_05_df)}"
)

print(
    f"Substantive ROI predictors: {len(roi_columns)}"
)

print("\nROI predictors:")
for roi in roi_columns:
    print(f"  {roi}")

print("\nMotor nuisance variables:")
for variable in MOTOR_COVARIATES:
    print(f"  {variable}")

analysis_05_df.to_csv(
    OOF_NBI_DATA_DIR
    / "analysis_05_complete_case_subject_data.csv",
    index=False
)

pd.DataFrame(
    {
        "ROI": roi_columns
    }
).to_csv(
    OOF_NBI_TABLE_DIR
    / "analysis_05_roi_predictor_list.csv",
    index=False
)


# ============================================================
# 6. RUN OBSERVED NESTED OUT-OF-FOLD ANALYSIS
# ============================================================

print("\n" + "=" * 100)
print("RUNNING OBSERVED NESTED MOTOR-RESIDUALIZED LASSO")
print("=" * 100)

observed_nested_results = (
    run_nested_motor_residualized_lasso(
        dataframe=analysis_05_df,
        roi_columns=roi_columns,
        outcome_column=OUTCOME_COLUMN,
        motor_columns=MOTOR_COVARIATES,
        alpha_grid=LASSO_ALPHAS,
        inner_folds=INNER_CV_FOLDS,
        inner_random_state=INNER_CV_RANDOM_STATE,
        max_iter=LASSO_MAX_ITER,
        coefficient_tolerance=(
            COEFFICIENT_ZERO_TOLERANCE
        ),
        return_fold_details=True
    )
)


# ============================================================
# 7. BUILD SUBJECT-LEVEL OOF PREDICTION TABLE
# ============================================================

motor_residualized_lasso_oof_predictions_df = (
    observed_nested_results[
        "Fold_Details"
    ]
    .copy()
)

# Add original raw performance and motor variables for later
# residual-beta regression.
prediction_merge_columns = [
    SUBJECT_COLUMN,
    OUTCOME_COLUMN,
    *MOTOR_COVARIATES
]

prediction_covariate_df = (
    analysis_05_df[
        prediction_merge_columns
    ]
    .rename(
        columns={
            SUBJECT_COLUMN: "Heldout_Subject"
        }
    )
)

motor_residualized_lasso_oof_predictions_df = (
    motor_residualized_lasso_oof_predictions_df
    .merge(
        prediction_covariate_df,
        on="Heldout_Subject",
        how="left",
        validate="one_to_one"
    )
)

motor_residualized_lasso_oof_predictions_df.to_csv(
    OOF_NBI_DATA_DIR
    / "motor_residualized_lasso_oof_predictions.csv",
    index=False
)


# ============================================================
# 8. CALCULATE OBSERVED OOF METRICS
# ============================================================

observed_metric_rows = []

prediction_definitions = {
    "Nested_Motor_Residualized_LASSO": (
        "Predicted_LASSO_Performance_z"
    ),

    "Fold_Derived_LASSO_NBI": (
        "Predicted_Fold_Derived_NBI_Performance_z"
    ),

    "Fixed_Original_NBI_After_Motor_Residualization": (
        "Predicted_Original_Fixed_NBI_Performance_z"
    ),

    "Fixed_Full_LASSO_NBI_After_Motor_Residualization": (
        "Predicted_Full_Fixed_NBI_Performance_z"
    )
}

observed_column = (
    "Observed_Motor_Adjusted_Performance_z"
)

for analysis_name, prediction_column in (
    prediction_definitions.items()
):
    metrics = calculate_cross_validated_metrics(
        observed=(
            motor_residualized_lasso_oof_predictions_df[
                observed_column
            ]
        ),
        predicted=(
            motor_residualized_lasso_oof_predictions_df[
                prediction_column
            ]
        )
    )

    observed_metric_rows.append(
        {
            "Analysis": analysis_name,
            **metrics
        }
    )

motor_residualized_lasso_summary_df = pd.DataFrame(
    observed_metric_rows
)


# ============================================================
# 9. FOLD COEFFICIENTS AND ROI-SELECTION STABILITY
# ============================================================

motor_residualized_lasso_fold_coefficients_df = (
    observed_nested_results[
        "Coefficient_Details"
    ]
    .copy()
)

motor_residualized_lasso_fold_coefficients_df.to_csv(
    OOF_NBI_DATA_DIR
    / "motor_residualized_lasso_fold_coefficients.csv",
    index=False
)

if not motor_residualized_lasso_fold_coefficients_df.empty:

    motor_residualized_lasso_selection_stability_df = (
        motor_residualized_lasso_fold_coefficients_df
        .groupby("ROI")
        .agg(
            N_Folds=(
                "Outer_Fold",
                "nunique"
            ),
            Selection_Frequency=(
                "Selected",
                "mean"
            ),
            Positive_Selection_Frequency=(
                "Coefficient",
                lambda values: np.mean(
                    np.asarray(values)
                    > COEFFICIENT_ZERO_TOLERANCE
                )
            ),
            Negative_Selection_Frequency=(
                "Coefficient",
                lambda values: np.mean(
                    np.asarray(values)
                    < -COEFFICIENT_ZERO_TOLERANCE
                )
            ),
            Mean_Coefficient=(
                "Coefficient",
                "mean"
            ),
            Median_Coefficient=(
                "Coefficient",
                "median"
            ),
            Coefficient_SD=(
                "Coefficient",
                "std"
            ),
            Mean_Absolute_Coefficient=(
                "Absolute_Coefficient",
                "mean"
            )
        )
        .reset_index()
    )

    motor_residualized_lasso_selection_stability_df[
        "Dominant_Sign"
    ] = np.where(
        motor_residualized_lasso_selection_stability_df[
            "Positive_Selection_Frequency"
        ]
        >
        motor_residualized_lasso_selection_stability_df[
            "Negative_Selection_Frequency"
        ],
        "Positive",
        np.where(
            motor_residualized_lasso_selection_stability_df[
                "Negative_Selection_Frequency"
            ]
            >
            motor_residualized_lasso_selection_stability_df[
                "Positive_Selection_Frequency"
            ],
            "Negative",
            "Mixed_or_Zero"
        )
    )

    motor_residualized_lasso_selection_stability_df = (
        motor_residualized_lasso_selection_stability_df
        .sort_values(
            [
                "Selection_Frequency",
                "Mean_Absolute_Coefficient"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

else:
    motor_residualized_lasso_selection_stability_df = (
        pd.DataFrame()
    )

motor_residualized_lasso_selection_stability_df.to_csv(
    OOF_NBI_TABLE_DIR
    / "motor_residualized_lasso_roi_selection_stability.csv",
    index=False
)


# ============================================================
# 10. TEST OOF NBI AGAINST RAW PERFORMANCE WITH MOTOR
#     COVARIATES INCLUDED
#
# This is a complementary subject-level regression:
#
# Performance ~ OOF fold-derived NBI
#             + MotorPC1 + MotorPC2 + MotorPC3
#             + Left motor ROI ISC
#
# Because each subject's NBI was constructed without using that
# subject in LASSO selection or coefficient estimation, this
# provides an out-of-fold NBI residual-beta test.
# ============================================================

oof_regression_df = (
    motor_residualized_lasso_oof_predictions_df[
        [
            "Heldout_Subject",
            OUTCOME_COLUMN,
            "Fold_Derived_NBI_Score",
            "Full_Fixed_NBI_Score",
            "Original_Fixed_NBI_Score",
            *MOTOR_COVARIATES
        ]
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .copy()
)

oof_regression_rows = []
oof_model_objects = {}

oof_score_definitions = {
    "Fold_Derived_LASSO_NBI": (
        "Fold_Derived_NBI_Score"
    ),
    "Fixed_Full_LASSO_NBI": (
        "Full_Fixed_NBI_Score"
    ),
    "Fixed_Original_NBI": (
        "Original_Fixed_NBI_Score"
    )
}

for score_name, score_column in (
    oof_score_definitions.items()
):

    model_df = (
        oof_regression_df[
            [
                OUTCOME_COLUMN,
                score_column,
                *MOTOR_COVARIATES
            ]
        ]
        .dropna()
        .copy()
    )

    if len(model_df) < MINIMUM_SUBJECTS:
        continue

    # Standardize within the final complete-case comparison.
    standardized_df = pd.DataFrame(
        index=model_df.index
    )

    standardization_columns = [
        OUTCOME_COLUMN,
        score_column,
        *MOTOR_COVARIATES
    ]

    for column in standardization_columns:
        values = pd.to_numeric(
            model_df[column],
            errors="coerce"
        )

        standard_deviation = values.std(
            ddof=1
        )

        if (
            not np.isfinite(
                standard_deviation
            )
            or standard_deviation <= 1e-12
        ):
            standardized_df[column] = np.nan
        else:
            standardized_df[column] = (
                values - values.mean()
            ) / standard_deviation

    standardized_df = standardized_df.dropna()

    motor_formula = (
        f"{OUTCOME_COLUMN} ~ "
        + " + ".join(MOTOR_COVARIATES)
    )

    combined_formula = (
        f"{OUTCOME_COLUMN} ~ "
        + " + ".join(
            MOTOR_COVARIATES
            + [score_column]
        )
    )

    motor_only_model = smf.ols(
        motor_formula,
        data=standardized_df
    ).fit(cov_type="HC3")

    combined_model = smf.ols(
        combined_formula,
        data=standardized_df
    ).fit(cov_type="HC3")

    # Classical OLS versions for valid nested F comparison.
    motor_only_model_classical = smf.ols(
        motor_formula,
        data=standardized_df
    ).fit()

    combined_model_classical = smf.ols(
        combined_formula,
        data=standardized_df
    ).fit()

    nested_f, nested_p, nested_df = (
        combined_model_classical.compare_f_test(
            motor_only_model_classical
        )
    )

    reduced_sse = np.sum(
        motor_only_model_classical.resid ** 2
    )

    full_sse = np.sum(
        combined_model_classical.resid ** 2
    )

    partial_r2 = (
        (reduced_sse - full_sse)
        / reduced_sse
        if reduced_sse > 0
        else np.nan
    )

    confidence_interval = (
        combined_model
        .conf_int()
        .loc[score_column]
    )

    oof_regression_rows.append(
        {
            "OOF_Index": score_name,
            "Score_Column": score_column,
            "N": len(standardized_df),
            "Standardized_Beta": (
                combined_model.params[
                    score_column
                ]
            ),
            "HC3_SE": (
                combined_model.bse[
                    score_column
                ]
            ),
            "HC3_T": (
                combined_model.tvalues[
                    score_column
                ]
            ),
            "HC3_P": (
                combined_model.pvalues[
                    score_column
                ]
            ),
            "CI_Lower": (
                confidence_interval.iloc[0]
            ),
            "CI_Upper": (
                confidence_interval.iloc[1]
            ),
            "Motor_Only_R2": (
                motor_only_model_classical.rsquared
            ),
            "Combined_R2": (
                combined_model_classical.rsquared
            ),
            "Incremental_R2": (
                combined_model_classical.rsquared
                - motor_only_model_classical.rsquared
            ),
            "Partial_R2": partial_r2,
            "Nested_F": nested_f,
            "Nested_DF": nested_df,
            "Nested_P": nested_p
        }
    )

    oof_model_objects[
        score_name
    ] = {
        "MotorOnly": motor_only_model,
        "Combined": combined_model,
        "Data": standardized_df
    }

    with open(
        OOF_NBI_MODEL_DIR
        / (
            f"{safe_filename(score_name)}"
            "_motor_adjusted_regression.txt"
        ),
        "w",
        encoding="utf-8"
    ) as model_file:
        model_file.write(
            f"OOF index: {score_name}\n"
            f"Formula: {combined_formula}\n\n"
        )
        model_file.write(
            combined_model.summary().as_text()
        )

motor_residualized_oof_nbi_regression_df = (
    pd.DataFrame(
        oof_regression_rows
    )
)

motor_residualized_oof_nbi_regression_df.to_csv(
    OOF_NBI_TABLE_DIR
    / "oof_nbi_motor_adjusted_regression_results.csv",
    index=False
)


# ============================================================
# 11. PERMUTATION TEST OF THE COMPLETE NESTED PIPELINE
# ============================================================

permutation_results_df = pd.DataFrame()

if RUN_PERMUTATIONS:

    print("\n" + "=" * 100)
    print(
        "RUNNING COMPLETE-PIPELINE PERMUTATION TEST"
    )
    print("=" * 100)

    permutation_rng = np.random.default_rng(
        PERMUTATION_RANDOM_STATE
    )

    observed_y = (
        analysis_05_df[
            OUTCOME_COLUMN
        ]
        .to_numpy(dtype=float)
    )

    permutation_rows = []

    for permutation_index in range(
        N_PERMUTATIONS
    ):

        permuted_df = analysis_05_df.copy()

        permuted_df[
            OUTCOME_COLUMN
        ] = permutation_rng.permutation(
            observed_y
        )

        permuted_results = (
            run_nested_motor_residualized_lasso(
                dataframe=permuted_df,
                roi_columns=roi_columns,
                outcome_column=OUTCOME_COLUMN,
                motor_columns=MOTOR_COVARIATES,
                alpha_grid=LASSO_ALPHAS,
                inner_folds=INNER_CV_FOLDS,
                inner_random_state=(
                    INNER_CV_RANDOM_STATE
                ),
                max_iter=LASSO_MAX_ITER,
                coefficient_tolerance=(
                    COEFFICIENT_ZERO_TOLERANCE
                ),
                return_fold_details=False
            )
        )

        permutation_prediction_df = (
            permuted_results[
                "Fold_Details"
            ]
        )

        permutation_observed = (
            permutation_prediction_df[
                "Observed_Motor_Adjusted_Performance_z"
            ]
        )

        permutation_prediction_columns = {
            "Nested_Motor_Residualized_LASSO": (
                "Predicted_LASSO_Performance_z"
            ),
            "Fold_Derived_LASSO_NBI": (
                "Predicted_Fold_Derived_NBI_Performance_z"
            ),
            "Fixed_Original_NBI_After_Motor_Residualization": (
                "Predicted_Original_Fixed_NBI_Performance_z"
            ),
            "Fixed_Full_LASSO_NBI_After_Motor_Residualization": (
                "Predicted_Full_Fixed_NBI_Performance_z"
            )
        }

        for (
            analysis_name,
            prediction_column
        ) in permutation_prediction_columns.items():

            metrics = (
                calculate_cross_validated_metrics(
                    observed=permutation_observed,
                    predicted=(
                        permutation_prediction_df[
                            prediction_column
                        ]
                    )
                )
            )

            permutation_rows.append(
                {
                    "Permutation": (
                        permutation_index + 1
                    ),
                    "Analysis": analysis_name,
                    "CV_R2": metrics["CV_R2"],
                    "Pearson_r": (
                        metrics["Pearson_r"]
                    )
                }
            )

        if (
            (permutation_index + 1)
            % PERMUTATION_PROGRESS_EVERY
            == 0
            or (
                permutation_index + 1
                == N_PERMUTATIONS
            )
        ):
            print(
                "Completed "
                f"{permutation_index + 1}"
                f"/{N_PERMUTATIONS} permutations."
            )

    permutation_results_df = pd.DataFrame(
        permutation_rows
    )

    permutation_results_df.to_csv(
        OOF_NBI_PERMUTATION_DIR
        / "complete_pipeline_permutation_results.csv",
        index=False
    )

    permutation_summary_rows = []

    for analysis_name in (
        motor_residualized_lasso_summary_df[
            "Analysis"
        ]
    ):

        observed_row = (
            motor_residualized_lasso_summary_df[
                motor_residualized_lasso_summary_df[
                    "Analysis"
                ] == analysis_name
            ]
            .iloc[0]
        )

        null_df = permutation_results_df[
            permutation_results_df[
                "Analysis"
            ] == analysis_name
        ]

        observed_r2 = observed_row[
            "CV_R2"
        ]

        observed_r = observed_row[
            "Pearson_r"
        ]

        valid_null_r2 = (
            null_df["CV_R2"]
            .dropna()
            .to_numpy(dtype=float)
        )

        valid_null_r = (
            null_df["Pearson_r"]
            .dropna()
            .to_numpy(dtype=float)
        )

        permutation_p_r2 = (
            (
                1
                + np.sum(
                    valid_null_r2
                    >= observed_r2
                )
            )
            /
            (
                len(valid_null_r2)
                + 1
            )
            if (
                np.isfinite(observed_r2)
                and len(valid_null_r2) > 0
            )
            else np.nan
        )

        permutation_p_r = (
            (
                1
                + np.sum(
                    np.abs(valid_null_r)
                    >= abs(observed_r)
                )
            )
            /
            (
                len(valid_null_r)
                + 1
            )
            if (
                np.isfinite(observed_r)
                and len(valid_null_r) > 0
            )
            else np.nan
        )

        permutation_summary_rows.append(
            {
                "Analysis": analysis_name,
                "Observed_CV_R2": observed_r2,
                "Permutation_P_CV_R2": (
                    permutation_p_r2
                ),
                "Observed_Pearson_r": (
                    observed_r
                ),
                "Permutation_P_Pearson_r": (
                    permutation_p_r
                ),
                "N_Valid_Permutations_R2": (
                    len(valid_null_r2)
                ),
                "N_Valid_Permutations_r": (
                    len(valid_null_r)
                ),
                "Null_R2_Mean": (
                    np.mean(valid_null_r2)
                    if len(valid_null_r2) > 0
                    else np.nan
                ),
                "Null_R2_95th_Percentile": (
                    np.percentile(
                        valid_null_r2,
                        95
                    )
                    if len(valid_null_r2) > 0
                    else np.nan
                ),
                "Null_r_Mean": (
                    np.mean(valid_null_r)
                    if len(valid_null_r) > 0
                    else np.nan
                ),
                "Null_Absolute_r_95th_Percentile": (
                    np.percentile(
                        np.abs(valid_null_r),
                        95
                    )
                    if len(valid_null_r) > 0
                    else np.nan
                )
            }
        )

    permutation_summary_df = pd.DataFrame(
        permutation_summary_rows
    )

    motor_residualized_lasso_summary_df = (
        motor_residualized_lasso_summary_df
        .merge(
            permutation_summary_df[
                [
                    "Analysis",
                    "Permutation_P_CV_R2",
                    "Permutation_P_Pearson_r",
                    "N_Valid_Permutations_R2",
                    "N_Valid_Permutations_r",
                    "Null_R2_Mean",
                    "Null_R2_95th_Percentile",
                    "Null_r_Mean",
                    "Null_Absolute_r_95th_Percentile"
                ]
            ],
            on="Analysis",
            how="left",
            validate="one_to_one"
        )
    )


# ============================================================
# 12. SAVE FINAL SUMMARY TABLE
# ============================================================

motor_residualized_lasso_summary_df.to_csv(
    OOF_NBI_TABLE_DIR
    / "motor_residualized_nested_oof_summary.csv",
    index=False
)


# ============================================================
# 13. FIGURE: OBSERVED VS OOF PREDICTED PERFORMANCE
# ============================================================

for (
    analysis_name,
    prediction_column
) in prediction_definitions.items():

    plot_df = (
        motor_residualized_lasso_oof_predictions_df[
            [
                observed_column,
                prediction_column
            ]
        ]
        .dropna()
        .copy()
    )

    if len(plot_df) < 4:
        continue

    observed_values = plot_df[
        observed_column
    ].to_numpy(dtype=float)

    predicted_values = plot_df[
        prediction_column
    ].to_numpy(dtype=float)

    metrics = calculate_cross_validated_metrics(
        observed=observed_values,
        predicted=predicted_values
    )

    figure, axis = plt.subplots(
        figsize=(7.5, 6.5)
    )

    axis.scatter(
        observed_values,
        predicted_values,
        alpha=0.80,
        s=65
    )

    minimum_value = min(
        np.min(observed_values),
        np.min(predicted_values)
    )

    maximum_value = max(
        np.max(observed_values),
        np.max(predicted_values)
    )

    axis.plot(
        [
            minimum_value,
            maximum_value
        ],
        [
            minimum_value,
            maximum_value
        ],
        linestyle="--",
        linewidth=1.5,
        label="Identity line"
    )

    axis.set_xlabel(
        "Observed motor-adjusted performance (z)"
    )

    axis.set_ylabel(
        "Out-of-fold predicted motor-adjusted performance (z)"
    )

    axis.set_title(
        f"{analysis_name.replace('_', ' ')}\n"
        f"CV $R^2$ = {metrics['CV_R2']:.3f}, "
        f"$r$ = {metrics['Pearson_r']:.3f}"
    )

    axis.legend(
        frameon=False
    )

    figure.tight_layout()

    figure.savefig(
        OOF_NBI_FIGURE_DIR
        / (
            f"{safe_filename(analysis_name)}"
            "_observed_vs_predicted.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(figure)


# ============================================================
# 14. FIGURE: ROI-SELECTION STABILITY
# ============================================================

if (
    not
    motor_residualized_lasso_selection_stability_df
    .empty
):

    selection_plot_df = (
        motor_residualized_lasso_selection_stability_df
        .sort_values(
            "Selection_Frequency",
            ascending=True
        )
        .copy()
    )

    figure, axis = plt.subplots(
        figsize=(
            9,
            max(
                6,
                0.45 * len(selection_plot_df)
            )
        )
    )

    axis.barh(
        selection_plot_df["ROI"],
        selection_plot_df[
            "Selection_Frequency"
        ]
    )

    axis.axvline(
        0.50,
        linestyle="--",
        linewidth=1.5,
        label="Selected in 50% of folds"
    )

    axis.set_xlim(
        0,
        1
    )

    axis.set_xlabel(
        "Selection frequency across outer LOOCV folds"
    )

    axis.set_ylabel(
        "ROI"
    )

    axis.set_title(
        "Motor-residualized LASSO ROI-selection stability"
    )

    axis.legend(
        frameon=False
    )

    figure.tight_layout()

    figure.savefig(
        OOF_NBI_FIGURE_DIR
        / "motor_residualized_lasso_selection_frequency.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(figure)


# ============================================================
# 15. FIGURE: MEAN FOLD COEFFICIENTS
# ============================================================

if (
    not
    motor_residualized_lasso_selection_stability_df
    .empty
):

    coefficient_plot_df = (
        motor_residualized_lasso_selection_stability_df
        .sort_values(
            "Mean_Coefficient"
        )
        .copy()
    )

    figure, axis = plt.subplots(
        figsize=(
            9,
            max(
                6,
                0.45 * len(coefficient_plot_df)
            )
        )
    )

    axis.barh(
        coefficient_plot_df["ROI"],
        coefficient_plot_df[
            "Mean_Coefficient"
        ]
    )

    axis.axvline(
        0,
        linewidth=1
    )

    axis.set_xlabel(
        "Mean standardized LASSO coefficient across folds"
    )

    axis.set_ylabel(
        "ROI"
    )

    axis.set_title(
        "Motor-residualized LASSO coefficient pattern"
    )

    figure.tight_layout()

    figure.savefig(
        OOF_NBI_FIGURE_DIR
        / "motor_residualized_lasso_mean_coefficients.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(figure)


# ============================================================
# 16. FIGURE: FOLD-DERIVED OOF NBI VS MOTOR-ADJUSTED
#     PERFORMANCE
# ============================================================

nbi_scatter_df = (
    motor_residualized_lasso_oof_predictions_df[
        [
            "Fold_Derived_NBI_Score",
            observed_column
        ]
    ]
    .dropna()
    .copy()
)

if len(nbi_scatter_df) >= 4:

    nbi_r, nbi_p = safe_pearsonr(
        nbi_scatter_df[
            "Fold_Derived_NBI_Score"
        ],
        nbi_scatter_df[
            observed_column
        ]
    )

    figure, axis = plt.subplots(
        figsize=(7.5, 6.5)
    )

    axis.scatter(
        nbi_scatter_df[
            "Fold_Derived_NBI_Score"
        ],
        nbi_scatter_df[
            observed_column
        ],
        alpha=0.80,
        s=65
    )

    regression_coefficients = np.polyfit(
        nbi_scatter_df[
            "Fold_Derived_NBI_Score"
        ],
        nbi_scatter_df[
            observed_column
        ],
        deg=1
    )

    x_line = np.linspace(
        nbi_scatter_df[
            "Fold_Derived_NBI_Score"
        ].min(),
        nbi_scatter_df[
            "Fold_Derived_NBI_Score"
        ].max(),
        200
    )

    y_line = (
        regression_coefficients[0]
        * x_line
        + regression_coefficients[1]
    )

    axis.plot(
        x_line,
        y_line,
        linewidth=2
    )

    axis.set_xlabel(
        "Out-of-fold fold-derived motor-residualized NBI"
    )

    axis.set_ylabel(
        "Observed motor-adjusted performance (z)"
    )

    axis.set_title(
        "Fold-derived Network Balance after motor residualization\n"
        f"$r$ = {nbi_r:.3f}, $p$ = {nbi_p:.4f}"
    )

    figure.tight_layout()

    figure.savefig(
        OOF_NBI_FIGURE_DIR
        / "fold_derived_oof_nbi_vs_motor_adjusted_performance.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(figure)


# ============================================================
# 17. FIGURE: PERMUTATION NULLS
# ============================================================

if (
    RUN_PERMUTATIONS
    and not permutation_results_df.empty
):

    for analysis_name in (
        motor_residualized_lasso_summary_df[
            "Analysis"
        ]
    ):

        observed_row = (
            motor_residualized_lasso_summary_df[
                motor_residualized_lasso_summary_df[
                    "Analysis"
                ] == analysis_name
            ]
            .iloc[0]
        )

        null_df = (
            permutation_results_df[
                permutation_results_df[
                    "Analysis"
                ] == analysis_name
            ]
        )

        # CV R² null
        figure, axis = plt.subplots(
            figsize=(7.5, 6)
        )

        axis.hist(
            null_df["CV_R2"].dropna(),
            bins=30,
            alpha=0.80
        )

        axis.axvline(
            observed_row["CV_R2"],
            linestyle="--",
            linewidth=2,
            label=(
                "Observed CV "
                f"$R^2$ = "
                f"{observed_row['CV_R2']:.3f}"
            )
        )

        axis.set_xlabel(
            "Permuted complete-pipeline CV $R^2$"
        )

        axis.set_ylabel(
            "Frequency"
        )

        axis.set_title(
            f"{analysis_name.replace('_', ' ')}\n"
            "Complete-pipeline permutation null"
        )

        axis.legend(
            frameon=False
        )

        figure.tight_layout()

        figure.savefig(
            OOF_NBI_PERMUTATION_DIR
            / (
                f"{safe_filename(analysis_name)}"
                "_permutation_null_cv_r2.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(figure)

        # Pearson correlation null
        figure, axis = plt.subplots(
            figsize=(7.5, 6)
        )

        axis.hist(
            null_df["Pearson_r"].dropna(),
            bins=30,
            alpha=0.80
        )

        axis.axvline(
            observed_row["Pearson_r"],
            linestyle="--",
            linewidth=2,
            label=(
                "Observed "
                f"$r$ = "
                f"{observed_row['Pearson_r']:.3f}"
            )
        )

        axis.axvline(
            -observed_row["Pearson_r"],
            linestyle="--",
            linewidth=1
        )

        axis.set_xlabel(
            "Permuted observed–predicted correlation"
        )

        axis.set_ylabel(
            "Frequency"
        )

        axis.set_title(
            f"{analysis_name.replace('_', ' ')}\n"
            "Complete-pipeline permutation null"
        )

        axis.legend(
            frameon=False
        )

        figure.tight_layout()

        figure.savefig(
            OOF_NBI_PERMUTATION_DIR
            / (
                f"{safe_filename(analysis_name)}"
                "_permutation_null_correlation.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(figure)


# ============================================================
# 18. SAVE ANALYSIS SETTINGS
# ============================================================

analysis_05_settings = {
    "Complete_Case_N": int(
        len(analysis_05_df)
    ),
    "N_ROI_Predictors": int(
        len(roi_columns)
    ),
    "ROI_Predictors": roi_columns,
    "Motor_Covariates": MOTOR_COVARIATES,
    "Outer_CV": "LeaveOneOut",
    "Inner_CV_Folds": int(
        INNER_CV_FOLDS
    ),
    "Inner_CV_Shuffled": True,
    "Alpha_Minimum": float(
        np.min(LASSO_ALPHAS)
    ),
    "Alpha_Maximum": float(
        np.max(LASSO_ALPHAS)
    ),
    "N_Alpha_Values": int(
        len(LASSO_ALPHAS)
    ),
    "Run_Permutations": bool(
        RUN_PERMUTATIONS
    ),
    "N_Permutations": int(
        N_PERMUTATIONS
        if RUN_PERMUTATIONS
        else 0
    ),
    "Random_State": int(
        INNER_CV_RANDOM_STATE
    ),
    "Coefficient_Zero_Tolerance": float(
        COEFFICIENT_ZERO_TOLERANCE
    )
}

with open(
    OOF_NBI_OUTPUT_DIR
    / "analysis_05_settings.json",
    "w",
    encoding="utf-8"
) as settings_file:
    json.dump(
        analysis_05_settings,
        settings_file,
        indent=4
    )


# ============================================================
# 19. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 110)
print(
    "ANALYSIS 05 COMPLETE:"
    " OUT-OF-FOLD MOTOR-RESIDUALIZED NBI"
)
print("=" * 110)

print("\nNested out-of-fold prediction summary:")
display(
    motor_residualized_lasso_summary_df
    .round(5)
)

print(
    "\nOut-of-fold NBI regression after retaining "
    "motor covariates:"
)

if (
    motor_residualized_oof_nbi_regression_df
    .empty
):
    print(
        "No valid OOF NBI regression was available."
    )
else:
    display(
        motor_residualized_oof_nbi_regression_df
        .round(5)
    )

print("\nROI-selection stability:")
if (
    motor_residualized_lasso_selection_stability_df
    .empty
):
    print(
        "No coefficient stability table was generated."
    )
else:
    display(
        motor_residualized_lasso_selection_stability_df[
            [
                "ROI",
                "Selection_Frequency",
                "Positive_Selection_Frequency",
                "Negative_Selection_Frequency",
                "Mean_Coefficient",
                "Median_Coefficient",
                "Dominant_Sign"
            ]
        ]
        .round(4)
    )

print("\nOuter-fold details:")
display(
    observed_nested_results[
        "Fold_Details"
    ]
    .round(5)
)

print(
    "\nPrimary interpretation targets:\n"
    "  1. Nested_Motor_Residualized_LASSO:\n"
    "     Tests whether motor-residualized ROI ISC predicts\n"
    "     held-out motor-adjusted performance.\n\n"
    "  2. Fold_Derived_LASSO_NBI:\n"
    "     Tests a newly reconstructed NBI whose positive and\n"
    "     negative ROI memberships are determined separately\n"
    "     inside every training fold.\n\n"
    "  3. Fixed_Full_LASSO_NBI_After_Motor_Residualization:\n"
    "     Tests whether the previously observed full-LASSO\n"
    "     architecture predicts after both ROI and performance\n"
    "     have been motor-residualized inside each fold.\n\n"
    "  4. OOF NBI motor-adjusted regression:\n"
    "     Tests whether the genuinely out-of-fold NBI remains\n"
    "     associated with raw performance when MotorPC1–3 and\n"
    "     left motor ROI ISC are included as covariates."
)

print(
    "\nCreated notebook objects:\n"
    "  analysis_05_df\n"
    "  motor_residualized_lasso_oof_predictions_df\n"
    "  motor_residualized_lasso_summary_df\n"
    "  motor_residualized_lasso_fold_coefficients_df\n"
    "  motor_residualized_lasso_selection_stability_df\n"
    "  motor_residualized_oof_nbi_regression_df\n"
    "  observed_nested_results\n"
    "  permutation_results_df\n"
    "  oof_model_objects"
)

print(
    "\nSaved all outputs to:\n"
    f"{OOF_NBI_OUTPUT_DIR}"
)

In [ ]:
# ============================================================
#  NETWORK BALANCE ARCHITECTURE STABILITY
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Evaluate whether the ROI architecture underlying the Network
# Balance Index is reproducible rather than dependent on one
# full-sample LASSO fit.
#
# ANALYSES
# ------------------------------------------------------------
# 1. Full-sample reference LASSO
#
# 2. Leave-one-subject-out coefficient stability:
#       - hold one participant out
#       - standardize using training participants only
#       - retune LASSO alpha within the training sample
#       - fit the training-fold LASSO
#       - save every ROI coefficient and its direction
#
# 3. Sign and selection consistency:
#       - selected-frequency
#       - positive-frequency
#       - negative-frequency
#       - expected-sign consistency
#       - mean and median coefficients
#       - bootstrap confidence intervals
#
# 4. Stability selection:
#       - repeatedly sample 70% of participants
#       - fit LASSO using a fixed reference penalty
#       - record how often every ROI is selected
#       - record positive and negative selection probabilities
#
# 5. Whole-network reproducibility:
#       - correlate every LOOCV coefficient vector with the
#         full-sample reference coefficient vector
#       - correlate coefficient vectors between folds
#       - calculate sign agreement and cosine similarity
#
# 6. Fixed-index LOOCV prediction:
#       - Original Network Balance Index
#       - Full LASSO Network Balance Index
#
# REQUIRED NOTEBOOK OBJECTS
# ------------------------------------------------------------
#   roi_isc_df:
#       Subject + one column per ROI
#
#   mean_corr_df:
#       Subject + Performance
#
# OPTIONAL NOTEBOOK OBJECT
# ------------------------------------------------------------
#   roi_cols
#
# OUTPUT FOLDER
# ------------------------------------------------------------
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\
# Emotion Perception Data\Motor_Control\06_network_stability
# ============================================================

import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cosine

from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 1. OUTPUT DIRECTORIES
# ============================================================

NETWORK_STABILITY_OUT = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\06_network_stability"
)

NETWORK_STABILITY_DATA_DIR = NETWORK_STABILITY_OUT / "data"
NETWORK_STABILITY_TABLE_DIR = NETWORK_STABILITY_OUT / "tables"
NETWORK_STABILITY_FIGURE_DIR = NETWORK_STABILITY_OUT / "figures"

for folder in [
    NETWORK_STABILITY_OUT,
    NETWORK_STABILITY_DATA_DIR,
    NETWORK_STABILITY_TABLE_DIR,
    NETWORK_STABILITY_FIGURE_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(
    "Analysis 06 outputs will be saved to:\n"
    f"{NETWORK_STABILITY_OUT}"
)


# ============================================================
# 2. USER-ADJUSTABLE SETTINGS
# ============================================================

SUBJECT_COLUMN = "Subject"
OUTCOME_COLUMN = "Performance"

# Same general alpha range used in the original notebook.
ALPHA_GRID = np.logspace(-3, 1, 100)

INNER_CV_FOLDS = 5
RANDOM_SEED = 2026
LASSO_MAX_ITER = 200000

# Coefficients smaller than this are treated as zero.
COEFFICIENT_ZERO_TOLERANCE = 1e-8

# Repeated subsample stability selection.
N_STABILITY_REPETITIONS = 2000
STABILITY_SAMPLE_FRACTION = 0.70

# Bootstrap confidence intervals for mean LOOCV coefficients.
N_BOOTSTRAP = 5000

MINIMUM_COMPLETE_SUBJECTS = 20

RNG = np.random.default_rng(RANDOM_SEED)


# ============================================================
# 3. PRESPECIFIED NETWORK ARCHITECTURES
# ============================================================

# Original four-ROI architecture.
ORIGINAL_POSITIVE_ROIS = [
    "Right VLPFC",
    "Left TPJ"
]

ORIGINAL_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Right TPJ"
]

# Full LASSO architecture used in the notebook.
FULL_LASSO_POSITIVE_ROIS = [
    "Left TPJ",
    "Right VLPFC",
    "Left OFC",
    "Left STS"
]

FULL_LASSO_NEGATIVE_ROIS = [
    "Left VLPFC",
    "Left DLPFC",
    "Right TPJ"
]

NETWORK_ARCHITECTURES = {
    "Original_Network_Balance": {
        "positive": ORIGINAL_POSITIVE_ROIS,
        "negative": ORIGINAL_NEGATIVE_ROIS
    },

    "Full_LASSO_Network_Balance": {
        "positive": FULL_LASSO_POSITIVE_ROIS,
        "negative": FULL_LASSO_NEGATIVE_ROIS
    }
}


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """Normalize participant IDs before merging."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def safe_filename(value):
    """Make a Windows-safe file name."""
    value = str(value)

    for character in '<>:"/\\|?*':
        value = value.replace(character, "_")

    return value.strip(" .")


def safe_pearsonr(x, y):
    """Pearson correlation with basic validity checks."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 4
        or np.std(x, ddof=1) <= 1e-12
        or np.std(y, ddof=1) <= 1e-12
    ):
        return np.nan, np.nan

    return pearsonr(x, y)


def safe_spearmanr(x, y):
    """Spearman correlation with basic validity checks."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 4
        or np.std(x, ddof=1) <= 1e-12
        or np.std(y, ddof=1) <= 1e-12
    ):
        return np.nan, np.nan

    result = spearmanr(x, y)

    return float(result.statistic), float(result.pvalue)


def coefficient_sign(
    coefficients,
    tolerance=COEFFICIENT_ZERO_TOLERANCE
):
    """
    Convert coefficient values to:
        +1 = positive
         0 = shrunk to zero
        -1 = negative
    """
    coefficients = np.asarray(coefficients, dtype=float)

    signs = np.zeros(
        coefficients.shape,
        dtype=int
    )

    signs[
        coefficients > tolerance
    ] = 1

    signs[
        coefficients < -tolerance
    ] = -1

    return signs


def bootstrap_mean_ci(
    values,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED
):
    """Bootstrap 95% CI for the mean."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)

    bootstrap_means = np.empty(
        n_bootstrap,
        dtype=float
    )

    for bootstrap_index in range(n_bootstrap):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True
        )

        bootstrap_means[
            bootstrap_index
        ] = np.mean(sample)

    return (
        np.percentile(bootstrap_means, 2.5),
        np.percentile(bootstrap_means, 97.5)
    )


def calculate_fixed_network_score(
    standardized_roi_matrix,
    roi_names,
    positive_rois,
    negative_rois
):
    """
    Calculate an unweighted positive-minus-negative NBI from
    standardized ROI values.
    """
    roi_names = list(roi_names)

    missing_rois = [
        roi
        for roi in positive_rois + negative_rois
        if roi not in roi_names
    ]

    if missing_rois:
        raise ValueError(
            "The following ROIs required for a fixed network "
            f"architecture are missing:\n{missing_rois}"
        )

    positive_indices = [
        roi_names.index(roi)
        for roi in positive_rois
    ]

    negative_indices = [
        roi_names.index(roi)
        for roi in negative_rois
    ]

    positive_mean = np.mean(
        standardized_roi_matrix[
            :,
            positive_indices
        ],
        axis=1
    )

    negative_mean = np.mean(
        standardized_roi_matrix[
            :,
            negative_indices
        ],
        axis=1
    )

    return positive_mean - negative_mean


def calculate_prediction_metrics(
    observed,
    predicted
):
    """Calculate out-of-sample prediction metrics."""
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    valid = (
        np.isfinite(observed)
        & np.isfinite(predicted)
    )

    observed = observed[valid]
    predicted = predicted[valid]

    if len(observed) < 4:
        return {
            "N": len(observed),
            "CV_R2": np.nan,
            "Pearson_r": np.nan,
            "Pearson_p": np.nan,
            "Spearman_rho": np.nan,
            "Spearman_p": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan
        }

    pearson_r, pearson_p = safe_pearsonr(
        observed,
        predicted
    )

    spearman_rho, spearman_p = safe_spearmanr(
        observed,
        predicted
    )

    return {
        "N": len(observed),
        "CV_R2": r2_score(
            observed,
            predicted
        ),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p,
        "RMSE": np.sqrt(
            mean_squared_error(
                observed,
                predicted
            )
        ),
        "MAE": mean_absolute_error(
            observed,
            predicted
        )
    }


# ============================================================
# 5. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "roi_isc_df",
    "mean_corr_df"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "The following required notebook objects were not found:\n"
        f"{missing_objects}\n\n"
        "Run the original ROI ISC and behavioral-performance "
        "cells before running Analysis 06."
    )


# ============================================================
# 6. PREPARE SUBJECT-LEVEL ROI AND PERFORMANCE TABLE
# ============================================================

roi_subject_df = roi_isc_df.copy()
behavior_subject_df = mean_corr_df.copy()

if SUBJECT_COLUMN not in roi_subject_df.columns:
    raise ValueError(
        "roi_isc_df does not contain a Subject column."
    )

if SUBJECT_COLUMN not in behavior_subject_df.columns:
    raise ValueError(
        "mean_corr_df does not contain a Subject column."
    )

if OUTCOME_COLUMN not in behavior_subject_df.columns:
    if {
        "Mean_Valence_Corr",
        "Mean_Arousal_Corr"
    }.issubset(behavior_subject_df.columns):

        behavior_subject_df[
            OUTCOME_COLUMN
        ] = behavior_subject_df[
            [
                "Mean_Valence_Corr",
                "Mean_Arousal_Corr"
            ]
        ].mean(axis=1)

        print(
            "Performance was not found, so it was computed as "
            "the mean of Mean_Valence_Corr and "
            "Mean_Arousal_Corr."
        )

    else:
        raise ValueError(
            "mean_corr_df must contain Performance or both "
            "Mean_Valence_Corr and Mean_Arousal_Corr."
        )

roi_subject_df[
    SUBJECT_COLUMN
] = roi_subject_df[
    SUBJECT_COLUMN
].apply(normalize_subject_id)

behavior_subject_df[
    SUBJECT_COLUMN
] = behavior_subject_df[
    SUBJECT_COLUMN
].apply(normalize_subject_id)

# Prefer the notebook's existing roi_cols variable.
if "roi_cols" in globals():
    candidate_roi_columns = [
        column
        for column in list(roi_cols)
        if column in roi_subject_df.columns
    ]

elif "roi_cols_local" in globals():
    candidate_roi_columns = [
        column
        for column in list(roi_cols_local)
        if column in roi_subject_df.columns
    ]

else:
    candidate_roi_columns = [
        column
        for column in roi_subject_df.columns
        if column != SUBJECT_COLUMN
    ]

# Convert potential ROI values to numeric.
for column in candidate_roi_columns:
    roi_subject_df[column] = pd.to_numeric(
        roi_subject_df[column],
        errors="coerce"
    )

# Keep nonconstant numeric ROI predictors.
roi_columns = []

for column in candidate_roi_columns:
    values = roi_subject_df[column]

    if (
        values.notna().sum() >= MINIMUM_COMPLETE_SUBJECTS
        and np.isfinite(values.std(ddof=1))
        and values.std(ddof=1) > 1e-12
    ):
        roi_columns.append(column)

if len(roi_columns) < 2:
    raise ValueError(
        "Fewer than two usable ROI columns were found.\n"
        f"Candidate columns were:\n{candidate_roi_columns}"
    )

# Check that both prespecified NBI architectures are available.
all_required_network_rois = sorted(
    set(
        ORIGINAL_POSITIVE_ROIS
        + ORIGINAL_NEGATIVE_ROIS
        + FULL_LASSO_POSITIVE_ROIS
        + FULL_LASSO_NEGATIVE_ROIS
    )
)

missing_network_rois = [
    roi
    for roi in all_required_network_rois
    if roi not in roi_columns
]

if missing_network_rois:
    raise ValueError(
        "The following prespecified network ROIs are missing "
        "from roi_isc_df:\n"
        f"{missing_network_rois}"
    )

# One subject-level row per participant.
roi_subject_df = (
    roi_subject_df[
        [SUBJECT_COLUMN] + roi_columns
    ]
    .groupby(
        SUBJECT_COLUMN,
        as_index=False
    )[roi_columns]
    .mean()
)

behavior_subject_df[
    OUTCOME_COLUMN
] = pd.to_numeric(
    behavior_subject_df[
        OUTCOME_COLUMN
    ],
    errors="coerce"
)

behavior_subject_df = (
    behavior_subject_df[
        [
            SUBJECT_COLUMN,
            OUTCOME_COLUMN
        ]
    ]
    .groupby(
        SUBJECT_COLUMN,
        as_index=False
    )[OUTCOME_COLUMN]
    .mean()
)

network_stability_analysis_df = pd.merge(
    roi_subject_df,
    behavior_subject_df,
    on=SUBJECT_COLUMN,
    how="inner",
    validate="one_to_one"
)

network_stability_analysis_df = (
    network_stability_analysis_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna(
        subset=[
            OUTCOME_COLUMN,
            *roi_columns
        ]
    )
    .sort_values(
        SUBJECT_COLUMN
    )
    .reset_index(drop=True)
)

if (
    len(network_stability_analysis_df)
    < MINIMUM_COMPLETE_SUBJECTS
):
    raise ValueError(
        "Too few complete subjects remained.\n"
        f"Complete-case N = "
        f"{len(network_stability_analysis_df)}"
    )

network_stability_analysis_df.to_csv(
    NETWORK_STABILITY_DATA_DIR
    / "network_stability_complete_case_data.csv",
    index=False
)

pd.DataFrame(
    {
        "ROI": roi_columns
    }
).to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "network_stability_roi_list.csv",
    index=False
)

print("\n" + "=" * 90)
print("NETWORK-STABILITY SAMPLE")
print("=" * 90)
print(
    f"Complete-case subjects: "
    f"{len(network_stability_analysis_df)}"
)
print(
    f"ROI predictors: {len(roi_columns)}"
)
print("\nROIs:")
for roi in roi_columns:
    print(f"  {roi}")


# ============================================================
# 7. EXTRACT MATRICES
# ============================================================

subjects = network_stability_analysis_df[
    SUBJECT_COLUMN
].astype(str).to_numpy()

X_raw = network_stability_analysis_df[
    roi_columns
].to_numpy(dtype=float)

y_raw = network_stability_analysis_df[
    OUTCOME_COLUMN
].to_numpy(dtype=float)

n_subjects = len(subjects)
n_rois = len(roi_columns)


# ============================================================
# 8. FULL-SAMPLE REFERENCE LASSO
# ============================================================

full_X_scaler = StandardScaler()
full_y_scaler = StandardScaler()

X_full_z = full_X_scaler.fit_transform(
    X_raw
)

y_full_z = (
    full_y_scaler
    .fit_transform(
        y_raw.reshape(-1, 1)
    )
    .reshape(-1)
)

full_inner_cv = KFold(
    n_splits=min(
        INNER_CV_FOLDS,
        n_subjects
    ),
    shuffle=True,
    random_state=RANDOM_SEED
)

full_lasso_cv = LassoCV(
    alphas=ALPHA_GRID,
    cv=full_inner_cv,
    max_iter=LASSO_MAX_ITER,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

full_lasso_cv.fit(
    X_full_z,
    y_full_z
)

reference_alpha = float(
    full_lasso_cv.alpha_
)

full_lasso_model = Lasso(
    alpha=reference_alpha,
    max_iter=LASSO_MAX_ITER,
    random_state=RANDOM_SEED
)

full_lasso_model.fit(
    X_full_z,
    y_full_z
)

reference_coefficients = (
    full_lasso_model.coef_
    .astype(float)
)

reference_signs = coefficient_sign(
    reference_coefficients
)

full_sample_reference_df = pd.DataFrame(
    {
        "ROI": roi_columns,
        "Full_Sample_Coefficient": (
            reference_coefficients
        ),
        "Full_Sample_Absolute_Coefficient": (
            np.abs(reference_coefficients)
        ),
        "Full_Sample_Sign": reference_signs,
        "Full_Sample_Selected": (
            np.abs(reference_coefficients)
            > COEFFICIENT_ZERO_TOLERANCE
        )
    }
)

full_sample_reference_df[
    "Full_Sample_Direction"
] = full_sample_reference_df[
    "Full_Sample_Sign"
].map(
    {
        1: "Positive",
        0: "Zero",
        -1: "Negative"
    }
)

full_sample_reference_df = (
    full_sample_reference_df
    .sort_values(
        "Full_Sample_Absolute_Coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

full_sample_reference_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "full_sample_reference_lasso_coefficients.csv",
    index=False
)

print("\n" + "=" * 90)
print("FULL-SAMPLE REFERENCE LASSO")
print("=" * 90)
print(f"Selected alpha: {reference_alpha:.6f}")
display(
    full_sample_reference_df.round(5)
)


# ============================================================
# 9. OUTER LOOCV COEFFICIENT STABILITY
# ============================================================

outer_loo = LeaveOneOut()

loocv_coefficient_rows = []
loocv_fold_rows = []

# Matrix: folds × ROI coefficients.
loocv_coefficient_matrix = np.full(
    (n_subjects, n_rois),
    np.nan
)

for fold_index, (
    train_indices,
    test_indices
) in enumerate(
    outer_loo.split(X_raw),
    start=1
):

    heldout_index = int(
        test_indices[0]
    )

    heldout_subject = subjects[
        heldout_index
    ]

    X_train_raw = X_raw[
        train_indices
    ]

    y_train_raw = y_raw[
        train_indices
    ]

    # Training-only standardization.
    fold_X_scaler = StandardScaler()
    fold_y_scaler = StandardScaler()

    X_train_z = fold_X_scaler.fit_transform(
        X_train_raw
    )

    y_train_z = (
        fold_y_scaler
        .fit_transform(
            y_train_raw.reshape(-1, 1)
        )
        .reshape(-1)
    )

    # Retune alpha inside this outer training fold.
    effective_inner_folds = min(
        INNER_CV_FOLDS,
        len(train_indices)
    )

    fold_inner_cv = KFold(
        n_splits=effective_inner_folds,
        shuffle=True,
        random_state=(
            RANDOM_SEED + fold_index
        )
    )

    fold_lasso_cv = LassoCV(
        alphas=ALPHA_GRID,
        cv=fold_inner_cv,
        max_iter=LASSO_MAX_ITER,
        random_state=(
            RANDOM_SEED + fold_index
        ),
        n_jobs=-1
    )

    fold_lasso_cv.fit(
        X_train_z,
        y_train_z
    )

    fold_alpha = float(
        fold_lasso_cv.alpha_
    )

    fold_lasso_model = Lasso(
        alpha=fold_alpha,
        max_iter=LASSO_MAX_ITER,
        random_state=(
            RANDOM_SEED + fold_index
        )
    )

    fold_lasso_model.fit(
        X_train_z,
        y_train_z
    )

    fold_coefficients = (
        fold_lasso_model.coef_
        .astype(float)
    )

    fold_signs = coefficient_sign(
        fold_coefficients
    )

    loocv_coefficient_matrix[
        heldout_index,
        :
    ] = fold_coefficients

    # Whole-vector similarity to the full-sample coefficients.
    coefficient_r, coefficient_p = safe_pearsonr(
        fold_coefficients,
        reference_coefficients
    )

    coefficient_rho, coefficient_rho_p = safe_spearmanr(
        fold_coefficients,
        reference_coefficients
    )

    if (
        np.linalg.norm(fold_coefficients) > 1e-12
        and np.linalg.norm(reference_coefficients) > 1e-12
    ):
        cosine_similarity = (
            1
            - cosine(
                fold_coefficients,
                reference_coefficients
            )
        )
    else:
        cosine_similarity = np.nan

    union_selected = (
        (np.abs(fold_coefficients)
         > COEFFICIENT_ZERO_TOLERANCE)
        |
        (np.abs(reference_coefficients)
         > COEFFICIENT_ZERO_TOLERANCE)
    )

    if union_selected.sum() > 0:
        sign_agreement_union = np.mean(
            fold_signs[union_selected]
            ==
            reference_signs[union_selected]
        )
    else:
        sign_agreement_union = np.nan

    original_expected_signs = {
        **{
            roi: 1
            for roi in ORIGINAL_POSITIVE_ROIS
        },
        **{
            roi: -1
            for roi in ORIGINAL_NEGATIVE_ROIS
        }
    }

    full_expected_signs = {
        **{
            roi: 1
            for roi in FULL_LASSO_POSITIVE_ROIS
        },
        **{
            roi: -1
            for roi in FULL_LASSO_NEGATIVE_ROIS
        }
    }

    original_fold_agreements = []
    full_fold_agreements = []

    for roi, expected_sign in (
        original_expected_signs.items()
    ):
        roi_index = roi_columns.index(roi)

        original_fold_agreements.append(
            fold_signs[roi_index]
            == expected_sign
        )

    for roi, expected_sign in (
        full_expected_signs.items()
    ):
        roi_index = roi_columns.index(roi)

        full_fold_agreements.append(
            fold_signs[roi_index]
            == expected_sign
        )

    loocv_fold_rows.append(
        {
            "Outer_Fold": fold_index,
            "Heldout_Subject": heldout_subject,
            "Selected_Alpha": fold_alpha,
            "N_Selected_ROIs": int(
                np.sum(
                    np.abs(fold_coefficients)
                    > COEFFICIENT_ZERO_TOLERANCE
                )
            ),
            "N_Positive_ROIs": int(
                np.sum(
                    fold_coefficients
                    > COEFFICIENT_ZERO_TOLERANCE
                )
            ),
            "N_Negative_ROIs": int(
                np.sum(
                    fold_coefficients
                    < -COEFFICIENT_ZERO_TOLERANCE
                )
            ),
            "Coefficient_Vector_Pearson_r_With_Full": (
                coefficient_r
            ),
            "Coefficient_Vector_Pearson_p": (
                coefficient_p
            ),
            "Coefficient_Vector_Spearman_rho_With_Full": (
                coefficient_rho
            ),
            "Coefficient_Vector_Spearman_p": (
                coefficient_rho_p
            ),
            "Coefficient_Vector_Cosine_Similarity_With_Full": (
                cosine_similarity
            ),
            "Sign_Agreement_With_Full_Among_Union_Selected": (
                sign_agreement_union
            ),
            "Original_NBI_Expected_Sign_Proportion": (
                np.mean(
                    original_fold_agreements
                )
            ),
            "Full_LASSO_NBI_Expected_Sign_Proportion": (
                np.mean(
                    full_fold_agreements
                )
            )
        }
    )

    for roi_index, roi in enumerate(
        roi_columns
    ):
        coefficient = fold_coefficients[
            roi_index
        ]

        sign = fold_signs[
            roi_index
        ]

        loocv_coefficient_rows.append(
            {
                "Outer_Fold": fold_index,
                "Heldout_Subject": heldout_subject,
                "ROI": roi,
                "Coefficient": coefficient,
                "Absolute_Coefficient": abs(
                    coefficient
                ),
                "Sign": sign,
                "Direction": (
                    "Positive"
                    if sign == 1
                    else (
                        "Negative"
                        if sign == -1
                        else "Zero"
                    )
                ),
                "Selected": bool(
                    abs(coefficient)
                    > COEFFICIENT_ZERO_TOLERANCE
                ),
                "Selected_Alpha": fold_alpha,
                "Full_Sample_Coefficient": (
                    reference_coefficients[
                        roi_index
                    ]
                ),
                "Full_Sample_Sign": (
                    reference_signs[
                        roi_index
                    ]
                ),
                "Matches_Full_Sample_Sign": bool(
                    sign
                    == reference_signs[
                        roi_index
                    ]
                )
            }
        )

loocv_network_coefficients_df = pd.DataFrame(
    loocv_coefficient_rows
)

loocv_network_fold_similarity_df = pd.DataFrame(
    loocv_fold_rows
)

loocv_network_coefficients_df.to_csv(
    NETWORK_STABILITY_DATA_DIR
    / "loocv_fold_roi_coefficients.csv",
    index=False
)

loocv_network_fold_similarity_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "loocv_fold_network_similarity.csv",
    index=False
)


# ============================================================
# 10. SUMMARIZE ROI SELECTION AND SIGN CONSISTENCY
# ============================================================

roi_stability_rows = []

expected_sign_lookup = {}

for roi in ORIGINAL_POSITIVE_ROIS:
    expected_sign_lookup.setdefault(
        roi,
        []
    ).append(
        "Original_Positive"
    )

for roi in ORIGINAL_NEGATIVE_ROIS:
    expected_sign_lookup.setdefault(
        roi,
        []
    ).append(
        "Original_Negative"
    )

for roi in FULL_LASSO_POSITIVE_ROIS:
    expected_sign_lookup.setdefault(
        roi,
        []
    ).append(
        "Full_LASSO_Positive"
    )

for roi in FULL_LASSO_NEGATIVE_ROIS:
    expected_sign_lookup.setdefault(
        roi,
        []
    ).append(
        "Full_LASSO_Negative"
    )

for roi_index, roi in enumerate(
    roi_columns
):
    roi_values = loocv_coefficient_matrix[
        :,
        roi_index
    ]

    roi_signs = coefficient_sign(
        roi_values
    )

    selected_mask = (
        np.abs(roi_values)
        > COEFFICIENT_ZERO_TOLERANCE
    )

    positive_mask = (
        roi_values
        > COEFFICIENT_ZERO_TOLERANCE
    )

    negative_mask = (
        roi_values
        < -COEFFICIENT_ZERO_TOLERANCE
    )

    ci_lower, ci_upper = bootstrap_mean_ci(
        roi_values,
        n_bootstrap=N_BOOTSTRAP,
        seed=RANDOM_SEED + roi_index
    )

    reference_sign = reference_signs[
        roi_index
    ]

    if reference_sign != 0:
        reference_sign_consistency = np.mean(
            roi_signs == reference_sign
        )

        conditional_reference_sign_consistency = (
            np.mean(
                roi_signs[selected_mask]
                == reference_sign
            )
            if selected_mask.sum() > 0
            else np.nan
        )
    else:
        reference_sign_consistency = np.mean(
            roi_signs == 0
        )

        conditional_reference_sign_consistency = (
            np.nan
        )

    # Expected directions based on the fixed architectures.
    full_expected_sign = (
        1
        if roi in FULL_LASSO_POSITIVE_ROIS
        else (
            -1
            if roi in FULL_LASSO_NEGATIVE_ROIS
            else 0
        )
    )

    original_expected_sign = (
        1
        if roi in ORIGINAL_POSITIVE_ROIS
        else (
            -1
            if roi in ORIGINAL_NEGATIVE_ROIS
            else 0
        )
    )

    full_expected_sign_frequency = (
        np.mean(
            roi_signs == full_expected_sign
        )
        if full_expected_sign != 0
        else np.nan
    )

    original_expected_sign_frequency = (
        np.mean(
            roi_signs
            == original_expected_sign
        )
        if original_expected_sign != 0
        else np.nan
    )

    roi_stability_rows.append(
        {
            "ROI": roi,
            "Network_Membership": "; ".join(
                expected_sign_lookup.get(
                    roi,
                    ["Not_In_Fixed_NBI"]
                )
            ),
            "Full_Sample_Coefficient": (
                reference_coefficients[
                    roi_index
                ]
            ),
            "Full_Sample_Direction": (
                "Positive"
                if reference_sign == 1
                else (
                    "Negative"
                    if reference_sign == -1
                    else "Zero"
                )
            ),
            "Mean_LOOCV_Coefficient": (
                np.mean(roi_values)
            ),
            "Median_LOOCV_Coefficient": (
                np.median(roi_values)
            ),
            "SD_LOOCV_Coefficient": (
                np.std(
                    roi_values,
                    ddof=1
                )
            ),
            "Mean_Beta_CI_Lower": ci_lower,
            "Mean_Beta_CI_Upper": ci_upper,
            "LOOCV_Selection_Frequency": (
                np.mean(selected_mask)
            ),
            "LOOCV_Positive_Frequency": (
                np.mean(positive_mask)
            ),
            "LOOCV_Negative_Frequency": (
                np.mean(negative_mask)
            ),
            "LOOCV_Zero_Frequency": (
                np.mean(~selected_mask)
            ),
            "Sign_Consistency_With_Full_Sample": (
                reference_sign_consistency
            ),
            "Sign_Consistency_Given_Selected": (
                conditional_reference_sign_consistency
            ),
            "Full_LASSO_Expected_Sign_Frequency": (
                full_expected_sign_frequency
            ),
            "Original_NBI_Expected_Sign_Frequency": (
                original_expected_sign_frequency
            )
        }
    )

loocv_roi_stability_summary_df = pd.DataFrame(
    roi_stability_rows
)

loocv_roi_stability_summary_df = (
    loocv_roi_stability_summary_df
    .sort_values(
        [
            "LOOCV_Selection_Frequency",
            "Sign_Consistency_With_Full_Sample"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

loocv_roi_stability_summary_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "loocv_roi_selection_and_sign_stability.csv",
    index=False
)


# ============================================================
# 11. PAIRWISE FOLD NETWORK REPRODUCIBILITY
# ============================================================

pairwise_fold_correlation_matrix = np.full(
    (n_subjects, n_subjects),
    np.nan
)

pairwise_fold_cosine_matrix = np.full(
    (n_subjects, n_subjects),
    np.nan
)

pairwise_fold_sign_agreement_matrix = np.full(
    (n_subjects, n_subjects),
    np.nan
)

for first_fold in range(n_subjects):
    for second_fold in range(n_subjects):

        first_coefficients = (
            loocv_coefficient_matrix[
                first_fold,
                :
            ]
        )

        second_coefficients = (
            loocv_coefficient_matrix[
                second_fold,
                :
            ]
        )

        fold_r, _ = safe_pearsonr(
            first_coefficients,
            second_coefficients
        )

        pairwise_fold_correlation_matrix[
            first_fold,
            second_fold
        ] = fold_r

        if (
            np.linalg.norm(
                first_coefficients
            ) > 1e-12
            and np.linalg.norm(
                second_coefficients
            ) > 1e-12
        ):
            pairwise_fold_cosine_matrix[
                first_fold,
                second_fold
            ] = (
                1
                - cosine(
                    first_coefficients,
                    second_coefficients
                )
            )

        first_signs = coefficient_sign(
            first_coefficients
        )

        second_signs = coefficient_sign(
            second_coefficients
        )

        selected_union = (
            (first_signs != 0)
            |
            (second_signs != 0)
        )

        if selected_union.sum() > 0:
            pairwise_fold_sign_agreement_matrix[
                first_fold,
                second_fold
            ] = np.mean(
                first_signs[selected_union]
                ==
                second_signs[selected_union]
            )

pairwise_fold_correlation_df = pd.DataFrame(
    pairwise_fold_correlation_matrix,
    index=subjects,
    columns=subjects
)

pairwise_fold_cosine_df = pd.DataFrame(
    pairwise_fold_cosine_matrix,
    index=subjects,
    columns=subjects
)

pairwise_fold_sign_agreement_df = pd.DataFrame(
    pairwise_fold_sign_agreement_matrix,
    index=subjects,
    columns=subjects
)

pairwise_fold_correlation_df.to_csv(
    NETWORK_STABILITY_DATA_DIR
    / "pairwise_fold_coefficient_correlations.csv"
)

pairwise_fold_cosine_df.to_csv(
    NETWORK_STABILITY_DATA_DIR
    / "pairwise_fold_coefficient_cosine_similarity.csv"
)

pairwise_fold_sign_agreement_df.to_csv(
    NETWORK_STABILITY_DATA_DIR
    / "pairwise_fold_sign_agreement.csv"
)

upper_triangle_indices = np.triu_indices(
    n_subjects,
    k=1
)

pairwise_network_summary_df = pd.DataFrame(
    [
        {
            "Metric": (
                "Pairwise coefficient-vector Pearson r"
            ),
            "Mean": np.nanmean(
                pairwise_fold_correlation_matrix[
                    upper_triangle_indices
                ]
            ),
            "Median": np.nanmedian(
                pairwise_fold_correlation_matrix[
                    upper_triangle_indices
                ]
            ),
            "SD": np.nanstd(
                pairwise_fold_correlation_matrix[
                    upper_triangle_indices
                ],
                ddof=1
            ),
            "Minimum": np.nanmin(
                pairwise_fold_correlation_matrix[
                    upper_triangle_indices
                ]
            ),
            "Maximum": np.nanmax(
                pairwise_fold_correlation_matrix[
                    upper_triangle_indices
                ]
            )
        },
        {
            "Metric": (
                "Pairwise coefficient-vector cosine similarity"
            ),
            "Mean": np.nanmean(
                pairwise_fold_cosine_matrix[
                    upper_triangle_indices
                ]
            ),
            "Median": np.nanmedian(
                pairwise_fold_cosine_matrix[
                    upper_triangle_indices
                ]
            ),
            "SD": np.nanstd(
                pairwise_fold_cosine_matrix[
                    upper_triangle_indices
                ],
                ddof=1
            ),
            "Minimum": np.nanmin(
                pairwise_fold_cosine_matrix[
                    upper_triangle_indices
                ]
            ),
            "Maximum": np.nanmax(
                pairwise_fold_cosine_matrix[
                    upper_triangle_indices
                ]
            )
        },
        {
            "Metric": (
                "Pairwise sign agreement among selected-union ROIs"
            ),
            "Mean": np.nanmean(
                pairwise_fold_sign_agreement_matrix[
                    upper_triangle_indices
                ]
            ),
            "Median": np.nanmedian(
                pairwise_fold_sign_agreement_matrix[
                    upper_triangle_indices
                ]
            ),
            "SD": np.nanstd(
                pairwise_fold_sign_agreement_matrix[
                    upper_triangle_indices
                ],
                ddof=1
            ),
            "Minimum": np.nanmin(
                pairwise_fold_sign_agreement_matrix[
                    upper_triangle_indices
                ]
            ),
            "Maximum": np.nanmax(
                pairwise_fold_sign_agreement_matrix[
                    upper_triangle_indices
                ]
            )
        }
    ]
)

pairwise_network_summary_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "pairwise_fold_network_reproducibility_summary.csv",
    index=False
)


# ============================================================
# 12. REPEATED SUBSAMPLE STABILITY SELECTION
# ============================================================
#
# The penalty is fixed at the alpha selected in the complete
# sample. This separates instability in feature selection from
# instability introduced by retuning alpha in every subsample.
# ============================================================

subsample_size = max(
    10,
    int(
        np.floor(
            n_subjects
            * STABILITY_SAMPLE_FRACTION
        )
    )
)

stability_coefficients = np.zeros(
    (
        N_STABILITY_REPETITIONS,
        n_rois
    ),
    dtype=float
)

stability_selection_rows = []

print("\n" + "=" * 90)
print("RUNNING REPEATED SUBSAMPLE STABILITY SELECTION")
print("=" * 90)
print(
    f"Repetitions: {N_STABILITY_REPETITIONS}"
)
print(
    f"Participants per repetition: "
    f"{subsample_size}/{n_subjects}"
)
print(
    f"Fixed reference alpha: {reference_alpha:.6f}"
)

for repetition in range(
    N_STABILITY_REPETITIONS
):
    sampled_indices = RNG.choice(
        np.arange(n_subjects),
        size=subsample_size,
        replace=False
    )

    X_subsample_raw = X_raw[
        sampled_indices
    ]

    y_subsample_raw = y_raw[
        sampled_indices
    ]

    X_subsample_scaler = StandardScaler()
    y_subsample_scaler = StandardScaler()

    X_subsample_z = (
        X_subsample_scaler
        .fit_transform(
            X_subsample_raw
        )
    )

    y_subsample_z = (
        y_subsample_scaler
        .fit_transform(
            y_subsample_raw.reshape(
                -1,
                1
            )
        )
        .reshape(-1)
    )

    subsample_lasso = Lasso(
        alpha=reference_alpha,
        max_iter=LASSO_MAX_ITER,
        random_state=(
            RANDOM_SEED + repetition
        )
    )

    subsample_lasso.fit(
        X_subsample_z,
        y_subsample_z
    )

    coefficients = (
        subsample_lasso.coef_
        .astype(float)
    )

    stability_coefficients[
        repetition,
        :
    ] = coefficients

    if (
        (repetition + 1) % 100 == 0
        or repetition + 1
        == N_STABILITY_REPETITIONS
    ):
        print(
            f"Completed "
            f"{repetition + 1}/"
            f"{N_STABILITY_REPETITIONS}"
        )

for roi_index, roi in enumerate(
    roi_columns
):
    coefficients = stability_coefficients[
        :,
        roi_index
    ]

    signs = coefficient_sign(
        coefficients
    )

    selected = (
        np.abs(coefficients)
        > COEFFICIENT_ZERO_TOLERANCE
    )

    reference_sign = reference_signs[
        roi_index
    ]

    if reference_sign != 0:
        same_reference_sign = np.mean(
            signs == reference_sign
        )

        same_reference_sign_if_selected = (
            np.mean(
                signs[selected]
                == reference_sign
            )
            if selected.sum() > 0
            else np.nan
        )
    else:
        same_reference_sign = np.mean(
            signs == 0
        )

        same_reference_sign_if_selected = np.nan

    stability_selection_rows.append(
        {
            "ROI": roi,
            "Full_Sample_Coefficient": (
                reference_coefficients[
                    roi_index
                ]
            ),
            "Full_Sample_Direction": (
                "Positive"
                if reference_sign == 1
                else (
                    "Negative"
                    if reference_sign == -1
                    else "Zero"
                )
            ),
            "Selection_Probability": (
                np.mean(selected)
            ),
            "Positive_Selection_Probability": (
                np.mean(
                    coefficients
                    > COEFFICIENT_ZERO_TOLERANCE
                )
            ),
            "Negative_Selection_Probability": (
                np.mean(
                    coefficients
                    < -COEFFICIENT_ZERO_TOLERANCE
                )
            ),
            "Zero_Probability": (
                np.mean(~selected)
            ),
            "Reference_Sign_Probability": (
                same_reference_sign
            ),
            "Reference_Sign_Probability_Given_Selected": (
                same_reference_sign_if_selected
            ),
            "Mean_Subsample_Coefficient": (
                np.mean(coefficients)
            ),
            "Median_Subsample_Coefficient": (
                np.median(coefficients)
            ),
            "SD_Subsample_Coefficient": (
                np.std(
                    coefficients,
                    ddof=1
                )
            )
        }
    )

stability_selection_summary_df = pd.DataFrame(
    stability_selection_rows
)

stability_selection_summary_df = (
    stability_selection_summary_df
    .sort_values(
        [
            "Selection_Probability",
            "Reference_Sign_Probability"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

stability_selection_summary_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "subsample_stability_selection_summary.csv",
    index=False
)

stability_coefficients_df = pd.DataFrame(
    stability_coefficients,
    columns=roi_columns
)

stability_coefficients_df.insert(
    0,
    "Repetition",
    np.arange(
        1,
        N_STABILITY_REPETITIONS + 1
    )
)

stability_coefficients_df.to_csv(
    NETWORK_STABILITY_DATA_DIR
    / "subsample_stability_selection_coefficients.csv",
    index=False
)


# ============================================================
# 13. FIXED ORIGINAL AND FULL-LASSO NBI LOOCV PREDICTION
# ============================================================
#
# The ROI membership and positive/negative directions remain
# fixed. Only:
#       - ROI scaling
#       - score-to-performance slope/intercept
# are estimated from the training participants.
# ============================================================

fixed_nbi_fold_rows = []
fixed_nbi_summary_rows = []

for network_name, architecture in (
    NETWORK_ARCHITECTURES.items()
):
    positive_rois = architecture[
        "positive"
    ]

    negative_rois = architecture[
        "negative"
    ]

    observed_values = np.full(
        n_subjects,
        np.nan
    )

    predicted_values = np.full(
        n_subjects,
        np.nan
    )

    heldout_scores = np.full(
        n_subjects,
        np.nan
    )

    for fold_index, (
        train_indices,
        test_indices
    ) in enumerate(
        outer_loo.split(X_raw),
        start=1
    ):
        heldout_index = int(
            test_indices[0]
        )

        X_train_raw = X_raw[
            train_indices
        ]

        X_test_raw = X_raw[
            test_indices
        ]

        y_train = y_raw[
            train_indices
        ]

        y_test = y_raw[
            test_indices
        ]

        # Training-only ROI standardization.
        fold_scaler = StandardScaler()

        X_train_z = fold_scaler.fit_transform(
            X_train_raw
        )

        X_test_z = fold_scaler.transform(
            X_test_raw
        )

        train_scores = (
            calculate_fixed_network_score(
                standardized_roi_matrix=(
                    X_train_z
                ),
                roi_names=roi_columns,
                positive_rois=positive_rois,
                negative_rois=negative_rois
            )
        )

        test_score = (
            calculate_fixed_network_score(
                standardized_roi_matrix=(
                    X_test_z
                ),
                roi_names=roi_columns,
                positive_rois=positive_rois,
                negative_rois=negative_rois
            )[0]
        )

        score_to_performance_model = (
            LinearRegression()
        )

        score_to_performance_model.fit(
            train_scores.reshape(-1, 1),
            y_train
        )

        heldout_prediction = float(
            score_to_performance_model.predict(
                np.array(
                    [[test_score]]
                )
            )[0]
        )

        observed_values[
            heldout_index
        ] = y_test[0]

        predicted_values[
            heldout_index
        ] = heldout_prediction

        heldout_scores[
            heldout_index
        ] = test_score

        fixed_nbi_fold_rows.append(
            {
                "Network": network_name,
                "Outer_Fold": fold_index,
                "Heldout_Subject": (
                    subjects[
                        heldout_index
                    ]
                ),
                "Observed_Performance": (
                    y_test[0]
                ),
                "Predicted_Performance": (
                    heldout_prediction
                ),
                "Heldout_NBI_Score": (
                    test_score
                ),
                "Training_Slope": (
                    score_to_performance_model
                    .coef_[0]
                ),
                "Training_Intercept": (
                    score_to_performance_model
                    .intercept_
                ),
                "Positive_ROIs": ", ".join(
                    positive_rois
                ),
                "Negative_ROIs": ", ".join(
                    negative_rois
                )
            }
        )

    metrics = calculate_prediction_metrics(
        observed=observed_values,
        predicted=predicted_values
    )

    fixed_nbi_summary_rows.append(
        {
            "Network": network_name,
            "Positive_ROIs": ", ".join(
                positive_rois
            ),
            "Negative_ROIs": ", ".join(
                negative_rois
            ),
            **metrics
        }
    )

fixed_nbi_loocv_fold_df = pd.DataFrame(
    fixed_nbi_fold_rows
)

fixed_nbi_loocv_summary_df = pd.DataFrame(
    fixed_nbi_summary_rows
)

fixed_nbi_loocv_fold_df.to_csv(
    NETWORK_STABILITY_DATA_DIR
    / "fixed_nbi_loocv_fold_predictions.csv",
    index=False
)

fixed_nbi_loocv_summary_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "fixed_nbi_loocv_prediction_summary.csv",
    index=False
)


# ============================================================
# 14. COMBINED ROI VALIDATION TABLE
# ============================================================

network_roi_validation_df = (
    loocv_roi_stability_summary_df
    .merge(
        stability_selection_summary_df[
            [
                "ROI",
                "Selection_Probability",
                "Positive_Selection_Probability",
                "Negative_Selection_Probability",
                "Reference_Sign_Probability",
                "Reference_Sign_Probability_Given_Selected",
                "Mean_Subsample_Coefficient"
            ]
        ],
        on="ROI",
        how="left",
        validate="one_to_one"
    )
)

network_roi_validation_df[
    "In_Original_NBI"
] = network_roi_validation_df[
    "ROI"
].isin(
    ORIGINAL_POSITIVE_ROIS
    + ORIGINAL_NEGATIVE_ROIS
)

network_roi_validation_df[
    "In_Full_LASSO_NBI"
] = network_roi_validation_df[
    "ROI"
].isin(
    FULL_LASSO_POSITIVE_ROIS
    + FULL_LASSO_NEGATIVE_ROIS
)

network_roi_validation_df[
    "Expected_Original_Direction"
] = network_roi_validation_df[
    "ROI"
].map(
    {
        **{
            roi: "Positive"
            for roi in ORIGINAL_POSITIVE_ROIS
        },
        **{
            roi: "Negative"
            for roi in ORIGINAL_NEGATIVE_ROIS
        }
    }
)

network_roi_validation_df[
    "Expected_Full_LASSO_Direction"
] = network_roi_validation_df[
    "ROI"
].map(
    {
        **{
            roi: "Positive"
            for roi in FULL_LASSO_POSITIVE_ROIS
        },
        **{
            roi: "Negative"
            for roi in FULL_LASSO_NEGATIVE_ROIS
        }
    }
)

network_roi_validation_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "combined_network_roi_validation_summary.csv",
    index=False
)


# ============================================================
# 15. NETWORK-LEVEL SIGN VALIDATION SUMMARY
# ============================================================

network_level_rows = []

for network_name, architecture in (
    NETWORK_ARCHITECTURES.items()
):
    expected_signs = {
        **{
            roi: 1
            for roi in architecture[
                "positive"
            ]
        },
        **{
            roi: -1
            for roi in architecture[
                "negative"
            ]
        }
    }

    roi_rows = []

    for roi, expected_sign in (
        expected_signs.items()
    ):
        row = network_roi_validation_df[
            network_roi_validation_df[
                "ROI"
            ] == roi
        ]

        if row.empty:
            continue

        row = row.iloc[0]

        positive_frequency = row[
            "LOOCV_Positive_Frequency"
        ]

        negative_frequency = row[
            "LOOCV_Negative_Frequency"
        ]

        loocv_expected_frequency = (
            positive_frequency
            if expected_sign == 1
            else negative_frequency
        )

        stability_expected_frequency = (
            row[
                "Positive_Selection_Probability"
            ]
            if expected_sign == 1
            else row[
                "Negative_Selection_Probability"
            ]
        )

        roi_rows.append(
            {
                "ROI": roi,
                "Expected_Sign": expected_sign,
                "LOOCV_Expected_Sign_Frequency": (
                    loocv_expected_frequency
                ),
                "Subsample_Expected_Sign_Probability": (
                    stability_expected_frequency
                )
            }
        )

    roi_network_df = pd.DataFrame(
        roi_rows
    )

    network_level_rows.append(
        {
            "Network": network_name,
            "N_ROIs": len(roi_network_df),
            "Mean_LOOCV_Expected_Sign_Frequency": (
                roi_network_df[
                    "LOOCV_Expected_Sign_Frequency"
                ].mean()
            ),
            "Median_LOOCV_Expected_Sign_Frequency": (
                roi_network_df[
                    "LOOCV_Expected_Sign_Frequency"
                ].median()
            ),
            "Mean_Subsample_Expected_Sign_Probability": (
                roi_network_df[
                    "Subsample_Expected_Sign_Probability"
                ].mean()
            ),
            "Median_Subsample_Expected_Sign_Probability": (
                roi_network_df[
                    "Subsample_Expected_Sign_Probability"
                ].median()
            ),
            "All_ROIs_LOOCV_Expected_Sign_At_Least_50pct": bool(
                (
                    roi_network_df[
                        "LOOCV_Expected_Sign_Frequency"
                    ] >= 0.50
                ).all()
            ),
            "All_ROIs_Stability_Expected_Sign_At_Least_50pct": bool(
                (
                    roi_network_df[
                        "Subsample_Expected_Sign_Probability"
                    ] >= 0.50
                ).all()
            )
        }
    )

network_level_sign_validation_df = pd.DataFrame(
    network_level_rows
)

network_level_sign_validation_df.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "network_level_sign_validation_summary.csv",
    index=False
)


# ============================================================
# 16. FIGURE: FULL-SAMPLE COEFFICIENTS WITH LOOCV CIs
# ============================================================

coefficient_plot_df = (
    network_roi_validation_df
    .sort_values(
        "Mean_LOOCV_Coefficient"
    )
    .copy()
)

y_positions = np.arange(
    len(coefficient_plot_df)
)

mean_coefficients = coefficient_plot_df[
    "Mean_LOOCV_Coefficient"
].to_numpy(dtype=float)

lower_errors = (
    mean_coefficients
    -
    coefficient_plot_df[
        "Mean_Beta_CI_Lower"
    ].to_numpy(dtype=float)
)

upper_errors = (
    coefficient_plot_df[
        "Mean_Beta_CI_Upper"
    ].to_numpy(dtype=float)
    -
    mean_coefficients
)

figure, axis = plt.subplots(
    figsize=(
        10,
        max(
            6,
            0.55 * len(coefficient_plot_df)
        )
    )
)

axis.errorbar(
    mean_coefficients,
    y_positions,
    xerr=np.vstack(
        [
            lower_errors,
            upper_errors
        ]
    ),
    fmt="o",
    capsize=4,
    linewidth=1.5
)

axis.scatter(
    coefficient_plot_df[
        "Full_Sample_Coefficient"
    ],
    y_positions,
    marker="x",
    s=65,
    label="Full-sample coefficient"
)

axis.axvline(
    0,
    linestyle="--",
    linewidth=1
)

axis.set_yticks(
    y_positions
)

axis.set_yticklabels(
    coefficient_plot_df[
        "ROI"
    ]
)

axis.set_xlabel(
    "Standardized LASSO coefficient"
)

axis.set_ylabel(
    "ROI"
)

axis.set_title(
    "LOOCV coefficient stability\n"
    "Mean training-fold coefficient with bootstrap 95% CI"
)

axis.legend(
    frameon=False
)

figure.tight_layout()

figure.savefig(
    NETWORK_STABILITY_FIGURE_DIR
    / "loocv_coefficients_with_bootstrap_ci.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ============================================================
# 17. FIGURE: SIGN-CONSISTENCY HEATMAP
# ============================================================

sign_matrix = coefficient_sign(
    loocv_coefficient_matrix
).T

# Sort ROIs by the reference coefficient.
sign_order = np.argsort(
    reference_coefficients
)

sign_matrix_sorted = sign_matrix[
    sign_order,
    :
]

sorted_roi_names = np.asarray(
    roi_columns
)[sign_order]

figure, axis = plt.subplots(
    figsize=(
        max(
            12,
            0.34 * n_subjects
        ),
        max(
            6,
            0.55 * n_rois
        )
    )
)

image = axis.imshow(
    sign_matrix_sorted,
    aspect="auto",
    interpolation="nearest",
    cmap="bwr",
    vmin=-1,
    vmax=1
)

axis.set_yticks(
    np.arange(n_rois)
)

axis.set_yticklabels(
    sorted_roi_names
)

axis.set_xticks(
    np.arange(n_subjects)
)

axis.set_xticklabels(
    subjects,
    rotation=90
)

axis.set_xlabel(
    "Held-out participant"
)

axis.set_ylabel(
    "ROI"
)

axis.set_title(
    "LOOCV LASSO coefficient directions\n"
    "Blue = negative, white = zero, red = positive"
)

colorbar = figure.colorbar(
    image,
    ax=axis,
    shrink=0.75,
    ticks=[
        -1,
        0,
        1
    ]
)

colorbar.ax.set_yticklabels(
    [
        "Negative",
        "Zero",
        "Positive"
    ]
)

figure.tight_layout()

figure.savefig(
    NETWORK_STABILITY_FIGURE_DIR
    / "loocv_coefficient_sign_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ============================================================
# 18. FIGURE: LOOCV SELECTION AND SIGN PROBABILITIES
# ============================================================

probability_plot_df = (
    network_roi_validation_df
    .sort_values(
        "LOOCV_Selection_Frequency",
        ascending=True
    )
    .copy()
)

y_positions = np.arange(
    len(probability_plot_df)
)

figure, axis = plt.subplots(
    figsize=(
        10,
        max(
            6,
            0.55 * len(probability_plot_df)
        )
    )
)

axis.barh(
    y_positions,
    probability_plot_df[
        "LOOCV_Positive_Frequency"
    ],
    label="Positive"
)

axis.barh(
    y_positions,
    -probability_plot_df[
        "LOOCV_Negative_Frequency"
    ],
    label="Negative"
)

axis.axvline(
    0,
    linewidth=1
)

axis.axvline(
    0.50,
    linestyle="--",
    linewidth=1
)

axis.axvline(
    -0.50,
    linestyle="--",
    linewidth=1
)

axis.set_yticks(
    y_positions
)

axis.set_yticklabels(
    probability_plot_df[
        "ROI"
    ]
)

axis.set_xlim(
    -1,
    1
)

axis.set_xlabel(
    "Proportion of LOOCV folds\n"
    "Negative direction ← 0 → Positive direction"
)

axis.set_ylabel(
    "ROI"
)

axis.set_title(
    "ROI direction consistency across held-out participants"
)

axis.legend(
    frameon=False
)

figure.tight_layout()

figure.savefig(
    NETWORK_STABILITY_FIGURE_DIR
    / "loocv_roi_positive_negative_frequencies.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ============================================================
# 19. FIGURE: STABILITY-SELECTION PROBABILITIES
# ============================================================

stability_plot_df = (
    stability_selection_summary_df
    .sort_values(
        "Selection_Probability",
        ascending=True
    )
    .copy()
)

y_positions = np.arange(
    len(stability_plot_df)
)

figure, axis = plt.subplots(
    figsize=(
        10,
        max(
            6,
            0.55 * len(stability_plot_df)
        )
    )
)

axis.barh(
    y_positions,
    stability_plot_df[
        "Positive_Selection_Probability"
    ],
    label="Positive selection"
)

axis.barh(
    y_positions,
    -stability_plot_df[
        "Negative_Selection_Probability"
    ],
    label="Negative selection"
)

axis.axvline(
    0,
    linewidth=1
)

axis.axvline(
    0.50,
    linestyle="--",
    linewidth=1
)

axis.axvline(
    -0.50,
    linestyle="--",
    linewidth=1
)

axis.set_yticks(
    y_positions
)

axis.set_yticklabels(
    stability_plot_df[
        "ROI"
    ]
)

axis.set_xlim(
    -1,
    1
)

axis.set_xlabel(
    "Subsample selection probability\n"
    "Negative direction ← 0 → Positive direction"
)

axis.set_ylabel(
    "ROI"
)

axis.set_title(
    f"LASSO stability selection across "
    f"{N_STABILITY_REPETITIONS} subsamples"
)

axis.legend(
    frameon=False
)

figure.tight_layout()

figure.savefig(
    NETWORK_STABILITY_FIGURE_DIR
    / "subsample_stability_selection_probabilities.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ============================================================
# 20. FIGURE: PAIRWISE FOLD COEFFICIENT CORRELATIONS
# ============================================================

figure, axis = plt.subplots(
    figsize=(10, 9)
)

image = axis.imshow(
    pairwise_fold_correlation_matrix,
    aspect="equal",
    interpolation="nearest",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

axis.set_xticks(
    np.arange(n_subjects)
)

axis.set_xticklabels(
    subjects,
    rotation=90,
    fontsize=8
)

axis.set_yticks(
    np.arange(n_subjects)
)

axis.set_yticklabels(
    subjects,
    fontsize=8
)

axis.set_xlabel(
    "Held-out participant"
)

axis.set_ylabel(
    "Held-out participant"
)

axis.set_title(
    "Pairwise similarity of LOOCV LASSO coefficient vectors"
)

figure.colorbar(
    image,
    ax=axis,
    shrink=0.75,
    label="Pearson correlation"
)

figure.tight_layout()

figure.savefig(
    NETWORK_STABILITY_FIGURE_DIR
    / "pairwise_fold_coefficient_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ============================================================
# 21. FIGURE: FOLD SIMILARITY TO FULL-SAMPLE ARCHITECTURE
# ============================================================

fold_similarity_plot_df = (
    loocv_network_fold_similarity_df
    .sort_values(
        "Coefficient_Vector_Pearson_r_With_Full"
    )
    .reset_index(drop=True)
)

figure, axis = plt.subplots(
    figsize=(
        max(
            12,
            0.38 * len(
                fold_similarity_plot_df
            )
        ),
        6
    )
)

axis.plot(
    np.arange(
        len(fold_similarity_plot_df)
    ),
    fold_similarity_plot_df[
        "Coefficient_Vector_Pearson_r_With_Full"
    ],
    marker="o"
)

axis.axhline(
    0,
    linewidth=1
)

axis.axhline(
    0.50,
    linestyle="--",
    linewidth=1,
    label="r = .50"
)

axis.set_xticks(
    np.arange(
        len(fold_similarity_plot_df)
    )
)

axis.set_xticklabels(
    fold_similarity_plot_df[
        "Heldout_Subject"
    ],
    rotation=90
)

axis.set_ylim(
    -1.05,
    1.05
)

axis.set_xlabel(
    "Held-out participant"
)

axis.set_ylabel(
    "Correlation with full-sample coefficient vector"
)

axis.set_title(
    "Reproducibility of the full network architecture"
)

axis.legend(
    frameon=False
)

figure.tight_layout()

figure.savefig(
    NETWORK_STABILITY_FIGURE_DIR
    / "fold_coefficient_similarity_to_full_sample.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)


# ============================================================
# 22. FIGURE: FIXED NBI LOOCV PREDICTION
# ============================================================

for network_name in (
    fixed_nbi_loocv_fold_df[
        "Network"
    ].unique()
):
    plot_df = (
        fixed_nbi_loocv_fold_df[
            fixed_nbi_loocv_fold_df[
                "Network"
            ] == network_name
        ]
        .dropna(
            subset=[
                "Observed_Performance",
                "Predicted_Performance"
            ]
        )
        .copy()
    )

    summary_row = (
        fixed_nbi_loocv_summary_df[
            fixed_nbi_loocv_summary_df[
                "Network"
            ] == network_name
        ]
        .iloc[0]
    )

    observed = plot_df[
        "Observed_Performance"
    ].to_numpy(dtype=float)

    predicted = plot_df[
        "Predicted_Performance"
    ].to_numpy(dtype=float)

    figure, axis = plt.subplots(
        figsize=(7.5, 6.5)
    )

    axis.scatter(
        observed,
        predicted,
        s=70,
        alpha=0.80
    )

    minimum_value = min(
        np.min(observed),
        np.min(predicted)
    )

    maximum_value = max(
        np.max(observed),
        np.max(predicted)
    )

    axis.plot(
        [
            minimum_value,
            maximum_value
        ],
        [
            minimum_value,
            maximum_value
        ],
        linestyle="--",
        linewidth=1.5,
        label="Identity line"
    )

    axis.set_xlabel(
        "Observed performance"
    )

    axis.set_ylabel(
        "LOOCV predicted performance"
    )

    axis.set_title(
        f"{network_name.replace('_', ' ')}\n"
        f"CV $R^2$ = "
        f"{summary_row['CV_R2']:.3f}, "
        f"$r$ = "
        f"{summary_row['Pearson_r']:.3f}"
    )

    axis.legend(
        frameon=False
    )

    figure.tight_layout()

    figure.savefig(
        NETWORK_STABILITY_FIGURE_DIR
        / (
            f"{safe_filename(network_name)}"
            "_fixed_architecture_loocv_prediction.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(figure)


# ============================================================
# 23. SAVE SETTINGS
# ============================================================

analysis_06_settings = {
    "Complete_Case_N": int(
        n_subjects
    ),
    "N_ROIs": int(
        n_rois
    ),
    "ROI_Columns": roi_columns,
    "Full_Sample_Reference_Alpha": (
        reference_alpha
    ),
    "Alpha_Grid_Minimum": float(
        np.min(ALPHA_GRID)
    ),
    "Alpha_Grid_Maximum": float(
        np.max(ALPHA_GRID)
    ),
    "Alpha_Grid_Size": int(
        len(ALPHA_GRID)
    ),
    "LOOCV_Alpha_Retuned_Per_Fold": True,
    "Inner_CV_Folds": int(
        INNER_CV_FOLDS
    ),
    "Stability_Repetitions": int(
        N_STABILITY_REPETITIONS
    ),
    "Stability_Sample_Fraction": float(
        STABILITY_SAMPLE_FRACTION
    ),
    "Stability_Alpha": float(
        reference_alpha
    ),
    "Bootstrap_Repetitions": int(
        N_BOOTSTRAP
    ),
    "Coefficient_Zero_Tolerance": float(
        COEFFICIENT_ZERO_TOLERANCE
    ),
    "Original_Positive_ROIs": (
        ORIGINAL_POSITIVE_ROIS
    ),
    "Original_Negative_ROIs": (
        ORIGINAL_NEGATIVE_ROIS
    ),
    "Full_LASSO_Positive_ROIs": (
        FULL_LASSO_POSITIVE_ROIS
    ),
    "Full_LASSO_Negative_ROIs": (
        FULL_LASSO_NEGATIVE_ROIS
    )
}

with open(
    NETWORK_STABILITY_OUT
    / "analysis_06_settings.json",
    "w",
    encoding="utf-8"
) as settings_file:
    json.dump(
        analysis_06_settings,
        settings_file,
        indent=4
    )


# ============================================================
# 24. PRINT MAIN RESULTS
# ============================================================

print("\n" + "=" * 110)
print("ANALYSIS 06 COMPLETE: NETWORK ARCHITECTURE STABILITY")
print("=" * 110)

print("\n1. Fixed NBI held-out prediction:")
display(
    fixed_nbi_loocv_summary_df.round(5)
)

print("\n2. ROI-level LOOCV selection and direction stability:")
display_columns = [
    "ROI",
    "Network_Membership",
    "Full_Sample_Coefficient",
    "Mean_LOOCV_Coefficient",
    "Mean_Beta_CI_Lower",
    "Mean_Beta_CI_Upper",
    "LOOCV_Selection_Frequency",
    "LOOCV_Positive_Frequency",
    "LOOCV_Negative_Frequency",
    "Sign_Consistency_With_Full_Sample",
    "Sign_Consistency_Given_Selected"
]

display(
    network_roi_validation_df[
        display_columns
    ]
    .sort_values(
        "LOOCV_Selection_Frequency",
        ascending=False
    )
    .round(4)
)

print("\n3. Repeated subsample stability selection:")
display(
    stability_selection_summary_df[
        [
            "ROI",
            "Full_Sample_Direction",
            "Selection_Probability",
            "Positive_Selection_Probability",
            "Negative_Selection_Probability",
            "Reference_Sign_Probability",
            "Reference_Sign_Probability_Given_Selected",
            "Mean_Subsample_Coefficient"
        ]
    ]
    .round(4)
)

print("\n4. Network-level expected-sign validation:")
display(
    network_level_sign_validation_df
    .round(4)
)

print("\n5. Fold-to-full-sample network similarity:")
fold_similarity_descriptives = pd.DataFrame(
    [
        {
            "Metric": "Pearson r with full-sample coefficient vector",
            "Mean": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Pearson_r_With_Full"
                ].mean()
            ),
            "Median": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Pearson_r_With_Full"
                ].median()
            ),
            "Minimum": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Pearson_r_With_Full"
                ].min()
            ),
            "Maximum": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Pearson_r_With_Full"
                ].max()
            )
        },
        {
            "Metric": "Cosine similarity with full-sample coefficient vector",
            "Mean": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Cosine_Similarity_With_Full"
                ].mean()
            ),
            "Median": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Cosine_Similarity_With_Full"
                ].median()
            ),
            "Minimum": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Cosine_Similarity_With_Full"
                ].min()
            ),
            "Maximum": (
                loocv_network_fold_similarity_df[
                    "Coefficient_Vector_Cosine_Similarity_With_Full"
                ].max()
            )
        },
        {
            "Metric": "Sign agreement with full-sample vector",
            "Mean": (
                loocv_network_fold_similarity_df[
                    "Sign_Agreement_With_Full_Among_Union_Selected"
                ].mean()
            ),
            "Median": (
                loocv_network_fold_similarity_df[
                    "Sign_Agreement_With_Full_Among_Union_Selected"
                ].median()
            ),
            "Minimum": (
                loocv_network_fold_similarity_df[
                    "Sign_Agreement_With_Full_Among_Union_Selected"
                ].min()
            ),
            "Maximum": (
                loocv_network_fold_similarity_df[
                    "Sign_Agreement_With_Full_Among_Union_Selected"
                ].max()
            )
        }
    ]
)

display(
    fold_similarity_descriptives
    .round(4)
)

fold_similarity_descriptives.to_csv(
    NETWORK_STABILITY_TABLE_DIR
    / "fold_to_full_sample_similarity_summary.csv",
    index=False
)

print("\n6. Pairwise fold network reproducibility:")
display(
    pairwise_network_summary_df
    .round(4)
)

print(
    "\nPrimary output tables:\n"
    "  combined_network_roi_validation_summary.csv\n"
    "  loocv_roi_selection_and_sign_stability.csv\n"
    "  subsample_stability_selection_summary.csv\n"
    "  network_level_sign_validation_summary.csv\n"
    "  fold_to_full_sample_similarity_summary.csv\n"
    "  pairwise_fold_network_reproducibility_summary.csv\n"
    "  fixed_nbi_loocv_prediction_summary.csv"
)

print(
    "\nCreated notebook objects:\n"
    "  network_stability_analysis_df\n"
    "  full_sample_reference_df\n"
    "  loocv_network_coefficients_df\n"
    "  loocv_network_fold_similarity_df\n"
    "  loocv_roi_stability_summary_df\n"
    "  stability_selection_summary_df\n"
    "  network_roi_validation_df\n"
    "  network_level_sign_validation_df\n"
    "  pairwise_network_summary_df\n"
    "  fixed_nbi_loocv_fold_df\n"
    "  fixed_nbi_loocv_summary_df"
)

print(
    "\nAll outputs saved to:\n"
    f"{NETWORK_STABILITY_OUT}"
)

In [ ]:
# ============================================================
#  LEFT MOTOR ROI VALIDATION
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Empirically test whether the anatomically defined left motor
# control ROI ISC is related to participants' mouse-movement
# behavior.
#
# For each Subject × Video observation:
#
#   Left Motor ROI ISC
#       correlated / modeled with
#   MotorPC1
#   MotorPC2
#   MotorPC3
#
# Analyses:
#   1. Prepare and merge subject × video motor-PC scores with
#      left motor ROI ISC
#   2. Pearson + Spearman correlations
#   3. Crossed mixed-effects models:
#
#        LeftMotorISC ~ MotorPCk
#                     + (1 | Subject)
#                     + (1 | Video)
#
#      fitted separately for Valence and Arousal
#
#   4. Video-preserving permutation tests
#   5. FDR correction across MotorPC1–3 within modality
#   6. Scatterplots and summary figures
#
# REQUIRED NOTEBOOK OBJECTS
# ------------------------------------------------------------
#   motor_pca_scores_df
#
# and ONE of:
#   motor_control_isc_wide_df
#   motor_video_roi_isc_df
#
# OUTPUT:
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\
# Emotion Perception Data\Motor_Control\07_motor_roi_validation
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests


warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)


# ============================================================
# 1. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

if "motor_pca_scores_df" not in globals():
    raise ValueError(
        "motor_pca_scores_df was not found.\n"
        "Run the motor-feature PCA analysis first."
    )

if (
    "motor_control_isc_wide_df" not in globals()
    and
    "motor_video_roi_isc_df" not in globals()
):
    raise ValueError(
        "Neither motor_control_isc_wide_df nor "
        "motor_video_roi_isc_df was found.\n\n"
        "Run the motor-control ROI ISC pipeline first."
    )


# ============================================================
# 2. OUTPUT DIRECTORIES
# ============================================================

MOTOR_VALIDATION_OUT = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\07_motor_roi_validation"
)

MOTOR_VALIDATION_TABLE_DIR = (
    MOTOR_VALIDATION_OUT / "tables"
)

MOTOR_VALIDATION_FIGURE_DIR = (
    MOTOR_VALIDATION_OUT / "figures"
)

MOTOR_VALIDATION_MODEL_DIR = (
    MOTOR_VALIDATION_OUT / "model_summaries"
)

for folder in [
    MOTOR_VALIDATION_OUT,
    MOTOR_VALIDATION_TABLE_DIR,
    MOTOR_VALIDATION_FIGURE_DIR,
    MOTOR_VALIDATION_MODEL_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Motor ROI validation outputs will be saved to:\n"
    f"{MOTOR_VALIDATION_OUT}"
)


# ============================================================
# 3. USER-ADJUSTABLE SETTINGS
# ============================================================

MOTOR_PC_COLUMNS = [
    "MotorPC1",
    "MotorPC2",
    "MotorPC3"
]

N_PERMUTATIONS = 10000

RANDOM_SEED = 2026

MINIMUM_N = 20

FIGURE_DPI = 300

RNG = np.random.default_rng(
    RANDOM_SEED
)


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """
    Normalize participant IDs.
    """

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_trigger(value):
    """
    Make 1, 1.0, and '1' equivalent.
    """

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(
                int(numeric_value)
            )

    except (ValueError, TypeError):
        pass

    return value


def normalize_video_name(value):
    """
    Remove only the final modality suffix from a video name.

    Example:
        031SoloC_baseline_valence
        -> 031SoloC_baseline
    """

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"_(valence|arousal)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def fisher_z(values):
    """
    Fisher-z transform correlations safely.
    """

    values = pd.to_numeric(
        values,
        errors="coerce"
    )

    values = values.clip(
        lower=-0.999999,
        upper=0.999999
    )

    return np.arctanh(
        values
    )


def first_existing_column(
    dataframe,
    candidates
):
    """
    Return first candidate column found.
    """

    for column in candidates:

        if column in dataframe.columns:
            return column

    return None


def safe_pearson(x, y):
    """
    Pearson correlation with validity checks.
    """

    temp = pd.DataFrame({
        "x": x,
        "y": y
    }).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(temp) < MINIMUM_N:
        return np.nan, np.nan, len(temp)

    if (
        temp["x"].std(ddof=1) == 0
        or
        temp["y"].std(ddof=1) == 0
    ):
        return np.nan, np.nan, len(temp)

    r_value, p_value = pearsonr(
        temp["x"],
        temp["y"]
    )

    return (
        float(r_value),
        float(p_value),
        len(temp)
    )


def safe_spearman(x, y):
    """
    Spearman correlation with validity checks.
    """

    temp = pd.DataFrame({
        "x": x,
        "y": y
    }).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(temp) < MINIMUM_N:
        return np.nan, np.nan, len(temp)

    rho, p_value = spearmanr(
        temp["x"],
        temp["y"]
    )

    return (
        float(rho),
        float(p_value),
        len(temp)
    )


# ============================================================
# 5. PREPARE MOTOR PCA TABLE
# ============================================================

motor_pc_df = (
    motor_pca_scores_df
    .copy()
)

required_motor_columns = [
    "Subject",
    "BaseVideo",
    "Modality"
] + MOTOR_PC_COLUMNS

missing_motor_columns = [
    column
    for column in required_motor_columns
    if column not in motor_pc_df.columns
]

if missing_motor_columns:
    raise ValueError(
        "motor_pca_scores_df is missing required columns:\n"
        f"{missing_motor_columns}"
    )


motor_pc_df["Subject"] = (
    motor_pc_df["Subject"]
    .apply(normalize_subject_id)
)

motor_pc_df["BaseVideo"] = (
    motor_pc_df["BaseVideo"]
    .apply(normalize_video_name)
)

motor_pc_df["Modality"] = (
    motor_pc_df["Modality"]
    .astype(str)
    .str.strip()
    .str.capitalize()
)

for pc in MOTOR_PC_COLUMNS:

    motor_pc_df[pc] = pd.to_numeric(
        motor_pc_df[pc],
        errors="coerce"
    )


# ============================================================
# 6. RECOVER TRIGGER NUMBERS
#
# Prefer Trigger already present.
# Otherwise use video_mapping.csv.
# ============================================================

if "Trigger" in motor_pc_df.columns:

    motor_pc_df["Trigger"] = (
        motor_pc_df["Trigger"]
        .apply(normalize_trigger)
    )

else:

    VIDEO_MAPPING_CSV = Path(
        r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
        r"\Emotion Perception Data\video_mapping.csv"
    )

    if not VIDEO_MAPPING_CSV.exists():

        raise FileNotFoundError(
            "motor_pca_scores_df does not contain Trigger and "
            "video_mapping.csv could not be found at:\n"
            f"{VIDEO_MAPPING_CSV}"
        )

    video_mapping_df = pd.read_csv(
        VIDEO_MAPPING_CSV
    )

    mapping_video_column = (
        first_existing_column(
            video_mapping_df,
            [
                "video",
                "Video",
                "BaseVideo",
                "base_video"
            ]
        )
    )

    mapping_trigger_column = (
        first_existing_column(
            video_mapping_df,
            [
                "trigger_number",
                "Trigger",
                "trigger",
                "TriggerNumber"
            ]
        )
    )

    if (
        mapping_video_column is None
        or
        mapping_trigger_column is None
    ):
        raise ValueError(
            "Could not identify video and trigger columns "
            "in video_mapping.csv.\n\n"
            f"Available columns:\n"
            f"{list(video_mapping_df.columns)}"
        )

    video_mapping_df["BaseVideo"] = (
        video_mapping_df[
            mapping_video_column
        ]
        .apply(normalize_video_name)
    )

    video_mapping_df["Trigger"] = (
        video_mapping_df[
            mapping_trigger_column
        ]
        .apply(normalize_trigger)
    )

    video_mapping_df = (
        video_mapping_df[
            [
                "BaseVideo",
                "Trigger"
            ]
        ]
        .drop_duplicates()
    )

    motor_pc_df = (
        motor_pc_df
        .merge(
            video_mapping_df,
            on="BaseVideo",
            how="left",
            validate="many_to_one"
        )
    )


print(
    "\nMotor-PC table:"
)

print(
    f"  Rows: {len(motor_pc_df)}"
)

print(
    f"  Subjects: "
    f"{motor_pc_df['Subject'].nunique()}"
)

print(
    f"  Videos: "
    f"{motor_pc_df['BaseVideo'].nunique()}"
)

print(
    f"  Modalities: "
    f"{sorted(motor_pc_df['Modality'].dropna().unique())}"
)

print(
    f"  Missing trigger mappings: "
    f"{motor_pc_df['Trigger'].isna().sum()}"
)


# ============================================================
# 7. PREPARE LEFT MOTOR ROI ISC TABLE
# ============================================================

if "motor_control_isc_wide_df" in globals():

    motor_roi_df = (
        motor_control_isc_wide_df
        .copy()
    )

    required_columns = [
        "Subject",
        "Trigger"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in motor_roi_df.columns
    ]

    if missing_columns:
        raise ValueError(
            "motor_control_isc_wide_df is missing:\n"
            f"{missing_columns}"
        )

    motor_roi_df["Subject"] = (
        motor_roi_df["Subject"]
        .apply(normalize_subject_id)
    )

    motor_roi_df["Trigger"] = (
        motor_roi_df["Trigger"]
        .apply(normalize_trigger)
    )

    left_z_column = first_existing_column(
        motor_roi_df,
        [
            "Left_Motor_ROI_ISC_z",
            "Left_Motor_Control_ROI_ISC_z",
            "Left Motor Control ROI_z",
            "Left_Motor_ISC_z"
        ]
    )

    left_raw_column = first_existing_column(
        motor_roi_df,
        [
            "Left_Motor_ROI_ISC",
            "Left_Motor_Control_ROI_ISC",
            "Left Motor Control ROI",
            "Left_Motor_ISC"
        ]
    )

    if left_z_column is not None:

        motor_roi_df[
            "Left_Motor_ROI_ISC_z"
        ] = pd.to_numeric(
            motor_roi_df[left_z_column],
            errors="coerce"
        )

    elif left_raw_column is not None:

        motor_roi_df[
            "Left_Motor_ROI_ISC_z"
        ] = fisher_z(
            motor_roi_df[left_raw_column]
        )

    else:

        raise ValueError(
            "Could not locate the left motor ROI ISC "
            "column in motor_control_isc_wide_df.\n\n"
            f"Available columns:\n"
            f"{list(motor_roi_df.columns)}"
        )

    motor_roi_df = (
        motor_roi_df[
            [
                "Subject",
                "Trigger",
                "Left_Motor_ROI_ISC_z"
            ]
        ]
        .groupby(
            [
                "Subject",
                "Trigger"
            ],
            as_index=False
        )
        .mean()
    )


# ------------------------------------------------------------
# Fallback: long-format motor ROI ISC table
# ------------------------------------------------------------

else:

    motor_roi_long_df = (
        motor_video_roi_isc_df
        .copy()
    )

    required_columns = [
        "Subject",
        "Trigger",
        "ROI",
        "ISC"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in motor_roi_long_df.columns
    ]

    if missing_columns:
        raise ValueError(
            "motor_video_roi_isc_df is missing:\n"
            f"{missing_columns}"
        )

    motor_roi_long_df["Subject"] = (
        motor_roi_long_df["Subject"]
        .apply(normalize_subject_id)
    )

    motor_roi_long_df["Trigger"] = (
        motor_roi_long_df["Trigger"]
        .apply(normalize_trigger)
    )

    motor_roi_long_df["ROI"] = (
        motor_roi_long_df["ROI"]
        .astype(str)
        .str.strip()
    )

    motor_roi_long_df["ISC"] = pd.to_numeric(
        motor_roi_long_df["ISC"],
        errors="coerce"
    )

    available_motor_rois = sorted(
        motor_roi_long_df[
            "ROI"
        ]
        .dropna()
        .unique()
    )

    left_motor_candidates = [
        roi
        for roi in available_motor_rois
        if (
            "left" in roi.lower()
            and
            "motor" in roi.lower()
        )
    ]

    if len(left_motor_candidates) == 0:

        raise ValueError(
            "No ROI containing both 'Left' and 'Motor' "
            "was found.\n\n"
            f"Available ROI labels:\n"
            f"{available_motor_rois}"
        )

    selected_left_motor_roi = (
        left_motor_candidates[0]
    )

    print(
        "\nUsing left motor ROI label:"
    )

    print(
        selected_left_motor_roi
    )

    motor_roi_df = (
        motor_roi_long_df.loc[
            motor_roi_long_df["ROI"]
            == selected_left_motor_roi,
            [
                "Subject",
                "Trigger",
                "ISC"
            ]
        ]
        .groupby(
            [
                "Subject",
                "Trigger"
            ],
            as_index=False
        )
        .mean()
    )

    motor_roi_df[
        "Left_Motor_ROI_ISC_z"
    ] = fisher_z(
        motor_roi_df["ISC"]
    )

    motor_roi_df = (
        motor_roi_df[
            [
                "Subject",
                "Trigger",
                "Left_Motor_ROI_ISC_z"
            ]
        ]
    )


# ============================================================
# 8. MERGE MOTOR BEHAVIOR WITH MOTOR ROI ISC
# ============================================================

motor_validation_df = (
    motor_pc_df
    .merge(
        motor_roi_df,
        on=[
            "Subject",
            "Trigger"
        ],
        how="left",
        validate="many_to_one"
    )
)


motor_validation_df = (
    motor_validation_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


motor_validation_df.to_csv(
    MOTOR_VALIDATION_TABLE_DIR
    / "subject_video_motor_roi_validation_data.csv",
    index=False
)


print(
    "\nMerged motor-validation table:"
)

print(
    f"  Total rows: "
    f"{len(motor_validation_df)}"
)

print(
    f"  Subjects: "
    f"{motor_validation_df['Subject'].nunique()}"
)

print(
    f"  Videos: "
    f"{motor_validation_df['BaseVideo'].nunique()}"
)

print(
    f"  Rows with valid left motor ROI ISC: "
    f"{motor_validation_df['Left_Motor_ROI_ISC_z'].notna().sum()}"
)

print(
    f"  Rows missing left motor ROI ISC: "
    f"{motor_validation_df['Left_Motor_ROI_ISC_z'].isna().sum()}"
)


# ============================================================
# 9. POOLED PEARSON / SPEARMAN CORRELATIONS
#
# These are descriptive because subject × video observations
# are not independent.
# ============================================================

correlation_rows = []


for modality in [
    "Valence",
    "Arousal"
]:

    modality_df = (
        motor_validation_df.loc[
            motor_validation_df["Modality"]
            == modality
        ]
        .copy()
    )

    for motor_pc in MOTOR_PC_COLUMNS:

        temp = (
            modality_df[
                [
                    "Subject",
                    "BaseVideo",
                    motor_pc,
                    "Left_Motor_ROI_ISC_z"
                ]
            ]
            .dropna()
        )

        if len(temp) < MINIMUM_N:
            continue

        pearson_r, pearson_p, n_value = (
            safe_pearson(
                temp[motor_pc],
                temp[
                    "Left_Motor_ROI_ISC_z"
                ]
            )
        )

        spearman_rho, spearman_p, _ = (
            safe_spearman(
                temp[motor_pc],
                temp[
                    "Left_Motor_ROI_ISC_z"
                ]
            )
        )

        correlation_rows.append({
            "Modality": modality,
            "MotorPC": motor_pc,
            "N_SubjectVideo": n_value,
            "N_Subjects": (
                temp["Subject"].nunique()
            ),
            "N_Videos": (
                temp["BaseVideo"].nunique()
            ),
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_rho,
            "Spearman_p": spearman_p
        })


motor_roi_correlation_df = pd.DataFrame(
    correlation_rows
)


# ============================================================
# 10. VIDEO-PRESERVING PERMUTATION TEST
#
# Within each video, shuffle the participant labels associated
# with Left Motor ROI ISC.
#
# This preserves:
#   - each video's motor-ISC distribution
#   - each video's Motor-PC distribution
#
# while destroying:
#   - the participant-level correspondence between movement
#     and motor-region ISC.
#
# Run separately by modality and Motor PC.
# ============================================================

permutation_results = []


for modality in [
    "Valence",
    "Arousal"
]:

    modality_df = (
        motor_validation_df.loc[
            motor_validation_df["Modality"]
            == modality
        ]
        .copy()
    )

    for motor_pc in MOTOR_PC_COLUMNS:

        temp = (
            modality_df[
                [
                    "Subject",
                    "BaseVideo",
                    motor_pc,
                    "Left_Motor_ROI_ISC_z"
                ]
            ]
            .dropna()
            .reset_index(drop=True)
        )

        if len(temp) < MINIMUM_N:
            continue

        observed_r = pearsonr(
            temp[motor_pc],
            temp[
                "Left_Motor_ROI_ISC_z"
            ]
        )[0]

        video_indices = {
            video: group.index.to_numpy()
            for video, group in temp.groupby(
                "BaseVideo",
                sort=False
            )
        }

        original_motor_isc = (
            temp[
                "Left_Motor_ROI_ISC_z"
            ]
            .to_numpy(dtype=float)
        )

        motor_pc_values = (
            temp[motor_pc]
            .to_numpy(dtype=float)
        )

        null_correlations = np.empty(
            N_PERMUTATIONS,
            dtype=float
        )

        for permutation_index in range(
            N_PERMUTATIONS
        ):

            permuted_motor_isc = (
                original_motor_isc.copy()
            )

            for indices in (
                video_indices.values()
            ):

                if len(indices) > 1:

                    permuted_motor_isc[
                        indices
                    ] = RNG.permutation(
                        permuted_motor_isc[
                            indices
                        ]
                    )

            null_correlations[
                permutation_index
            ] = pearsonr(
                motor_pc_values,
                permuted_motor_isc
            )[0]

        permutation_p = (
            1
            +
            np.sum(
                np.abs(
                    null_correlations
                )
                >= abs(
                    observed_r
                )
            )
        ) / (
            N_PERMUTATIONS
            + 1
        )

        permutation_results.append({
            "Modality": modality,
            "MotorPC": motor_pc,
            "Observed_Pearson_r": observed_r,
            "Permutation_p": permutation_p,
            "N_Permutations": N_PERMUTATIONS
        })


motor_roi_permutation_df = pd.DataFrame(
    permutation_results
)


# ============================================================
# 11. CROSSED MIXED-EFFECTS MODELS
#
# Primary inference:
#
# Left Motor ISC ~ Motor PC
#                  + random intercept Subject
#                  + random intercept Video
#
# statsmodels implementation:
#
# groups = Subject
# vc_formula gives crossed Video random intercept.
# ============================================================

mixed_model_rows = []


for modality in [
    "Valence",
    "Arousal"
]:

    modality_df = (
        motor_validation_df.loc[
            motor_validation_df["Modality"]
            == modality
        ]
        .copy()
    )

    for motor_pc in MOTOR_PC_COLUMNS:

        temp = (
            modality_df[
                [
                    "Subject",
                    "BaseVideo",
                    motor_pc,
                    "Left_Motor_ROI_ISC_z"
                ]
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
            .copy()
        )

        if (
            len(temp) < MINIMUM_N
            or
            temp["Subject"].nunique() < 5
            or
            temp["BaseVideo"].nunique() < 5
        ):
            continue

        # ----------------------------------------------------
        # Standardize within analysis sample.
        #
        # This makes beta directly interpretable as:
        # SD change in motor ROI ISC associated with a
        # 1-SD increase in the motor PC.
        # ----------------------------------------------------

        temp[
            "MotorPC_z"
        ] = (
            temp[motor_pc]
            - temp[motor_pc].mean()
        ) / temp[motor_pc].std(
            ddof=1
        )

        temp[
            "LeftMotorISC_zstd"
        ] = (
            temp["Left_Motor_ROI_ISC_z"]
            - temp[
                "Left_Motor_ROI_ISC_z"
            ].mean()
        ) / temp[
            "Left_Motor_ROI_ISC_z"
        ].std(
            ddof=1
        )

        formula = (
            "LeftMotorISC_zstd "
            "~ MotorPC_z"
        )

        model = smf.mixedlm(
            formula=formula,
            data=temp,
            groups=temp[
                "Subject"
            ],
            re_formula="1",
            vc_formula={
                "Video": (
                    "0 + C(BaseVideo)"
                )
            }
        )

        fit_result = None
        fit_method = None

        # ----------------------------------------------------
        # Try several optimizers for robustness.
        # ----------------------------------------------------

        for method in [
            "lbfgs",
            "powell",
            "nm"
        ]:

            try:

                candidate_result = (
                    model.fit(
                        reml=False,
                        method=method,
                        maxiter=5000,
                        disp=False
                    )
                )

                fit_result = (
                    candidate_result
                )

                fit_method = method

                break

            except Exception:
                continue

        if fit_result is None:

            print(
                f"\nWARNING: Mixed model failed for "
                f"{modality} / {motor_pc}"
            )

            continue

        beta = fit_result.params[
            "MotorPC_z"
        ]

        se = fit_result.bse[
            "MotorPC_z"
        ]

        z_value = (
            beta / se
        )

        p_value = fit_result.pvalues[
            "MotorPC_z"
        ]

        ci = fit_result.conf_int().loc[
            "MotorPC_z"
        ]

        mixed_model_rows.append({
            "Modality": modality,
            "MotorPC": motor_pc,
            "N_SubjectVideo": len(temp),
            "N_Subjects": (
                temp["Subject"].nunique()
            ),
            "N_Videos": (
                temp["BaseVideo"].nunique()
            ),
            "Standardized_Beta": beta,
            "SE": se,
            "Z": z_value,
            "P": p_value,
            "CI95_Lower": ci.iloc[0],
            "CI95_Upper": ci.iloc[1],
            "Optimizer": fit_method,
            "Converged": bool(
                getattr(
                    fit_result,
                    "converged",
                    False
                )
            )
        })

        # ----------------------------------------------------
        # Save full model summary.
        # ----------------------------------------------------

        summary_path = (
            MOTOR_VALIDATION_MODEL_DIR
            /
            (
                f"{modality}_"
                f"{motor_pc}_"
                f"left_motor_roi_mixed_model.txt"
            )
        )

        with open(
            summary_path,
            "w",
            encoding="utf-8"
        ) as file:

            file.write(
                fit_result.summary().as_text()
            )


motor_roi_mixed_model_df = pd.DataFrame(
    mixed_model_rows
)


# ============================================================
# 12. FDR CORRECTION
#
# Correct PC1–PC3 tests separately within Valence and Arousal.
# ============================================================

if not motor_roi_correlation_df.empty:

    motor_roi_correlation_df[
        "Pearson_p_FDR"
    ] = np.nan

    motor_roi_correlation_df[
        "Spearman_p_FDR"
    ] = np.nan

    for modality in (
        motor_roi_correlation_df[
            "Modality"
        ].unique()
    ):

        mask = (
            motor_roi_correlation_df[
                "Modality"
            ]
            == modality
        )

        motor_roi_correlation_df.loc[
            mask,
            "Pearson_p_FDR"
        ] = multipletests(
            motor_roi_correlation_df.loc[
                mask,
                "Pearson_p"
            ],
            method="fdr_bh"
        )[1]

        motor_roi_correlation_df.loc[
            mask,
            "Spearman_p_FDR"
        ] = multipletests(
            motor_roi_correlation_df.loc[
                mask,
                "Spearman_p"
            ],
            method="fdr_bh"
        )[1]


if not motor_roi_permutation_df.empty:

    motor_roi_permutation_df[
        "Permutation_p_FDR"
    ] = np.nan

    for modality in (
        motor_roi_permutation_df[
            "Modality"
        ].unique()
    ):

        mask = (
            motor_roi_permutation_df[
                "Modality"
            ]
            == modality
        )

        motor_roi_permutation_df.loc[
            mask,
            "Permutation_p_FDR"
        ] = multipletests(
            motor_roi_permutation_df.loc[
                mask,
                "Permutation_p"
            ],
            method="fdr_bh"
        )[1]


if not motor_roi_mixed_model_df.empty:

    motor_roi_mixed_model_df[
        "P_FDR"
    ] = np.nan

    for modality in (
        motor_roi_mixed_model_df[
            "Modality"
        ].unique()
    ):

        mask = (
            motor_roi_mixed_model_df[
                "Modality"
            ]
            == modality
        )

        motor_roi_mixed_model_df.loc[
            mask,
            "P_FDR"
        ] = multipletests(
            motor_roi_mixed_model_df.loc[
                mask,
                "P"
            ],
            method="fdr_bh"
        )[1]


# ============================================================
# 13. MERGE SUMMARY TABLE
# ============================================================

motor_roi_validation_summary_df = (
    motor_roi_correlation_df
    .merge(
        motor_roi_permutation_df,
        on=[
            "Modality",
            "MotorPC"
        ],
        how="outer"
    )
    .merge(
        motor_roi_mixed_model_df,
        on=[
            "Modality",
            "MotorPC"
        ],
        how="outer",
        suffixes=(
            "_Correlation",
            "_MixedModel"
        )
    )
)


# ============================================================
# 14. SAVE TABLES
# ============================================================

motor_roi_correlation_df.to_csv(
    MOTOR_VALIDATION_TABLE_DIR
    / "motor_roi_motor_pc_correlations.csv",
    index=False
)

motor_roi_permutation_df.to_csv(
    MOTOR_VALIDATION_TABLE_DIR
    / "motor_roi_motor_pc_permutation_tests.csv",
    index=False
)

motor_roi_mixed_model_df.to_csv(
    MOTOR_VALIDATION_TABLE_DIR
    / "motor_roi_motor_pc_mixed_models.csv",
    index=False
)

motor_roi_validation_summary_df.to_csv(
    MOTOR_VALIDATION_TABLE_DIR
    / "motor_roi_validation_complete_summary.csv",
    index=False
)


# ============================================================
# 15. SCATTERPLOTS
# ============================================================

for modality in [
    "Valence",
    "Arousal"
]:

    modality_df = (
        motor_validation_df.loc[
            motor_validation_df["Modality"]
            == modality
        ]
    )

    for motor_pc in MOTOR_PC_COLUMNS:

        plot_df = (
            modality_df[
                [
                    motor_pc,
                    "Left_Motor_ROI_ISC_z"
                ]
            ]
            .dropna()
        )

        if len(plot_df) < MINIMUM_N:
            continue

        r_value, p_value = pearsonr(
            plot_df[motor_pc],
            plot_df[
                "Left_Motor_ROI_ISC_z"
            ]
        )

        x = (
            plot_df[motor_pc]
            .to_numpy(dtype=float)
        )

        y = (
            plot_df[
                "Left_Motor_ROI_ISC_z"
            ]
            .to_numpy(dtype=float)
        )

        slope, intercept = np.polyfit(
            x,
            y,
            1
        )

        x_line = np.linspace(
            np.min(x),
            np.max(x),
            200
        )

        y_line = (
            intercept
            + slope * x_line
        )

        fig, ax = plt.subplots(
            figsize=(7, 6)
        )

        ax.scatter(
            x,
            y,
            alpha=0.55
        )

        ax.plot(
            x_line,
            y_line,
            linewidth=2
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1
        )

        ax.set_xlabel(
            motor_pc
        )

        ax.set_ylabel(
            "Left motor ROI ISC (Fisher z)"
        )

        ax.set_title(
            f"{modality}: Left motor ROI ISC × {motor_pc}\n"
            f"pooled Pearson r = {r_value:.3f}, "
            f"p = {p_value:.4f}"
        )

        fig.tight_layout()

        fig.savefig(
            MOTOR_VALIDATION_FIGURE_DIR
            /
            (
                f"{modality.lower()}_"
                f"{motor_pc.lower()}_"
                f"left_motor_roi_scatter.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 16. MIXED-MODEL EFFECT SUMMARY FIGURE
# ============================================================

if not motor_roi_mixed_model_df.empty:

    plot_df = (
        motor_roi_mixed_model_df
        .copy()
    )

    plot_df[
        "Label"
    ] = (
        plot_df["Modality"]
        + " | "
        + plot_df["MotorPC"]
    )

    x_positions = np.arange(
        len(plot_df)
    )

    beta_values = (
        plot_df[
            "Standardized_Beta"
        ].to_numpy()
    )

    lower_errors = (
        beta_values
        -
        plot_df[
            "CI95_Lower"
        ].to_numpy()
    )

    upper_errors = (
        plot_df[
            "CI95_Upper"
        ].to_numpy()
        -
        beta_values
    )

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.errorbar(
        x_positions,
        beta_values,
        yerr=[
            lower_errors,
            upper_errors
        ],
        fmt="o",
        capsize=5
    )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        plot_df["Label"],
        rotation=35,
        ha="right"
    )

    ax.set_ylabel(
        "Standardized association with left motor ROI ISC"
    )

    ax.set_title(
        "Motor behavior predicts left motor-control ROI ISC\n"
        "Crossed subject × video mixed-effects models"
    )

    fig.tight_layout()

    fig.savefig(
        MOTOR_VALIDATION_FIGURE_DIR
        / "motor_roi_mixed_model_effect_summary.png",
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 17. PRINT RESULTS
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "LEFT MOTOR ROI VALIDATION COMPLETE"
)

print(
    "=" * 90
)


print(
    "\nDESCRIPTIVE POOLED CORRELATIONS"
)

if not motor_roi_correlation_df.empty:

    display(
        motor_roi_correlation_df[
            [
                "Modality",
                "MotorPC",
                "N_SubjectVideo",
                "N_Subjects",
                "N_Videos",
                "Pearson_r",
                "Pearson_p",
                "Pearson_p_FDR",
                "Spearman_rho",
                "Spearman_p",
                "Spearman_p_FDR"
            ]
        ]
        .round(4)
    )


print(
    "\nVIDEO-PRESERVING PERMUTATION TESTS"
)

if not motor_roi_permutation_df.empty:

    display(
        motor_roi_permutation_df[
            [
                "Modality",
                "MotorPC",
                "Observed_Pearson_r",
                "Permutation_p",
                "Permutation_p_FDR"
            ]
        ]
        .round(4)
    )


print(
    "\nPRIMARY CROSSED MIXED-EFFECTS MODELS"
)

if not motor_roi_mixed_model_df.empty:

    display(
        motor_roi_mixed_model_df[
            [
                "Modality",
                "MotorPC",
                "N_SubjectVideo",
                "N_Subjects",
                "N_Videos",
                "Standardized_Beta",
                "SE",
                "Z",
                "P",
                "P_FDR",
                "CI95_Lower",
                "CI95_Upper",
                "Converged"
            ]
        ]
        .round(4)
    )


# ============================================================
# 18. AUTOMATIC INTERPRETATION FLAGS
# ============================================================

if not motor_roi_mixed_model_df.empty:

    interpretation_df = (
        motor_roi_mixed_model_df[
            [
                "Modality",
                "MotorPC",
                "Standardized_Beta",
                "P",
                "P_FDR",
                "CI95_Lower",
                "CI95_Upper"
            ]
        ]
        .copy()
    )

    interpretation_df[
        "Significant_Uncorrected"
    ] = (
        interpretation_df["P"]
        < 0.05
    )

    interpretation_df[
        "Significant_FDR"
    ] = (
        interpretation_df["P_FDR"]
        < 0.05
    )

    interpretation_df[
        "CI_Excludes_Zero"
    ] = (
        (
            interpretation_df[
                "CI95_Lower"
            ] > 0
        )
        |
        (
            interpretation_df[
                "CI95_Upper"
            ] < 0
        )
    )

    interpretation_df.to_csv(
        MOTOR_VALIDATION_TABLE_DIR
        / "motor_roi_validation_significance_flags.csv",
        index=False
    )

    print(
        "\nSIGNIFICANCE SUMMARY"
    )

    display(
        interpretation_df
        .round(4)
    )


print(
    "\nInterpretation guide:"
)

print(
    """
1. The pooled Pearson/Spearman correlations are descriptive,
   because subject × video observations are repeated.

2. The PRIMARY inference is the crossed mixed-effects model,
   because it accounts for repeated measurements within both
   participants and videos.

3. A positive standardized beta means that higher scores on
   that Motor PC are associated with greater left motor ROI ISC.

4. A negative standardized beta means that higher scores on
   that Motor PC are associated with lower left motor ROI ISC.
   Whether this represents 'more' or 'less' motor movement
   depends on the PCA loadings for that component.

5. The video-preserving permutation test asks whether the
   observed MotorPC–motor-ROI association is stronger than
   expected after breaking the participant-specific coupling
   between mouse movement and motor-region ISC while preserving
   each video's distribution.

6. Evidence that at least one retained Motor PC significantly
   predicts left motor ROI ISC—particularly if supported by the
   mixed model and permutation test—would empirically validate
   the motor-control ROI as carrying behaviorally relevant
   motor information.
"""
)


print(
    "\nSaved all outputs to:"
)

print(
    MOTOR_VALIDATION_OUT
)

In [ ]:
# ============================================================
# 
# DIRECT FUNCTIONAL VALIDATION OF LEFT MOTOR-CONTROL ROI
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# This analysis provides two complementary tests of whether the
# anatomically defined left motor-control ROI contains information
# related to actual mouse movement.
#
#
# PART A — INDIVIDUAL KINEMATIC FEATURES × LEFT MOTOR ROI ISC
# ------------------------------------------------------------
# For every Subject × Video × Modality observation:
#
#     Left motor-control ROI ISC
#         ~ individual hand-kinematic feature
#
# Tests:
#   1. Pooled Pearson correlation
#   2. Pooled Spearman correlation
#   3. Crossed subject × video mixed-effects model:
#
#        LeftMotorISC_zstd ~ Feature_z
#                          + (1 | Subject)
#                          + (1 | Video)
#
#   4. Video-preserving permutation test
#   5. FDR correction across all kinematic features
#
#
# PART B — CONTINUOUS MOUSE MOVEMENT × LEFT MOTOR HbO
# ------------------------------------------------------------
# Uses the actual preprocessed left motor-control ROI HbO
# time series stored in:
#
#     motor_roi_ts[Subject][Trigger]["Left Motor Control ROI"]
#
# and the cleaned behavioral trajectories stored in:
#
#     trajectory_cache[(Subject, Video)]
#
# Mouse behavior is interpolated onto the 1.5-Hz fNIRS time grid.
#
# Continuous predictors:
#   AbsVelocity
#   AbsAcceleration
#   MovementOnset
#
# For each Subject × Video × Modality, correlations are calculated
# at neural lags from 0 to 8 seconds.
#
# Positive lag means:
#
#     mouse movement occurs FIRST
#     neural HbO is sampled LAG seconds later
#
# Example:
#     lag = 5 s tests whether movement at time t predicts
#     left-motor HbO at approximately t + 5 s.
#
#
# GROUP-LEVEL CONTINUOUS TESTS
# ------------------------------------------------------------
# 1. Save subject × video correlations at every lag.
# 2. Fisher-z transform within-trial correlations.
# 3. Fit crossed subject × video mixed models at each lag:
#
#        FisherZ_r ~ 1
#                  + (1 | Subject)
#                  + (1 | Video)
#
#    The intercept tests whether the mean within-trial
#    movement–HbO coupling differs from zero.
#
# 4. FDR correct across lag tests within each modality ×
#    continuous predictor.
#
# 5. Circular-shift permutation test:
#    the movement trajectory is circularly shifted relative to
#    HbO, preserving its temporal autocorrelation while breaking
#    the true alignment.
#
#
# REQUIRED NOTEBOOK OBJECTS
# ------------------------------------------------------------
#   motor_kinematic_features_df   <- Analysis 01
#   trajectory_cache             <- Analysis 01
#   motor_roi_ts                 <- bilateral motor ROI pipeline
#
# and one of:
#   motor_control_isc_wide_df
#   motor_video_roi_isc_df
#
#
# REQUIRED FILE
# ------------------------------------------------------------
#   video_mapping.csv
#
#
# OUTPUT
# ------------------------------------------------------------
# C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\
# Emotion Perception Data\Motor_Control\
# 08_direct_motor_roi_validation
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    pearsonr,
    spearmanr
)

import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests


warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)


# ============================================================
# 1. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "motor_kinematic_features_df",
    "trajectory_cache",
    "motor_roi_ts"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:

    raise ValueError(
        "The following required notebook objects were not found:\n"
        f"{missing_objects}\n\n"
        "Run the hand-kinematic feature extraction and bilateral "
        "motor-control ROI pipeline first."
    )


if (
    "motor_control_isc_wide_df" not in globals()
    and
    "motor_video_roi_isc_df" not in globals()
):

    raise ValueError(
        "Neither motor_control_isc_wide_df nor "
        "motor_video_roi_isc_df was found.\n\n"
        "Run the bilateral motor-control ROI ISC pipeline first."
    )


# ============================================================
# 2. PATHS
# ============================================================

VIDEO_MAPPING_CSV = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\video_mapping.csv"
)

if not VIDEO_MAPPING_CSV.exists():

    alternative_paths = [

        Path.cwd() / "video_mapping.csv",

        Path(
            r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
            r"\Emotion Perception Data\Motor_Control"
            r"\video_mapping.csv"
        ),

        Path(
            r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
            r"\Emotion Perception Data\Behavioral Data"
            r"\video_mapping.csv"
        )
    ]

    matching_path = next(
        (
            path
            for path in alternative_paths
            if path.exists()
        ),
        None
    )

    if matching_path is None:

        raise FileNotFoundError(
            "video_mapping.csv could not be found.\n\n"
            "Update VIDEO_MAPPING_CSV near the top of this cell."
        )

    VIDEO_MAPPING_CSV = matching_path


DIRECT_MOTOR_OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\Motor_Control"
    r"\08_direct_motor_roi_validation"
)

FEATURE_OUTPUT_DIR = (
    DIRECT_MOTOR_OUTPUT_DIR
    / "01_individual_kinematic_features"
)

CONTINUOUS_OUTPUT_DIR = (
    DIRECT_MOTOR_OUTPUT_DIR
    / "02_continuous_motor_tracking"
)

TABLE_DIR = (
    DIRECT_MOTOR_OUTPUT_DIR
    / "tables"
)

FIGURE_DIR = (
    DIRECT_MOTOR_OUTPUT_DIR
    / "figures"
)

FEATURE_FIGURE_DIR = (
    FIGURE_DIR
    / "individual_features"
)

CONTINUOUS_FIGURE_DIR = (
    FIGURE_DIR
    / "continuous_tracking"
)

MODEL_DIR = (
    DIRECT_MOTOR_OUTPUT_DIR
    / "model_summaries"
)

for folder in [
    DIRECT_MOTOR_OUTPUT_DIR,
    FEATURE_OUTPUT_DIR,
    CONTINUOUS_OUTPUT_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    FEATURE_FIGURE_DIR,
    CONTINUOUS_FIGURE_DIR,
    MODEL_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "Direct motor ROI validation outputs will be saved to:\n"
    f"{DIRECT_MOTOR_OUTPUT_DIR}"
)

print(
    "\nVideo mapping file:\n"
    f"{VIDEO_MAPPING_CSV}"
)


# ============================================================
# 3. USER-ADJUSTABLE SETTINGS
# ============================================================

RANDOM_SEED = 2026

RNG = np.random.default_rng(
    RANDOM_SEED
)

FIGURE_DPI = 300


# ------------------------------------------------------------
# Part A settings
# ------------------------------------------------------------

MINIMUM_FEATURE_N = 20

# 10,000 for final analysis.
N_FEATURE_PERMUTATIONS = 10000


# ------------------------------------------------------------
# Part B settings
# ------------------------------------------------------------

LEFT_MOTOR_ROI_NAME = (
    "Left Motor Control ROI"
)

# Your motor preprocessing pipeline explicitly resamples to 1.5 Hz.
FNIRS_SAMPLING_RATE_HZ = 1.5

# Positive lag:
# movement precedes neural HbO.
#
# 0, 0.5, 1.0 ... 8.0 s
LAG_SECONDS = np.arange(
    0.0,
    8.0 + 0.001,
    0.5
)

CONTINUOUS_PREDICTORS = [
    "AbsVelocity",
    "AbsAcceleration",
    "MovementOnset"
]

MIN_CONTINUOUS_SAMPLES = 30

# Movement threshold copied from the hand-kinematic extraction cell.
MOVEMENT_THRESHOLD = (
    MOVEMENT_VELOCITY_THRESHOLD
    if "MOVEMENT_VELOCITY_THRESHOLD" in globals()
    else 0.05
)

# Minimum period the participant needs to remain below threshold
# before the next upward transition counts as an onset.
MIN_ONSET_QUIET_SEC = 0.25

# Number of circular-shift permutations.
#
# Start with 1000 for speed.
# Increase to 5000–10000 for final inference.
N_CONTINUOUS_PERMUTATIONS = 1000

# To avoid trivial shifts almost identical to the original alignment,
# circular shifts must move behavior by at least this many seconds.
MIN_CIRCULAR_SHIFT_SEC = 5.0


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_subject_id(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"\.(csv|snirf)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_video_name(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(
        r"_(valence|arousal)$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


def normalize_trigger(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    try:

        numeric_value = float(
            value
        )

        if numeric_value.is_integer():

            return str(
                int(numeric_value)
            )

    except (
        ValueError,
        TypeError
    ):

        pass

    return value


def safe_filename(value):

    value = str(value)

    value = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        value
    )

    value = re.sub(
        r"\s+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    return value.strip(
        "._ "
    )


def zscore_series(series):

    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    sd = series.std(
        ddof=1
    )

    if (
        not np.isfinite(sd)
        or
        sd <= 1e-12
    ):

        return pd.Series(
            np.nan,
            index=series.index
        )

    return (
        series
        - series.mean()
    ) / sd


def safe_pearson(
    x,
    y,
    minimum_n=5
):

    temp = pd.DataFrame({
        "x": x,
        "y": y
    }).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    n = len(
        temp
    )

    if n < minimum_n:

        return (
            np.nan,
            np.nan,
            n
        )

    if (
        temp["x"].std(ddof=1) <= 1e-12
        or
        temp["y"].std(ddof=1) <= 1e-12
    ):

        return (
            np.nan,
            np.nan,
            n
        )

    r_value, p_value = pearsonr(
        temp["x"],
        temp["y"]
    )

    return (
        float(r_value),
        float(p_value),
        n
    )


def safe_spearman(
    x,
    y,
    minimum_n=5
):

    temp = pd.DataFrame({
        "x": x,
        "y": y
    }).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    n = len(
        temp
    )

    if n < minimum_n:

        return (
            np.nan,
            np.nan,
            n
        )

    rho_value, p_value = spearmanr(
        temp["x"],
        temp["y"]
    )

    return (
        float(rho_value),
        float(p_value),
        n
    )


def fisher_z_from_r(r_value):

    if not np.isfinite(
        r_value
    ):
        return np.nan

    return np.arctanh(
        np.clip(
            r_value,
            -0.999999,
            0.999999
        )
    )


def extract_modality_from_video(
    video_name
):

    match = re.search(
        r"_(valence|arousal)$",
        str(video_name),
        flags=re.IGNORECASE
    )

    if match is None:
        return None

    return (
        match.group(1)
        .capitalize()
    )


def find_existing_column(
    dataframe,
    candidates
):

    for column in candidates:

        if column in dataframe.columns:

            return column

    return None


# ============================================================
# 5. LOAD VIDEO MAPPING
# ============================================================

video_mapping_raw = pd.read_csv(
    VIDEO_MAPPING_CSV
)

video_column = find_existing_column(
    video_mapping_raw,
    [
        "video",
        "Video",
        "BaseVideo",
        "video_name",
        "VideoName"
    ]
)

trigger_column = find_existing_column(
    video_mapping_raw,
    [
        "trigger_number",
        "Trigger",
        "trigger",
        "TriggerNumber"
    ]
)

if (
    video_column is None
    or
    trigger_column is None
):

    raise ValueError(
        "video_mapping.csv must contain video and trigger columns.\n\n"
        f"Columns found:\n{list(video_mapping_raw.columns)}"
    )


video_mapping_df = pd.DataFrame({

    "BaseVideo":
        video_mapping_raw[
            video_column
        ].map(
            normalize_video_name
        ),

    "Trigger":
        video_mapping_raw[
            trigger_column
        ].map(
            normalize_trigger
        )

})

video_mapping_df = (
    video_mapping_df
    .dropna()
    .drop_duplicates(
        subset=[
            "BaseVideo"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    f"\nLoaded {len(video_mapping_df)} "
    "video-to-trigger mappings."
)

display(
    video_mapping_df
)


# ============================================================
# ============================================================
# PART A
# INDIVIDUAL KINEMATIC VARIABLES × LEFT MOTOR ROI ISC
# ============================================================
# ============================================================


# ============================================================
# 6. PREPARE LEFT MOTOR ISC TABLE
# ============================================================

if "motor_control_isc_wide_df" in globals():

    left_motor_source_df = (
        motor_control_isc_wide_df
        .copy()
    )

else:

    left_motor_source_df = (
        motor_video_roi_isc_df
        .copy()
    )


# ------------------------------------------------------------
# Convert long ROI table to wide if necessary
# ------------------------------------------------------------

if (
    "ROI" in left_motor_source_df.columns
    and
    "ISC" in left_motor_source_df.columns
):

    left_motor_long = (
        left_motor_source_df[
            left_motor_source_df["ROI"]
            == LEFT_MOTOR_ROI_NAME
        ]
        .copy()
    )

    left_motor_feature_df = (
        left_motor_long[
            [
                "Subject",
                "Trigger",
                "ISC"
            ]
        ]
        .rename(
            columns={
                "ISC":
                "Left_Motor_ROI_ISC"
            }
        )
    )

else:

    left_motor_column = find_existing_column(
        left_motor_source_df,
        [
            "Left_Motor_ROI_ISC_z",
            "Left_Motor_ROI_ISC",
            "Left Motor Control ROI",
            "Left_Motor_Control_ROI"
        ]
    )

    if left_motor_column is None:

        raise ValueError(
            "Could not identify the left motor ROI ISC column.\n\n"
            f"Available columns:\n"
            f"{list(left_motor_source_df.columns)}"
        )

    left_motor_feature_df = (
        left_motor_source_df[
            [
                "Subject",
                "Trigger",
                left_motor_column
            ]
        ]
        .rename(
            columns={
                left_motor_column:
                "Left_Motor_ROI_ISC"
            }
        )
    )


left_motor_feature_df[
    "Subject"
] = (
    left_motor_feature_df[
        "Subject"
    ].map(
        normalize_subject_id
    )
)

left_motor_feature_df[
    "Trigger"
] = (
    left_motor_feature_df[
        "Trigger"
    ].map(
        normalize_trigger
    )
)


# Fisher z only when source is in correlation units.
isc_values = pd.to_numeric(
    left_motor_feature_df[
        "Left_Motor_ROI_ISC"
    ],
    errors="coerce"
)

if (
    np.nanmax(
        np.abs(
            isc_values
        )
    )
    <= 1.000001
):

    left_motor_feature_df[
        "LeftMotorISC_z"
    ] = np.arctanh(
        np.clip(
            isc_values,
            -0.999999,
            0.999999
        )
    )

else:

    # Already appears to be Fisher-z transformed.
    left_motor_feature_df[
        "LeftMotorISC_z"
    ] = isc_values


# ============================================================
# 7. PREPARE KINEMATIC FEATURE TABLE
# ============================================================

feature_validation_df = (
    motor_kinematic_features_df
    .copy()
)

feature_validation_df[
    "Subject"
] = (
    feature_validation_df[
        "Subject"
    ].map(
        normalize_subject_id
    )
)

feature_validation_df[
    "BaseVideo"
] = (
    feature_validation_df[
        "BaseVideo"
    ].map(
        normalize_video_name
    )
)

feature_validation_df = pd.merge(
    feature_validation_df,
    video_mapping_df,
    on="BaseVideo",
    how="left"
)


feature_validation_df = pd.merge(
    feature_validation_df,
    left_motor_feature_df[
        [
            "Subject",
            "Trigger",
            "LeftMotorISC_z"
        ]
    ],
    on=[
        "Subject",
        "Trigger"
    ],
    how="left"
)


feature_validation_df[
    "LeftMotorISC_zstd"
] = (
    feature_validation_df
    .groupby(
        "Modality"
    )[
        "LeftMotorISC_z"
    ]
    .transform(
        zscore_series
    )
)


feature_validation_df.to_csv(
    FEATURE_OUTPUT_DIR
    / "individual_kinematic_features_with_left_motor_isc.csv",
    index=False
)


# ============================================================
# 8. DEFINE ACTUAL KINEMATIC FEATURES
# ============================================================

# These match the feature extraction cell in your notebook.
candidate_kinematic_features = [

    # First order
    "MeanAbsVelocity",
    "MedianAbsVelocity",
    "PeakAbsVelocity",
    "VelocitySD",
    "AbsVelocitySD",
    "ProportionMoving",

    # Second order
    "MeanAbsAcceleration",
    "PeakAbsAcceleration",
    "AccelerationSD",

    # Third order
    "MeanAbsJerk",
    "RMSJerk",
    "PeakAbsJerk",

    # Path / direction
    "PathLength",
    "PathLengthPerSecond",
    "NetDisplacementSigned",
    "NetDisplacementAbsolute",
    "PathEfficiency",
    "ReversalCount",
    "ReversalRate",

    # Pausing
    "PauseCount",
    "PauseRate",
    "TotalPauseDuration",
    "MeanPauseDuration",
    "PauseProportion",

    # Variability / complexity
    "PositionSD",
    "SmoothedPositionSD",
    "DimensionlessJerk",
    "Smoothness",
    "MovementEntropy"
]

kinematic_features = [
    feature
    for feature in candidate_kinematic_features
    if feature in feature_validation_df.columns
]

print(
    "\nIndividual hand-kinematic features tested:"
)

for feature in kinematic_features:
    print(
        f"  {feature}"
    )

print(
    f"\nTotal features: "
    f"{len(kinematic_features)}"
)


# ============================================================
# 9. FEATURE-LEVEL CORRELATIONS + MIXED MODELS
# ============================================================

feature_result_rows = []
feature_model_objects = {}


for modality in [
    "Valence",
    "Arousal"
]:

    modality_df = (
        feature_validation_df[
            feature_validation_df[
                "Modality"
            ] == modality
        ]
        .copy()
    )

    for feature in kinematic_features:

        temp = (
            modality_df[
                [
                    "Subject",
                    "BaseVideo",
                    "LeftMotorISC_zstd",
                    feature
                ]
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
            .copy()
        )

        if len(temp) < MINIMUM_FEATURE_N:
            continue

        if (
            temp[feature].std(
                ddof=1
            )
            <= 1e-12
        ):
            continue

        temp[
            "Feature_z"
        ] = zscore_series(
            temp[
                feature
            ]
        )

        # ----------------------------------------------
        # Pearson / Spearman
        # ----------------------------------------------

        (
            pearson_r,
            pearson_p,
            n_pearson
        ) = safe_pearson(
            temp[
                "Feature_z"
            ],
            temp[
                "LeftMotorISC_zstd"
            ],
            minimum_n=MINIMUM_FEATURE_N
        )

        (
            spearman_rho,
            spearman_p,
            _
        ) = safe_spearman(
            temp[
                "Feature_z"
            ],
            temp[
                "LeftMotorISC_zstd"
            ],
            minimum_n=MINIMUM_FEATURE_N
        )


        # ----------------------------------------------
        # Crossed mixed-effects model
        #
        # Left motor ISC ~ kinematic feature
        #                  + subject random intercept
        #                  + video random intercept
        # ----------------------------------------------

        mixed_beta = np.nan
        mixed_se = np.nan
        mixed_p = np.nan
        mixed_ci_low = np.nan
        mixed_ci_high = np.nan
        mixed_converged = False

        try:

            model = smf.mixedlm(

                "LeftMotorISC_zstd ~ Feature_z",

                data=temp,

                groups=temp[
                    "Subject"
                ],

                re_formula="1",

                vc_formula={
                    "Video":
                    "0 + C(BaseVideo)"
                }
            )

            result = model.fit(
                reml=False,
                method="lbfgs",
                maxiter=3000,
                disp=False
            )

            mixed_beta = float(
                result.params[
                    "Feature_z"
                ]
            )

            mixed_se = float(
                result.bse[
                    "Feature_z"
                ]
            )

            mixed_p = float(
                result.pvalues[
                    "Feature_z"
                ]
            )

            mixed_ci = result.conf_int().loc[
                "Feature_z"
            ]

            mixed_ci_low = float(
                mixed_ci.iloc[0]
            )

            mixed_ci_high = float(
                mixed_ci.iloc[1]
            )

            mixed_converged = bool(
                result.converged
            )

            feature_model_objects[
                (
                    modality,
                    feature
                )
            ] = result

            model_filename = (
                safe_filename(
                    f"{modality}_{feature}_mixed_model"
                )
                + ".txt"
            )

            with open(
                MODEL_DIR / model_filename,
                "w",
                encoding="utf-8"
            ) as output_file:

                output_file.write(
                    result.summary().as_text()
                )

        except Exception as error:

            print(
                f"Mixed model failed: "
                f"{modality} | {feature}\n"
                f"{error}"
            )


        # ----------------------------------------------
        # Video-preserving permutation
        #
        # Shuffle the feature ACROSS PARTICIPANTS WITHIN
        # EACH VIDEO.
        #
        # This preserves video-level movement structure
        # while breaking participant-specific coupling.
        # ----------------------------------------------

        observed_beta = (
            mixed_beta
            if np.isfinite(
                mixed_beta
            )
            else pearson_r
        )

        permutation_statistics = []

        if np.isfinite(
            observed_beta
        ):

            for permutation_number in range(
                N_FEATURE_PERMUTATIONS
            ):

                permuted = (
                    temp
                    .copy()
                )

                permuted[
                    "Feature_z"
                ] = (
                    permuted
                    .groupby(
                        "BaseVideo"
                    )[
                        "Feature_z"
                    ]
                    .transform(
                        lambda x:
                        RNG.permutation(
                            x.to_numpy()
                        )
                    )
                )

                # Faster permutation statistic:
                # pooled Pearson after preserving video blocks.
                #
                # Inferential mixed model remains the primary
                # coefficient test above.
                perm_r, _, _ = safe_pearson(
                    permuted[
                        "Feature_z"
                    ],
                    permuted[
                        "LeftMotorISC_zstd"
                    ],
                    minimum_n=MINIMUM_FEATURE_N
                )

                permutation_statistics.append(
                    perm_r
                )

            permutation_statistics = np.asarray(
                permutation_statistics,
                dtype=float
            )

            permutation_p = (
                1
                +
                np.sum(
                    np.abs(
                        permutation_statistics
                    )
                    >=
                    abs(
                        pearson_r
                    )
                )
            ) / (
                N_FEATURE_PERMUTATIONS
                + 1
            )

        else:

            permutation_p = np.nan


        feature_result_rows.append({

            "Modality":
                modality,

            "KinematicFeature":
                feature,

            "N_SubjectVideo":
                len(
                    temp
                ),

            "N_Subjects":
                temp[
                    "Subject"
                ].nunique(),

            "N_Videos":
                temp[
                    "BaseVideo"
                ].nunique(),

            "Pearson_r":
                pearson_r,

            "Pearson_p":
                pearson_p,

            "Spearman_rho":
                spearman_rho,

            "Spearman_p":
                spearman_p,

            "Mixed_Beta":
                mixed_beta,

            "Mixed_SE":
                mixed_se,

            "Mixed_CI_Low":
                mixed_ci_low,

            "Mixed_CI_High":
                mixed_ci_high,

            "Mixed_p":
                mixed_p,

            "Mixed_Converged":
                mixed_converged,

            "Permutation_p":
                permutation_p
        })


feature_motor_roi_results_df = pd.DataFrame(
    feature_result_rows
)


# ============================================================
# 10. FDR CORRECTION FOR FEATURE TESTS
# ============================================================

for modality in (
    feature_motor_roi_results_df[
        "Modality"
    ]
    .dropna()
    .unique()
):

    modality_mask = (
        feature_motor_roi_results_df[
            "Modality"
        ]
        == modality
    )

    for p_column in [
        "Pearson_p",
        "Mixed_p",
        "Permutation_p"
    ]:

        valid_mask = (
            modality_mask
            &
            feature_motor_roi_results_df[
                p_column
            ].notna()
        )

        if valid_mask.sum() == 0:
            continue

        corrected = multipletests(

            feature_motor_roi_results_df.loc[
                valid_mask,
                p_column
            ],

            method="fdr_bh"

        )[1]

        feature_motor_roi_results_df.loc[
            valid_mask,
            f"{p_column}_FDR"
        ] = corrected


feature_motor_roi_results_df.to_csv(
    TABLE_DIR
    / "individual_kinematic_features_left_motor_roi_results.csv",
    index=False
)


print(
    "\n"
    + "=" * 90
)

print(
    "PART A: INDIVIDUAL KINEMATIC FEATURES × LEFT MOTOR ROI ISC"
)

print(
    "=" * 90
)

display(
    feature_motor_roi_results_df
    .sort_values(
        [
            "Modality",
            "Mixed_p"
        ]
    )
    .round(
        4
    )
)


# ============================================================
# 11. FEATURE SUMMARY FIGURE
# ============================================================

for modality in [
    "Valence",
    "Arousal"
]:

    plot_df = (
        feature_motor_roi_results_df[
            feature_motor_roi_results_df[
                "Modality"
            ] == modality
        ]
        .dropna(
            subset=[
                "Mixed_Beta"
            ]
        )
        .sort_values(
            "Mixed_Beta"
        )
        .copy()
    )

    if plot_df.empty:
        continue

    y_positions = np.arange(
        len(
            plot_df
        )
    )

    lower_error = (
        plot_df[
            "Mixed_Beta"
        ]
        -
        plot_df[
            "Mixed_CI_Low"
        ]
    )

    upper_error = (
        plot_df[
            "Mixed_CI_High"
        ]
        -
        plot_df[
            "Mixed_Beta"
        ]
    )

    fig, ax = plt.subplots(
        figsize=(
            9,
            max(
                7,
                0.35
                * len(
                    plot_df
                )
            )
        )
    )

    ax.errorbar(
        plot_df[
            "Mixed_Beta"
        ],
        y_positions,
        xerr=np.vstack(
            [
                lower_error,
                upper_error
            ]
        ),
        fmt="o",
        capsize=3
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_yticks(
        y_positions
    )

    ax.set_yticklabels(
        plot_df[
            "KinematicFeature"
        ]
    )

    ax.set_xlabel(
        "Standardized association with left motor ROI ISC"
    )

    ax.set_ylabel(
        "Hand-kinematic feature"
    )

    ax.set_title(
        f"{modality}: individual hand kinematics × "
        "left motor-control ROI ISC"
    )

    plt.tight_layout()

    fig.savefig(
        FEATURE_FIGURE_DIR
        / safe_filename(
            f"{modality}_individual_kinematic_feature_effects.png"
        ),
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# ============================================================
# PART B
# CONTINUOUS MOVEMENT × LEFT MOTOR HbO
# ============================================================
# ============================================================


# ============================================================
# 12. BUILD MOVEMENT-ONSET VECTOR
# ============================================================

def calculate_movement_onsets(
    time_sec,
    abs_velocity,
    threshold,
    quiet_duration_sec
):
    """
    Binary movement-onset vector.

    An onset is defined as:
        current abs velocity >= threshold
    after the trajectory has remained below threshold for at
    least quiet_duration_sec.
    """

    time_sec = np.asarray(
        time_sec,
        dtype=float
    )

    abs_velocity = np.asarray(
        abs_velocity,
        dtype=float
    )

    onset = np.zeros(
        len(
            abs_velocity
        ),
        dtype=float
    )

    if len(
        abs_velocity
    ) < 3:
        return onset

    median_dt = np.nanmedian(
        np.diff(
            time_sec
        )
    )

    if (
        not np.isfinite(
            median_dt
        )
        or
        median_dt <= 0
    ):
        return onset

    quiet_samples = max(
        1,
        int(
            round(
                quiet_duration_sec
                /
                median_dt
            )
        )
    )

    moving = (
        abs_velocity
        >= threshold
    )

    for index in range(
        quiet_samples,
        len(
            moving
        )
    ):

        if not moving[
            index
        ]:
            continue

        previous_window = moving[
            index
            - quiet_samples:
            index
        ]

        if not np.any(
            previous_window
        ):

            onset[
                index
            ] = 1.0

    return onset


# ============================================================
# 13. ALIGN BEHAVIOR TO fNIRS TIME GRID
# ============================================================

def align_behavior_and_fnirs(
    behavioral_df,
    fnirs_hbo
):
    """
    Interpolate continuous behavioral derivatives onto the
    1.5-Hz fNIRS time grid.
    """

    behavioral_df = (
        behavioral_df
        .copy()
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    required_columns = [
        "Time",
        "AbsVelocity",
        "Acceleration"
    ]

    if not all(
        column in behavioral_df.columns
        for column in required_columns
    ):

        return None

    behavioral_df = (
        behavioral_df[
            required_columns
        ]
        .dropna(
            subset=[
                "Time"
            ]
        )
        .sort_values(
            "Time"
        )
        .drop_duplicates(
            subset=[
                "Time"
            ]
        )
    )

    if len(
        behavioral_df
    ) < MIN_CONTINUOUS_SAMPLES:
        return None

    behavior_time = (
        behavioral_df[
            "Time"
        ].to_numpy(
            dtype=float
        )
    )

    abs_velocity = (
        behavioral_df[
            "AbsVelocity"
        ].to_numpy(
            dtype=float
        )
    )

    acceleration = (
        behavioral_df[
            "Acceleration"
        ].to_numpy(
            dtype=float
        )
    )

    abs_acceleration = np.abs(
        acceleration
    )

    movement_onset = calculate_movement_onsets(
        time_sec=behavior_time,
        abs_velocity=abs_velocity,
        threshold=MOVEMENT_THRESHOLD,
        quiet_duration_sec=MIN_ONSET_QUIET_SEC
    )

    fnirs_hbo = np.asarray(
        fnirs_hbo,
        dtype=float
    ).ravel()

    fnirs_time = (
        np.arange(
            len(
                fnirs_hbo
            ),
            dtype=float
        )
        /
        FNIRS_SAMPLING_RATE_HZ
    )

    # Restrict to time represented in both signals.
    maximum_common_time = min(
        np.nanmax(
            behavior_time
        ),
        np.nanmax(
            fnirs_time
        )
    )

    fnirs_mask = (
        fnirs_time
        <= maximum_common_time
    )

    fnirs_time = fnirs_time[
        fnirs_mask
    ]

    fnirs_hbo = fnirs_hbo[
        fnirs_mask
    ]

    if len(
        fnirs_hbo
    ) < MIN_CONTINUOUS_SAMPLES:

        return None

    velocity_interp = np.interp(
        fnirs_time,
        behavior_time,
        abs_velocity
    )

    acceleration_interp = np.interp(
        fnirs_time,
        behavior_time,
        abs_acceleration
    )

    # Nearest-neighbor-style interpolation for binary events.
    onset_interp = np.interp(
        fnirs_time,
        behavior_time,
        movement_onset
    )

    onset_interp = (
        onset_interp
        > 0.5
    ).astype(
        float
    )

    aligned = pd.DataFrame({

        "Time":
            fnirs_time,

        "LeftMotorHbO":
            fnirs_hbo,

        "AbsVelocity":
            velocity_interp,

        "AbsAcceleration":
            acceleration_interp,

        "MovementOnset":
            onset_interp
    })

    return aligned


# ============================================================
# 14. LAGGED CORRELATION FUNCTION
# ============================================================

def calculate_lagged_correlation(
    dataframe,
    predictor,
    lag_sec
):
    """
    Positive lag means movement precedes HbO.

    Predictor(t) is correlated with HbO(t + lag).
    """

    lag_samples = int(
        round(
            lag_sec
            * FNIRS_SAMPLING_RATE_HZ
        )
    )

    x = dataframe[
        predictor
    ].to_numpy(
        dtype=float
    )

    y = dataframe[
        "LeftMotorHbO"
    ].to_numpy(
        dtype=float
    )

    if lag_samples > 0:

        x = x[
            :-lag_samples
        ]

        y = y[
            lag_samples:
        ]

    minimum_length = min(
        len(
            x
        ),
        len(
            y
        )
    )

    x = x[
        :minimum_length
    ]

    y = y[
        :minimum_length
    ]

    valid = (
        np.isfinite(
            x
        )
        &
        np.isfinite(
            y
        )
    )

    x = x[
        valid
    ]

    y = y[
        valid
    ]

    if len(
        x
    ) < MIN_CONTINUOUS_SAMPLES:

        return (
            np.nan,
            len(
                x
            )
        )

    if (
        np.std(
            x
        ) <= 1e-12
        or
        np.std(
            y
        ) <= 1e-12
    ):

        return (
            np.nan,
            len(
                x
            )
        )

    r_value = pearsonr(
        x,
        y
    )[0]

    return (
        float(
            r_value
        ),
        len(
            x
        )
    )


# ============================================================
# 15. BUILD CONTINUOUS DATASET
# ============================================================

continuous_lag_rows = []
continuous_aligned_cache = {}


for (
    cache_key,
    behavioral_trajectory
) in trajectory_cache.items():

    subject_id = normalize_subject_id(
        cache_key[
            0
        ]
    )

    behavioral_video_name = str(
        cache_key[
            1
        ]
    )

    base_video = normalize_video_name(
        behavioral_video_name
    )

    modality = extract_modality_from_video(
        behavioral_video_name
    )

    if modality not in [
        "Valence",
        "Arousal"
    ]:
        continue

    mapping_match = (
        video_mapping_df[
            video_mapping_df[
                "BaseVideo"
            ] == base_video
        ]
    )

    if mapping_match.empty:
        continue

    trigger = normalize_trigger(
        mapping_match.iloc[
            0
        ][
            "Trigger"
        ]
    )

    # --------------------------------------------------------
    # Find corresponding participant in motor_roi_ts
    # --------------------------------------------------------

    matching_motor_subject = None

    for available_subject in motor_roi_ts.keys():

        if (
            normalize_subject_id(
                available_subject
            )
            == subject_id
        ):

            matching_motor_subject = available_subject
            break

    if matching_motor_subject is None:
        continue

    # --------------------------------------------------------
    # Find trigger
    # --------------------------------------------------------

    participant_trigger_dict = motor_roi_ts[
        matching_motor_subject
    ]

    matching_trigger = None

    for available_trigger in participant_trigger_dict.keys():

        if (
            normalize_trigger(
                available_trigger
            )
            == trigger
        ):

            matching_trigger = available_trigger
            break

    if matching_trigger is None:
        continue

    roi_dictionary = participant_trigger_dict[
        matching_trigger
    ]

    if LEFT_MOTOR_ROI_NAME not in roi_dictionary:
        continue

    fnirs_hbo = roi_dictionary[
        LEFT_MOTOR_ROI_NAME
    ]

    aligned_df = align_behavior_and_fnirs(
        behavioral_df=behavioral_trajectory,
        fnirs_hbo=fnirs_hbo
    )

    if aligned_df is None:
        continue

    continuous_aligned_cache[
        (
            subject_id,
            base_video,
            modality
        )
    ] = aligned_df

    for predictor in CONTINUOUS_PREDICTORS:

        for lag_sec in LAG_SECONDS:

            correlation_value, n_samples = (
                calculate_lagged_correlation(

                    dataframe=aligned_df,

                    predictor=predictor,

                    lag_sec=lag_sec
                )
            )

            continuous_lag_rows.append({

                "Subject":
                    subject_id,

                "BaseVideo":
                    base_video,

                "Trigger":
                    trigger,

                "Modality":
                    modality,

                "Predictor":
                    predictor,

                "LagSec":
                    lag_sec,

                "N_Timepoints":
                    n_samples,

                "Pearson_r":
                    correlation_value,

                "FisherZ_r":
                    fisher_z_from_r(
                        correlation_value
                    )
            })


continuous_lagged_correlations_df = pd.DataFrame(
    continuous_lag_rows
)


if continuous_lagged_correlations_df.empty:

    raise RuntimeError(
        "No continuous motor–fNIRS observations were created.\n\n"
        "Check Subject IDs, video_mapping.csv, trajectory_cache, "
        "and motor_roi_ts."
    )


continuous_lagged_correlations_df.to_csv(
    CONTINUOUS_OUTPUT_DIR
    / "subject_video_continuous_motor_hbo_lag_correlations.csv",
    index=False
)


print(
    "\n"
    + "=" * 90
)

print(
    "CONTINUOUS DATA MATCHING SUMMARY"
)

print(
    "=" * 90
)

print(
    "Aligned Subject × Video × Modality observations: "
    f"{len(continuous_aligned_cache)}"
)

print(
    "Unique subjects: "
    f"{continuous_lagged_correlations_df['Subject'].nunique()}"
)

print(
    "Unique videos: "
    f"{continuous_lagged_correlations_df['BaseVideo'].nunique()}"
)

print(
    "Lag tests created: "
    f"{len(continuous_lagged_correlations_df)}"
)


# ============================================================
# 16. GROUP-LEVEL MIXED MODELS AT EACH LAG
# ============================================================

continuous_group_rows = []
continuous_model_objects = {}


for modality in [
    "Valence",
    "Arousal"
]:

    for predictor in CONTINUOUS_PREDICTORS:

        for lag_sec in LAG_SECONDS:

            temp = (
                continuous_lagged_correlations_df[
                    (
                        continuous_lagged_correlations_df[
                            "Modality"
                        ]
                        == modality
                    )
                    &
                    (
                        continuous_lagged_correlations_df[
                            "Predictor"
                        ]
                        == predictor
                    )
                    &
                    np.isclose(
                        continuous_lagged_correlations_df[
                            "LagSec"
                        ],
                        lag_sec
                    )
                ]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .dropna(
                    subset=[
                        "FisherZ_r"
                    ]
                )
                .copy()
            )

            if len(
                temp
            ) < MINIMUM_FEATURE_N:

                continue

            estimate = np.nan
            se = np.nan
            p_value = np.nan
            ci_low = np.nan
            ci_high = np.nan
            converged = False

            try:

                model = smf.mixedlm(

                    "FisherZ_r ~ 1",

                    data=temp,

                    groups=temp[
                        "Subject"
                    ],

                    re_formula="1",

                    vc_formula={
                        "Video":
                        "0 + C(BaseVideo)"
                    }
                )

                result = model.fit(
                    reml=False,
                    method="lbfgs",
                    maxiter=3000,
                    disp=False
                )

                estimate = float(
                    result.params[
                        "Intercept"
                    ]
                )

                se = float(
                    result.bse[
                        "Intercept"
                    ]
                )

                p_value = float(
                    result.pvalues[
                        "Intercept"
                    ]
                )

                confidence_interval = (
                    result.conf_int()
                    .loc[
                        "Intercept"
                    ]
                )

                ci_low = float(
                    confidence_interval.iloc[
                        0
                    ]
                )

                ci_high = float(
                    confidence_interval.iloc[
                        1
                    ]
                )

                converged = bool(
                    result.converged
                )

                continuous_model_objects[
                    (
                        modality,
                        predictor,
                        lag_sec
                    )
                ] = result

            except Exception:

                # Fallback intercept-only OLS if variance
                # components become singular at one lag.
                try:

                    result = smf.ols(
                        "FisherZ_r ~ 1",
                        data=temp
                    ).fit()

                    estimate = float(
                        result.params[
                            "Intercept"
                        ]
                    )

                    se = float(
                        result.bse[
                            "Intercept"
                        ]
                    )

                    p_value = float(
                        result.pvalues[
                            "Intercept"
                        ]
                    )

                    confidence_interval = (
                        result.conf_int()
                        .loc[
                            "Intercept"
                        ]
                    )

                    ci_low = float(
                        confidence_interval.iloc[
                            0
                        ]
                    )

                    ci_high = float(
                        confidence_interval.iloc[
                            1
                        ]
                    )

                except Exception:
                    pass


            continuous_group_rows.append({

                "Modality":
                    modality,

                "Predictor":
                    predictor,

                "LagSec":
                    lag_sec,

                "N_SubjectVideo":
                    len(
                        temp
                    ),

                "N_Subjects":
                    temp[
                        "Subject"
                    ].nunique(),

                "N_Videos":
                    temp[
                        "BaseVideo"
                    ].nunique(),

                "Mean_FisherZ":
                    temp[
                        "FisherZ_r"
                    ].mean(),

                "Mean_r":
                    temp[
                        "Pearson_r"
                    ].mean(),

                "Median_r":
                    temp[
                        "Pearson_r"
                    ].median(),

                "Mixed_Intercept_FisherZ":
                    estimate,

                "Mixed_SE":
                    se,

                "CI_Low_FisherZ":
                    ci_low,

                "CI_High_FisherZ":
                    ci_high,

                "Mixed_p":
                    p_value,

                "Mixed_Converged":
                    converged
            })


continuous_motor_hbo_group_df = pd.DataFrame(
    continuous_group_rows
)


# ============================================================
# 17. FDR CORRECT LAG TESTS
# ============================================================

continuous_motor_hbo_group_df[
    "Mixed_p_FDR"
] = np.nan


for (
    modality,
    predictor
), group_indices in (
    continuous_motor_hbo_group_df
    .groupby(
        [
            "Modality",
            "Predictor"
        ]
    )
    .groups
    .items()
):

    indices = list(
        group_indices
    )

    valid_indices = [
        index
        for index in indices
        if pd.notna(
            continuous_motor_hbo_group_df.loc[
                index,
                "Mixed_p"
            ]
        )
    ]

    if len(
        valid_indices
    ) == 0:
        continue

    corrected_p = multipletests(

        continuous_motor_hbo_group_df.loc[
            valid_indices,
            "Mixed_p"
        ],

        method="fdr_bh"

    )[1]

    continuous_motor_hbo_group_df.loc[
        valid_indices,
        "Mixed_p_FDR"
    ] = corrected_p


continuous_motor_hbo_group_df.to_csv(
    TABLE_DIR
    / "continuous_motor_hbo_lag_mixed_models.csv",
    index=False
)


# ============================================================
# 18. IDENTIFY PEAK LAG
# ============================================================

peak_rows = []


for (
    modality,
    predictor
), temp in (
    continuous_motor_hbo_group_df
    .groupby(
        [
            "Modality",
            "Predictor"
        ]
    )
):

    temp = (
        temp
        .dropna(
            subset=[
                "Mixed_Intercept_FisherZ"
            ]
        )
        .copy()
    )

    if temp.empty:
        continue

    # Motor validation expects positive movement-related HbO
    # coupling, so select the largest positive group effect.
    peak_index = (
        temp[
            "Mixed_Intercept_FisherZ"
        ]
        .idxmax()
    )

    peak_row = (
        temp
        .loc[
            peak_index
        ]
        .copy()
    )

    peak_rows.append(
        peak_row
    )


continuous_peak_lag_df = pd.DataFrame(
    peak_rows
)


continuous_peak_lag_df.to_csv(
    TABLE_DIR
    / "continuous_motor_hbo_peak_lags.csv",
    index=False
)


print(
    "\n"
    + "=" * 90
)

print(
    "PART B: CONTINUOUS MOVEMENT × LEFT MOTOR HbO"
)

print(
    "=" * 90
)

display(
    continuous_peak_lag_df
    .round(
        4
    )
)


# ============================================================
# 19. CIRCULAR-SHIFT PERMUTATION TEST AT PEAK LAGS
# ============================================================
#
# IMPORTANT:
#
# Each behavioral trajectory is circularly shifted relative
# to the neural time series.
#
# This preserves:
#   - the movement trajectory's autocorrelation
#   - its amplitude distribution
#   - its temporal smoothness
#
# while destroying its true temporal alignment with HbO.
#
# The statistic is the mean Fisher-z within-trial correlation.
# ============================================================

continuous_permutation_rows = []


for _, peak_row in continuous_peak_lag_df.iterrows():

    modality = peak_row[
        "Modality"
    ]

    predictor = peak_row[
        "Predictor"
    ]

    peak_lag = float(
        peak_row[
            "LagSec"
        ]
    )

    observed_rows = (
        continuous_lagged_correlations_df[
            (
                continuous_lagged_correlations_df[
                    "Modality"
                ]
                == modality
            )
            &
            (
                continuous_lagged_correlations_df[
                    "Predictor"
                ]
                == predictor
            )
            &
            np.isclose(
                continuous_lagged_correlations_df[
                    "LagSec"
                ],
                peak_lag
            )
        ]
        .dropna(
            subset=[
                "FisherZ_r"
            ]
        )
        .copy()
    )

    observed_statistic = (
        observed_rows[
            "FisherZ_r"
        ].mean()
    )

    null_statistics = []


    for permutation_number in range(
        N_CONTINUOUS_PERMUTATIONS
    ):

        permuted_fisher_z_values = []


        for (
            subject_id,
            base_video,
            observation_modality
        ), aligned_df in continuous_aligned_cache.items():

            if observation_modality != modality:
                continue

            x = (
                aligned_df[
                    predictor
                ]
                .to_numpy(
                    dtype=float
                )
            )

            y = (
                aligned_df[
                    "LeftMotorHbO"
                ]
                .to_numpy(
                    dtype=float
                )
            )

            n_samples = min(
                len(
                    x
                ),
                len(
                    y
                )
            )

            if n_samples < MIN_CONTINUOUS_SAMPLES:
                continue

            x = x[
                :n_samples
            ]

            y = y[
                :n_samples
            ]

            minimum_shift_samples = int(
                round(
                    MIN_CIRCULAR_SHIFT_SEC
                    *
                    FNIRS_SAMPLING_RATE_HZ
                )
            )

            if (
                n_samples
                <=
                2 * minimum_shift_samples
            ):
                continue

            shift = RNG.integers(
                minimum_shift_samples,
                n_samples
                - minimum_shift_samples
            )

            x_shifted = np.roll(
                x,
                shift
            )

            permuted_df = pd.DataFrame({

                predictor:
                    x_shifted,

                "LeftMotorHbO":
                    y
            })

            permuted_r, _ = (
                calculate_lagged_correlation(

                    dataframe=permuted_df,

                    predictor=predictor,

                    lag_sec=peak_lag
                )
            )

            if np.isfinite(
                permuted_r
            ):

                permuted_fisher_z_values.append(
                    fisher_z_from_r(
                        permuted_r
                    )
                )


        if len(
            permuted_fisher_z_values
        ) > 0:

            null_statistics.append(
                np.mean(
                    permuted_fisher_z_values
                )
            )


    null_statistics = np.asarray(
        null_statistics,
        dtype=float
    )

    if len(
        null_statistics
    ) > 0:

        permutation_p = (
            1
            +
            np.sum(
                np.abs(
                    null_statistics
                )
                >=
                abs(
                    observed_statistic
                )
            )
        ) / (
            len(
                null_statistics
            )
            + 1
        )

    else:

        permutation_p = np.nan


    continuous_permutation_rows.append({

        "Modality":
            modality,

        "Predictor":
            predictor,

        "PeakLagSec":
            peak_lag,

        "ObservedMeanFisherZ":
            observed_statistic,

        "ObservedApproxMeanR":
            np.tanh(
                observed_statistic
            ),

        "N_Permutations":
            len(
                null_statistics
            ),

        "CircularShiftPermutation_p":
            permutation_p
    })


continuous_motor_hbo_permutation_df = pd.DataFrame(
    continuous_permutation_rows
)


if not continuous_motor_hbo_permutation_df.empty:

    valid = (
        continuous_motor_hbo_permutation_df[
            "CircularShiftPermutation_p"
        ]
        .notna()
    )

    if valid.sum() > 0:

        continuous_motor_hbo_permutation_df.loc[
            valid,
            "CircularShiftPermutation_p_FDR"
        ] = multipletests(

            continuous_motor_hbo_permutation_df.loc[
                valid,
                "CircularShiftPermutation_p"
            ],

            method="fdr_bh"

        )[1]


continuous_motor_hbo_permutation_df.to_csv(
    TABLE_DIR
    / "continuous_motor_hbo_circular_shift_permutation.csv",
    index=False
)


print(
    "\n=== Circular-shift permutation tests ==="
)

display(
    continuous_motor_hbo_permutation_df
    .round(
        4
    )
)


# ============================================================
# 20. LAG-RESPONSE FIGURES
# ============================================================

for modality in [
    "Valence",
    "Arousal"
]:

    for predictor in CONTINUOUS_PREDICTORS:

        plot_df = (
            continuous_motor_hbo_group_df[
                (
                    continuous_motor_hbo_group_df[
                        "Modality"
                    ]
                    == modality
                )
                &
                (
                    continuous_motor_hbo_group_df[
                        "Predictor"
                    ]
                    == predictor
                )
            ]
            .sort_values(
                "LagSec"
            )
            .copy()
        )

        if plot_df.empty:
            continue

        fig, ax = plt.subplots(
            figsize=(
                8,
                5
            )
        )

        ax.plot(
            plot_df[
                "LagSec"
            ],
            plot_df[
                "Mixed_Intercept_FisherZ"
            ],
            marker="o"
        )

        ax.fill_between(
            plot_df[
                "LagSec"
            ],
            plot_df[
                "CI_Low_FisherZ"
            ],
            plot_df[
                "CI_High_FisherZ"
            ],
            alpha=0.20
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1
        )

        ax.set_xlabel(
            "Movement → neural lag (seconds)"
        )

        ax.set_ylabel(
            "Mean movement–left motor HbO coupling\n"
            "(Fisher-z correlation)"
        )

        ax.set_title(
            f"{modality}: Left motor HbO × {predictor}"
        )

        plt.tight_layout()

        fig.savefig(
            CONTINUOUS_FIGURE_DIR
            / safe_filename(
                f"{modality}_{predictor}_lag_response.png"
            ),
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

        plt.show()


# ============================================================
# 21. SUMMARY FIGURE OF PEAK CONTINUOUS EFFECTS
# ============================================================

if not continuous_peak_lag_df.empty:

    summary_df = (
        continuous_peak_lag_df
        .copy()
        .reset_index(
            drop=True
        )
    )

    summary_df[
        "Label"
    ] = (
        summary_df[
            "Modality"
        ]
        +
        " | "
        +
        summary_df[
            "Predictor"
        ]
        +
        "\nlag="
        +
        summary_df[
            "LagSec"
        ]
        .round(
            1
        )
        .astype(
            str
        )
        +
        " s"
    )

    y = summary_df[
        "Mixed_Intercept_FisherZ"
    ].to_numpy()

    lower = (
        y
        -
        summary_df[
            "CI_Low_FisherZ"
        ].to_numpy()
    )

    upper = (
        summary_df[
            "CI_High_FisherZ"
        ].to_numpy()
        -
        y
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            5
        )
    )

    x_positions = np.arange(
        len(
            summary_df
        )
    )

    ax.errorbar(
        x_positions,
        y,
        yerr=np.vstack(
            [
                lower,
                upper
            ]
        ),
        fmt="o",
        capsize=4
    )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        summary_df[
            "Label"
        ],
        rotation=35,
        ha="right"
    )

    ax.set_ylabel(
        "Peak movement–left motor HbO coupling\n"
        "(Fisher-z correlation)"
    )

    ax.set_title(
        "Continuous functional validation of left motor-control ROI"
    )

    plt.tight_layout()

    fig.savefig(
        CONTINUOUS_FIGURE_DIR
        / "continuous_motor_roi_peak_effect_summary.png",
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 22. MASTER SUMMARY
# ============================================================

master_summary_rows = []


# Part A significant individual features
if not feature_motor_roi_results_df.empty:

    for _, row in feature_motor_roi_results_df.iterrows():

        master_summary_rows.append({

            "Analysis":
                "Individual feature × motor ROI ISC",

            "Modality":
                row[
                    "Modality"
                ],

            "Predictor":
                row[
                    "KinematicFeature"
                ],

            "Effect":
                row[
                    "Mixed_Beta"
                ],

            "P":
                row[
                    "Mixed_p"
                ],

            "P_FDR":
                row.get(
                    "Mixed_p_FDR",
                    np.nan
                ),

            "LagSec":
                np.nan
        })


# Part B continuous effects
if not continuous_peak_lag_df.empty:

    for _, row in continuous_peak_lag_df.iterrows():

        permutation_match = (
            continuous_motor_hbo_permutation_df[
                (
                    continuous_motor_hbo_permutation_df[
                        "Modality"
                    ]
                    == row[
                        "Modality"
                    ]
                )
                &
                (
                    continuous_motor_hbo_permutation_df[
                        "Predictor"
                    ]
                    == row[
                        "Predictor"
                    ]
                )
            ]
        )

        if not permutation_match.empty:

            permutation_p = (
                permutation_match.iloc[
                    0
                ][
                    "CircularShiftPermutation_p"
                ]
            )

            permutation_p_fdr = (
                permutation_match.iloc[
                    0
                ].get(
                    "CircularShiftPermutation_p_FDR",
                    np.nan
                )
            )

        else:

            permutation_p = np.nan
            permutation_p_fdr = np.nan


        master_summary_rows.append({

            "Analysis":
                "Continuous movement × left motor HbO",

            "Modality":
                row[
                    "Modality"
                ],

            "Predictor":
                row[
                    "Predictor"
                ],

            "Effect":
                row[
                    "Mixed_Intercept_FisherZ"
                ],

            "P":
                row[
                    "Mixed_p"
                ],

            "P_FDR":
                row[
                    "Mixed_p_FDR"
                ],

            "LagSec":
                row[
                    "LagSec"
                ],

            "CircularShiftPermutation_p":
                permutation_p,

            "CircularShiftPermutation_p_FDR":
                permutation_p_fdr
        })


direct_motor_roi_validation_summary_df = pd.DataFrame(
    master_summary_rows
)


direct_motor_roi_validation_summary_df.to_csv(
    TABLE_DIR
    / "direct_motor_roi_validation_master_summary.csv",
    index=False
)


# ============================================================
# 23. FINAL PRINTED SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DIRECT LEFT MOTOR ROI FUNCTIONAL VALIDATION COMPLETE"
)

print(
    "=" * 100
)

print(
    "\nPART A:"
)

print(
    f"Tested {len(kinematic_features)} individual "
    "hand-kinematic variables against left motor ROI ISC."
)

print(
    "\nPART B:"
)

print(
    "Tested continuous left motor HbO against:"
)

for predictor in CONTINUOUS_PREDICTORS:

    print(
        f"  {predictor}"
    )

print(
    "\nNeural lag range:"
)

print(
    f"  {LAG_SECONDS.min():.1f}–"
    f"{LAG_SECONDS.max():.1f} s"
)

print(
    "\nPeak continuous effects:"
)

display(
    continuous_peak_lag_df[
        [
            "Modality",
            "Predictor",
            "LagSec",
            "Mean_r",
            "Mixed_Intercept_FisherZ",
            "CI_Low_FisherZ",
            "CI_High_FisherZ",
            "Mixed_p",
            "Mixed_p_FDR"
        ]
    ]
    .round(
        4
    )
)

print(
    "\nCircular-shift permutation results:"
)

display(
    continuous_motor_hbo_permutation_df
    .round(
        4
    )
)

print(
    "\nAll outputs saved to:"
)

print(
    DIRECT_MOTOR_OUTPUT_DIR
)

# Counterfactual Analysis

In [ ]:
# ============================================================
# BEHAVIORAL COUNTERFACTUAL TRAJECTORY / PC STRUCTURE
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import pearsonr
from factor_analyzer import FactorAnalyzer

if "data_by_video" not in globals():
    raise ValueError("data_by_video not found. Run your behavioral analysis cell first.")

if "analysis_out" not in globals():
    analysis_out = os.path.join(os.getcwd(), "counterfactual_pc_analysis")
    os.makedirs(analysis_out, exist_ok=True)

pc_out = os.path.join(analysis_out, "counterfactual_pc_by_video")
fig_out = os.path.join(pc_out, "figures")
os.makedirs(pc_out, exist_ok=True)
os.makedirs(fig_out, exist_ok=True)

N_FACTORS = 3
N_COMPONENTS = 3

BEST_R_MIN = 0.20
BEST_MARGIN_MIN = 0.05

PC2_VAR_THRESHOLD = 0.15
PC2_ALIGNED_PROP_THRESHOLD = 0.20

PLOT_ALL_VIDEOS = True
PLOT_TOP_N_ALT_VIDEOS = 10


def CCT_components(df, n_factors=3, n_components=3, return_ev=False):
    fa = FactorAnalyzer(rotation=None)
    fa.fit(df, n_factors)
    ev, _ = fa.get_eigenvalues()
    comps = fa.transform(df)[:, :n_components]
    return (comps, ev) if return_ev else comps


def clean_for_pca(subj_dict):
    min_len = min(len(v) for v in subj_dict.values())
    trimmed = {subj: np.array(arr[:min_len]) for subj, arr in subj_dict.items()}
    df = pd.DataFrame(trimmed)
    df = df.dropna(axis=0, how="any")
    return df


def zscore_cols(df):
    out = df.copy().astype(float)
    for col in out.columns:
        x = out[col].values
        sd = np.nanstd(x, ddof=1)
        if sd == 0 or not np.isfinite(sd):
            out[col] = np.nan
        else:
            out[col] = (x - np.nanmean(x)) / sd

    out = out.dropna(axis=1, how="any")
    out = out.dropna(axis=0, how="any")
    return out


def safe_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    min_len = min(len(x), len(y))
    x = x[:min_len]
    y = y[:min_len]

    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 5:
        return np.nan, np.nan

    x = x[mask]
    y = y[mask]

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return pearsonr(x, y)


def extract_trigger_id(video_name):
    nums = re.findall(r"\d+", str(video_name))
    if len(nums) == 0:
        return str(video_name)
    return str(int(nums[-1]))


def base_video_name(video_name):
    trig = extract_trigger_id(video_name)
    return trig


def align_pc_signs_to_group_mean(comps, df_z):
    comps = comps.copy()
    group_mean = df_z.mean(axis=1).values

    for k in range(comps.shape[1]):
        r, _ = safe_corr(comps[:, k], group_mean)
        if np.isfinite(r) and r < 0:
            comps[:, k] *= -1

    return comps


def run_video_pc_analysis(df, modality, video_name):
    if df.shape[1] < 5:
        return None, [], None, None

    df_z = zscore_cols(df)

    if df_z.shape[1] < 5 or df_z.shape[0] < 10:
        return None, [], None, None

    try:
        comps, ev = CCT_components(
            df_z,
            n_factors=N_FACTORS,
            n_components=N_COMPONENTS,
            return_ev=True
        )
    except Exception as e:
        print(f"PCA error for {modality} {video_name}: {e}")
        return None, [], None, None

    comps = align_pc_signs_to_group_mean(comps, df_z)
    ev = np.asarray(ev, dtype=float)

    ev_sum = np.nansum(ev)
    ev_prop = ev / ev_sum if ev_sum > 0 else np.full_like(ev, np.nan)

    pc1_var = ev_prop[0] if len(ev_prop) > 0 else np.nan
    pc2_var = ev_prop[1] if len(ev_prop) > 1 else np.nan
    pc3_var = ev_prop[2] if len(ev_prop) > 2 else np.nan

    video_base = base_video_name(video_name)
    trigger = extract_trigger_id(video_name)

    comp_df = pd.DataFrame({
        "Timepoint": np.arange(comps.shape[0]),
        "PC1": comps[:, 0],
        "PC2": comps[:, 1],
        "PC3": comps[:, 2],
        "Modality": modality,
        "Video": video_name,
        "Video_Base": video_base,
        "Trigger": trigger
    })

    align_rows = []

    for subj in df_z.columns:
        subj_ts = df_z[subj].values

        r_pc1, p_pc1 = safe_corr(subj_ts, comps[:, 0])
        r_pc2, p_pc2 = safe_corr(subj_ts, comps[:, 1])
        r_pc3, p_pc3 = safe_corr(subj_ts, comps[:, 2])

        signed_rs = np.array([r_pc1, r_pc2, r_pc3], dtype=float)
        abs_rs = np.abs(signed_rs)

        if np.all(~np.isfinite(abs_rs)):
            best_pc = np.nan
            best_r = np.nan
            best_abs_r = np.nan
            second_best_abs_r = np.nan
            best_margin = np.nan
            confident_assignment = False
        else:
            order = np.argsort(abs_rs)[::-1]
            best_idx = order[0]
            second_idx = order[1]

            best_pc = f"PC{best_idx + 1}"
            best_r = signed_rs[best_idx]
            best_abs_r = abs_rs[best_idx]
            second_best_abs_r = abs_rs[second_idx]
            best_margin = best_abs_r - second_best_abs_r

            confident_assignment = (
                np.isfinite(best_abs_r)
                and best_abs_r >= BEST_R_MIN
                and best_margin >= BEST_MARGIN_MIN
            )

        align_rows.append({
            "Subject": subj,
            "Video": video_name,
            "Video_Base": video_base,
            "Trigger": trigger,
            "Modality": modality,
            "r_PC1": r_pc1,
            "r_PC2": r_pc2,
            "r_PC3": r_pc3,
            "abs_r_PC1": abs(r_pc1) if np.isfinite(r_pc1) else np.nan,
            "abs_r_PC2": abs(r_pc2) if np.isfinite(r_pc2) else np.nan,
            "abs_r_PC3": abs(r_pc3) if np.isfinite(r_pc3) else np.nan,
            "Best_PC": best_pc,
            "Best_PC_r": best_r,
            "Best_PC_abs_r": best_abs_r,
            "Second_Best_PC_abs_r": second_best_abs_r,
            "Best_PC_margin": best_margin,
            "Confident_PC_Assignment": confident_assignment
        })

    align_df_tmp = pd.DataFrame(align_rows)

    pc_counts = align_df_tmp["Best_PC"].value_counts(normalize=True)
    pc_counts_n = align_df_tmp["Best_PC"].value_counts()

    summary = {
        "Video": video_name,
        "Video_Base": video_base,
        "Trigger": trigger,
        "Modality": modality,
        "N_subjects": df_z.shape[1],
        "N_timepoints": df_z.shape[0],
        "PC1_var": pc1_var,
        "PC2_var": pc2_var,
        "PC3_var": pc3_var,
        "PC2_over_PC1": pc2_var / pc1_var if pc1_var and np.isfinite(pc1_var) else np.nan,
        "PC3_over_PC1": pc3_var / pc1_var if pc1_var and np.isfinite(pc1_var) else np.nan,
        "PC1_PC3_cumulative_var": np.nansum([pc1_var, pc2_var, pc3_var]),
        "Prop_Best_PC1": pc_counts.get("PC1", 0),
        "Prop_Best_PC2": pc_counts.get("PC2", 0),
        "Prop_Best_PC3": pc_counts.get("PC3", 0),
        "N_Best_PC1": pc_counts_n.get("PC1", 0),
        "N_Best_PC2": pc_counts_n.get("PC2", 0),
        "N_Best_PC3": pc_counts_n.get("PC3", 0),
        "Mean_abs_r_PC1": align_df_tmp["abs_r_PC1"].mean(),
        "Mean_abs_r_PC2": align_df_tmp["abs_r_PC2"].mean(),
        "Mean_abs_r_PC3": align_df_tmp["abs_r_PC3"].mean()
    }

    summary["Multiple_Trajectory_Flag"] = (
        summary["PC2_var"] >= PC2_VAR_THRESHOLD
        and summary["Prop_Best_PC2"] >= PC2_ALIGNED_PROP_THRESHOLD
    )

    return summary, align_rows, comp_df, df_z


def plot_pc_trajectories(video_name, modality, comp_df, df_z, align_df_video, out_dir):
    color_map = {"PC1": "tab:blue", "PC2": "tab:orange", "PC3": "tab:green"}

    plt.figure(figsize=(12, 6))

    for subj in df_z.columns:
        best_pc = align_df_video.loc[align_df_video["Subject"] == subj, "Best_PC"]
        c = color_map.get(best_pc.iloc[0], "gray") if len(best_pc) else "gray"

        plt.plot(
            np.arange(df_z.shape[0]),
            df_z[subj].values,
            color=c,
            alpha=0.18,
            linewidth=1
        )

    plt.plot(comp_df["Timepoint"], stats.zscore(comp_df["PC1"]), color="tab:blue", linewidth=3, label="PC1 consensus")
    plt.plot(comp_df["Timepoint"], stats.zscore(comp_df["PC2"]), color="tab:orange", linewidth=3, label="PC2 alternative")
    plt.plot(comp_df["Timepoint"], stats.zscore(comp_df["PC3"]), color="tab:green", linewidth=3, label="PC3 alternative")

    plt.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)

    plt.title(f"{modality} | Video {video_name}: PC Consensus / Counterfactual Trajectories")
    plt.xlabel("Timepoint")
    plt.ylabel("Z-scored rating / component trajectory")
    plt.legend(frameon=False)
    plt.tight_layout()

    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", f"{modality}_{video_name}")
    out_path = os.path.join(out_dir, f"pc_trajectories_{safe_name}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    return out_path


# ============================================================
# Run behavioral PC analysis
# ============================================================

valence_videos = {k: v for k, v in data_by_video.items() if "valence" in str(k).lower()}
arousal_videos = {k: v for k, v in data_by_video.items() if "arousal" in str(k).lower()}

print(f"Found {len(valence_videos)} valence videos and {len(arousal_videos)} arousal videos.")

video_summary_rows = []
alignment_rows = []
component_tables = []
cleaned_behavior_by_video = {}

for modality, video_dict in [("Valence", valence_videos), ("Arousal", arousal_videos)]:
    for video_name, subj_dict in video_dict.items():

        try:
            df_raw = clean_for_pca(subj_dict)
        except Exception as e:
            print(f"Could not clean {modality} {video_name}: {e}")
            continue

        summary, align_rows, comp_df, df_z = run_video_pc_analysis(
            df_raw,
            modality=modality,
            video_name=video_name
        )

        if summary is None:
            continue

        video_summary_rows.append(summary)
        alignment_rows.extend(align_rows)
        component_tables.append(comp_df)
        cleaned_behavior_by_video[(modality, video_name)] = df_z

pc_video_summary_df = pd.DataFrame(video_summary_rows)
pc_alignment_df = pd.DataFrame(alignment_rows)
pc_component_df = pd.concat(component_tables, ignore_index=True) if component_tables else pd.DataFrame()

pc_video_summary_df.to_csv(os.path.join(pc_out, "counterfactual_pc_video_summary.csv"), index=False)
pc_alignment_df.to_csv(os.path.join(pc_out, "counterfactual_pc_subject_video_alignment.csv"), index=False)
pc_component_df.to_csv(os.path.join(pc_out, "counterfactual_pc_component_trajectories.csv"), index=False)

print("\n=== PC video summary ===")
display(pc_video_summary_df.sort_values(["Multiple_Trajectory_Flag", "PC2_var"], ascending=False).round(4))

print("\n=== Subject × video PC alignment ===")
display(pc_alignment_df.head().round(4))

# ============================================================
# Plot PC trajectories
# ============================================================

plot_paths = []

if PLOT_ALL_VIDEOS:
    videos_to_plot = pc_video_summary_df.copy()
else:
    videos_to_plot = (
        pc_video_summary_df
        .sort_values(["Multiple_Trajectory_Flag", "PC2_var"], ascending=False)
        .head(PLOT_TOP_N_ALT_VIDEOS)
    )

for _, row in videos_to_plot.iterrows():
    modality = row["Modality"]
    video_name = row["Video"]

    comp_df = pc_component_df[
        (pc_component_df["Modality"] == modality) &
        (pc_component_df["Video"] == video_name)
    ]

    df_z = cleaned_behavior_by_video.get((modality, video_name), None)

    align_df_video = pc_alignment_df[
        (pc_alignment_df["Modality"] == modality) &
        (pc_alignment_df["Video"] == video_name)
    ]

    if df_z is None or comp_df.empty:
        continue

    path = plot_pc_trajectories(
        video_name=video_name,
        modality=modality,
        comp_df=comp_df,
        df_z=df_z,
        align_df_video=align_df_video,
        out_dir=fig_out
    )

    plot_paths.append(path)

multi_traj_videos = pc_video_summary_df[
    pc_video_summary_df["Multiple_Trajectory_Flag"] == True
].copy()

print(f"\nSaved {len(plot_paths)} PC trajectory plots to: {fig_out}")

print("\n=== Videos flagged as having multiple plausible trajectories ===")
display(
    multi_traj_videos[
        [
            "Modality", "Video", "Trigger", "N_subjects",
            "PC1_var", "PC2_var", "PC3_var",
            "PC2_over_PC1", "Prop_Best_PC1", "Prop_Best_PC2", "Prop_Best_PC3"
        ]
    ].sort_values("PC2_var", ascending=False).round(4)
)

print("\nBehavioral PC analysis complete.")
print(f"Saved outputs to: {pc_out}")

In [ ]:
# ============================================================
#  NEURAL ISC × BEHAVIORAL PC ALIGNMENT ANALYSIS
#
# Requires Cell 1 outputs:
#   pc_alignment_df
#
# Requires fNIRS output:
#   video_roi_isc_df
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr, ttest_ind
from statsmodels.stats.multitest import multipletests

if "pc_alignment_df" not in globals():
    raise ValueError("pc_alignment_df not found. Run Cell 1 first.")

if "video_roi_isc_df" not in globals():
    raise ValueError("video_roi_isc_df not found. Run your fNIRS ISC pipeline first.")

if "pc_out" not in globals():
    if "analysis_out" not in globals():
        analysis_out = os.path.join(os.getcwd(), "counterfactual_pc_analysis")
    pc_out = os.path.join(analysis_out, "counterfactual_pc_by_video")
    os.makedirs(pc_out, exist_ok=True)

N_PERM = 10000
RNG = np.random.default_rng(42)

# ============================================================
# Step 1: Prepare neural ISC table
# ============================================================

neural_df = video_roi_isc_df.copy()
neural_df["Subject"] = neural_df["Subject"].astype(str)
neural_df["Trigger"] = neural_df["Trigger"].astype(str)

video_roi_wide = (
    neural_df
    .pivot_table(
        index=["Subject", "Trigger"],
        columns="ROI",
        values="ISC",
        aggfunc="mean"
    )
    .reset_index()
)

roi_cols_neural = [c for c in video_roi_wide.columns if c not in ["Subject", "Trigger"]]
video_roi_wide["Mean_ROI_ISC"] = video_roi_wide[roi_cols_neural].mean(axis=1)

needed_original = ["Right VLPFC", "Left TPJ", "Left VLPFC", "Right TPJ"]

if all(c in video_roi_wide.columns for c in needed_original):
    video_roi_wide["Original_Network_Balance"] = (
        video_roi_wide[["Right VLPFC", "Left TPJ"]].mean(axis=1) -
        video_roi_wide[["Left VLPFC", "Right TPJ"]].mean(axis=1)
    )

positive_lasso_rois = ["Left TPJ", "Right VLPFC", "Left OFC", "Left STS"]
negative_lasso_rois = ["Left VLPFC", "Left DLPFC", "Right TPJ"]

if all(c in video_roi_wide.columns for c in positive_lasso_rois + negative_lasso_rois):
    video_roi_wide["Full_LASSO_Network_Balance"] = (
        video_roi_wide[positive_lasso_rois].mean(axis=1) -
        video_roi_wide[negative_lasso_rois].mean(axis=1)
    )

# ============================================================
# Step 2: Merge behavioral PC alignment with neural ISC
# ============================================================

pc_alignment_merge = pc_alignment_df.copy()
pc_alignment_merge["Subject"] = pc_alignment_merge["Subject"].astype(str)
pc_alignment_merge["Trigger"] = pc_alignment_merge["Trigger"].astype(str)

pc_neural_df = pd.merge(
    pc_alignment_merge,
    video_roi_wide,
    on=["Subject", "Trigger"],
    how="left"
)

pc_neural_df.to_csv(
    os.path.join(pc_out, "counterfactual_pc_alignment_with_neural_isc.csv"),
    index=False
)

print("\n=== PC alignment merged with neural ISC ===")
display(pc_neural_df.head().round(4))

# ============================================================
# Step 3: Correlate PC alignment with neural ISC / network balance
# ============================================================

alignment_vars = [
    "abs_r_PC1",
    "abs_r_PC2",
    "abs_r_PC3",
    "Best_PC_abs_r"
]

neural_predictors = ["Mean_ROI_ISC"]

for c in ["Original_Network_Balance", "Full_LASSO_Network_Balance"]:
    if c in pc_neural_df.columns:
        neural_predictors.append(c)

neural_predictors.extend(roi_cols_neural)

corr_rows = []

for modality in pc_neural_df["Modality"].dropna().unique():
    df_mod = pc_neural_df[pc_neural_df["Modality"] == modality].copy()

    for neural_var in neural_predictors:
        if neural_var not in df_mod.columns:
            continue

        for align_var in alignment_vars:
            tmp = df_mod[[neural_var, align_var]].dropna()

            if len(tmp) < 8:
                continue

            x = tmp[neural_var].values
            y = tmp[align_var].values

            if np.std(x) == 0 or np.std(y) == 0:
                continue

            r, p = pearsonr(x, y)
            rho, p_s = spearmanr(x, y)

            perm_rs = np.empty(N_PERM)
            for i in range(N_PERM):
                y_perm = RNG.permutation(y)
                perm_rs[i] = pearsonr(x, y_perm)[0]

            p_perm = (1 + np.sum(np.abs(perm_rs) >= abs(r))) / (N_PERM + 1)

            corr_rows.append({
                "Modality": modality,
                "Neural_Predictor": neural_var,
                "Alignment_Outcome": align_var,
                "N_subject_video": len(tmp),
                "Pearson_r": r,
                "Pearson_p": p,
                "Permutation_p": p_perm,
                "Spearman_rho": rho,
                "Spearman_p": p_s
            })

pc_neural_corr_df = pd.DataFrame(corr_rows)

if not pc_neural_corr_df.empty:
    pc_neural_corr_df["Pearson_p_FDR"] = multipletests(
        pc_neural_corr_df["Pearson_p"],
        method="fdr_bh"
    )[1]

    pc_neural_corr_df["Permutation_p_FDR"] = multipletests(
        pc_neural_corr_df["Permutation_p"],
        method="fdr_bh"
    )[1]

    pc_neural_corr_df.to_csv(
        os.path.join(pc_out, "counterfactual_pc_alignment_neural_correlations.csv"),
        index=False
    )

    print("\n=== Neural ISC / network balance correlations with PC alignment ===")
    display(pc_neural_corr_df.sort_values("Permutation_p").round(4))
else:
    print("No neural-PC correlations were computed. Check merge keys / missing data.")

# ============================================================
# Step 4: Compare neural ISC by Best_PC group
# ============================================================

group_rows = []

group_neural_vars = [
    "Mean_ROI_ISC",
    "Original_Network_Balance",
    "Full_LASSO_Network_Balance"
]

group_neural_vars = [c for c in group_neural_vars if c in pc_neural_df.columns]

for modality in pc_neural_df["Modality"].dropna().unique():
    df_mod = pc_neural_df[pc_neural_df["Modality"] == modality].copy()

    for neural_var in group_neural_vars:
        for comparison_pc in ["PC2", "PC3"]:

            g_alt = df_mod.loc[df_mod["Best_PC"] == comparison_pc, neural_var].dropna().values
            g_pc1 = df_mod.loc[df_mod["Best_PC"] == "PC1", neural_var].dropna().values

            if len(g_alt) < 5 or len(g_pc1) < 5:
                continue

            t_stat, p_t = ttest_ind(g_alt, g_pc1, equal_var=False)

            mean_alt = np.mean(g_alt)
            mean_pc1 = np.mean(g_pc1)
            diff = mean_alt - mean_pc1

            pooled = np.sqrt(
                ((len(g_alt) - 1) * np.var(g_alt, ddof=1) +
                 (len(g_pc1) - 1) * np.var(g_pc1, ddof=1)) /
                (len(g_alt) + len(g_pc1) - 2)
            )

            d = diff / pooled if pooled > 0 else np.nan

            group_rows.append({
                "Modality": modality,
                "Neural_Variable": neural_var,
                "Comparison": f"{comparison_pc}_aligned_vs_PC1_aligned",
                "N_alt": len(g_alt),
                "N_PC1": len(g_pc1),
                "Mean_alt": mean_alt,
                "Mean_PC1": mean_pc1,
                "Difference_alt_minus_PC1": diff,
                "T": t_stat,
                "P": p_t,
                "Cohen_d": d
            })

pc_group_neural_df = pd.DataFrame(group_rows)

if not pc_group_neural_df.empty:
    pc_group_neural_df["P_FDR"] = multipletests(
        pc_group_neural_df["P"],
        method="fdr_bh"
    )[1]

    pc_group_neural_df.to_csv(
        os.path.join(pc_out, "counterfactual_best_pc_group_neural_comparisons.csv"),
        index=False
    )

    print("\n=== Neural ISC / network balance by Best-PC group ===")
    display(pc_group_neural_df.sort_values("P").round(4))
else:
    print("No Best-PC neural group comparisons were computed.")

print("\nNeural PC-alignment analysis complete.")
print(f"Saved outputs to: {pc_out}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist, squareform


def plot_rater_clustering(df, n_perm=1000, alpha=0.05, seed=0):
    """DataFrame (timepoints × raters) → PCA + Horn's parallel analysis → reordered distance heatmap."""

    # Drop raters with no variance (constant columns would break z-scoring)
    df = df.loc[:, df.std(axis=0, ddof=1) > 1e-9]

    # Z-score each rater before PCA 
    M = (df.values - df.values.mean(0)) / df.values.std(0, ddof=1)

    # Fit PCA
    k   = min(M.shape[1] - 1, M.shape[0] - 1, 5)
    rng = np.random.default_rng(seed)
    pca = PCA(n_components=k).fit(M)

    # Horn's parallel analysis: permuted PCA, helps us assess how many components are significant
    null_evr = np.array([
        PCA(n_components=k).fit(
            np.column_stack([rng.permutation(M[:, j]) for j in range(M.shape[1])])
        ).explained_variance_ratio_
        for _ in range(n_perm)
    ])

    # find which PCs are signficicant
    sig_idx = np.where(pca.explained_variance_ratio_ > np.percentile(null_evr, 100 * (1 - alpha), axis=0))[0]
    print(f'n significant PCs (alpha={alpha}): {len(sig_idx)}')
    if len(sig_idx) == 0:
        return None

    # cluster participants by PC scores with Ward linkage
    loadings   = pca.components_[sig_idx, :].T
    link       = linkage(pdist(loadings, metric='euclidean'), method='ward') 
    leaves     = leaves_list(link)                                  # rater order from the dendrogram
    reordered  = squareform(pdist(loadings))[np.ix_(leaves, leaves)]  # rater × rater distance, reordered

    # find where the boundaries between groups are
    clusters   = fcluster(link, t=max(2, len(sig_idx)), criterion='maxclust')
    boundaries = np.where(np.diff(clusters[leaves]) != 0)[0] + 0.5

    # plot ordered  matrix
    fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
    im = ax.imshow(reordered, cmap='magma_r', aspect='equal')
    for b in boundaries:
        ax.axhline(b, color='white', lw=1.5)
        ax.axvline(b, color='white', lw=1.5)
    ax.set(xticks=[], yticks=[])
    fig.colorbar(im, ax=ax, shrink=0.7, label='Euclidean distance')
    plt.show()

# run plotting figure
plot_rater_clustering(df_example)

In [ ]:
# ============================================================
# 
# COUNTERFACTUAL TRAJECTORY / PC STRUCTURE
# Saves all tables, summaries, and figures
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Emotion Perception Data\counterfactual"
)

CELL1_DIR = OUTPUT_DIR / "01_counterfactual_pc_structure"
CELL1_FIG_DIR = CELL1_DIR / "figures"
CELL1_TABLE_DIR = CELL1_DIR / "tables"

for folder in [
    OUTPUT_DIR,
    CELL1_DIR,
    CELL1_FIG_DIR,
    CELL1_TABLE_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Stage 1 output directory:\n{OUTPUT_DIR}")


# ------------------------------------------------------------
# Required input
# ------------------------------------------------------------
if "data_by_video" not in globals():
    raise ValueError(
        "data_by_video was not found. Run the existing "
        "Behavioral Analysis CCT cell first."
    )


# ------------------------------------------------------------
# User-adjustable settings
# ------------------------------------------------------------
N_PCS = 3

MIN_SUBJECTS_PER_VIDEO = 5
MIN_TIMEPOINTS_PER_VIDEO = 20

PC2_VARIANCE_THRESHOLD = 0.15
PC2_ALIGNMENT_PROPORTION_THRESHOLD = 0.20

N_VIDEOS_TO_PLOT = 8
FIGURE_DPI = 300


# ------------------------------------------------------------
# General helper functions
# ------------------------------------------------------------
def safe_filename(value, max_length=150):
    """
    Convert text into a Windows-safe filename.
    """
    value = str(value)

    value = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        value
    )

    value = re.sub(
        r"\s+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    value = value.strip("._ ")

    if len(value) > max_length:
        value = value[:max_length]

    return value or "unnamed"


def save_dataframe(df, path, index=False):
    """
    Save a DataFrame and print its location.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        path,
        index=index
    )

    print(f"Saved table: {path}")


def save_figure(fig, path, dpi=300):
    """
    Save a Matplotlib figure and print its location.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight"
    )

    print(f"Saved figure: {path}")


def identify_modality(video_name):
    """
    Identify whether a behavioral column contains valence or arousal.
    """
    name = str(video_name).lower()

    if "valence" in name:
        return "Valence"

    if "arousal" in name:
        return "Arousal"

    return "Unknown"


def remove_modality_suffix(video_name):
    """
    Return the underlying video identifier by removing only the
    trailing modality suffix.

    Examples
    --------
    video_17_valence -> video_17
    video_17_arousal -> video_17

    The numeric portion is retained because it is part of the
    video identifier and is not assumed to be a trigger.
    """
    return re.sub(
        r"_(valence|arousal)$",
        "",
        str(video_name),
        flags=re.IGNORECASE
    )


def get_video_trigger(video_name, trigger_mapping=None):
    """
    Return the real experimental trigger only when an explicit,
    validated mapping has been supplied.

    Parameters
    ----------
    video_name : str
        Full behavioral trajectory name, such as video_17_valence.

    trigger_mapping : dict or None
        Dictionary mapping the underlying video identifier to its
        actual experimental trigger.

        Example:
        {
            "video_17": "5",
            "video_18": "12"
        }

    Returns
    -------
    str or pandas.NA
        The actual trigger when available. Otherwise pandas.NA.

    Important
    ---------
    This function never extracts a number from the video name,
    because the number is part of the video identity rather than
    necessarily the trigger.
    """
    base_video = remove_modality_suffix(video_name)

    if trigger_mapping is None:
        return pd.NA

    return trigger_mapping.get(
        base_video,
        pd.NA
    )


def safe_pearsonr(x, y, minimum_n=5):
    """
    Pearson correlation with finite-value and variance checks.
    """
    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    n_common = min(
        len(x),
        len(y)
    )

    x = x[:n_common]
    y = y[:n_common]

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    if valid.sum() < minimum_n:
        return np.nan, np.nan

    x = x[valid]
    y = y[valid]

    if (
        np.std(x, ddof=1) == 0
        or np.std(y, ddof=1) == 0
    ):
        return np.nan, np.nan

    return pearsonr(x, y)


def prepare_video_matrix(subject_dictionary):
    """
    Convert one video's subject dictionary into a
    timepoint × subject matrix.

    Processing:
      1. Keep numeric one-dimensional trajectories.
      2. Trim participants to the shortest trajectory.
      3. Remove timepoints containing missing values.
      4. Remove zero-variance participants.
      5. Z-score each participant across time.
    """
    usable = {}

    for subject, trajectory in subject_dictionary.items():

        x = pd.to_numeric(
            pd.Series(
                np.asarray(trajectory).ravel()
            ),
            errors="coerce"
        ).to_numpy(dtype=float)

        if len(x) >= MIN_TIMEPOINTS_PER_VIDEO:
            usable[str(subject)] = x

    if len(usable) < MIN_SUBJECTS_PER_VIDEO:
        return None

    minimum_length = min(
        len(x)
        for x in usable.values()
    )

    matrix = pd.DataFrame({
        subject: trajectory[:minimum_length]
        for subject, trajectory in usable.items()
    })

    matrix = matrix.replace(
        [np.inf, -np.inf],
        np.nan
    )

    matrix = matrix.dropna(
        axis=0,
        how="any"
    )

    standard_deviations = matrix.std(
        axis=0,
        ddof=1
    )

    valid_subjects = (
        standard_deviations.notna()
        & (standard_deviations > 0)
    )

    matrix = matrix.loc[
        :,
        valid_subjects
    ]

    if (
        matrix.shape[1] < MIN_SUBJECTS_PER_VIDEO
        or matrix.shape[0] < MIN_TIMEPOINTS_PER_VIDEO
    ):
        return None

    matrix_z = (
        matrix - matrix.mean(axis=0)
    ) / matrix.std(axis=0, ddof=1)

    matrix_z = matrix_z.replace(
        [np.inf, -np.inf],
        np.nan
    )

    matrix_z = matrix_z.dropna(
        axis=0,
        how="any"
    )

    matrix_z = matrix_z.dropna(
        axis=1,
        how="any"
    )

    if (
        matrix_z.shape[1] < MIN_SUBJECTS_PER_VIDEO
        or matrix_z.shape[0] < MIN_TIMEPOINTS_PER_VIDEO
    ):
        return None

    return matrix_z


def align_pc_signs(pc_scores, behavior_matrix):
    """
    Flip each PC when necessary so that it correlates positively
    with the group-average behavioral trajectory.
    """
    aligned = np.asarray(
        pc_scores,
        dtype=float
    ).copy()

    group_mean = behavior_matrix.mean(
        axis=1
    ).to_numpy(dtype=float)

    for pc_index in range(
        aligned.shape[1]
    ):
        r_value, _ = safe_pearsonr(
            aligned[:, pc_index],
            group_mean
        )

        if np.isfinite(r_value) and r_value < 0:
            aligned[:, pc_index] *= -1

    return aligned


# ------------------------------------------------------------
# Run PCA separately for every video × modality
# ------------------------------------------------------------
pc_video_rows = []
pc_alignment_rows = []
pc_trajectory_tables = []

behavior_by_video_clean = {}

skipped_video_rows = []

for video_name, subject_dictionary in data_by_video.items():

    modality = identify_modality(
        video_name
    )

    if modality == "Unknown":
        continue

    behavior_matrix = prepare_video_matrix(
        subject_dictionary
    )

    if behavior_matrix is None:

        skipped_video_rows.append({
            "Video": str(video_name),
            "Modality": modality,
            "Reason": (
                "Insufficient usable participants or timepoints"
            )
        })

        print(
            f"Skipped {video_name}: insufficient usable "
            f"participants or timepoints."
        )

        continue

    maximum_components = min(
        N_PCS,
        behavior_matrix.shape[0],
        behavior_matrix.shape[1]
    )

    if maximum_components < 2:

        skipped_video_rows.append({
            "Video": str(video_name),
            "Modality": modality,
            "Reason": "Fewer than two estimable PCs"
        })

        print(
            f"Skipped {video_name}: fewer than two estimable PCs."
        )

        continue

    pca = PCA(
        n_components=maximum_components,
        svd_solver="full"
    )

    pc_scores = pca.fit_transform(
        behavior_matrix.to_numpy()
    )

    pc_scores = align_pc_signs(
        pc_scores,
        behavior_matrix
    )

    explained_variance = np.full(
        N_PCS,
        np.nan,
        dtype=float
    )

    explained_variance[
        :maximum_components
    ] = pca.explained_variance_ratio_

    pc_matrix = np.full(
        (
            behavior_matrix.shape[0],
            N_PCS
        ),
        np.nan,
        dtype=float
    )

    pc_matrix[
        :,
        :maximum_components
    ] = pc_scores

    base_video = remove_modality_suffix(
        video_name
    )

    trigger = extract_trigger(
        video_name
    )

    trajectory_df = pd.DataFrame({
        "Timepoint": np.arange(
            behavior_matrix.shape[0]
        ),
        "PC1": pc_matrix[:, 0],
        "PC2": pc_matrix[:, 1],
        "PC3": pc_matrix[:, 2],
        "Modality": modality,
        "Video": str(video_name),
        "BaseVideo": base_video,
        "Trigger": trigger
    })

    pc_trajectory_tables.append(
        trajectory_df
    )

    video_alignment_rows = []

    for subject in behavior_matrix.columns:

        trajectory = behavior_matrix[
            subject
        ].to_numpy(dtype=float)

        correlations = {}
        p_values = {}

        for pc_number in range(
            1,
            N_PCS + 1
        ):

            component = pc_matrix[
                :,
                pc_number - 1
            ]

            if np.all(
                ~np.isfinite(component)
            ):
                r_value = np.nan
                p_value = np.nan

            else:
                r_value, p_value = safe_pearsonr(
                    trajectory,
                    component
                )

            correlations[
                f"r_PC{pc_number}"
            ] = r_value

            p_values[
                f"p_PC{pc_number}"
            ] = p_value

        signed_r = np.array([
            correlations["r_PC1"],
            correlations["r_PC2"],
            correlations["r_PC3"]
        ], dtype=float)

        absolute_r = np.abs(
            signed_r
        )

        if np.all(
            ~np.isfinite(absolute_r)
        ):
            best_pc = np.nan
            best_pc_r = np.nan
            best_pc_abs_r = np.nan
            second_best_abs_r = np.nan
            best_pc_margin = np.nan

        else:
            ranking_values = np.where(
                np.isfinite(absolute_r),
                absolute_r,
                -np.inf
            )

            order = np.argsort(
                ranking_values
            )[::-1]

            best_index = int(
                order[0]
            )

            valid_order = [
                index
                for index in order
                if np.isfinite(
                    absolute_r[index]
                )
            ]

            best_pc = f"PC{best_index + 1}"
            best_pc_r = signed_r[
                best_index
            ]

            best_pc_abs_r = absolute_r[
                best_index
            ]

            if len(valid_order) >= 2:
                second_best_abs_r = absolute_r[
                    valid_order[1]
                ]

                best_pc_margin = (
                    best_pc_abs_r
                    - second_best_abs_r
                )

            else:
                second_best_abs_r = np.nan
                best_pc_margin = np.nan

        row = {
            "Subject": str(subject),
            "Modality": modality,
            "Video": str(video_name),
            "BaseVideo": base_video,
            "Trigger": trigger,
            **correlations,
            **p_values,
            "abs_r_PC1": (
                abs(correlations["r_PC1"])
                if np.isfinite(
                    correlations["r_PC1"]
                )
                else np.nan
            ),
            "abs_r_PC2": (
                abs(correlations["r_PC2"])
                if np.isfinite(
                    correlations["r_PC2"]
                )
                else np.nan
            ),
            "abs_r_PC3": (
                abs(correlations["r_PC3"])
                if np.isfinite(
                    correlations["r_PC3"]
                )
                else np.nan
            ),
            "Best_PC": best_pc,
            "Best_PC_r": best_pc_r,
            "Best_PC_abs_r": best_pc_abs_r,
            "Second_Best_PC_abs_r": second_best_abs_r,
            "Best_PC_margin": best_pc_margin
        }

        video_alignment_rows.append(
            row
        )

        pc_alignment_rows.append(
            row
        )

    video_alignment_df = pd.DataFrame(
        video_alignment_rows
    )

    best_pc_counts = (
        video_alignment_df["Best_PC"]
        .value_counts(dropna=True)
    )

    best_pc_proportions = (
        video_alignment_df["Best_PC"]
        .value_counts(
            normalize=True,
            dropna=True
        )
    )

    pc1_variance = explained_variance[0]
    pc2_variance = explained_variance[1]
    pc3_variance = explained_variance[2]

    multiple_trajectory_flag = bool(
        np.isfinite(pc2_variance)
        and (
            pc2_variance
            >= PC2_VARIANCE_THRESHOLD
        )
        and (
            best_pc_proportions.get(
                "PC2",
                0
            )
            >= PC2_ALIGNMENT_PROPORTION_THRESHOLD
        )
    )

    pc_video_rows.append({
        "Modality": modality,
        "Video": str(video_name),
        "BaseVideo": base_video,
        "Trigger": trigger,
        "N_Subjects": behavior_matrix.shape[1],
        "N_Timepoints": behavior_matrix.shape[0],
        "PC1_Variance": pc1_variance,
        "PC2_Variance": pc2_variance,
        "PC3_Variance": pc3_variance,
        "PC1_to_PC3_Cumulative_Variance": np.nansum(
            explained_variance
        ),
        "PC2_to_PC1_Ratio": (
            pc2_variance / pc1_variance
            if (
                np.isfinite(pc1_variance)
                and pc1_variance > 0
            )
            else np.nan
        ),
        "PC3_to_PC1_Ratio": (
            pc3_variance / pc1_variance
            if (
                np.isfinite(pc1_variance)
                and pc1_variance > 0
            )
            else np.nan
        ),
        "N_Best_PC1": best_pc_counts.get(
            "PC1",
            0
        ),
        "N_Best_PC2": best_pc_counts.get(
            "PC2",
            0
        ),
        "N_Best_PC3": best_pc_counts.get(
            "PC3",
            0
        ),
        "Prop_Best_PC1": best_pc_proportions.get(
            "PC1",
            0
        ),
        "Prop_Best_PC2": best_pc_proportions.get(
            "PC2",
            0
        ),
        "Prop_Best_PC3": best_pc_proportions.get(
            "PC3",
            0
        ),
        "Mean_Abs_r_PC1": video_alignment_df[
            "abs_r_PC1"
        ].mean(),
        "Mean_Abs_r_PC2": video_alignment_df[
            "abs_r_PC2"
        ].mean(),
        "Mean_Abs_r_PC3": video_alignment_df[
            "abs_r_PC3"
        ].mean(),
        "Multiple_Trajectory_Flag": (
            multiple_trajectory_flag
        )
    })

    behavior_by_video_clean[
        (
            modality,
            str(video_name)
        )
    ] = behavior_matrix


# ------------------------------------------------------------
# Final Cell 1 outputs
# ------------------------------------------------------------
pc_video_summary_df = pd.DataFrame(
    pc_video_rows
)

pc_alignment_df = pd.DataFrame(
    pc_alignment_rows
)

pc_component_df = (
    pd.concat(
        pc_trajectory_tables,
        ignore_index=True
    )
    if len(pc_trajectory_tables) > 0
    else pd.DataFrame()
)

skipped_pc_videos_df = pd.DataFrame(
    skipped_video_rows
)

if pc_video_summary_df.empty:
    raise RuntimeError(
        "No videos could be analyzed. "
        "Check the structure of data_by_video."
    )

pc_video_summary_df = (
    pc_video_summary_df
    .sort_values(
        [
            "Multiple_Trajectory_Flag",
            "PC2_Variance"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Save Cell 1 tables
# ------------------------------------------------------------
save_dataframe(
    pc_video_summary_df,
    CELL1_TABLE_DIR
    / "pc_video_summary.csv"
)

save_dataframe(
    pc_alignment_df,
    CELL1_TABLE_DIR
    / "subject_video_pc_alignment.csv"
)

save_dataframe(
    pc_component_df,
    CELL1_TABLE_DIR
    / "pc_temporal_trajectories.csv"
)

if not skipped_pc_videos_df.empty:
    save_dataframe(
        skipped_pc_videos_df,
        CELL1_TABLE_DIR
        / "skipped_videos.csv"
    )


# Save each cleaned behavioral matrix
CLEAN_MATRIX_DIR = (
    CELL1_TABLE_DIR
    / "cleaned_behavior_matrices"
)

CLEAN_MATRIX_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for (
    modality,
    video_name
), behavior_matrix in behavior_by_video_clean.items():

    matrix_filename = (
        f"{safe_filename(modality)}__"
        f"{safe_filename(video_name)}"
        f"__cleaned_behavior_matrix.csv"
    )

    behavior_matrix.to_csv(
        CLEAN_MATRIX_DIR
        / matrix_filename,
        index=True,
        index_label="Timepoint"
    )


# ------------------------------------------------------------
# Print and save text summary
# ------------------------------------------------------------
n_analyzed = len(
    pc_video_summary_df
)

n_flagged = int(
    pc_video_summary_df[
        "Multiple_Trajectory_Flag"
    ].sum()
)

summary_lines = [
    "=" * 90,
    "COUNTERFACTUAL TRAJECTORY / PCA RESULTS",
    "=" * 90,
    "",
    (
        f"Analyzed {n_analyzed} "
        "video × modality datasets."
    ),
    (
        "Videos meeting the provisional "
        "multiple-trajectory criteria: "
        f"{n_flagged} of {n_analyzed}"
    ),
    "",
    "Thresholds:",
    (
        f"PC2 variance threshold = "
        f"{PC2_VARIANCE_THRESHOLD:.3f}"
    ),
    (
        "PC2 best-alignment proportion threshold = "
        f"{PC2_ALIGNMENT_PROPORTION_THRESHOLD:.3f}"
    ),
    "",
    "Mean variance explained:",
    (
        f"PC1 = "
        f"{pc_video_summary_df['PC1_Variance'].mean():.4f}"
    ),
    (
        f"PC2 = "
        f"{pc_video_summary_df['PC2_Variance'].mean():.4f}"
    ),
    (
        f"PC3 = "
        f"{pc_video_summary_df['PC3_Variance'].mean():.4f}"
    )
]

summary_text = "\n".join(
    summary_lines
)

print(
    "\n" + summary_text
)

with open(
    CELL1_DIR
    / "counterfactual_pc_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(
        summary_text
    )


print("\nVideo-level PCA summary:")

display(
    pc_video_summary_df[
        [
            "Modality",
            "Video",
            "N_Subjects",
            "N_Timepoints",
            "PC1_Variance",
            "PC2_Variance",
            "PC3_Variance",
            "PC1_to_PC3_Cumulative_Variance",
            "PC2_to_PC1_Ratio",
            "Prop_Best_PC1",
            "Prop_Best_PC2",
            "Prop_Best_PC3",
            "Multiple_Trajectory_Flag"
        ]
    ].round(4)
)

print("\nSubject × video PC-alignment preview:")

display(
    pc_alignment_df.head(
        20
    ).round(4)
)


# ------------------------------------------------------------
# Plot 1: explained variance across all videos
# ------------------------------------------------------------
plot_df = (
    pc_video_summary_df
    .sort_values(
        "PC2_Variance",
        ascending=False
    )
    .reset_index(drop=True)
)

x_positions = np.arange(
    len(plot_df)
)

fig, ax = plt.subplots(
    figsize=(
        max(
            12,
            len(plot_df) * 0.35
        ),
        6
    )
)

ax.plot(
    x_positions,
    plot_df["PC1_Variance"],
    marker="o",
    label="PC1"
)

ax.plot(
    x_positions,
    plot_df["PC2_Variance"],
    marker="o",
    label="PC2"
)

ax.plot(
    x_positions,
    plot_df["PC3_Variance"],
    marker="o",
    label="PC3"
)

ax.axhline(
    PC2_VARIANCE_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"PC2 flag threshold = "
        f"{PC2_VARIANCE_THRESHOLD:.2f}"
    )
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    (
        plot_df["Modality"].str[0]
        + ":"
        + plot_df["BaseVideo"].astype(str)
    ),
    rotation=90
)

ax.set_xlabel(
    "Video × modality"
)

ax.set_ylabel(
    "Proportion of variance explained"
)

ax.set_title(
    "Behavioral PC structure across videos"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

save_figure(
    fig,
    CELL1_FIG_DIR
    / "pc_explained_variance_across_videos.png",
    dpi=FIGURE_DPI
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Plot 2: strongest alternative-trajectory videos
# ------------------------------------------------------------
videos_to_plot = (
    pc_video_summary_df
    .sort_values(
        [
            "Multiple_Trajectory_Flag",
            "PC2_Variance"
        ],
        ascending=[
            False,
            False
        ]
    )
    .head(
        N_VIDEOS_TO_PLOT
    )
)

for _, video_row in videos_to_plot.iterrows():

    modality = video_row[
        "Modality"
    ]

    video_name = video_row[
        "Video"
    ]

    behavior_matrix = behavior_by_video_clean[
        (
            modality,
            video_name
        )
    ]

    component_subset = (
        pc_component_df[
            (
                pc_component_df[
                    "Modality"
                ] == modality
            )
            & (
                pc_component_df[
                    "Video"
                ] == video_name
            )
        ]
        .sort_values(
            "Timepoint"
        )
    )

    alignment_subset = pc_alignment_df[
        (
            pc_alignment_df[
                "Modality"
            ] == modality
        )
        & (
            pc_alignment_df[
                "Video"
            ] == video_name
        )
    ]

    fig, ax = plt.subplots(
        figsize=(12, 5.5)
    )

    pc_colors = {
        "PC1": "tab:blue",
        "PC2": "tab:orange",
        "PC3": "tab:green"
    }

    for subject in behavior_matrix.columns:

        subject_best_pc = (
            alignment_subset.loc[
                alignment_subset[
                    "Subject"
                ] == str(subject),
                "Best_PC"
            ]
        )

        color = (
            pc_colors.get(
                subject_best_pc.iloc[0],
                "gray"
            )
            if len(subject_best_pc) > 0
            else "gray"
        )

        ax.plot(
            behavior_matrix[
                subject
            ].to_numpy(),
            linewidth=0.9,
            alpha=0.16,
            color=color
        )

    for pc_name in [
        "PC1",
        "PC2",
        "PC3"
    ]:

        values = component_subset[
            pc_name
        ].to_numpy(dtype=float)

        if np.all(
            ~np.isfinite(values)
        ):
            continue

        value_sd = np.nanstd(
            values,
            ddof=1
        )

        if (
            not np.isfinite(value_sd)
            or value_sd == 0
        ):
            continue

        values_z = (
            values
            - np.nanmean(values)
        ) / value_sd

        ax.plot(
            component_subset[
                "Timepoint"
            ],
            values_z,
            linewidth=3,
            color=pc_colors[
                pc_name
            ],
            label=(
                f"{pc_name}: "
                f"{video_row[f'{pc_name}_Variance']:.1%} variance"
            )
        )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "Timepoint"
    )

    ax.set_ylabel(
        "Standardized behavioral trajectory"
    )

    ax.set_title(
        f"{modality} | {video_name}\n"
        f"PC1={video_row['PC1_Variance']:.1%}, "
        f"PC2={video_row['PC2_Variance']:.1%}, "
        f"PC3={video_row['PC3_Variance']:.1%}"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_filename = (
        f"{safe_filename(modality)}__"
        f"{safe_filename(video_name)}"
        f"__counterfactual_pc_trajectories.png"
    )

    save_figure(
        fig,
        CELL1_FIG_DIR
        / figure_filename,
        dpi=FIGURE_DPI
    )

    plt.show()
    plt.close(fig)


print("\nCell 1 complete.")
print(f"Outputs saved in:\n{CELL1_DIR}")
print("\nCreated in memory:")
print("  pc_video_summary_df")
print("  pc_alignment_df")
print("  pc_component_df")
print("  behavior_by_video_clean")

In [ ]:
# ============================================================
# 
# PER-VIDEO HIERARCHICAL CLUSTERING
# Saves all tables, summaries, matrices, and figures
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from scipy.cluster.hierarchy import (
    linkage,
    fcluster,
    dendrogram,
    leaves_list
)

from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score


# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
CELL2_DIR = OUTPUT_DIR / "02_hierarchical_clustering"
CELL2_FIG_DIR = CELL2_DIR / "figures"
CELL2_TABLE_DIR = CELL2_DIR / "tables"
CELL2_MATRIX_DIR = CELL2_TABLE_DIR / "correlation_distance_matrices"

for folder in [
    CELL2_DIR,
    CELL2_FIG_DIR,
    CELL2_TABLE_DIR,
    CELL2_MATRIX_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Check Cell 1 outputs
# ------------------------------------------------------------
if "behavior_by_video_clean" not in globals():
    raise ValueError(
        "behavior_by_video_clean was not found. "
        "Run Stage 1 Cell 1 first."
    )


# ------------------------------------------------------------
# User-adjustable settings
# ------------------------------------------------------------
CANDIDATE_CLUSTER_COUNTS = (
    2,
    3
)

MIN_CLUSTER_SIZE = 2
N_CLUSTER_VIDEOS_TO_PLOT = 8
FIGURE_DPI = 300


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def build_correlation_and_distance_matrices(
    behavior_matrix
):
    """
    Compute subject × subject correlations and correlation distances.

    Distance:
        (1 - r) / 2

    Mapping:
        r =  1 -> distance 0
        r =  0 -> distance 0.5
        r = -1 -> distance 1
    """
    correlation_df = behavior_matrix.corr(
        method="pearson"
    )

    correlation_matrix = correlation_df.to_numpy(
        dtype=float
    )

    correlation_matrix = np.where(
        np.isfinite(correlation_matrix),
        correlation_matrix,
        0.0
    )

    correlation_matrix = np.clip(
        correlation_matrix,
        -1.0,
        1.0
    )

    correlation_matrix = (
        correlation_matrix
        + correlation_matrix.T
    ) / 2.0

    np.fill_diagonal(
        correlation_matrix,
        1.0
    )

    distance_matrix = (
        1.0
        - correlation_matrix
    ) / 2.0

    distance_matrix = np.clip(
        distance_matrix,
        0.0,
        1.0
    )

    distance_matrix = (
        distance_matrix
        + distance_matrix.T
    ) / 2.0

    np.fill_diagonal(
        distance_matrix,
        0.0
    )

    return (
        correlation_matrix,
        distance_matrix
    )


def evaluate_cluster_solution(
    distance_matrix,
    linkage_matrix,
    n_clusters,
    minimum_cluster_size=2
):
    """
    Evaluate one hierarchical-clustering solution.
    """
    labels = fcluster(
        linkage_matrix,
        t=n_clusters,
        criterion="maxclust"
    ).astype(int)

    unique_labels, cluster_sizes = np.unique(
        labels,
        return_counts=True
    )

    if len(unique_labels) != n_clusters:
        return (
            labels,
            np.nan,
            cluster_sizes
        )

    if np.min(
        cluster_sizes
    ) < minimum_cluster_size:
        return (
            labels,
            np.nan,
            cluster_sizes
        )

    try:
        score = silhouette_score(
            distance_matrix,
            labels,
            metric="precomputed"
        )

    except Exception:
        score = np.nan

    return (
        labels,
        score,
        cluster_sizes
    )


# ------------------------------------------------------------
# Run clustering for each video × modality
# ------------------------------------------------------------
clustering_summary_rows = []
clustering_label_rows = []
skipped_cluster_rows = []

cluster_objects_by_video = {}

for (
    modality,
    video_name
), behavior_matrix in behavior_by_video_clean.items():

    subjects = (
        behavior_matrix.columns
        .astype(str)
        .tolist()
    )

    if len(subjects) < (
        max(CANDIDATE_CLUSTER_COUNTS) + 1
    ):

        skipped_cluster_rows.append({
            "Modality": modality,
            "Video": video_name,
            "N_Subjects": len(subjects),
            "Reason": "Too few participants for candidate solutions"
        })

        print(
            f"Skipped clustering for {modality} | "
            f"{video_name}: only {len(subjects)} participants."
        )

        continue

    (
        correlation_matrix,
        distance_matrix
    ) = build_correlation_and_distance_matrices(
        behavior_matrix
    )

    condensed_distance = squareform(
        distance_matrix,
        checks=False
    )

    linkage_matrix = linkage(
        condensed_distance,
        method="average",
        optimal_ordering=True
    )

    solution_results = {}

    for n_clusters in CANDIDATE_CLUSTER_COUNTS:

        (
            labels,
            silhouette,
            sizes
        ) = evaluate_cluster_solution(
            distance_matrix=distance_matrix,
            linkage_matrix=linkage_matrix,
            n_clusters=n_clusters,
            minimum_cluster_size=MIN_CLUSTER_SIZE
        )

        solution_results[
            n_clusters
        ] = {
            "labels": labels,
            "silhouette": silhouette,
            "sizes": sizes
        }

    valid_solutions = {
        k: result
        for k, result in solution_results.items()
        if np.isfinite(
            result["silhouette"]
        )
    }

    if len(valid_solutions) == 0:

        selected_k = 2
        selected_result = solution_results[
            selected_k
        ]

        selection_note = (
            "No candidate met the minimum-size criterion; "
            "the 2-cluster solution was retained descriptively."
        )

    else:
        selected_k = max(
            valid_solutions,
            key=lambda k: valid_solutions[
                k
            ]["silhouette"]
        )

        selected_result = valid_solutions[
            selected_k
        ]

        selection_note = (
            "Selected by maximum valid silhouette score."
        )

    selected_labels = selected_result[
        "labels"
    ]

    selected_silhouette = selected_result[
        "silhouette"
    ]

    silhouette_2 = solution_results[
        2
    ]["silhouette"]

    silhouette_3 = solution_results[
        3
    ]["silhouette"]

    base_video = remove_modality_suffix(
        video_name
    )

    trigger = extract_trigger(
        video_name
    )

    clustering_summary_rows.append({
        "Modality": modality,
        "Video": video_name,
        "BaseVideo": base_video,
        "Trigger": trigger,
        "N_Subjects": len(subjects),
        "Silhouette_K2": silhouette_2,
        "Silhouette_K3": silhouette_3,
        "Selected_K": selected_k,
        "Selected_Silhouette": selected_silhouette,
        "Cluster_Sizes": tuple(
            pd.Series(
                selected_labels
            )
            .value_counts()
            .sort_index()
            .tolist()
        ),
        "Selection_Note": selection_note
    })

    for subject, label in zip(
        subjects,
        selected_labels
    ):
        clustering_label_rows.append({
            "Subject": subject,
            "Modality": modality,
            "Video": video_name,
            "BaseVideo": base_video,
            "Trigger": trigger,
            "Raw_Cluster": int(label),
            "Selected_K": int(selected_k),
            "Selected_Silhouette": selected_silhouette
        })

    cluster_objects_by_video[
        (
            modality,
            video_name
        )
    ] = {
        "subjects": subjects,
        "behavior_matrix": behavior_matrix,
        "correlation_matrix": correlation_matrix,
        "distance_matrix": distance_matrix,
        "linkage_matrix": linkage_matrix,
        "labels": selected_labels,
        "selected_k": selected_k,
        "selected_silhouette": selected_silhouette
    }

    # --------------------------------------------------------
    # Save correlation and distance matrices
    # --------------------------------------------------------
    correlation_df = pd.DataFrame(
        correlation_matrix,
        index=subjects,
        columns=subjects
    )

    distance_df = pd.DataFrame(
        distance_matrix,
        index=subjects,
        columns=subjects
    )

    matrix_prefix = (
        f"{safe_filename(modality)}__"
        f"{safe_filename(video_name)}"
    )

    correlation_df.to_csv(
        CELL2_MATRIX_DIR
        / f"{matrix_prefix}__observer_correlation_matrix.csv",
        index=True,
        index_label="Subject"
    )

    distance_df.to_csv(
        CELL2_MATRIX_DIR
        / f"{matrix_prefix}__observer_distance_matrix.csv",
        index=True,
        index_label="Subject"
    )

    linkage_df = pd.DataFrame(
        linkage_matrix,
        columns=[
            "Cluster_1",
            "Cluster_2",
            "Distance",
            "N_Observations"
        ]
    )

    linkage_df.to_csv(
        CELL2_MATRIX_DIR
        / f"{matrix_prefix}__hierarchical_linkage_matrix.csv",
        index=False
    )


# ------------------------------------------------------------
# Final Cell 2 outputs
# ------------------------------------------------------------
clustering_summary_df = pd.DataFrame(
    clustering_summary_rows
)

hierarchical_cluster_labels_raw_df = pd.DataFrame(
    clustering_label_rows
)

skipped_clustering_videos_df = pd.DataFrame(
    skipped_cluster_rows
)

if clustering_summary_df.empty:
    raise RuntimeError(
        "No hierarchical-clustering analyses were completed."
    )

clustering_summary_df = (
    clustering_summary_df
    .sort_values(
        "Selected_Silhouette",
        ascending=False,
        na_position="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Save Cell 2 tables
# ------------------------------------------------------------
save_dataframe(
    clustering_summary_df,
    CELL2_TABLE_DIR
    / "per_video_hierarchical_clustering_summary.csv"
)

save_dataframe(
    hierarchical_cluster_labels_raw_df,
    CELL2_TABLE_DIR
    / "raw_hierarchical_cluster_labels.csv"
)

if not skipped_clustering_videos_df.empty:
    save_dataframe(
        skipped_clustering_videos_df,
        CELL2_TABLE_DIR
        / "skipped_clustering_videos.csv"
    )


# ------------------------------------------------------------
# Print and save summary
# ------------------------------------------------------------
cluster_count_table = (
    clustering_summary_df[
        "Selected_K"
    ]
    .value_counts()
    .rename_axis(
        "Selected_K"
    )
    .reset_index(
        name="N_Videos"
    )
)

save_dataframe(
    cluster_count_table,
    CELL2_TABLE_DIR
    / "selected_cluster_count_frequencies.csv"
)

summary_lines = [
    "=" * 90,
    "PER-VIDEO HIERARCHICAL CLUSTERING RESULTS",
    "=" * 90,
    "",
    (
        "Completed clustering for "
        f"{len(clustering_summary_df)} "
        "video × modality datasets."
    ),
    "",
    (
        f"Videos selecting two clusters: "
        f"{int((clustering_summary_df['Selected_K'] == 2).sum())}"
    ),
    (
        f"Videos selecting three clusters: "
        f"{int((clustering_summary_df['Selected_K'] == 3).sum())}"
    ),
    (
        "Mean selected silhouette score: "
        f"{clustering_summary_df['Selected_Silhouette'].mean():.4f}"
    ),
    (
        "Median selected silhouette score: "
        f"{clustering_summary_df['Selected_Silhouette'].median():.4f}"
    )
]

summary_text = "\n".join(
    summary_lines
)

print(
    "\n" + summary_text
)

with open(
    CELL2_DIR
    / "hierarchical_clustering_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(
        summary_text
    )


print("\nSelected cluster count frequencies:")

display(
    cluster_count_table
)

print("\nPer-video clustering summary:")

display(
    clustering_summary_df[
        [
            "Modality",
            "Video",
            "N_Subjects",
            "Silhouette_K2",
            "Silhouette_K3",
            "Selected_K",
            "Selected_Silhouette",
            "Cluster_Sizes"
        ]
    ].round(4)
)


# ------------------------------------------------------------
# Plot silhouette scores for all videos
# ------------------------------------------------------------
silhouette_plot_df = (
    clustering_summary_df
    .sort_values(
        "Selected_Silhouette",
        ascending=False,
        na_position="last"
    )
    .reset_index(drop=True)
)

x_positions = np.arange(
    len(silhouette_plot_df)
)

fig, ax = plt.subplots(
    figsize=(
        max(
            12,
            len(silhouette_plot_df) * 0.38
        ),
        6
    )
)

ax.plot(
    x_positions,
    silhouette_plot_df[
        "Silhouette_K2"
    ],
    marker="o",
    label="2-cluster solution"
)

ax.plot(
    x_positions,
    silhouette_plot_df[
        "Silhouette_K3"
    ],
    marker="o",
    label="3-cluster solution"
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    (
        silhouette_plot_df[
            "Modality"
        ].str[0]
        + ":"
        + silhouette_plot_df[
            "BaseVideo"
        ].astype(str)
    ),
    rotation=90
)

ax.set_xlabel(
    "Video × modality"
)

ax.set_ylabel(
    "Silhouette score"
)

ax.set_title(
    "Two- versus three-cluster solutions across videos"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

save_figure(
    fig,
    CELL2_FIG_DIR
    / "silhouette_scores_across_videos.png",
    dpi=FIGURE_DPI
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Plot strongest cluster solutions
# ------------------------------------------------------------
videos_to_plot = clustering_summary_df.head(
    N_CLUSTER_VIDEOS_TO_PLOT
)

for _, row in videos_to_plot.iterrows():

    key = (
        row["Modality"],
        row["Video"]
    )

    cluster_object = cluster_objects_by_video[
        key
    ]

    subjects = cluster_object[
        "subjects"
    ]

    correlation_matrix = cluster_object[
        "correlation_matrix"
    ]

    linkage_matrix = cluster_object[
        "linkage_matrix"
    ]

    labels = cluster_object[
        "labels"
    ]

    figure_prefix = (
        f"{safe_filename(row['Modality'])}__"
        f"{safe_filename(row['Video'])}"
    )

    # --------------------------------------------------------
    # Dendrogram
    # --------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(12, 5)
    )

    dendrogram(
        linkage_matrix,
        labels=subjects,
        leaf_rotation=90,
        leaf_font_size=8,
        ax=ax
    )

    ax.set_ylabel(
        "Correlation distance"
    )

    ax.set_xlabel(
        "Participant"
    )

    silhouette_text = (
        f"{row['Selected_Silhouette']:.3f}"
        if np.isfinite(
            row["Selected_Silhouette"]
        )
        else "NA"
    )

    ax.set_title(
        f"{row['Modality']} | {row['Video']}\n"
        f"Hierarchical clustering: selected "
        f"k={row['Selected_K']}, "
        f"silhouette={silhouette_text}"
    )

    fig.tight_layout()

    save_figure(
        fig,
        CELL2_FIG_DIR
        / f"{figure_prefix}__dendrogram.png",
        dpi=FIGURE_DPI
    )

    plt.show()
    plt.close(fig)

    # --------------------------------------------------------
    # Ordered observer-correlation matrix
    # --------------------------------------------------------
    dendrogram_order = leaves_list(
        linkage_matrix
    )

    ordering_table = pd.DataFrame({
        "Original_Index": np.arange(
            len(subjects)
        ),
        "Dendrogram_Order": np.argsort(
            dendrogram_order
        ),
        "Cluster": labels
    })

    ordered_indices = (
        ordering_table
        .sort_values(
            [
                "Cluster",
                "Dendrogram_Order"
            ]
        )[
            "Original_Index"
        ]
        .to_numpy(dtype=int)
    )

    ordered_corr = correlation_matrix[
        np.ix_(
            ordered_indices,
            ordered_indices
        )
    ]

    ordered_subjects = [
        subjects[index]
        for index in ordered_indices
    ]

    ordered_labels = labels[
        ordered_indices
    ]

    ordered_matrix_df = pd.DataFrame(
        ordered_corr,
        index=ordered_subjects,
        columns=ordered_subjects
    )

    ordered_matrix_df.to_csv(
        CELL2_MATRIX_DIR
        / f"{figure_prefix}__ordered_observer_correlation_matrix.csv",
        index=True,
        index_label="Subject"
    )

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

    image = ax.imshow(
        ordered_corr,
        aspect="auto",
        vmin=-1,
        vmax=1,
        cmap="coolwarm"
    )

    fig.colorbar(
        image,
        ax=ax,
        label=(
            "Observer–observer behavioral correlation"
        )
    )

    ax.set_xticks(
        np.arange(
            len(ordered_subjects)
        )
    )

    ax.set_xticklabels(
        ordered_subjects,
        rotation=90,
        fontsize=7
    )

    ax.set_yticks(
        np.arange(
            len(ordered_subjects)
        )
    )

    ax.set_yticklabels(
        ordered_subjects,
        fontsize=7
    )

    cluster_change_locations = (
        np.where(
            np.diff(
                ordered_labels
            ) != 0
        )[0]
        + 0.5
    )

    for boundary in cluster_change_locations:
        ax.axhline(
            boundary,
            linewidth=2
        )

        ax.axvline(
            boundary,
            linewidth=2
        )

    ax.set_title(
        f"{row['Modality']} | {row['Video']}\n"
        "Observer correlation matrix ordered by cluster"
    )

    fig.tight_layout()

    save_figure(
        fig,
        CELL2_FIG_DIR
        / f"{figure_prefix}__ordered_correlation_matrix.png",
        dpi=FIGURE_DPI
    )

    plt.show()
    plt.close(fig)


print("\nCell 2 complete.")
print(f"Outputs saved in:\n{CELL2_DIR}")
print("\nCreated in memory:")
print("  clustering_summary_df")
print("  hierarchical_cluster_labels_raw_df")
print("  cluster_objects_by_video")

In [ ]:
# ============================================================
# 
# BEHAVIORAL SUBGROUP LABELING
# Saves all tables, summaries, and figures
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
CELL3_DIR = OUTPUT_DIR / "03_behavioral_subgroup_labeling"
CELL3_FIG_DIR = CELL3_DIR / "figures"
CELL3_TABLE_DIR = CELL3_DIR / "tables"

for folder in [
    CELL3_DIR,
    CELL3_FIG_DIR,
    CELL3_TABLE_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Check prior outputs
# ------------------------------------------------------------
required_objects = [
    "cluster_objects_by_video",
    "hierarchical_cluster_labels_raw_df",
    "pc_alignment_df"
]

for required_object in required_objects:

    if required_object not in globals():
        raise ValueError(
            f"{required_object} was not found. "
            "Run Stage 1 Cells 1 and 2 first."
        )


N_SUBGROUP_VIDEOS_TO_PLOT = 8
FIGURE_DPI = 300


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------
def leave_one_out_mean_correlation(
    behavior_matrix,
    subject,
    comparison_subjects
):
    """
    Correlate a participant with the mean trajectory of the
    remaining participants in the comparison group.
    """
    comparison_subjects = [
        comparison_subject
        for comparison_subject in comparison_subjects
        if comparison_subject != subject
    ]

    if len(comparison_subjects) < 1:
        return np.nan

    target = behavior_matrix[
        subject
    ].to_numpy(dtype=float)

    comparison_mean = (
        behavior_matrix[
            comparison_subjects
        ]
        .mean(axis=1)
        .to_numpy(dtype=float)
    )

    r_value, _ = safe_pearsonr(
        target,
        comparison_mean
    )

    return r_value


# ------------------------------------------------------------
# Create dominant and alternative subgroup labels
# ------------------------------------------------------------
subgroup_label_rows = []
cluster_summary_rows = []
cluster_consensus_tables = []

for key, cluster_object in cluster_objects_by_video.items():

    modality, video_name = key

    behavior_matrix = cluster_object[
        "behavior_matrix"
    ].copy()

    subjects = cluster_object[
        "subjects"
    ]

    raw_labels = np.asarray(
        cluster_object[
            "labels"
        ],
        dtype=int
    )

    subject_label_df = pd.DataFrame({
        "Subject": subjects,
        "Raw_Cluster": raw_labels
    })

    cluster_sizes = (
        subject_label_df[
            "Raw_Cluster"
        ]
        .value_counts()
        .sort_values(
            ascending=False
        )
    )

    # Resolve equal-size clusters using mean within-cluster correlation
    cluster_tie_break_scores = {}

    for raw_cluster in cluster_sizes.index:

        cluster_subjects = (
            subject_label_df.loc[
                subject_label_df[
                    "Raw_Cluster"
                ] == raw_cluster,
                "Subject"
            ]
            .tolist()
        )

        if len(cluster_subjects) < 2:
            cluster_tie_break_scores[
                raw_cluster
            ] = -np.inf

            continue

        cluster_corr = (
            behavior_matrix[
                cluster_subjects
            ]
            .corr()
            .to_numpy(dtype=float)
        )

        upper_triangle = cluster_corr[
            np.triu_indices_from(
                cluster_corr,
                k=1
            )
        ]

        cluster_tie_break_scores[
            raw_cluster
        ] = np.nanmean(
            upper_triangle
        )

    ordered_clusters = sorted(
        cluster_sizes.index,
        key=lambda cluster: (
            cluster_sizes.loc[
                cluster
            ],
            cluster_tie_break_scores.get(
                cluster,
                -np.inf
            )
        ),
        reverse=True
    )

    interpretable_names = {
        raw_cluster: (
            "Dominant_Consensus"
            if rank == 0
            else f"Alternative_{rank}"
        )
        for rank, raw_cluster
        in enumerate(
            ordered_clusters
        )
    }

    full_group_mean = (
        behavior_matrix
        .mean(axis=1)
        .to_numpy(dtype=float)
    )

    # --------------------------------------------------------
    # Participant-level subgroup metrics
    # --------------------------------------------------------
    for subject, raw_cluster in zip(
        subjects,
        raw_labels
    ):

        cluster_name = interpretable_names[
            raw_cluster
        ]

        cluster_subjects = (
            subject_label_df.loc[
                subject_label_df[
                    "Raw_Cluster"
                ] == raw_cluster,
                "Subject"
            ]
            .tolist()
        )

        target = behavior_matrix[
            subject
        ].to_numpy(dtype=float)

        cluster_mean = (
            behavior_matrix[
                cluster_subjects
            ]
            .mean(axis=1)
            .to_numpy(dtype=float)
        )

        r_group_mean, _ = safe_pearsonr(
            target,
            full_group_mean
        )

        (
            r_cluster_mean_including_self,
            _
        ) = safe_pearsonr(
            target,
            cluster_mean
        )

        r_cluster_mean_loo = (
            leave_one_out_mean_correlation(
                behavior_matrix=behavior_matrix,
                subject=subject,
                comparison_subjects=cluster_subjects
            )
        )

        subgroup_label_rows.append({
            "Subject": str(subject),
            "Modality": modality,
            "Video": video_name,
            "BaseVideo": remove_modality_suffix(
                video_name
            ),
            "Trigger": extract_trigger(
                video_name
            ),
            "Raw_Cluster": int(
                raw_cluster
            ),
            "Behavioral_Subgroup": cluster_name,
            "Is_Dominant_Consensus": (
                cluster_name
                == "Dominant_Consensus"
            ),
            "Is_Alternative_Consensus": (
                cluster_name
                != "Dominant_Consensus"
            ),
            "Cluster_Size": len(
                cluster_subjects
            ),
            "r_Full_Group_Mean": r_group_mean,
            "r_Cluster_Mean_Including_Self": (
                r_cluster_mean_including_self
            ),
            "r_Cluster_Mean_LOO": (
                r_cluster_mean_loo
            )
        })

    # --------------------------------------------------------
    # Cluster-level metrics and consensus trajectories
    # --------------------------------------------------------
    for raw_cluster in ordered_clusters:

        cluster_name = interpretable_names[
            raw_cluster
        ]

        cluster_subjects = (
            subject_label_df.loc[
                subject_label_df[
                    "Raw_Cluster"
                ] == raw_cluster,
                "Subject"
            ]
            .tolist()
        )

        cluster_matrix = behavior_matrix[
            cluster_subjects
        ]

        consensus = (
            cluster_matrix
            .mean(axis=1)
            .to_numpy(dtype=float)
        )

        cluster_corr = (
            cluster_matrix
            .corr()
            .to_numpy(dtype=float)
        )

        if len(cluster_subjects) >= 2:

            upper_triangle = cluster_corr[
                np.triu_indices_from(
                    cluster_corr,
                    k=1
                )
            ]

            mean_within_cluster_r = np.nanmean(
                upper_triangle
            )

        else:
            mean_within_cluster_r = np.nan

        (
            r_cluster_to_group,
            _
        ) = safe_pearsonr(
            consensus,
            full_group_mean
        )

        cluster_summary_rows.append({
            "Modality": modality,
            "Video": video_name,
            "BaseVideo": remove_modality_suffix(
                video_name
            ),
            "Trigger": extract_trigger(
                video_name
            ),
            "Raw_Cluster": int(
                raw_cluster
            ),
            "Behavioral_Subgroup": cluster_name,
            "N_Subjects": len(
                cluster_subjects
            ),
            "Mean_Within_Cluster_r": (
                mean_within_cluster_r
            ),
            "r_Cluster_Consensus_to_Group_Mean": (
                r_cluster_to_group
            )
        })

        cluster_consensus_tables.append(
            pd.DataFrame({
                "Timepoint": np.arange(
                    len(consensus)
                ),
                "Consensus_Trajectory": consensus,
                "Modality": modality,
                "Video": video_name,
                "BaseVideo": remove_modality_suffix(
                    video_name
                ),
                "Trigger": extract_trigger(
                    video_name
                ),
                "Raw_Cluster": int(
                    raw_cluster
                ),
                "Behavioral_Subgroup": cluster_name,
                "N_Subjects": len(
                    cluster_subjects
                )
            })
        )


# ------------------------------------------------------------
# Final Cell 3 outputs
# ------------------------------------------------------------
behavioral_subgroup_df = pd.DataFrame(
    subgroup_label_rows
)

behavioral_cluster_summary_df = pd.DataFrame(
    cluster_summary_rows
)

behavioral_cluster_consensus_df = (
    pd.concat(
        cluster_consensus_tables,
        ignore_index=True
    )
    if len(cluster_consensus_tables) > 0
    else pd.DataFrame()
)


# Merge PC alignment into subgroup table
pc_merge_columns = [
    "Subject",
    "Modality",
    "Video",
    "r_PC1",
    "r_PC2",
    "r_PC3",
    "abs_r_PC1",
    "abs_r_PC2",
    "abs_r_PC3",
    "Best_PC",
    "Best_PC_r",
    "Best_PC_abs_r",
    "Best_PC_margin"
]

behavioral_subgroup_df = (
    behavioral_subgroup_df
    .merge(
        pc_alignment_df[
            pc_merge_columns
        ],
        on=[
            "Subject",
            "Modality",
            "Video"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Save Cell 3 tables
# ------------------------------------------------------------
save_dataframe(
    behavioral_subgroup_df,
    CELL3_TABLE_DIR
    / "participant_video_behavioral_subgroups.csv"
)

save_dataframe(
    behavioral_cluster_summary_df,
    CELL3_TABLE_DIR
    / "behavioral_cluster_summary.csv"
)

save_dataframe(
    behavioral_cluster_consensus_df,
    CELL3_TABLE_DIR
    / "behavioral_cluster_consensus_trajectories.csv"
)


subgroup_frequency_df = (
    behavioral_subgroup_df[
        "Behavioral_Subgroup"
    ]
    .value_counts()
    .rename_axis(
        "Behavioral_Subgroup"
    )
    .reset_index(
        name="N_Subject_Video_Observations"
    )
)

save_dataframe(
    subgroup_frequency_df,
    CELL3_TABLE_DIR
    / "behavioral_subgroup_frequencies.csv"
)


# ------------------------------------------------------------
# Print and save summary
# ------------------------------------------------------------
n_dominant = int(
    behavioral_subgroup_df[
        "Is_Dominant_Consensus"
    ].sum()
)

n_alternative = int(
    behavioral_subgroup_df[
        "Is_Alternative_Consensus"
    ].sum()
)

summary_lines = [
    "=" * 90,
    "BEHAVIORAL SUBGROUP LABELING RESULTS",
    "=" * 90,
    "",
    (
        "Total subject × video observations: "
        f"{len(behavioral_subgroup_df)}"
    ),
    (
        "Dominant-consensus observations: "
        f"{n_dominant}"
    ),
    (
        "Alternative-consensus observations: "
        f"{n_alternative}"
    ),
    (
        "Proportion alternative-consensus: "
        f"{n_alternative / len(behavioral_subgroup_df):.4f}"
    ),
    "",
    (
        "Mean leave-one-out within-cluster correlation: "
        f"{behavioral_subgroup_df['r_Cluster_Mean_LOO'].mean():.4f}"
    ),
    (
        "Mean correlation with full group trajectory: "
        f"{behavioral_subgroup_df['r_Full_Group_Mean'].mean():.4f}"
    )
]

summary_text = "\n".join(
    summary_lines
)

print(
    "\n" + summary_text
)

with open(
    CELL3_DIR
    / "behavioral_subgroup_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(
        summary_text
    )


print("\nSubgroup frequency:")

display(
    subgroup_frequency_df
)

print("\nCluster-level subgroup summary:")

display(
    behavioral_cluster_summary_df
    .sort_values(
        [
            "Modality",
            "Video",
            "Behavioral_Subgroup"
        ]
    )
    .round(4)
)

print("\nParticipant-level subgroup-label preview:")

display(
    behavioral_subgroup_df[
        [
            "Subject",
            "Modality",
            "Video",
            "Behavioral_Subgroup",
            "Cluster_Size",
            "r_Full_Group_Mean",
            "r_Cluster_Mean_LOO",
            "r_PC1",
            "r_PC2",
            "r_PC3",
            "Best_PC"
        ]
    ]
    .head(30)
    .round(4)
)


# ------------------------------------------------------------
# Plot overall subgroup frequencies
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    subgroup_frequency_df[
        "Behavioral_Subgroup"
    ],
    subgroup_frequency_df[
        "N_Subject_Video_Observations"
    ]
)

ax.set_xlabel(
    "Behavioral subgroup"
)

ax.set_ylabel(
    "Number of subject × video observations"
)

ax.set_title(
    "Behavioral subgroup frequencies"
)

ax.tick_params(
    axis="x",
    rotation=30
)

fig.tight_layout()

save_figure(
    fig,
    CELL3_FIG_DIR
    / "behavioral_subgroup_frequencies.png",
    dpi=FIGURE_DPI
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Plot cluster consensus trajectories
# ------------------------------------------------------------
plot_video_order = (
    clustering_summary_df
    .sort_values(
        "Selected_Silhouette",
        ascending=False,
        na_position="last"
    )
    .head(
        N_SUBGROUP_VIDEOS_TO_PLOT
    )
)

for _, row in plot_video_order.iterrows():

    modality = row[
        "Modality"
    ]

    video_name = row[
        "Video"
    ]

    trajectory_subset = (
        behavioral_cluster_consensus_df[
            (
                behavioral_cluster_consensus_df[
                    "Modality"
                ] == modality
            )
            & (
                behavioral_cluster_consensus_df[
                    "Video"
                ] == video_name
            )
        ]
    )

    behavior_matrix = behavior_by_video_clean[
        (
            modality,
            video_name
        )
    ]

    subject_subset = behavioral_subgroup_df[
        (
            behavioral_subgroup_df[
                "Modality"
            ] == modality
        )
        & (
            behavioral_subgroup_df[
                "Video"
            ] == video_name
        )
    ]

    fig, ax = plt.subplots(
        figsize=(12, 5.5)
    )

    subgroup_names = (
        subject_subset[
            "Behavioral_Subgroup"
        ]
        .drop_duplicates()
        .tolist()
    )

    for subgroup_name in subgroup_names:

        subgroup_subjects = (
            subject_subset.loc[
                subject_subset[
                    "Behavioral_Subgroup"
                ] == subgroup_name,
                "Subject"
            ]
            .tolist()
        )

        for subject in subgroup_subjects:
            ax.plot(
                behavior_matrix[
                    subject
                ].to_numpy(),
                alpha=0.12,
                linewidth=0.8
            )

    for subgroup_name in subgroup_names:

        subgroup_trajectory = (
            trajectory_subset[
                trajectory_subset[
                    "Behavioral_Subgroup"
                ] == subgroup_name
            ]
            .sort_values(
                "Timepoint"
            )
        )

        if subgroup_trajectory.empty:
            continue

        n_subjects = int(
            subgroup_trajectory[
                "N_Subjects"
            ].iloc[0]
        )

        ax.plot(
            subgroup_trajectory[
                "Timepoint"
            ],
            subgroup_trajectory[
                "Consensus_Trajectory"
            ],
            linewidth=3,
            label=(
                f"{subgroup_name}, "
                f"n={n_subjects}"
            )
        )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "Timepoint"
    )

    ax.set_ylabel(
        "Mean standardized behavioral rating"
    )

    ax.set_title(
        f"{modality} | {video_name}\n"
        "Dominant and alternative behavioral "
        "consensus trajectories"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_filename = (
        f"{safe_filename(modality)}__"
        f"{safe_filename(video_name)}"
        f"__behavioral_subgroup_consensus.png"
    )

    save_figure(
        fig,
        CELL3_FIG_DIR
        / figure_filename,
        dpi=FIGURE_DPI
    )

    plt.show()
    plt.close(fig)


print("\nCell 3 complete.")
print(f"Outputs saved in:\n{CELL3_DIR}")
print("\nCreated in memory:")
print("  behavioral_subgroup_df")
print("  behavioral_cluster_summary_df")
print("  behavioral_cluster_consensus_df")

In [ ]:
# ============================================================
# 
# WRONG-BUT-CONSISTENT OBSERVER IDENTIFICATION
# Saves all tables, summaries, and figures
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import pearsonr, spearmanr


# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
CELL4_DIR = OUTPUT_DIR / "04_wrong_but_consistent_observers"
CELL4_FIG_DIR = CELL4_DIR / "figures"
CELL4_TABLE_DIR = CELL4_DIR / "tables"

for folder in [
    CELL4_DIR,
    CELL4_FIG_DIR,
    CELL4_TABLE_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Check Cell 3 output
# ------------------------------------------------------------
if "behavioral_subgroup_df" not in globals():
    raise ValueError(
        "behavioral_subgroup_df was not found. "
        "Run Stage 1 Cell 3 first."
    )


# ------------------------------------------------------------
# Identification criteria
# ------------------------------------------------------------
MIN_ALTERNATIVE_CLUSTER_SIZE = 3

MIN_WITHIN_CLUSTER_LOO_R = 0.50

MAX_ABS_PC1_ALIGNMENT = 0.30

FIGURE_DPI = 300


# ------------------------------------------------------------
# Classify wrong-but-consistent observations
# ------------------------------------------------------------
wrong_consistent_df = behavioral_subgroup_df.copy()

wrong_consistent_df[
    "Meets_Alternative_Cluster_Criterion"
] = wrong_consistent_df[
    "Is_Alternative_Consensus"
].fillna(False)

wrong_consistent_df[
    "Meets_Cluster_Size_Criterion"
] = (
    wrong_consistent_df[
        "Cluster_Size"
    ] >= MIN_ALTERNATIVE_CLUSTER_SIZE
)

wrong_consistent_df[
    "Meets_Within_Cluster_Consistency_Criterion"
] = (
    wrong_consistent_df[
        "r_Cluster_Mean_LOO"
    ] >= MIN_WITHIN_CLUSTER_LOO_R
)

wrong_consistent_df[
    "Meets_Weak_PC1_Criterion"
] = (
    wrong_consistent_df[
        "abs_r_PC1"
    ] <= MAX_ABS_PC1_ALIGNMENT
)

wrong_consistent_df[
    "Wrong_But_Consistent"
] = (
    wrong_consistent_df[
        "Meets_Alternative_Cluster_Criterion"
    ]
    & wrong_consistent_df[
        "Meets_Cluster_Size_Criterion"
    ]
    & wrong_consistent_df[
        "Meets_Within_Cluster_Consistency_Criterion"
    ]
    & wrong_consistent_df[
        "Meets_Weak_PC1_Criterion"
    ]
)


# ------------------------------------------------------------
# Candidate-level table
# ------------------------------------------------------------
wrong_but_consistent_candidates_df = (
    wrong_consistent_df[
        wrong_consistent_df[
            "Wrong_But_Consistent"
        ]
    ]
    .copy()
    .sort_values(
        [
            "Modality",
            "Video",
            "r_Cluster_Mean_LOO"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Video-level summary
# ------------------------------------------------------------
wrong_consistent_video_summary_df = (
    wrong_consistent_df
    .groupby(
        [
            "Modality",
            "Video",
            "BaseVideo",
            "Trigger"
        ],
        as_index=False
    )
    .agg(
        N_Subjects=(
            "Subject",
            "nunique"
        ),
        N_Alternative_Observers=(
            "Is_Alternative_Consensus",
            "sum"
        ),
        N_Wrong_But_Consistent=(
            "Wrong_But_Consistent",
            "sum"
        ),
        Mean_Abs_PC1_Alignment=(
            "abs_r_PC1",
            "mean"
        ),
        Mean_Within_Cluster_LOO_r=(
            "r_Cluster_Mean_LOO",
            "mean"
        )
    )
)

wrong_consistent_video_summary_df[
    "Proportion_Wrong_But_Consistent"
] = (
    wrong_consistent_video_summary_df[
        "N_Wrong_But_Consistent"
    ]
    / wrong_consistent_video_summary_df[
        "N_Subjects"
    ]
)


# ------------------------------------------------------------
# Subject-level recurrence summary
# ------------------------------------------------------------
wrong_consistent_subject_summary_df = (
    wrong_consistent_df
    .groupby(
        "Subject",
        as_index=False
    )
    .agg(
        N_Video_Modality_Observations=(
            "Video",
            "size"
        ),
        N_Alternative_Assignments=(
            "Is_Alternative_Consensus",
            "sum"
        ),
        N_Wrong_But_Consistent=(
            "Wrong_But_Consistent",
            "sum"
        ),
        Mean_Abs_PC1_Alignment=(
            "abs_r_PC1",
            "mean"
        ),
        Mean_Within_Cluster_LOO_r=(
            "r_Cluster_Mean_LOO",
            "mean"
        )
    )
)

wrong_consistent_subject_summary_df[
    "Wrong_Consistent_Rate"
] = (
    wrong_consistent_subject_summary_df[
        "N_Wrong_But_Consistent"
    ]
    / wrong_consistent_subject_summary_df[
        "N_Video_Modality_Observations"
    ]
)

wrong_consistent_subject_summary_df = (
    wrong_consistent_subject_summary_df
    .sort_values(
        [
            "N_Wrong_But_Consistent",
            "Wrong_Consistent_Rate"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Correlation between PC1 alignment and cluster consistency
# ------------------------------------------------------------
association_data = (
    wrong_consistent_df[
        [
            "abs_r_PC1",
            "r_Cluster_Mean_LOO"
        ]
    ]
    .dropna()
)

if (
    len(association_data) >= 5
    and association_data[
        "abs_r_PC1"
    ].std(ddof=1) > 0
    and association_data[
        "r_Cluster_Mean_LOO"
    ].std(ddof=1) > 0
):

    pearson_r, pearson_p = pearsonr(
        association_data[
            "abs_r_PC1"
        ],
        association_data[
            "r_Cluster_Mean_LOO"
        ]
    )

    (
        spearman_rho,
        spearman_p
    ) = spearmanr(
        association_data[
            "abs_r_PC1"
        ],
        association_data[
            "r_Cluster_Mean_LOO"
        ]
    )

else:
    pearson_r = np.nan
    pearson_p = np.nan
    spearman_rho = np.nan
    spearman_p = np.nan


association_results_df = pd.DataFrame({
    "Test": [
        "Pearson",
        "Spearman"
    ],
    "Statistic": [
        pearson_r,
        spearman_rho
    ],
    "P_Value": [
        pearson_p,
        spearman_p
    ],
    "N_Observations": [
        len(association_data),
        len(association_data)
    ]
})


# ------------------------------------------------------------
# Save Cell 4 tables
# ------------------------------------------------------------
save_dataframe(
    wrong_consistent_df,
    CELL4_TABLE_DIR
    / "all_subject_video_wrong_consistent_classifications.csv"
)

save_dataframe(
    wrong_but_consistent_candidates_df,
    CELL4_TABLE_DIR
    / "wrong_but_consistent_candidates.csv"
)

save_dataframe(
    wrong_consistent_video_summary_df,
    CELL4_TABLE_DIR
    / "wrong_but_consistent_video_summary.csv"
)

save_dataframe(
    wrong_consistent_subject_summary_df,
    CELL4_TABLE_DIR
    / "wrong_but_consistent_subject_summary.csv"
)

save_dataframe(
    association_results_df,
    CELL4_TABLE_DIR
    / "pc1_alignment_within_cluster_association.csv"
)


# ------------------------------------------------------------
# Print and save summary
# ------------------------------------------------------------
n_candidates = len(
    wrong_but_consistent_candidates_df
)

n_unique_candidates = (
    wrong_but_consistent_candidates_df[
        "Subject"
    ].nunique()
)

n_candidate_videos = (
    wrong_consistent_video_summary_df.loc[
        wrong_consistent_video_summary_df[
            "N_Wrong_But_Consistent"
        ] > 0
    ]
    .shape[0]
)

summary_lines = [
    "=" * 90,
    "WRONG-BUT-CONSISTENT OBSERVER IDENTIFICATION",
    "=" * 90,
    "",
    "Operational criteria:",
    "1. Alternative behavioral-cluster membership",
    (
        "2. Alternative-cluster size >= "
        f"{MIN_ALTERNATIVE_CLUSTER_SIZE}"
    ),
    (
        "3. Leave-one-out within-cluster correlation >= "
        f"{MIN_WITHIN_CLUSTER_LOO_R:.2f}"
    ),
    (
        "4. Absolute PC1 alignment <= "
        f"{MAX_ABS_PC1_ALIGNMENT:.2f}"
    ),
    "",
    (
        "Wrong-but-consistent subject × video observations: "
        f"{n_candidates}"
    ),
    (
        "Unique participants meeting the criterion at least once: "
        f"{n_unique_candidates}"
    ),
    (
        "Video × modality datasets containing at least one candidate: "
        f"{n_candidate_videos}"
    ),
    "",
    "PC1 alignment versus within-cluster consistency:",
    (
        f"Pearson r = {pearson_r:.4f}, "
        f"p = {pearson_p:.6g}"
    ),
    (
        f"Spearman rho = {spearman_rho:.4f}, "
        f"p = {spearman_p:.6g}"
    )
]

summary_text = "\n".join(
    summary_lines
)

print(
    "\n" + summary_text
)

with open(
    CELL4_DIR
    / "wrong_but_consistent_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(
        summary_text
    )


print("\nWrong-but-consistent candidate observations:")

if wrong_but_consistent_candidates_df.empty:

    print(
        "No observations met all current thresholds. "
        "This is a valid result. Inspect the scatterplot before "
        "changing the thresholds."
    )

else:
    display(
        wrong_but_consistent_candidates_df[
            [
                "Subject",
                "Modality",
                "Video",
                "Behavioral_Subgroup",
                "Cluster_Size",
                "r_Cluster_Mean_LOO",
                "r_Full_Group_Mean",
                "r_PC1",
                "abs_r_PC1",
                "r_PC2",
                "r_PC3",
                "Best_PC"
            ]
        ].round(4)
    )


print("\nVideo-level identification summary:")

display(
    wrong_consistent_video_summary_df
    .sort_values(
        [
            "N_Wrong_But_Consistent",
            "Proportion_Wrong_But_Consistent"
        ],
        ascending=False
    )
    .round(4)
)


print("\nParticipants most frequently showing this pattern:")

display(
    wrong_consistent_subject_summary_df
    .head(20)
    .round(4)
)


# ------------------------------------------------------------
# Plot 1: consistency versus PC1 alignment
# ------------------------------------------------------------
plot_data = (
    wrong_consistent_df
    .dropna(
        subset=[
            "abs_r_PC1",
            "r_Cluster_Mean_LOO"
        ]
    )
    .copy()
)

dominant_data = plot_data[
    plot_data[
        "Is_Dominant_Consensus"
    ]
]

alternative_data = plot_data[
    plot_data[
        "Is_Alternative_Consensus"
    ]
]

candidate_data = plot_data[
    plot_data[
        "Wrong_But_Consistent"
    ]
]

fig, ax = plt.subplots(
    figsize=(9, 7)
)

ax.scatter(
    dominant_data[
        "abs_r_PC1"
    ],
    dominant_data[
        "r_Cluster_Mean_LOO"
    ],
    alpha=0.40,
    label="Dominant-consensus observers"
)

ax.scatter(
    alternative_data[
        "abs_r_PC1"
    ],
    alternative_data[
        "r_Cluster_Mean_LOO"
    ],
    alpha=0.55,
    label="Alternative-consensus observers"
)

if not candidate_data.empty:

    ax.scatter(
        candidate_data[
            "abs_r_PC1"
        ],
        candidate_data[
            "r_Cluster_Mean_LOO"
        ],
        s=110,
        marker="*",
        label="Wrong-but-consistent candidates"
    )

ax.axvline(
    MAX_ABS_PC1_ALIGNMENT,
    linestyle="--",
    linewidth=1.5,
    label=(
        "Weak PC1 threshold = "
        f"{MAX_ABS_PC1_ALIGNMENT:.2f}"
    )
)

ax.axhline(
    MIN_WITHIN_CLUSTER_LOO_R,
    linestyle="--",
    linewidth=1.5,
    label=(
        "Within-cluster threshold = "
        f"{MIN_WITHIN_CLUSTER_LOO_R:.2f}"
    )
)

ax.set_xlabel(
    "Absolute alignment with PC1"
)

ax.set_ylabel(
    "Leave-one-out correlation with own cluster consensus"
)

ax.set_title(
    "Identification of wrong-but-consistent observers\n"
    "Strong alternative-cluster agreement despite weak PC1 alignment"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

save_figure(
    fig,
    CELL4_FIG_DIR
    / "wrong_consistent_identification_scatterplot.png",
    dpi=FIGURE_DPI
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Plot 2: number of candidates by video
# ------------------------------------------------------------
video_plot_df = (
    wrong_consistent_video_summary_df
    .sort_values(
        "N_Wrong_But_Consistent",
        ascending=False
    )
    .copy()
)

video_plot_df[
    "Plot_Label"
] = (
    video_plot_df[
        "Modality"
    ].str[0]
    + ":"
    + video_plot_df[
        "BaseVideo"
    ].astype(str)
)

fig, ax = plt.subplots(
    figsize=(
        max(
            12,
            len(video_plot_df) * 0.38
        ),
        6
    )
)

ax.bar(
    np.arange(
        len(video_plot_df)
    ),
    video_plot_df[
        "N_Wrong_But_Consistent"
    ]
)

ax.set_xticks(
    np.arange(
        len(video_plot_df)
    )
)

ax.set_xticklabels(
    video_plot_df[
        "Plot_Label"
    ],
    rotation=90
)

ax.set_ylabel(
    "Number of wrong-but-consistent observers"
)

ax.set_xlabel(
    "Video × modality"
)

ax.set_title(
    "Wrong-but-consistent observers by video"
)

fig.tight_layout()

save_figure(
    fig,
    CELL4_FIG_DIR
    / "wrong_consistent_observers_by_video.png",
    dpi=FIGURE_DPI
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Plot 3: recurring candidates by participant
# ------------------------------------------------------------
subject_plot_df = (
    wrong_consistent_subject_summary_df[
        wrong_consistent_subject_summary_df[
            "N_Wrong_But_Consistent"
        ] > 0
    ]
    .copy()
)

if not subject_plot_df.empty:

    fig, ax = plt.subplots(
        figsize=(
            max(
                10,
                len(subject_plot_df) * 0.45
            ),
            6
        )
    )

    ax.bar(
        np.arange(
            len(subject_plot_df)
        ),
        subject_plot_df[
            "N_Wrong_But_Consistent"
        ]
    )

    ax.set_xticks(
        np.arange(
            len(subject_plot_df)
        )
    )

    ax.set_xticklabels(
        subject_plot_df[
            "Subject"
        ],
        rotation=90
    )

    ax.set_xlabel(
        "Participant"
    )

    ax.set_ylabel(
        "Number of wrong-but-consistent observations"
    )

    ax.set_title(
        "Recurrence of wrong-but-consistent behavior by participant"
    )

    fig.tight_layout()

    save_figure(
        fig,
        CELL4_FIG_DIR
        / "wrong_consistent_recurrence_by_participant.png",
        dpi=FIGURE_DPI
    )

    plt.show()
    plt.close(fig)


# ------------------------------------------------------------
# Plot 4: candidate trajectories against cluster consensuses
# ------------------------------------------------------------
candidate_video_keys = list(
    wrong_but_consistent_candidates_df[
        [
            "Modality",
            "Video"
        ]
    ]
    .drop_duplicates()
    .itertuples(
        index=False,
        name=None
    )
)

for (
    modality,
    video_name
) in candidate_video_keys:

    behavior_matrix = behavior_by_video_clean[
        (
            modality,
            video_name
        )
    ]

    subgroup_subset = wrong_consistent_df[
        (
            wrong_consistent_df[
                "Modality"
            ] == modality
        )
        & (
            wrong_consistent_df[
                "Video"
            ] == video_name
        )
    ]

    consensus_subset = (
        behavioral_cluster_consensus_df[
            (
                behavioral_cluster_consensus_df[
                    "Modality"
                ] == modality
            )
            & (
                behavioral_cluster_consensus_df[
                    "Video"
                ] == video_name
            )
        ]
    )

    candidates = (
        subgroup_subset.loc[
            subgroup_subset[
                "Wrong_But_Consistent"
            ],
            "Subject"
        ]
        .tolist()
    )

    fig, ax = plt.subplots(
        figsize=(12, 5.5)
    )

    for subgroup_name in (
        consensus_subset[
            "Behavioral_Subgroup"
        ]
        .drop_duplicates()
    ):

        trajectory = (
            consensus_subset[
                consensus_subset[
                    "Behavioral_Subgroup"
                ] == subgroup_name
            ]
            .sort_values(
                "Timepoint"
            )
        )

        ax.plot(
            trajectory[
                "Timepoint"
            ],
            trajectory[
                "Consensus_Trajectory"
            ],
            linewidth=3,
            label=(
                f"{subgroup_name} consensus"
            )
        )

    for subject in candidates:

        ax.plot(
            behavior_matrix[
                subject
            ].to_numpy(),
            linewidth=1.7,
            alpha=0.85,
            linestyle="--",
            label=f"Candidate {subject}"
        )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "Timepoint"
    )

    ax.set_ylabel(
        "Standardized behavioral trajectory"
    )

    ax.set_title(
        f"{modality} | {video_name}\n"
        "Wrong-but-consistent candidate trajectories"
    )

    ax.legend(
        frameon=False,
        bbox_to_anchor=(
            1.02,
            1
        ),
        loc="upper left"
    )

    fig.tight_layout()

    figure_filename = (
        f"{safe_filename(modality)}__"
        f"{safe_filename(video_name)}"
        f"__wrong_consistent_candidate_trajectories.png"
    )

    save_figure(
        fig,
        CELL4_FIG_DIR
        / figure_filename,
        dpi=FIGURE_DPI
    )

    plt.show()
    plt.close(fig)


print("\nCell 4 complete.")
print(f"Outputs saved in:\n{CELL4_DIR}")
print("\nCreated in memory:")
print("  wrong_consistent_df")
print("  wrong_but_consistent_candidates_df")
print("  wrong_consistent_video_summary_df")
print("  wrong_consistent_subject_summary_df")

print("\n" + "=" * 90)
print("ALL STAGE 1 ANALYSES COMPLETE")
print("=" * 90)
print(f"All Stage 1 outputs are located in:\n{OUTPUT_DIR}")